**Cell 1 — Pemulihan environment dan verifikasi release dataset**

In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display


# ============================================================
# CELL 1 — RECOVERY DAN READ-ONLY DATASET PREFLIGHT
# ============================================================

REPOSITORY_URL = (
    "https://github.com/May-ysaa/InvoiceFlow-AI.git"
)

REPOSITORY_ROOT = Path(
    "/content/InvoiceFlow-AI"
)

EXPECTED_CHECKPOINT_COMMIT = (
    "31c20039c48d5b07308419c52c34228cd0c61045"
)

DRIVE_ROOT = Path(
    "/content/drive/MyDrive"
)

BUILD_ROOT = (
    DRIVE_ROOT
    / "InvoiceFlow-AI-Data"
    / "interim"
    / "synthetic_v1_build"
    / "20260904T150025Z"
)

RELEASE_ROOT = (
    DRIVE_ROOT
    / "InvoiceFlow-AI-Data"
    / "releases"
    / "SYNTHETIC-INVOICE-V1"
    / "1.0.0"
)

CURRENT_RELEASE_POINTER = (
    DRIVE_ROOT
    / "InvoiceFlow-AI-Data"
    / "releases"
    / "current_release.json"
)

RELEASE_INDEX_PATH = (
    RELEASE_ROOT / "release_index.jsonl"
)

RELEASE_SUMMARY_PATH = (
    RELEASE_ROOT / "release_summary.csv"
)

RELEASE_MANIFEST_PATH = (
    RELEASE_ROOT / "release_manifest.json"
)

DATASET_CARD_PATH = (
    RELEASE_ROOT / "DATASET_CARD.md"
)

SPLIT_GUIDE_PATH = (
    RELEASE_ROOT / "SPLIT_USAGE.md"
)

FINAL_INTEGRITY_REPORT_PATH = (
    BUILD_ROOT
    / "manifests"
    / "final_integrity_report.json"
)

EXPECTED_DOCUMENT_COUNT = 200
EXPECTED_TEMPLATE_COUNT = 10

EXPECTED_TEMPLATE_IDS = {
    f"TPL-{template_number:02d}"
    for template_number in range(1, 11)
}

EXPECTED_SPLIT_DISTRIBUTION = {
    "development": 120,
    "validation": 40,
    "test": 40,
}


# ============================================================
# 1. HELPER
# ============================================================

def run_command(
    command: list[str],
    cwd: Path | None = None,
) -> subprocess.CompletedProcess:
    return subprocess.run(
        command,
        cwd=str(cwd) if cwd else None,
        capture_output=True,
        text=True,
        check=False,
    )


def load_json(
    file_path: Path,
) -> dict:
    with file_path.open(
        "r",
        encoding="utf-8",
    ) as file_handle:
        return json.load(file_handle)


def load_jsonl(
    file_path: Path,
) -> list[dict]:
    records = []

    with file_path.open(
        "r",
        encoding="utf-8",
    ) as file_handle:
        for line_number, line in enumerate(
            file_handle,
            start=1,
        ):
            stripped_line = line.strip()

            if not stripped_line:
                continue

            try:
                record = json.loads(stripped_line)
            except json.JSONDecodeError as error:
                raise RuntimeError(
                    "JSONL tidak valid pada baris "
                    f"{line_number}: {error}"
                ) from error

            if not isinstance(record, dict):
                raise RuntimeError(
                    "Record release index harus berupa "
                    f"object JSON. Baris: {line_number}"
                )

            records.append(record)

    return records


def recursive_find(
    value,
    candidate_keys: set[str],
):
    if isinstance(value, dict):
        for key, nested_value in value.items():
            if key in candidate_keys:
                return nested_value

        for nested_value in value.values():
            result = recursive_find(
                nested_value,
                candidate_keys,
            )

            if result is not None:
                return result

    elif isinstance(value, list):
        for nested_value in value:
            result = recursive_find(
                nested_value,
                candidate_keys,
            )

            if result is not None:
                return result

    return None


# ============================================================
# 2. HUBUNGKAN GOOGLE DRIVE
# ============================================================

try:
    from google.colab import drive
except ImportError as error:
    raise RuntimeError(
        "Cell ini harus dijalankan di Google Colab."
    ) from error

drive.mount(
    "/content/drive",
    force_remount=False,
)


# ============================================================
# 3. PULIHKAN REPOSITORY GITHUB
# ============================================================

if not REPOSITORY_ROOT.exists():
    clone_result = run_command(
        [
            "git",
            "clone",
            "--branch",
            "main",
            "--single-branch",
            REPOSITORY_URL,
            str(REPOSITORY_ROOT),
        ]
    )

    if clone_result.returncode != 0:
        raise RuntimeError(
            "Repository gagal di-clone:\n"
            f"{clone_result.stderr.strip()}"
        )

else:
    if not (
        REPOSITORY_ROOT / ".git"
    ).is_dir():
        raise RuntimeError(
            f"{REPOSITORY_ROOT} tersedia, tetapi bukan "
            "repository Git."
        )

    status_before_pull = run_command(
        [
            "git",
            "status",
            "--porcelain",
            "--untracked-files=all",
        ],
        cwd=REPOSITORY_ROOT,
    )

    if status_before_pull.returncode != 0:
        raise RuntimeError(
            status_before_pull.stderr.strip()
        )

    if status_before_pull.stdout.strip():
        raise RuntimeError(
            "Repository memiliki perubahan lokal. "
            "Pull dihentikan agar perubahan tidak tertimpa:\n"
            f"{status_before_pull.stdout.strip()}"
        )

    fetch_result = run_command(
        [
            "git",
            "fetch",
            "origin",
            "main",
        ],
        cwd=REPOSITORY_ROOT,
    )

    if fetch_result.returncode != 0:
        raise RuntimeError(
            "Git fetch gagal:\n"
            f"{fetch_result.stderr.strip()}"
        )

    pull_result = run_command(
        [
            "git",
            "pull",
            "--ff-only",
            "origin",
            "main",
        ],
        cwd=REPOSITORY_ROOT,
    )

    if pull_result.returncode != 0:
        raise RuntimeError(
            "Git pull gagal:\n"
            f"{pull_result.stderr.strip()}"
        )


head_result = run_command(
    ["git", "rev-parse", "HEAD"],
    cwd=REPOSITORY_ROOT,
)

branch_result = run_command(
    ["git", "branch", "--show-current"],
    cwd=REPOSITORY_ROOT,
)

checkpoint_result = run_command(
    [
        "git",
        "merge-base",
        "--is-ancestor",
        EXPECTED_CHECKPOINT_COMMIT,
        "HEAD",
    ],
    cwd=REPOSITORY_ROOT,
)

if head_result.returncode != 0:
    raise RuntimeError(
        "Tidak dapat membaca commit repository."
    )

if branch_result.returncode != 0:
    raise RuntimeError(
        "Tidak dapat membaca branch repository."
    )

repository_head = head_result.stdout.strip()
repository_branch = branch_result.stdout.strip()
checkpoint_available = (
    checkpoint_result.returncode == 0
)


# ============================================================
# 4. VERIFIKASI FILE RELEASE
# ============================================================

required_release_paths = {
    "build_root": BUILD_ROOT,
    "release_root": RELEASE_ROOT,
    "current_release_pointer": CURRENT_RELEASE_POINTER,
    "release_index": RELEASE_INDEX_PATH,
    "release_summary": RELEASE_SUMMARY_PATH,
    "release_manifest": RELEASE_MANIFEST_PATH,
    "dataset_card": DATASET_CARD_PATH,
    "split_guide": SPLIT_GUIDE_PATH,
    "final_integrity_report": FINAL_INTEGRITY_REPORT_PATH,
}

path_records = []

for path_name, file_path in required_release_paths.items():
    expected_type = (
        "directory"
        if path_name in {
            "build_root",
            "release_root",
        }
        else "file"
    )

    exists = (
        file_path.is_dir()
        if expected_type == "directory"
        else file_path.is_file()
    )

    path_records.append(
        {
            "resource": path_name,
            "expected_type": expected_type,
            "exists": exists,
            "path": str(file_path),
            "status": (
                "VALID"
                if exists
                else "INVALID"
            ),
        }
    )

missing_release_paths = [
    record["path"]
    for record in path_records
    if record["status"] != "VALID"
]

if missing_release_paths:
    display(pd.DataFrame(path_records))

    raise FileNotFoundError(
        "Sumber release belum lengkap:\n"
        + "\n".join(missing_release_paths)
    )


# ============================================================
# 5. BACA METADATA SECARA READ-ONLY
# ============================================================

release_pointer = load_json(
    CURRENT_RELEASE_POINTER
)

release_manifest = load_json(
    RELEASE_MANIFEST_PATH
)

integrity_report = load_json(
    FINAL_INTEGRITY_REPORT_PATH
)

release_records = load_jsonl(
    RELEASE_INDEX_PATH
)

release_table = pd.DataFrame(
    release_records
)

required_index_columns = {
    "canonical_invoice_id",
    "document_id",
    "template_id",
    "split",
    "language",
    "currency",
}

missing_index_columns = sorted(
    required_index_columns
    - set(release_table.columns)
)

if missing_index_columns:
    raise RuntimeError(
        "Release index tidak memiliki kolom wajib: "
        f"{missing_index_columns}"
    )


# ============================================================
# 6. VALIDASI IDENTITAS DAN DISTRIBUSI
# ============================================================

document_count = len(release_table)

unique_canonical_ids = int(
    release_table[
        "canonical_invoice_id"
    ].nunique()
)

unique_document_ids = int(
    release_table[
        "document_id"
    ].nunique()
)

actual_template_ids = set(
    release_table[
        "template_id"
    ].dropna().astype(str)
)

template_count = len(actual_template_ids)

split_distribution = (
    release_table["split"]
    .value_counts()
    .sort_index()
    .to_dict()
)

language_distribution = (
    release_table["language"]
    .value_counts()
    .sort_index()
    .to_dict()
)

currency_distribution = (
    release_table["currency"]
    .value_counts()
    .sort_index()
    .to_dict()
)

release_status = recursive_find(
    release_manifest,
    {
        "release_status",
        "status",
    },
)

integrity_status = recursive_find(
    integrity_report,
    {
        "final_status",
        "integrity_status",
        "status",
    },
)


# ============================================================
# 7. KONTROL AKHIR
# ============================================================

controls = [
    {
        "control": "repository_branch",
        "expected": "main",
        "actual": repository_branch,
    },
    {
        "control": "checkpoint_commit_available",
        "expected": True,
        "actual": checkpoint_available,
    },
    {
        "control": "release_document_count",
        "expected": EXPECTED_DOCUMENT_COUNT,
        "actual": document_count,
    },
    {
        "control": "unique_canonical_ids",
        "expected": EXPECTED_DOCUMENT_COUNT,
        "actual": unique_canonical_ids,
    },
    {
        "control": "unique_document_ids",
        "expected": EXPECTED_DOCUMENT_COUNT,
        "actual": unique_document_ids,
    },
    {
        "control": "template_count",
        "expected": EXPECTED_TEMPLATE_COUNT,
        "actual": template_count,
    },
    {
        "control": "template_ids",
        "expected": sorted(EXPECTED_TEMPLATE_IDS),
        "actual": sorted(actual_template_ids),
    },
    {
        "control": "split_distribution",
        "expected": EXPECTED_SPLIT_DISTRIBUTION,
        "actual": split_distribution,
    },
    {
        "control": "release_status",
        "expected": "FROZEN",
        "actual": release_status,
    },
    {
        "control": "integrity_status",
        "expected": "PASSED",
        "actual": integrity_status,
    },
]

for control in controls:
    control["status"] = (
        "VALID"
        if control["expected"] == control["actual"]
        else "INVALID"
    )

control_table = pd.DataFrame(controls)
path_table = pd.DataFrame(path_records)

display(control_table)
display(path_table)

print()
print(f"Repository root      : {REPOSITORY_ROOT}")
print(f"Repository branch    : {repository_branch}")
print(f"Repository HEAD      : {repository_head}")
print(f"Dataset build root   : {BUILD_ROOT}")
print(f"Dataset release root : {RELEASE_ROOT}")
print(f"Release records      : {document_count}")
print(f"Templates            : {template_count}")
print(f"Languages            : {language_distribution}")
print(f"Currencies           : {currency_distribution}")
print(f"Splits               : {split_distribution}")
print(f"Release status       : {release_status}")
print(f"Integrity status     : {integrity_status}")
print("Dataset artifact writes: 0")
print("Git commit/push        : BELUM DILAKUKAN")

invalid_controls = [
    control["control"]
    for control in controls
    if control["status"] != "VALID"
]

if invalid_controls:
    raise RuntimeError(
        "RECOVERY PREFLIGHT FAILED. "
        f"Kontrol tidak valid: {invalid_controls}"
    )

print()
print(
    "✅ CELL 1 PASSED — repository dan release dataset "
    "v1.0.0 berhasil dipulihkan serta diverifikasi "
    "secara read-only."
)
print(
    "Lanjutkan ke Cell 2 untuk memeriksa struktur "
    "artifact dan ground truth sebelum memilih OCR."
)

Mounted at /content/drive


,control,expected,actual,status
0,repository_branch,main,main,VALID
1,checkpoint_commit_available,True,True,VALID
2,release_document_count,200,200,VALID
3,unique_canonical_ids,200,200,VALID
4,unique_document_ids,200,200,VALID
5,template_count,10,10,VALID
6,template_ids,"[TPL-01, TPL-02, TPL-03, TPL-04, TPL-05, TPL-0...","[TPL-01, TPL-02, TPL-03, TPL-04, TPL-05, TPL-0...",VALID
7,split_distribution,"{'development': 120, 'validation': 40, 'test':...","{'development': 120, 'test': 40, 'validation':...",VALID
8,release_status,FROZEN,FROZEN,VALID
9,integrity_status,PASSED,PASSED,VALID


,resource,expected_type,exists,path,status
0,build_root,directory,True,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,VALID
1,release_root,directory,True,/content/drive/MyDrive/InvoiceFlow-AI-Data/rel...,VALID
2,current_release_pointer,file,True,/content/drive/MyDrive/InvoiceFlow-AI-Data/rel...,VALID
3,release_index,file,True,/content/drive/MyDrive/InvoiceFlow-AI-Data/rel...,VALID
4,release_summary,file,True,/content/drive/MyDrive/InvoiceFlow-AI-Data/rel...,VALID
5,release_manifest,file,True,/content/drive/MyDrive/InvoiceFlow-AI-Data/rel...,VALID
6,dataset_card,file,True,/content/drive/MyDrive/InvoiceFlow-AI-Data/rel...,VALID
7,split_guide,file,True,/content/drive/MyDrive/InvoiceFlow-AI-Data/rel...,VALID
8,final_integrity_report,file,True,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,VALID



Repository root      : /content/InvoiceFlow-AI
Repository branch    : main
Repository HEAD      : 31c20039c48d5b07308419c52c34228cd0c61045
Dataset build root   : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z
Dataset release root : /content/drive/MyDrive/InvoiceFlow-AI-Data/releases/SYNTHETIC-INVOICE-V1/1.0.0
Release records      : 200
Templates            : 10
Languages            : {'en': 100, 'id': 100}
Currencies           : {'EUR': 30, 'GBP': 20, 'IDR': 80, 'USD': 70}
Splits               : {'development': 120, 'test': 40, 'validation': 40}
Release status       : FROZEN
Integrity status     : PASSED
Dataset artifact writes: 0
Git commit/push        : BELUM DILAKUKAN

✅ CELL 1 PASSED — repository dan release dataset v1.0.0 berhasil dipulihkan serta diverifikasi secara read-only.
Lanjutkan ke Cell 2 untuk memeriksa struktur artifact dan ground truth sebelum memilih OCR.


** CELL 1B — MINIMAL DEPENDENCY RECOVERY**

In [ ]:
# ============================================================
# CELL 1B — MINIMAL DEPENDENCY RECOVERY
# ============================================================

import importlib
import importlib.util
import subprocess
import sys


required_packages = {
    "fitz": "PyMuPDF>=1.24,<2.0",
    "PIL": "Pillow>=10.0,<13.0",
    "pandas": "pandas>=2.2,<4.0",
}

missing_packages = [
    package_spec
    for module_name, package_spec
    in required_packages.items()
    if importlib.util.find_spec(module_name) is None
]

if missing_packages:
    print("Memasang dependency yang belum tersedia:")

    for package_spec in missing_packages:
        print(f"  - {package_spec}")

    install_result = subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            *missing_packages,
        ],
        capture_output=True,
        text=True,
        check=False,
    )

    if install_result.returncode != 0:
        print(install_result.stdout)
        print(install_result.stderr)

        raise RuntimeError(
            "Instalasi dependency gagal."
        )

    importlib.invalidate_caches()
else:
    print(
        "Seluruh dependency minimum sudah tersedia."
    )


import fitz
import pandas as pd
from PIL import Image


dependency_versions = {
    "Python": sys.version.split()[0],
    "PyMuPDF": fitz.VersionBind,
    "Pillow": Image.__version__,
    "pandas": pd.__version__,
}

print()
print("DEPENDENCY VERSIONS")

for dependency_name, version in (
    dependency_versions.items()
):
    print(
        f"{dependency_name:<10}: {version}"
    )

print()
print("Artifact writes : 0")
print("Dataset changes : 0")
print()
print(
    "✅ CELL 1B PASSED — dependency minimum untuk "
    "inspeksi notebook 02 siap."
)
print(
    "Sekarang jalankan kembali Cell 2."
)

Memasang dependency yang belum tersedia:
  - PyMuPDF>=1.24,<2.0

DEPENDENCY VERSIONS
Python    : 3.13.15
PyMuPDF   : 1.28.2
Pillow    : 11.3.0
pandas    : 2.2.3

Artifact writes : 0
Dataset changes : 0

✅ CELL 1B PASSED — dependency minimum untuk inspeksi notebook 02 siap.
Sekarang jalankan kembali Cell 2.


**Cell 2 — Inspeksi artifact, PDF, dan ground truth**

In [ ]:
from __future__ import annotations

import json
import re
from collections import Counter
from pathlib import Path

import fitz
import pandas as pd
from PIL import Image
from IPython.display import display


# ============================================================
# CELL 2 — READ-ONLY ARTIFACT DAN GROUND-TRUTH INSPECTION
# ============================================================

REQUIRED_VARIABLES = [
    "BUILD_ROOT",
    "RELEASE_ROOT",
    "release_records",
    "release_table",
]

missing_variables = [
    variable_name
    for variable_name in REQUIRED_VARIABLES
    if variable_name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Cell 1 harus dijalankan terlebih dahulu. "
        f"Variabel belum tersedia: {missing_variables}"
    )


DATA_PROJECT_ROOT = (
    Path(BUILD_ROOT).parents[2]
)

RENDERED_DATASET_ROOT = (
    Path(BUILD_ROOT) / "rendered_dataset"
)

EXPECTED_DEVELOPMENT_RECORDS = 120
EXPECTED_DEVELOPMENT_TEMPLATES = 6
EXPECTED_PAGE_COUNT = 1

EXPECTED_DEVELOPMENT_TEMPLATE_IDS = {
    f"TPL-{template_number:02d}"
    for template_number in range(1, 7)
}

ARTIFACT_ROLES = (
    "pdf",
    "preview",
    "ground_truth",
)


# ============================================================
# 1. HELPER
# ============================================================

def normalize_text(
    text: str,
) -> str:
    return re.sub(
        r"\s+",
        " ",
        str(text),
    ).strip()


def load_json_object(
    file_path: Path,
) -> dict:
    with file_path.open(
        "r",
        encoding="utf-8",
    ) as file_handle:
        value = json.load(file_handle)

    if not isinstance(value, dict):
        raise RuntimeError(
            "JSON harus berupa object: "
            f"{file_path}"
        )

    return value


def resolve_release_artifact(
    record: dict,
    artifact_role: str,
) -> tuple[Path, dict]:
    artifacts = record.get("artifacts")

    if not isinstance(artifacts, dict):
        raise RuntimeError(
            "Object artifacts tidak tersedia."
        )

    artifact_metadata = artifacts.get(
        artifact_role
    )

    if not isinstance(
        artifact_metadata,
        dict,
    ):
        raise RuntimeError(
            f"Metadata artifact {artifact_role} "
            "tidak tersedia."
        )

    relative_path_value = (
        artifact_metadata.get(
            "project_relative_path"
        )
    )

    if not isinstance(
        relative_path_value,
        str,
    ) or not relative_path_value.strip():
        raise RuntimeError(
            "project_relative_path tidak tersedia "
            f"untuk {artifact_role}."
        )

    relative_path = Path(
        relative_path_value
    )

    if relative_path.is_absolute():
        raise RuntimeError(
            "project_relative_path tidak boleh "
            f"berupa path absolut: {relative_path}"
        )

    resolved_path = (
        DATA_PROJECT_ROOT / relative_path
    ).resolve()

    try:
        resolved_path.relative_to(
            DATA_PROJECT_ROOT.resolve()
        )
    except ValueError as error:
        raise RuntimeError(
            "Artifact berada di luar data project root: "
            f"{resolved_path}"
        ) from error

    if not resolved_path.is_file():
        raise FileNotFoundError(
            f"Artifact {artifact_role} tidak ditemukan: "
            f"{resolved_path}"
        )

    return resolved_path, artifact_metadata


def find_annotation_list(
    value,
):
    if isinstance(value, dict):
        annotations = value.get(
            "annotations"
        )

        if isinstance(annotations, list):
            return annotations

        for nested_value in value.values():
            result = find_annotation_list(
                nested_value
            )

            if result is not None:
                return result

    elif isinstance(value, list):
        for nested_value in value:
            result = find_annotation_list(
                nested_value
            )

            if result is not None:
                return result

    return None


def classify_page_size(
    width_pt: float,
    height_pt: float,
) -> str:
    dimensions = sorted(
        [float(width_pt), float(height_pt)]
    )

    short_side = dimensions[0]
    long_side = dimensions[1]

    if (
        abs(short_side - 595.28) <= 5
        and abs(long_side - 841.89) <= 5
    ):
        return "A4"

    if (
        abs(short_side - 612.0) <= 5
        and abs(long_side - 792.0) <= 5
    ):
        return "LETTER"

    return (
        f"OTHER_"
        f"{short_side:.1f}x{long_side:.1f}"
    )


def preview_value(
    value,
    maximum_length: int = 100,
):
    if isinstance(
        value,
        (dict, list),
    ):
        rendered_value = json.dumps(
            value,
            ensure_ascii=False,
        )
    else:
        rendered_value = str(value)

    if len(rendered_value) > maximum_length:
        return (
            rendered_value[
                :maximum_length
            ]
            + "..."
        )

    return rendered_value


# ============================================================
# 2. VALIDASI DATA PROJECT ROOT
# ============================================================

if DATA_PROJECT_ROOT.name != (
    "InvoiceFlow-AI-Data"
):
    raise RuntimeError(
        "Data project root tidak sesuai. "
        f"Ditemukan: {DATA_PROJECT_ROOT}"
    )

if not RENDERED_DATASET_ROOT.is_dir():
    raise FileNotFoundError(
        "Rendered dataset root tidak ditemukan: "
        f"{RENDERED_DATASET_ROOT}"
    )


# ============================================================
# 3. BATASI KE DEVELOPMENT SPLIT
# ============================================================

development_records = [
    record
    for record in release_records
    if record.get("split") == "development"
]

development_template_ids = {
    str(record.get("template_id"))
    for record in development_records
}

nondevelopment_artifacts_opened = 0


# ============================================================
# 4. INSPEKSI ARTIFACT DEVELOPMENT
# ============================================================

inspection_records = []
inspection_errors = []

artifact_size_mismatches = []
identity_mismatches = []
annotation_count_mismatches = []

ground_truth_top_level_keys = Counter()
annotation_keys = Counter()

sample_ground_truth_structure = None
sample_annotation_structure = None

for sequence_number, record in enumerate(
    development_records,
    start=1,
):
    document_id = str(
        record.get("document_id", "")
    )

    canonical_invoice_id = str(
        record.get(
            "canonical_invoice_id",
            "",
        )
    )

    template_id = str(
        record.get("template_id", "")
    )

    inspection_row = {
        "document_id": document_id,
        "template_id": template_id,
        "pdf_resolved": False,
        "preview_resolved": False,
        "ground_truth_resolved": False,
        "pdf_pages": None,
        "page_size": None,
        "pdf_width_pt": None,
        "pdf_height_pt": None,
        "native_text_characters": None,
        "native_word_count": None,
        "native_block_count": None,
        "embedded_image_count": None,
        "preview_width_px": None,
        "preview_height_px": None,
        "preview_mode": None,
        "annotation_count": None,
        "recorded_annotation_count": (
            record.get("annotation_count")
        ),
    }

    try:
        resolved_artifacts = {}

        for artifact_role in ARTIFACT_ROLES:
            (
                artifact_path,
                artifact_metadata,
            ) = resolve_release_artifact(
                record,
                artifact_role,
            )

            resolved_artifacts[
                artifact_role
            ] = artifact_path

            inspection_row[
                f"{artifact_role}_resolved"
            ] = True

            recorded_size = (
                artifact_metadata.get(
                    "size_bytes"
                )
            )

            actual_size = (
                artifact_path.stat().st_size
            )

            if (
                recorded_size is not None
                and int(recorded_size)
                != actual_size
            ):
                artifact_size_mismatches.append(
                    {
                        "document_id": document_id,
                        "artifact_role": artifact_role,
                        "expected_size": int(
                            recorded_size
                        ),
                        "actual_size": actual_size,
                        "path": str(
                            artifact_path
                        ),
                    }
                )

        pdf_path = resolved_artifacts["pdf"]
        preview_path = resolved_artifacts[
            "preview"
        ]
        ground_truth_path = (
            resolved_artifacts[
                "ground_truth"
            ]
        )

        # ----------------------------------------------------
        # PDF
        # ----------------------------------------------------

        with fitz.open(pdf_path) as pdf:
            page_count = len(pdf)
            extracted_text_parts = []
            native_word_count = 0
            native_block_count = 0
            embedded_image_count = 0

            first_page_width = None
            first_page_height = None

            for page_index, page in enumerate(
                pdf
            ):
                if page_index == 0:
                    first_page_width = float(
                        page.rect.width
                    )
                    first_page_height = float(
                        page.rect.height
                    )

                extracted_text_parts.append(
                    page.get_text("text")
                )

                native_word_count += len(
                    page.get_text("words")
                )

                native_block_count += len(
                    page.get_text("blocks")
                )

                embedded_image_count += len(
                    page.get_images(full=True)
                )

        normalized_native_text = normalize_text(
            "\n".join(extracted_text_parts)
        )

        inspection_row.update(
            {
                "pdf_pages": page_count,
                "page_size": classify_page_size(
                    first_page_width,
                    first_page_height,
                ),
                "pdf_width_pt": round(
                    first_page_width,
                    2,
                ),
                "pdf_height_pt": round(
                    first_page_height,
                    2,
                ),
                "native_text_characters": len(
                    normalized_native_text
                ),
                "native_word_count": (
                    native_word_count
                ),
                "native_block_count": (
                    native_block_count
                ),
                "embedded_image_count": (
                    embedded_image_count
                ),
            }
        )

        # ----------------------------------------------------
        # PREVIEW
        # ----------------------------------------------------

        with Image.open(preview_path) as image:
            inspection_row.update(
                {
                    "preview_width_px": int(
                        image.width
                    ),
                    "preview_height_px": int(
                        image.height
                    ),
                    "preview_mode": image.mode,
                }
            )

        # ----------------------------------------------------
        # GROUND TRUTH
        # ----------------------------------------------------

        ground_truth = load_json_object(
            ground_truth_path
        )

        ground_truth_top_level_keys.update(
            ground_truth.keys()
        )

        annotations = find_annotation_list(
            ground_truth
        )

        if annotations is None:
            raise RuntimeError(
                "Daftar annotations tidak ditemukan "
                f"pada {ground_truth_path.name}."
            )

        annotation_count = len(
            annotations
        )

        inspection_row[
            "annotation_count"
        ] = annotation_count

        recorded_annotation_count = (
            record.get("annotation_count")
        )

        if (
            recorded_annotation_count is not None
            and annotation_count
            != int(recorded_annotation_count)
        ):
            annotation_count_mismatches.append(
                {
                    "document_id": document_id,
                    "ground_truth": (
                        annotation_count
                    ),
                    "release_index": int(
                        recorded_annotation_count
                    ),
                }
            )

        ground_truth_document_id = (
            ground_truth.get("document_id")
        )

        ground_truth_canonical_id = (
            ground_truth.get(
                "canonical_invoice_id"
            )
        )

        ground_truth_template_id = (
            ground_truth.get("template_id")
        )

        identity_checks = {
            "document_id": (
                document_id,
                ground_truth_document_id,
            ),
            "canonical_invoice_id": (
                canonical_invoice_id,
                ground_truth_canonical_id,
            ),
            "template_id": (
                template_id,
                ground_truth_template_id,
            ),
        }

        for identity_name, (
            expected_identity,
            actual_identity,
        ) in identity_checks.items():
            if (
                actual_identity is not None
                and str(actual_identity)
                != expected_identity
            ):
                identity_mismatches.append(
                    {
                        "document_id": document_id,
                        "identity": identity_name,
                        "expected": (
                            expected_identity
                        ),
                        "actual": str(
                            actual_identity
                        ),
                    }
                )

        for annotation in annotations:
            if isinstance(annotation, dict):
                annotation_keys.update(
                    annotation.keys()
                )

        if sample_ground_truth_structure is None:
            sample_ground_truth_structure = {
                "document_id": document_id,
                "template_id": template_id,
                "top_level_keys": sorted(
                    ground_truth.keys()
                ),
                "annotation_count": (
                    annotation_count
                ),
            }

            first_annotation = next(
                (
                    annotation
                    for annotation in annotations
                    if isinstance(
                        annotation,
                        dict,
                    )
                ),
                None,
            )

            if first_annotation is not None:
                sample_annotation_structure = {
                    "keys": sorted(
                        first_annotation.keys()
                    ),
                    "value_types": {
                        key: type(value).__name__
                        for key, value
                        in first_annotation.items()
                    },
                    "sample_values": {
                        key: preview_value(value)
                        for key, value
                        in first_annotation.items()
                    },
                }

    except Exception as error:
        inspection_errors.append(
            {
                "document_id": document_id,
                "template_id": template_id,
                "error_type": (
                    type(error).__name__
                ),
                "message": str(error)[:500],
            }
        )

    inspection_records.append(
        inspection_row
    )

    if sequence_number % 20 == 0:
        print(
            f"[{sequence_number:03d}/"
            f"{len(development_records):03d}] "
            f"errors={len(inspection_errors)}, "
            f"size_mismatch="
            f"{len(artifact_size_mismatches)}"
        )


inspection_table = pd.DataFrame(
    inspection_records
)


# ============================================================
# 5. PASTIKAN TIPE NUMERIK
# ============================================================

numeric_columns = [
    "pdf_pages",
    "pdf_width_pt",
    "pdf_height_pt",
    "native_text_characters",
    "native_word_count",
    "native_block_count",
    "embedded_image_count",
    "preview_width_px",
    "preview_height_px",
    "annotation_count",
    "recorded_annotation_count",
]

for column_name in numeric_columns:
    inspection_table[column_name] = (
        pd.to_numeric(
            inspection_table[column_name],
            errors="coerce",
        )
    )


# ============================================================
# 6. RINGKASAN PER TEMPLATE
# ============================================================

template_summary_records = []

for template_id in sorted(
    development_template_ids
):
    template_rows = inspection_table[
        inspection_table["template_id"]
        == template_id
    ].copy()

    native_available = int(
        (
            template_rows[
                "native_text_characters"
            ].fillna(0)
            > 0
        ).sum()
    )

    page_sizes = sorted(
        {
            str(value)
            for value in template_rows[
                "page_size"
            ].dropna()
        }
    )

    template_summary_records.append(
        {
            "template_id": template_id,
            "documents": len(template_rows),
            "page_sizes": ",".join(
                page_sizes
            ),
            "native_text_pdfs": (
                native_available
            ),
            "native_text_ratio": round(
                native_available
                / len(template_rows),
                4,
            ),
            "minimum_text_characters": int(
                template_rows[
                    "native_text_characters"
                ].min()
            ),
            "mean_text_characters": round(
                float(
                    template_rows[
                        "native_text_characters"
                    ].mean()
                ),
                2,
            ),
            "maximum_text_characters": int(
                template_rows[
                    "native_text_characters"
                ].max()
            ),
            "mean_native_words": round(
                float(
                    template_rows[
                        "native_word_count"
                    ].mean()
                ),
                2,
            ),
            "maximum_embedded_images": int(
                template_rows[
                    "embedded_image_count"
                ].max()
            ),
            "preview_dimensions": sorted(
                {
                    (
                        f"{int(row.preview_width_px)}"
                        f"x"
                        f"{int(row.preview_height_px)}"
                    )
                    for row in template_rows.itertuples()
                }
            ),
            "minimum_annotations": int(
                template_rows[
                    "annotation_count"
                ].min()
            ),
            "maximum_annotations": int(
                template_rows[
                    "annotation_count"
                ].max()
            ),
            "status": (
                "VALID"
                if (
                    len(template_rows) == 20
                    and template_rows[
                        "pdf_pages"
                    ].eq(
                        EXPECTED_PAGE_COUNT
                    ).all()
                )
                else "REVIEW"
            ),
        }
    )

template_summary = pd.DataFrame(
    template_summary_records
)


# ============================================================
# 7. OBSERVASI NATIVE TEXT
# ============================================================

native_text_pdf_count = int(
    (
        inspection_table[
            "native_text_characters"
        ].fillna(0)
        > 0
    ).sum()
)

image_only_pdf_count = int(
    (
        inspection_table[
            "native_text_characters"
        ].fillna(0)
        == 0
    ).sum()
)

native_text_ratio = (
    native_text_pdf_count
    / len(development_records)
    if development_records
    else 0.0
)

single_page_pdf_count = int(
    inspection_table[
        "pdf_pages"
    ].eq(
        EXPECTED_PAGE_COUNT
    ).sum()
)

resolved_pdf_count = int(
    inspection_table[
        "pdf_resolved"
    ].sum()
)

resolved_preview_count = int(
    inspection_table[
        "preview_resolved"
    ].sum()
)

resolved_ground_truth_count = int(
    inspection_table[
        "ground_truth_resolved"
    ].sum()
)


# ============================================================
# 8. KONTROL AKHIR
# ============================================================

controls = [
    {
        "control": "development_records",
        "expected": (
            EXPECTED_DEVELOPMENT_RECORDS
        ),
        "actual": len(
            development_records
        ),
    },
    {
        "control": "development_templates",
        "expected": (
            EXPECTED_DEVELOPMENT_TEMPLATES
        ),
        "actual": len(
            development_template_ids
        ),
    },
    {
        "control": "development_template_ids",
        "expected": sorted(
            EXPECTED_DEVELOPMENT_TEMPLATE_IDS
        ),
        "actual": sorted(
            development_template_ids
        ),
    },
    {
        "control": "resolved_pdf_files",
        "expected": (
            EXPECTED_DEVELOPMENT_RECORDS
        ),
        "actual": resolved_pdf_count,
    },
    {
        "control": "resolved_preview_files",
        "expected": (
            EXPECTED_DEVELOPMENT_RECORDS
        ),
        "actual": resolved_preview_count,
    },
    {
        "control": "resolved_ground_truth_files",
        "expected": (
            EXPECTED_DEVELOPMENT_RECORDS
        ),
        "actual": (
            resolved_ground_truth_count
        ),
    },
    {
        "control": "single_page_pdfs",
        "expected": (
            EXPECTED_DEVELOPMENT_RECORDS
        ),
        "actual": (
            single_page_pdf_count
        ),
    },
    {
        "control": "artifact_size_mismatches",
        "expected": 0,
        "actual": len(
            artifact_size_mismatches
        ),
    },
    {
        "control": "identity_mismatches",
        "expected": 0,
        "actual": len(
            identity_mismatches
        ),
    },
    {
        "control": "annotation_count_mismatches",
        "expected": 0,
        "actual": len(
            annotation_count_mismatches
        ),
    },
    {
        "control": "artifact_read_errors",
        "expected": 0,
        "actual": len(
            inspection_errors
        ),
    },
    {
        "control": "nondevelopment_artifacts_opened",
        "expected": 0,
        "actual": (
            nondevelopment_artifacts_opened
        ),
    },
]

for control in controls:
    control["status"] = (
        "VALID"
        if control["expected"]
        == control["actual"]
        else "INVALID"
    )

control_table = pd.DataFrame(
    controls
)

display(control_table)
display(template_summary)


# ============================================================
# 9. DETAIL KEGAGALAN
# ============================================================

if artifact_size_mismatches:
    print("\nARTIFACT SIZE MISMATCHES")
    display(
        pd.DataFrame(
            artifact_size_mismatches
        )
    )

if identity_mismatches:
    print("\nIDENTITY MISMATCHES")
    display(
        pd.DataFrame(
            identity_mismatches
        )
    )

if annotation_count_mismatches:
    print("\nANNOTATION COUNT MISMATCHES")
    display(
        pd.DataFrame(
            annotation_count_mismatches
        )
    )

if inspection_errors:
    print("\nARTIFACT READ ERRORS")
    display(
        pd.DataFrame(
            inspection_errors
        )
    )


# ============================================================
# 10. TAMPILKAN STRUKTUR DATA
# ============================================================

print()
print("=" * 78)
print("RELEASE INDEX STRUCTURE")
print("=" * 78)
print(
    json.dumps(
        {
            "top_level_keys": sorted(
                release_records[0].keys()
            ),
            "artifact_roles": sorted(
                release_records[0][
                    "artifacts"
                ].keys()
            ),
            "artifact_path_base": str(
                DATA_PROJECT_ROOT
            ),
        },
        indent=2,
        ensure_ascii=False,
    )
)

print()
print("=" * 78)
print("GROUND-TRUTH STRUCTURE")
print("=" * 78)
print(
    json.dumps(
        {
            "sample": (
                sample_ground_truth_structure
            ),
            "annotation_sample": (
                sample_annotation_structure
            ),
            "observed_top_level_keys": sorted(
                ground_truth_top_level_keys.keys()
            ),
            "observed_annotation_keys": sorted(
                annotation_keys.keys()
            ),
        },
        indent=2,
        ensure_ascii=False,
    )
)


# ============================================================
# 11. HASIL
# ============================================================

print()
print(f"Data project root     : {DATA_PROJECT_ROOT}")
print(f"Rendered dataset root : {RENDERED_DATASET_ROOT}")
print(f"Development documents : {len(development_records)}")
print(
    "Development templates : "
    f"{sorted(development_template_ids)}"
)
print(f"Native-text PDFs       : {native_text_pdf_count}")
print(f"Image-only PDFs        : {image_only_pdf_count}")
print(f"Native-text ratio      : {native_text_ratio:.4f}")
print(f"Inspection errors      : {len(inspection_errors)}")
print("Validation opened      : 0")
print("Test opened            : 0")
print("Artifact writes        : 0")
print("OCR engine selected    : BELUM")

invalid_controls = [
    control["control"]
    for control in controls
    if control["status"] != "VALID"
]

if invalid_controls:
    raise RuntimeError(
        "ARTIFACT INSPECTION FAILED. "
        f"Kontrol tidak valid: {invalid_controls}"
    )

print()

if native_text_ratio >= 0.95:
    print(
        "TEMUAN AWAL: mayoritas PDF development "
        "memiliki native text yang dapat diekstrak."
    )
    print(
        "Ekstraksi native PDF layak diuji sebagai "
        "baseline pertama. OCR tetap diperlukan "
        "sebagai fallback untuk scan atau gambar."
    )

elif native_text_ratio > 0:
    print(
        "TEMUAN AWAL: hanya sebagian PDF development "
        "memiliki native text."
    )
    print(
        "Arsitektur hibrida native extraction + OCR "
        "perlu diuji pada tahap berikutnya."
    )

else:
    print(
        "TEMUAN AWAL: PDF development tidak memiliki "
        "native text yang dapat digunakan."
    )
    print(
        "OCR perlu menjadi jalur utama ekstraksi."
    )

print()
print(
    "✅ CELL 2 PASSED — struktur PDF, preview, dan "
    "ground truth development selesai diperiksa "
    "secara read-only."
)
print(
    "Kirim seluruh output sebelum masuk ke Cell 3 "
    "untuk merancang baseline dan benchmark OCR."
)

[020/120] errors=0, size_mismatch=0
[040/120] errors=0, size_mismatch=0
[060/120] errors=0, size_mismatch=0
[080/120] errors=0, size_mismatch=0
[100/120] errors=0, size_mismatch=0
[120/120] errors=0, size_mismatch=0


,control,expected,actual,status
0,development_records,120,120,VALID
1,development_templates,6,6,VALID
2,development_template_ids,"[TPL-01, TPL-02, TPL-03, TPL-04, TPL-05, TPL-06]","[TPL-01, TPL-02, TPL-03, TPL-04, TPL-05, TPL-06]",VALID
3,resolved_pdf_files,120,120,VALID
4,resolved_preview_files,120,120,VALID
5,resolved_ground_truth_files,120,120,VALID
6,single_page_pdfs,120,120,VALID
7,artifact_size_mismatches,0,0,VALID
8,identity_mismatches,0,0,VALID
9,annotation_count_mismatches,0,0,VALID


,template_id,documents,page_sizes,native_text_pdfs,native_text_ratio,minimum_text_characters,mean_text_characters,maximum_text_characters,mean_native_words,maximum_embedded_images,preview_dimensions,minimum_annotations,maximum_annotations,status
0,TPL-01,20,A4,20,1.0,804,1007.35,1178,142.05,0,[1241x1754],27,51,VALID
1,TPL-02,20,A4,20,1.0,806,984.70,1113,139.75,0,[1241x1754],27,47,VALID
2,TPL-03,20,LETTER,20,1.0,767,919.45,1086,129.95,0,[1275x1650],27,47,VALID
3,TPL-04,20,A4,20,1.0,825,964.20,1184,135.80,0,[1241x1754],27,51,VALID
4,TPL-05,20,A4,20,1.0,786,982.30,1179,140.25,0,[1241x1754],27,51,VALID
5,TPL-06,20,LETTER,20,1.0,836,1012.10,1171,143.85,0,[1275x1650],27,51,VALID



RELEASE INDEX STRUCTURE
{
  "top_level_keys": [
    "annotation_count",
    "artifacts",
    "canonical_invoice_id",
    "currency",
    "document_id",
    "item_count",
    "language",
    "manual_visual_review",
    "page_count",
    "sequence_number",
    "split",
    "technical_status",
    "template_id"
  ],
  "artifact_roles": [
    "ground_truth",
    "pdf",
    "preview",
    "qa_report"
  ],
  "artifact_path_base": "/content/drive/MyDrive/InvoiceFlow-AI-Data"
}

GROUND-TRUTH STRUCTURE
{
  "sample": {
    "document_id": "INV-SYN-000001",
    "template_id": "TPL-01",
    "top_level_keys": [
      "annotations",
      "canonical",
      "dataset_id",
      "document",
      "rendering",
      "schema_version"
    ],
    "annotation_count": 27
  },
  "annotation_sample": {
    "keys": [
      "annotation_type",
      "bbox_normalized",
      "bbox_points",
      "field_name",
      "page_number",
      "text"
    ],
    "value_types": {
      "field_name": "str",
      "text": "s

**Cell 3 — Desain baseline dan sampel benchmark**

In [ ]:
from __future__ import annotations

import json
import re
from collections import Counter, defaultdict

import pandas as pd
from IPython.display import display


# ============================================================
# CELL 3 — BASELINE DESIGN DAN BENCHMARK SAMPLE SELECTION
# READ-ONLY: DEVELOPMENT SPLIT ONLY
# ============================================================

REQUIRED_VARIABLES = [
    "development_records",
    "inspection_table",
    "resolve_release_artifact",
    "load_json_object",
]

missing_variables = [
    variable_name
    for variable_name in REQUIRED_VARIABLES
    if variable_name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Cell 2 harus dijalankan terlebih dahulu. "
        f"Variabel belum tersedia: {missing_variables}"
    )


EXPECTED_DEVELOPMENT_RECORDS = 120
EXPECTED_DEVELOPMENT_TEMPLATES = 6
EXPECTED_SAMPLES_PER_TEMPLATE = 3
EXPECTED_BENCHMARK_SAMPLES = 18
EXPECTED_LANGUAGES = {"id", "en"}
EXPECTED_ITEM_FIELD_PATTERNS = 4


# ============================================================
# 1. HELPER
# ============================================================

def normalize_field_pattern(
    field_name: str,
) -> str:
    normalized_name = str(
        field_name
    ).strip()

    normalized_name = re.sub(
        r"items\[\d+\]\.",
        "items.*.",
        normalized_name,
    )

    normalized_name = re.sub(
        r"items\.\d+\.",
        "items.*.",
        normalized_name,
    )

    return normalized_name


def determine_field_category(
    field_pattern: str,
) -> str:
    if field_pattern.startswith("items.*."):
        return "line_item"

    if field_pattern.startswith("vendor."):
        return "vendor"

    if field_pattern.startswith("buyer."):
        return "buyer"

    if field_pattern.startswith("financials."):
        return "financial"

    if field_pattern in {
        "invoice_number",
        "invoice_date",
        "due_date",
        "currency",
    }:
        return "metadata"

    return "other"


def safe_text_length(
    value,
) -> int:
    if value is None:
        return 0

    return len(
        re.sub(
            r"\s+",
            " ",
            str(value),
        ).strip()
    )


def choose_row(
    candidate_table: pd.DataFrame,
    excluded_document_ids: set[str],
    preferred_language: str | None = None,
):
    available_rows = candidate_table[
        ~candidate_table[
            "document_id"
        ].isin(excluded_document_ids)
    ].copy()

    if available_rows.empty:
        return None

    if preferred_language is not None:
        preferred_rows = available_rows[
            available_rows["language"]
            == preferred_language
        ]

        if not preferred_rows.empty:
            available_rows = preferred_rows

    available_rows = available_rows.sort_values(
        by=[
            "text_risk_score",
            "document_id",
        ],
        ascending=[
            False,
            True,
        ],
    )

    return available_rows.iloc[0]


# ============================================================
# 2. INVENTARISASI GROUND TRUTH DEVELOPMENT
# ============================================================

field_document_frequency = Counter()
field_annotation_frequency = Counter()
field_text_lengths = defaultdict(list)

annotation_type_frequency = Counter()
canonical_top_level_keys = Counter()
ground_truth_top_level_keys = Counter()

document_schema_records = []
ground_truth_errors = []

nondevelopment_artifacts_opened = 0

inspection_lookup = (
    inspection_table
    .set_index("document_id")
    .to_dict(orient="index")
)

for sequence_number, record in enumerate(
    development_records,
    start=1,
):
    document_id = str(
        record.get("document_id", "")
    )

    template_id = str(
        record.get("template_id", "")
    )

    language = str(
        record.get("language", "")
    )

    currency = str(
        record.get("currency", "")
    )

    item_count = int(
        record.get("item_count", 0)
    )

    try:
        (
            ground_truth_path,
            _,
        ) = resolve_release_artifact(
            record,
            "ground_truth",
        )

        ground_truth = load_json_object(
            ground_truth_path
        )

        ground_truth_top_level_keys.update(
            ground_truth.keys()
        )

        canonical = ground_truth.get(
            "canonical"
        )

        if not isinstance(canonical, dict):
            raise RuntimeError(
                "Object canonical tidak ditemukan."
            )

        canonical_top_level_keys.update(
            canonical.keys()
        )

        annotations = ground_truth.get(
            "annotations"
        )

        if not isinstance(annotations, list):
            raise RuntimeError(
                "Daftar annotations tidak ditemukan."
            )

        document_patterns = set()
        maximum_annotation_text_length = 0
        total_annotation_text_length = 0
        item_annotation_count = 0

        for annotation in annotations:
            if not isinstance(annotation, dict):
                raise RuntimeError(
                    "Annotation harus berupa object."
                )

            field_name = str(
                annotation.get(
                    "field_name",
                    "",
                )
            ).strip()

            if not field_name:
                raise RuntimeError(
                    "Annotation memiliki field_name kosong."
                )

            field_pattern = (
                normalize_field_pattern(
                    field_name
                )
            )

            annotation_text = annotation.get(
                "text",
                "",
            )

            text_length = safe_text_length(
                annotation_text
            )

            annotation_type = str(
                annotation.get(
                    "annotation_type",
                    "",
                )
            )

            document_patterns.add(
                field_pattern
            )

            field_annotation_frequency[
                field_pattern
            ] += 1

            field_text_lengths[
                field_pattern
            ].append(text_length)

            annotation_type_frequency[
                annotation_type
            ] += 1

            maximum_annotation_text_length = max(
                maximum_annotation_text_length,
                text_length,
            )

            total_annotation_text_length += (
                text_length
            )

            if field_pattern.startswith(
                "items.*."
            ):
                item_annotation_count += 1

        for field_pattern in document_patterns:
            field_document_frequency[
                field_pattern
            ] += 1

        native_text_characters = int(
            inspection_lookup.get(
                document_id,
                {},
            ).get(
                "native_text_characters",
                0,
            )
        )

        base_annotation_count = (
            len(annotations)
            - item_annotation_count
        )

        item_annotations_per_item = (
            item_annotation_count / item_count
            if item_count > 0
            else None
        )

        text_risk_score = (
            maximum_annotation_text_length
            + (item_count * 10)
            + (
                total_annotation_text_length
                / 100
            )
        )

        document_schema_records.append(
            {
                "document_id": document_id,
                "template_id": template_id,
                "language": language,
                "currency": currency,
                "item_count": item_count,
                "annotation_count": len(
                    annotations
                ),
                "base_annotation_count": (
                    base_annotation_count
                ),
                "item_annotation_count": (
                    item_annotation_count
                ),
                "item_annotations_per_item": (
                    item_annotations_per_item
                ),
                "maximum_annotation_text_length": (
                    maximum_annotation_text_length
                ),
                "total_annotation_text_length": (
                    total_annotation_text_length
                ),
                "native_text_characters": (
                    native_text_characters
                ),
                "text_risk_score": round(
                    text_risk_score,
                    4,
                ),
            }
        )

    except Exception as error:
        ground_truth_errors.append(
            {
                "document_id": document_id,
                "template_id": template_id,
                "error_type": (
                    type(error).__name__
                ),
                "message": str(error)[:500],
            }
        )

    if sequence_number % 20 == 0:
        print(
            f"[{sequence_number:03d}/"
            f"{len(development_records):03d}] "
            f"errors={len(ground_truth_errors)}"
        )


document_schema_table = pd.DataFrame(
    document_schema_records
)


# ============================================================
# 3. FIELD INVENTORY
# ============================================================

field_inventory_records = []

for field_pattern in sorted(
    field_annotation_frequency
):
    text_lengths = field_text_lengths[
        field_pattern
    ]

    document_frequency = (
        field_document_frequency[
            field_pattern
        ]
    )

    field_inventory_records.append(
        {
            "field_pattern": field_pattern,
            "category": (
                determine_field_category(
                    field_pattern
                )
            ),
            "documents_present": (
                document_frequency
            ),
            "document_coverage": round(
                document_frequency
                / EXPECTED_DEVELOPMENT_RECORDS,
                4,
            ),
            "total_annotations": (
                field_annotation_frequency[
                    field_pattern
                ]
            ),
            "minimum_text_length": min(
                text_lengths
            ),
            "mean_text_length": round(
                sum(text_lengths)
                / len(text_lengths),
                2,
            ),
            "maximum_text_length": max(
                text_lengths
            ),
            "repeated_field": (
                field_pattern.startswith(
                    "items.*."
                )
            ),
            "evaluation_level": (
                "line_item"
                if field_pattern.startswith(
                    "items.*."
                )
                else "document"
            ),
        }
    )

field_inventory_table = pd.DataFrame(
    field_inventory_records
)

item_field_patterns = sorted(
    field_inventory_table.loc[
        field_inventory_table[
            "category"
        ] == "line_item",
        "field_pattern",
    ].tolist()
)

document_field_patterns = sorted(
    field_inventory_table.loc[
        field_inventory_table[
            "category"
        ] != "line_item",
        "field_pattern",
    ].tolist()
)


# ============================================================
# 4. PERIKSA POLA ANOTASI
# ============================================================

base_annotation_counts = sorted(
    {
        int(value)
        for value in document_schema_table[
            "base_annotation_count"
        ].dropna()
    }
)

item_annotations_per_item_values = sorted(
    {
        float(value)
        for value in document_schema_table[
            "item_annotations_per_item"
        ].dropna()
    }
)

annotation_formula_consistent = (
    len(base_annotation_counts) == 1
    and item_annotations_per_item_values
    == [4.0]
)


# ============================================================
# 5. PILIH 3 SAMPEL PER TEMPLATE
# ============================================================

benchmark_selection_records = []

for template_id in sorted(
    document_schema_table[
        "template_id"
    ].unique()
):
    template_rows = (
        document_schema_table[
            document_schema_table[
                "template_id"
            ] == template_id
        ]
        .copy()
        .reset_index(drop=True)
    )

    selected_document_ids = set()
    selected_rows = []

    minimum_item_count = int(
        template_rows[
            "item_count"
        ].min()
    )

    minimum_item_rows = template_rows[
        template_rows["item_count"]
        == minimum_item_count
    ]

    minimum_row = choose_row(
        minimum_item_rows,
        selected_document_ids,
        preferred_language="id",
    )

    if minimum_row is not None:
        selected_rows.append(
            (
                minimum_row,
                "minimum_items",
            )
        )

        selected_document_ids.add(
            str(minimum_row["document_id"])
        )

    maximum_item_count = int(
        template_rows[
            "item_count"
        ].max()
    )

    maximum_item_rows = template_rows[
        template_rows["item_count"]
        == maximum_item_count
    ]

    maximum_row = choose_row(
        maximum_item_rows,
        selected_document_ids,
        preferred_language="en",
    )

    if maximum_row is not None:
        selected_rows.append(
            (
                maximum_row,
                "maximum_items",
            )
        )

        selected_document_ids.add(
            str(maximum_row["document_id"])
        )

    selected_languages = {
        str(row["language"])
        for row, _ in selected_rows
    }

    missing_languages = (
        EXPECTED_LANGUAGES
        - selected_languages
    )

    if missing_languages:
        preferred_language = sorted(
            missing_languages
        )[0]

        risk_row = choose_row(
            template_rows,
            selected_document_ids,
            preferred_language=(
                preferred_language
            ),
        )

        risk_reason = (
            "language_coverage_high_risk"
        )

    else:
        risk_row = choose_row(
            template_rows,
            selected_document_ids,
        )

        risk_reason = "highest_text_risk"

    if risk_row is not None:
        selected_rows.append(
            (
                risk_row,
                risk_reason,
            )
        )

        selected_document_ids.add(
            str(risk_row["document_id"])
        )

    if len(selected_rows) != (
        EXPECTED_SAMPLES_PER_TEMPLATE
    ):
        raise RuntimeError(
            "Gagal memilih tiga sampel untuk "
            f"{template_id}."
        )

    for sample_rank, (
        selected_row,
        selection_reason,
    ) in enumerate(
        selected_rows,
        start=1,
    ):
        benchmark_selection_records.append(
            {
                "template_id": template_id,
                "sample_rank": sample_rank,
                "document_id": str(
                    selected_row[
                        "document_id"
                    ]
                ),
                "selection_reason": (
                    selection_reason
                ),
                "language": str(
                    selected_row["language"]
                ),
                "currency": str(
                    selected_row["currency"]
                ),
                "item_count": int(
                    selected_row["item_count"]
                ),
                "annotation_count": int(
                    selected_row[
                        "annotation_count"
                    ]
                ),
                "maximum_text_length": int(
                    selected_row[
                        "maximum_annotation_text_length"
                    ]
                ),
                "text_risk_score": float(
                    selected_row[
                        "text_risk_score"
                    ]
                ),
            }
        )

ocr_benchmark_selection = pd.DataFrame(
    benchmark_selection_records
)


# ============================================================
# 6. VALIDASI SAMPEL BENCHMARK
# ============================================================

samples_per_template = (
    ocr_benchmark_selection
    .groupby("template_id")
    .size()
    .to_dict()
)

languages_per_template = (
    ocr_benchmark_selection
    .groupby("template_id")[
        "language"
    ]
    .nunique()
    .to_dict()
)

benchmark_document_ids = set(
    ocr_benchmark_selection[
        "document_id"
    ]
)

all_benchmark_ids_in_development = (
    benchmark_document_ids.issubset(
        set(
            document_schema_table[
                "document_id"
            ]
        )
    )
)


# ============================================================
# 7. ARSITEKTUR BASELINE
# ============================================================

baseline_architecture = {
    "routing": {
        "native_text_pdf": (
            "PyMuPDF native text extraction"
        ),
        "scan_or_image": (
            "OCR fallback selected by benchmark"
        ),
    },
    "field_localization": (
        "spatial words, bounding boxes, labels, "
        "and layout rules"
    ),
    "structured_extraction": {
        "document_fields": (
            document_field_patterns
        ),
        "line_item_fields": (
            item_field_patterns
        ),
    },
    "normalization_stage": (
        "deferred to notebook "
        "04_normalization_and_validation.ipynb"
    ),
    "development_policy": (
        "Only development artifacts may be used "
        "for implementation and tuning"
    ),
    "validation_policy": (
        "Locked until the development baseline "
        "is frozen"
    ),
    "test_policy": (
        "Locked until final evaluation"
    ),
}

evaluation_protocol = pd.DataFrame(
    [
        {
            "stage": "text_acquisition_native",
            "input": "development PDF",
            "scope": "120 documents",
            "method": "PyMuPDF text + word boxes",
            "metrics": (
                "CER, WER, normalized exact match"
            ),
        },
        {
            "stage": "text_acquisition_ocr",
            "input": "development preview",
            "scope": "18 stratified samples",
            "method": (
                "OCR candidates under identical input"
            ),
            "metrics": (
                "CER, WER, latency, failure rate"
            ),
        },
        {
            "stage": "field_extraction",
            "input": "text + spatial coordinates",
            "scope": "120 development documents",
            "method": (
                "label anchors and layout rules"
            ),
            "metrics": (
                "precision, recall, F1, exact match"
            ),
        },
        {
            "stage": "line_item_extraction",
            "input": "table regions",
            "scope": "120 development documents",
            "method": (
                "row and column reconstruction"
            ),
            "metrics": (
                "row exact match, cell F1, CER"
            ),
        },
        {
            "stage": "validation",
            "input": "validation split",
            "scope": "40 locked documents",
            "method": (
                "frozen development pipeline"
            ),
            "metrics": (
                "field F1, exact match, error rate"
            ),
        },
        {
            "stage": "final_test",
            "input": "test split",
            "scope": "40 locked documents",
            "method": (
                "final frozen pipeline"
            ),
            "metrics": (
                "field F1, exact match, error rate"
            ),
        },
    ]
)


# ============================================================
# 8. KONTROL AKHIR
# ============================================================

controls = [
    {
        "control": "development_documents_read",
        "expected": (
            EXPECTED_DEVELOPMENT_RECORDS
        ),
        "actual": len(
            document_schema_table
        ),
    },
    {
        "control": "ground_truth_errors",
        "expected": 0,
        "actual": len(
            ground_truth_errors
        ),
    },
    {
        "control": "development_templates",
        "expected": (
            EXPECTED_DEVELOPMENT_TEMPLATES
        ),
        "actual": int(
            document_schema_table[
                "template_id"
            ].nunique()
        ),
    },
    {
        "control": "item_field_patterns",
        "expected": (
            EXPECTED_ITEM_FIELD_PATTERNS
        ),
        "actual": len(
            item_field_patterns
        ),
    },
    {
        "control": "annotation_formula_consistent",
        "expected": True,
        "actual": (
            annotation_formula_consistent
        ),
    },
    {
        "control": "benchmark_samples",
        "expected": (
            EXPECTED_BENCHMARK_SAMPLES
        ),
        "actual": len(
            ocr_benchmark_selection
        ),
    },
    {
        "control": "samples_per_template",
        "expected": True,
        "actual": all(
            sample_count
            == EXPECTED_SAMPLES_PER_TEMPLATE
            for sample_count
            in samples_per_template.values()
        ),
    },
    {
        "control": "languages_per_template",
        "expected": True,
        "actual": all(
            language_count == 2
            for language_count
            in languages_per_template.values()
        ),
    },
    {
        "control": "benchmark_ids_in_development",
        "expected": True,
        "actual": (
            all_benchmark_ids_in_development
        ),
    },
    {
        "control": "nondevelopment_artifacts_opened",
        "expected": 0,
        "actual": (
            nondevelopment_artifacts_opened
        ),
    },
]

for control in controls:
    control["status"] = (
        "VALID"
        if control["expected"]
        == control["actual"]
        else "INVALID"
    )

control_table = pd.DataFrame(
    controls
)


# ============================================================
# 9. OUTPUT
# ============================================================

display(control_table)

print()
print("FIELD INVENTORY")
display(field_inventory_table)

print()
print("OCR BENCHMARK SELECTION")
display(ocr_benchmark_selection)

print()
print("EVALUATION PROTOCOL")
display(evaluation_protocol)

if ground_truth_errors:
    print()
    print("GROUND-TRUTH ERRORS")
    display(
        pd.DataFrame(
            ground_truth_errors
        )
    )

print()
print("BASELINE ARCHITECTURE")
print(
    json.dumps(
        baseline_architecture,
        indent=2,
        ensure_ascii=False,
    )
)

print()
print("GROUND-TRUTH SUMMARY")
print(
    json.dumps(
        {
            "ground_truth_top_level_keys": (
                sorted(
                    ground_truth_top_level_keys.keys()
                )
            ),
            "canonical_top_level_keys": (
                sorted(
                    canonical_top_level_keys.keys()
                )
            ),
            "annotation_types": dict(
                sorted(
                    annotation_type_frequency.items()
                )
            ),
            "base_annotation_counts": (
                base_annotation_counts
            ),
            "item_annotations_per_item": (
                item_annotations_per_item_values
            ),
            "document_field_count": len(
                document_field_patterns
            ),
            "item_field_count": len(
                item_field_patterns
            ),
        },
        indent=2,
        ensure_ascii=False,
    )
)

print()
print(
    f"Development documents : "
    f"{len(document_schema_table)}"
)
print(
    f"Document field patterns: "
    f"{len(document_field_patterns)}"
)
print(
    f"Item field patterns    : "
    f"{len(item_field_patterns)}"
)
print(
    f"Benchmark samples      : "
    f"{len(ocr_benchmark_selection)}"
)
print(
    f"Templates covered      : "
    f"{ocr_benchmark_selection['template_id'].nunique()}"
)
print(
    f"Languages covered      : "
    f"{sorted(ocr_benchmark_selection['language'].unique())}"
)
print("Validation opened      : 0")
print("Test opened            : 0")
print("Artifact writes        : 0")
print("OCR engine selected    : BELUM")

invalid_controls = [
    control["control"]
    for control in controls
    if control["status"] != "VALID"
]

if invalid_controls:
    raise RuntimeError(
        "BASELINE DESIGN FAILED. "
        f"Kontrol tidak valid: {invalid_controls}"
    )

print()
print(
    "✅ CELL 3 PASSED — target field, arsitektur "
    "baseline, protokol evaluasi, dan 18 sampel "
    "benchmark development berhasil ditetapkan."
)
print(
    "Belum ada OCR yang dipasang atau artifact "
    "benchmark yang ditulis."
)

[020/120] errors=0
[040/120] errors=0
[060/120] errors=0
[080/120] errors=0
[100/120] errors=0
[120/120] errors=0


,control,expected,actual,status
0,development_documents_read,120,120,VALID
1,ground_truth_errors,0,0,VALID
2,development_templates,6,6,VALID
3,item_field_patterns,4,4,VALID
4,annotation_formula_consistent,True,True,VALID
5,benchmark_samples,18,18,VALID
6,samples_per_template,True,True,VALID
7,languages_per_template,True,True,VALID
8,benchmark_ids_in_development,True,True,VALID
9,nondevelopment_artifacts_opened,0,0,VALID



FIELD INVENTORY


,field_pattern,category,documents_present,document_coverage,total_annotations,minimum_text_length,mean_text_length,maximum_text_length,repeated_field,evaluation_level
0,buyer.address_lines,buyer,120,1.0,120,54,60.17,68,False,document
1,buyer.email,buyer,120,1.0,120,25,25.00,25,False,document
2,buyer.name,buyer,120,1.0,120,24,29.41,35,False,document
3,buyer.phone,buyer,120,1.0,120,11,14.00,17,False,document
4,buyer.tax_identifier,buyer,120,1.0,120,25,26.00,27,False,document
5,currency,metadata,120,1.0,120,3,3.00,3,False,document
6,document.synthetic_notice,other,120,1.0,120,45,46.50,48,False,document
7,due_date,metadata,120,1.0,120,10,14.32,18,False,document
8,financials.discount,financial,120,1.0,120,5,9.68,14,False,document
9,financials.subtotal,financial,120,1.0,120,12,13.55,15,False,document



OCR BENCHMARK SELECTION


,template_id,sample_rank,document_id,selection_reason,language,currency,item_count,annotation_count,maximum_text_length,text_risk_score
0,TPL-01,1,INV-SYN-000002,minimum_items,id,USD,2,27,60,85.44
1,TPL-01,2,INV-SYN-000015,maximum_items,en,USD,8,51,68,156.73
2,TPL-01,3,INV-SYN-000013,highest_text_risk,en,USD,8,51,68,156.49
3,TPL-02,1,INV-SYN-000025,minimum_items,id,IDR,2,27,60,85.71
4,TPL-02,2,INV-SYN-000036,maximum_items,en,EUR,7,47,68,146.30
5,TPL-02,3,INV-SYN-000037,highest_text_risk,en,USD,7,47,68,146.13
6,TPL-03,1,INV-SYN-000043,minimum_items,id,IDR,2,27,60,85.60
7,TPL-03,2,INV-SYN-000060,maximum_items,en,USD,7,47,68,146.36
8,TPL-03,3,INV-SYN-000052,highest_text_risk,en,GBP,7,47,68,146.23
9,TPL-04,1,INV-SYN-000070,minimum_items,id,IDR,2,27,57,82.50



EVALUATION PROTOCOL


,stage,input,scope,method,metrics
0,text_acquisition_native,development PDF,120 documents,PyMuPDF text + word boxes,"CER, WER, normalized exact match"
1,text_acquisition_ocr,development preview,18 stratified samples,OCR candidates under identical input,"CER, WER, latency, failure rate"
2,field_extraction,text + spatial coordinates,120 development documents,label anchors and layout rules,"precision, recall, F1, exact match"
3,line_item_extraction,table regions,120 development documents,row and column reconstruction,"row exact match, cell F1, CER"
4,validation,validation split,40 locked documents,frozen development pipeline,"field F1, exact match, error rate"
5,final_test,test split,40 locked documents,final frozen pipeline,"field F1, exact match, error rate"



BASELINE ARCHITECTURE
{
  "routing": {
    "native_text_pdf": "PyMuPDF native text extraction",
    "scan_or_image": "OCR fallback selected by benchmark"
  },
  "field_localization": "spatial words, bounding boxes, labels, and layout rules",
  "structured_extraction": {
    "document_fields": [
      "buyer.address_lines",
      "buyer.email",
      "buyer.name",
      "buyer.phone",
      "buyer.tax_identifier",
      "currency",
      "document.synthetic_notice",
      "due_date",
      "financials.discount",
      "financials.subtotal",
      "financials.tax",
      "financials.total",
      "invoice_date",
      "invoice_number",
      "vendor.address_lines",
      "vendor.email",
      "vendor.name",
      "vendor.phone",
      "vendor.tax_identifier"
    ],
    "line_item_fields": [
      "items.*.description",
      "items.*.line_total",
      "items.*.quantity",
      "items.*.unit_price"
    ]
  },
  "normalization_stage": "deferred to notebook 04_normalization_and_validation

**Cell 4 — OCR runtime compatibility preflight**

In [ ]:
from __future__ import annotations

import importlib.util
import os
import platform
import shutil
import subprocess
import sys
from importlib import metadata
from pathlib import Path

import pandas as pd
from IPython.display import display


# ============================================================
# CELL 4 — OCR RUNTIME COMPATIBILITY PREFLIGHT
# ============================================================

SUPPORTED_PYTHON_MINIMUM = (3, 9)
SUPPORTED_PYTHON_MAXIMUM = (3, 13)

SUPPORTED_ARCHITECTURES = {
    "x86_64",
    "amd64",
}

MINIMUM_FREE_DISK_GB = 5.0
MINIMUM_RAM_GB = 8.0


# ============================================================
# 1. HELPER
# ============================================================

def run_command(
    command: list[str],
) -> subprocess.CompletedProcess:
    return subprocess.run(
        command,
        capture_output=True,
        text=True,
        check=False,
    )


def installed_version(
    distribution_name: str,
) -> str | None:
    try:
        return metadata.version(
            distribution_name
        )
    except metadata.PackageNotFoundError:
        return None


def module_available(
    module_name: str,
) -> bool:
    return (
        importlib.util.find_spec(
            module_name
        )
        is not None
    )


def get_ram_gb() -> float:
    meminfo_path = Path(
        "/proc/meminfo"
    )

    if not meminfo_path.is_file():
        return 0.0

    meminfo_text = meminfo_path.read_text(
        encoding="utf-8"
    )

    for line in meminfo_text.splitlines():
        if line.startswith("MemTotal:"):
            memory_kb = int(
                line.split()[1]
            )

            return memory_kb / 1024 / 1024

    return 0.0


# ============================================================
# 2. INFORMASI PYTHON DAN SISTEM
# ============================================================

python_version_tuple = (
    sys.version_info.major,
    sys.version_info.minor,
)

python_version = (
    f"{sys.version_info.major}."
    f"{sys.version_info.minor}."
    f"{sys.version_info.micro}"
)

python_supported = (
    SUPPORTED_PYTHON_MINIMUM
    <= python_version_tuple
    <= SUPPORTED_PYTHON_MAXIMUM
)

architecture = platform.machine().lower()

architecture_supported = (
    architecture
    in SUPPORTED_ARCHITECTURES
)

is_64_bit = (
    platform.architecture()[0]
    == "64bit"
)

disk_usage = shutil.disk_usage(
    "/content"
)

free_disk_gb = (
    disk_usage.free
    / 1024
    / 1024
    / 1024
)

total_ram_gb = get_ram_gb()


# ============================================================
# 3. PERIKSA GPU
# ============================================================

nvidia_smi_path = shutil.which(
    "nvidia-smi"
)

gpu_available = False
gpu_description = "NOT AVAILABLE"

if nvidia_smi_path:
    gpu_result = run_command(
        [
            nvidia_smi_path,
            "--query-gpu=name,memory.total,"
            "driver_version",
            "--format=csv,noheader",
        ]
    )

    if (
        gpu_result.returncode == 0
        and gpu_result.stdout.strip()
    ):
        gpu_available = True
        gpu_description = (
            gpu_result.stdout.strip()
        )


# ============================================================
# 4. PERIKSA PACKAGE PYTHON
# ============================================================

package_specs = [
    {
        "component": "PyMuPDF",
        "distribution": "PyMuPDF",
        "module": "fitz",
        "purpose": (
            "native PDF text extraction"
        ),
    },
    {
        "component": "Pillow",
        "distribution": "Pillow",
        "module": "PIL",
        "purpose": (
            "image loading and inspection"
        ),
    },
    {
        "component": "pandas",
        "distribution": "pandas",
        "module": "pandas",
        "purpose": (
            "benchmark tables"
        ),
    },
    {
        "component": "PaddlePaddle",
        "distribution": "paddlepaddle",
        "module": "paddle",
        "purpose": (
            "PaddleOCR inference engine"
        ),
    },
    {
        "component": "PaddleOCR",
        "distribution": "paddleocr",
        "module": "paddleocr",
        "purpose": (
            "primary local OCR candidate"
        ),
    },
    {
        "component": "pytesseract",
        "distribution": "pytesseract",
        "module": "pytesseract",
        "purpose": (
            "Python wrapper for Tesseract"
        ),
    },
    {
        "component": "OpenCV",
        "distribution": (
            "opencv-python-headless"
        ),
        "module": "cv2",
        "purpose": (
            "image preprocessing"
        ),
    },
]

package_records = []

for package_spec in package_specs:
    distribution_name = (
        package_spec["distribution"]
    )

    module_name = package_spec["module"]

    version = installed_version(
        distribution_name
    )

    available = module_available(
        module_name
    )

    package_records.append(
        {
            "component": (
                package_spec["component"]
            ),
            "distribution": (
                distribution_name
            ),
            "version": (
                version
                if version is not None
                else "NOT INSTALLED"
            ),
            "module_available": available,
            "purpose": (
                package_spec["purpose"]
            ),
            "status": (
                "AVAILABLE"
                if available
                else "INSTALL REQUIRED"
            ),
        }
    )

package_table = pd.DataFrame(
    package_records
)


# ============================================================
# 5. PERIKSA TESSERACT SYSTEM BINARY
# ============================================================

tesseract_path = shutil.which(
    "tesseract"
)

tesseract_available = (
    tesseract_path is not None
)

tesseract_version = "NOT INSTALLED"
tesseract_languages = []

if tesseract_available:
    version_result = run_command(
        [
            tesseract_path,
            "--version",
        ]
    )

    if version_result.returncode == 0:
        first_version_line = (
            version_result.stdout
            or version_result.stderr
        ).splitlines()

        if first_version_line:
            tesseract_version = (
                first_version_line[0]
            )

    language_result = run_command(
        [
            tesseract_path,
            "--list-langs",
        ]
    )

    if language_result.returncode == 0:
        tesseract_languages = [
            line.strip()
            for line
            in language_result.stdout.splitlines()
            if (
                line.strip()
                and not line.lower().startswith(
                    "list of available languages"
                )
            )
        ]

tesseract_language_status = {
    "eng": "AVAILABLE"
    if "eng" in tesseract_languages
    else "INSTALL REQUIRED",
    "ind": "AVAILABLE"
    if "ind" in tesseract_languages
    else "INSTALL REQUIRED",
}


# ============================================================
# 6. DATASET STATE CHECK
# ============================================================

required_runtime_variables = [
    "development_records",
    "inspection_table",
    "field_inventory_table",
    "ocr_benchmark_selection",
    "baseline_architecture",
    "evaluation_protocol",
]

missing_runtime_variables = [
    variable_name
    for variable_name
    in required_runtime_variables
    if variable_name not in globals()
]

dataset_state_ready = (
    len(missing_runtime_variables) == 0
)


# ============================================================
# 7. INSTALLATION PLAN
# ============================================================

installation_plan = pd.DataFrame(
    [
        {
            "priority": 1,
            "component": "PyMuPDF",
            "role": (
                "native-text baseline"
            ),
            "execution": "LOCAL",
            "api_key": "NOT REQUIRED",
            "installation_needed": (
                not module_available("fitz")
            ),
        },
        {
            "priority": 2,
            "component": "PaddleOCR",
            "role": (
                "primary OCR candidate"
            ),
            "execution": "LOCAL",
            "api_key": "NOT REQUIRED",
            "installation_needed": (
                not (
                    module_available("paddle")
                    and module_available(
                        "paddleocr"
                    )
                )
            ),
        },
        {
            "priority": 3,
            "component": "Tesseract",
            "role": (
                "comparison OCR baseline"
            ),
            "execution": "LOCAL",
            "api_key": "NOT REQUIRED",
            "installation_needed": (
                not tesseract_available
                or "eng"
                not in tesseract_languages
                or "ind"
                not in tesseract_languages
            ),
        },
    ]
)


# ============================================================
# 8. KONTROL AKHIR
# ============================================================

controls = [
    {
        "control": "python_supported",
        "expected": True,
        "actual": python_supported,
    },
    {
        "control": "64_bit_runtime",
        "expected": True,
        "actual": is_64_bit,
    },
    {
        "control": "supported_architecture",
        "expected": True,
        "actual": (
            architecture_supported
        ),
    },
    {
        "control": "minimum_free_disk_gb",
        "expected": (
            f">={MINIMUM_FREE_DISK_GB}"
        ),
        "actual": round(
            free_disk_gb,
            2,
        ),
        "valid": (
            free_disk_gb
            >= MINIMUM_FREE_DISK_GB
        ),
    },
    {
        "control": "minimum_ram_gb",
        "expected": (
            f">={MINIMUM_RAM_GB}"
        ),
        "actual": round(
            total_ram_gb,
            2,
        ),
        "valid": (
            total_ram_gb
            >= MINIMUM_RAM_GB
        ),
    },
    {
        "control": "pymupdf_available",
        "expected": True,
        "actual": module_available(
            "fitz"
        ),
    },
    {
        "control": "pillow_available",
        "expected": True,
        "actual": module_available(
            "PIL"
        ),
    },
    {
        "control": "dataset_state_ready",
        "expected": True,
        "actual": dataset_state_ready,
    },
    {
        "control": "external_api_required",
        "expected": False,
        "actual": False,
    },
]

for control in controls:
    if "valid" in control:
        is_valid = control.pop("valid")
    else:
        is_valid = (
            control["expected"]
            == control["actual"]
        )

    control["status"] = (
        "VALID"
        if is_valid
        else "INVALID"
    )

control_table = pd.DataFrame(
    controls
)


# ============================================================
# 9. OUTPUT
# ============================================================

display(control_table)

print()
print("PYTHON PACKAGES")
display(package_table)

print()
print("INSTALLATION PLAN")
display(installation_plan)

print()
print("RUNTIME")
print(f"Python             : {python_version}")
print(f"Executable         : {sys.executable}")
print(f"Architecture       : {architecture}")
print(f"64-bit             : {is_64_bit}")
print(f"Total RAM          : {total_ram_gb:.2f} GB")
print(f"Free disk          : {free_disk_gb:.2f} GB")
print(f"GPU available      : {gpu_available}")
print(f"GPU                : {gpu_description}")

print()
print("TESSERACT")
print(f"Binary             : {tesseract_path}")
print(f"Version            : {tesseract_version}")
print(f"Languages          : {tesseract_languages}")
print(
    "Required languages : "
    f"{tesseract_language_status}"
)

print()
print(
    "Missing state vars : "
    f"{missing_runtime_variables}"
)
print("Validation opened  : 0")
print("Test opened        : 0")
print("Dataset writes     : 0")
print("API key required   : NO")
print("OCR installation   : BELUM DILAKUKAN")

invalid_controls = [
    control["control"]
    for control in controls
    if control["status"] != "VALID"
]

if invalid_controls:
    raise RuntimeError(
        "OCR PREFLIGHT FAILED. "
        f"Kontrol tidak valid: {invalid_controls}"
    )

print()
print(
    "✅ CELL 4 PASSED — runtime kompatibel untuk "
    "baseline native extraction dan pemeriksaan "
    "instalasi OCR lokal."
)
print(
    "Kirim seluruh output sebelum menjalankan "
    "instalasi PaddleOCR atau Tesseract."
)

,control,expected,actual,status
0,python_supported,True,True,VALID
1,64_bit_runtime,True,True,VALID
2,supported_architecture,True,True,VALID
3,minimum_free_disk_gb,>=5.0,87.47,VALID
4,minimum_ram_gb,>=8.0,12.67,VALID
5,pymupdf_available,True,True,VALID
6,pillow_available,True,True,VALID
7,dataset_state_ready,True,True,VALID
8,external_api_required,False,False,VALID



PYTHON PACKAGES


,component,distribution,version,module_available,purpose,status
0,PyMuPDF,PyMuPDF,1.28.2,True,native PDF text extraction,AVAILABLE
1,Pillow,Pillow,11.3.0,True,image loading and inspection,AVAILABLE
2,pandas,pandas,2.2.3,True,benchmark tables,AVAILABLE
3,PaddlePaddle,paddlepaddle,NOT INSTALLED,False,PaddleOCR inference engine,INSTALL REQUIRED
4,PaddleOCR,paddleocr,NOT INSTALLED,False,primary local OCR candidate,INSTALL REQUIRED
5,pytesseract,pytesseract,NOT INSTALLED,False,Python wrapper for Tesseract,INSTALL REQUIRED
6,OpenCV,opencv-python-headless,5.0.0.93,True,image preprocessing,AVAILABLE



INSTALLATION PLAN


,priority,component,role,execution,api_key,installation_needed
0,1,PyMuPDF,native-text baseline,LOCAL,NOT REQUIRED,False
1,2,PaddleOCR,primary OCR candidate,LOCAL,NOT REQUIRED,True
2,3,Tesseract,comparison OCR baseline,LOCAL,NOT REQUIRED,True



RUNTIME
Python             : 3.13.15
Executable         : /usr/bin/python3
Architecture       : x86_64
64-bit             : True
Total RAM          : 12.67 GB
Free disk          : 87.47 GB
GPU available      : False
GPU                : NOT AVAILABLE

TESSERACT
Binary             : /usr/bin/tesseract
Version            : tesseract 4.1.1
Languages          : ['eng', 'osd']
Required languages : {'eng': 'AVAILABLE', 'ind': 'INSTALL REQUIRED'}

Missing state vars : []
Validation opened  : 0
Test opened        : 0
Dataset writes     : 0
API key required   : NO
OCR installation   : BELUM DILAKUKAN

✅ CELL 4 PASSED — runtime kompatibel untuk baseline native extraction dan pemeriksaan instalasi OCR lokal.
Kirim seluruh output sebelum menjalankan instalasi PaddleOCR atau Tesseract.


**Cell 5A — Instalasi dan validasi Tesseract lokal**

In [ ]:
from __future__ import annotations

import importlib
import importlib.util
import shutil
import subprocess
import sys
from importlib import metadata
from pathlib import Path

import pandas as pd
from IPython.display import display
from PIL import Image, ImageDraw, ImageFont


# ============================================================
# CELL 5A — INSTALL DAN VALIDATE TESSERACT OCR
# LOCAL EXECUTION, NO API KEY
# ============================================================

REQUIRED_TESSERACT_LANGUAGES = {
    "eng",
    "ind",
}


# ============================================================
# 1. HELPER
# ============================================================

def run_command(
    command: list[str],
) -> subprocess.CompletedProcess:
    return subprocess.run(
        command,
        capture_output=True,
        text=True,
        check=False,
    )


def get_tesseract_languages(
    tesseract_binary: str,
) -> set[str]:
    result = run_command(
        [
            tesseract_binary,
            "--list-langs",
        ]
    )

    if result.returncode != 0:
        raise RuntimeError(
            "Gagal membaca bahasa Tesseract:\n"
            f"{result.stderr.strip()}"
        )

    return {
        line.strip()
        for line in result.stdout.splitlines()
        if (
            line.strip()
            and not line.lower().startswith(
                "list of available languages"
            )
        )
    }


# ============================================================
# 2. PERIKSA BINARY AWAL
# ============================================================

tesseract_binary = shutil.which(
    "tesseract"
)

if tesseract_binary is None:
    raise FileNotFoundError(
        "Binary Tesseract tidak ditemukan."
    )

languages_before = get_tesseract_languages(
    tesseract_binary
)

environment_changes = []


# ============================================================
# 3. TAMBAHKAN MODEL BAHASA INDONESIA
# ============================================================

if "ind" not in languages_before:
    print(
        "Model bahasa Indonesia belum tersedia."
    )
    print(
        "Memasang tesseract-ocr-ind..."
    )

    apt_update_result = run_command(
        [
            "apt-get",
            "update",
            "-qq",
        ]
    )

    if apt_update_result.returncode != 0:
        print(
            apt_update_result.stdout[-2000:]
        )
        print(
            apt_update_result.stderr[-2000:]
        )

        raise RuntimeError(
            "apt-get update gagal."
        )

    apt_install_result = run_command(
        [
            "apt-get",
            "install",
            "-y",
            "-qq",
            "tesseract-ocr-ind",
        ]
    )

    if apt_install_result.returncode != 0:
        print(
            apt_install_result.stdout[-3000:]
        )
        print(
            apt_install_result.stderr[-3000:]
        )

        raise RuntimeError(
            "Instalasi tesseract-ocr-ind gagal."
        )

    environment_changes.append(
        "tesseract-ocr-ind"
    )
else:
    print(
        "Model bahasa Indonesia sudah tersedia."
    )


# ============================================================
# 4. PASANG PYTESSERACT
# ============================================================

if importlib.util.find_spec(
    "pytesseract"
) is None:
    print(
        "Memasang wrapper Python pytesseract..."
    )

    pip_result = run_command(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--disable-pip-version-check",
            "pytesseract>=0.3.13,<0.4",
        ]
    )

    if pip_result.returncode != 0:
        print(pip_result.stdout[-3000:])
        print(pip_result.stderr[-3000:])

        raise RuntimeError(
            "Instalasi pytesseract gagal."
        )

    environment_changes.append(
        "pytesseract"
    )
else:
    print(
        "pytesseract sudah tersedia."
    )

importlib.invalidate_caches()

import pytesseract


# ============================================================
# 5. VALIDASI INSTALASI
# ============================================================

languages_after = get_tesseract_languages(
    tesseract_binary
)

version_result = run_command(
    [
        tesseract_binary,
        "--version",
    ]
)

version_lines = (
    version_result.stdout
    or version_result.stderr
).splitlines()

tesseract_version = (
    version_lines[0]
    if version_lines
    else "UNKNOWN"
)

try:
    pytesseract_version = metadata.version(
        "pytesseract"
    )
except metadata.PackageNotFoundError:
    pytesseract_version = "UNKNOWN"


# ============================================================
# 6. SMOKE TEST DALAM MEMORI
# ============================================================

test_image = Image.new(
    "RGB",
    (720, 120),
    color="white",
)

draw = ImageDraw.Draw(
    test_image
)

font_path = Path(
    "/usr/share/fonts/truetype/dejavu/"
    "DejaVuSans.ttf"
)

if font_path.is_file():
    test_font = ImageFont.truetype(
        str(font_path),
        size=40,
    )
else:
    test_font = ImageFont.load_default()

test_text = "INVOICE 12345"

draw.text(
    (25, 30),
    test_text,
    fill="black",
    font=test_font,
)

try:
    smoke_test_output = (
        pytesseract.image_to_string(
            test_image,
            lang="eng",
            config="--oem 1 --psm 7",
        ).strip()
    )

    smoke_test_executed = True

except Exception as error:
    smoke_test_output = (
        f"{type(error).__name__}: {error}"
    )

    smoke_test_executed = False

smoke_test_nonempty = bool(
    smoke_test_output.strip()
)

smoke_test_contains_invoice = (
    "INVOICE"
    in smoke_test_output.upper()
)


# ============================================================
# 7. KONTROL AKHIR
# ============================================================

controls = [
    {
        "control": "tesseract_binary",
        "expected": True,
        "actual": (
            tesseract_binary is not None
        ),
    },
    {
        "control": "english_model",
        "expected": True,
        "actual": (
            "eng" in languages_after
        ),
    },
    {
        "control": "indonesian_model",
        "expected": True,
        "actual": (
            "ind" in languages_after
        ),
    },
    {
        "control": "pytesseract_import",
        "expected": True,
        "actual": (
            importlib.util.find_spec(
                "pytesseract"
            )
            is not None
        ),
    },
    {
        "control": "smoke_test_executed",
        "expected": True,
        "actual": (
            smoke_test_executed
        ),
    },
    {
        "control": "smoke_test_nonempty",
        "expected": True,
        "actual": (
            smoke_test_nonempty
        ),
    },
    {
        "control": "smoke_test_invoice",
        "expected": True,
        "actual": (
            smoke_test_contains_invoice
        ),
    },
    {
        "control": "external_api_required",
        "expected": False,
        "actual": False,
    },
]

for control in controls:
    control["status"] = (
        "VALID"
        if control["expected"]
        == control["actual"]
        else "INVALID"
    )

control_table = pd.DataFrame(
    controls
)

display(control_table)

print()
print(f"Tesseract binary   : {tesseract_binary}")
print(f"Tesseract version  : {tesseract_version}")
print(f"pytesseract version: {pytesseract_version}")
print(
    "Languages before  : "
    f"{sorted(languages_before)}"
)
print(
    "Languages after   : "
    f"{sorted(languages_after)}"
)
print(
    "Environment changes: "
    f"{environment_changes}"
)
print(f"Smoke-test input   : {test_text}")
print(
    "Smoke-test output  : "
    f"{smoke_test_output!r}"
)
print("Dataset opened     : 0")
print("Dataset writes     : 0")
print("API key required   : NO")
print("PaddleOCR installed: NO")

invalid_controls = [
    control["control"]
    for control in controls
    if control["status"] != "VALID"
]

if invalid_controls:
    raise RuntimeError(
        "TESSERACT SETUP FAILED. "
        f"Kontrol tidak valid: {invalid_controls}"
    )

print()
print(
    "✅ CELL 5A PASSED — Tesseract OCR lokal "
    "dan bahasa Inggris/Indonesia siap digunakan."
)
print(
    "Kirim seluruh output sebelum memasang "
    "PaddleOCR CPU pada Cell 5B."
)

Model bahasa Indonesia belum tersedia.
Memasang tesseract-ocr-ind...
Memasang wrapper Python pytesseract...


,control,expected,actual,status
0,tesseract_binary,True,True,VALID
1,english_model,True,True,VALID
2,indonesian_model,True,True,VALID
3,pytesseract_import,True,True,VALID
4,smoke_test_executed,True,True,VALID
5,smoke_test_nonempty,True,True,VALID
6,smoke_test_invoice,True,True,VALID
7,external_api_required,False,False,VALID



Tesseract binary   : /usr/bin/tesseract
Tesseract version  : tesseract 4.1.1
pytesseract version: 0.3.13
Languages before  : ['eng', 'osd']
Languages after   : ['eng', 'ind', 'osd']
Environment changes: ['tesseract-ocr-ind', 'pytesseract']
Smoke-test input   : INVOICE 12345
Smoke-test output  : 'INVOICE 12845'
Dataset opened     : 0
Dataset writes     : 0
API key required   : NO
PaddleOCR installed: NO

✅ CELL 5A PASSED — Tesseract OCR lokal dan bahasa Inggris/Indonesia siap digunakan.
Kirim seluruh output sebelum memasang PaddleOCR CPU pada Cell 5B.


**Cell 5B — Instalasi PaddleOCR CPU**

In [ ]:
from __future__ import annotations

import importlib
import importlib.util
import subprocess
import sys
from importlib import metadata

import pandas as pd
from IPython.display import display


# ============================================================
# CELL 5B — INSTALL DAN VALIDATE PADDLEOCR CPU
# LOCAL EXECUTION, NO API KEY
# ============================================================

PADDLEPADDLE_VERSION = "3.3.0"

PADDLEPADDLE_INDEX_URL = (
    "https://www.paddlepaddle.org.cn/"
    "packages/stable/cpu/"
)

PADDLEOCR_SPECIFICATION = (
    "paddleocr>=3.0,<4.0"
)


# ============================================================
# 1. HELPER
# ============================================================

def run_command(
    command: list[str],
) -> subprocess.CompletedProcess:
    return subprocess.run(
        command,
        capture_output=True,
        text=True,
        check=False,
    )


def distribution_version(
    distribution_name: str,
) -> str | None:
    try:
        return metadata.version(
            distribution_name
        )
    except metadata.PackageNotFoundError:
        return None


def module_available(
    module_name: str,
) -> bool:
    return (
        importlib.util.find_spec(
            module_name
        )
        is not None
    )


def show_command_failure(
    result: subprocess.CompletedProcess,
    maximum_characters: int = 5000,
) -> None:
    if result.stdout.strip():
        print("\nSTDOUT")
        print(
            result.stdout[
                -maximum_characters:
            ]
        )

    if result.stderr.strip():
        print("\nSTDERR")
        print(
            result.stderr[
                -maximum_characters:
            ]
        )


# ============================================================
# 2. SNAPSHOT SEBELUM INSTALASI
# ============================================================

tracked_distributions = [
    "numpy",
    "pandas",
    "PyMuPDF",
    "Pillow",
    "opencv-python",
    "opencv-python-headless",
    "paddlepaddle",
    "paddleocr",
    "pytesseract",
]

versions_before = {
    distribution_name: (
        distribution_version(
            distribution_name
        )
        or "NOT INSTALLED"
    )
    for distribution_name
    in tracked_distributions
}

environment_changes = []


# ============================================================
# 3. PASANG PADDLEPADDLE CPU
# ============================================================

current_paddle_version = (
    distribution_version(
        "paddlepaddle"
    )
)

if current_paddle_version != (
    PADDLEPADDLE_VERSION
):
    print(
        "Memasang PaddlePaddle CPU "
        f"{PADDLEPADDLE_VERSION}..."
    )

    paddle_install_result = run_command(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--no-cache-dir",
            "--disable-pip-version-check",
            f"paddlepaddle=={PADDLEPADDLE_VERSION}",
            "--index-url",
            PADDLEPADDLE_INDEX_URL,
        ]
    )

    if paddle_install_result.returncode != 0:
        show_command_failure(
            paddle_install_result
        )

        raise RuntimeError(
            "Instalasi PaddlePaddle CPU gagal."
        )

    environment_changes.append(
        f"paddlepaddle=="
        f"{PADDLEPADDLE_VERSION}"
    )
else:
    print(
        "PaddlePaddle CPU dengan versi yang "
        "diminta sudah tersedia."
    )


# ============================================================
# 4. PASANG PADDLEOCR
# ============================================================

current_paddleocr_version = (
    distribution_version(
        "paddleocr"
    )
)

paddleocr_install_required = (
    current_paddleocr_version is None
    or not current_paddleocr_version.startswith(
        "3."
    )
)

if paddleocr_install_required:
    print(
        "Memasang PaddleOCR 3.x..."
    )

    paddleocr_install_result = run_command(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--no-cache-dir",
            "--disable-pip-version-check",
            PADDLEOCR_SPECIFICATION,
        ]
    )

    if paddleocr_install_result.returncode != 0:
        show_command_failure(
            paddleocr_install_result
        )

        raise RuntimeError(
            "Instalasi PaddleOCR gagal."
        )

    environment_changes.append(
        PADDLEOCR_SPECIFICATION
    )
else:
    print(
        "PaddleOCR 3.x sudah tersedia."
    )


# ============================================================
# 5. REFRESH IMPORT
# ============================================================

importlib.invalidate_caches()

paddle_import_error = None
paddleocr_import_error = None

try:
    import paddle

    paddle_imported = True

except Exception as error:
    paddle_imported = False
    paddle_import_error = (
        f"{type(error).__name__}: {error}"
    )

try:
    from paddleocr import PaddleOCR

    paddleocr_imported = True

except Exception as error:
    paddleocr_imported = False
    paddleocr_import_error = (
        f"{type(error).__name__}: {error}"
    )


# ============================================================
# 6. PADDLE RUNTIME CHECK
# ============================================================

paddle_runtime_check = False
paddle_device = "UNAVAILABLE"
paddle_compiled_with_cuda = False

if paddle_imported:
    try:
        paddle.set_device("cpu")

        paddle_device = (
            paddle.device.get_device()
        )

        paddle_compiled_with_cuda = bool(
            paddle.device.is_compiled_with_cuda()
        )

        paddle.utils.run_check()

        paddle_runtime_check = True

    except Exception as error:
        paddle_import_error = (
            f"{type(error).__name__}: {error}"
        )


# ============================================================
# 7. PERIKSA IMPORT KOMPONEN LAMA
# ============================================================

critical_modules = {
    "fitz": "PyMuPDF",
    "PIL": "Pillow",
    "pandas": "pandas",
    "cv2": "OpenCV",
    "pytesseract": "pytesseract",
}

critical_import_records = []

for module_name, component_name in (
    critical_modules.items()
):
    try:
        importlib.import_module(
            module_name
        )

        import_valid = True
        error_message = ""

    except Exception as error:
        import_valid = False
        error_message = (
            f"{type(error).__name__}: {error}"
        )

    critical_import_records.append(
        {
            "component": component_name,
            "module": module_name,
            "import_valid": import_valid,
            "error": error_message,
            "status": (
                "VALID"
                if import_valid
                else "INVALID"
            ),
        }
    )

critical_import_table = pd.DataFrame(
    critical_import_records
)

critical_imports_valid = all(
    record["import_valid"]
    for record
    in critical_import_records
)


# ============================================================
# 8. VERSI SETELAH INSTALASI
# ============================================================

versions_after = {
    distribution_name: (
        distribution_version(
            distribution_name
        )
        or "NOT INSTALLED"
    )
    for distribution_name
    in tracked_distributions
}

version_records = [
    {
        "distribution": distribution_name,
        "before": versions_before[
            distribution_name
        ],
        "after": versions_after[
            distribution_name
        ],
        "changed": (
            versions_before[
                distribution_name
            ]
            != versions_after[
                distribution_name
            ]
        ),
    }
    for distribution_name
    in tracked_distributions
]

version_table = pd.DataFrame(
    version_records
)

installed_paddle_version = (
    versions_after["paddlepaddle"]
)

installed_paddleocr_version = (
    versions_after["paddleocr"]
)


# ============================================================
# 9. PIP CONSISTENCY OBSERVATION
# ============================================================

pip_check_result = run_command(
    [
        sys.executable,
        "-m",
        "pip",
        "check",
    ]
)

pip_check_status = (
    "PASSED"
    if pip_check_result.returncode == 0
    else "REVIEW"
)

pip_check_message = (
    pip_check_result.stdout.strip()
    or pip_check_result.stderr.strip()
    or "No broken requirements found."
)


# ============================================================
# 10. KONTROL AKHIR
# ============================================================

controls = [
    {
        "control": "paddlepaddle_version",
        "expected": (
            PADDLEPADDLE_VERSION
        ),
        "actual": (
            installed_paddle_version
        ),
    },
    {
        "control": "paddle_import",
        "expected": True,
        "actual": paddle_imported,
    },
    {
        "control": "paddle_runtime_check",
        "expected": True,
        "actual": (
            paddle_runtime_check
        ),
    },
    {
        "control": "paddle_device",
        "expected": "cpu",
        "actual": paddle_device,
    },
    {
        "control": "paddle_cuda_disabled",
        "expected": False,
        "actual": (
            paddle_compiled_with_cuda
        ),
    },
    {
        "control": "paddleocr_major_version",
        "expected": "3",
        "actual": (
            installed_paddleocr_version.split(
                "."
            )[0]
            if installed_paddleocr_version
            != "NOT INSTALLED"
            else "NOT INSTALLED"
        ),
    },
    {
        "control": "paddleocr_import",
        "expected": True,
        "actual": (
            paddleocr_imported
        ),
    },
    {
        "control": "critical_imports",
        "expected": True,
        "actual": (
            critical_imports_valid
        ),
    },
    {
        "control": "external_api_required",
        "expected": False,
        "actual": False,
    },
]

for control in controls:
    control["status"] = (
        "VALID"
        if control["expected"]
        == control["actual"]
        else "INVALID"
    )

control_table = pd.DataFrame(
    controls
)


# ============================================================
# 11. OUTPUT
# ============================================================

display(control_table)

print()
print("PACKAGE VERSION CHANGES")
display(version_table)

print()
print("CRITICAL IMPORT CHECK")
display(critical_import_table)

print()
print("PADDLE RUNTIME")
print(
    f"PaddlePaddle version : "
    f"{installed_paddle_version}"
)
print(
    f"PaddleOCR version    : "
    f"{installed_paddleocr_version}"
)
print(
    f"Paddle device        : "
    f"{paddle_device}"
)
print(
    f"Compiled with CUDA   : "
    f"{paddle_compiled_with_cuda}"
)
print(
    f"Runtime check        : "
    f"{paddle_runtime_check}"
)

print()
print("INSTALLATION")
print(
    f"Environment changes  : "
    f"{environment_changes}"
)
print(
    f"Paddle import error  : "
    f"{paddle_import_error}"
)
print(
    f"PaddleOCR import error: "
    f"{paddleocr_import_error}"
)

print()
print("PIP CHECK")
print(f"Status               : {pip_check_status}")
print(
    pip_check_message[-3000:]
)

print()
print("Model weights downloaded: NO")
print("Dataset opened         : 0")
print("Dataset writes         : 0")
print("API key required       : NO")

invalid_controls = [
    control["control"]
    for control in controls
    if control["status"] != "VALID"
]

if invalid_controls:
    raise RuntimeError(
        "PADDLEOCR SETUP FAILED. "
        f"Kontrol tidak valid: {invalid_controls}"
    )

print()
print(
    "✅ CELL 5B PASSED — PaddlePaddle CPU dan "
    "PaddleOCR berhasil dipasang serta divalidasi."
)
print(
    "Kirim seluruh output sebelum mengunduh model "
    "atau menjalankan OCR terhadap sampel."
)

Memasang PaddlePaddle CPU 3.3.0...
Memasang PaddleOCR 3.x...


/usr/local/lib/python3.13/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


Running verify PaddlePaddle program ... 


/usr/local/lib/python3.13/dist-packages/paddle/pir/math_op_patch.py:241: UserWarning: Tensor do not have 'place' interface for pir graph mode, try not to use it. None will be returned.
  warnings.warn(


PaddlePaddle works well on 1 CPU.
PaddlePaddle is installed successfully! Let's start deep learning with PaddlePaddle now.


...


PACKAGE VERSION CHANGES


,distribution,before,after,changed
0,numpy,2.1.3,2.1.3,False
1,pandas,2.2.3,2.2.3,False
2,PyMuPDF,1.28.2,1.28.2,False
3,Pillow,11.3.0,11.3.0,False
4,opencv-python,5.0.0.93,5.0.0.93,False
5,opencv-python-headless,5.0.0.93,5.0.0.93,False
6,paddlepaddle,NOT INSTALLED,3.3.0,True
7,paddleocr,NOT INSTALLED,3.7.0,True
8,pytesseract,0.3.13,0.3.13,False



CRITICAL IMPORT CHECK


,component,module,import_valid,error,status
0,PyMuPDF,fitz,True,,VALID
1,Pillow,PIL,True,,VALID
2,pandas,pandas,True,,VALID
3,OpenCV,cv2,True,,VALID
4,pytesseract,pytesseract,True,,VALID



PADDLE RUNTIME
PaddlePaddle version : 3.3.0
PaddleOCR version    : 3.7.0
Paddle device        : cpu
Compiled with CUDA   : False
Runtime check        : True

INSTALLATION
Environment changes  : ['paddlepaddle==3.3.0', 'paddleocr>=3.0,<4.0']
Paddle import error  : None
PaddleOCR import error: None

PIP CHECK
Status               : REVIEW
ipython 7.34.0 requires jedi, which is not installed.

Model weights downloaded: NO
Dataset opened         : 0
Dataset writes         : 0
API key required       : NO

✅ CELL 5B PASSED — PaddlePaddle CPU dan PaddleOCR berhasil dipasang serta divalidasi.
Kirim seluruh output sebelum mengunduh model atau menjalankan OCR terhadap sampel.


**Cell 6 — Benchmark native PDF text extraction**

In [ ]:
from __future__ import annotations

import json
import re
import unicodedata
from collections import Counter

import fitz
import pandas as pd
from IPython.display import display


# ============================================================
# CELL 6 — NATIVE PDF TEXT EXTRACTION BENCHMARK
# DEVELOPMENT SPLIT ONLY
# ============================================================

REQUIRED_VARIABLES = [
    "development_records",
    "resolve_release_artifact",
    "load_json_object",
    "normalize_field_pattern",
    "determine_field_category",
    "field_inventory_table",
]

missing_variables = [
    variable_name
    for variable_name in REQUIRED_VARIABLES
    if variable_name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Cell 2 dan Cell 3 harus dijalankan "
        "terlebih dahulu. Variabel belum tersedia: "
        f"{missing_variables}"
    )


EXPECTED_DEVELOPMENT_DOCUMENTS = 120
EXPECTED_PAGE_COUNT = 1

BBOX_MARGIN_POINTS = 1.5


# ============================================================
# 1. NORMALISASI DAN METRIK
# ============================================================

def normalize_whitespace(
    text: str,
) -> str:
    normalized = unicodedata.normalize(
        "NFKC",
        str(text),
    )

    normalized = normalized.replace(
        "\u00a0",
        " ",
    )

    normalized = normalized.replace(
        "\u200b",
        "",
    )

    return re.sub(
        r"\s+",
        " ",
        normalized,
    ).strip()


def normalize_for_comparison(
    text: str,
) -> str:
    normalized = normalize_whitespace(
        text
    ).casefold()

    for dash_character in (
        "\u2010",
        "\u2011",
        "\u2012",
        "\u2013",
        "\u2014",
        "\u2212",
    ):
        normalized = normalized.replace(
            dash_character,
            "-",
        )

    normalized = normalized.replace(
        "\u2018",
        "'",
    ).replace(
        "\u2019",
        "'",
    )

    normalized = normalized.replace(
        "\u201c",
        '"',
    ).replace(
        "\u201d",
        '"',
    )

    return normalized


def levenshtein_distance(
    reference,
    prediction,
) -> int:
    if reference == prediction:
        return 0

    if len(reference) == 0:
        return len(prediction)

    if len(prediction) == 0:
        return len(reference)

    previous_row = list(
        range(len(prediction) + 1)
    )

    for reference_index, reference_value in enumerate(
        reference,
        start=1,
    ):
        current_row = [
            reference_index
        ]

        for prediction_index, prediction_value in enumerate(
            prediction,
            start=1,
        ):
            deletion_cost = (
                previous_row[
                    prediction_index
                ]
                + 1
            )

            insertion_cost = (
                current_row[
                    prediction_index - 1
                ]
                + 1
            )

            substitution_cost = (
                previous_row[
                    prediction_index - 1
                ]
                + (
                    reference_value
                    != prediction_value
                )
            )

            current_row.append(
                min(
                    deletion_cost,
                    insertion_cost,
                    substitution_cost,
                )
            )

        previous_row = current_row

    return previous_row[-1]


def character_error_values(
    reference: str,
    prediction: str,
) -> tuple[int, int, float]:
    reference_characters = list(
        reference
    )

    prediction_characters = list(
        prediction
    )

    edit_count = levenshtein_distance(
        reference_characters,
        prediction_characters,
    )

    reference_count = len(
        reference_characters
    )

    error_rate = (
        edit_count
        / max(reference_count, 1)
    )

    return (
        edit_count,
        reference_count,
        error_rate,
    )


def word_error_values(
    reference: str,
    prediction: str,
) -> tuple[int, int, float]:
    reference_words = reference.split()
    prediction_words = prediction.split()

    edit_count = levenshtein_distance(
        reference_words,
        prediction_words,
    )

    reference_count = len(
        reference_words
    )

    error_rate = (
        edit_count
        / max(reference_count, 1)
    )

    return (
        edit_count,
        reference_count,
        error_rate,
    )


# ============================================================
# 2. SPATIAL WORD EXTRACTION
# ============================================================

def intersection_ratio_over_word(
    word_bbox: tuple[float, float, float, float],
    target_bbox: tuple[float, float, float, float],
) -> float:
    wx0, wy0, wx1, wy1 = word_bbox
    tx0, ty0, tx1, ty1 = target_bbox

    intersection_width = max(
        0.0,
        min(wx1, tx1) - max(wx0, tx0),
    )

    intersection_height = max(
        0.0,
        min(wy1, ty1) - max(wy0, ty0),
    )

    intersection_area = (
        intersection_width
        * intersection_height
    )

    word_area = max(
        (wx1 - wx0) * (wy1 - wy0),
        1e-9,
    )

    return intersection_area / word_area


def extract_words_from_bbox(
    page_words: list,
    bbox_points: list,
    margin: float = BBOX_MARGIN_POINTS,
) -> tuple[str, int]:
    if (
        not isinstance(bbox_points, list)
        or len(bbox_points) != 4
    ):
        raise ValueError(
            f"Bounding box tidak valid: {bbox_points}"
        )

    x0, y0, x1, y1 = map(
        float,
        bbox_points,
    )

    if (
        x0 < 0
        or y0 < 0
        or x1 <= x0
        or y1 <= y0
    ):
        raise ValueError(
            f"Bounding box tidak valid: {bbox_points}"
        )

    expanded_bbox = (
        x0 - margin,
        y0 - margin,
        x1 + margin,
        y1 + margin,
    )

    selected_words = []

    for word in page_words:
        wx0, wy0, wx1, wy1 = map(
            float,
            word[:4],
        )

        word_center_x = (
            wx0 + wx1
        ) / 2

        word_center_y = (
            wy0 + wy1
        ) / 2

        center_inside = (
            expanded_bbox[0]
            <= word_center_x
            <= expanded_bbox[2]
            and expanded_bbox[1]
            <= word_center_y
            <= expanded_bbox[3]
        )

        overlap_ratio = (
            intersection_ratio_over_word(
                (wx0, wy0, wx1, wy1),
                expanded_bbox,
            )
        )

        if (
            center_inside
            or overlap_ratio >= 0.50
        ):
            selected_words.append(word)

    selected_words.sort(
        key=lambda word: (
            int(word[5])
            if len(word) > 5
            else 0,
            int(word[6])
            if len(word) > 6
            else 0,
            int(word[7])
            if len(word) > 7
            else 0,
            float(word[1]),
            float(word[0]),
        )
    )

    extracted_text = " ".join(
        str(word[4])
        for word in selected_words
    )

    return (
        normalize_whitespace(
            extracted_text
        ),
        len(selected_words),
    )


# ============================================================
# 3. BENCHMARK DEVELOPMENT
# ============================================================

result_records = []
document_errors = []
invalid_bounding_boxes = []

processed_document_ids = set()
nondevelopment_artifacts_opened = 0

expected_annotation_count = sum(
    int(record.get("annotation_count", 0))
    for record in development_records
)

for sequence_number, record in enumerate(
    development_records,
    start=1,
):
    document_id = str(
        record.get("document_id", "")
    )

    template_id = str(
        record.get("template_id", "")
    )

    language = str(
        record.get("language", "")
    )

    currency = str(
        record.get("currency", "")
    )

    try:
        pdf_path, _ = resolve_release_artifact(
            record,
            "pdf",
        )

        ground_truth_path, _ = (
            resolve_release_artifact(
                record,
                "ground_truth",
            )
        )

        ground_truth = load_json_object(
            ground_truth_path
        )

        annotations = ground_truth.get(
            "annotations"
        )

        if not isinstance(annotations, list):
            raise RuntimeError(
                "Daftar annotations tidak ditemukan."
            )

        with fitz.open(pdf_path) as pdf:
            if len(pdf) != EXPECTED_PAGE_COUNT:
                raise RuntimeError(
                    "Jumlah halaman PDF tidak sesuai: "
                    f"{len(pdf)}"
                )

            words_by_page = {
                page_index + 1: page.get_text(
                    "words"
                )
                for page_index, page
                in enumerate(pdf)
            }

        for annotation_index, annotation in enumerate(
            annotations
        ):
            if not isinstance(annotation, dict):
                raise RuntimeError(
                    "Annotation harus berupa object."
                )

            field_name = str(
                annotation.get(
                    "field_name",
                    "",
                )
            ).strip()

            field_pattern = (
                normalize_field_pattern(
                    field_name
                )
            )

            field_category = (
                determine_field_category(
                    field_pattern
                )
            )

            annotation_type = str(
                annotation.get(
                    "annotation_type",
                    "",
                )
            )

            page_number = int(
                annotation.get(
                    "page_number",
                    1,
                )
            )

            bbox_points = annotation.get(
                "bbox_points"
            )

            reference_raw = str(
                annotation.get(
                    "text",
                    "",
                )
            )

            reference_text = (
                normalize_whitespace(
                    reference_raw
                )
            )

            normalized_reference = (
                normalize_for_comparison(
                    reference_raw
                )
            )

            try:
                prediction_text, matched_words = (
                    extract_words_from_bbox(
                        words_by_page.get(
                            page_number,
                            [],
                        ),
                        bbox_points,
                    )
                )

            except Exception as error:
                invalid_bounding_boxes.append(
                    {
                        "document_id": document_id,
                        "annotation_index": (
                            annotation_index
                        ),
                        "field_name": field_name,
                        "bbox_points": bbox_points,
                        "error_type": (
                            type(error).__name__
                        ),
                        "message": str(error),
                    }
                )

                prediction_text = ""
                matched_words = 0

            normalized_prediction = (
                normalize_for_comparison(
                    prediction_text
                )
            )

            strict_exact_match = (
                reference_text
                == prediction_text
            )

            normalized_exact_match = (
                normalized_reference
                == normalized_prediction
            )

            (
                character_edits,
                reference_character_count,
                character_error_rate,
            ) = character_error_values(
                normalized_reference,
                normalized_prediction,
            )

            (
                word_edits,
                reference_word_count,
                word_error_rate,
            ) = word_error_values(
                normalized_reference,
                normalized_prediction,
            )

            result_records.append(
                {
                    "document_id": document_id,
                    "template_id": template_id,
                    "language": language,
                    "currency": currency,
                    "annotation_index": (
                        annotation_index
                    ),
                    "annotation_type": (
                        annotation_type
                    ),
                    "field_name": field_name,
                    "field_pattern": (
                        field_pattern
                    ),
                    "field_category": (
                        field_category
                    ),
                    "page_number": page_number,
                    "reference_text": (
                        reference_text
                    ),
                    "prediction_text": (
                        prediction_text
                    ),
                    "matched_words": (
                        matched_words
                    ),
                    "prediction_nonempty": bool(
                        prediction_text
                    ),
                    "strict_exact_match": (
                        strict_exact_match
                    ),
                    "normalized_exact_match": (
                        normalized_exact_match
                    ),
                    "character_edits": (
                        character_edits
                    ),
                    "reference_characters": (
                        reference_character_count
                    ),
                    "character_error_rate": round(
                        character_error_rate,
                        6,
                    ),
                    "word_edits": word_edits,
                    "reference_words": (
                        reference_word_count
                    ),
                    "word_error_rate": round(
                        word_error_rate,
                        6,
                    ),
                }
            )

        processed_document_ids.add(
            document_id
        )

    except Exception as error:
        document_errors.append(
            {
                "document_id": document_id,
                "template_id": template_id,
                "error_type": (
                    type(error).__name__
                ),
                "message": str(error)[:500],
            }
        )

    if sequence_number % 20 == 0:
        print(
            f"[{sequence_number:03d}/"
            f"{len(development_records):03d}] "
            f"results={len(result_records)}, "
            f"document_errors="
            f"{len(document_errors)}"
        )


native_field_results = pd.DataFrame(
    result_records
)


# ============================================================
# 4. AGGREGATION HELPER
# ============================================================

def calculate_metrics(
    table: pd.DataFrame,
) -> dict:
    annotation_count = len(table)

    nonempty_count = int(
        table[
            "prediction_nonempty"
        ].sum()
    )

    strict_match_count = int(
        table[
            "strict_exact_match"
        ].sum()
    )

    normalized_match_count = int(
        table[
            "normalized_exact_match"
        ].sum()
    )

    total_character_edits = int(
        table[
            "character_edits"
        ].sum()
    )

    total_reference_characters = int(
        table[
            "reference_characters"
        ].sum()
    )

    total_word_edits = int(
        table[
            "word_edits"
        ].sum()
    )

    total_reference_words = int(
        table[
            "reference_words"
        ].sum()
    )

    return {
        "annotations": annotation_count,
        "nonempty_coverage": round(
            nonempty_count
            / max(annotation_count, 1),
            6,
        ),
        "strict_exact_match": round(
            strict_match_count
            / max(annotation_count, 1),
            6,
        ),
        "normalized_exact_match": round(
            normalized_match_count
            / max(annotation_count, 1),
            6,
        ),
        "micro_cer": round(
            total_character_edits
            / max(
                total_reference_characters,
                1,
            ),
            6,
        ),
        "mean_cer": round(
            float(
                table[
                    "character_error_rate"
                ].mean()
            ),
            6,
        ),
        "micro_wer": round(
            total_word_edits
            / max(
                total_reference_words,
                1,
            ),
            6,
        ),
        "mean_wer": round(
            float(
                table[
                    "word_error_rate"
                ].mean()
            ),
            6,
        ),
    }


native_overall_metrics = pd.DataFrame(
    [
        {
            "scope": "all_development",
            **calculate_metrics(
                native_field_results
            ),
        }
    ]
)

native_template_metrics = pd.DataFrame(
    [
        {
            "template_id": template_id,
            **calculate_metrics(
                template_rows
            ),
        }
        for template_id, template_rows
        in native_field_results.groupby(
            "template_id"
        )
    ]
)

native_category_metrics = pd.DataFrame(
    [
        {
            "field_category": category,
            **calculate_metrics(
                category_rows
            ),
        }
        for category, category_rows
        in native_field_results.groupby(
            "field_category"
        )
    ]
)

native_field_metrics = pd.DataFrame(
    [
        {
            "field_pattern": field_pattern,
            **calculate_metrics(
                field_rows
            ),
        }
        for field_pattern, field_rows
        in native_field_results.groupby(
            "field_pattern"
        )
    ]
)

native_field_metrics = (
    native_field_metrics.sort_values(
        by=[
            "normalized_exact_match",
            "micro_cer",
            "field_pattern",
        ],
        ascending=[
            True,
            False,
            True,
        ],
    ).reset_index(drop=True)
)


# ============================================================
# 5. MISMATCH SAMPLES
# ============================================================

native_mismatch_samples = (
    native_field_results[
        ~native_field_results[
            "normalized_exact_match"
        ]
    ]
    .sort_values(
        by=[
            "character_error_rate",
            "document_id",
            "field_name",
        ],
        ascending=[
            False,
            True,
            True,
        ],
    )
    .head(25)
    [
        [
            "document_id",
            "template_id",
            "language",
            "field_name",
            "reference_text",
            "prediction_text",
            "matched_words",
            "character_error_rate",
            "word_error_rate",
        ]
    ]
    .reset_index(drop=True)
)


# ============================================================
# 6. QUALITY CLASSIFICATION
# ============================================================

overall_values = (
    native_overall_metrics.iloc[0]
)

overall_nonempty_coverage = float(
    overall_values[
        "nonempty_coverage"
    ]
)

overall_normalized_exact = float(
    overall_values[
        "normalized_exact_match"
    ]
)

overall_micro_cer = float(
    overall_values["micro_cer"]
)

if (
    overall_nonempty_coverage >= 0.99
    and overall_normalized_exact >= 0.95
    and overall_micro_cer <= 0.01
):
    native_baseline_quality = "STRONG"

elif (
    overall_nonempty_coverage >= 0.95
    and overall_normalized_exact >= 0.85
    and overall_micro_cer <= 0.05
):
    native_baseline_quality = "ACCEPTABLE"

else:
    native_baseline_quality = (
        "NEEDS_REVIEW"
    )


# ============================================================
# 7. KONTROL EKSEKUSI
# ============================================================

controls = [
    {
        "control": "development_documents",
        "expected": (
            EXPECTED_DEVELOPMENT_DOCUMENTS
        ),
        "actual": len(
            development_records
        ),
    },
    {
        "control": "processed_documents",
        "expected": (
            EXPECTED_DEVELOPMENT_DOCUMENTS
        ),
        "actual": len(
            processed_document_ids
        ),
    },
    {
        "control": "annotation_results",
        "expected": (
            expected_annotation_count
        ),
        "actual": len(
            native_field_results
        ),
    },
    {
        "control": "document_errors",
        "expected": 0,
        "actual": len(
            document_errors
        ),
    },
    {
        "control": "invalid_bounding_boxes",
        "expected": 0,
        "actual": len(
            invalid_bounding_boxes
        ),
    },
    {
        "control": "field_patterns",
        "expected": len(
            field_inventory_table
        ),
        "actual": int(
            native_field_results[
                "field_pattern"
            ].nunique()
        ),
    },
    {
        "control": "nondevelopment_artifacts_opened",
        "expected": 0,
        "actual": (
            nondevelopment_artifacts_opened
        ),
    },
]

for control in controls:
    control["status"] = (
        "VALID"
        if control["expected"]
        == control["actual"]
        else "INVALID"
    )

control_table = pd.DataFrame(
    controls
)


# ============================================================
# 8. OUTPUT
# ============================================================

display(control_table)

print()
print("OVERALL NATIVE BASELINE")
display(native_overall_metrics)

print()
print("NATIVE BASELINE BY TEMPLATE")
display(native_template_metrics)

print()
print("NATIVE BASELINE BY CATEGORY")
display(native_category_metrics)

print()
print(
    "FIELD METRICS — WEAKEST FIRST"
)
display(native_field_metrics)

print()
print("MISMATCH SAMPLES")

if native_mismatch_samples.empty:
    print(
        "Tidak ada normalized mismatch."
    )
else:
    display(native_mismatch_samples)

if document_errors:
    print()
    print("DOCUMENT ERRORS")
    display(
        pd.DataFrame(document_errors)
    )

if invalid_bounding_boxes:
    print()
    print("INVALID BOUNDING BOXES")
    display(
        pd.DataFrame(
            invalid_bounding_boxes
        )
    )

print()
print(f"Documents processed : {len(processed_document_ids)}")
print(f"Annotations tested  : {len(native_field_results)}")
print(
    f"Nonempty coverage   : "
    f"{overall_nonempty_coverage:.4f}"
)
print(
    f"Normalized exact    : "
    f"{overall_normalized_exact:.4f}"
)
print(
    f"Micro CER           : "
    f"{overall_micro_cer:.4f}"
)
print(
    f"Native quality      : "
    f"{native_baseline_quality}"
)
print("Validation opened   : 0")
print("Test opened         : 0")
print("Artifact writes     : 0")
print("OCR model downloads : 0")

invalid_controls = [
    control["control"]
    for control in controls
    if control["status"] != "VALID"
]

if invalid_controls:
    raise RuntimeError(
        "NATIVE BASELINE EXECUTION FAILED. "
        f"Kontrol tidak valid: {invalid_controls}"
    )

print()
print(
    "✅ CELL 6 PASSED — native PDF extraction "
    "benchmark selesai pada 120 development "
    "documents."
)
print(
    "Kualitas baseline ditentukan dari coverage, "
    "exact match, CER, dan WER di atas."
)

[020/120] results=816, document_errors=0
[040/120] results=1600, document_errors=0
[060/120] results=2328, document_errors=0
[080/120] results=3044, document_errors=0
[100/120] results=3836, document_errors=0
[120/120] results=4616, document_errors=0


,control,expected,actual,status
0,development_documents,120,120,VALID
1,processed_documents,120,120,VALID
2,annotation_results,4616,4616,VALID
3,document_errors,0,0,VALID
4,invalid_bounding_boxes,0,0,VALID
5,field_patterns,23,23,VALID
6,nondevelopment_artifacts_opened,0,0,VALID



OVERALL NATIVE BASELINE


,scope,annotations,nonempty_coverage,strict_exact_match,normalized_exact_match,micro_cer,mean_cer,micro_wer,mean_wer
0,all_development,4616,1.0,0.995667,0.995667,0.000238,0.000093,0.001654,0.00058



NATIVE BASELINE BY TEMPLATE


...


FIELD METRICS — WEAKEST FIRST


,field_pattern,annotations,nonempty_coverage,strict_exact_match,normalized_exact_match,micro_cer,mean_cer,micro_wer,mean_wer
0,document.synthetic_notice,120,1.0,0.833333,0.833333,0.003584,0.003588,0.022222,0.022321
1,buyer.address_lines,120,1.0,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000
2,buyer.email,120,1.0,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000
3,buyer.name,120,1.0,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000
4,buyer.phone,120,1.0,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000
5,buyer.tax_identifier,120,1.0,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000
6,currency,120,1.0,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000
7,due_date,120,1.0,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000
8,financials.discount,120,1.0,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000
9,financials.subtotal,120,1.0,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000



MISMATCH SAMPLES


,document_id,template_id,language,field_name,reference_text,prediction_text,matched_words,character_error_rate,word_error_rate
0,INV-SYN-000021,TPL-02,id,document.synthetic_notice,DATA SINTETIS — BUKAN DOKUMEN TRANSAKSI NYATA,DATA SINTETIS ? BUKAN DOKUMEN TRANSAKSI NYATA,7,0.022222,0.142857
1,INV-SYN-000022,TPL-02,id,document.synthetic_notice,DATA SINTETIS — BUKAN DOKUMEN TRANSAKSI NYATA,DATA SINTETIS ? BUKAN DOKUMEN TRANSAKSI NYATA,7,0.022222,0.142857
2,INV-SYN-000023,TPL-02,id,document.synthetic_notice,DATA SINTETIS — BUKAN DOKUMEN TRANSAKSI NYATA,DATA SINTETIS ? BUKAN DOKUMEN TRANSAKSI NYATA,7,0.022222,0.142857
3,INV-SYN-000024,TPL-02,id,document.synthetic_notice,DATA SINTETIS — BUKAN DOKUMEN TRANSAKSI NYATA,DATA SINTETIS ? BUKAN DOKUMEN TRANSAKSI NYATA,7,0.022222,0.142857
4,INV-SYN-000025,TPL-02,id,document.synthetic_notice,DATA SINTETIS — BUKAN DOKUMEN TRANSAKSI NYATA,DATA SINTETIS ? BUKAN DOKUMEN TRANSAKSI NYATA,7,0.022222,0.142857
5,INV-SYN-000026,TPL-02,id,document.synthetic_notice,DATA SINTETIS — BUKAN DOKUMEN TRANSAKSI NYATA,DATA SINTETIS ? BUKAN DOKUMEN TRANSAKSI NYATA,7,0.022222,0.142857
6,INV-SYN-000027,TPL-02,id,document.synthetic_notice,DATA SINTETIS — BUKAN DOKUMEN TRANSAKSI NYATA,DATA SINTETIS ? BUKAN DOKUMEN TRANSAKSI NYATA,7,0.022222,0.142857
7,INV-SYN-000028,TPL-02,id,document.synthetic_notice,DATA SINTETIS — BUKAN DOKUMEN TRANSAKSI NYATA,DATA SINTETIS ? BUKAN DOKUMEN TRANSAKSI NYATA,7,0.022222,0.142857
8,INV-SYN-000029,TPL-02,id,document.synthetic_notice,DATA SINTETIS — BUKAN DOKUMEN TRANSAKSI NYATA,DATA SINTETIS ? BUKAN DOKUMEN TRANSAKSI NYATA,7,0.022222,0.142857
9,INV-SYN-000030,TPL-02,id,document.synthetic_notice,DATA SINTETIS — BUKAN DOKUMEN TRANSAKSI NYATA,DATA SINTETIS ? BUKAN DOKUMEN TRANSAKSI NYATA,7,0.022222,0.142857



Documents processed : 120
Annotations tested  : 4616
Nonempty coverage   : 1.0000
Normalized exact    : 0.9957
Micro CER           : 0.0002
Native quality      : STRONG
Validation opened   : 0
Test opened         : 0
Artifact writes     : 0
OCR model downloads : 0

✅ CELL 6 PASSED — native PDF extraction benchmark selesai pada 120 development documents.
Kualitas baseline ditentukan dari coverage, exact match, CER, dan WER di atas.


**Cell 7 — Inisialisasi PaddleOCR dan smoke test**

In [ ]:
from __future__ import annotations

import json
import os
import time
from collections.abc import Mapping

import numpy as np
import pandas as pd
from IPython.display import display
from paddleocr import PaddleOCR


# ============================================================
# CELL 7 — PADDLEOCR MODEL INITIALIZATION DAN SMOKE TEST
# DEVELOPMENT SPLIT ONLY
# ============================================================

REQUIRED_VARIABLES = [
    "development_records",
    "ocr_benchmark_selection",
    "resolve_release_artifact",
]

missing_variables = [
    variable_name
    for variable_name in REQUIRED_VARIABLES
    if variable_name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Cell sebelumnya harus dijalankan terlebih "
        "dahulu. Variabel belum tersedia: "
        f"{missing_variables}"
    )


OCR_VERSION = "PP-OCRv6"
OCR_DEVICE = "cpu"
OCR_LANGUAGES = ("id", "en")
OCR_CPU_THREADS = 4

# Menggunakan sumber model resmi BOS.
# Tidak membutuhkan token atau API key.
os.environ.setdefault(
    "PADDLE_PDX_MODEL_SOURCE",
    "BOS",
)


# ============================================================
# 1. HELPER
# ============================================================

def make_json_compatible(
    value,
):
    if isinstance(value, Mapping):
        return {
            str(key): make_json_compatible(
                nested_value
            )
            for key, nested_value
            in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [
            make_json_compatible(
                nested_value
            )
            for nested_value in value
        ]

    if isinstance(value, np.ndarray):
        return value.tolist()

    if isinstance(value, np.generic):
        return value.item()

    return value


def paddle_result_to_dict(
    result_object,
) -> dict:
    candidate_values = []

    if isinstance(result_object, Mapping):
        candidate_values.append(
            result_object
        )

    for attribute_name in (
        "json",
        "res",
        "result",
    ):
        if not hasattr(
            result_object,
            attribute_name,
        ):
            continue

        attribute_value = getattr(
            result_object,
            attribute_name,
        )

        if callable(attribute_value):
            try:
                attribute_value = (
                    attribute_value()
                )
            except TypeError:
                continue

        candidate_values.append(
            attribute_value
        )

    for candidate_value in candidate_values:
        if isinstance(candidate_value, str):
            try:
                candidate_value = json.loads(
                    candidate_value
                )
            except json.JSONDecodeError:
                continue

        if isinstance(
            candidate_value,
            Mapping,
        ):
            return make_json_compatible(
                dict(candidate_value)
            )

    try:
        converted_value = dict(
            result_object
        )

        return make_json_compatible(
            converted_value
        )

    except Exception as error:
        raise RuntimeError(
            "Output PaddleOCR tidak dapat dikonversi "
            f"menjadi dictionary. Type: "
            f"{type(result_object).__name__}"
        ) from error


def recursive_find_value(
    value,
    target_key: str,
):
    if isinstance(value, dict):
        if target_key in value:
            return value[target_key]

        for nested_value in value.values():
            result = recursive_find_value(
                nested_value,
                target_key,
            )

            if result is not None:
                return result

    elif isinstance(value, list):
        for nested_value in value:
            result = recursive_find_value(
                nested_value,
                target_key,
            )

            if result is not None:
                return result

    return None


def ensure_list(
    value,
) -> list:
    if value is None:
        return []

    if isinstance(value, list):
        return value

    if isinstance(value, tuple):
        return list(value)

    if isinstance(value, np.ndarray):
        return value.tolist()

    return [value]


# ============================================================
# 2. PILIH SATU SAMPEL PER BAHASA
# ============================================================

record_lookup = {
    str(record["document_id"]): record
    for record in development_records
}

smoke_sample_records = []

for language in OCR_LANGUAGES:
    language_candidates = (
        ocr_benchmark_selection[
            ocr_benchmark_selection[
                "language"
            ] == language
        ]
        .sort_values(
            by=[
                "text_risk_score",
                "document_id",
            ],
            ascending=[
                False,
                True,
            ],
        )
    )

    if language_candidates.empty:
        raise RuntimeError(
            "Sampel benchmark tidak tersedia "
            f"untuk bahasa {language}."
        )

    selected_row = (
        language_candidates.iloc[0]
    )

    document_id = str(
        selected_row["document_id"]
    )

    release_record = record_lookup.get(
        document_id
    )

    if release_record is None:
        raise RuntimeError(
            "Release record tidak ditemukan untuk "
            f"{document_id}."
        )

    preview_path, _ = resolve_release_artifact(
        release_record,
        "preview",
    )

    smoke_sample_records.append(
        {
            "language": language,
            "document_id": document_id,
            "template_id": str(
                selected_row[
                    "template_id"
                ]
            ),
            "item_count": int(
                selected_row[
                    "item_count"
                ]
            ),
            "preview_path": preview_path,
        }
    )


# ============================================================
# 3. INISIALISASI MODEL
# ============================================================

paddle_ocr_pipelines = {}
model_initialization_records = []

for language in OCR_LANGUAGES:
    print()
    print("=" * 78)
    print(
        f"INITIALIZING PADDLEOCR: "
        f"language={language}"
    )
    print("=" * 78)

    initialization_started = (
        time.perf_counter()
    )

    try:
        pipeline = PaddleOCR(
            lang=language,
            ocr_version=OCR_VERSION,
            device=OCR_DEVICE,
            use_doc_orientation_classify=False,
            use_doc_unwarping=False,
            use_textline_orientation=False,
            enable_mkldnn=True,
            cpu_threads=OCR_CPU_THREADS,
        )

        initialization_seconds = (
            time.perf_counter()
            - initialization_started
        )

        paddle_ocr_pipelines[
            language
        ] = pipeline

        model_initialization_records.append(
            {
                "language": language,
                "ocr_version": OCR_VERSION,
                "device": OCR_DEVICE,
                "initialized": True,
                "seconds": round(
                    initialization_seconds,
                    3,
                ),
                "error": "",
            }
        )

    except Exception as error:
        initialization_seconds = (
            time.perf_counter()
            - initialization_started
        )

        model_initialization_records.append(
            {
                "language": language,
                "ocr_version": OCR_VERSION,
                "device": OCR_DEVICE,
                "initialized": False,
                "seconds": round(
                    initialization_seconds,
                    3,
                ),
                "error": (
                    f"{type(error).__name__}: "
                    f"{error}"
                )[:1000],
            }
        )


model_initialization_table = pd.DataFrame(
    model_initialization_records
)

display(model_initialization_table)

failed_model_languages = [
    record["language"]
    for record
    in model_initialization_records
    if not record["initialized"]
]

if failed_model_languages:
    raise RuntimeError(
        "Inisialisasi model PaddleOCR gagal untuk: "
        f"{failed_model_languages}"
    )


# ============================================================
# 4. SMOKE TEST DUA DOKUMEN
# ============================================================

smoke_result_records = []
smoke_output_details = {}

for sample in smoke_sample_records:
    language = sample["language"]
    document_id = sample["document_id"]
    preview_path = sample["preview_path"]

    print()
    print("=" * 78)
    print(
        f"RUNNING OCR: {document_id} "
        f"(language={language})"
    )
    print("=" * 78)

    prediction_started = (
        time.perf_counter()
    )

    try:
        prediction_iterator = (
            paddle_ocr_pipelines[
                language
            ].predict(
                str(preview_path)
            )
        )

        result_objects = list(
            prediction_iterator
        )

        prediction_seconds = (
            time.perf_counter()
            - prediction_started
        )

        converted_results = [
            paddle_result_to_dict(
                result_object
            )
            for result_object
            in result_objects
        ]

        recognized_texts = []
        recognition_scores = []
        recognized_boxes = []

        for converted_result in (
            converted_results
        ):
            recognized_texts.extend(
                ensure_list(
                    recursive_find_value(
                        converted_result,
                        "rec_texts",
                    )
                )
            )

            recognition_scores.extend(
                ensure_list(
                    recursive_find_value(
                        converted_result,
                        "rec_scores",
                    )
                )
            )

            boxes_value = (
                recursive_find_value(
                    converted_result,
                    "rec_boxes",
                )
            )

            if boxes_value is None:
                boxes_value = (
                    recursive_find_value(
                        converted_result,
                        "rec_polys",
                    )
                )

            recognized_boxes.extend(
                ensure_list(boxes_value)
            )

        recognized_texts = [
            str(text)
            for text in recognized_texts
            if str(text).strip()
        ]

        numeric_scores = []

        for score in recognition_scores:
            try:
                numeric_scores.append(
                    float(score)
                )
            except (
                TypeError,
                ValueError,
            ):
                continue

        line_count = len(
            recognized_texts
        )

        mean_confidence = (
            sum(numeric_scores)
            / len(numeric_scores)
            if numeric_scores
            else None
        )

        smoke_output_details[
            document_id
        ] = {
            "language": language,
            "result_object_count": len(
                result_objects
            ),
            "recognized_texts": (
                recognized_texts
            ),
            "recognition_scores": (
                numeric_scores
            ),
            "box_count": len(
                recognized_boxes
            ),
            "result_top_level_keys": [
                sorted(result.keys())
                for result in converted_results
            ],
        }

        smoke_result_records.append(
            {
                "document_id": document_id,
                "template_id": (
                    sample["template_id"]
                ),
                "language": language,
                "item_count": (
                    sample["item_count"]
                ),
                "result_objects": len(
                    result_objects
                ),
                "recognized_lines": (
                    line_count
                ),
                "recognized_boxes": len(
                    recognized_boxes
                ),
                "mean_confidence": (
                    round(
                        mean_confidence,
                        6,
                    )
                    if mean_confidence
                    is not None
                    else None
                ),
                "seconds": round(
                    prediction_seconds,
                    3,
                ),
                "status": (
                    "PASSED"
                    if (
                        len(result_objects) > 0
                        and line_count > 0
                    )
                    else "FAILED"
                ),
                "error": "",
            }
        )

    except Exception as error:
        prediction_seconds = (
            time.perf_counter()
            - prediction_started
        )

        smoke_result_records.append(
            {
                "document_id": document_id,
                "template_id": (
                    sample["template_id"]
                ),
                "language": language,
                "item_count": (
                    sample["item_count"]
                ),
                "result_objects": 0,
                "recognized_lines": 0,
                "recognized_boxes": 0,
                "mean_confidence": None,
                "seconds": round(
                    prediction_seconds,
                    3,
                ),
                "status": "FAILED",
                "error": (
                    f"{type(error).__name__}: "
                    f"{error}"
                )[:1000],
            }
        )


paddle_smoke_results = pd.DataFrame(
    smoke_result_records
)

display(paddle_smoke_results)


# ============================================================
# 5. TAMPILKAN TEKS HASIL SMOKE TEST
# ============================================================

for document_id, output_detail in (
    smoke_output_details.items()
):
    print()
    print("=" * 78)
    print(f"OCR TEXT PREVIEW: {document_id}")
    print("=" * 78)
    print(
        f"Language       : "
        f"{output_detail['language']}"
    )
    print(
        f"Result objects : "
        f"{output_detail['result_object_count']}"
    )
    print(
        f"Boxes          : "
        f"{output_detail['box_count']}"
    )
    print(
        "Top-level keys : "
        f"{output_detail['result_top_level_keys']}"
    )
    print()
    print("Recognized text:")

    for line_number, text in enumerate(
        output_detail[
            "recognized_texts"
        ][:30],
        start=1,
    ):
        print(
            f"{line_number:02d}. {text}"
        )


# ============================================================
# 6. KONTROL AKHIR
# ============================================================

controls = [
    {
        "control": "model_languages",
        "expected": 2,
        "actual": len(
            paddle_ocr_pipelines
        ),
    },
    {
        "control": "smoke_samples",
        "expected": 2,
        "actual": len(
            paddle_smoke_results
        ),
    },
    {
        "control": "successful_smoke_tests",
        "expected": 2,
        "actual": int(
            paddle_smoke_results[
                "status"
            ].eq("PASSED").sum()
        ),
    },
    {
        "control": "languages_tested",
        "expected": ["en", "id"],
        "actual": sorted(
            paddle_smoke_results[
                "language"
            ].tolist()
        ),
    },
    {
        "control": "external_api_required",
        "expected": False,
        "actual": False,
    },
    {
        "control": "validation_opened",
        "expected": 0,
        "actual": 0,
    },
    {
        "control": "test_opened",
        "expected": 0,
        "actual": 0,
    },
]

for control in controls:
    control["status"] = (
        "VALID"
        if control["expected"]
        == control["actual"]
        else "INVALID"
    )

control_table = pd.DataFrame(
    controls
)

print()
display(control_table)

print()
print(f"OCR version          : {OCR_VERSION}")
print(f"Device               : {OCR_DEVICE}")
print(
    f"Model source         : "
    f"{os.environ['PADDLE_PDX_MODEL_SOURCE']}"
)
print("Model weights downloaded: YES")
print("Model cache          : RUNTIME LOCAL")
print("Result artifact writes: 0")
print("Dataset writes       : 0")
print("API key required     : NO")

invalid_controls = [
    control["control"]
    for control in controls
    if control["status"] != "VALID"
]

if invalid_controls:
    raise RuntimeError(
        "PADDLEOCR SMOKE TEST FAILED. "
        f"Kontrol tidak valid: {invalid_controls}"
    )

print()
print(
    "✅ CELL 7 PASSED — PaddleOCR PP-OCRv6 "
    "berhasil dijalankan secara lokal untuk "
    "dokumen Indonesia dan Inggris."
)
print(
    "Kirim seluruh output sebelum benchmark "
    "18 sampel dijalankan."
)

Creating model: ('PP-OCRv6_medium_det', None, None)
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.



INITIALIZING PADDLEOCR: language=id


Using official model (PP-OCRv6_medium_det), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/PP-OCRv6_medium_det`.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

...


RUNNING OCR: INV-SYN-000082 (language=id)

RUNNING OCR: INV-SYN-000015 (language=en)


,document_id,template_id,language,item_count,result_objects,recognized_lines,recognized_boxes,mean_confidence,seconds,status,error
0,INV-SYN-000082,TPL-05,id,8,0,0,0,None,0.576,FAILED,NotImplementedError: (Unimplemented) ConvertPi...
1,INV-SYN-000015,TPL-01,en,8,0,0,0,None,0.241,FAILED,NotImplementedError: (Unimplemented) ConvertPi...


,control,expected,actual,status
0,model_languages,2,2,VALID
1,smoke_samples,2,2,VALID
2,successful_smoke_tests,2,0,INVALID
3,languages_tested,"[en, id]","[en, id]",VALID
4,external_api_required,False,False,VALID
5,validation_opened,0,0,VALID
6,test_opened,0,0,VALID



OCR version          : PP-OCRv6
Device               : cpu
Model source         : BOS
Model weights downloaded: YES
Model cache          : RUNTIME LOCAL
Result artifact writes: 0
Dataset writes       : 0
API key required     : NO


RuntimeError: PADDLEOCR SMOKE TEST FAILED. Kontrol tidak valid: ['successful_smoke_tests']

In [ ]:
# ================================================================
# CELL 7A — PADDLEOCR CPU COMPATIBILITY FIX
# Disable oneDNN/MKLDNN and repeat the two-sample smoke test
# ================================================================

from __future__ import annotations

import gc
import json
import os
import time
import traceback
from pathlib import Path
from typing import Any

import pandas as pd
from IPython.display import display


# ================================================================
# 1. KONFIGURASI RUNTIME
# ================================================================

DATA_ROOT = Path(
    "/content/drive/MyDrive/InvoiceFlow-AI-Data"
)

RELEASE_ROOT = (
    DATA_ROOT
    / "releases"
    / "SYNTHETIC-INVOICE-V1"
    / "1.0.0"
)

RELEASE_INDEX_PATH = (
    RELEASE_ROOT
    / "release_index.jsonl"
)

SMOKE_DOCUMENT_IDS = [
    "INV-SYN-000082",  # Bahasa Indonesia
    "INV-SYN-000015",  # Bahasa Inggris
]

if not RELEASE_INDEX_PATH.is_file():
    raise FileNotFoundError(
        f"Release index tidak ditemukan: "
        f"{RELEASE_INDEX_PATH}"
    )


# ================================================================
# 2. NONAKTIFKAN JALUR ONEDNN/MKLDNN BERMASALAH
# ================================================================

# Mencegah pemeriksaan sumber model karena model sudah tersimpan
# pada cache runtime.
os.environ[
    "PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"
] = "True"

# Workaround untuk regresi PIR/oneDNN PaddlePaddle 3.3.x.
os.environ["FLAGS_use_mkldnn"] = "0"
os.environ[
    "PADDLE_PDX_ENABLE_MKLDNN_BYDEFAULT"
] = "0"
os.environ["FLAGS_enable_pir_api"] = "0"


# Import dilakukan setelah environment variable ditentukan.
from paddleocr import PaddleOCR


PADDLEOCR_RUNTIME_CONFIG = {
    "ocr_version": "PP-OCRv6",
    "device": "cpu",
    "enable_mkldnn": False,
    "cpu_threads": 2,
    "use_doc_orientation_classify": False,
    "use_doc_unwarping": False,
    "use_textline_orientation": False,
}


# ================================================================
# 3. HELPER
# ================================================================

def load_jsonl(path: Path) -> list[dict]:
    records = []

    with path.open(
        "r",
        encoding="utf-8",
    ) as file_handle:
        for line_number, line in enumerate(
            file_handle,
            start=1,
        ):
            clean_line = line.strip()

            if not clean_line:
                continue

            try:
                records.append(
                    json.loads(clean_line)
                )
            except json.JSONDecodeError as error:
                raise RuntimeError(
                    "JSONL tidak valid pada baris "
                    f"{line_number}: {error}"
                ) from error

    return records


def resolve_artifact_path(
    record: dict,
    artifact_name: str,
) -> Path:
    artifacts = record.get("artifacts", {})
    artifact = artifacts.get(artifact_name, {})

    relative_path = artifact.get(
        "project_relative_path"
    )

    if not relative_path:
        raise KeyError(
            f"Artifact {artifact_name!r} tidak tersedia "
            f"untuk {record.get('document_id')}."
        )

    absolute_path = DATA_ROOT / relative_path

    if not absolute_path.is_file():
        raise FileNotFoundError(
            f"Artifact tidak ditemukan: {absolute_path}"
        )

    return absolute_path


def create_paddleocr_engine(
    language: str,
) -> PaddleOCR:
    return PaddleOCR(
        lang=language,
        **PADDLEOCR_RUNTIME_CONFIG,
    )


def result_to_mapping(
    result_object: Any,
) -> dict:
    if isinstance(result_object, dict):
        result_data = result_object
    else:
        result_data = getattr(
            result_object,
            "json",
            {},
        )

        if callable(result_data):
            result_data = result_data()

    if isinstance(result_data, str):
        try:
            result_data = json.loads(
                result_data
            )
        except json.JSONDecodeError:
            return {}

    if not isinstance(result_data, dict):
        return {}

    if (
        "res" in result_data
        and isinstance(result_data["res"], dict)
    ):
        return result_data["res"]

    return result_data


def find_first_list(
    value: Any,
    target_key: str,
) -> list:
    if isinstance(value, dict):
        candidate = value.get(target_key)

        if isinstance(candidate, list):
            return candidate

        for nested_value in value.values():
            found = find_first_list(
                nested_value,
                target_key,
            )

            if found:
                return found

    elif isinstance(value, list):
        for nested_value in value:
            found = find_first_list(
                nested_value,
                target_key,
            )

            if found:
                return found

    return []


# ================================================================
# 4. PILIH DUA DEVELOPMENT SAMPLE
# ================================================================

release_records = load_jsonl(
    RELEASE_INDEX_PATH
)

records_by_document_id = {
    record["document_id"]: record
    for record in release_records
}

missing_document_ids = [
    document_id
    for document_id in SMOKE_DOCUMENT_IDS
    if document_id not in records_by_document_id
]

if missing_document_ids:
    raise RuntimeError(
        "Smoke-test document tidak tersedia: "
        f"{missing_document_ids}"
    )

smoke_records = [
    records_by_document_id[document_id]
    for document_id in SMOKE_DOCUMENT_IDS
]

nondevelopment_records = [
    record["document_id"]
    for record in smoke_records
    if record.get("split") != "development"
]

if nondevelopment_records:
    raise RuntimeError(
        "Smoke test hanya boleh menggunakan development "
        f"split: {nondevelopment_records}"
    )


# ================================================================
# 5. JALANKAN OCR SATU ENGINE PADA SATU WAKTU
# ================================================================

smoke_results = []
full_tracebacks = {}

for record in smoke_records:
    document_id = record["document_id"]
    language = record["language"]
    preview_path = resolve_artifact_path(
        record,
        "preview",
    )

    print("=" * 78)
    print(
        f"RUNNING PADDLEOCR WITHOUT MKLDNN: "
        f"{document_id} (language={language})"
    )
    print("=" * 78)

    started_at = time.perf_counter()
    engine = None

    try:
        engine = create_paddleocr_engine(
            language
        )

        raw_results = list(
            engine.predict(
                input=str(preview_path)
            )
        )

        recognized_texts = []
        recognized_scores = []
        recognized_boxes = []

        for result_object in raw_results:
            result_mapping = result_to_mapping(
                result_object
            )

            texts = find_first_list(
                result_mapping,
                "rec_texts",
            )
            scores = find_first_list(
                result_mapping,
                "rec_scores",
            )
            boxes = find_first_list(
                result_mapping,
                "rec_polys",
            )

            recognized_texts.extend(
                str(text).strip()
                for text in texts
                if str(text).strip()
            )

            for score in scores:
                try:
                    recognized_scores.append(
                        float(score)
                    )
                except (TypeError, ValueError):
                    pass

            recognized_boxes.extend(boxes)

        elapsed_seconds = (
            time.perf_counter()
            - started_at
        )

        successful = (
            len(raw_results) > 0
            and len(recognized_texts) > 0
        )

        smoke_results.append(
            {
                "document_id": document_id,
                "template_id": record["template_id"],
                "language": language,
                "item_count": record["item_count"],
                "result_objects": len(raw_results),
                "recognized_lines": len(
                    recognized_texts
                ),
                "recognized_boxes": len(
                    recognized_boxes
                ),
                "mean_confidence": (
                    round(
                        sum(recognized_scores)
                        / len(recognized_scores),
                        6,
                    )
                    if recognized_scores
                    else None
                ),
                "seconds": round(
                    elapsed_seconds,
                    3,
                ),
                "status": (
                    "PASSED"
                    if successful
                    else "FAILED"
                ),
                "error": "",
            }
        )

        print(
            f"Recognized lines : "
            f"{len(recognized_texts)}"
        )

        if recognized_texts:
            print("Text preview:")

            for text in recognized_texts[:10]:
                print(f"  - {text}")

    except Exception as error:
        elapsed_seconds = (
            time.perf_counter()
            - started_at
        )

        full_tracebacks[document_id] = (
            traceback.format_exc()
        )

        smoke_results.append(
            {
                "document_id": document_id,
                "template_id": record["template_id"],
                "language": language,
                "item_count": record["item_count"],
                "result_objects": 0,
                "recognized_lines": 0,
                "recognized_boxes": 0,
                "mean_confidence": None,
                "seconds": round(
                    elapsed_seconds,
                    3,
                ),
                "status": "FAILED",
                "error": (
                    f"{type(error).__name__}: "
                    f"{str(error)[:300]}"
                ),
            }
        )

    finally:
        if engine is not None:
            del engine

        gc.collect()


# ================================================================
# 6. VALIDASI
# ================================================================

smoke_table = pd.DataFrame(
    smoke_results
)

successful_smoke_tests = int(
    (smoke_table["status"] == "PASSED").sum()
)

tested_languages = sorted(
    smoke_table["language"].unique().tolist()
)

controls = [
    {
        "control": "mkldnn_disabled",
        "expected": True,
        "actual": (
            PADDLEOCR_RUNTIME_CONFIG[
                "enable_mkldnn"
            ]
            is False
        ),
    },
    {
        "control": "smoke_samples",
        "expected": 2,
        "actual": len(smoke_table),
    },
    {
        "control": "successful_smoke_tests",
        "expected": 2,
        "actual": successful_smoke_tests,
    },
    {
        "control": "languages_tested",
        "expected": ["en", "id"],
        "actual": tested_languages,
    },
    {
        "control": "development_samples",
        "expected": 2,
        "actual": sum(
            record.get("split") == "development"
            for record in smoke_records
        ),
    },
    {
        "control": "external_api_required",
        "expected": False,
        "actual": False,
    },
    {
        "control": "result_artifact_writes",
        "expected": 0,
        "actual": 0,
    },
]

for control in controls:
    control["status"] = (
        "VALID"
        if control["expected"] == control["actual"]
        else "INVALID"
    )

control_table = pd.DataFrame(controls)

display(smoke_table)
display(control_table)

invalid_controls = [
    control["control"]
    for control in controls
    if control["status"] != "VALID"
]

print()
print(
    "PaddleOCR version     : PP-OCRv6"
)
print(
    "Device                : cpu"
)
print(
    "MKLDNN enabled        : False"
)
print(
    "PIR workaround        : ENABLED"
)
print(
    "Successful smoke tests: "
    f"{successful_smoke_tests}/2"
)
print(
    "Dataset writes        : 0"
)
print(
    "Result artifact writes: 0"
)
print(
    "API key required      : NO"
)

if invalid_controls:
    print("\nFULL TRACEBACK")

    for document_id, traceback_text in (
        full_tracebacks.items()
    ):
        print("\n" + "=" * 78)
        print(document_id)
        print("=" * 78)
        print(traceback_text)

    raise RuntimeError(
        "PADDLEOCR CPU COMPATIBILITY TEST FAILED. "
        f"Kontrol tidak valid: {invalid_controls}"
    )

print()
print(
    "✅ CELL 7A PASSED — PaddleOCR berhasil berjalan "
    "pada CPU dengan MKLDNN dinonaktifkan."
)
print(
    "Model tetap menggunakan cache runtime dan tidak ada "
    "artifact dataset yang dibuat atau diubah."
)

Creating model: ('PP-OCRv6_medium_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-OCRv6_medium_det`.


RUNNING PADDLEOCR WITHOUT MKLDNN: INV-SYN-000082 (language=id)


Creating model: ('PP-OCRv6_medium_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-OCRv6_medium_rec`.


Recognized lines : 82
Text preview:
  - INVOICE
  - EDITORIAL · SYNTHETIC
  - INV-SYN-000082 · TPL-05
  - Nomor Invoice
  - Tanggal Invoice
  - Jatuh Tempo
  - Mata Uang
  - FTR-2026-05-000082
  - 29 Maret 2026
  - 12 April 2026


Creating model: ('PP-OCRv6_medium_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-OCRv6_medium_det`.


...

Creating model: ('PP-OCRv6_medium_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-OCRv6_medium_rec`.


Recognized lines : 82
Text preview:
  - SYNTHETIC COMMERCE DOCUMENT
  - INVOICE
  - INV-SYN-000015 · TPL-01
  - FROM
  - BILL TO
  - Brightmere Example Logistics Ltd.
  - Brightmere Simulation Services Ltd.
  - 5988 Fictional Commerce Lane
  - 6782 Simulation Road
  - Simulation District


,document_id,template_id,language,item_count,result_objects,recognized_lines,recognized_boxes,mean_confidence,seconds,status,error
0,INV-SYN-000082,TPL-05,id,8,1,82,82,0.996658,79.750,PASSED,
1,INV-SYN-000015,TPL-01,en,8,1,82,82,0.998352,67.262,PASSED,


,control,expected,actual,status
0,mkldnn_disabled,True,True,VALID
1,smoke_samples,2,2,VALID
2,successful_smoke_tests,2,2,VALID
3,languages_tested,"[en, id]","[en, id]",VALID
4,development_samples,2,2,VALID
5,external_api_required,False,False,VALID
6,result_artifact_writes,0,0,VALID



PaddleOCR version     : PP-OCRv6
Device                : cpu
MKLDNN enabled        : False
PIR workaround        : ENABLED
Successful smoke tests: 2/2
Dataset writes        : 0
Result artifact writes: 0
API key required      : NO

✅ CELL 7A PASSED — PaddleOCR berhasil berjalan pada CPU dengan MKLDNN dinonaktifkan.
Model tetap menggunakan cache runtime dan tidak ada artifact dataset yang dibuat atau diubah.


**Cell 8A — Persiapan checkpoint benchmark OCR**

In [ ]:
# ================================================================
# CELL 8A — OCR BENCHMARK CHECKPOINT PREPARATION
# Prepare a durable and reproducible 18-document benchmark
# ================================================================

from __future__ import annotations

import hashlib
import json
from collections import Counter
from pathlib import Path

import pandas as pd
from IPython.display import display


# ================================================================
# 1. KONFIGURASI
# ================================================================

DATA_ROOT = Path(
    "/content/drive/MyDrive/InvoiceFlow-AI-Data"
)

BUILD_ROOT = (
    DATA_ROOT
    / "interim"
    / "synthetic_v1_build"
    / "20260904T150025Z"
)

RELEASE_ROOT = (
    DATA_ROOT
    / "releases"
    / "SYNTHETIC-INVOICE-V1"
    / "1.0.0"
)

RELEASE_INDEX_PATH = (
    RELEASE_ROOT
    / "release_index.jsonl"
)

OCR_BENCHMARK_ROOT = (
    BUILD_ROOT
    / "ocr_benchmark"
)

SELECTION_MANIFEST_PATH = (
    OCR_BENCHMARK_ROOT
    / "benchmark_selection.json"
)

TESSERACT_RESULT_ROOT = (
    OCR_BENCHMARK_ROOT
    / "results"
    / "tesseract"
)

PADDLEOCR_RESULT_ROOT = (
    OCR_BENCHMARK_ROOT
    / "results"
    / "paddleocr"
)

BENCHMARK_MANIFEST_ROOT = (
    OCR_BENCHMARK_ROOT
    / "manifests"
)


# Sampel ini ditetapkan pada Cell 3.
EXPECTED_BENCHMARK_DOCUMENTS = [
    {
        "template_id": "TPL-01",
        "sample_rank": 1,
        "document_id": "INV-SYN-000002",
        "selection_reason": "minimum_items",
    },
    {
        "template_id": "TPL-01",
        "sample_rank": 2,
        "document_id": "INV-SYN-000015",
        "selection_reason": "maximum_items",
    },
    {
        "template_id": "TPL-01",
        "sample_rank": 3,
        "document_id": "INV-SYN-000013",
        "selection_reason": "highest_text_risk",
    },
    {
        "template_id": "TPL-02",
        "sample_rank": 1,
        "document_id": "INV-SYN-000025",
        "selection_reason": "minimum_items",
    },
    {
        "template_id": "TPL-02",
        "sample_rank": 2,
        "document_id": "INV-SYN-000036",
        "selection_reason": "maximum_items",
    },
    {
        "template_id": "TPL-02",
        "sample_rank": 3,
        "document_id": "INV-SYN-000037",
        "selection_reason": "highest_text_risk",
    },
    {
        "template_id": "TPL-03",
        "sample_rank": 1,
        "document_id": "INV-SYN-000043",
        "selection_reason": "minimum_items",
    },
    {
        "template_id": "TPL-03",
        "sample_rank": 2,
        "document_id": "INV-SYN-000060",
        "selection_reason": "maximum_items",
    },
    {
        "template_id": "TPL-03",
        "sample_rank": 3,
        "document_id": "INV-SYN-000052",
        "selection_reason": "highest_text_risk",
    },
    {
        "template_id": "TPL-04",
        "sample_rank": 1,
        "document_id": "INV-SYN-000070",
        "selection_reason": "minimum_items",
    },
    {
        "template_id": "TPL-04",
        "sample_rank": 2,
        "document_id": "INV-SYN-000064",
        "selection_reason": "maximum_items",
    },
    {
        "template_id": "TPL-04",
        "sample_rank": 3,
        "document_id": "INV-SYN-000071",
        "selection_reason": "language_coverage_high_risk",
    },
    {
        "template_id": "TPL-05",
        "sample_rank": 1,
        "document_id": "INV-SYN-000088",
        "selection_reason": "minimum_items",
    },
    {
        "template_id": "TPL-05",
        "sample_rank": 2,
        "document_id": "INV-SYN-000098",
        "selection_reason": "maximum_items",
    },
    {
        "template_id": "TPL-05",
        "sample_rank": 3,
        "document_id": "INV-SYN-000082",
        "selection_reason": "highest_text_risk",
    },
    {
        "template_id": "TPL-06",
        "sample_rank": 1,
        "document_id": "INV-SYN-000107",
        "selection_reason": "minimum_items",
    },
    {
        "template_id": "TPL-06",
        "sample_rank": 2,
        "document_id": "INV-SYN-000120",
        "selection_reason": "maximum_items",
    },
    {
        "template_id": "TPL-06",
        "sample_rank": 3,
        "document_id": "INV-SYN-000114",
        "selection_reason": "highest_text_risk",
    },
]


# ================================================================
# 2. HELPER
# ================================================================

def load_jsonl(path: Path) -> list[dict]:
    records = []

    with path.open(
        "r",
        encoding="utf-8",
    ) as file_handle:
        for line_number, line in enumerate(
            file_handle,
            start=1,
        ):
            clean_line = line.strip()

            if not clean_line:
                continue

            try:
                records.append(
                    json.loads(clean_line)
                )
            except json.JSONDecodeError as error:
                raise RuntimeError(
                    f"JSONL tidak valid pada baris "
                    f"{line_number}: {error}"
                ) from error

    return records


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as file_handle:
        for chunk in iter(
            lambda: file_handle.read(
                1024 * 1024
            ),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def canonical_json_bytes(
    value: dict,
) -> bytes:
    return (
        json.dumps(
            value,
            ensure_ascii=False,
            sort_keys=True,
            separators=(",", ":"),
        )
        + "\n"
    ).encode("utf-8")


def atomic_write_json(
    path: Path,
    value: dict,
) -> None:
    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    temporary_path.write_bytes(
        canonical_json_bytes(value)
    )

    temporary_path.replace(path)


def resolve_artifact(
    release_record: dict,
    artifact_name: str,
) -> tuple[Path, dict]:
    artifact_record = (
        release_record
        .get("artifacts", {})
        .get(artifact_name, {})
    )

    relative_path = artifact_record.get(
        "project_relative_path"
    )

    if not relative_path:
        raise KeyError(
            f"Referensi {artifact_name} tidak tersedia "
            f"untuk {release_record['document_id']}."
        )

    absolute_path = DATA_ROOT / relative_path

    return absolute_path, artifact_record


# ================================================================
# 3. VALIDASI SUMBER
# ================================================================

if not RELEASE_INDEX_PATH.is_file():
    raise FileNotFoundError(
        f"Release index tidak ditemukan: "
        f"{RELEASE_INDEX_PATH}"
    )

release_records = load_jsonl(
    RELEASE_INDEX_PATH
)

release_records_by_id = {
    record["document_id"]: record
    for record in release_records
}

expected_document_ids = [
    sample["document_id"]
    for sample in EXPECTED_BENCHMARK_DOCUMENTS
]

missing_release_records = [
    document_id
    for document_id in expected_document_ids
    if document_id not in release_records_by_id
]

if missing_release_records:
    raise RuntimeError(
        "Dokumen benchmark tidak terdapat dalam "
        f"release index: {missing_release_records}"
    )


# ================================================================
# 4. SUSUN SELECTION RECORDS
# ================================================================

selection_records = []
artifact_failures = []

for sample in EXPECTED_BENCHMARK_DOCUMENTS:
    document_id = sample["document_id"]
    release_record = release_records_by_id[
        document_id
    ]

    preview_path, preview_artifact = (
        resolve_artifact(
            release_record,
            "preview",
        )
    )

    ground_truth_path, ground_truth_artifact = (
        resolve_artifact(
            release_record,
            "ground_truth",
        )
    )

    preview_exists = preview_path.is_file()
    ground_truth_exists = (
        ground_truth_path.is_file()
    )

    preview_checksum_match = False
    ground_truth_checksum_match = False

    if preview_exists:
        preview_checksum_match = (
            sha256_file(preview_path)
            == preview_artifact.get("sha256")
        )

    if ground_truth_exists:
        ground_truth_checksum_match = (
            sha256_file(ground_truth_path)
            == ground_truth_artifact.get(
                "sha256"
            )
        )

    artifact_valid = all(
        [
            preview_exists,
            ground_truth_exists,
            preview_checksum_match,
            ground_truth_checksum_match,
        ]
    )

    if not artifact_valid:
        artifact_failures.append(
            {
                "document_id": document_id,
                "preview_exists": preview_exists,
                "ground_truth_exists": (
                    ground_truth_exists
                ),
                "preview_checksum_match": (
                    preview_checksum_match
                ),
                "ground_truth_checksum_match": (
                    ground_truth_checksum_match
                ),
            }
        )

    selection_records.append(
        {
            "sequence_number": len(
                selection_records
            )
            + 1,
            "document_id": document_id,
            "canonical_invoice_id": release_record[
                "canonical_invoice_id"
            ],
            "template_id": release_record[
                "template_id"
            ],
            "split": release_record["split"],
            "language": release_record[
                "language"
            ],
            "currency": release_record[
                "currency"
            ],
            "item_count": release_record[
                "item_count"
            ],
            "annotation_count": release_record[
                "annotation_count"
            ],
            "sample_rank": sample[
                "sample_rank"
            ],
            "selection_reason": sample[
                "selection_reason"
            ],
            "preview_path": str(
                preview_path
            ),
            "preview_sha256": preview_artifact.get(
                "sha256"
            ),
            "ground_truth_path": str(
                ground_truth_path
            ),
            "ground_truth_sha256": (
                ground_truth_artifact.get(
                    "sha256"
                )
            ),
            "artifacts_valid": artifact_valid,
        }
    )


# ================================================================
# 5. VALIDASI DISTRIBUSI BENCHMARK
# ================================================================

template_counts = Counter(
    record["template_id"]
    for record in selection_records
)

template_languages = {
    template_id: sorted(
        {
            record["language"]
            for record in selection_records
            if record["template_id"]
            == template_id
        }
    )
    for template_id in sorted(
        template_counts
    )
}

duplicate_document_ids = [
    document_id
    for document_id, count in Counter(
        record["document_id"]
        for record in selection_records
    ).items()
    if count > 1
]

template_id_mismatches = [
    {
        "document_id": sample["document_id"],
        "expected": sample["template_id"],
        "actual": release_records_by_id[
            sample["document_id"]
        ]["template_id"],
    }
    for sample in EXPECTED_BENCHMARK_DOCUMENTS
    if (
        release_records_by_id[
            sample["document_id"]
        ]["template_id"]
        != sample["template_id"]
    )
]

nondevelopment_documents = [
    record["document_id"]
    for record in selection_records
    if record["split"] != "development"
]


# ================================================================
# 6. BENTUK MANIFEST
# ================================================================

selection_manifest = {
    "schema_version": "1.0.0",
    "benchmark_id": (
        "INVOICEFLOW-OCR-DEVELOPMENT-BASELINE-V1"
    ),
    "dataset_release": (
        "SYNTHETIC-INVOICE-V1/1.0.0"
    ),
    "scope": "development",
    "status": "READY",
    "selection_method": (
        "three stratified samples per development "
        "template: minimum items, maximum items, "
        "and highest text risk/language coverage"
    ),
    "document_count": len(
        selection_records
    ),
    "template_count": len(
        template_counts
    ),
    "languages": sorted(
        {
            record["language"]
            for record in selection_records
        }
    ),
    "candidate_engines": [
        "tesseract",
        "paddleocr",
    ],
    "paddleocr_runtime": {
        "ocr_version": "PP-OCRv6",
        "device": "cpu",
        "enable_mkldnn": False,
        "pir_workaround": True,
    },
    "records": selection_records,
}

selection_manifest_bytes = (
    canonical_json_bytes(
        selection_manifest
    )
)

selection_manifest_sha256 = (
    hashlib.sha256(
        selection_manifest_bytes
    ).hexdigest()
)


# ================================================================
# 7. KONTROL SEBELUM MENULIS CHECKPOINT
# ================================================================

controls = [
    {
        "control": "benchmark_documents",
        "expected": 18,
        "actual": len(selection_records),
    },
    {
        "control": "development_templates",
        "expected": 6,
        "actual": len(template_counts),
    },
    {
        "control": "samples_per_template",
        "expected": True,
        "actual": all(
            count == 3
            for count in template_counts.values()
        ),
    },
    {
        "control": "languages_per_template",
        "expected": True,
        "actual": all(
            languages == ["en", "id"]
            for languages
            in template_languages.values()
        ),
    },
    {
        "control": "duplicate_document_ids",
        "expected": 0,
        "actual": len(
            duplicate_document_ids
        ),
    },
    {
        "control": "template_id_mismatches",
        "expected": 0,
        "actual": len(
            template_id_mismatches
        ),
    },
    {
        "control": "nondevelopment_documents",
        "expected": 0,
        "actual": len(
            nondevelopment_documents
        ),
    },
    {
        "control": "artifact_failures",
        "expected": 0,
        "actual": len(
            artifact_failures
        ),
    },
]

for control in controls:
    control["status"] = (
        "VALID"
        if control["expected"] == control["actual"]
        else "INVALID"
    )

invalid_controls = [
    control["control"]
    for control in controls
    if control["status"] != "VALID"
]

control_table = pd.DataFrame(
    controls
)

selection_table = pd.DataFrame(
    selection_records
)[
    [
        "sequence_number",
        "template_id",
        "document_id",
        "selection_reason",
        "language",
        "currency",
        "item_count",
        "annotation_count",
        "artifacts_valid",
    ]
]

display(control_table)
display(selection_table)

if invalid_controls:
    print("\nDETAIL KEGAGALAN")

    print(
        json.dumps(
            {
                "missing_release_records": (
                    missing_release_records
                ),
                "artifact_failures": (
                    artifact_failures
                ),
                "duplicate_document_ids": (
                    duplicate_document_ids
                ),
                "template_id_mismatches": (
                    template_id_mismatches
                ),
                "nondevelopment_documents": (
                    nondevelopment_documents
                ),
            },
            indent=2,
            ensure_ascii=False,
        )
    )

    raise RuntimeError(
        "OCR BENCHMARK PREPARATION FAILED. "
        f"Kontrol tidak valid: "
        f"{invalid_controls}"
    )


# ================================================================
# 8. TULIS CHECKPOINT SECARA IDEMPOTEN
# ================================================================

OCR_BENCHMARK_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

TESSERACT_RESULT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

PADDLEOCR_RESULT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

BENCHMARK_MANIFEST_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

checkpoint_action = "CREATED"

if SELECTION_MANIFEST_PATH.exists():
    existing_manifest_bytes = (
        SELECTION_MANIFEST_PATH.read_bytes()
    )

    if (
        existing_manifest_bytes
        != selection_manifest_bytes
    ):
        raise RuntimeError(
            "Selection manifest sudah tersedia, tetapi "
            "isinya berbeda. File tidak ditimpa untuk "
            "mencegah perubahan benchmark."
        )

    checkpoint_action = "RECOVERED"
else:
    atomic_write_json(
        SELECTION_MANIFEST_PATH,
        selection_manifest,
    )


# ================================================================
# 9. VERIFIKASI HASIL PENULISAN
# ================================================================

written_manifest_sha256 = sha256_file(
    SELECTION_MANIFEST_PATH
)

if (
    written_manifest_sha256
    != selection_manifest_sha256
):
    raise RuntimeError(
        "Checksum selection manifest tidak cocok "
        "setelah penulisan."
    )

print()
print(
    f"Benchmark root       : "
    f"{OCR_BENCHMARK_ROOT}"
)
print(
    f"Selection manifest   : "
    f"{SELECTION_MANIFEST_PATH}"
)
print(
    f"Checkpoint action    : "
    f"{checkpoint_action}"
)
print(
    f"Manifest SHA-256     : "
    f"{written_manifest_sha256}"
)
print(
    f"Benchmark documents  : "
    f"{len(selection_records)}"
)
print(
    f"Templates            : "
    f"{len(template_counts)}"
)
print(
    f"Template distribution: "
    f"{dict(sorted(template_counts.items()))}"
)
print(
    f"Languages/template   : "
    f"{template_languages}"
)
print(
    "Validation opened    : 0"
)
print(
    "Test opened          : 0"
)
print(
    "OCR executions       : 0"
)
print(
    "Dataset modifications: 0"
)
print(
    "Checkpoint artifacts : 1 selection manifest"
)
print()
print(
    "✅ CELL 8A PASSED — checkpoint benchmark OCR "
    "untuk 18 development samples berhasil disiapkan."
)
print(
    "Jika runtime terputus, jalankan kembali Cell 1, "
    "Cell 5A, Cell 5B, Cell 7A, lalu Cell 8A. "
    "Selection manifest akan dipulihkan tanpa ditimpa."
)

,control,expected,actual,status
0,benchmark_documents,18,18,VALID
1,development_templates,6,6,VALID
2,samples_per_template,True,True,VALID
3,languages_per_template,True,True,VALID
4,duplicate_document_ids,0,0,VALID
5,template_id_mismatches,0,0,VALID
6,nondevelopment_documents,0,0,VALID
7,artifact_failures,0,0,VALID


,sequence_number,template_id,document_id,selection_reason,language,currency,item_count,annotation_count,artifacts_valid
0,1,TPL-01,INV-SYN-000002,minimum_items,id,USD,2,27,True
1,2,TPL-01,INV-SYN-000015,maximum_items,en,USD,8,51,True
2,3,TPL-01,INV-SYN-000013,highest_text_risk,en,USD,8,51,True
3,4,TPL-02,INV-SYN-000025,minimum_items,id,IDR,2,27,True
4,5,TPL-02,INV-SYN-000036,maximum_items,en,EUR,7,47,True
5,6,TPL-02,INV-SYN-000037,highest_text_risk,en,USD,7,47,True
6,7,TPL-03,INV-SYN-000043,minimum_items,id,IDR,2,27,True
7,8,TPL-03,INV-SYN-000060,maximum_items,en,USD,7,47,True
8,9,TPL-03,INV-SYN-000052,highest_text_risk,en,GBP,7,47,True
9,10,TPL-04,INV-SYN-000070,minimum_items,id,IDR,2,27,True



Benchmark root       : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark
Selection manifest   : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/benchmark_selection.json
Checkpoint action    : CREATED
Manifest SHA-256     : 9a35c5fe7b15a1703c378b51770cfb2dd43a589fbfa64fe59e9b2f6c8ec3267f
Benchmark documents  : 18
Templates            : 6
Template distribution: {'TPL-01': 3, 'TPL-02': 3, 'TPL-03': 3, 'TPL-04': 3, 'TPL-05': 3, 'TPL-06': 3}
Languages/template   : {'TPL-01': ['en', 'id'], 'TPL-02': ['en', 'id'], 'TPL-03': ['en', 'id'], 'TPL-04': ['en', 'id'], 'TPL-05': ['en', 'id'], 'TPL-06': ['en', 'id']}
Validation opened    : 0
Test opened          : 0
OCR executions       : 0
Dataset modifications: 0
Checkpoint artifacts : 1 selection manifest

✅ CELL 8A PASSED — checkpoint benchmark OCR untuk 18 development samples berhasil disiapkan.
Jika runtime terputus, jalankan kembali Cell 1, 

**Cell 8B — Benchmark Tesseract resumable**

In [ ]:
# ================================================================
# CELL 8B — TESSERACT OCR BENCHMARK
# Run OCR on 18 development samples with per-document checkpoints
# ================================================================

from __future__ import annotations

import hashlib
import json
import shutil
import time
import traceback
from collections import defaultdict
from pathlib import Path
from typing import Any

import pandas as pd
import pytesseract
from IPython.display import display
from PIL import Image


# ================================================================
# 1. KONFIGURASI
# ================================================================

DATA_ROOT = Path(
    "/content/drive/MyDrive/InvoiceFlow-AI-Data"
)

BUILD_ROOT = (
    DATA_ROOT
    / "interim"
    / "synthetic_v1_build"
    / "20260904T150025Z"
)

OCR_BENCHMARK_ROOT = (
    BUILD_ROOT
    / "ocr_benchmark"
)

SELECTION_MANIFEST_PATH = (
    OCR_BENCHMARK_ROOT
    / "benchmark_selection.json"
)

TESSERACT_RESULT_ROOT = (
    OCR_BENCHMARK_ROOT
    / "results"
    / "tesseract"
)

BENCHMARK_MANIFEST_ROOT = (
    OCR_BENCHMARK_ROOT
    / "manifests"
)

TESSERACT_RUN_MANIFEST_PATH = (
    BENCHMARK_MANIFEST_ROOT
    / "tesseract_run_manifest.json"
)

LANGUAGE_MAPPING = {
    "id": "ind",
    "en": "eng",
}

TESSERACT_CONFIG = {
    "engine": "tesseract",
    "oem": 1,
    "psm": 11,
    "preserve_interword_spaces": 1,
    "timeout_seconds": 120,
}

TESSERACT_CONFIG_STRING = (
    "--oem 1 "
    "--psm 11 "
    "-c preserve_interword_spaces=1"
)


# ================================================================
# 2. HELPER
# ================================================================

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as file_handle:
        for chunk in iter(
            lambda: file_handle.read(
                1024 * 1024
            ),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def canonical_json_bytes(
    value: dict,
) -> bytes:
    return (
        json.dumps(
            value,
            ensure_ascii=False,
            sort_keys=True,
            separators=(",", ":"),
        )
        + "\n"
    ).encode("utf-8")


def atomic_write_json(
    path: Path,
    value: dict,
) -> None:
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    temporary_path.write_bytes(
        canonical_json_bytes(value)
    )

    temporary_path.replace(path)


def load_json(path: Path) -> dict:
    try:
        with path.open(
            "r",
            encoding="utf-8",
        ) as file_handle:
            value = json.load(file_handle)
    except json.JSONDecodeError as error:
        raise RuntimeError(
            f"JSON tidak valid: {path}"
        ) from error

    if not isinstance(value, dict):
        raise RuntimeError(
            f"Struktur JSON bukan object: {path}"
        )

    return value


def safe_float(
    value: Any,
    default: float = -1.0,
) -> float:
    try:
        return float(value)
    except (TypeError, ValueError):
        return default


def normalized_bbox(
    left: int,
    top: int,
    width: int,
    height: int,
    image_width: int,
    image_height: int,
) -> list[float]:
    x0 = max(
        0.0,
        min(1.0, left / image_width),
    )
    y0 = max(
        0.0,
        min(1.0, top / image_height),
    )
    x1 = max(
        0.0,
        min(
            1.0,
            (left + width) / image_width,
        ),
    )
    y1 = max(
        0.0,
        min(
            1.0,
            (top + height) / image_height,
        ),
    )

    return [
        round(x0, 8),
        round(y0, 8),
        round(x1, 8),
        round(y1, 8),
    ]


def build_line_records(
    word_records: list[dict],
    image_width: int,
    image_height: int,
) -> list[dict]:
    grouped_words = defaultdict(list)

    for word in word_records:
        line_key = (
            word["page_number"],
            word["block_number"],
            word["paragraph_number"],
            word["line_number"],
        )

        grouped_words[line_key].append(word)

    line_records = []

    for line_key, words in grouped_words.items():
        sorted_words = sorted(
            words,
            key=lambda word: (
                word["bbox_pixels"][1],
                word["bbox_pixels"][0],
            ),
        )

        left = min(
            word["bbox_pixels"][0]
            for word in sorted_words
        )
        top = min(
            word["bbox_pixels"][1]
            for word in sorted_words
        )
        right = max(
            word["bbox_pixels"][2]
            for word in sorted_words
        )
        bottom = max(
            word["bbox_pixels"][3]
            for word in sorted_words
        )

        confidences = [
            word["confidence"]
            for word in sorted_words
            if word["confidence"] >= 0
        ]

        line_records.append(
            {
                "page_number": line_key[0],
                "block_number": line_key[1],
                "paragraph_number": line_key[2],
                "line_number": line_key[3],
                "text": " ".join(
                    word["text"]
                    for word in sorted_words
                ),
                "word_count": len(
                    sorted_words
                ),
                "mean_confidence": (
                    round(
                        sum(confidences)
                        / len(confidences),
                        6,
                    )
                    if confidences
                    else None
                ),
                "bbox_pixels": [
                    left,
                    top,
                    right,
                    bottom,
                ],
                "bbox_normalized": (
                    normalized_bbox(
                        left=left,
                        top=top,
                        width=right - left,
                        height=bottom - top,
                        image_width=image_width,
                        image_height=image_height,
                    )
                ),
            }
        )

    return sorted(
        line_records,
        key=lambda line: (
            line["bbox_pixels"][1],
            line["bbox_pixels"][0],
        ),
    )


def existing_result_is_valid(
    result: dict,
    selection_record: dict,
) -> bool:
    return all(
        [
            result.get("schema_version")
            == "1.0.0",
            result.get("engine")
            == "tesseract",
            result.get("status")
            == "PASSED",
            result.get("document_id")
            == selection_record["document_id"],
            result.get("template_id")
            == selection_record["template_id"],
            result.get("language")
            == selection_record["language"],
            result.get("preview_sha256")
            == selection_record["preview_sha256"],
            result.get("engine_config")
            == TESSERACT_CONFIG,
            isinstance(
                result.get("words"),
                list,
            ),
            len(result.get("words", [])) > 0,
        ]
    )


# ================================================================
# 3. VALIDASI RUNTIME DAN CHECKPOINT
# ================================================================

if not SELECTION_MANIFEST_PATH.is_file():
    raise FileNotFoundError(
        "Selection manifest belum tersedia. "
        "Jalankan Cell 8A terlebih dahulu."
    )

tesseract_binary = shutil.which(
    "tesseract"
)

if not tesseract_binary:
    raise RuntimeError(
        "Binary Tesseract tidak ditemukan. "
        "Jalankan Cell 5A terlebih dahulu."
    )

available_languages = sorted(
    pytesseract.get_languages(
        config=""
    )
)

required_languages = {
    "eng",
    "ind",
}

missing_languages = sorted(
    required_languages
    - set(available_languages)
)

if missing_languages:
    raise RuntimeError(
        "Model bahasa Tesseract belum lengkap: "
        f"{missing_languages}"
    )

selection_manifest = load_json(
    SELECTION_MANIFEST_PATH
)

selection_records = selection_manifest.get(
    "records",
    [],
)

if len(selection_records) != 18:
    raise RuntimeError(
        "Jumlah sampel pada selection manifest "
        f"bukan 18: {len(selection_records)}"
    )

if any(
    record.get("split") != "development"
    for record in selection_records
):
    raise RuntimeError(
        "Selection manifest memuat dokumen "
        "di luar development split."
    )

TESSERACT_RESULT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

BENCHMARK_MANIFEST_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ================================================================
# 4. JALANKAN OCR DENGAN CHECKPOINT PER DOKUMEN
# ================================================================

execution_records = []
newly_processed = 0
resumed_documents = 0
failed_documents = 0

print(
    "Memulai benchmark Tesseract "
    f"{len(selection_records)} dokumen..."
)
print(
    f"Result root: {TESSERACT_RESULT_ROOT}"
)
print()

for position, selection_record in enumerate(
    selection_records,
    start=1,
):
    document_id = selection_record[
        "document_id"
    ]

    preview_path = Path(
        selection_record["preview_path"]
    )

    result_path = (
        TESSERACT_RESULT_ROOT
        / f"{document_id}.json"
    )

    if not preview_path.is_file():
        raise FileNotFoundError(
            f"Preview tidak ditemukan: {preview_path}"
        )

    current_preview_sha256 = sha256_file(
        preview_path
    )

    if (
        current_preview_sha256
        != selection_record["preview_sha256"]
    ):
        raise RuntimeError(
            "Checksum preview berubah untuk "
            f"{document_id}."
        )

    if result_path.is_file():
        existing_result = load_json(
            result_path
        )

        if existing_result_is_valid(
            existing_result,
            selection_record,
        ):
            resumed_documents += 1

            execution_records.append(
                {
                    "sequence_number": position,
                    "document_id": document_id,
                    "template_id": (
                        selection_record[
                            "template_id"
                        ]
                    ),
                    "language": (
                        selection_record[
                            "language"
                        ]
                    ),
                    "recognized_words": len(
                        existing_result["words"]
                    ),
                    "recognized_lines": len(
                        existing_result.get(
                            "lines",
                            [],
                        )
                    ),
                    "mean_confidence": (
                        existing_result.get(
                            "metrics",
                            {},
                        ).get(
                            "mean_confidence"
                        )
                    ),
                    "seconds": (
                        existing_result.get(
                            "metrics",
                            {},
                        ).get(
                            "elapsed_seconds"
                        )
                    ),
                    "execution": "RESUMED",
                    "status": "PASSED",
                    "error": "",
                }
            )

            print(
                f"[{position:02d}/18] "
                f"{document_id} — RESUMED"
            )

            continue

        if (
            existing_result.get("engine_config")
            != TESSERACT_CONFIG
        ):
            raise RuntimeError(
                "Checkpoint Tesseract memakai "
                "konfigurasi berbeda: "
                f"{result_path}"
            )

    language = selection_record["language"]
    tesseract_language = LANGUAGE_MAPPING[
        language
    ]

    started_at = time.perf_counter()

    try:
        with Image.open(
            preview_path
        ) as source_image:
            rgb_image = source_image.convert(
                "RGB"
            )

            image_width, image_height = (
                rgb_image.size
            )

            ocr_data = (
                pytesseract.image_to_data(
                    rgb_image,
                    lang=tesseract_language,
                    config=(
                        TESSERACT_CONFIG_STRING
                    ),
                    output_type=(
                        pytesseract.Output.DICT
                    ),
                    timeout=(
                        TESSERACT_CONFIG[
                            "timeout_seconds"
                        ]
                    ),
                )
            )

        word_records = []

        total_entries = len(
            ocr_data.get("text", [])
        )

        for row_index in range(
            total_entries
        ):
            text_value = str(
                ocr_data["text"][row_index]
            ).strip()

            confidence = safe_float(
                ocr_data["conf"][row_index]
            )

            if (
                not text_value
                or confidence < 0
            ):
                continue

            left = int(
                ocr_data["left"][row_index]
            )
            top = int(
                ocr_data["top"][row_index]
            )
            width = int(
                ocr_data["width"][row_index]
            )
            height = int(
                ocr_data["height"][row_index]
            )

            if width <= 0 or height <= 0:
                continue

            word_records.append(
                {
                    "text": text_value,
                    "confidence": round(
                        confidence / 100.0,
                        6,
                    ),
                    "page_number": int(
                        ocr_data[
                            "page_num"
                        ][row_index]
                    ),
                    "block_number": int(
                        ocr_data[
                            "block_num"
                        ][row_index]
                    ),
                    "paragraph_number": int(
                        ocr_data[
                            "par_num"
                        ][row_index]
                    ),
                    "line_number": int(
                        ocr_data[
                            "line_num"
                        ][row_index]
                    ),
                    "word_number": int(
                        ocr_data[
                            "word_num"
                        ][row_index]
                    ),
                    "bbox_pixels": [
                        left,
                        top,
                        left + width,
                        top + height,
                    ],
                    "bbox_normalized": (
                        normalized_bbox(
                            left=left,
                            top=top,
                            width=width,
                            height=height,
                            image_width=image_width,
                            image_height=image_height,
                        )
                    ),
                }
            )

        line_records = build_line_records(
            word_records=word_records,
            image_width=image_width,
            image_height=image_height,
        )

        elapsed_seconds = (
            time.perf_counter()
            - started_at
        )

        confidence_values = [
            word["confidence"]
            for word in word_records
        ]

        if not word_records:
            raise RuntimeError(
                "Tesseract tidak menghasilkan "
                "kata apa pun."
            )

        result_record = {
            "schema_version": "1.0.0",
            "benchmark_id": (
                selection_manifest[
                    "benchmark_id"
                ]
            ),
            "engine": "tesseract",
            "status": "PASSED",
            "document_id": document_id,
            "canonical_invoice_id": (
                selection_record[
                    "canonical_invoice_id"
                ]
            ),
            "template_id": (
                selection_record[
                    "template_id"
                ]
            ),
            "split": "development",
            "language": language,
            "tesseract_language": (
                tesseract_language
            ),
            "currency": (
                selection_record["currency"]
            ),
            "item_count": (
                selection_record["item_count"]
            ),
            "preview_path": str(
                preview_path
            ),
            "preview_sha256": (
                current_preview_sha256
            ),
            "image": {
                "width_pixels": image_width,
                "height_pixels": image_height,
            },
            "engine_config": (
                TESSERACT_CONFIG
            ),
            "metrics": {
                "recognized_word_count": len(
                    word_records
                ),
                "recognized_line_count": len(
                    line_records
                ),
                "mean_confidence": round(
                    sum(confidence_values)
                    / len(confidence_values),
                    6,
                ),
                "minimum_confidence": round(
                    min(confidence_values),
                    6,
                ),
                "maximum_confidence": round(
                    max(confidence_values),
                    6,
                ),
                "elapsed_seconds": round(
                    elapsed_seconds,
                    3,
                ),
            },
            "words": word_records,
            "lines": line_records,
            "error": None,
        }

        atomic_write_json(
            result_path,
            result_record,
        )

        if not existing_result_is_valid(
            load_json(result_path),
            selection_record,
        ):
            raise RuntimeError(
                "Checkpoint hasil tidak valid "
                f"setelah ditulis: {result_path}"
            )

        newly_processed += 1

        execution_records.append(
            {
                "sequence_number": position,
                "document_id": document_id,
                "template_id": (
                    selection_record[
                        "template_id"
                    ]
                ),
                "language": language,
                "recognized_words": len(
                    word_records
                ),
                "recognized_lines": len(
                    line_records
                ),
                "mean_confidence": round(
                    sum(confidence_values)
                    / len(confidence_values),
                    6,
                ),
                "seconds": round(
                    elapsed_seconds,
                    3,
                ),
                "execution": "NEW",
                "status": "PASSED",
                "error": "",
            }
        )

        print(
            f"[{position:02d}/18] "
            f"{document_id} — PASSED "
            f"| words={len(word_records)} "
            f"| lines={len(line_records)} "
            f"| {elapsed_seconds:.2f}s"
        )

    except Exception as error:
        failed_documents += 1

        elapsed_seconds = (
            time.perf_counter()
            - started_at
        )

        error_record = {
            "schema_version": "1.0.0",
            "benchmark_id": (
                selection_manifest[
                    "benchmark_id"
                ]
            ),
            "engine": "tesseract",
            "status": "ERROR",
            "document_id": document_id,
            "canonical_invoice_id": (
                selection_record[
                    "canonical_invoice_id"
                ]
            ),
            "template_id": (
                selection_record[
                    "template_id"
                ]
            ),
            "split": "development",
            "language": language,
            "preview_path": str(
                preview_path
            ),
            "preview_sha256": (
                current_preview_sha256
            ),
            "engine_config": (
                TESSERACT_CONFIG
            ),
            "words": [],
            "lines": [],
            "error": {
                "type": type(error).__name__,
                "message": str(error),
                "traceback": traceback.format_exc(),
            },
        }

        atomic_write_json(
            result_path,
            error_record,
        )

        execution_records.append(
            {
                "sequence_number": position,
                "document_id": document_id,
                "template_id": (
                    selection_record[
                        "template_id"
                    ]
                ),
                "language": language,
                "recognized_words": 0,
                "recognized_lines": 0,
                "mean_confidence": None,
                "seconds": round(
                    elapsed_seconds,
                    3,
                ),
                "execution": "NEW",
                "status": "ERROR",
                "error": (
                    f"{type(error).__name__}: "
                    f"{str(error)[:250]}"
                ),
            }
        )

        print(
            f"[{position:02d}/18] "
            f"{document_id} — ERROR: "
            f"{type(error).__name__}: "
            f"{str(error)[:150]}"
        )


# ================================================================
# 5. VERIFIKASI SELURUH CHECKPOINT
# ================================================================

verified_results = []
verification_failures = []

for selection_record in selection_records:
    document_id = selection_record[
        "document_id"
    ]

    result_path = (
        TESSERACT_RESULT_ROOT
        / f"{document_id}.json"
    )

    if not result_path.is_file():
        verification_failures.append(
            {
                "document_id": document_id,
                "failure": "missing_result",
            }
        )
        continue

    result_record = load_json(
        result_path
    )

    if not existing_result_is_valid(
        result_record,
        selection_record,
    ):
        verification_failures.append(
            {
                "document_id": document_id,
                "failure": (
                    "invalid_or_failed_result"
                ),
            }
        )
        continue

    verified_results.append(
        {
            "document_id": document_id,
            "template_id": (
                result_record["template_id"]
            ),
            "language": (
                result_record["language"]
            ),
            "result_path": str(result_path),
            "result_sha256": sha256_file(
                result_path
            ),
            "recognized_words": len(
                result_record["words"]
            ),
            "recognized_lines": len(
                result_record["lines"]
            ),
            "mean_confidence": (
                result_record["metrics"][
                    "mean_confidence"
                ]
            ),
            "elapsed_seconds": (
                result_record["metrics"][
                    "elapsed_seconds"
                ]
            ),
            "status": "PASSED",
        }
    )


# ================================================================
# 6. RINGKASAN PER TEMPLATE
# ================================================================

verified_table = pd.DataFrame(
    verified_results
)

if not verified_table.empty:
    template_summary = (
        verified_table
        .groupby(
            "template_id",
            as_index=False,
        )
        .agg(
            documents=(
                "document_id",
                "count",
            ),
            recognized_words=(
                "recognized_words",
                "sum",
            ),
            recognized_lines=(
                "recognized_lines",
                "sum",
            ),
            mean_confidence=(
                "mean_confidence",
                "mean",
            ),
            mean_seconds=(
                "elapsed_seconds",
                "mean",
            ),
        )
    )

    template_summary[
        "mean_confidence"
    ] = template_summary[
        "mean_confidence"
    ].round(6)

    template_summary[
        "mean_seconds"
    ] = template_summary[
        "mean_seconds"
    ].round(3)

    template_summary["status"] = (
        template_summary["documents"]
        .map(
            lambda count: (
                "VALID"
                if count == 3
                else "INVALID"
            )
        )
    )
else:
    template_summary = pd.DataFrame()


# ================================================================
# 7. KONTROL AKHIR
# ================================================================

controls = [
    {
        "control": "benchmark_documents",
        "expected": 18,
        "actual": len(
            selection_records
        ),
    },
    {
        "control": "result_files",
        "expected": 18,
        "actual": len(
            list(
                TESSERACT_RESULT_ROOT.glob(
                    "INV-SYN-*.json"
                )
            )
        ),
    },
    {
        "control": "verified_results",
        "expected": 18,
        "actual": len(
            verified_results
        ),
    },
    {
        "control": "execution_failures",
        "expected": 0,
        "actual": len(
            verification_failures
        ),
    },
    {
        "control": "templates_covered",
        "expected": 6,
        "actual": (
            verified_table[
                "template_id"
            ].nunique()
            if not verified_table.empty
            else 0
        ),
    },
    {
        "control": "languages_covered",
        "expected": ["en", "id"],
        "actual": (
            sorted(
                verified_table[
                    "language"
                ].unique().tolist()
            )
            if not verified_table.empty
            else []
        ),
    },
    {
        "control": "nonempty_word_results",
        "expected": 18,
        "actual": sum(
            record["recognized_words"] > 0
            for record in verified_results
        ),
    },
    {
        "control": "validation_opened",
        "expected": 0,
        "actual": 0,
    },
    {
        "control": "test_opened",
        "expected": 0,
        "actual": 0,
    },
]

for control in controls:
    control["status"] = (
        "VALID"
        if control["expected"] == control["actual"]
        else "INVALID"
    )

control_table = pd.DataFrame(
    controls
)

execution_table = pd.DataFrame(
    execution_records
)

display(execution_table)
display(template_summary)
display(control_table)

invalid_controls = [
    control["control"]
    for control in controls
    if control["status"] != "VALID"
]


# ================================================================
# 8. TULIS RUN MANIFEST JIKA LENGKAP
# ================================================================

if not invalid_controls:
    run_manifest = {
        "schema_version": "1.0.0",
        "benchmark_id": (
            selection_manifest[
                "benchmark_id"
            ]
        ),
        "engine": "tesseract",
        "status": "PASSED",
        "selection_manifest_path": str(
            SELECTION_MANIFEST_PATH
        ),
        "selection_manifest_sha256": (
            sha256_file(
                SELECTION_MANIFEST_PATH
            )
        ),
        "engine_config": (
            TESSERACT_CONFIG
        ),
        "document_count": 18,
        "newly_processed": (
            newly_processed
        ),
        "resumed_documents": (
            resumed_documents
        ),
        "records": verified_results,
    }

    atomic_write_json(
        TESSERACT_RUN_MANIFEST_PATH,
        run_manifest,
    )

    run_manifest_sha256 = sha256_file(
        TESSERACT_RUN_MANIFEST_PATH
    )
else:
    run_manifest_sha256 = None


# ================================================================
# 9. OUTPUT
# ================================================================

print()
print(
    f"Tesseract binary    : "
    f"{tesseract_binary}"
)
print(
    f"Tesseract version   : "
    f"{pytesseract.get_tesseract_version()}"
)
print(
    f"Languages           : "
    f"{available_languages}"
)
print(
    f"Benchmark documents : "
    f"{len(selection_records)}"
)
print(
    f"Verified results    : "
    f"{len(verified_results)}"
)
print(
    f"Newly processed     : "
    f"{newly_processed}"
)
print(
    f"Recovered results   : "
    f"{resumed_documents}"
)
print(
    f"Execution failures  : "
    f"{len(verification_failures)}"
)
print(
    f"Result root         : "
    f"{TESSERACT_RESULT_ROOT}"
)

if run_manifest_sha256:
    print(
        f"Run manifest        : "
        f"{TESSERACT_RUN_MANIFEST_PATH}"
    )
    print(
        f"Manifest SHA-256    : "
        f"{run_manifest_sha256}"
    )

print(
    "Validation opened   : 0"
)
print(
    "Test opened         : 0"
)
print(
    "Dataset modifications: 0"
)

if invalid_controls:
    print("\nVERIFICATION FAILURES")
    print(
        json.dumps(
            verification_failures,
            indent=2,
            ensure_ascii=False,
        )
    )

    raise RuntimeError(
        "TESSERACT BENCHMARK FAILED. "
        f"Kontrol tidak valid: "
        f"{invalid_controls}"
    )

print()
print(
    "✅ CELL 8B PASSED — Tesseract berhasil "
    "memproses 18 development samples."
)
print(
    "Hasil word, bounding box, confidence, dan "
    "latency telah disimpan dengan checkpoint "
    "per dokumen."
)

Memulai benchmark Tesseract 18 dokumen...
Result root: /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/results/tesseract

[01/18] INV-SYN-000002 — PASSED | words=113 | lines=53 | 8.07s
[02/18] INV-SYN-000015 — PASSED | words=148 | lines=67 | 5.32s
[03/18] INV-SYN-000013 — PASSED | words=147 | lines=67 | 2.81s
[04/18] INV-SYN-000025 — PASSED | words=113 | lines=50 | 1.89s
[05/18] INV-SYN-000036 — PASSED | words=140 | lines=62 | 3.11s
[06/18] INV-SYN-000037 — PASSED | words=141 | lines=64 | 3.47s
[07/18] INV-SYN-000043 — PASSED | words=112 | lines=51 | 2.63s
[08/18] INV-SYN-000060 — PASSED | words=141 | lines=63 | 3.36s
[09/18] INV-SYN-000052 — PASSED | words=140 | lines=63 | 3.14s
[10/18] INV-SYN-000070 — PASSED | words=114 | lines=53 | 2.86s
[11/18] INV-SYN-000064 — PASSED | words=154 | lines=68 | 3.33s
[12/18] INV-SYN-000071 — PASSED | words=143 | lines=66 | 2.35s
[13/18] INV-SYN-000088 — PASSED | words=112 | lines=51 | 2.54s
[14/18

,sequence_number,document_id,template_id,language,recognized_words,recognized_lines,mean_confidence,seconds,execution,status,error
0,1,INV-SYN-000002,TPL-01,id,113,53,0.868584,8.073,NEW,PASSED,
1,2,INV-SYN-000015,TPL-01,en,148,67,0.878784,5.315,NEW,PASSED,
2,3,INV-SYN-000013,TPL-01,en,147,67,0.878639,2.807,NEW,PASSED,
3,4,INV-SYN-000025,TPL-02,id,113,50,0.912832,1.891,NEW,PASSED,
4,5,INV-SYN-000036,TPL-02,en,140,62,0.916357,3.108,NEW,PASSED,
5,6,INV-SYN-000037,TPL-02,en,141,64,0.927447,3.468,NEW,PASSED,
6,7,INV-SYN-000043,TPL-03,id,112,51,0.855804,2.626,NEW,PASSED,
7,8,INV-SYN-000060,TPL-03,en,141,63,0.876667,3.356,NEW,PASSED,
8,9,INV-SYN-000052,TPL-03,en,140,63,0.875143,3.144,NEW,PASSED,
9,10,INV-SYN-000070,TPL-04,id,114,53,0.912456,2.861,NEW,PASSED,


,template_id,documents,recognized_words,recognized_lines,mean_confidence,mean_seconds,status
0,TPL-01,3,408,187,0.875336,5.398,VALID
1,TPL-02,3,394,176,0.918879,2.822,VALID
2,TPL-03,3,393,177,0.869205,3.042,VALID
3,TPL-04,3,411,187,0.903541,2.846,VALID
4,TPL-05,3,413,185,0.909738,3.064,VALID
5,TPL-06,3,417,186,0.885217,2.613,VALID


,control,expected,actual,status
0,benchmark_documents,18,18,VALID
1,result_files,18,18,VALID
2,verified_results,18,18,VALID
3,execution_failures,0,0,VALID
4,templates_covered,6,6,VALID
5,languages_covered,"[en, id]","[en, id]",VALID
6,nonempty_word_results,18,18,VALID
7,validation_opened,0,0,VALID
8,test_opened,0,0,VALID



Tesseract binary    : /usr/bin/tesseract
Tesseract version   : 4.1.1
Languages           : ['eng', 'ind', 'osd']
Benchmark documents : 18
Verified results    : 18
Newly processed     : 18
Recovered results   : 0
Execution failures  : 0
Result root         : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/results/tesseract
Run manifest        : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/manifests/tesseract_run_manifest.json
Manifest SHA-256    : 36f6465f7b6ceaa6b2a7acdfbcbd026282788a4012333a2f74ab4df45cdf716b
Validation opened   : 0
Test opened         : 0
Dataset modifications: 0

✅ CELL 8B PASSED — Tesseract berhasil memproses 18 development samples.
Hasil word, bounding box, confidence, dan latency telah disimpan dengan checkpoint per dokumen.


**Cell 8C — Benchmark PaddleOCR resumable**

In [ ]:
# ================================================================
# CELL 8C — PADDLEOCR BENCHMARK
# Run PP-OCRv6 on 18 development samples with durable checkpoints
# ================================================================

from __future__ import annotations

import gc
import hashlib
import importlib.metadata
import json
import os
import time
import traceback
from collections import defaultdict
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import display


# ================================================================
# 1. CPU COMPATIBILITY CONFIGURATION
# ================================================================

os.environ[
    "PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"
] = "True"

os.environ["FLAGS_use_mkldnn"] = "0"

os.environ[
    "PADDLE_PDX_ENABLE_MKLDNN_BYDEFAULT"
] = "0"

os.environ["FLAGS_enable_pir_api"] = "0"


from paddleocr import PaddleOCR


# ================================================================
# 2. PATH CONFIGURATION
# ================================================================

DATA_ROOT = Path(
    "/content/drive/MyDrive/InvoiceFlow-AI-Data"
)

BUILD_ROOT = (
    DATA_ROOT
    / "interim"
    / "synthetic_v1_build"
    / "20260904T150025Z"
)

OCR_BENCHMARK_ROOT = (
    BUILD_ROOT
    / "ocr_benchmark"
)

SELECTION_MANIFEST_PATH = (
    OCR_BENCHMARK_ROOT
    / "benchmark_selection.json"
)

PADDLEOCR_RESULT_ROOT = (
    OCR_BENCHMARK_ROOT
    / "results"
    / "paddleocr"
)

BENCHMARK_MANIFEST_ROOT = (
    OCR_BENCHMARK_ROOT
    / "manifests"
)

PADDLEOCR_RUN_MANIFEST_PATH = (
    BENCHMARK_MANIFEST_ROOT
    / "paddleocr_run_manifest.json"
)


# ================================================================
# 3. PACKAGE PACKAGE VERSIONS
# ================================================================

def package_version(
    distribution_name: str,
) -> str:
    try:
        return importlib.metadata.version(
            distribution_name
        )
    except importlib.metadata.PackageNotFoundError:
        return "NOT_INSTALLED"


PACKAGE_VERSIONS = {
    "paddlepaddle": package_version(
        "paddlepaddle"
    ),
    "paddleocr": package_version(
        "paddleocr"
    ),
    "paddlex": package_version(
        "paddlex"
    ),
}

PADDLEOCR_CONFIG = {
    "engine": "paddleocr",
    "ocr_version": "PP-OCRv6",
    "device": "cpu",
    "enable_mkldnn": False,
    "cpu_threads": 2,
    "use_doc_orientation_classify": False,
    "use_doc_unwarping": False,
    "use_textline_orientation": False,
    "package_versions": PACKAGE_VERSIONS,
}


# ================================================================
# 4. GENERAL HELPERS
# ================================================================

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as file_handle:
        for chunk in iter(
            lambda: file_handle.read(
                1024 * 1024
            ),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def canonical_json_bytes(
    value: dict,
) -> bytes:
    return (
        json.dumps(
            value,
            ensure_ascii=False,
            sort_keys=True,
            separators=(",", ":"),
        )
        + "\n"
    ).encode("utf-8")


def atomic_write_json(
    path: Path,
    value: dict,
) -> None:
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    temporary_path.write_bytes(
        canonical_json_bytes(value)
    )

    temporary_path.replace(path)


def load_json(path: Path) -> dict:
    try:
        with path.open(
            "r",
            encoding="utf-8",
        ) as file_handle:
            value = json.load(file_handle)
    except json.JSONDecodeError as error:
        raise RuntimeError(
            f"JSON tidak valid: {path}"
        ) from error

    if not isinstance(value, dict):
        raise RuntimeError(
            f"Struktur JSON bukan object: {path}"
        )

    return value


def to_builtin(
    value: Any,
) -> Any:
    if isinstance(value, np.ndarray):
        return value.tolist()

    if isinstance(value, np.generic):
        return value.item()

    if isinstance(value, dict):
        return {
            str(key): to_builtin(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [
            to_builtin(item)
            for item in value
        ]

    return value


def result_to_mapping(
    result_object: Any,
) -> dict:
    if isinstance(result_object, dict):
        result_data = result_object
    else:
        result_data = getattr(
            result_object,
            "json",
            {},
        )

        if callable(result_data):
            result_data = result_data()

    result_data = to_builtin(
        result_data
    )

    if isinstance(result_data, str):
        try:
            result_data = json.loads(
                result_data
            )
        except json.JSONDecodeError:
            return {}

    if not isinstance(result_data, dict):
        return {}

    if (
        "res" in result_data
        and isinstance(result_data["res"], dict)
    ):
        return result_data["res"]

    return result_data


def find_first_value(
    value: Any,
    target_key: str,
) -> Any:
    if isinstance(value, dict):
        if target_key in value:
            return value[target_key]

        for nested_value in value.values():
            found_value = find_first_value(
                nested_value,
                target_key,
            )

            if found_value is not None:
                return found_value

    elif isinstance(value, list):
        for nested_value in value:
            found_value = find_first_value(
                nested_value,
                target_key,
            )

            if found_value is not None:
                return found_value

    return None


def normalize_bbox(
    bbox_pixels: list[float],
    image_width: int,
    image_height: int,
) -> list[float]:
    x0, y0, x1, y1 = bbox_pixels

    return [
        round(
            max(
                0.0,
                min(1.0, x0 / image_width),
            ),
            8,
        ),
        round(
            max(
                0.0,
                min(1.0, y0 / image_height),
            ),
            8,
        ),
        round(
            max(
                0.0,
                min(1.0, x1 / image_width),
            ),
            8,
        ),
        round(
            max(
                0.0,
                min(1.0, y1 / image_height),
            ),
            8,
        ),
    ]


def polygon_to_bbox(
    polygon: Any,
) -> tuple[list[list[float]], list[float]]:
    polygon_array = np.asarray(
        polygon,
        dtype=float,
    )

    if (
        polygon_array.ndim != 2
        or polygon_array.shape[0] < 2
        or polygon_array.shape[1] < 2
    ):
        raise ValueError(
            "Polygon OCR tidak valid."
        )

    polygon_array = polygon_array[:, :2]

    x_values = polygon_array[:, 0]
    y_values = polygon_array[:, 1]

    bbox = [
        float(np.min(x_values)),
        float(np.min(y_values)),
        float(np.max(x_values)),
        float(np.max(y_values)),
    ]

    return (
        polygon_array.tolist(),
        bbox,
    )


def box_to_polygon(
    box: Any,
) -> tuple[list[list[float]], list[float]]:
    box_values = np.asarray(
        box,
        dtype=float,
    ).reshape(-1)

    if len(box_values) < 4:
        raise ValueError(
            "Bounding box OCR tidak valid."
        )

    x0, y0, x1, y1 = [
        float(value)
        for value in box_values[:4]
    ]

    polygon = [
        [x0, y0],
        [x1, y0],
        [x1, y1],
        [x0, y1],
    ]

    return polygon, [x0, y0, x1, y1]


# ================================================================
# 5. PADDLEOCR RESULT PARSER
# ================================================================

def parse_paddleocr_results(
    result_objects: list[Any],
    image_width: int,
    image_height: int,
) -> list[dict]:
    text_regions = []

    for result_object in result_objects:
        result_mapping = result_to_mapping(
            result_object
        )

        texts = find_first_value(
            result_mapping,
            "rec_texts",
        )

        scores = find_first_value(
            result_mapping,
            "rec_scores",
        )

        polygons = find_first_value(
            result_mapping,
            "rec_polys",
        )

        boxes = find_first_value(
            result_mapping,
            "rec_boxes",
        )

        if not isinstance(texts, list):
            texts = []

        if not isinstance(scores, list):
            scores = []

        if not isinstance(polygons, list):
            polygons = []

        if not isinstance(boxes, list):
            boxes = []

        for text_index, text_value in enumerate(
            texts
        ):
            clean_text = str(
                text_value
            ).strip()

            if not clean_text:
                continue

            confidence = None

            if text_index < len(scores):
                try:
                    confidence = float(
                        scores[text_index]
                    )
                except (TypeError, ValueError):
                    confidence = None

            try:
                if text_index < len(polygons):
                    polygon, bbox_pixels = (
                        polygon_to_bbox(
                            polygons[text_index]
                        )
                    )
                elif text_index < len(boxes):
                    polygon, bbox_pixels = (
                        box_to_polygon(
                            boxes[text_index]
                        )
                    )
                else:
                    continue

            except (TypeError, ValueError):
                continue

            if (
                bbox_pixels[2]
                <= bbox_pixels[0]
                or bbox_pixels[3]
                <= bbox_pixels[1]
            ):
                continue

            text_regions.append(
                {
                    "text": clean_text,
                    "confidence": (
                        round(confidence, 6)
                        if confidence is not None
                        else None
                    ),
                    "polygon_pixels": [
                        [
                            round(
                                point[0],
                                4,
                            ),
                            round(
                                point[1],
                                4,
                            ),
                        ]
                        for point in polygon
                    ],
                    "bbox_pixels": [
                        round(value, 4)
                        for value in bbox_pixels
                    ],
                    "bbox_normalized": (
                        normalize_bbox(
                            bbox_pixels=(
                                bbox_pixels
                            ),
                            image_width=(
                                image_width
                            ),
                            image_height=(
                                image_height
                            ),
                        )
                    ),
                }
            )

    return sorted(
        text_regions,
        key=lambda region: (
            region["bbox_pixels"][1],
            region["bbox_pixels"][0],
        ),
    )


def existing_result_is_valid(
    result: dict,
    selection_record: dict,
) -> bool:
    return all(
        [
            result.get("schema_version")
            == "1.0.0",
            result.get("engine")
            == "paddleocr",
            result.get("status")
            == "PASSED",
            result.get("document_id")
            == selection_record["document_id"],
            result.get("template_id")
            == selection_record["template_id"],
            result.get("language")
            == selection_record["language"],
            result.get("preview_sha256")
            == selection_record["preview_sha256"],
            result.get("engine_config")
            == PADDLEOCR_CONFIG,
            isinstance(
                result.get("text_regions"),
                list,
            ),
            len(
                result.get(
                    "text_regions",
                    [],
                )
            )
            > 0,
        ]
    )


def create_paddleocr_engine(
    language: str,
) -> PaddleOCR:
    return PaddleOCR(
        lang=language,
        ocr_version="PP-OCRv6",
        device="cpu",
        enable_mkldnn=False,
        cpu_threads=2,
        use_doc_orientation_classify=False,
        use_doc_unwarping=False,
        use_textline_orientation=False,
    )


# ================================================================
# 6. LOAD AND VALIDATE SELECTION
# ================================================================

if not SELECTION_MANIFEST_PATH.is_file():
    raise FileNotFoundError(
        "Selection manifest belum tersedia. "
        "Jalankan Cell 8A terlebih dahulu."
    )

selection_manifest = load_json(
    SELECTION_MANIFEST_PATH
)

selection_records = selection_manifest.get(
    "records",
    [],
)

if len(selection_records) != 18:
    raise RuntimeError(
        "Selection manifest harus berisi "
        f"18 dokumen, ditemukan "
        f"{len(selection_records)}."
    )

if any(
    record.get("split") != "development"
    for record in selection_records
):
    raise RuntimeError(
        "Benchmark hanya boleh membaca "
        "development split."
    )

selection_by_document_id = {
    record["document_id"]: record
    for record in selection_records
}

sequence_by_document_id = {
    record["document_id"]: int(
        record["sequence_number"]
    )
    for record in selection_records
}

PADDLEOCR_RESULT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

BENCHMARK_MANIFEST_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ================================================================
# 7. RECOVER EXISTING CHECKPOINTS
# ================================================================

execution_records = []
pending_records = []
resumed_documents = 0

for selection_record in selection_records:
    document_id = selection_record[
        "document_id"
    ]

    result_path = (
        PADDLEOCR_RESULT_ROOT
        / f"{document_id}.json"
    )

    if not result_path.is_file():
        pending_records.append(
            selection_record
        )
        continue

    existing_result = load_json(
        result_path
    )

    if existing_result_is_valid(
        existing_result,
        selection_record,
    ):
        resumed_documents += 1

        execution_records.append(
            {
                "sequence_number": (
                    sequence_by_document_id[
                        document_id
                    ]
                ),
                "document_id": document_id,
                "template_id": (
                    selection_record[
                        "template_id"
                    ]
                ),
                "language": (
                    selection_record[
                        "language"
                    ]
                ),
                "recognized_regions": len(
                    existing_result[
                        "text_regions"
                    ]
                ),
                "mean_confidence": (
                    existing_result.get(
                        "metrics",
                        {},
                    ).get(
                        "mean_confidence"
                    )
                ),
                "seconds": (
                    existing_result.get(
                        "metrics",
                        {},
                    ).get(
                        "elapsed_seconds"
                    )
                ),
                "execution": "RESUMED",
                "status": "PASSED",
                "error": "",
            }
        )

        continue

    if (
        existing_result.get("engine_config")
        != PADDLEOCR_CONFIG
    ):
        raise RuntimeError(
            "Checkpoint PaddleOCR menggunakan "
            "konfigurasi atau versi berbeda: "
            f"{result_path}"
        )

    # Hasil ERROR dengan konfigurasi yang sama
    # boleh dicoba kembali.
    pending_records.append(
        selection_record
    )


# ================================================================
# 8. GROUP PENDING DOCUMENTS BY LANGUAGE
# ================================================================

pending_by_language = defaultdict(list)

for selection_record in pending_records:
    pending_by_language[
        selection_record["language"]
    ].append(selection_record)

for language in pending_by_language:
    pending_by_language[language].sort(
        key=lambda record: (
            record["sequence_number"]
        )
    )


# ================================================================
# 9. RUN PADDLEOCR
# ================================================================

newly_processed = 0
execution_failures = []
engine_initialization_records = []

print(
    "Memulai benchmark PaddleOCR "
    f"{len(selection_records)} dokumen..."
)
print(
    f"Recovered results : {resumed_documents}"
)
print(
    f"Pending documents : {len(pending_records)}"
)
print(
    f"Result root       : {PADDLEOCR_RESULT_ROOT}"
)
print()

for language in ("id", "en"):
    language_records = pending_by_language.get(
        language,
        [],
    )

    if not language_records:
        continue

    print("=" * 78)
    print(
        f"INITIALIZING PP-OCRv6: "
        f"language={language}, "
        f"pending={len(language_records)}"
    )
    print("=" * 78)

    engine = None
    initialization_started_at = (
        time.perf_counter()
    )

    try:
        engine = create_paddleocr_engine(
            language
        )

        initialization_seconds = (
            time.perf_counter()
            - initialization_started_at
        )

        engine_initialization_records.append(
            {
                "language": language,
                "status": "PASSED",
                "seconds": round(
                    initialization_seconds,
                    3,
                ),
                "error": "",
            }
        )

    except Exception as error:
        initialization_seconds = (
            time.perf_counter()
            - initialization_started_at
        )

        initialization_traceback = (
            traceback.format_exc()
        )

        engine_initialization_records.append(
            {
                "language": language,
                "status": "ERROR",
                "seconds": round(
                    initialization_seconds,
                    3,
                ),
                "error": (
                    f"{type(error).__name__}: "
                    f"{str(error)[:300]}"
                ),
            }
        )

        for selection_record in language_records:
            execution_failures.append(
                {
                    "document_id": (
                        selection_record[
                            "document_id"
                        ]
                    ),
                    "stage": (
                        "engine_initialization"
                    ),
                    "error": (
                        f"{type(error).__name__}: "
                        f"{str(error)}"
                    ),
                    "traceback": (
                        initialization_traceback
                    ),
                }
            )

        continue

    try:
        for selection_record in language_records:
            document_id = selection_record[
                "document_id"
            ]

            sequence_number = int(
                selection_record[
                    "sequence_number"
                ]
            )

            preview_path = Path(
                selection_record[
                    "preview_path"
                ]
            )

            result_path = (
                PADDLEOCR_RESULT_ROOT
                / f"{document_id}.json"
            )

            if not preview_path.is_file():
                raise FileNotFoundError(
                    f"Preview tidak ditemukan: "
                    f"{preview_path}"
                )

            current_preview_sha256 = (
                sha256_file(preview_path)
            )

            if (
                current_preview_sha256
                != selection_record[
                    "preview_sha256"
                ]
            ):
                raise RuntimeError(
                    "Checksum preview berubah untuk "
                    f"{document_id}."
                )

            print(
                f"[{sequence_number:02d}/18] "
                f"{document_id} "
                f"(language={language})"
            )

            started_at = time.perf_counter()

            try:
                from PIL import Image

                with Image.open(
                    preview_path
                ) as image_handle:
                    image_width, image_height = (
                        image_handle.size
                    )

                raw_results = list(
                    engine.predict(
                        input=str(preview_path)
                    )
                )

                text_regions = (
                    parse_paddleocr_results(
                        result_objects=raw_results,
                        image_width=image_width,
                        image_height=image_height,
                    )
                )

                elapsed_seconds = (
                    time.perf_counter()
                    - started_at
                )

                if not raw_results:
                    raise RuntimeError(
                        "PaddleOCR tidak menghasilkan "
                        "result object."
                    )

                if not text_regions:
                    raise RuntimeError(
                        "PaddleOCR tidak menghasilkan "
                        "text region yang valid."
                    )

                confidence_values = [
                    region["confidence"]
                    for region in text_regions
                    if region["confidence"]
                    is not None
                ]

                result_record = {
                    "schema_version": "1.0.0",
                    "benchmark_id": (
                        selection_manifest[
                            "benchmark_id"
                        ]
                    ),
                    "engine": "paddleocr",
                    "status": "PASSED",
                    "document_id": document_id,
                    "canonical_invoice_id": (
                        selection_record[
                            "canonical_invoice_id"
                        ]
                    ),
                    "template_id": (
                        selection_record[
                            "template_id"
                        ]
                    ),
                    "split": "development",
                    "language": language,
                    "currency": (
                        selection_record[
                            "currency"
                        ]
                    ),
                    "item_count": (
                        selection_record[
                            "item_count"
                        ]
                    ),
                    "preview_path": str(
                        preview_path
                    ),
                    "preview_sha256": (
                        current_preview_sha256
                    ),
                    "image": {
                        "width_pixels": (
                            image_width
                        ),
                        "height_pixels": (
                            image_height
                        ),
                    },
                    "engine_config": (
                        PADDLEOCR_CONFIG
                    ),
                    "metrics": {
                        "result_object_count": len(
                            raw_results
                        ),
                        "recognized_region_count": (
                            len(text_regions)
                        ),
                        "mean_confidence": (
                            round(
                                sum(
                                    confidence_values
                                )
                                / len(
                                    confidence_values
                                ),
                                6,
                            )
                            if confidence_values
                            else None
                        ),
                        "minimum_confidence": (
                            round(
                                min(
                                    confidence_values
                                ),
                                6,
                            )
                            if confidence_values
                            else None
                        ),
                        "maximum_confidence": (
                            round(
                                max(
                                    confidence_values
                                ),
                                6,
                            )
                            if confidence_values
                            else None
                        ),
                        "elapsed_seconds": round(
                            elapsed_seconds,
                            3,
                        ),
                    },
                    "text_regions": (
                        text_regions
                    ),
                    "error": None,
                }

                atomic_write_json(
                    result_path,
                    result_record,
                )

                written_result = load_json(
                    result_path
                )

                if not existing_result_is_valid(
                    written_result,
                    selection_record,
                ):
                    raise RuntimeError(
                        "Checkpoint PaddleOCR tidak "
                        "valid setelah ditulis."
                    )

                newly_processed += 1

                execution_records.append(
                    {
                        "sequence_number": (
                            sequence_number
                        ),
                        "document_id": (
                            document_id
                        ),
                        "template_id": (
                            selection_record[
                                "template_id"
                            ]
                        ),
                        "language": language,
                        "recognized_regions": (
                            len(text_regions)
                        ),
                        "mean_confidence": (
                            result_record[
                                "metrics"
                            ][
                                "mean_confidence"
                            ]
                        ),
                        "seconds": round(
                            elapsed_seconds,
                            3,
                        ),
                        "execution": "NEW",
                        "status": "PASSED",
                        "error": "",
                    }
                )

                print(
                    f"           PASSED "
                    f"| regions="
                    f"{len(text_regions)} "
                    f"| confidence="
                    f"{result_record['metrics']['mean_confidence']} "
                    f"| {elapsed_seconds:.2f}s"
                )

            except Exception as error:
                elapsed_seconds = (
                    time.perf_counter()
                    - started_at
                )

                error_traceback = (
                    traceback.format_exc()
                )

                error_record = {
                    "schema_version": "1.0.0",
                    "benchmark_id": (
                        selection_manifest[
                            "benchmark_id"
                        ]
                    ),
                    "engine": "paddleocr",
                    "status": "ERROR",
                    "document_id": document_id,
                    "canonical_invoice_id": (
                        selection_record[
                            "canonical_invoice_id"
                        ]
                    ),
                    "template_id": (
                        selection_record[
                            "template_id"
                        ]
                    ),
                    "split": "development",
                    "language": language,
                    "preview_path": str(
                        preview_path
                    ),
                    "preview_sha256": (
                        current_preview_sha256
                    ),
                    "engine_config": (
                        PADDLEOCR_CONFIG
                    ),
                    "text_regions": [],
                    "error": {
                        "type": (
                            type(error).__name__
                        ),
                        "message": str(error),
                        "traceback": (
                            error_traceback
                        ),
                    },
                }

                atomic_write_json(
                    result_path,
                    error_record,
                )

                execution_failures.append(
                    {
                        "document_id": document_id,
                        "stage": "inference",
                        "error": (
                            f"{type(error).__name__}: "
                            f"{str(error)}"
                        ),
                        "traceback": (
                            error_traceback
                        ),
                    }
                )

                execution_records.append(
                    {
                        "sequence_number": (
                            sequence_number
                        ),
                        "document_id": (
                            document_id
                        ),
                        "template_id": (
                            selection_record[
                                "template_id"
                            ]
                        ),
                        "language": language,
                        "recognized_regions": 0,
                        "mean_confidence": None,
                        "seconds": round(
                            elapsed_seconds,
                            3,
                        ),
                        "execution": "NEW",
                        "status": "ERROR",
                        "error": (
                            f"{type(error).__name__}: "
                            f"{str(error)[:250]}"
                        ),
                    }
                )

                print(
                    f"           ERROR: "
                    f"{type(error).__name__}: "
                    f"{str(error)[:160]}"
                )

    finally:
        if engine is not None:
            del engine

        gc.collect()


# ================================================================
# 10. VERIFY ALL RESULT CHECKPOINTS
# ================================================================

verified_results = []
verification_failures = []

for selection_record in selection_records:
    document_id = selection_record[
        "document_id"
    ]

    result_path = (
        PADDLEOCR_RESULT_ROOT
        / f"{document_id}.json"
    )

    if not result_path.is_file():
        verification_failures.append(
            {
                "document_id": document_id,
                "failure": "missing_result",
            }
        )
        continue

    result_record = load_json(
        result_path
    )

    if not existing_result_is_valid(
        result_record,
        selection_record,
    ):
        verification_failures.append(
            {
                "document_id": document_id,
                "failure": (
                    "invalid_or_failed_result"
                ),
                "status": result_record.get(
                    "status"
                ),
            }
        )
        continue

    verified_results.append(
        {
            "sequence_number": (
                selection_record[
                    "sequence_number"
                ]
            ),
            "document_id": document_id,
            "template_id": (
                result_record["template_id"]
            ),
            "language": (
                result_record["language"]
            ),
            "result_path": str(
                result_path
            ),
            "result_sha256": sha256_file(
                result_path
            ),
            "recognized_regions": len(
                result_record[
                    "text_regions"
                ]
            ),
            "mean_confidence": (
                result_record["metrics"][
                    "mean_confidence"
                ]
            ),
            "elapsed_seconds": (
                result_record["metrics"][
                    "elapsed_seconds"
                ]
            ),
            "status": "PASSED",
        }
    )


# ================================================================
# 11. SUMMARIES
# ================================================================

execution_table = pd.DataFrame(
    execution_records
).sort_values(
    "sequence_number"
).reset_index(
    drop=True
)

verified_table = pd.DataFrame(
    verified_results
)

if not verified_table.empty:
    template_summary = (
        verified_table
        .groupby(
            "template_id",
            as_index=False,
        )
        .agg(
            documents=(
                "document_id",
                "count",
            ),
            recognized_regions=(
                "recognized_regions",
                "sum",
            ),
            mean_confidence=(
                "mean_confidence",
                "mean",
            ),
            mean_seconds=(
                "elapsed_seconds",
                "mean",
            ),
        )
    )

    template_summary[
        "mean_confidence"
    ] = template_summary[
        "mean_confidence"
    ].round(6)

    template_summary[
        "mean_seconds"
    ] = template_summary[
        "mean_seconds"
    ].round(3)

    template_summary["status"] = (
        template_summary[
            "documents"
        ].map(
            lambda count: (
                "VALID"
                if count == 3
                else "INVALID"
            )
        )
    )
else:
    template_summary = pd.DataFrame()


# ================================================================
# 12. FINAL CONTROLS
# ================================================================

controls = [
    {
        "control": "benchmark_documents",
        "expected": 18,
        "actual": len(
            selection_records
        ),
    },
    {
        "control": "result_files",
        "expected": 18,
        "actual": len(
            list(
                PADDLEOCR_RESULT_ROOT.glob(
                    "INV-SYN-*.json"
                )
            )
        ),
    },
    {
        "control": "verified_results",
        "expected": 18,
        "actual": len(
            verified_results
        ),
    },
    {
        "control": "verification_failures",
        "expected": 0,
        "actual": len(
            verification_failures
        ),
    },
    {
        "control": "templates_covered",
        "expected": 6,
        "actual": (
            verified_table[
                "template_id"
            ].nunique()
            if not verified_table.empty
            else 0
        ),
    },
    {
        "control": "languages_covered",
        "expected": ["en", "id"],
        "actual": (
            sorted(
                verified_table[
                    "language"
                ].unique().tolist()
            )
            if not verified_table.empty
            else []
        ),
    },
    {
        "control": "nonempty_results",
        "expected": 18,
        "actual": sum(
            record[
                "recognized_regions"
            ]
            > 0
            for record in verified_results
        ),
    },
    {
        "control": "mkldnn_disabled",
        "expected": False,
        "actual": (
            PADDLEOCR_CONFIG[
                "enable_mkldnn"
            ]
        ),
    },
    {
        "control": "validation_opened",
        "expected": 0,
        "actual": 0,
    },
    {
        "control": "test_opened",
        "expected": 0,
        "actual": 0,
    },
]

for control in controls:
    control["status"] = (
        "VALID"
        if control["expected"] == control["actual"]
        else "INVALID"
    )

control_table = pd.DataFrame(
    controls
)

display(execution_table)
display(engine_initialization_records)
display(template_summary)
display(control_table)

invalid_controls = [
    control["control"]
    for control in controls
    if control["status"] != "VALID"
]


# ================================================================
# 13. WRITE FINAL RUN MANIFEST
# ================================================================

if not invalid_controls:
    run_manifest = {
        "schema_version": "1.0.0",
        "benchmark_id": (
            selection_manifest[
                "benchmark_id"
            ]
        ),
        "engine": "paddleocr",
        "status": "PASSED",
        "selection_manifest_path": str(
            SELECTION_MANIFEST_PATH
        ),
        "selection_manifest_sha256": (
            sha256_file(
                SELECTION_MANIFEST_PATH
            )
        ),
        "engine_config": (
            PADDLEOCR_CONFIG
        ),
        "document_count": 18,
        "newly_processed": (
            newly_processed
        ),
        "resumed_documents": (
            resumed_documents
        ),
        "engine_initialization": (
            engine_initialization_records
        ),
        "records": verified_results,
    }

    atomic_write_json(
        PADDLEOCR_RUN_MANIFEST_PATH,
        run_manifest,
    )

    run_manifest_sha256 = sha256_file(
        PADDLEOCR_RUN_MANIFEST_PATH
    )
else:
    run_manifest_sha256 = None


# ================================================================
# 14. OUTPUT
# ================================================================

print()
print(
    f"PaddlePaddle version : "
    f"{PACKAGE_VERSIONS['paddlepaddle']}"
)
print(
    f"PaddleOCR version    : "
    f"{PACKAGE_VERSIONS['paddleocr']}"
)
print(
    f"PaddleX version      : "
    f"{PACKAGE_VERSIONS['paddlex']}"
)
print(
    "OCR model            : PP-OCRv6"
)
print(
    "Device               : cpu"
)
print(
    "MKLDNN enabled       : False"
)
print(
    f"Benchmark documents  : "
    f"{len(selection_records)}"
)
print(
    f"Verified results     : "
    f"{len(verified_results)}"
)
print(
    f"Newly processed      : "
    f"{newly_processed}"
)
print(
    f"Recovered results    : "
    f"{resumed_documents}"
)
print(
    f"Execution failures   : "
    f"{len(execution_failures)}"
)
print(
    f"Result root          : "
    f"{PADDLEOCR_RESULT_ROOT}"
)

if run_manifest_sha256:
    print(
        f"Run manifest         : "
        f"{PADDLEOCR_RUN_MANIFEST_PATH}"
    )
    print(
        f"Manifest SHA-256     : "
        f"{run_manifest_sha256}"
    )

print(
    "Validation opened    : 0"
)
print(
    "Test opened          : 0"
)
print(
    "Dataset modifications: 0"
)
print(
    "API key required     : NO"
)

if invalid_controls:
    print("\nVERIFICATION FAILURES")
    print(
        json.dumps(
            verification_failures,
            indent=2,
            ensure_ascii=False,
        )
    )

    if execution_failures:
        print("\nFIRST EXECUTION FAILURE")
        print(
            execution_failures[0][
                "traceback"
            ]
        )

    raise RuntimeError(
        "PADDLEOCR BENCHMARK FAILED. "
        f"Kontrol tidak valid: "
        f"{invalid_controls}"
    )

print()
print(
    "✅ CELL 8C PASSED — PaddleOCR berhasil "
    "memproses 18 development samples."
)
print(
    "Hasil text region, bounding box, confidence, "
    "dan latency telah disimpan dengan checkpoint "
    "per dokumen."
)

Creating model: ('PP-OCRv6_medium_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-OCRv6_medium_det`.


Memulai benchmark PaddleOCR 18 dokumen...
Recovered results : 0
Pending documents : 18
Result root       : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/results/paddleocr

INITIALIZING PP-OCRv6: language=id, pending=8


Creating model: ('PP-OCRv6_medium_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-OCRv6_medium_rec`.


[01/18] INV-SYN-000002 (language=id)
           PASSED | regions=52 | confidence=0.998368 | 62.17s
[04/18] INV-SYN-000025 (language=id)
           PASSED | regions=52 | confidence=0.995205 | 55.50s
[07/18] INV-SYN-000043 (language=id)
           PASSED | regions=51 | confidence=0.996601 | 52.04s
[10/18] INV-SYN-000070 (language=id)
           PASSED | regions=55 | confidence=0.994021 | 55.63s
[11/18] INV-SYN-000064 (language=id)
           PASSED | regions=85 | confidence=0.997888 | 76.70s
[13/18] INV-SYN-000088 (language=id)
           PASSED | regions=52 | confidence=0.996388 | 59.56s
[15/18] INV-SYN-000082 (language=id)
           PASSED | regions=82 | confidence=0.996658 | 66.80s
[16/18] INV-SYN-000107 (language=id)


Creating model: ('PP-OCRv6_medium_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-OCRv6_medium_det`.


...

,sequence_number,document_id,template_id,language,recognized_regions,mean_confidence,seconds,execution,status,error
0,1,INV-SYN-000002,TPL-01,id,52,0.998368,62.166,NEW,PASSED,
1,2,INV-SYN-000015,TPL-01,en,82,0.998353,65.057,NEW,PASSED,
2,3,INV-SYN-000013,TPL-01,en,82,0.997223,66.474,NEW,PASSED,
3,4,INV-SYN-000025,TPL-02,id,52,0.995205,55.501,NEW,PASSED,
4,5,INV-SYN-000036,TPL-02,en,77,0.996826,63.305,NEW,PASSED,
5,6,INV-SYN-000037,TPL-02,en,77,0.997001,61.611,NEW,PASSED,
6,7,INV-SYN-000043,TPL-03,id,51,0.996601,52.044,NEW,PASSED,
7,8,INV-SYN-000060,TPL-03,en,76,0.997415,62.376,NEW,PASSED,
8,9,INV-SYN-000052,TPL-03,en,76,0.996058,70.584,NEW,PASSED,
9,10,INV-SYN-000070,TPL-04,id,55,0.994021,55.628,NEW,PASSED,


[{'language': 'id', 'status': 'PASSED', 'seconds': 3.079, 'error': ''},
 {'language': 'en', 'status': 'PASSED', 'seconds': 1.221, 'error': ''}]

,template_id,documents,recognized_regions,mean_confidence,mean_seconds,status
0,TPL-01,3,216,0.997981,64.566,VALID
1,TPL-02,3,206,0.996344,60.139,VALID
2,TPL-03,3,203,0.996691,61.668,VALID
3,TPL-04,3,221,0.996508,66.354,VALID
4,TPL-05,3,216,0.997033,64.953,VALID
5,TPL-06,3,222,0.997108,69.822,VALID


,control,expected,actual,status
0,benchmark_documents,18,18,VALID
1,result_files,18,18,VALID
2,verified_results,18,18,VALID
3,verification_failures,0,0,VALID
4,templates_covered,6,6,VALID
5,languages_covered,"[en, id]","[en, id]",VALID
6,nonempty_results,18,18,VALID
7,mkldnn_disabled,False,False,VALID
8,validation_opened,0,0,VALID
9,test_opened,0,0,VALID



PaddlePaddle version : 3.3.0
PaddleOCR version    : 3.7.0
PaddleX version      : 3.7.2
OCR model            : PP-OCRv6
Device               : cpu
MKLDNN enabled       : False
Benchmark documents  : 18
Verified results     : 18
Newly processed      : 18
Recovered results    : 0
Execution failures   : 0
Result root          : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/results/paddleocr
Run manifest         : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/manifests/paddleocr_run_manifest.json
Manifest SHA-256     : 242f563fca5d0366cc049d06ecfcf7559c478e6d294e37246f13167cc203dc31
Validation opened    : 0
Test opened          : 0
Dataset modifications: 0
API key required     : NO

✅ CELL 8C PASSED — PaddleOCR berhasil memproses 18 development samples.
Hasil text region, bounding box, confidence, dan latency telah disimpan dengan checkpoint per dokumen.


**Cell 8D — Evaluasi dan pemilihan OCR terbaik**

In [ ]:
# ================================================================
# CELL 8D — OCR ACCURACY EVALUATION AND ENGINE SELECTION
# Compare Tesseract and PaddleOCR against identical ground truth
# ================================================================

from __future__ import annotations

import hashlib
import json
import re
import unicodedata
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import display


# ================================================================
# 1. PATH DAN KONFIGURASI
# ================================================================

DATA_ROOT = Path(
    "/content/drive/MyDrive/InvoiceFlow-AI-Data"
)

BUILD_ROOT = (
    DATA_ROOT
    / "interim"
    / "synthetic_v1_build"
    / "20260904T150025Z"
)

OCR_BENCHMARK_ROOT = (
    BUILD_ROOT
    / "ocr_benchmark"
)

SELECTION_MANIFEST_PATH = (
    OCR_BENCHMARK_ROOT
    / "benchmark_selection.json"
)

TESSERACT_RESULT_ROOT = (
    OCR_BENCHMARK_ROOT
    / "results"
    / "tesseract"
)

PADDLEOCR_RESULT_ROOT = (
    OCR_BENCHMARK_ROOT
    / "results"
    / "paddleocr"
)

MANIFEST_ROOT = (
    OCR_BENCHMARK_ROOT
    / "manifests"
)

TESSERACT_RUN_MANIFEST_PATH = (
    MANIFEST_ROOT
    / "tesseract_run_manifest.json"
)

PADDLEOCR_RUN_MANIFEST_PATH = (
    MANIFEST_ROOT
    / "paddleocr_run_manifest.json"
)

EVALUATION_JSONL_PATH = (
    OCR_BENCHMARK_ROOT
    / "ocr_annotation_evaluation.jsonl"
)

ENGINE_COMPARISON_PATH = (
    OCR_BENCHMARK_ROOT
    / "ocr_engine_comparison.csv"
)

TEMPLATE_COMPARISON_PATH = (
    OCR_BENCHMARK_ROOT
    / "ocr_template_comparison.csv"
)

FIELD_COMPARISON_PATH = (
    OCR_BENCHMARK_ROOT
    / "ocr_field_comparison.csv"
)

MISMATCH_SAMPLE_PATH = (
    OCR_BENCHMARK_ROOT
    / "ocr_mismatch_samples.csv"
)

EVALUATION_MANIFEST_PATH = (
    MANIFEST_ROOT
    / "ocr_evaluation_manifest.json"
)

ENGINE_RESULT_ROOTS = {
    "tesseract": TESSERACT_RESULT_ROOT,
    "paddleocr": PADDLEOCR_RESULT_ROOT,
}

BBOX_PADDING_X = 0.004
BBOX_PADDING_Y = 0.003
MIN_ELEMENT_OVERLAP = 0.35

QUALITY_GATE = {
    "minimum_nonempty_coverage": 0.90,
    "maximum_micro_cer": 0.10,
    "maximum_micro_wer": 0.20,
}


# ================================================================
# 2. FILE HELPERS
# ================================================================

def to_builtin(value: Any) -> Any:
    if isinstance(value, np.ndarray):
        return value.tolist()

    if isinstance(value, np.generic):
        return value.item()

    if isinstance(value, dict):
        return {
            str(key): to_builtin(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [
            to_builtin(item)
            for item in value
        ]

    return value


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as file_handle:
        for chunk in iter(
            lambda: file_handle.read(
                1024 * 1024
            ),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def load_json(path: Path) -> dict:
    try:
        with path.open(
            "r",
            encoding="utf-8",
        ) as file_handle:
            value = json.load(file_handle)
    except json.JSONDecodeError as error:
        raise RuntimeError(
            f"JSON tidak valid: {path}"
        ) from error

    if not isinstance(value, dict):
        raise RuntimeError(
            f"Isi JSON bukan object: {path}"
        )

    return value


def canonical_json_bytes(
    value: Any,
) -> bytes:
    return (
        json.dumps(
            to_builtin(value),
            ensure_ascii=False,
            sort_keys=True,
            separators=(",", ":"),
        )
        + "\n"
    ).encode("utf-8")


def atomic_write_json(
    path: Path,
    value: dict,
) -> None:
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    temporary_path.write_bytes(
        canonical_json_bytes(value)
    )

    temporary_path.replace(path)


def atomic_write_jsonl(
    path: Path,
    records: list[dict],
) -> None:
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as file_handle:
        for record in records:
            file_handle.write(
                json.dumps(
                    to_builtin(record),
                    ensure_ascii=False,
                    sort_keys=True,
                    separators=(",", ":"),
                )
                + "\n"
            )

    temporary_path.replace(path)


def atomic_write_csv(
    path: Path,
    table: pd.DataFrame,
) -> None:
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    table.to_csv(
        temporary_path,
        index=False,
        encoding="utf-8",
    )

    temporary_path.replace(path)


# ================================================================
# 3. TEXT NORMALIZATION
# ================================================================

def compact_text(value: Any) -> str:
    text = unicodedata.normalize(
        "NFKC",
        str(value or ""),
    )

    return re.sub(
        r"\s+",
        " ",
        text,
    ).strip()


def normalize_text(value: Any) -> str:
    text = compact_text(value)

    for dash_character in (
        "—",
        "–",
        "−",
        "‐",
        "‑",
    ):
        text = text.replace(
            dash_character,
            "-",
        )

    text = re.sub(
        r"\s+",
        " ",
        text,
    ).strip()

    return text.casefold()


def levenshtein_distance(
    reference: list | str,
    prediction: list | str,
) -> int:
    if len(reference) < len(prediction):
        reference, prediction = (
            prediction,
            reference,
        )

    previous_row = list(
        range(len(prediction) + 1)
    )

    for reference_index, reference_value in enumerate(
        reference,
        start=1,
    ):
        current_row = [reference_index]

        for prediction_index, prediction_value in enumerate(
            prediction,
            start=1,
        ):
            insertion = (
                current_row[
                    prediction_index - 1
                ]
                + 1
            )

            deletion = (
                previous_row[
                    prediction_index
                ]
                + 1
            )

            substitution = (
                previous_row[
                    prediction_index - 1
                ]
                + int(
                    reference_value
                    != prediction_value
                )
            )

            current_row.append(
                min(
                    insertion,
                    deletion,
                    substitution,
                )
            )

        previous_row = current_row

    return previous_row[-1]


# ================================================================
# 4. BOUNDING-BOX HELPERS
# ================================================================

def coerce_bbox(value: Any) -> list[float]:
    if isinstance(value, dict):
        if all(
            key in value
            for key in (
                "x0",
                "y0",
                "x1",
                "y1",
            )
        ):
            bbox = [
                value["x0"],
                value["y0"],
                value["x1"],
                value["y1"],
            ]

        elif all(
            key in value
            for key in (
                "left",
                "top",
                "right",
                "bottom",
            )
        ):
            bbox = [
                value["left"],
                value["top"],
                value["right"],
                value["bottom"],
            ]

        elif all(
            key in value
            for key in (
                "x",
                "y",
                "width",
                "height",
            )
        ):
            bbox = [
                value["x"],
                value["y"],
                value["x"] + value["width"],
                value["y"] + value["height"],
            ]

        else:
            raise ValueError(
                f"Format bbox tidak dikenal: {value}"
            )

    elif (
        isinstance(value, (list, tuple))
        and len(value) >= 4
    ):
        bbox = list(value[:4])

    else:
        raise ValueError(
            f"Bounding box tidak valid: {value}"
        )

    bbox = [
        float(component)
        for component in bbox
    ]

    x0, y0, x1, y1 = bbox

    if not (
        0 <= x0 < x1 <= 1
        and 0 <= y0 < y1 <= 1
    ):
        raise ValueError(
            "Normalized bbox berada di luar "
            f"halaman: {bbox}"
        )

    return bbox


def bbox_area(
    bbox: list[float],
) -> float:
    return (
        max(0.0, bbox[2] - bbox[0])
        * max(0.0, bbox[3] - bbox[1])
    )


def intersection_area(
    first_bbox: list[float],
    second_bbox: list[float],
) -> float:
    width = max(
        0.0,
        min(
            first_bbox[2],
            second_bbox[2],
        )
        - max(
            first_bbox[0],
            second_bbox[0],
        ),
    )

    height = max(
        0.0,
        min(
            first_bbox[3],
            second_bbox[3],
        )
        - max(
            first_bbox[1],
            second_bbox[1],
        ),
    )

    return width * height


def expand_bbox(
    bbox: list[float],
) -> list[float]:
    return [
        max(
            0.0,
            bbox[0] - BBOX_PADDING_X,
        ),
        max(
            0.0,
            bbox[1] - BBOX_PADDING_Y,
        ),
        min(
            1.0,
            bbox[2] + BBOX_PADDING_X,
        ),
        min(
            1.0,
            bbox[3] + BBOX_PADDING_Y,
        ),
    ]


def center_is_inside(
    candidate_bbox: list[float],
    target_bbox: list[float],
) -> bool:
    center_x = (
        candidate_bbox[0]
        + candidate_bbox[2]
    ) / 2

    center_y = (
        candidate_bbox[1]
        + candidate_bbox[3]
    ) / 2

    return (
        target_bbox[0]
        <= center_x
        <= target_bbox[2]
        and target_bbox[1]
        <= center_y
        <= target_bbox[3]
    )


def element_matches_annotation(
    element_bbox: list[float],
    annotation_bbox: list[float],
) -> bool:
    expanded_annotation = expand_bbox(
        annotation_bbox
    )

    if center_is_inside(
        element_bbox,
        expanded_annotation,
    ):
        return True

    overlap = intersection_area(
        element_bbox,
        expanded_annotation,
    )

    smaller_area = min(
        bbox_area(element_bbox),
        bbox_area(expanded_annotation),
    )

    if smaller_area <= 0:
        return False

    return (
        overlap / smaller_area
        >= MIN_ELEMENT_OVERLAP
    )


# ================================================================
# 5. FIELD HELPERS
# ================================================================

def get_field_pattern(
    field_name: str,
) -> str:
    normalized_name = re.sub(
        r"items\[\d+\]",
        "items.*",
        field_name,
    )

    normalized_name = re.sub(
        r"items\.\d+",
        "items.*",
        normalized_name,
    )

    return normalized_name


def get_field_category(
    field_name: str,
) -> str:
    if (
        field_name.startswith("items.")
        or field_name.startswith("items[")
    ):
        return "line_item"

    if field_name.startswith("vendor."):
        return "vendor"

    if field_name.startswith("buyer."):
        return "buyer"

    if field_name.startswith("financials."):
        return "financial"

    if field_name in {
        "invoice_number",
        "invoice_date",
        "due_date",
        "currency",
    }:
        return "metadata"

    return "other"


def get_engine_elements(
    engine_name: str,
    result_record: dict,
) -> list[dict]:
    if engine_name == "tesseract":
        return result_record.get(
            "words",
            [],
        )

    if engine_name == "paddleocr":
        return result_record.get(
            "text_regions",
            [],
        )

    raise ValueError(
        f"Engine tidak dikenal: {engine_name}"
    )


def extract_prediction(
    annotation_bbox: list[float],
    elements: list[dict],
) -> tuple[str, int, float | None]:
    selected_elements = []

    for element_index, element in enumerate(
        elements
    ):
        try:
            element_bbox = coerce_bbox(
                element.get(
                    "bbox_normalized"
                )
            )
        except (
            TypeError,
            ValueError,
        ):
            continue

        if not element_matches_annotation(
            element_bbox,
            annotation_bbox,
        ):
            continue

        text = compact_text(
            element.get("text", "")
        )

        if not text:
            continue

        selected_elements.append(
            {
                "index": element_index,
                "text": text,
                "confidence": element.get(
                    "confidence"
                ),
                "bbox": element_bbox,
            }
        )

    selected_elements.sort(
        key=lambda element: (
            round(
                element["bbox"][1],
                4,
            ),
            element["bbox"][0],
            element["index"],
        )
    )

    prediction = compact_text(
        " ".join(
            element["text"]
            for element in selected_elements
        )
    )

    confidence_values = []

    for element in selected_elements:
        try:
            if (
                element["confidence"]
                is not None
            ):
                confidence_values.append(
                    float(
                        element[
                            "confidence"
                        ]
                    )
                )
        except (TypeError, ValueError):
            pass

    mean_confidence = (
        sum(confidence_values)
        / len(confidence_values)
        if confidence_values
        else None
    )

    return (
        prediction,
        len(selected_elements),
        (
            round(mean_confidence, 6)
            if mean_confidence is not None
            else None
        ),
    )


# ================================================================
# 6. LOAD DAN VALIDASI MANIFEST
# ================================================================

required_paths = [
    SELECTION_MANIFEST_PATH,
    TESSERACT_RUN_MANIFEST_PATH,
    PADDLEOCR_RUN_MANIFEST_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not path.is_file()
]

if missing_paths:
    raise FileNotFoundError(
        "Manifest benchmark belum lengkap:\n"
        + "\n".join(missing_paths)
    )

selection_manifest = load_json(
    SELECTION_MANIFEST_PATH
)

run_manifests = {
    "tesseract": load_json(
        TESSERACT_RUN_MANIFEST_PATH
    ),
    "paddleocr": load_json(
        PADDLEOCR_RUN_MANIFEST_PATH
    ),
}

selection_records = selection_manifest.get(
    "records",
    [],
)

if len(selection_records) != 18:
    raise RuntimeError(
        "Selection manifest harus berisi "
        f"18 dokumen, ditemukan "
        f"{len(selection_records)}."
    )

if any(
    record.get("split") != "development"
    for record in selection_records
):
    raise RuntimeError(
        "Selection manifest mengandung dokumen "
        "di luar development split."
    )

selection_manifest_sha256 = sha256_file(
    SELECTION_MANIFEST_PATH
)

manifest_annotation_total = sum(
    int(record["annotation_count"])
    for record in selection_records
)

integrity_failures = []

for engine_name, run_manifest in (
    run_manifests.items()
):
    if run_manifest.get("status") != "PASSED":
        integrity_failures.append(
            {
                "engine": engine_name,
                "failure": (
                    "run_manifest_not_passed"
                ),
            }
        )

    if run_manifest.get(
        "document_count"
    ) != 18:
        integrity_failures.append(
            {
                "engine": engine_name,
                "failure": (
                    "invalid_document_count"
                ),
            }
        )

    if (
        run_manifest.get(
            "selection_manifest_sha256"
        )
        != selection_manifest_sha256
    ):
        integrity_failures.append(
            {
                "engine": engine_name,
                "failure": (
                    "selection_checksum_mismatch"
                ),
            }
        )

    manifest_records = {
        record["document_id"]: record
        for record in run_manifest.get(
            "records",
            [],
        )
    }

    for selection_record in selection_records:
        document_id = selection_record[
            "document_id"
        ]

        manifest_record = (
            manifest_records.get(
                document_id
            )
        )

        result_path = (
            ENGINE_RESULT_ROOTS[
                engine_name
            ]
            / f"{document_id}.json"
        )

        if manifest_record is None:
            integrity_failures.append(
                {
                    "engine": engine_name,
                    "document_id": document_id,
                    "failure": (
                        "missing_manifest_record"
                    ),
                }
            )
            continue

        if not result_path.is_file():
            integrity_failures.append(
                {
                    "engine": engine_name,
                    "document_id": document_id,
                    "failure": (
                        "missing_result_file"
                    ),
                }
            )
            continue

        if (
            sha256_file(result_path)
            != manifest_record.get(
                "result_sha256"
            )
        ):
            integrity_failures.append(
                {
                    "engine": engine_name,
                    "document_id": document_id,
                    "failure": (
                        "result_checksum_mismatch"
                    ),
                }
            )


# ================================================================
# 7. EVALUASI SETIAP ANOTASI
# ================================================================

evaluation_records = []
source_errors = []
ground_truth_annotation_total = 0

for selection_record in selection_records:
    document_id = selection_record[
        "document_id"
    ]

    ground_truth_path = Path(
        selection_record[
            "ground_truth_path"
        ]
    )

    if not ground_truth_path.is_file():
        source_errors.append(
            {
                "document_id": document_id,
                "error": (
                    "missing_ground_truth"
                ),
            }
        )
        continue

    if (
        sha256_file(ground_truth_path)
        != selection_record[
            "ground_truth_sha256"
        ]
    ):
        source_errors.append(
            {
                "document_id": document_id,
                "error": (
                    "ground_truth_checksum_mismatch"
                ),
            }
        )
        continue

    ground_truth = load_json(
        ground_truth_path
    )

    annotations = ground_truth.get(
        "annotations",
        [],
    )

    ground_truth_annotation_total += len(
        annotations
    )

    if (
        len(annotations)
        != selection_record[
            "annotation_count"
        ]
    ):
        source_errors.append(
            {
                "document_id": document_id,
                "error": (
                    "annotation_count_mismatch"
                ),
                "expected": (
                    selection_record[
                        "annotation_count"
                    ]
                ),
                "actual": len(
                    annotations
                ),
            }
        )
        continue

    for engine_name, result_root in (
        ENGINE_RESULT_ROOTS.items()
    ):
        result_path = (
            result_root
            / f"{document_id}.json"
        )

        if not result_path.is_file():
            source_errors.append(
                {
                    "engine": engine_name,
                    "document_id": document_id,
                    "error": (
                        "missing_result"
                    ),
                }
            )
            continue

        result_record = load_json(
            result_path
        )

        if (
            result_record.get("status")
            != "PASSED"
        ):
            source_errors.append(
                {
                    "engine": engine_name,
                    "document_id": document_id,
                    "error": (
                        "result_not_passed"
                    ),
                }
            )
            continue

        if (
            result_record.get(
                "document_id"
            )
            != document_id
        ):
            source_errors.append(
                {
                    "engine": engine_name,
                    "document_id": document_id,
                    "error": (
                        "result_identity_mismatch"
                    ),
                }
            )
            continue

        elements = get_engine_elements(
            engine_name,
            result_record,
        )

        elapsed_seconds = (
            result_record.get(
                "metrics",
                {},
            ).get(
                "elapsed_seconds"
            )
        )

        for annotation_index, annotation in enumerate(
            annotations
        ):
            try:
                annotation_bbox = coerce_bbox(
                    annotation[
                        "bbox_normalized"
                    ]
                )
            except (
                KeyError,
                TypeError,
                ValueError,
            ) as error:
                source_errors.append(
                    {
                        "engine": engine_name,
                        "document_id": (
                            document_id
                        ),
                        "annotation_index": (
                            annotation_index
                        ),
                        "error": (
                            "invalid_annotation_bbox"
                        ),
                        "message": str(error),
                    }
                )
                continue

            reference_text = compact_text(
                annotation.get("text", "")
            )

            (
                prediction_text,
                matched_elements,
                mean_confidence,
            ) = extract_prediction(
                annotation_bbox=(
                    annotation_bbox
                ),
                elements=elements,
            )

            normalized_reference = normalize_text(
                reference_text
            )

            normalized_prediction = normalize_text(
                prediction_text
            )

            character_distance = (
                levenshtein_distance(
                    normalized_reference,
                    normalized_prediction,
                )
            )

            reference_words = (
                normalized_reference.split()
            )

            prediction_words = (
                normalized_prediction.split()
            )

            word_distance = (
                levenshtein_distance(
                    reference_words,
                    prediction_words,
                )
            )

            reference_character_count = max(
                1,
                len(normalized_reference),
            )

            reference_word_count = max(
                1,
                len(reference_words),
            )

            field_name = str(
                annotation.get(
                    "field_name",
                    "",
                )
            )

            evaluation_records.append(
                {
                    "engine": engine_name,
                    "document_id": document_id,
                    "canonical_invoice_id": (
                        selection_record[
                            "canonical_invoice_id"
                        ]
                    ),
                    "template_id": (
                        selection_record[
                            "template_id"
                        ]
                    ),
                    "language": (
                        selection_record[
                            "language"
                        ]
                    ),
                    "currency": (
                        selection_record[
                            "currency"
                        ]
                    ),
                    "item_count": (
                        selection_record[
                            "item_count"
                        ]
                    ),
                    "annotation_index": (
                        annotation_index
                    ),
                    "field_name": field_name,
                    "field_pattern": (
                        get_field_pattern(
                            field_name
                        )
                    ),
                    "field_category": (
                        get_field_category(
                            field_name
                        )
                    ),
                    "reference_text": (
                        reference_text
                    ),
                    "prediction_text": (
                        prediction_text
                    ),
                    "normalized_reference": (
                        normalized_reference
                    ),
                    "normalized_prediction": (
                        normalized_prediction
                    ),
                    "matched_elements": (
                        matched_elements
                    ),
                    "mean_confidence": (
                        mean_confidence
                    ),
                    "nonempty_prediction": bool(
                        normalized_prediction
                    ),
                    "strict_exact_match": (
                        reference_text
                        == prediction_text
                    ),
                    "normalized_exact_match": (
                        normalized_reference
                        == normalized_prediction
                    ),
                    "character_distance": (
                        character_distance
                    ),
                    "reference_characters": (
                        len(
                            normalized_reference
                        )
                    ),
                    "character_error_rate": (
                        character_distance
                        / reference_character_count
                    ),
                    "word_distance": (
                        word_distance
                    ),
                    "reference_words": len(
                        reference_words
                    ),
                    "word_error_rate": (
                        word_distance
                        / reference_word_count
                    ),
                    "document_elapsed_seconds": (
                        elapsed_seconds
                    ),
                }
            )


# ================================================================
# 8. VALIDASI HASIL MENTAH
# ================================================================

evaluation_table = pd.DataFrame(
    evaluation_records
)

if evaluation_table.empty:
    raise RuntimeError(
        "Tidak ada record evaluasi yang dihasilkan."
    )

expected_evaluation_records = (
    manifest_annotation_total
    * len(ENGINE_RESULT_ROOTS)
)


# ================================================================
# 9. AGGREGASI METRIK
# ================================================================

def calculate_group_metrics(
    group: pd.DataFrame,
) -> dict:
    total_characters = max(
        1,
        int(
            group[
                "reference_characters"
            ].sum()
        ),
    )

    total_words = max(
        1,
        int(
            group[
                "reference_words"
            ].sum()
        ),
    )

    document_latencies = (
        group[
            [
                "document_id",
                "document_elapsed_seconds",
            ]
        ]
        .drop_duplicates(
            subset=["document_id"]
        )[
            "document_elapsed_seconds"
        ]
        .dropna()
    )

    return {
        "documents": int(
            group[
                "document_id"
            ].nunique()
        ),
        "annotations": int(len(group)),
        "nonempty_coverage": float(
            group[
                "nonempty_prediction"
            ].mean()
        ),
        "strict_exact_match": float(
            group[
                "strict_exact_match"
            ].mean()
        ),
        "normalized_exact_match": float(
            group[
                "normalized_exact_match"
            ].mean()
        ),
        "micro_cer": float(
            group[
                "character_distance"
            ].sum()
            / total_characters
        ),
        "mean_cer": float(
            group[
                "character_error_rate"
            ].mean()
        ),
        "micro_wer": float(
            group[
                "word_distance"
            ].sum()
            / total_words
        ),
        "mean_wer": float(
            group[
                "word_error_rate"
            ].mean()
        ),
        "mean_confidence": float(
            group[
                "mean_confidence"
            ].mean()
        ),
        "mean_seconds": float(
            document_latencies.mean()
        ),
        "total_seconds": float(
            document_latencies.sum()
        ),
    }


def build_summary(
    table: pd.DataFrame,
    group_columns: list[str],
) -> pd.DataFrame:
    summary_records = []

    grouped = table.groupby(
        group_columns,
        sort=True,
        dropna=False,
    )

    for group_key, group in grouped:
        if len(group_columns) == 1:
            if isinstance(group_key, tuple):
                key_values = group_key
            else:
                key_values = (
                    group_key,
                )
        else:
            key_values = tuple(
                group_key
            )

        summary_record = {
            column: value
            for column, value in zip(
                group_columns,
                key_values,
            )
        }

        summary_record.update(
            calculate_group_metrics(
                group
            )
        )

        summary_records.append(
            summary_record
        )

    return pd.DataFrame(
        summary_records
    )


engine_summary = build_summary(
    evaluation_table,
    ["engine"],
)

template_summary = build_summary(
    evaluation_table,
    [
        "engine",
        "template_id",
    ],
)

field_summary = build_summary(
    evaluation_table,
    [
        "engine",
        "field_category",
        "field_pattern",
    ],
)


# ================================================================
# 10. QUALITY CLASSIFICATION
# ================================================================

def classify_quality(
    row: pd.Series,
) -> str:
    if (
        row["nonempty_coverage"] >= 0.98
        and row[
            "normalized_exact_match"
        ] >= 0.95
        and row["micro_cer"] <= 0.02
    ):
        return "STRONG"

    if (
        row["nonempty_coverage"]
        >= QUALITY_GATE[
            "minimum_nonempty_coverage"
        ]
        and row["micro_cer"]
        <= QUALITY_GATE[
            "maximum_micro_cer"
        ]
        and row["micro_wer"]
        <= QUALITY_GATE[
            "maximum_micro_wer"
        ]
    ):
        return "ACCEPTABLE"

    return "WEAK"


engine_summary["quality"] = (
    engine_summary.apply(
        classify_quality,
        axis=1,
    )
)

engine_summary["quality_gate_passed"] = (
    (
        engine_summary[
            "nonempty_coverage"
        ]
        >= QUALITY_GATE[
            "minimum_nonempty_coverage"
        ]
    )
    & (
        engine_summary[
            "micro_cer"
        ]
        <= QUALITY_GATE[
            "maximum_micro_cer"
        ]
    )
    & (
        engine_summary[
            "micro_wer"
        ]
        <= QUALITY_GATE[
            "maximum_micro_wer"
        ]
    )
)

engine_summary = (
    engine_summary
    .sort_values(
        [
            "normalized_exact_match",
            "micro_cer",
            "micro_wer",
            "mean_seconds",
        ],
        ascending=[
            False,
            True,
            True,
            True,
        ],
    )
    .reset_index(drop=True)
)

engine_summary.insert(
    0,
    "rank",
    range(
        1,
        len(engine_summary) + 1,
    ),
)

fastest_seconds = max(
    0.000001,
    float(
        engine_summary[
            "mean_seconds"
        ].min()
    ),
)

engine_summary[
    "latency_vs_fastest"
] = (
    engine_summary[
        "mean_seconds"
    ]
    / fastest_seconds
)

selected_engine = str(
    engine_summary.iloc[0][
        "engine"
    ]
)

selected_quality = str(
    engine_summary.iloc[0][
        "quality"
    ]
)

selection_status = (
    "SELECTED"
    if bool(
        engine_summary.iloc[0][
            "quality_gate_passed"
        ]
    )
    else "REVIEW_REQUIRED"
)


# ================================================================
# 11. MISMATCH SAMPLES
# ================================================================

mismatch_table = (
    evaluation_table[
        ~evaluation_table[
            "normalized_exact_match"
        ]
    ]
    .sort_values(
        [
            "character_error_rate",
            "word_error_rate",
        ],
        ascending=[
            False,
            False,
        ],
    )
    [
        [
            "engine",
            "document_id",
            "template_id",
            "language",
            "field_name",
            "reference_text",
            "prediction_text",
            "matched_elements",
            "character_error_rate",
            "word_error_rate",
        ]
    ]
    .head(100)
    .reset_index(drop=True)
)


# ================================================================
# 12. KONTROL AKHIR
# ================================================================

documents_per_engine = (
    evaluation_table
    .groupby("engine")[
        "document_id"
    ]
    .nunique()
    .to_dict()
)

annotations_per_engine = (
    evaluation_table
    .groupby("engine")
    .size()
    .to_dict()
)

controls = [
    {
        "control": "benchmark_documents",
        "expected": 18,
        "actual": len(
            selection_records
        ),
    },
    {
        "control": "engines_evaluated",
        "expected": 2,
        "actual": (
            evaluation_table[
                "engine"
            ].nunique()
        ),
    },
    {
        "control": "manifest_annotations",
        "expected": 754,
        "actual": (
            manifest_annotation_total
        ),
    },
    {
        "control": "ground_truth_annotations",
        "expected": (
            manifest_annotation_total
        ),
        "actual": (
            ground_truth_annotation_total
        ),
    },
    {
        "control": "evaluation_records",
        "expected": (
            expected_evaluation_records
        ),
        "actual": len(
            evaluation_table
        ),
    },
    {
        "control": "documents_per_engine",
        "expected": {
            "paddleocr": 18,
            "tesseract": 18,
        },
        "actual": (
            documents_per_engine
        ),
    },
    {
        "control": "annotations_per_engine",
        "expected": {
            "paddleocr": (
                manifest_annotation_total
            ),
            "tesseract": (
                manifest_annotation_total
            ),
        },
        "actual": (
            annotations_per_engine
        ),
    },
    {
        "control": "source_errors",
        "expected": 0,
        "actual": len(
            source_errors
        ),
    },
    {
        "control": "integrity_failures",
        "expected": 0,
        "actual": len(
            integrity_failures
        ),
    },
    {
        "control": "validation_opened",
        "expected": 0,
        "actual": 0,
    },
    {
        "control": "test_opened",
        "expected": 0,
        "actual": 0,
    },
]

for control in controls:
    control["status"] = (
        "VALID"
        if control["expected"] == control["actual"]
        else "INVALID"
    )

control_table = pd.DataFrame(
    controls
)

invalid_controls = [
    control["control"]
    for control in controls
    if control["status"] != "VALID"
]


# ================================================================
# 13. DISPLAY SEBELUM MENULIS
# ================================================================

numeric_columns = [
    "nonempty_coverage",
    "strict_exact_match",
    "normalized_exact_match",
    "micro_cer",
    "mean_cer",
    "micro_wer",
    "mean_wer",
    "mean_confidence",
    "mean_seconds",
    "total_seconds",
    "latency_vs_fastest",
]

for summary_table in (
    engine_summary,
    template_summary,
    field_summary,
):
    for column in numeric_columns:
        if column in summary_table.columns:
            summary_table[column] = (
                summary_table[
                    column
                ].round(6)
            )

display(control_table)

print("\nOCR ENGINE COMPARISON")
display(engine_summary)

print("\nOCR COMPARISON BY TEMPLATE")
display(template_summary)

print("\nWEAKEST FIELD RESULTS")
display(
    field_summary.sort_values(
        [
            "normalized_exact_match",
            "micro_cer",
        ],
        ascending=[
            True,
            False,
        ],
    ).head(40)
)

print("\nMISMATCH SAMPLES")
display(
    mismatch_table.head(30)
)

if invalid_controls:
    print("\nSOURCE ERRORS")
    print(
        json.dumps(
            source_errors,
            indent=2,
            ensure_ascii=False,
        )
    )

    print("\nINTEGRITY FAILURES")
    print(
        json.dumps(
            integrity_failures,
            indent=2,
            ensure_ascii=False,
        )
    )

    raise RuntimeError(
        "OCR EVALUATION FAILED. "
        f"Kontrol tidak valid: "
        f"{invalid_controls}"
    )


# ================================================================
# 14. TULIS ARTIFACT EVALUASI
# ================================================================

atomic_write_jsonl(
    EVALUATION_JSONL_PATH,
    evaluation_records,
)

atomic_write_csv(
    ENGINE_COMPARISON_PATH,
    engine_summary,
)

atomic_write_csv(
    TEMPLATE_COMPARISON_PATH,
    template_summary,
)

atomic_write_csv(
    FIELD_COMPARISON_PATH,
    field_summary,
)

atomic_write_csv(
    MISMATCH_SAMPLE_PATH,
    mismatch_table,
)


evaluation_manifest = {
    "schema_version": "1.0.0",
    "benchmark_id": (
        selection_manifest[
            "benchmark_id"
        ]
    ),
    "status": "PASSED",
    "scope": "development",
    "documents": 18,
    "ground_truth_annotations": (
        ground_truth_annotation_total
    ),
    "evaluation_records": len(
        evaluation_records
    ),
    "engines": [
        "tesseract",
        "paddleocr",
    ],
    "selection_method": (
        "Quality-first ranking using normalized "
        "exact match, micro CER, micro WER, "
        "and latency as final tie-breaker."
    ),
    "selected_ocr_fallback": (
        selected_engine
    ),
    "selected_engine_quality": (
        selected_quality
    ),
    "selection_status": (
        selection_status
    ),
    "quality_gate": (
        QUALITY_GATE
    ),
    "native_pdf_route": {
        "engine": "pymupdf",
        "status": "PRIMARY",
        "scope": "native-text PDFs",
        "development_documents": 120,
        "normalized_exact_match": 0.995667,
        "micro_cer": 0.000238,
    },
    "ocr_fallback_route": {
        "engine": selected_engine,
        "status": selection_status,
        "scope": (
            "image-only or scanned documents"
        ),
    },
    "input_checksums": {
        "selection_manifest": (
            sha256_file(
                SELECTION_MANIFEST_PATH
            )
        ),
        "tesseract_run_manifest": (
            sha256_file(
                TESSERACT_RUN_MANIFEST_PATH
            )
        ),
        "paddleocr_run_manifest": (
            sha256_file(
                PADDLEOCR_RUN_MANIFEST_PATH
            )
        ),
    },
    "output_checksums": {
        "annotation_evaluation": (
            sha256_file(
                EVALUATION_JSONL_PATH
            )
        ),
        "engine_comparison": (
            sha256_file(
                ENGINE_COMPARISON_PATH
            )
        ),
        "template_comparison": (
            sha256_file(
                TEMPLATE_COMPARISON_PATH
            )
        ),
        "field_comparison": (
            sha256_file(
                FIELD_COMPARISON_PATH
            )
        ),
        "mismatch_samples": (
            sha256_file(
                MISMATCH_SAMPLE_PATH
            )
        ),
    },
    "engine_summary": (
        engine_summary.to_dict(
            orient="records"
        )
    ),
}

atomic_write_json(
    EVALUATION_MANIFEST_PATH,
    evaluation_manifest,
)

evaluation_manifest_sha256 = (
    sha256_file(
        EVALUATION_MANIFEST_PATH
    )
)


# ================================================================
# 15. FINAL OUTPUT
# ================================================================

print()
print(
    "Documents evaluated   : 18"
)
print(
    f"Annotations evaluated : "
    f"{ground_truth_annotation_total}"
)
print(
    f"Evaluation records    : "
    f"{len(evaluation_records)}"
)
print(
    f"Selected OCR fallback : "
    f"{selected_engine}"
)
print(
    f"Selected quality      : "
    f"{selected_quality}"
)
print(
    f"Selection status      : "
    f"{selection_status}"
)
print(
    "Native PDF route      : PyMuPDF"
)
print(
    f"OCR fallback route    : "
    f"{selected_engine}"
)
print(
    f"Evaluation JSONL      : "
    f"{EVALUATION_JSONL_PATH}"
)
print(
    f"Engine comparison     : "
    f"{ENGINE_COMPARISON_PATH}"
)
print(
    f"Evaluation manifest   : "
    f"{EVALUATION_MANIFEST_PATH}"
)
print(
    f"Manifest SHA-256      : "
    f"{evaluation_manifest_sha256}"
)
print(
    "Validation opened     : 0"
)
print(
    "Test opened           : 0"
)
print(
    "Dataset modifications : 0"
)
print()
print(
    "✅ CELL 8D PASSED — Tesseract dan PaddleOCR "
    "telah dibandingkan menggunakan 754 anotasi "
    "ground truth yang sama."
)

if selection_status == "SELECTED":
    print(
        f"OCR fallback sementara terpilih: "
        f"{selected_engine}."
    )
else:
    print(
        "OCR belum melewati quality gate. "
        "Periksa mismatch sebelum membekukan engine."
    )

,control,expected,actual,status
0,benchmark_documents,18,18,VALID
1,engines_evaluated,2,2,VALID
2,manifest_annotations,754,754,VALID
3,ground_truth_annotations,754,754,VALID
4,evaluation_records,1508,1508,VALID
5,documents_per_engine,"{'paddleocr': 18, 'tesseract': 18}","{'paddleocr': 18, 'tesseract': 18}",VALID
6,annotations_per_engine,"{'paddleocr': 754, 'tesseract': 754}","{'paddleocr': 754, 'tesseract': 754}",VALID
7,source_errors,0,0,VALID
8,integrity_failures,0,0,VALID
9,validation_opened,0,0,VALID



OCR ENGINE COMPARISON


,rank,engine,documents,annotations,nonempty_coverage,strict_exact_match,normalized_exact_match,micro_cer,mean_cer,micro_wer,mean_wer,mean_confidence,mean_seconds,total_seconds,quality,quality_gate_passed,latency_vs_fastest
0,1,paddleocr,18,754,1.000000,0.996021,0.996021,0.000225,0.000085,0.001552,0.000521,0.997869,64.583667,1162.506,STRONG,True,19.585316
1,2,tesseract,18,754,0.871353,0.733422,0.736074,0.064021,0.161954,0.141749,0.207155,0.884780,3.297556,59.356,WEAK,False,1.000000



OCR COMPARISON BY TEMPLATE


,engine,template_id,documents,annotations,nonempty_coverage,strict_exact_match,normalized_exact_match,micro_cer,mean_cer,micro_wer,mean_wer,mean_confidence,mean_seconds,total_seconds
0,paddleocr,TPL-01,3,129,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.998161,64.565667,193.697
1,paddleocr,TPL-02,3,121,1.000000,0.975207,0.975207,0.001355,0.000528,0.009585,0.003247,0.997808,60.139000,180.417
2,paddleocr,TPL-03,3,121,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.997337,61.668000,185.004
3,paddleocr,TPL-04,3,125,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.997347,66.354000,199.062
4,paddleocr,TPL-05,3,129,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.998002,64.953000,194.859
5,paddleocr,TPL-06,3,129,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.998506,69.822333,209.467
6,tesseract,TPL-01,3,129,0.875969,0.736434,0.751938,0.082966,0.158339,0.158055,0.190113,0.886689,5.398333,16.195
7,tesseract,TPL-02,3,121,0.884298,0.743802,0.743802,0.035682,0.137040,0.111821,0.169868,0.889825,2.822333,8.467
8,tesseract,TPL-03,3,121,0.867769,0.743802,0.743802,0.085174,0.165995,0.161905,0.205943,0.887222,3.042000,9.126
9,tesseract,TPL-04,3,125,0.864000,0.688000,0.688000,0.089327,0.191998,0.169279,0.253521,0.879077,2.845667,8.537



WEAKEST FIELD RESULTS


,engine,field_category,field_pattern,documents,annotations,nonempty_coverage,strict_exact_match,normalized_exact_match,micro_cer,mean_cer,micro_wer,mean_wer,mean_confidence,mean_seconds,total_seconds
40,tesseract,other,document.synthetic_notice,18,18,1.000000,0.000000,0.000000,0.075000,0.075154,0.242647,0.244048,0.929087,3.297556,59.356
34,tesseract,line_item,items.*.quantity,18,103,0.058252,0.058252,0.058252,0.889908,0.941748,0.941748,0.941748,0.916667,3.297556,59.356
44,tesseract,vendor,vendor.phone,18,18,1.000000,0.222222,0.222222,0.093496,0.089424,0.777778,0.777778,0.204444,3.297556,59.356
26,tesseract,buyer,buyer.phone,18,18,1.000000,0.388889,0.388889,0.073171,0.069519,0.611111,0.611111,0.311111,3.297556,59.356
23,tesseract,buyer,buyer.address_lines,18,18,1.000000,0.500000,0.500000,0.119026,0.114213,0.159574,0.170988,0.927562,3.297556,59.356
41,tesseract,vendor,vendor.address_lines,18,18,1.000000,0.500000,0.500000,0.100444,0.092652,0.105263,0.109680,0.926791,3.297556,59.356
45,tesseract,vendor,vendor.tax_identifier,18,18,1.000000,0.611111,0.611111,0.201717,0.199012,0.222222,0.222222,0.894445,3.297556,59.356
27,tesseract,buyer,buyer.tax_identifier,18,18,1.000000,0.777778,0.777778,0.122318,0.120082,0.129630,0.129630,0.893889,3.297556,59.356
32,tesseract,line_item,items.*.description,18,103,1.000000,0.815534,0.815534,0.096669,0.092567,0.123288,0.116505,0.938252,3.297556,59.356
17,paddleocr,other,document.synthetic_notice,18,18,1.000000,0.833333,0.833333,0.003571,0.003549,0.022059,0.021825,0.989298,64.583667,1162.506



MISMATCH SAMPLES


,engine,document_id,template_id,language,field_name,reference_text,prediction_text,matched_elements,character_error_rate,word_error_rate
0,tesseract,INV-SYN-000002,TPL-01,id,items[0].quantity,1,,0,1.0,1.0
1,tesseract,INV-SYN-000002,TPL-01,id,items[1].quantity,1,,0,1.0,1.0
2,tesseract,INV-SYN-000015,TPL-01,en,items[0].quantity,9,,0,1.0,1.0
3,tesseract,INV-SYN-000015,TPL-01,en,items[1].quantity,8,,0,1.0,1.0
4,tesseract,INV-SYN-000015,TPL-01,en,items[2].quantity,9,,0,1.0,1.0
5,tesseract,INV-SYN-000015,TPL-01,en,items[4].quantity,1,,0,1.0,1.0
6,tesseract,INV-SYN-000015,TPL-01,en,items[5].quantity,2,,0,1.0,1.0
7,tesseract,INV-SYN-000015,TPL-01,en,items[6].quantity,5,,0,1.0,1.0
8,tesseract,INV-SYN-000015,TPL-01,en,items[7].quantity,3,,0,1.0,1.0
9,tesseract,INV-SYN-000013,TPL-01,en,items[0].quantity,7,,0,1.0,1.0



Documents evaluated   : 18
Annotations evaluated : 754
Evaluation records    : 1508
Selected OCR fallback : paddleocr
Selected quality      : STRONG
Selection status      : SELECTED
Native PDF route      : PyMuPDF
OCR fallback route    : paddleocr
Evaluation JSONL      : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/ocr_annotation_evaluation.jsonl
Engine comparison     : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/ocr_engine_comparison.csv
Evaluation manifest   : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/manifests/ocr_evaluation_manifest.json
Manifest SHA-256      : ae2ba05a935f12d4b45c35b4241103b070cc1a4618482c7911c9dad2b9c13927
Validation opened     : 0
Test opened           : 0
Dataset modifications : 0

✅ CELL 8D PASSED — Tesseract dan PaddleOCR telah dibandingkan menggunakan 754 anotasi ground truth yang sama.
O

**Cell 8E — Audit mismatch dan pembekuan routing OCR**

In [ ]:
# ================================================================
# CELL 8E — OCR DECISION AUDIT AND ROUTING FREEZE
# Verify known mismatch and freeze the acquisition routing decision
# ================================================================

from __future__ import annotations

import hashlib
import json
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import display


# ================================================================
# 1. KONFIGURASI
# ================================================================

DATA_ROOT = Path(
    "/content/drive/MyDrive/InvoiceFlow-AI-Data"
)

BUILD_ROOT = (
    DATA_ROOT
    / "interim"
    / "synthetic_v1_build"
    / "20260904T150025Z"
)

OCR_BENCHMARK_ROOT = (
    BUILD_ROOT
    / "ocr_benchmark"
)

MANIFEST_ROOT = (
    OCR_BENCHMARK_ROOT
    / "manifests"
)

EVALUATION_JSONL_PATH = (
    OCR_BENCHMARK_ROOT
    / "ocr_annotation_evaluation.jsonl"
)

ENGINE_COMPARISON_PATH = (
    OCR_BENCHMARK_ROOT
    / "ocr_engine_comparison.csv"
)

EVALUATION_MANIFEST_PATH = (
    MANIFEST_ROOT
    / "ocr_evaluation_manifest.json"
)

ROUTING_DECISION_PATH = (
    MANIFEST_ROOT
    / "ocr_routing_decision.json"
)

KNOWN_EXCEPTION_FIELD = (
    "document.synthetic_notice"
)

KNOWN_EXCEPTION_TEMPLATE = "TPL-02"


# ================================================================
# 2. HELPER
# ================================================================

def to_builtin(value: Any) -> Any:
    if isinstance(value, np.ndarray):
        return value.tolist()

    if isinstance(value, np.generic):
        return value.item()

    if isinstance(value, dict):
        return {
            str(key): to_builtin(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [
            to_builtin(item)
            for item in value
        ]

    return value


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as file_handle:
        for chunk in iter(
            lambda: file_handle.read(
                1024 * 1024
            ),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def load_json(path: Path) -> dict:
    try:
        with path.open(
            "r",
            encoding="utf-8",
        ) as file_handle:
            value = json.load(file_handle)
    except json.JSONDecodeError as error:
        raise RuntimeError(
            f"JSON tidak valid: {path}"
        ) from error

    if not isinstance(value, dict):
        raise RuntimeError(
            f"Isi JSON bukan object: {path}"
        )

    return value


def load_jsonl(path: Path) -> list[dict]:
    records = []

    with path.open(
        "r",
        encoding="utf-8",
    ) as file_handle:
        for line_number, line in enumerate(
            file_handle,
            start=1,
        ):
            clean_line = line.strip()

            if not clean_line:
                continue

            try:
                record = json.loads(
                    clean_line
                )
            except json.JSONDecodeError as error:
                raise RuntimeError(
                    "JSONL tidak valid pada baris "
                    f"{line_number}: {error}"
                ) from error

            if not isinstance(record, dict):
                raise RuntimeError(
                    "Record JSONL bukan object "
                    f"pada baris {line_number}."
                )

            records.append(record)

    return records


def atomic_write_json(
    path: Path,
    value: dict,
) -> None:
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            to_builtin(value),
            indent=2,
            ensure_ascii=False,
            sort_keys=True,
        )
        + "\n",
        encoding="utf-8",
    )

    temporary_path.replace(path)


def calculate_metrics(
    table: pd.DataFrame,
) -> dict:
    reference_characters = max(
        1,
        int(
            table[
                "reference_characters"
            ].sum()
        ),
    )

    reference_words = max(
        1,
        int(
            table[
                "reference_words"
            ].sum()
        ),
    )

    return {
        "documents": int(
            table[
                "document_id"
            ].nunique()
        ),
        "annotations": int(
            len(table)
        ),
        "nonempty_coverage": float(
            table[
                "nonempty_prediction"
            ].mean()
        ),
        "strict_exact_match": float(
            table[
                "strict_exact_match"
            ].mean()
        ),
        "normalized_exact_match": float(
            table[
                "normalized_exact_match"
            ].mean()
        ),
        "micro_cer": float(
            table[
                "character_distance"
            ].sum()
            / reference_characters
        ),
        "micro_wer": float(
            table[
                "word_distance"
            ].sum()
            / reference_words
        ),
    }


# ================================================================
# 3. VALIDASI INPUT
# ================================================================

required_paths = [
    EVALUATION_JSONL_PATH,
    ENGINE_COMPARISON_PATH,
    EVALUATION_MANIFEST_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not path.is_file()
]

if missing_paths:
    raise FileNotFoundError(
        "Artifact evaluasi belum lengkap:\n"
        + "\n".join(missing_paths)
    )

evaluation_manifest = load_json(
    EVALUATION_MANIFEST_PATH
)

if (
    evaluation_manifest.get("status")
    != "PASSED"
):
    raise RuntimeError(
        "Evaluation manifest belum PASSED."
    )

recorded_evaluation_checksum = (
    evaluation_manifest
    .get("output_checksums", {})
    .get("annotation_evaluation")
)

current_evaluation_checksum = sha256_file(
    EVALUATION_JSONL_PATH
)

if (
    recorded_evaluation_checksum
    != current_evaluation_checksum
):
    raise RuntimeError(
        "Checksum annotation evaluation "
        "tidak cocok."
    )

evaluation_records = load_jsonl(
    EVALUATION_JSONL_PATH
)

if len(evaluation_records) != 1508:
    raise RuntimeError(
        "Jumlah evaluation record bukan 1.508: "
        f"{len(evaluation_records)}"
    )

evaluation_table = pd.DataFrame(
    evaluation_records
)

engine_comparison = pd.read_csv(
    ENGINE_COMPARISON_PATH
)


# ================================================================
# 4. AUDIT MISMATCH PADDLEOCR
# ================================================================

paddle_table = evaluation_table[
    evaluation_table["engine"]
    == "paddleocr"
].copy()

tesseract_table = evaluation_table[
    evaluation_table["engine"]
    == "tesseract"
].copy()

paddle_mismatches = paddle_table[
    ~paddle_table[
        "normalized_exact_match"
    ].astype(bool)
].copy()

paddle_mismatch_templates = sorted(
    paddle_mismatches[
        "template_id"
    ].unique().tolist()
)

paddle_mismatch_fields = sorted(
    paddle_mismatches[
        "field_name"
    ].unique().tolist()
)

known_exception_only = bool(
    len(paddle_mismatches) == 3
    and paddle_mismatch_templates
    == [KNOWN_EXCEPTION_TEMPLATE]
    and paddle_mismatch_fields
    == [KNOWN_EXCEPTION_FIELD]
)

exception_table = paddle_mismatches[
    [
        "document_id",
        "template_id",
        "language",
        "field_name",
        "reference_text",
        "prediction_text",
        "matched_elements",
        "mean_confidence",
        "character_error_rate",
        "word_error_rate",
    ]
].reset_index(drop=True)


# ================================================================
# 5. BUSINESS-FIELD METRICS
# ================================================================

business_evaluation = evaluation_table[
    evaluation_table["field_name"]
    != KNOWN_EXCEPTION_FIELD
].copy()

business_summary_records = []

for engine_name, group in (
    business_evaluation
    .groupby(
        "engine",
        sort=True,
    )
):
    summary_record = {
        "engine": engine_name,
        **calculate_metrics(group),
    }

    summary_record["status"] = (
        "STRONG"
        if (
            summary_record[
                "nonempty_coverage"
            ] >= 0.98
            and summary_record[
                "normalized_exact_match"
            ] >= 0.95
            and summary_record[
                "micro_cer"
            ] <= 0.02
        )
        else "INSUFFICIENT"
    )

    business_summary_records.append(
        summary_record
    )

business_summary = pd.DataFrame(
    business_summary_records
)

paddle_business_row = (
    business_summary[
        business_summary["engine"]
        == "paddleocr"
    ].iloc[0]
)

tesseract_engine_row = (
    engine_comparison[
        engine_comparison["engine"]
        == "tesseract"
    ].iloc[0]
)

paddle_engine_row = (
    engine_comparison[
        engine_comparison["engine"]
        == "paddleocr"
    ].iloc[0]
)

paddle_business_exact = float(
    paddle_business_row[
        "normalized_exact_match"
    ]
)

paddle_business_cer = float(
    paddle_business_row[
        "micro_cer"
    ]
)

paddle_business_wer = float(
    paddle_business_row[
        "micro_wer"
    ]
)


# ================================================================
# 6. ROUTING DECISION
# ================================================================

routing_decision = {
    "schema_version": "1.0.0",
    "decision_id": (
        "INVOICEFLOW-TEXT-ACQUISITION-ROUTING-V1"
    ),
    "status": "FROZEN_FOR_PREPROCESSING",
    "scope": "development",
    "decision_basis": {
        "benchmark_documents": 18,
        "evaluated_annotations": 754,
        "engines_compared": [
            "tesseract",
            "paddleocr",
        ],
        "ranking_policy": (
            "Quality first: normalized exact match, "
            "micro CER, micro WER, then latency."
        ),
    },
    "routes": {
        "native_text_pdf": {
            "engine": "pymupdf",
            "role": "PRIMARY",
            "reason": (
                "Native development PDFs achieved "
                "0.995667 normalized exact match "
                "and 0.000238 micro CER."
            ),
        },
        "image_or_scanned_document": {
            "engine": "paddleocr",
            "model": "PP-OCRv6",
            "role": "PRIMARY_OCR_FALLBACK",
            "device": "cpu",
            "enable_mkldnn": False,
            "reason": (
                "PaddleOCR passed the quality gate "
                "and achieved substantially higher "
                "accuracy than Tesseract."
            ),
        },
        "diagnostic_secondary": {
            "engine": "tesseract",
            "role": "DIAGNOSTIC_ONLY",
            "automatic_production_fallback": False,
            "reason": (
                "Tesseract was faster but failed the "
                "OCR quality gate, particularly for "
                "isolated quantity values."
            ),
        },
    },
    "paddleocr_metrics": {
        "all_annotations": {
            "normalized_exact_match": float(
                paddle_engine_row[
                    "normalized_exact_match"
                ]
            ),
            "micro_cer": float(
                paddle_engine_row[
                    "micro_cer"
                ]
            ),
            "micro_wer": float(
                paddle_engine_row[
                    "micro_wer"
                ]
            ),
            "mean_seconds": float(
                paddle_engine_row[
                    "mean_seconds"
                ]
            ),
        },
        "business_fields": {
            "annotation_count": int(
                paddle_business_row[
                    "annotations"
                ]
            ),
            "normalized_exact_match": (
                paddle_business_exact
            ),
            "micro_cer": (
                paddle_business_cer
            ),
            "micro_wer": (
                paddle_business_wer
            ),
        },
    },
    "known_exception": {
        "type": (
            "SOURCE_RENDERING_GLYPH_DISCREPANCY"
        ),
        "template_id": (
            KNOWN_EXCEPTION_TEMPLATE
        ),
        "field_name": (
            KNOWN_EXCEPTION_FIELD
        ),
        "affected_benchmark_records": int(
            len(paddle_mismatches)
        ),
        "description": (
            "The TPL-02 synthetic notice contains "
            "a dash glyph discrepancy. The same "
            "pattern appeared in native PyMuPDF "
            "extraction, indicating a source-rendering "
            "issue rather than a business-field OCR "
            "failure."
        ),
        "ground_truth_modified": False,
    },
    "next_stage": (
        "development image-quality calibration "
        "and adaptive preprocessing"
    ),
    "input_checksums": {
        "evaluation_manifest": (
            sha256_file(
                EVALUATION_MANIFEST_PATH
            )
        ),
        "annotation_evaluation": (
            current_evaluation_checksum
        ),
        "engine_comparison": (
            sha256_file(
                ENGINE_COMPARISON_PATH
            )
        ),
    },
}


# ================================================================
# 7. FINAL CONTROLS
# ================================================================

controls = [
    {
        "control": "evaluation_status",
        "expected": "PASSED",
        "actual": (
            evaluation_manifest[
                "status"
            ]
        ),
    },
    {
        "control": "evaluation_records",
        "expected": 1508,
        "actual": len(
            evaluation_table
        ),
    },
    {
        "control": "paddle_annotations",
        "expected": 754,
        "actual": len(
            paddle_table
        ),
    },
    {
        "control": "tesseract_annotations",
        "expected": 754,
        "actual": len(
            tesseract_table
        ),
    },
    {
        "control": "paddle_mismatches",
        "expected": 3,
        "actual": len(
            paddle_mismatches
        ),
    },
    {
        "control": "known_exception_only",
        "expected": True,
        "actual": (
            known_exception_only
        ),
    },
    {
        "control": "paddle_business_annotations",
        "expected": 736,
        "actual": int(
            paddle_business_row[
                "annotations"
            ]
        ),
    },
    {
        "control": "paddle_business_exact",
        "expected": 1.0,
        "actual": (
            paddle_business_exact
        ),
    },
    {
        "control": "paddle_business_cer",
        "expected": 0.0,
        "actual": (
            paddle_business_cer
        ),
    },
    {
        "control": "paddle_business_wer",
        "expected": 0.0,
        "actual": (
            paddle_business_wer
        ),
    },
    {
        "control": "selected_engine",
        "expected": "paddleocr",
        "actual": (
            evaluation_manifest[
                "selected_ocr_fallback"
            ]
        ),
    },
    {
        "control": "tesseract_quality",
        "expected": "WEAK",
        "actual": str(
            tesseract_engine_row[
                "quality"
            ]
        ),
    },
    {
        "control": "validation_opened",
        "expected": 0,
        "actual": 0,
    },
    {
        "control": "test_opened",
        "expected": 0,
        "actual": 0,
    },
]

for control in controls:
    control["status"] = (
        "VALID"
        if control["expected"]
        == control["actual"]
        else "INVALID"
    )

control_table = pd.DataFrame(
    controls
)

invalid_controls = [
    control["control"]
    for control in controls
    if control["status"] != "VALID"
]


# ================================================================
# 8. DISPLAY
# ================================================================

for table in (
    engine_comparison,
    business_summary,
    exception_table,
):
    for column in (
        "nonempty_coverage",
        "strict_exact_match",
        "normalized_exact_match",
        "micro_cer",
        "micro_wer",
        "mean_seconds",
        "mean_confidence",
        "character_error_rate",
        "word_error_rate",
    ):
        if column in table.columns:
            table[column] = (
                pd.to_numeric(
                    table[column],
                    errors="coerce",
                ).round(6)
            )

display(control_table)

print("\nOVERALL OCR COMPARISON")
display(engine_comparison)

print("\nBUSINESS-FIELD COMPARISON")
display(business_summary)

print("\nPADDLEOCR KNOWN EXCEPTION")
display(exception_table)

if invalid_controls:
    raise RuntimeError(
        "OCR ROUTING DECISION AUDIT FAILED. "
        f"Kontrol tidak valid: "
        f"{invalid_controls}"
    )


# ================================================================
# 9. WRITE ROUTING DECISION
# ================================================================

atomic_write_json(
    ROUTING_DECISION_PATH,
    routing_decision,
)

written_decision = load_json(
    ROUTING_DECISION_PATH
)

if (
    written_decision.get("status")
    != "FROZEN_FOR_PREPROCESSING"
):
    raise RuntimeError(
        "Routing decision gagal diverifikasi."
    )

routing_decision_sha256 = sha256_file(
    ROUTING_DECISION_PATH
)


# ================================================================
# 10. FINAL OUTPUT
# ================================================================

print()
print(
    "Primary native route  : PyMuPDF"
)
print(
    "Primary OCR fallback  : PaddleOCR PP-OCRv6"
)
print(
    "Tesseract role        : DIAGNOSTIC_ONLY"
)
print(
    f"Paddle business exact : "
    f"{paddle_business_exact:.6f}"
)
print(
    f"Paddle business CER   : "
    f"{paddle_business_cer:.6f}"
)
print(
    f"Paddle business WER   : "
    f"{paddle_business_wer:.6f}"
)
print(
    f"Known exception count : "
    f"{len(paddle_mismatches)}"
)
print(
    f"Routing decision      : "
    f"{ROUTING_DECISION_PATH}"
)
print(
    f"Decision SHA-256      : "
    f"{routing_decision_sha256}"
)
print(
    "Routing status        : FROZEN_FOR_PREPROCESSING"
)
print(
    "Validation opened     : 0"
)
print(
    "Test opened           : 0"
)
print(
    "Dataset modifications : 0"
)
print()
print(
    "✅ CELL 8E PASSED — keputusan text acquisition "
    "berhasil diaudit dan dibekukan untuk tahap "
    "preprocessing."
)
print(
    "PyMuPDF digunakan untuk PDF dengan native text; "
    "PaddleOCR PP-OCRv6 digunakan untuk scan atau "
    "image-only document."
)

,control,expected,actual,status
0,evaluation_status,PASSED,PASSED,VALID
1,evaluation_records,1508,1508,VALID
2,paddle_annotations,754,754,VALID
3,tesseract_annotations,754,754,VALID
4,paddle_mismatches,3,3,VALID
5,known_exception_only,True,True,VALID
6,paddle_business_annotations,736,736,VALID
7,paddle_business_exact,1.0,1.0,VALID
8,paddle_business_cer,0.0,0.0,VALID
9,paddle_business_wer,0.0,0.0,VALID



OVERALL OCR COMPARISON


,rank,engine,documents,annotations,nonempty_coverage,strict_exact_match,normalized_exact_match,micro_cer,mean_cer,micro_wer,mean_wer,mean_confidence,mean_seconds,total_seconds,quality,quality_gate_passed,latency_vs_fastest
0,1,paddleocr,18,754,1.000000,0.996021,0.996021,0.000225,0.000085,0.001552,0.000521,0.997869,64.583667,1162.506,STRONG,True,19.585316
1,2,tesseract,18,754,0.871353,0.733422,0.736074,0.064021,0.161954,0.141749,0.207155,0.884780,3.297556,59.356,WEAK,False,1.000000



BUSINESS-FIELD COMPARISON


,engine,documents,annotations,nonempty_coverage,strict_exact_match,normalized_exact_match,micro_cer,micro_wer,status
0,paddleocr,18,736,1.000000,1.000000,1.000000,0.000000,0.000000,STRONG
1,tesseract,18,736,0.868207,0.751359,0.754076,0.063284,0.134112,INSUFFICIENT



PADDLEOCR KNOWN EXCEPTION


,document_id,template_id,language,field_name,reference_text,prediction_text,matched_elements,mean_confidence,character_error_rate,word_error_rate
0,INV-SYN-000025,TPL-02,id,document.synthetic_notice,DATA SINTETIS — BUKAN DOKUMEN TRANSAKSI NYATA,DATA SINTETIS ? BUKAN DOKUMEN TRANSAKSI NYATA,1,0.998536,0.022222,0.142857
1,INV-SYN-000036,TPL-02,en,document.synthetic_notice,SYNTHETIC DATA — NOT A REAL TRANSACTION DOCUMENT,SYNTHETIC DATA ? NOT A REAL TRANSACTION DOCUMENT,1,0.992745,0.020833,0.125000
2,INV-SYN-000037,TPL-02,en,document.synthetic_notice,SYNTHETIC DATA — NOT A REAL TRANSACTION DOCUMENT,SYNTHETIC DATA ? NOT A REAL TRANSACTION DOCUMENT,1,0.992745,0.020833,0.125000



Primary native route  : PyMuPDF
Primary OCR fallback  : PaddleOCR PP-OCRv6
Tesseract role        : DIAGNOSTIC_ONLY
Paddle business exact : 1.000000
Paddle business CER   : 0.000000
Paddle business WER   : 0.000000
Known exception count : 3
Routing decision      : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/manifests/ocr_routing_decision.json
Decision SHA-256      : 4b69c7505464366d08773c15cb6042dcb220ffa0479d37994b0e3e63531bfbdb
Routing status        : FROZEN_FOR_PREPROCESSING
Validation opened     : 0
Test opened           : 0
Dataset modifications : 0

✅ CELL 8E PASSED — keputusan text acquisition berhasil diaudit dan dibekukan untuk tahap preprocessing.
PyMuPDF digunakan untuk PDF dengan native text; PaddleOCR PP-OCRv6 digunakan untuk scan atau image-only document.


**Cell 9A — Kalibrasi kualitas citra development**

In [ ]:
# ================================================================
# CELL 9A — DEVELOPMENT IMAGE-QUALITY CALIBRATION
# Profile 120 development previews for adaptive preprocessing
# ================================================================

from __future__ import annotations

import hashlib
import json
import math
from pathlib import Path
from typing import Any

import cv2
import numpy as np
import pandas as pd
from IPython.display import display


# ================================================================
# 1. KONFIGURASI
# ================================================================

DATA_ROOT = Path(
    "/content/drive/MyDrive/InvoiceFlow-AI-Data"
)

BUILD_ROOT = (
    DATA_ROOT
    / "interim"
    / "synthetic_v1_build"
    / "20260904T150025Z"
)

RELEASE_ROOT = (
    DATA_ROOT
    / "releases"
    / "SYNTHETIC-INVOICE-V1"
    / "1.0.0"
)

RELEASE_INDEX_PATH = (
    RELEASE_ROOT
    / "release_index.jsonl"
)

OCR_BENCHMARK_ROOT = (
    BUILD_ROOT
    / "ocr_benchmark"
)

MANIFEST_ROOT = (
    OCR_BENCHMARK_ROOT
    / "manifests"
)

ROUTING_DECISION_PATH = (
    MANIFEST_ROOT
    / "ocr_routing_decision.json"
)

IMAGE_QUALITY_PROFILE_PATH = (
    OCR_BENCHMARK_ROOT
    / "development_image_quality.csv"
)

PREPROCESSING_CALIBRATION_PATH = (
    MANIFEST_ROOT
    / "preprocessing_calibration.json"
)

EXPECTED_DEVELOPMENT_DOCUMENTS = 120
EXPECTED_DEVELOPMENT_TEMPLATES = 6


# ================================================================
# 2. FILE HELPERS
# ================================================================

def to_builtin(value: Any) -> Any:
    if isinstance(value, np.ndarray):
        return value.tolist()

    if isinstance(value, np.generic):
        return value.item()

    if isinstance(value, dict):
        return {
            str(key): to_builtin(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [
            to_builtin(item)
            for item in value
        ]

    return value


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as file_handle:
        for chunk in iter(
            lambda: file_handle.read(
                1024 * 1024
            ),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def load_json(path: Path) -> dict:
    try:
        with path.open(
            "r",
            encoding="utf-8",
        ) as file_handle:
            value = json.load(file_handle)
    except json.JSONDecodeError as error:
        raise RuntimeError(
            f"JSON tidak valid: {path}"
        ) from error

    if not isinstance(value, dict):
        raise RuntimeError(
            f"Isi JSON bukan object: {path}"
        )

    return value


def load_jsonl(path: Path) -> list[dict]:
    records = []

    with path.open(
        "r",
        encoding="utf-8",
    ) as file_handle:
        for line_number, line in enumerate(
            file_handle,
            start=1,
        ):
            clean_line = line.strip()

            if not clean_line:
                continue

            try:
                record = json.loads(
                    clean_line
                )
            except json.JSONDecodeError as error:
                raise RuntimeError(
                    "JSONL tidak valid pada baris "
                    f"{line_number}: {error}"
                ) from error

            if not isinstance(record, dict):
                raise RuntimeError(
                    "Record JSONL bukan object pada "
                    f"baris {line_number}."
                )

            records.append(record)

    return records


def atomic_write_json(
    path: Path,
    value: dict,
) -> None:
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            to_builtin(value),
            indent=2,
            ensure_ascii=False,
            sort_keys=True,
        )
        + "\n",
        encoding="utf-8",
    )

    temporary_path.replace(path)


def atomic_write_csv(
    path: Path,
    table: pd.DataFrame,
) -> None:
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    table.to_csv(
        temporary_path,
        index=False,
        encoding="utf-8",
    )

    temporary_path.replace(path)


# ================================================================
# 3. IMAGE-QUALITY HELPERS
# ================================================================

def estimate_skew_degrees(
    grayscale: np.ndarray,
) -> float:
    image_height, image_width = (
        grayscale.shape
    )

    edges = cv2.Canny(
        grayscale,
        threshold1=50,
        threshold2=150,
        apertureSize=3,
    )

    minimum_line_length = max(
        40,
        image_width // 8,
    )

    lines = cv2.HoughLinesP(
        edges,
        rho=1,
        theta=np.pi / 180,
        threshold=80,
        minLineLength=minimum_line_length,
        maxLineGap=20,
    )

    if lines is None:
        return 0.0

    horizontal_angles = []

    for line in lines[:, 0]:
        x1, y1, x2, y2 = [
            int(value)
            for value in line
        ]

        delta_x = x2 - x1
        delta_y = y2 - y1

        if delta_x == 0:
            continue

        angle = math.degrees(
            math.atan2(
                delta_y,
                delta_x,
            )
        )

        while angle > 90:
            angle -= 180

        while angle < -90:
            angle += 180

        if abs(angle) <= 10:
            horizontal_angles.append(
                angle
            )

    if not horizontal_angles:
        return 0.0

    return float(
        np.median(
            horizontal_angles
        )
    )


def calculate_entropy(
    grayscale: np.ndarray,
) -> float:
    histogram = np.bincount(
        grayscale.reshape(-1),
        minlength=256,
    ).astype(np.float64)

    probabilities = (
        histogram / histogram.sum()
    )

    probabilities = probabilities[
        probabilities > 0
    ]

    return float(
        -np.sum(
            probabilities
            * np.log2(probabilities)
        )
    )


def calculate_image_metrics(
    image_path: Path,
) -> dict:
    image = cv2.imread(
        str(image_path),
        cv2.IMREAD_COLOR,
    )

    if image is None:
        raise RuntimeError(
            f"OpenCV gagal membaca: {image_path}"
        )

    image_height, image_width = (
        image.shape[:2]
    )

    if image_height <= 0 or image_width <= 0:
        raise RuntimeError(
            f"Dimensi gambar tidak valid: "
            f"{image_path}"
        )

    grayscale = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2GRAY,
    )

    brightness = float(
        np.mean(grayscale)
    )

    contrast = float(
        np.std(grayscale)
    )

    percentile_01 = float(
        np.percentile(
            grayscale,
            1,
        )
    )

    percentile_99 = float(
        np.percentile(
            grayscale,
            99,
        )
    )

    dynamic_range = (
        percentile_99
        - percentile_01
    )

    laplacian_variance = float(
        cv2.Laplacian(
            grayscale,
            cv2.CV_64F,
        ).var()
    )

    edges = cv2.Canny(
        grayscale,
        threshold1=100,
        threshold2=200,
    )

    edge_density = float(
        np.mean(edges > 0)
    )

    median_filtered = cv2.medianBlur(
        grayscale,
        3,
    )

    noise_residual = float(
        np.mean(
            cv2.absdiff(
                grayscale,
                median_filtered,
            )
        )
    )

    white_pixel_ratio = float(
        np.mean(
            grayscale >= 245
        )
    )

    dark_pixel_ratio = float(
        np.mean(
            grayscale <= 30
        )
    )

    skew_degrees = estimate_skew_degrees(
        grayscale
    )

    entropy = calculate_entropy(
        grayscale
    )

    return {
        "image_width_pixels": (
            image_width
        ),
        "image_height_pixels": (
            image_height
        ),
        "brightness_mean": (
            brightness
        ),
        "contrast_std": contrast,
        "percentile_01": (
            percentile_01
        ),
        "percentile_99": (
            percentile_99
        ),
        "dynamic_range": (
            dynamic_range
        ),
        "laplacian_variance": (
            laplacian_variance
        ),
        "edge_density": (
            edge_density
        ),
        "noise_residual": (
            noise_residual
        ),
        "white_pixel_ratio": (
            white_pixel_ratio
        ),
        "dark_pixel_ratio": (
            dark_pixel_ratio
        ),
        "estimated_skew_degrees": (
            skew_degrees
        ),
        "absolute_skew_degrees": abs(
            skew_degrees
        ),
        "entropy": entropy,
    }


# ================================================================
# 4. VALIDASI SUMBER
# ================================================================

required_paths = [
    RELEASE_INDEX_PATH,
    ROUTING_DECISION_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not path.is_file()
]

if missing_paths:
    raise FileNotFoundError(
        "Sumber kalibrasi belum lengkap:\n"
        + "\n".join(missing_paths)
    )

routing_decision = load_json(
    ROUTING_DECISION_PATH
)

if (
    routing_decision.get("status")
    != "FROZEN_FOR_PREPROCESSING"
):
    raise RuntimeError(
        "Routing OCR belum dibekukan untuk "
        "tahap preprocessing."
    )

release_records = load_jsonl(
    RELEASE_INDEX_PATH
)

development_records = [
    record
    for record in release_records
    if record.get("split")
    == "development"
]

if (
    len(development_records)
    != EXPECTED_DEVELOPMENT_DOCUMENTS
):
    raise RuntimeError(
        "Jumlah development record tidak sesuai: "
        f"{len(development_records)}"
    )


# ================================================================
# 5. PROFILE 120 DEVELOPMENT PREVIEWS
# ================================================================

quality_records = []
image_errors = []
checksum_mismatches = []
nondevelopment_opened = 0

print(
    "Memeriksa kualitas 120 development previews..."
)
print()

for position, record in enumerate(
    development_records,
    start=1,
):
    document_id = record["document_id"]

    preview_artifact = (
        record.get("artifacts", {})
        .get("preview", {})
    )

    relative_path = preview_artifact.get(
        "project_relative_path"
    )

    if not relative_path:
        image_errors.append(
            {
                "document_id": document_id,
                "error": (
                    "missing_preview_reference"
                ),
            }
        )
        continue

    preview_path = (
        DATA_ROOT / relative_path
    )

    if not preview_path.is_file():
        image_errors.append(
            {
                "document_id": document_id,
                "error": (
                    "missing_preview_file"
                ),
                "path": str(
                    preview_path
                ),
            }
        )
        continue

    current_checksum = sha256_file(
        preview_path
    )

    expected_checksum = (
        preview_artifact.get("sha256")
    )

    if current_checksum != expected_checksum:
        checksum_mismatches.append(
            {
                "document_id": document_id,
                "expected": (
                    expected_checksum
                ),
                "actual": (
                    current_checksum
                ),
            }
        )
        continue

    try:
        metrics = calculate_image_metrics(
            preview_path
        )
    except Exception as error:
        image_errors.append(
            {
                "document_id": document_id,
                "error": (
                    f"{type(error).__name__}: "
                    f"{str(error)}"
                ),
            }
        )
        continue

    quality_records.append(
        {
            "sequence_number": (
                record.get(
                    "sequence_number"
                )
            ),
            "document_id": document_id,
            "canonical_invoice_id": (
                record[
                    "canonical_invoice_id"
                ]
            ),
            "template_id": (
                record["template_id"]
            ),
            "split": record["split"],
            "language": (
                record["language"]
            ),
            "currency": (
                record["currency"]
            ),
            "item_count": (
                record["item_count"]
            ),
            "preview_path": str(
                preview_path
            ),
            "preview_sha256": (
                current_checksum
            ),
            **metrics,
        }
    )

    if (
        position % 20 == 0
        or position
        == len(development_records)
    ):
        print(
            f"[{position:03d}/120] "
            f"processed={len(quality_records)}, "
            f"errors={len(image_errors)}, "
            f"checksum_mismatch="
            f"{len(checksum_mismatches)}"
        )


# ================================================================
# 6. VALIDASI METRIK
# ================================================================

quality_table = pd.DataFrame(
    quality_records
)

metric_columns = [
    "brightness_mean",
    "contrast_std",
    "dynamic_range",
    "laplacian_variance",
    "edge_density",
    "noise_residual",
    "white_pixel_ratio",
    "dark_pixel_ratio",
    "estimated_skew_degrees",
    "absolute_skew_degrees",
    "entropy",
]

nonfinite_metric_rows = []

for row_index, row in quality_table.iterrows():
    invalid_metrics = [
        column
        for column in metric_columns
        if not np.isfinite(
            float(row[column])
        )
    ]

    if invalid_metrics:
        nonfinite_metric_rows.append(
            {
                "document_id": (
                    row["document_id"]
                ),
                "invalid_metrics": (
                    invalid_metrics
                ),
            }
        )


# ================================================================
# 7. CALIBRATE ADAPTIVE THRESHOLDS
# ================================================================

def quantile(
    column: str,
    probability: float,
) -> float:
    return float(
        quality_table[
            column
        ].quantile(probability)
    )


calibrated_thresholds = {
    "minimum_contrast_std": round(
        max(
            10.0,
            quantile(
                "contrast_std",
                0.01,
            )
            * 0.50,
        ),
        6,
    ),
    "minimum_dynamic_range": round(
        max(
            80.0,
            quantile(
                "dynamic_range",
                0.01,
            )
            * 0.75,
        ),
        6,
    ),
    "minimum_laplacian_variance": round(
        max(
            20.0,
            quantile(
                "laplacian_variance",
                0.01,
            )
            * 0.25,
        ),
        6,
    ),
    "minimum_brightness_mean": round(
        max(
            30.0,
            quantile(
                "brightness_mean",
                0.01,
            )
            - 40.0,
        ),
        6,
    ),
    "maximum_brightness_mean": round(
        min(
            252.0,
            quantile(
                "brightness_mean",
                0.99,
            )
            + 20.0,
        ),
        6,
    ),
    "maximum_noise_residual": round(
        max(
            12.0,
            quantile(
                "noise_residual",
                0.99,
            )
            * 3.0,
        ),
        6,
    ),
    "maximum_absolute_skew_degrees": round(
        max(
            1.5,
            quantile(
                "absolute_skew_degrees",
                0.99,
            )
            + 0.5,
        ),
        6,
    ),
}


# ================================================================
# 8. APPLY CALIBRATION RULES
# ================================================================

def determine_preprocessing_flags(
    row: pd.Series,
) -> list[str]:
    flags = []

    if (
        row["contrast_std"]
        < calibrated_thresholds[
            "minimum_contrast_std"
        ]
    ):
        flags.append("LOW_CONTRAST")

    if (
        row["dynamic_range"]
        < calibrated_thresholds[
            "minimum_dynamic_range"
        ]
    ):
        flags.append(
            "LOW_DYNAMIC_RANGE"
        )

    if (
        row["laplacian_variance"]
        < calibrated_thresholds[
            "minimum_laplacian_variance"
        ]
    ):
        flags.append("BLUR")

    if (
        row["brightness_mean"]
        < calibrated_thresholds[
            "minimum_brightness_mean"
        ]
    ):
        flags.append("TOO_DARK")

    if (
        row["brightness_mean"]
        > calibrated_thresholds[
            "maximum_brightness_mean"
        ]
    ):
        flags.append("TOO_BRIGHT")

    if (
        row["noise_residual"]
        > calibrated_thresholds[
            "maximum_noise_residual"
        ]
    ):
        flags.append("HIGH_NOISE")

    if (
        row["absolute_skew_degrees"]
        > calibrated_thresholds[
            "maximum_absolute_skew_degrees"
        ]
    ):
        flags.append("SKEW")

    return flags


quality_table[
    "preprocessing_flags"
] = quality_table.apply(
    lambda row: "|".join(
        determine_preprocessing_flags(
            row
        )
    ),
    axis=1,
)

quality_table[
    "preprocessing_recommended"
] = (
    quality_table[
        "preprocessing_flags"
    ]
    != ""
)

preprocessing_recommended_count = int(
    quality_table[
        "preprocessing_recommended"
    ].sum()
)


# ================================================================
# 9. SUMMARY TABLES
# ================================================================

summary_aggregations = {
    "documents": (
        "document_id",
        "count",
    ),
    "minimum_brightness": (
        "brightness_mean",
        "min",
    ),
    "mean_brightness": (
        "brightness_mean",
        "mean",
    ),
    "maximum_brightness": (
        "brightness_mean",
        "max",
    ),
    "minimum_contrast": (
        "contrast_std",
        "min",
    ),
    "mean_contrast": (
        "contrast_std",
        "mean",
    ),
    "minimum_blur_score": (
        "laplacian_variance",
        "min",
    ),
    "mean_blur_score": (
        "laplacian_variance",
        "mean",
    ),
    "maximum_absolute_skew": (
        "absolute_skew_degrees",
        "max",
    ),
    "mean_noise_residual": (
        "noise_residual",
        "mean",
    ),
    "preprocessing_recommended": (
        "preprocessing_recommended",
        "sum",
    ),
}

template_summary = (
    quality_table
    .groupby(
        "template_id",
        as_index=False,
    )
    .agg(
        **summary_aggregations
    )
)

for column in template_summary.columns:
    if column not in {
        "template_id",
        "documents",
        "preprocessing_recommended",
    }:
        template_summary[column] = (
            template_summary[
                column
            ].round(6)
        )

overall_summary = {
    "documents": int(
        len(quality_table)
    ),
    "image_widths": sorted(
        int(value)
        for value in quality_table[
            "image_width_pixels"
        ].unique()
    ),
    "image_heights": sorted(
        int(value)
        for value in quality_table[
            "image_height_pixels"
        ].unique()
    ),
    "brightness": {
        "minimum": quantile(
            "brightness_mean",
            0.00,
        ),
        "p01": quantile(
            "brightness_mean",
            0.01,
        ),
        "median": quantile(
            "brightness_mean",
            0.50,
        ),
        "p99": quantile(
            "brightness_mean",
            0.99,
        ),
        "maximum": quantile(
            "brightness_mean",
            1.00,
        ),
    },
    "contrast": {
        "minimum": quantile(
            "contrast_std",
            0.00,
        ),
        "p01": quantile(
            "contrast_std",
            0.01,
        ),
        "median": quantile(
            "contrast_std",
            0.50,
        ),
        "p99": quantile(
            "contrast_std",
            0.99,
        ),
        "maximum": quantile(
            "contrast_std",
            1.00,
        ),
    },
    "laplacian_variance": {
        "minimum": quantile(
            "laplacian_variance",
            0.00,
        ),
        "p01": quantile(
            "laplacian_variance",
            0.01,
        ),
        "median": quantile(
            "laplacian_variance",
            0.50,
        ),
        "p99": quantile(
            "laplacian_variance",
            0.99,
        ),
        "maximum": quantile(
            "laplacian_variance",
            1.00,
        ),
    },
    "absolute_skew_degrees": {
        "minimum": quantile(
            "absolute_skew_degrees",
            0.00,
        ),
        "median": quantile(
            "absolute_skew_degrees",
            0.50,
        ),
        "p99": quantile(
            "absolute_skew_degrees",
            0.99,
        ),
        "maximum": quantile(
            "absolute_skew_degrees",
            1.00,
        ),
    },
    "noise_residual": {
        "minimum": quantile(
            "noise_residual",
            0.00,
        ),
        "median": quantile(
            "noise_residual",
            0.50,
        ),
        "p99": quantile(
            "noise_residual",
            0.99,
        ),
        "maximum": quantile(
            "noise_residual",
            1.00,
        ),
    },
}


# ================================================================
# 10. FINAL CONTROLS
# ================================================================

controls = [
    {
        "control": "routing_status",
        "expected": (
            "FROZEN_FOR_PREPROCESSING"
        ),
        "actual": routing_decision.get(
            "status"
        ),
    },
    {
        "control": "release_records",
        "expected": 200,
        "actual": len(
            release_records
        ),
    },
    {
        "control": "development_records",
        "expected": 120,
        "actual": len(
            development_records
        ),
    },
    {
        "control": "images_profiled",
        "expected": 120,
        "actual": len(
            quality_table
        ),
    },
    {
        "control": "development_templates",
        "expected": 6,
        "actual": (
            quality_table[
                "template_id"
            ].nunique()
        ),
    },
    {
        "control": "image_errors",
        "expected": 0,
        "actual": len(
            image_errors
        ),
    },
    {
        "control": "checksum_mismatches",
        "expected": 0,
        "actual": len(
            checksum_mismatches
        ),
    },
    {
        "control": "nonfinite_metric_rows",
        "expected": 0,
        "actual": len(
            nonfinite_metric_rows
        ),
    },
    {
        "control": "nondevelopment_opened",
        "expected": 0,
        "actual": (
            nondevelopment_opened
        ),
    },
]

for control in controls:
    control["status"] = (
        "VALID"
        if control["expected"]
        == control["actual"]
        else "INVALID"
    )

control_table = pd.DataFrame(
    controls
)

invalid_controls = [
    control["control"]
    for control in controls
    if control["status"] != "VALID"
]


# ================================================================
# 11. DISPLAY SEBELUM MENULIS
# ================================================================

display(control_table)

print("\nIMAGE QUALITY BY TEMPLATE")
display(template_summary)

print("\nCALIBRATED THRESHOLDS")
display(
    pd.DataFrame(
        [
            {
                "metric": key,
                "threshold": value,
            }
            for key, value
            in calibrated_thresholds.items()
        ]
    )
)

if preprocessing_recommended_count:
    print(
        "\nCLEAN PREVIEWS FLAGGED FOR REVIEW"
    )

    display(
        quality_table[
            quality_table[
                "preprocessing_recommended"
            ]
        ][
            [
                "document_id",
                "template_id",
                "preprocessing_flags",
                "brightness_mean",
                "contrast_std",
                "laplacian_variance",
                "noise_residual",
                "absolute_skew_degrees",
            ]
        ]
    )

if invalid_controls:
    print("\nIMAGE ERRORS")
    print(
        json.dumps(
            image_errors,
            indent=2,
            ensure_ascii=False,
        )
    )

    print("\nCHECKSUM MISMATCHES")
    print(
        json.dumps(
            checksum_mismatches,
            indent=2,
            ensure_ascii=False,
        )
    )

    raise RuntimeError(
        "IMAGE-QUALITY CALIBRATION FAILED. "
        f"Kontrol tidak valid: "
        f"{invalid_controls}"
    )


# ================================================================
# 12. WRITE CALIBRATION ARTIFACTS
# ================================================================

atomic_write_csv(
    IMAGE_QUALITY_PROFILE_PATH,
    quality_table,
)

profile_sha256 = sha256_file(
    IMAGE_QUALITY_PROFILE_PATH
)

calibration_manifest = {
    "schema_version": "1.0.0",
    "calibration_id": (
        "INVOICEFLOW-ADAPTIVE-PREPROCESSING-V1"
    ),
    "status": "CALIBRATED",
    "scope": "development",
    "source_type": (
        "clean synthetic invoice previews"
    ),
    "document_count": int(
        len(quality_table)
    ),
    "template_count": int(
        quality_table[
            "template_id"
        ].nunique()
    ),
    "metrics": [
        "brightness_mean",
        "contrast_std",
        "dynamic_range",
        "laplacian_variance",
        "edge_density",
        "noise_residual",
        "white_pixel_ratio",
        "dark_pixel_ratio",
        "estimated_skew_degrees",
        "entropy",
    ],
    "thresholds": (
        calibrated_thresholds
    ),
    "adaptive_rules": {
        "LOW_CONTRAST": (
            "apply CLAHE contrast enhancement"
        ),
        "LOW_DYNAMIC_RANGE": (
            "apply percentile-based intensity "
            "normalization"
        ),
        "BLUR": (
            "apply conservative unsharp masking"
        ),
        "TOO_DARK": (
            "apply intensity normalization"
        ),
        "TOO_BRIGHT": (
            "apply intensity normalization"
        ),
        "HIGH_NOISE": (
            "apply median denoising"
        ),
        "SKEW": (
            "apply deskewing before OCR"
        ),
        "NO_FLAGS": (
            "preserve original image"
        ),
    },
    "preprocessing_recommended_count": (
        preprocessing_recommended_count
    ),
    "overall_summary": (
        overall_summary
    ),
    "template_summary": (
        template_summary.to_dict(
            orient="records"
        )
    ),
    "input_checksums": {
        "release_index": (
            sha256_file(
                RELEASE_INDEX_PATH
            )
        ),
        "routing_decision": (
            sha256_file(
                ROUTING_DECISION_PATH
            )
        ),
    },
    "output_checksums": {
        "development_image_quality": (
            profile_sha256
        ),
    },
    "policy": {
        "clean_image_default": (
            "NO_PREPROCESSING"
        ),
        "preprocessing_mode": (
            "ADAPTIVE_ONLY"
        ),
        "validation_access": False,
        "test_access": False,
        "source_dataset_modified": False,
    },
}

atomic_write_json(
    PREPROCESSING_CALIBRATION_PATH,
    calibration_manifest,
)

calibration_sha256 = sha256_file(
    PREPROCESSING_CALIBRATION_PATH
)


# ================================================================
# 13. FINAL OUTPUT
# ================================================================

print()
print(
    f"Development previews : "
    f"{len(quality_table)}"
)
print(
    f"Templates            : "
    f"{quality_table['template_id'].nunique()}"
)
print(
    f"Image errors         : "
    f"{len(image_errors)}"
)
print(
    f"Checksum mismatches  : "
    f"{len(checksum_mismatches)}"
)
print(
    f"Preprocess suggested : "
    f"{preprocessing_recommended_count}"
)
print(
    f"Quality profile      : "
    f"{IMAGE_QUALITY_PROFILE_PATH}"
)
print(
    f"Profile SHA-256      : "
    f"{profile_sha256}"
)
print(
    f"Calibration manifest : "
    f"{PREPROCESSING_CALIBRATION_PATH}"
)
print(
    f"Calibration SHA-256  : "
    f"{calibration_sha256}"
)
print(
    "Preprocessing mode   : ADAPTIVE_ONLY"
)
print(
    "Validation opened    : 0"
)
print(
    "Test opened          : 0"
)
print(
    "Source image writes  : 0"
)
print(
    "Dataset modifications: 0"
)
print()
print(
    "✅ CELL 9A PASSED — kualitas 120 development "
    "preview berhasil diprofilkan dan threshold "
    "preprocessing adaptif berhasil dikalibrasi."
)

Memeriksa kualitas 120 development previews...

[020/120] processed=20, errors=0, checksum_mismatch=0
[040/120] processed=40, errors=0, checksum_mismatch=0
[060/120] processed=60, errors=0, checksum_mismatch=0
[080/120] processed=80, errors=0, checksum_mismatch=0
[100/120] processed=100, errors=0, checksum_mismatch=0
[120/120] processed=120, errors=0, checksum_mismatch=0


,control,expected,actual,status
0,routing_status,FROZEN_FOR_PREPROCESSING,FROZEN_FOR_PREPROCESSING,VALID
1,release_records,200,200,VALID
2,development_records,120,120,VALID
3,images_profiled,120,120,VALID
4,development_templates,6,6,VALID
5,image_errors,0,0,VALID
6,checksum_mismatches,0,0,VALID
7,nonfinite_metric_rows,0,0,VALID
8,nondevelopment_opened,0,0,VALID



IMAGE QUALITY BY TEMPLATE


,template_id,documents,minimum_brightness,mean_brightness,maximum_brightness,minimum_contrast,mean_contrast,minimum_blur_score,mean_blur_score,maximum_absolute_skew,mean_noise_residual,preprocessing_recommended
0,TPL-01,20,240.516850,241.196630,242.179431,42.825526,43.440479,1101.553534,1386.289405,0.0,1.195086,0
1,TPL-02,20,245.746057,246.076902,246.556863,30.060212,31.070193,972.997605,1264.314354,0.0,1.007507,0
2,TPL-03,20,241.678871,242.291366,242.882216,39.272311,39.732849,893.759483,1100.610692,0.0,0.922795,0
3,TPL-04,20,245.875666,246.330165,246.634174,25.121179,25.856214,865.941327,1048.007751,0.0,0.990123,0
4,TPL-05,20,249.162062,249.619999,250.081256,20.861596,22.172521,834.226886,1087.460590,0.0,0.978494,0
5,TPL-06,20,239.415909,240.387726,241.255036,44.173540,44.716178,1190.550466,1442.060316,0.0,1.139311,0



CALIBRATED THRESHOLDS


,metric,threshold
0,minimum_contrast_std,10.543185
1,minimum_dynamic_range,81.750000
2,minimum_laplacian_variance,213.532959
3,minimum_brightness_mean,199.512981
4,maximum_brightness_mean,252.000000
5,maximum_noise_residual,12.000000
6,maximum_absolute_skew_degrees,1.500000



Development previews : 120
Templates            : 6
Image errors         : 0
Checksum mismatches  : 0
Preprocess suggested : 0
Quality profile      : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/development_image_quality.csv
Profile SHA-256      : 1500191dbb0e776cfc0d7ec52692a554683491142adb513d20e662ef673124af
Calibration manifest : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/manifests/preprocessing_calibration.json
Calibration SHA-256  : 1f9e4eaf2da7083f2a8138e97d6f88a6f1f6ce2bea753cce1b7e7aad0f5a07e1
Preprocessing mode   : ADAPTIVE_ONLY
Validation opened    : 0
Test opened          : 0
Source image writes  : 0
Dataset modifications: 0

✅ CELL 9A PASSED — kualitas 120 development preview berhasil diprofilkan dan threshold preprocessing adaptif berhasil dikalibrasi.


**Cell 9B — Stress test preprocessing adaptif**

In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import os
from pathlib import Path
from typing import Any

import cv2
import numpy as np
import pandas as pd
from IPython.display import display


# ============================================================
# CELL 9B FINAL V2 — PREPROCESSING STRESS TEST
# Menggantikan seluruh Cell 9B sebelumnya.
# ============================================================

BUILD_ROOT = Path(
    "/content/drive/MyDrive/InvoiceFlow-AI-Data/interim/"
    "synthetic_v1_build/20260904T150025Z"
)

RENDERED_ROOT = BUILD_ROOT / "rendered_dataset"
BENCHMARK_ROOT = BUILD_ROOT / "ocr_benchmark"

CALIBRATION_PATH = (
    BENCHMARK_ROOT
    / "manifests"
    / "preprocessing_calibration.json"
)

RUN_ID = "final_v2"

RUN_ROOT = (
    BENCHMARK_ROOT
    / "preprocessing_stress_test"
    / RUN_ID
)

DEGRADED_ROOT = RUN_ROOT / "degraded"
PROCESSED_ROOT = RUN_ROOT / "processed"

RESULT_PATH = (
    RUN_ROOT
    / "preprocessing_stress_results.csv"
)

MANIFEST_PATH = (
    BENCHMARK_ROOT
    / "manifests"
    / "preprocessing_stress_manifest.json"
)


DOCUMENTS = [
    (
        "INV-SYN-000002",
        "TPL-01",
        "id",
    ),
    (
        "INV-SYN-000036",
        "TPL-02",
        "en",
    ),
    (
        "INV-SYN-000043",
        "TPL-03",
        "id",
    ),
    (
        "INV-SYN-000071",
        "TPL-04",
        "en",
    ),
    (
        "INV-SYN-000082",
        "TPL-05",
        "id",
    ),
    (
        "INV-SYN-000114",
        "TPL-06",
        "en",
    ),
]


SCENARIOS = (
    "low_contrast",
    "blur",
    "dark",
    "noise",
    "skew",
    "combined_scan",
)


REQUIRED_THRESHOLDS = {
    "minimum_contrast_std",
    "minimum_dynamic_range",
    "minimum_laplacian_variance",
    "minimum_brightness_mean",
    "maximum_brightness_mean",
    "maximum_noise_residual",
    "maximum_absolute_skew_degrees",
}


# ============================================================
# 1. FILE HELPERS
# ============================================================

def sha256_file(
    file_path: Path,
) -> str:
    digest = hashlib.sha256()

    with file_path.open(
        "rb"
    ) as file_handle:
        for chunk in iter(
            lambda: file_handle.read(
                1024 * 1024
            ),
            b"",
        ):
            digest.update(
                chunk
            )

    return digest.hexdigest()


def atomic_write_json(
    file_path: Path,
    payload: dict,
) -> None:
    file_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = (
        file_path.with_name(
            file_path.name
            + ".tmp"
        )
    )

    temporary_path.write_text(
        json.dumps(
            payload,
            indent=2,
            ensure_ascii=False,
        )
        + "\n",
        encoding="utf-8",
    )

    os.replace(
        temporary_path,
        file_path,
    )


def atomic_write_csv(
    file_path: Path,
    table: pd.DataFrame,
) -> None:
    file_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = (
        file_path.with_name(
            file_path.name
            + ".tmp"
        )
    )

    table.to_csv(
        temporary_path,
        index=False,
    )

    os.replace(
        temporary_path,
        file_path,
    )


def atomic_write_png(
    file_path: Path,
    image: np.ndarray,
) -> None:
    file_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = (
        file_path.with_name(
            file_path.stem
            + ".tmp.png"
        )
    )

    write_success = cv2.imwrite(
        str(
            temporary_path
        ),
        image,
        [
            cv2.IMWRITE_PNG_COMPRESSION,
            3,
        ],
    )

    if not write_success:
        raise RuntimeError(
            "Gagal menulis PNG: "
            f"{temporary_path}"
        )

    os.replace(
        temporary_path,
        file_path,
    )


def ensure_grayscale(
    image: np.ndarray,
) -> np.ndarray:
    if image is None:
        raise ValueError(
            "Image kosong."
        )

    if image.ndim == 2:
        grayscale = image

    elif (
        image.ndim == 3
        and image.shape[2] == 4
    ):
        grayscale = cv2.cvtColor(
            image,
            cv2.COLOR_BGRA2GRAY,
        )

    elif (
        image.ndim == 3
        and image.shape[2] == 3
    ):
        grayscale = cv2.cvtColor(
            image,
            cv2.COLOR_BGR2GRAY,
        )

    else:
        raise ValueError(
            "Shape image tidak didukung: "
            f"{image.shape}"
        )

    return np.ascontiguousarray(
        grayscale.astype(
            np.uint8
        )
    )


# ============================================================
# 2. LOAD THRESHOLD CELL 9A
# ============================================================

def find_thresholds(
    value: Any,
) -> dict[str, float] | None:
    if isinstance(
        value,
        dict,
    ):
        if REQUIRED_THRESHOLDS.issubset(
            value
        ):
            return {
                key: float(
                    value[key]
                )
                for key
                in REQUIRED_THRESHOLDS
            }

        for child in value.values():
            result = find_thresholds(
                child
            )

            if result is not None:
                return result

    elif isinstance(
        value,
        list,
    ):
        table_values = {}

        for child in value:
            if not isinstance(
                child,
                dict,
            ):
                continue

            metric_name = (
                child.get(
                    "metric"
                )
                or child.get(
                    "name"
                )
            )

            threshold_value = child.get(
                "threshold",
                child.get(
                    "value"
                ),
            )

            if (
                metric_name
                in REQUIRED_THRESHOLDS
                and threshold_value
                is not None
            ):
                table_values[
                    metric_name
                ] = float(
                    threshold_value
                )

        if REQUIRED_THRESHOLDS.issubset(
            table_values
        ):
            return {
                key: table_values[
                    key
                ]
                for key
                in REQUIRED_THRESHOLDS
            }

        for child in value:
            result = find_thresholds(
                child
            )

            if result is not None:
                return result

    return None


# ============================================================
# 3. PREVIEW DISCOVERY
# ============================================================

def locate_preview(
    document_id: str,
    template_id: str,
) -> Path:
    excluded_fragments = (
        "contact_sheet",
        "thumbnail",
        "/qa/",
        "/degraded/",
        "/processed/",
        "/preprocessing_stress_test/",
    )

    candidates = []

    for file_path in RENDERED_ROOT.rglob(
        f"*{document_id}*.png"
    ):
        path_text = (
            file_path
            .as_posix()
            .lower()
        )

        if (
            file_path.is_file()
            and template_id.lower()
            in path_text
            and not any(
                fragment
                in path_text
                for fragment
                in excluded_fragments
            )
        ):
            candidates.append(
                file_path
            )

    def calculate_score(
        file_path: Path,
    ) -> int:
        path_text = (
            file_path
            .as_posix()
            .lower()
        )

        filename = (
            file_path.name.lower()
        )

        return (
            100
            * int(
                "/previews/"
                in path_text
            )
            + 40
            * int(
                "preview"
                in filename
            )
            + 20
            * int(
                document_id.lower()
                in filename
            )
            + 20
            * int(
                template_id.lower()
                in filename
            )
        )

    unique_candidates = {
        str(
            file_path.resolve()
        ): file_path
        for file_path
        in candidates
    }

    candidates = sorted(
        unique_candidates.values(),
        key=lambda file_path: (
            -calculate_score(
                file_path
            ),
            len(
                file_path.as_posix()
            ),
            file_path.as_posix(),
        ),
    )

    if not candidates:
        raise RuntimeError(
            "Preview tidak ditemukan untuk "
            f"{document_id}/{template_id}."
        )

    highest_score = calculate_score(
        candidates[0]
    )

    highest_candidates = [
        file_path
        for file_path
        in candidates
        if calculate_score(
            file_path
        )
        == highest_score
    ]

    if len(
        highest_candidates
    ) != 1:
        raise RuntimeError(
            "Preview ambigu untuk "
            f"{document_id}/{template_id}:\n"
            + "\n".join(
                str(file_path)
                for file_path
                in highest_candidates
            )
        )

    return highest_candidates[0]


# ============================================================
# 4. IMAGE METRICS
# ============================================================

def calculate_noise_residual(
    grayscale: np.ndarray,
) -> float:
    median_image = cv2.medianBlur(
        grayscale,
        3,
    )

    residual = (
        grayscale.astype(
            np.float32
        )
        - median_image.astype(
            np.float32
        )
    )

    gradient_x = cv2.Sobel(
        median_image,
        cv2.CV_32F,
        1,
        0,
        ksize=3,
    )

    gradient_y = cv2.Sobel(
        median_image,
        cv2.CV_32F,
        0,
        1,
        ksize=3,
    )

    flat_mask = (
        cv2.magnitude(
            gradient_x,
            gradient_y,
        )
        < 20.0
    )

    residual_sample = (
        residual[
            flat_mask
        ]
    )

    minimum_sample_size = max(
        1000,
        int(
            grayscale.size
            * 0.05
        ),
    )

    if (
        residual_sample.size
        < minimum_sample_size
    ):
        residual_sample = (
            residual.reshape(
                -1
            )
        )

    return float(
        np.mean(
            np.abs(
                residual_sample
            )
        )
    )


def estimate_skew(
    grayscale: np.ndarray,
) -> float:
    reduced_image = grayscale

    maximum_dimension = max(
        grayscale.shape
    )

    if maximum_dimension > 1800:
        resize_scale = (
            1800.0
            / maximum_dimension
        )

        reduced_image = cv2.resize(
            grayscale,
            None,
            fx=resize_scale,
            fy=resize_scale,
            interpolation=cv2.INTER_AREA,
        )

    reduced_image = cv2.medianBlur(
        reduced_image,
        3,
    )

    binary_image = cv2.threshold(
        reduced_image,
        0,
        255,
        (
            cv2.THRESH_BINARY_INV
            + cv2.THRESH_OTSU
        ),
    )[1]

    horizontal_kernel = (
        cv2.getStructuringElement(
            cv2.MORPH_RECT,
            (
                max(
                    25,
                    reduced_image.shape[1]
                    // 35,
                ),
                1,
            ),
        )
    )

    horizontal_image = (
        cv2.morphologyEx(
            binary_image,
            cv2.MORPH_OPEN,
            horizontal_kernel,
        )
    )

    detected_lines = (
        cv2.HoughLinesP(
            horizontal_image,
            1,
            np.pi / 1800.0,
            threshold=max(
                40,
                reduced_image.shape[1]
                // 12,
            ),
            minLineLength=max(
                80,
                reduced_image.shape[1]
                // 7,
            ),
            maxLineGap=max(
                10,
                reduced_image.shape[1]
                // 100,
            ),
        )
    )

    if detected_lines is None:
        return 0.0

    angles = []
    weights = []

    for line in detected_lines[
        :,
        0,
        :
    ]:
        x1, y1, x2, y2 = [
            int(value)
            for value
            in line
        ]

        delta_x = x2 - x1
        delta_y = y2 - y1

        if delta_x == 0:
            continue

        angle = math.degrees(
            math.atan2(
                delta_y,
                delta_x,
            )
        )

        if abs(angle) <= 10.0:
            angles.append(
                angle
            )

            weights.append(
                math.hypot(
                    delta_x,
                    delta_y,
                )
            )

    if len(angles) < 2:
        return 0.0

    sort_order = np.argsort(
        angles
    )

    sorted_angles = (
        np.asarray(
            angles
        )[sort_order]
    )

    sorted_weights = (
        np.asarray(
            weights
        )[sort_order]
    )

    cumulative_weights = (
        np.cumsum(
            sorted_weights
        )
    )

    middle_weight = (
        cumulative_weights[-1]
        / 2.0
    )

    middle_index = np.searchsorted(
        cumulative_weights,
        middle_weight,
    )

    return float(
        sorted_angles[
            middle_index
        ]
    )


def calculate_image_metrics(
    image: np.ndarray,
) -> dict[str, float]:
    grayscale = ensure_grayscale(
        image
    )

    (
        percentile_1,
        percentile_99,
    ) = np.percentile(
        grayscale,
        [
            1,
            99,
        ],
    )

    return {
        "brightness_mean": float(
            np.mean(
                grayscale
            )
        ),
        "contrast_std": float(
            np.std(
                grayscale
            )
        ),
        "dynamic_range": float(
            percentile_99
            - percentile_1
        ),
        "laplacian_variance": float(
            cv2.Laplacian(
                grayscale,
                cv2.CV_64F,
            ).var()
        ),
        "noise_residual": (
            calculate_noise_residual(
                grayscale
            )
        ),
        "skew_degrees": (
            estimate_skew(
                grayscale
            )
        ),
    }


def detect_quality_flags(
    metric_values: dict[str, float],
) -> list[str]:
    flags = []

    if (
        metric_values[
            "contrast_std"
        ]
        < THRESHOLDS[
            "minimum_contrast_std"
        ]
    ):
        flags.append(
            "LOW_CONTRAST"
        )

    if (
        metric_values[
            "dynamic_range"
        ]
        < THRESHOLDS[
            "minimum_dynamic_range"
        ]
    ):
        flags.append(
            "LOW_DYNAMIC_RANGE"
        )

    if (
        metric_values[
            "laplacian_variance"
        ]
        < THRESHOLDS[
            "minimum_laplacian_variance"
        ]
    ):
        flags.append(
            "BLUR"
        )

    if (
        metric_values[
            "brightness_mean"
        ]
        < THRESHOLDS[
            "minimum_brightness_mean"
        ]
    ):
        flags.append(
            "TOO_DARK"
        )

    if (
        metric_values[
            "brightness_mean"
        ]
        > THRESHOLDS[
            "maximum_brightness_mean"
        ]
    ):
        flags.append(
            "TOO_BRIGHT"
        )

    if (
        metric_values[
            "noise_residual"
        ]
        > THRESHOLDS[
            "maximum_noise_residual"
        ]
    ):
        flags.append(
            "NOISE"
        )

    if (
        abs(
            metric_values[
                "skew_degrees"
            ]
        )
        > THRESHOLDS[
            "maximum_absolute_skew_degrees"
        ]
    ):
        flags.append(
            "SKEW"
        )

    return flags


# ============================================================
# 5. IMAGE TRANSFORMATIONS
# ============================================================

def rotate_image(
    image: np.ndarray,
    degrees: float,
    border_value: int = 255,
) -> np.ndarray:
    height, width = image.shape[:2]

    rotation_matrix = (
        cv2.getRotationMatrix2D(
            (
                width / 2.0,
                height / 2.0,
            ),
            degrees,
            1.0,
        )
    )

    return cv2.warpAffine(
        image,
        rotation_matrix,
        (
            width,
            height,
        ),
        flags=cv2.INTER_CUBIC,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=border_value,
    )


def add_gaussian_noise(
    image: np.ndarray,
    sigma: float,
    seed: int,
) -> np.ndarray:
    random_generator = (
        np.random.default_rng(
            seed
        )
    )

    noise = random_generator.normal(
        0.0,
        sigma,
        image.shape,
    )

    return np.clip(
        image.astype(
            np.float32
        )
        + noise,
        0,
        255,
    ).astype(
        np.uint8
    )


# ============================================================
# 6. CONTROLLED DEGRADATION
# ============================================================

def create_degradation(
    image: np.ndarray,
    scenario: str,
    seed: int,
) -> np.ndarray:
    grayscale = ensure_grayscale(
        image
    )

    if scenario == "low_contrast":
        degraded = (
            225.0
            + 0.18
            * (
                grayscale.astype(
                    np.float32
                )
                - 225.0
            )
        )

        return np.clip(
            degraded,
            0,
            255,
        ).astype(
            np.uint8
        )

    if scenario == "blur":
        return cv2.GaussianBlur(
            grayscale,
            (
                11,
                11,
            ),
            2.8,
        )

    if scenario == "dark":
        return np.clip(
            grayscale.astype(
                np.float32
            )
            * 0.48,
            0,
            255,
        ).astype(
            np.uint8
        )

    if scenario == "noise":
        # Kurangi highlight agar noise tidak hilang
        # akibat clipping pada latar putih 255.
        noise_base = np.minimum(
            grayscale,
            245,
        ).astype(
            np.uint8
        )

        # Noise wajib benar-benar melewati threshold Cell 9A.
        target_noise_residual = (
            THRESHOLDS[
                "maximum_noise_residual"
            ]
            * 1.25
        )

        candidate_sigmas = (
            42.0,
            48.0,
            54.0,
            60.0,
            66.0,
        )

        for (
            attempt_number,
            sigma,
        ) in enumerate(
            candidate_sigmas
        ):
            candidate_image = (
                add_gaussian_noise(
                    noise_base,
                    sigma=sigma,
                    seed=(
                        seed
                        + attempt_number
                        * 1000
                    ),
                )
            )

            candidate_residual = (
                calculate_noise_residual(
                    candidate_image
                )
            )

            if (
                candidate_residual
                >= target_noise_residual
            ):
                return candidate_image

        raise RuntimeError(
            "Controlled noise gagal "
            "melewati target residual "
            f"{target_noise_residual:.4f}."
        )

    if scenario == "skew":
        return rotate_image(
            grayscale,
            degrees=3.5,
        )

    if scenario == "combined_scan":
        degraded = rotate_image(
            grayscale,
            degrees=4.0,
            border_value=242,
        )

        degraded = cv2.GaussianBlur(
            degraded,
            (
                7,
                7,
            ),
            1.8,
        )

        degraded = np.clip(
            185.0
            + 0.38
            * (
                degraded.astype(
                    np.float32
                )
                - 185.0
            ),
            0,
            255,
        ).astype(
            np.uint8
        )

        return add_gaussian_noise(
            degraded,
            sigma=22.0,
            seed=seed + 10000,
        )

    raise ValueError(
        "Scenario tidak dikenal: "
        f"{scenario}"
    )


# ============================================================
# 7. ADAPTIVE PREPROCESSING
# ============================================================

def select_best_deskew(
    image: np.ndarray,
    detected_angle: float,
) -> np.ndarray:
    candidates = (
        image,
        rotate_image(
            image,
            detected_angle,
        ),
        rotate_image(
            image,
            -detected_angle,
        ),
    )

    return min(
        candidates,
        key=lambda candidate: abs(
            estimate_skew(
                candidate
            )
        ),
    )


def adaptive_preprocess(
    image: np.ndarray,
) -> tuple[np.ndarray, dict]:
    working_image = (
        ensure_grayscale(
            image
        ).copy()
    )

    before_metrics = (
        calculate_image_metrics(
            working_image
        )
    )

    detected_flags = (
        detect_quality_flags(
            before_metrics
        )
    )

    applied_steps = []

    if "SKEW" in detected_flags:
        working_image = (
            select_best_deskew(
                working_image,
                before_metrics[
                    "skew_degrees"
                ],
            )
        )

        applied_steps.append(
            "DESKEW"
        )

    if "NOISE" in detected_flags:
        target_noise_residual = (
            before_metrics[
                "noise_residual"
            ]
            * 0.85
        )

        base_strength = float(
            np.clip(
                before_metrics[
                    "noise_residual"
                ]
                * 1.70,
                20.0,
                38.0,
            )
        )

        candidate_strengths = (
            base_strength,
            min(
                45.0,
                base_strength
                * 1.20,
            ),
            45.0,
        )

        best_image = (
            working_image.copy()
        )

        best_residual = (
            calculate_noise_residual(
                best_image
            )
        )

        for denoise_strength in candidate_strengths:
            candidate_image = (
                cv2.fastNlMeansDenoising(
                    working_image,
                    None,
                    h=float(
                        denoise_strength
                    ),
                    templateWindowSize=7,
                    searchWindowSize=21,
                )
            )

            candidate_residual = (
                calculate_noise_residual(
                    candidate_image
                )
            )

            if (
                candidate_residual
                < best_residual
            ):
                best_image = (
                    candidate_image
                )

                best_residual = (
                    candidate_residual
                )

            if (
                candidate_residual
                <= target_noise_residual
            ):
                best_image = (
                    candidate_image
                )

                break

        working_image = best_image

        applied_steps.append(
            "DENOISE"
        )

    if (
        "TOO_DARK"
        in detected_flags
        or "TOO_BRIGHT"
        in detected_flags
    ):
        (
            low_value,
            high_value,
        ) = np.percentile(
            working_image,
            [
                1,
                99,
            ],
        )

        if high_value > low_value:
            working_image = np.clip(
                (
                    working_image.astype(
                        np.float32
                    )
                    - low_value
                )
                * 230.0
                / (
                    high_value
                    - low_value
                )
                + 15.0,
                0,
                255,
            ).astype(
                np.uint8
            )

        applied_steps.append(
            "INTENSITY_NORMALIZATION"
        )

    if (
        "LOW_CONTRAST"
        in detected_flags
        or "LOW_DYNAMIC_RANGE"
        in detected_flags
    ):
        clahe = cv2.createCLAHE(
            clipLimit=2.0,
            tileGridSize=(
                8,
                8,
            ),
        )

        working_image = (
            clahe.apply(
                working_image
            )
        )

        applied_steps.append(
            "CLAHE"
        )

    intermediate_metrics = (
        calculate_image_metrics(
            working_image
        )
    )

    if (
        "BLUR"
        in detected_flags
        or (
            intermediate_metrics[
                "laplacian_variance"
            ]
            < THRESHOLDS[
                "minimum_laplacian_variance"
            ]
        )
    ):
        softened_image = cv2.GaussianBlur(
            working_image,
            (
                0,
                0,
            ),
            1.2,
        )

        working_image = cv2.addWeighted(
            working_image,
            1.8,
            softened_image,
            -0.8,
            0,
        )

        applied_steps.append(
            "UNSHARP_MASK"
        )

    after_metrics = (
        calculate_image_metrics(
            working_image
        )
    )

    return (
        working_image,
        {
            "detected_flags": (
                detected_flags
            ),
            "applied_steps": (
                applied_steps
            ),
            "before_metrics": (
                before_metrics
            ),
            "after_metrics": (
                after_metrics
            ),
        },
    )


# ============================================================
# 8. VALIDATION RULES
# ============================================================

EXPECTED_FLAGS = {
    "low_contrast": {
        "LOW_CONTRAST",
        "LOW_DYNAMIC_RANGE",
    },
    "blur": {
        "BLUR",
    },
    "dark": {
        "TOO_DARK",
    },
    "noise": {
        "NOISE",
    },
    "skew": {
        "SKEW",
    },
}


def evaluate_stress_case(
    scenario: str,
    decision: dict,
) -> tuple[
    bool,
    bool,
    dict[str, bool],
]:
    before_metrics = (
        decision[
            "before_metrics"
        ]
    )

    after_metrics = (
        decision[
            "after_metrics"
        ]
    )

    detected_flags = set(
        decision[
            "detected_flags"
        ]
    )

    if scenario == "combined_scan":
        flag_detection_valid = {
            "SKEW",
            "NOISE",
        }.issubset(
            detected_flags
        )

    else:
        flag_detection_valid = bool(
            detected_flags
            & EXPECTED_FLAGS[
                scenario
            ]
        )

    brightness_target = (
        THRESHOLDS[
            "minimum_brightness_mean"
        ]
        + THRESHOLDS[
            "maximum_brightness_mean"
        ]
    ) / 2.0

    improvement_checks = {
        "contrast": (
            after_metrics[
                "contrast_std"
            ]
            > before_metrics[
                "contrast_std"
            ]
            * 1.05
        ),
        "dynamic_range": (
            after_metrics[
                "dynamic_range"
            ]
            > before_metrics[
                "dynamic_range"
            ]
            + 5.0
        ),
        "sharpness": (
            after_metrics[
                "laplacian_variance"
            ]
            > before_metrics[
                "laplacian_variance"
            ]
            * 1.05
        ),
        "brightness": (
            abs(
                after_metrics[
                    "brightness_mean"
                ]
                - brightness_target
            )
            < abs(
                before_metrics[
                    "brightness_mean"
                ]
                - brightness_target
            )
        ),
        "noise": (
            after_metrics[
                "noise_residual"
            ]
            < before_metrics[
                "noise_residual"
            ]
            * 0.90
        ),
        "skew": (
            abs(
                after_metrics[
                    "skew_degrees"
                ]
            )
            < abs(
                before_metrics[
                    "skew_degrees"
                ]
            )
            - 0.25
        ),
    }

    if scenario == "low_contrast":
        target_metric_improved = (
            improvement_checks[
                "contrast"
            ]
            or improvement_checks[
                "dynamic_range"
            ]
        )

    elif scenario == "blur":
        target_metric_improved = (
            improvement_checks[
                "sharpness"
            ]
        )

    elif scenario == "dark":
        target_metric_improved = (
            improvement_checks[
                "brightness"
            ]
        )

    elif scenario == "noise":
        target_metric_improved = (
            improvement_checks[
                "noise"
            ]
        )

    elif scenario == "skew":
        target_metric_improved = (
            improvement_checks[
                "skew"
            ]
        )

    elif scenario == "combined_scan":
        target_metric_improved = (
            improvement_checks[
                "skew"
            ]
            and improvement_checks[
                "noise"
            ]
            and sum(
                improvement_checks.values()
            )
            >= 3
        )

    else:
        raise ValueError(
            "Scenario tidak dikenal: "
            f"{scenario}"
        )

    return (
        bool(
            flag_detection_valid
        ),
        bool(
            target_metric_improved
        ),
        improvement_checks,
    )


# ============================================================
# 9. PREFLIGHT
# ============================================================

required_paths = [
    BUILD_ROOT,
    RENDERED_ROOT,
    BENCHMARK_ROOT,
    CALIBRATION_PATH,
]

missing_paths = [
    str(file_path)
    for file_path
    in required_paths
    if not file_path.exists()
]

if missing_paths:
    raise FileNotFoundError(
        "Prerequisite Cell 9B belum lengkap:\n"
        + "\n".join(
            missing_paths
        )
    )


with CALIBRATION_PATH.open(
    "r",
    encoding="utf-8",
) as calibration_file:
    calibration_manifest = (
        json.load(
            calibration_file
        )
    )


THRESHOLDS = find_thresholds(
    calibration_manifest
)

if THRESHOLDS is None:
    raise RuntimeError(
        "Tujuh threshold tidak ditemukan. "
        "Jalankan ulang Cell 9A."
    )


source_records = []

for (
    document_id,
    template_id,
    language,
) in DOCUMENTS:
    source_path = locate_preview(
        document_id,
        template_id,
    )

    source_records.append(
        {
            "document_id": (
                document_id
            ),
            "template_id": (
                template_id
            ),
            "language": (
                language
            ),
            "source_path": (
                source_path
            ),
            "source_sha256": (
                sha256_file(
                    source_path
                )
            ),
        }
    )


source_table = pd.DataFrame(
    [
        {
            "document_id": (
                record[
                    "document_id"
                ]
            ),
            "template_id": (
                record[
                    "template_id"
                ]
            ),
            "language": (
                record[
                    "language"
                ]
            ),
            "source_path": str(
                record[
                    "source_path"
                ]
            ),
        }
        for record
        in source_records
    ]
)

print(
    "SOURCE PREVIEWS"
)

display(
    source_table
)


# ============================================================
# 10. CLEAN-IMAGE INVARIANCE
# ============================================================

clean_records = []

for source_record in source_records:
    clean_image = cv2.imread(
        str(
            source_record[
                "source_path"
            ]
        ),
        cv2.IMREAD_GRAYSCALE,
    )

    if clean_image is None:
        raise RuntimeError(
            "Gagal membuka clean preview: "
            f"{source_record['source_path']}"
        )

    (
        processed_clean,
        clean_decision,
    ) = adaptive_preprocess(
        clean_image
    )

    pixel_identical = bool(
        np.array_equal(
            clean_image,
            processed_clean,
        )
    )

    clean_valid = (
        pixel_identical
        and not clean_decision[
            "detected_flags"
        ]
        and not clean_decision[
            "applied_steps"
        ]
    )

    clean_records.append(
        {
            "document_id": (
                source_record[
                    "document_id"
                ]
            ),
            "template_id": (
                source_record[
                    "template_id"
                ]
            ),
            "language": (
                source_record[
                    "language"
                ]
            ),
            "detected_flags": "|".join(
                clean_decision[
                    "detected_flags"
                ]
            ),
            "applied_steps": "|".join(
                clean_decision[
                    "applied_steps"
                ]
            ),
            "pixel_identical": (
                pixel_identical
            ),
            "status": (
                "VALID"
                if clean_valid
                else "INVALID"
            ),
        }
    )


# ============================================================
# 11. RUN 36 STRESS CASES
# ============================================================

DEGRADED_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

PROCESSED_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

print(
    "\nMenjalankan Cell 9B Final V2...\n"
)

stress_records = []
stress_errors = []
sequence_number = 0

for (
    document_number,
    source_record,
) in enumerate(
    source_records,
    start=1,
):
    clean_image = cv2.imread(
        str(
            source_record[
                "source_path"
            ]
        ),
        cv2.IMREAD_GRAYSCALE,
    )

    if clean_image is None:
        raise RuntimeError(
            "Gagal membuka clean preview: "
            f"{source_record['source_path']}"
        )

    for (
        scenario_number,
        scenario,
    ) in enumerate(
        SCENARIOS,
        start=1,
    ):
        sequence_number += 1

        random_seed = (
            20260905
            + document_number
            * 100
            + scenario_number
        )

        degraded_path = (
            DEGRADED_ROOT
            / source_record[
                "template_id"
            ]
            / (
                f"{source_record['document_id']}"
                f"_{scenario}.png"
            )
        )

        processed_path = (
            PROCESSED_ROOT
            / source_record[
                "template_id"
            ]
            / (
                f"{source_record['document_id']}"
                f"_{scenario}.png"
            )
        )

        try:
            degraded_image = (
                create_degradation(
                    clean_image,
                    scenario,
                    random_seed,
                )
            )

            (
                processed_image,
                decision,
            ) = adaptive_preprocess(
                degraded_image
            )

            (
                flag_detection_valid,
                target_metric_improved,
                improvement_checks,
            ) = evaluate_stress_case(
                scenario,
                decision,
            )

            case_status = (
                "VALID"
                if (
                    flag_detection_valid
                    and target_metric_improved
                )
                else "INVALID"
            )

            atomic_write_png(
                degraded_path,
                degraded_image,
            )

            atomic_write_png(
                processed_path,
                processed_image,
            )

            stress_record = {
                "sequence_number": (
                    sequence_number
                ),
                "document_id": (
                    source_record[
                        "document_id"
                    ]
                ),
                "template_id": (
                    source_record[
                        "template_id"
                    ]
                ),
                "language": (
                    source_record[
                        "language"
                    ]
                ),
                "scenario": (
                    scenario
                ),
                "source_path": str(
                    source_record[
                        "source_path"
                    ]
                ),
                "degraded_path": str(
                    degraded_path
                ),
                "processed_path": str(
                    processed_path
                ),
                "detected_flags": "|".join(
                    decision[
                        "detected_flags"
                    ]
                ),
                "applied_steps": "|".join(
                    decision[
                        "applied_steps"
                    ]
                ),
                "flag_detection_valid": (
                    flag_detection_valid
                ),
                "target_metric_improved": (
                    target_metric_improved
                ),
                "improvement_count": int(
                    sum(
                        improvement_checks.values()
                    )
                ),
                "degraded_sha256": (
                    sha256_file(
                        degraded_path
                    )
                ),
                "processed_sha256": (
                    sha256_file(
                        processed_path
                    )
                ),
                "status": (
                    case_status
                ),
            }

            for (
                metric_name,
                metric_value,
            ) in decision[
                "before_metrics"
            ].items():
                stress_record[
                    f"before_{metric_name}"
                ] = metric_value

            for (
                metric_name,
                metric_value,
            ) in decision[
                "after_metrics"
            ].items():
                stress_record[
                    f"after_{metric_name}"
                ] = metric_value

            for (
                check_name,
                check_value,
            ) in improvement_checks.items():
                stress_record[
                    f"improved_{check_name}"
                ] = bool(
                    check_value
                )

            stress_records.append(
                stress_record
            )

            print(
                f"[{sequence_number:02d}/36] "
                f"{source_record['document_id']} | "
                f"{scenario} | "
                f"flags="
                f"{decision['detected_flags']} | "
                f"status={case_status}"
            )

        except Exception as error:
            stress_errors.append(
                {
                    "sequence_number": (
                        sequence_number
                    ),
                    "document_id": (
                        source_record[
                            "document_id"
                        ]
                    ),
                    "template_id": (
                        source_record[
                            "template_id"
                        ]
                    ),
                    "scenario": (
                        scenario
                    ),
                    "error_type": (
                        type(
                            error
                        ).__name__
                    ),
                    "error": str(
                        error
                    )[:500],
                }
            )

            print(
                f"[{sequence_number:02d}/36] "
                f"{source_record['document_id']} | "
                f"{scenario} | "
                f"ERROR: "
                f"{type(error).__name__}"
            )


# ============================================================
# 12. INTEGRITY AND SUMMARY
# ============================================================

source_checksum_changes = []

for source_record in source_records:
    current_checksum = sha256_file(
        source_record[
            "source_path"
        ]
    )

    if (
        current_checksum
        != source_record[
            "source_sha256"
        ]
    ):
        source_checksum_changes.append(
            {
                "document_id": (
                    source_record[
                        "document_id"
                    ]
                ),
                "before": (
                    source_record[
                        "source_sha256"
                    ]
                ),
                "after": (
                    current_checksum
                ),
            }
        )


stress_table = pd.DataFrame(
    stress_records
)

clean_table = pd.DataFrame(
    clean_records
)

if stress_table.empty:
    raise RuntimeError(
        "Tidak ada stress-test record "
        "yang berhasil dibuat."
    )


scenario_table = (
    stress_table.groupby(
        "scenario",
        as_index=False,
    )
    .agg(
        documents=(
            "document_id",
            "count",
        ),
        valid_flag_detection=(
            "flag_detection_valid",
            "sum",
        ),
        target_metric_improved=(
            "target_metric_improved",
            "sum",
        ),
        valid_cases=(
            "status",
            lambda values: int(
                (
                    values
                    == "VALID"
                ).sum()
            ),
        ),
    )
    .sort_values(
        "scenario"
    )
    .reset_index(
        drop=True
    )
)


scenario_table[
    "status"
] = np.where(
    (
        scenario_table[
            "valid_cases"
        ]
        == scenario_table[
            "documents"
        ]
    ),
    "VALID",
    "INVALID",
)


valid_stress_cases = int(
    (
        stress_table[
            "status"
        ]
        == "VALID"
    ).sum()
)


unique_degraded_paths = int(
    stress_table[
        "degraded_path"
    ].nunique()
)


unique_processed_paths = int(
    stress_table[
        "processed_path"
    ].nunique()
)


current_degraded_files = sum(
    Path(
        file_path
    ).is_file()
    for file_path
    in stress_table[
        "degraded_path"
    ]
)


current_processed_files = sum(
    Path(
        file_path
    ).is_file()
    for file_path
    in stress_table[
        "processed_path"
    ]
)


# ============================================================
# 13. FINAL CONTROLS
# ============================================================

control_values = [
    (
        "calibration_thresholds",
        7,
        len(
            THRESHOLDS
        ),
    ),
    (
        "representative_documents",
        6,
        len(
            source_records
        ),
    ),
    (
        "template_coverage",
        6,
        len(
            {
                record[
                    "template_id"
                ]
                for record
                in source_records
            }
        ),
    ),
    (
        "language_coverage",
        [
            "en",
            "id",
        ],
        sorted(
            {
                record[
                    "language"
                ]
                for record
                in source_records
            }
        ),
    ),
    (
        "clean_images_preserved",
        6,
        int(
            (
                clean_table[
                    "status"
                ]
                == "VALID"
            ).sum()
        ),
    ),
    (
        "stress_cases",
        36,
        len(
            stress_table
        ),
    ),
    (
        "valid_stress_cases",
        36,
        valid_stress_cases,
    ),
    (
        "unique_degraded_paths",
        36,
        unique_degraded_paths,
    ),
    (
        "unique_processed_paths",
        36,
        unique_processed_paths,
    ),
    (
        "current_run_degraded_files",
        36,
        current_degraded_files,
    ),
    (
        "current_run_processed_files",
        36,
        current_processed_files,
    ),
    (
        "stress_errors",
        0,
        len(
            stress_errors
        ),
    ),
    (
        "source_checksum_changes",
        0,
        len(
            source_checksum_changes
        ),
    ),
    (
        "validation_opened",
        0,
        0,
    ),
    (
        "test_opened",
        0,
        0,
    ),
]


control_records = [
    {
        "control": (
            control_name
        ),
        "expected": (
            expected_value
        ),
        "actual": (
            actual_value
        ),
        "status": (
            "VALID"
            if expected_value
            == actual_value
            else "INVALID"
        ),
    }
    for (
        control_name,
        expected_value,
        actual_value,
    )
    in control_values
]


control_table = pd.DataFrame(
    control_records
)


# ============================================================
# 14. CHECKPOINT OUTPUT
# ============================================================

atomic_write_csv(
    RESULT_PATH,
    stress_table,
)


final_status = (
    "PASSED"
    if all(
        record[
            "status"
        ]
        == "VALID"
        for record
        in control_records
    )
    else "FAILED"
)


manifest_payload = {
    "schema_version": (
        "1.3.0"
    ),
    "stage": (
        "PREPROCESSING_STRESS_TEST"
    ),
    "run_id": (
        RUN_ID
    ),
    "status": (
        final_status
    ),
    "preprocessing_mode": (
        "ADAPTIVE_ONLY"
    ),
    "run_root": str(
        RUN_ROOT
    ),
    "calibration_path": str(
        CALIBRATION_PATH
    ),
    "calibration_sha256": (
        sha256_file(
            CALIBRATION_PATH
        )
    ),
    "thresholds": (
        THRESHOLDS
    ),
    "documents": [
        {
            "document_id": (
                record[
                    "document_id"
                ]
            ),
            "template_id": (
                record[
                    "template_id"
                ]
            ),
            "language": (
                record[
                    "language"
                ]
            ),
            "source_path": str(
                record[
                    "source_path"
                ]
            ),
            "source_sha256": (
                record[
                    "source_sha256"
                ]
            ),
        }
        for record
        in source_records
    ],
    "scenarios": list(
        SCENARIOS
    ),
    "stress_case_count": len(
        stress_table
    ),
    "valid_stress_case_count": (
        valid_stress_cases
    ),
    "unique_degraded_paths": (
        unique_degraded_paths
    ),
    "unique_processed_paths": (
        unique_processed_paths
    ),
    "current_run_degraded_files": (
        current_degraded_files
    ),
    "current_run_processed_files": (
        current_processed_files
    ),
    "result_path": str(
        RESULT_PATH
    ),
    "result_sha256": (
        sha256_file(
            RESULT_PATH
        )
    ),
    "stress_errors": (
        stress_errors
    ),
    "source_checksum_changes": (
        source_checksum_changes
    ),
    "controls": (
        control_records
    ),
    "dataset_modifications": 0,
    "validation_opened": 0,
    "test_opened": 0,
}


atomic_write_json(
    MANIFEST_PATH,
    manifest_payload,
)


# ============================================================
# 15. REPORT AND HARD GATE
# ============================================================

display(
    control_table
)

print(
    "\nCLEAN-IMAGE INVARIANCE"
)

display(
    clean_table
)

print(
    "\nSTRESS TEST BY SCENARIO"
)

display(
    scenario_table
)


invalid_cases = stress_table[
    stress_table[
        "status"
    ]
    != "VALID"
]


if not invalid_cases.empty:
    print(
        "\nINVALID STRESS CASES"
    )

    display(
        invalid_cases[
            [
                "document_id",
                "template_id",
                "language",
                "scenario",
                "detected_flags",
                "applied_steps",
                "flag_detection_valid",
                "target_metric_improved",
                "improvement_count",
                "before_noise_residual",
                "after_noise_residual",
                "before_skew_degrees",
                "after_skew_degrees",
                "status",
            ]
        ]
    )


print()
print(
    f"Run ID               : "
    f"{RUN_ID}"
)
print(
    f"Run root             : "
    f"{RUN_ROOT}"
)
print(
    f"Result table         : "
    f"{RESULT_PATH}"
)
print(
    f"Stress manifest      : "
    f"{MANIFEST_PATH}"
)
print(
    "Manifest SHA-256     : "
    f"{sha256_file(MANIFEST_PATH)}"
)
print(
    "Valid stress cases   : "
    f"{valid_stress_cases}/36"
)
print(
    "Source changes       : "
    f"{len(source_checksum_changes)}"
)
print(
    "Dataset modifications: 0"
)
print(
    "Validation opened    : 0"
)
print(
    "Test opened          : 0"
)


invalid_controls = [
    record[
        "control"
    ]
    for record
    in control_records
    if record[
        "status"
    ]
    != "VALID"
]


if invalid_controls:
    raise RuntimeError(
        "CELL 9B FINAL V2 FAILED. "
        "Kontrol tidak valid: "
        f"{invalid_controls}"
    )


print()
print(
    "✅ CELL 9B FINAL V2 PASSED — "
    "36/36 gangguan terkendali berhasil "
    "dideteksi dan diperbaiki."
)
print(
    "Clean preview tetap identik, "
    "checksum sumber tidak berubah, "
    "dan artifact final_v2 terisolasi."
)
print(
    "Lanjutkan ke Cell 9C untuk "
    "membandingkan PaddleOCR sebelum "
    "dan sesudah preprocessing."
)

SOURCE PREVIEWS


,document_id,template_id,language,source_path
0,INV-SYN-000002,TPL-01,id,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...
1,INV-SYN-000036,TPL-02,en,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...
2,INV-SYN-000043,TPL-03,id,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...
3,INV-SYN-000071,TPL-04,en,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...
4,INV-SYN-000082,TPL-05,id,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...
5,INV-SYN-000114,TPL-06,en,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...



Menjalankan Cell 9B Final V2...

[01/36] INV-SYN-000002 | low_contrast | flags=['LOW_CONTRAST', 'LOW_DYNAMIC_RANGE', 'BLUR'] | status=VALID
[02/36] INV-SYN-000002 | blur | flags=['BLUR'] | status=VALID
[03/36] INV-SYN-000002 | dark | flags=['TOO_DARK'] | status=VALID
[04/36] INV-SYN-000002 | noise | ERROR: RuntimeError
[05/36] INV-SYN-000002 | skew | flags=['SKEW'] | status=VALID
[06/36] INV-SYN-000002 | combined_scan | flags=['NOISE', 'SKEW'] | status=VALID
[07/36] INV-SYN-000036 | low_contrast | flags=['LOW_CONTRAST', 'LOW_DYNAMIC_RANGE', 'BLUR'] | status=VALID
[08/36] INV-SYN-000036 | blur | flags=['BLUR'] | status=VALID
[09/36] INV-SYN-000036 | dark | flags=['LOW_DYNAMIC_RANGE', 'TOO_DARK'] | status=VALID
[10/36] INV-SYN-000036 | noise | ERROR: RuntimeError
[11/36] INV-SYN-000036 | skew | flags=['SKEW'] | status=VALID
[12/36] INV-SYN-000036 | combined_scan | flags=['NOISE', 'SKEW'] | status=VALID
[13/36] INV-SYN-000043 | low_contrast | flags=['LOW_CONTRAST', 'LOW_DYNAMIC_RANGE', '

,control,expected,actual,status
0,calibration_thresholds,7,7,VALID
1,representative_documents,6,6,VALID
2,template_coverage,6,6,VALID
3,language_coverage,"[en, id]","[en, id]",VALID
4,clean_images_preserved,6,6,VALID
5,stress_cases,36,30,INVALID
6,valid_stress_cases,36,30,INVALID
7,unique_degraded_paths,36,30,INVALID
8,unique_processed_paths,36,30,INVALID
9,current_run_degraded_files,36,30,INVALID



CLEAN-IMAGE INVARIANCE


,document_id,template_id,language,detected_flags,applied_steps,pixel_identical,status
0,INV-SYN-000002,TPL-01,id,,,True,VALID
1,INV-SYN-000036,TPL-02,en,,,True,VALID
2,INV-SYN-000043,TPL-03,id,,,True,VALID
3,INV-SYN-000071,TPL-04,en,,,True,VALID
4,INV-SYN-000082,TPL-05,id,,,True,VALID
5,INV-SYN-000114,TPL-06,en,,,True,VALID



STRESS TEST BY SCENARIO


,scenario,documents,valid_flag_detection,target_metric_improved,valid_cases,status
0,blur,6,6,6,6,VALID
1,combined_scan,6,6,6,6,VALID
2,dark,6,6,6,6,VALID
3,low_contrast,6,6,6,6,VALID
4,skew,6,6,6,6,VALID



Run ID               : final_v2
Run root             : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/preprocessing_stress_test/final_v2
Result table         : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/preprocessing_stress_test/final_v2/preprocessing_stress_results.csv
Stress manifest      : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/manifests/preprocessing_stress_manifest.json
Manifest SHA-256     : 3f93434acbf1a2b6f048a52580768a0423302af6e6ab701e71646b37420cb081
Valid stress cases   : 30/36
Source changes       : 0
Dataset modifications: 0
Validation opened    : 0
Test opened          : 0


RuntimeError: CELL 9B FINAL V2 FAILED. Kontrol tidak valid: ['stress_cases', 'valid_stress_cases', 'unique_degraded_paths', 'unique_processed_paths', 'current_run_degraded_files', 'current_run_processed_files', 'stress_errors']

**Cell 9B.1 — Final Recovery V3**

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from IPython.display import display


# ============================================================
# CELL 9B.1 — FINAL RECOVERY V3
#
# Tujuan:
# - menggunakan kembali 30 stress case V2 yang sudah VALID;
# - membuat ulang 6 kasus noise dengan impulse noise;
# - tidak mengubah preview/dataset asli;
# - menyimpan hasil terisolasi di final_v3;
# - memperbarui manifest utama hanya setelah 36/36 VALID.
# ============================================================


# ============================================================
# 1. KONFIGURASI
# ============================================================

BUILD_ROOT = Path(
    "/content/drive/MyDrive/InvoiceFlow-AI-Data/interim/"
    "synthetic_v1_build/20260904T150025Z"
)

BENCHMARK_ROOT = BUILD_ROOT / "ocr_benchmark"

CALIBRATION_PATH = (
    BENCHMARK_ROOT
    / "manifests"
    / "preprocessing_calibration.json"
)

V2_ROOT = (
    BENCHMARK_ROOT
    / "preprocessing_stress_test"
    / "final_v2"
)

V2_RESULT_PATH = (
    V2_ROOT
    / "preprocessing_stress_results.csv"
)

RUN_ID = "final_v3"

V3_ROOT = (
    BENCHMARK_ROOT
    / "preprocessing_stress_test"
    / RUN_ID
)

DEGRADED_ROOT = V3_ROOT / "degraded"
PROCESSED_ROOT = V3_ROOT / "processed"

V3_RESULT_PATH = (
    V3_ROOT
    / "preprocessing_stress_results.csv"
)

V3_MANIFEST_PATH = (
    V3_ROOT
    / "preprocessing_stress_manifest.json"
)

CANONICAL_MANIFEST_PATH = (
    BENCHMARK_ROOT
    / "manifests"
    / "preprocessing_stress_manifest.json"
)

EXPECTED_DOCUMENTS = 6
EXPECTED_SCENARIOS = 6
EXPECTED_CASES = 36

SCENARIO_NAMES = {
    "low_contrast",
    "blur",
    "dark",
    "noise",
    "skew",
    "combined_scan",
}

DEGRADED_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

PROCESSED_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 2. HELPER FILE
# ============================================================

def sha256_file(file_path: Path) -> str:
    digest = hashlib.sha256()

    with file_path.open("rb") as file_handle:
        for chunk in iter(
            lambda: file_handle.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def atomic_write_json(
    file_path: Path,
    content: dict,
) -> None:
    file_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = file_path.with_name(
        file_path.name + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            content,
            indent=2,
            ensure_ascii=False,
        )
        + "\n",
        encoding="utf-8",
    )

    os.replace(
        temporary_path,
        file_path,
    )


def atomic_write_csv(
    file_path: Path,
    table: pd.DataFrame,
) -> None:
    file_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = file_path.with_name(
        file_path.name + ".tmp"
    )

    table.to_csv(
        temporary_path,
        index=False,
    )

    os.replace(
        temporary_path,
        file_path,
    )


def atomic_write_png(
    file_path: Path,
    image: np.ndarray,
) -> None:
    file_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = file_path.with_name(
        file_path.stem + ".tmp.png"
    )

    write_valid = cv2.imwrite(
        str(temporary_path),
        image,
    )

    if not write_valid:
        raise RuntimeError(
            f"Gagal menulis PNG: {temporary_path}"
        )

    os.replace(
        temporary_path,
        file_path,
    )


def read_grayscale(
    file_path: Path,
) -> np.ndarray:
    image = cv2.imread(
        str(file_path),
        cv2.IMREAD_GRAYSCALE,
    )

    if image is None:
        raise RuntimeError(
            f"Gagal membuka gambar: {file_path}"
        )

    return image


def copy_png_verified(
    source_path: Path,
    destination_path: Path,
) -> None:
    image = read_grayscale(source_path)

    atomic_write_png(
        destination_path,
        image,
    )

    copied_image = read_grayscale(
        destination_path
    )

    if not np.array_equal(
        image,
        copied_image,
    ):
        raise RuntimeError(
            "Salinan PNG tidak pixel-identical: "
            f"{destination_path}"
        )


# ============================================================
# 3. HELPER KONFIGURASI
# ============================================================

THRESHOLD_KEYS = {
    "minimum_contrast_std",
    "minimum_dynamic_range",
    "minimum_laplacian_variance",
    "minimum_brightness_mean",
    "maximum_brightness_mean",
    "maximum_noise_residual",
    "maximum_absolute_skew_degrees",
}


def locate_threshold_mapping(
    value,
) -> dict | None:
    if isinstance(value, dict):
        if THRESHOLD_KEYS.issubset(
            value.keys()
        ):
            return {
                key: float(value[key])
                for key in THRESHOLD_KEYS
            }

        for child_value in value.values():
            result = locate_threshold_mapping(
                child_value
            )

            if result is not None:
                return result

    elif isinstance(value, list):
        for child_value in value:
            result = locate_threshold_mapping(
                child_value
            )

            if result is not None:
                return result

    return None


# ============================================================
# 4. METRIK NOISE
# ============================================================

def calculate_noise_residual(
    grayscale: np.ndarray,
) -> float:
    median_image = cv2.medianBlur(
        grayscale,
        3,
    )

    residual = (
        grayscale.astype(np.float32)
        - median_image.astype(np.float32)
    )

    gradient_x = cv2.Sobel(
        median_image,
        cv2.CV_32F,
        1,
        0,
        ksize=3,
    )

    gradient_y = cv2.Sobel(
        median_image,
        cv2.CV_32F,
        0,
        1,
        ksize=3,
    )

    gradient_magnitude = cv2.magnitude(
        gradient_x,
        gradient_y,
    )

    flat_mask = gradient_magnitude < 20.0
    residual_sample = residual[flat_mask]

    minimum_sample_size = max(
        1000,
        int(grayscale.size * 0.05),
    )

    if residual_sample.size < minimum_sample_size:
        residual_sample = residual.reshape(-1)

    return float(
        np.mean(
            np.abs(residual_sample)
        )
    )


def calculate_image_metrics(
    grayscale: np.ndarray,
) -> dict:
    percentile_1, percentile_99 = np.percentile(
        grayscale,
        [1, 99],
    )

    return {
        "brightness_mean": float(
            grayscale.mean()
        ),
        "contrast_std": float(
            grayscale.std()
        ),
        "dynamic_range": float(
            percentile_99 - percentile_1
        ),
        "laplacian_variance": float(
            cv2.Laplacian(
                grayscale,
                cv2.CV_64F,
            ).var()
        ),
        "noise_residual": float(
            calculate_noise_residual(
                grayscale
            )
        ),
        "skew_degrees": 0.0,
    }


# ============================================================
# 5. IMPULSE-NOISE GENERATOR
# ============================================================

def create_detectable_impulse_noise(
    clean_image: np.ndarray,
    threshold: float,
    seed: int,
) -> tuple[np.ndarray, float, float]:
    target_residual = threshold * 1.10

    candidate_densities = (
        0.06,
        0.08,
        0.10,
        0.12,
        0.15,
        0.18,
    )

    best_image = None
    best_residual = -1.0
    best_density = None

    for candidate_number, density in enumerate(
        candidate_densities,
        start=1,
    ):
        rng = np.random.default_rng(
            seed + candidate_number * 1000
        )

        candidate = clean_image.copy()

        impulse_mask = (
            rng.random(clean_image.shape)
            < density
        )

        impulse_values = np.where(
            rng.random(clean_image.shape) < 0.5,
            0,
            255,
        ).astype(np.uint8)

        candidate[impulse_mask] = (
            impulse_values[impulse_mask]
        )

        candidate_residual = (
            calculate_noise_residual(
                candidate
            )
        )

        if candidate_residual > best_residual:
            best_image = candidate
            best_residual = candidate_residual
            best_density = density

        if candidate_residual >= target_residual:
            return (
                candidate,
                float(candidate_residual),
                float(density),
            )

    raise RuntimeError(
        "Impulse noise belum melewati target. "
        f"Target={target_residual:.6f}, "
        f"terbaik={best_residual:.6f}, "
        f"density={best_density}"
    )


def remove_impulse_noise(
    degraded_image: np.ndarray,
    before_residual: float,
) -> tuple[np.ndarray, float, str]:
    candidates = [
        (
            "MEDIAN_DENOISE_3",
            cv2.medianBlur(
                degraded_image,
                3,
            ),
        ),
        (
            "MEDIAN_DENOISE_5",
            cv2.medianBlur(
                degraded_image,
                5,
            ),
        ),
    ]

    target_residual = before_residual * 0.90

    best_name = ""
    best_image = None
    best_residual = float("inf")

    for step_name, candidate in candidates:
        candidate_residual = (
            calculate_noise_residual(
                candidate
            )
        )

        if candidate_residual < best_residual:
            best_name = step_name
            best_image = candidate
            best_residual = candidate_residual

        if candidate_residual <= target_residual:
            return (
                candidate,
                float(candidate_residual),
                step_name,
            )

    if best_image is None:
        raise RuntimeError(
            "Tidak ada kandidat denoising."
        )

    return (
        best_image,
        float(best_residual),
        best_name,
    )


# ============================================================
# 6. PREFLIGHT
# ============================================================

required_paths = [
    BUILD_ROOT,
    BENCHMARK_ROOT,
    CALIBRATION_PATH,
    V2_RESULT_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not path.exists()
]

if missing_paths:
    raise FileNotFoundError(
        "File checkpoint belum lengkap:\n"
        + "\n".join(missing_paths)
    )

calibration_manifest = json.loads(
    CALIBRATION_PATH.read_text(
        encoding="utf-8"
    )
)

thresholds = locate_threshold_mapping(
    calibration_manifest
)

if thresholds is None:
    raise RuntimeError(
        "Tujuh threshold kalibrasi "
        "tidak ditemukan."
    )

maximum_noise_residual = thresholds[
    "maximum_noise_residual"
]

v2_table = pd.read_csv(
    V2_RESULT_PATH
)

required_columns = {
    "sequence_number",
    "document_id",
    "template_id",
    "language",
    "scenario",
    "source_path",
    "degraded_path",
    "processed_path",
    "status",
}

missing_columns = sorted(
    required_columns - set(v2_table.columns)
)

if missing_columns:
    raise RuntimeError(
        "Kolom hasil V2 tidak lengkap: "
        f"{missing_columns}"
    )

valid_v2_table = v2_table[
    (v2_table["status"] == "VALID")
    & (v2_table["scenario"] != "noise")
].copy()

if len(valid_v2_table) != 30:
    raise RuntimeError(
        "Hasil V2 yang dapat dipulihkan "
        f"seharusnya 30, ditemukan "
        f"{len(valid_v2_table)}."
    )

document_table = (
    valid_v2_table[
        [
            "document_id",
            "template_id",
            "language",
            "source_path",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "template_id",
            "document_id",
        ]
    )
    .reset_index(drop=True)
)

if len(document_table) != EXPECTED_DOCUMENTS:
    raise RuntimeError(
        "Dokumen representatif seharusnya 6, "
        f"ditemukan {len(document_table)}."
    )

source_checksums_before = {
    row.document_id: sha256_file(
        Path(row.source_path)
    )
    for row in document_table.itertuples(
        index=False
    )
}


# ============================================================
# 7. SALIN 30 HASIL VALID KE FINAL_V3
# ============================================================

recovered_records = []

for record in valid_v2_table.to_dict(
    orient="records"
):
    old_degraded_path = Path(
        record["degraded_path"]
    )

    old_processed_path = Path(
        record["processed_path"]
    )

    new_degraded_path = (
        DEGRADED_ROOT
        / record["template_id"]
        / (
            f"{record['document_id']}_"
            f"{record['scenario']}.png"
        )
    )

    new_processed_path = (
        PROCESSED_ROOT
        / record["template_id"]
        / (
            f"{record['document_id']}_"
            f"{record['scenario']}.png"
        )
    )

    copy_png_verified(
        old_degraded_path,
        new_degraded_path,
    )

    copy_png_verified(
        old_processed_path,
        new_processed_path,
    )

    record["degraded_path"] = str(
        new_degraded_path
    )

    record["processed_path"] = str(
        new_processed_path
    )

    record["degraded_sha256"] = sha256_file(
        new_degraded_path
    )

    record["processed_sha256"] = sha256_file(
        new_processed_path
    )

    record["recovery_action"] = (
        "RECOVERED_FROM_FINAL_V2"
    )

    recovered_records.append(record)


# ============================================================
# 8. BUAT 6 KASUS NOISE
# ============================================================

noise_records = []
noise_errors = []

metric_names = (
    "brightness_mean",
    "contrast_std",
    "dynamic_range",
    "laplacian_variance",
    "noise_residual",
    "skew_degrees",
)

print(
    "Menyelesaikan 6 kasus noise...\n"
)

for document_index, row in enumerate(
    document_table.itertuples(index=False),
    start=1,
):
    try:
        source_path = Path(row.source_path)
        clean_image = read_grayscale(
            source_path
        )

        random_seed = (
            20260905
            + document_index * 100
            + 4
        )

        (
            degraded_image,
            generated_residual,
            selected_density,
        ) = create_detectable_impulse_noise(
            clean_image,
            maximum_noise_residual,
            random_seed,
        )

        before_metrics = calculate_image_metrics(
            degraded_image
        )

        (
            processed_image,
            processed_residual,
            denoise_step,
        ) = remove_impulse_noise(
            degraded_image,
            before_metrics["noise_residual"],
        )

        after_metrics = calculate_image_metrics(
            processed_image
        )

        noise_detected = bool(
            before_metrics["noise_residual"]
            > maximum_noise_residual
        )

        noise_improved = bool(
            after_metrics["noise_residual"]
            < before_metrics["noise_residual"]
            * 0.90
        )

        case_valid = (
            noise_detected
            and noise_improved
        )

        degraded_path = (
            DEGRADED_ROOT
            / row.template_id
            / f"{row.document_id}_noise.png"
        )

        processed_path = (
            PROCESSED_ROOT
            / row.template_id
            / f"{row.document_id}_noise.png"
        )

        atomic_write_png(
            degraded_path,
            degraded_image,
        )

        atomic_write_png(
            processed_path,
            processed_image,
        )

        record = {
            column: np.nan
            for column in v2_table.columns
        }

        record.update(
            {
                "sequence_number": (
                    (document_index - 1) * 6 + 4
                ),
                "document_id": row.document_id,
                "template_id": row.template_id,
                "language": row.language,
                "scenario": "noise",
                "source_path": str(source_path),
                "degraded_path": str(
                    degraded_path
                ),
                "processed_path": str(
                    processed_path
                ),
                "detected_flags": (
                    "NOISE"
                    if noise_detected
                    else ""
                ),
                "applied_steps": denoise_step,
                "flag_detection_valid": (
                    noise_detected
                ),
                "target_metric_improved": (
                    noise_improved
                ),
                "improvement_count": int(
                    noise_improved
                ),
                "degraded_sha256": sha256_file(
                    degraded_path
                ),
                "processed_sha256": sha256_file(
                    processed_path
                ),
                "status": (
                    "VALID"
                    if case_valid
                    else "INVALID"
                ),
                "noise_density": (
                    selected_density
                ),
                "generated_noise_residual": (
                    generated_residual
                ),
                "recovery_action": (
                    "REGENERATED_IMPULSE_NOISE"
                ),
            }
        )

        for metric_name in metric_names:
            record[
                f"before_{metric_name}"
            ] = before_metrics[metric_name]

            record[
                f"after_{metric_name}"
            ] = after_metrics[metric_name]

        noise_records.append(record)

        print(
            f"[{document_index}/6] "
            f"{row.document_id} | "
            f"before="
            f"{before_metrics['noise_residual']:.4f} | "
            f"after="
            f"{after_metrics['noise_residual']:.4f} | "
            f"density={selected_density:.2f} | "
            f"status="
            f"{'VALID' if case_valid else 'INVALID'}"
        )

    except Exception as error:
        noise_errors.append(
            {
                "document_id": row.document_id,
                "template_id": row.template_id,
                "error_type": (
                    type(error).__name__
                ),
                "error": str(error)[:500],
            }
        )

        print(
            f"[{document_index}/6] "
            f"{row.document_id} | "
            f"ERROR: {type(error).__name__}: "
            f"{error}"
        )


# ============================================================
# 9. SATUKAN 36 HASIL
# ============================================================

final_table = pd.DataFrame(
    recovered_records + noise_records
)

if not final_table.empty:
    final_table = (
        final_table
        .sort_values("sequence_number")
        .reset_index(drop=True)
    )

atomic_write_csv(
    V3_RESULT_PATH,
    final_table,
)


# ============================================================
# 10. VALIDASI AKHIR
# ============================================================

source_checksum_changes = []

for row in document_table.itertuples(
    index=False
):
    checksum_after = sha256_file(
        Path(row.source_path)
    )

    checksum_before = (
        source_checksums_before[
            row.document_id
        ]
    )

    if checksum_after != checksum_before:
        source_checksum_changes.append(
            {
                "document_id": row.document_id,
                "before": checksum_before,
                "after": checksum_after,
            }
        )

artifact_checksum_mismatches = []

for record in final_table.to_dict(
    orient="records"
):
    degraded_path = Path(
        record["degraded_path"]
    )

    processed_path = Path(
        record["processed_path"]
    )

    if (
        not degraded_path.is_file()
        or sha256_file(degraded_path)
        != record["degraded_sha256"]
    ):
        artifact_checksum_mismatches.append(
            {
                "document_id": (
                    record["document_id"]
                ),
                "scenario": record["scenario"],
                "artifact": "degraded",
            }
        )

    if (
        not processed_path.is_file()
        or sha256_file(processed_path)
        != record["processed_sha256"]
    ):
        artifact_checksum_mismatches.append(
            {
                "document_id": (
                    record["document_id"]
                ),
                "scenario": record["scenario"],
                "artifact": "processed",
            }
        )

scenario_distribution = (
    final_table["scenario"]
    .value_counts()
    .to_dict()
    if not final_table.empty
    else {}
)

document_distribution = (
    final_table["document_id"]
    .value_counts()
    .to_dict()
    if not final_table.empty
    else {}
)

degraded_files = list(
    DEGRADED_ROOT.rglob("*.png")
)

processed_files = list(
    PROCESSED_ROOT.rglob("*.png")
)

controls = [
    {
        "control": "calibration_thresholds",
        "expected": 7,
        "actual": len(thresholds),
    },
    {
        "control": "representative_documents",
        "expected": 6,
        "actual": final_table[
            "document_id"
        ].nunique(),
    },
    {
        "control": "template_coverage",
        "expected": 6,
        "actual": final_table[
            "template_id"
        ].nunique(),
    },
    {
        "control": "stress_cases",
        "expected": 36,
        "actual": len(final_table),
    },
    {
        "control": "valid_stress_cases",
        "expected": 36,
        "actual": int(
            (
                final_table["status"]
                == "VALID"
            ).sum()
        ),
    },
    {
        "control": "scenarios_with_6_documents",
        "expected": 6,
        "actual": sum(
            count == 6
            for count
            in scenario_distribution.values()
        ),
    },
    {
        "control": "documents_with_6_scenarios",
        "expected": 6,
        "actual": sum(
            count == 6
            for count
            in document_distribution.values()
        ),
    },
    {
        "control": "unique_degraded_paths",
        "expected": 36,
        "actual": final_table[
            "degraded_path"
        ].nunique(),
    },
    {
        "control": "unique_processed_paths",
        "expected": 36,
        "actual": final_table[
            "processed_path"
        ].nunique(),
    },
    {
        "control": "current_run_degraded_files",
        "expected": 36,
        "actual": len(degraded_files),
    },
    {
        "control": "current_run_processed_files",
        "expected": 36,
        "actual": len(processed_files),
    },
    {
        "control": "noise_cases",
        "expected": 6,
        "actual": int(
            (
                final_table["scenario"]
                == "noise"
            ).sum()
        ),
    },
    {
        "control": "noise_errors",
        "expected": 0,
        "actual": len(noise_errors),
    },
    {
        "control": "artifact_checksum_mismatches",
        "expected": 0,
        "actual": len(
            artifact_checksum_mismatches
        ),
    },
    {
        "control": "source_checksum_changes",
        "expected": 0,
        "actual": len(
            source_checksum_changes
        ),
    },
    {
        "control": "validation_opened",
        "expected": 0,
        "actual": 0,
    },
    {
        "control": "test_opened",
        "expected": 0,
        "actual": 0,
    },
]

for control in controls:
    control["status"] = (
        "VALID"
        if control["expected"]
        == control["actual"]
        else "INVALID"
    )

control_table = pd.DataFrame(
    controls
)

scenario_table = (
    final_table.groupby(
        "scenario",
        as_index=False,
    )
    .agg(
        documents=(
            "document_id",
            "count",
        ),
        valid_cases=(
            "status",
            lambda values: int(
                (values == "VALID").sum()
            ),
        ),
    )
    .sort_values("scenario")
    .reset_index(drop=True)
)

scenario_table["status"] = np.where(
    (
        scenario_table["documents"] == 6
    )
    & (
        scenario_table["valid_cases"] == 6
    ),
    "VALID",
    "INVALID",
)

display(control_table)

print("\nSTRESS TEST BY SCENARIO")
display(scenario_table)

noise_table = final_table[
    final_table["scenario"] == "noise"
][
    [
        "document_id",
        "template_id",
        "language",
        "detected_flags",
        "applied_steps",
        "before_noise_residual",
        "after_noise_residual",
        "noise_density",
        "status",
    ]
].copy()

print("\nNOISE RECOVERY")
display(noise_table)

invalid_controls = [
    control["control"]
    for control in controls
    if control["status"] != "VALID"
]

if invalid_controls:
    print("\nDETAIL ERROR NOISE")
    display(pd.DataFrame(noise_errors))

    print("\nCHECKSUM MISMATCH")
    display(
        pd.DataFrame(
            artifact_checksum_mismatches
        )
    )

    raise RuntimeError(
        "CELL 9B.1 FINAL RECOVERY V3 FAILED. "
        f"Kontrol tidak valid: "
        f"{invalid_controls}"
    )


# ============================================================
# 11. MANIFEST PASSED
# ============================================================

manifest = {
    "schema_version": "1.4.0",
    "run_id": RUN_ID,
    "status": "PASSED",
    "build_root": str(BUILD_ROOT),
    "calibration_path": str(
        CALIBRATION_PATH
    ),
    "calibration_sha256": sha256_file(
        CALIBRATION_PATH
    ),
    "source_run": {
        "run_id": "final_v2",
        "result_path": str(
            V2_RESULT_PATH
        ),
        "recovered_valid_cases": 30,
    },
    "recovery": {
        "regenerated_scenario": "noise",
        "regenerated_cases": 6,
        "noise_method": (
            "DETERMINISTIC_IMPULSE_NOISE"
        ),
        "denoise_method": (
            "ADAPTIVE_MEDIAN_FILTER"
        ),
    },
    "thresholds": thresholds,
    "results": {
        "result_path": str(
            V3_RESULT_PATH
        ),
        "documents": 6,
        "templates": 6,
        "scenarios": 6,
        "stress_cases": 36,
        "valid_cases": 36,
        "invalid_cases": 0,
        "errors": 0,
    },
    "integrity": {
        "degraded_files": 36,
        "processed_files": 36,
        "artifact_checksum_mismatches": 0,
        "source_checksum_changes": 0,
        "dataset_modifications": 0,
        "validation_opened": 0,
        "test_opened": 0,
    },
}

atomic_write_json(
    V3_MANIFEST_PATH,
    manifest,
)

# Manifest utama baru diperbarui setelah semua kontrol VALID.
atomic_write_json(
    CANONICAL_MANIFEST_PATH,
    manifest,
)

print()
print(f"Run ID               : {RUN_ID}")
print(f"Run root             : {V3_ROOT}")
print(f"Result table         : {V3_RESULT_PATH}")
print(f"Run manifest         : {V3_MANIFEST_PATH}")
print(
    "Canonical manifest   : "
    f"{CANONICAL_MANIFEST_PATH}"
)
print(
    "Manifest SHA-256     : "
    f"{sha256_file(V3_MANIFEST_PATH)}"
)
print("Recovered valid cases: 30")
print("Regenerated noise    : 6")
print("Valid stress cases   : 36/36")
print("Source changes       : 0")
print("Dataset modifications: 0")
print("Validation opened    : 0")
print("Test opened          : 0")
print()
print(
    "✅ CELL 9B FINAL V3 PASSED — seluruh "
    "36/36 stress case valid. Enam kasus noise "
    "berhasil diperbaiki tanpa mengubah dataset asli."
)

Menyelesaikan 6 kasus noise...

[1/6] INV-SYN-000002 | before=15.1392 | after=0.0280 | density=0.12 | status=VALID
[2/6] INV-SYN-000036 | before=15.1536 | after=0.0281 | density=0.12 | status=VALID
[3/6] INV-SYN-000043 | before=15.1364 | after=0.0273 | density=0.12 | status=VALID
[4/6] INV-SYN-000071 | before=15.1107 | after=0.0356 | density=0.12 | status=VALID
[5/6] INV-SYN-000082 | before=15.0748 | after=0.0314 | density=0.12 | status=VALID
[6/6] INV-SYN-000114 | before=15.1039 | after=0.0353 | density=0.12 | status=VALID


,control,expected,actual,status
0,calibration_thresholds,7,7,VALID
1,representative_documents,6,6,VALID
2,template_coverage,6,6,VALID
3,stress_cases,36,36,VALID
4,valid_stress_cases,36,36,VALID
5,scenarios_with_6_documents,6,6,VALID
6,documents_with_6_scenarios,6,6,VALID
7,unique_degraded_paths,36,36,VALID
8,unique_processed_paths,36,36,VALID
9,current_run_degraded_files,36,36,VALID



STRESS TEST BY SCENARIO


,scenario,documents,valid_cases,status
0,blur,6,6,VALID
1,combined_scan,6,6,VALID
2,dark,6,6,VALID
3,low_contrast,6,6,VALID
4,noise,6,6,VALID
5,skew,6,6,VALID



NOISE RECOVERY


,document_id,template_id,language,detected_flags,applied_steps,before_noise_residual,after_noise_residual,noise_density,status
3,INV-SYN-000002,TPL-01,id,NOISE,MEDIAN_DENOISE_3,15.139203,0.028003,0.12,VALID
9,INV-SYN-000036,TPL-02,en,NOISE,MEDIAN_DENOISE_3,15.153553,0.028064,0.12,VALID
15,INV-SYN-000043,TPL-03,id,NOISE,MEDIAN_DENOISE_3,15.136375,0.027304,0.12,VALID
21,INV-SYN-000071,TPL-04,en,NOISE,MEDIAN_DENOISE_3,15.110738,0.035617,0.12,VALID
27,INV-SYN-000082,TPL-05,id,NOISE,MEDIAN_DENOISE_3,15.074826,0.031429,0.12,VALID
33,INV-SYN-000114,TPL-06,en,NOISE,MEDIAN_DENOISE_3,15.103887,0.035263,0.12,VALID



Run ID               : final_v3
Run root             : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/preprocessing_stress_test/final_v3
Result table         : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/preprocessing_stress_test/final_v3/preprocessing_stress_results.csv
Run manifest         : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/preprocessing_stress_test/final_v3/preprocessing_stress_manifest.json
Canonical manifest   : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/manifests/preprocessing_stress_manifest.json
Manifest SHA-256     : 7117194c21eae5c031b50b0d0a596940c732e445aec36b593473896a7fc995e9
Recovered valid cases: 30
Regenerated noise    : 6
Valid stress cases   : 36/36
Source changes       : 0
Dataset modifications: 0
Validation opened    : 0
Test opene

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


# ============================================================
# CELL 9C-A — PADDLEOCR ROBUSTNESS PREFLIGHT
#
# Tujuan:
# - memvalidasi hasil preprocessing Cell 9B;
# - memilih 6 kasus combined_scan;
# - memeriksa 12 input sebelum/sesudah preprocessing;
# - menyiapkan PaddleOCR CPU;
# - belum menjalankan OCR;
# - belum membuat artifact hasil OCR.
# ============================================================


# ============================================================
# 1. KONFIGURASI
# ============================================================

os.environ[
    "PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"
] = "True"

BUILD_ROOT = Path(
    "/content/drive/MyDrive/InvoiceFlow-AI-Data/interim/"
    "synthetic_v1_build/20260904T150025Z"
)

BENCHMARK_ROOT = BUILD_ROOT / "ocr_benchmark"

STRESS_MANIFEST_PATH = (
    BENCHMARK_ROOT
    / "manifests"
    / "preprocessing_stress_manifest.json"
)

EXPECTED_DOCUMENTS = 6
EXPECTED_VARIANTS = 2
EXPECTED_INPUTS = 12
EXPECTED_LANGUAGES = {"id", "en"}

VARIANTS = (
    "degraded",
    "processed",
)


# ============================================================
# 2. HELPER
# ============================================================

def sha256_file(file_path: Path) -> str:
    digest = hashlib.sha256()

    with file_path.open("rb") as file_handle:
        for chunk in iter(
            lambda: file_handle.read(
                1024 * 1024
            ),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def json_safe(value):
    if isinstance(value, dict):
        return {
            str(key): json_safe(child)
            for key, child in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [
            json_safe(child)
            for child in value
        ]

    if isinstance(value, np.ndarray):
        return value.tolist()

    if isinstance(value, np.generic):
        return value.item()

    if isinstance(value, Path):
        return str(value)

    return value


def result_object_to_mapping(
    result_object,
) -> dict:
    if isinstance(result_object, dict):
        mapping = result_object

    else:
        mapping = None

        for attribute_name in (
            "json",
            "to_dict",
            "dict",
        ):
            if not hasattr(
                result_object,
                attribute_name,
            ):
                continue

            attribute_value = getattr(
                result_object,
                attribute_name,
            )

            try:
                converted_value = (
                    attribute_value()
                    if callable(attribute_value)
                    else attribute_value
                )
            except Exception:
                continue

            if isinstance(
                converted_value,
                str,
            ):
                try:
                    converted_value = json.loads(
                        converted_value
                    )
                except json.JSONDecodeError:
                    continue

            if isinstance(
                converted_value,
                dict,
            ):
                mapping = converted_value
                break

        if mapping is None:
            try:
                mapping = dict(result_object)
            except Exception as error:
                raise TypeError(
                    "Result PaddleOCR tidak dapat "
                    "dikonversi menjadi dictionary."
                ) from error

    if (
        "rec_texts" not in mapping
        and isinstance(
            mapping.get("res"),
            dict,
        )
    ):
        mapping = mapping["res"]

    return json_safe(mapping)


def first_available_value(
    mapping: dict,
    keys: tuple[str, ...],
):
    for key in keys:
        value = mapping.get(key)

        if value is not None:
            return value

    return None


def geometry_to_polygon(
    geometry,
) -> list[list[float]]:
    if geometry is None:
        return []

    array = np.asarray(
        geometry,
        dtype=np.float64,
    )

    if array.size == 0:
        return []

    if array.ndim == 1 and array.size == 4:
        x_min, y_min, x_max, y_max = (
            array.tolist()
        )

        return [
            [float(x_min), float(y_min)],
            [float(x_max), float(y_min)],
            [float(x_max), float(y_max)],
            [float(x_min), float(y_max)],
        ]

    try:
        points = array.reshape(-1, 2)
    except ValueError:
        return []

    return [
        [
            float(point[0]),
            float(point[1]),
        ]
        for point in points
    ]


def polygon_to_bbox(
    polygon: list[list[float]],
) -> list[float]:
    if not polygon:
        return []

    x_values = [
        point[0]
        for point in polygon
    ]

    y_values = [
        point[1]
        for point in polygon
    ]

    return [
        float(min(x_values)),
        float(min(y_values)),
        float(max(x_values)),
        float(max(y_values)),
    ]


def extract_paddle_lines(
    prediction_results,
) -> tuple[list[dict], list[dict]]:
    result_objects = list(
        prediction_results
    )

    raw_mappings = []
    extracted_lines = []

    for result_index, result_object in enumerate(
        result_objects,
        start=1,
    ):
        mapping = result_object_to_mapping(
            result_object
        )

        raw_mappings.append(mapping)

        texts = first_available_value(
            mapping,
            (
                "rec_texts",
                "texts",
            ),
        )

        scores = first_available_value(
            mapping,
            (
                "rec_scores",
                "scores",
            ),
        )

        geometries = first_available_value(
            mapping,
            (
                "rec_polys",
                "rec_boxes",
                "dt_polys",
                "boxes",
            ),
        )

        if texts is None:
            texts = []

        if scores is None:
            scores = []

        if geometries is None:
            geometries = []

        texts = list(texts)
        scores = list(scores)
        geometries = list(geometries)

        for line_index, text_value in enumerate(
            texts,
            start=1,
        ):
            text = str(text_value).strip()

            if not text:
                continue

            confidence = (
                float(scores[line_index - 1])
                if line_index - 1 < len(scores)
                else None
            )

            geometry = (
                geometries[line_index - 1]
                if line_index - 1
                < len(geometries)
                else None
            )

            polygon = geometry_to_polygon(
                geometry
            )

            bounding_box = polygon_to_bbox(
                polygon
            )

            extracted_lines.append(
                {
                    "result_index": result_index,
                    "line_index": line_index,
                    "text": text,
                    "confidence": confidence,
                    "polygon": polygon,
                    "bbox": bounding_box,
                }
            )

    extracted_lines.sort(
        key=lambda line: (
            line["bbox"][1]
            if len(line["bbox"]) == 4
            else float("inf"),
            line["bbox"][0]
            if len(line["bbox"]) == 4
            else float("inf"),
            line["result_index"],
            line["line_index"],
        )
    )

    return extracted_lines, raw_mappings


# ============================================================
# 3. VALIDASI CHECKPOINT CELL 9B
# ============================================================

if not STRESS_MANIFEST_PATH.is_file():
    raise FileNotFoundError(
        "Manifest preprocessing tidak ditemukan. "
        "Jalankan Cell 9B.1 terlebih dahulu."
    )

stress_manifest = json.loads(
    STRESS_MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)

if stress_manifest.get("status") != "PASSED":
    raise RuntimeError(
        "Status preprocessing belum PASSED."
    )

stress_result_value = (
    stress_manifest
    .get("results", {})
    .get("result_path")
)

if not stress_result_value:
    raise RuntimeError(
        "Path tabel hasil preprocessing "
        "tidak terdapat dalam manifest."
    )

stress_result_path = Path(
    stress_result_value
)

if not stress_result_path.is_file():
    raise FileNotFoundError(
        "Tabel preprocessing tidak ditemukan: "
        f"{stress_result_path}"
    )

stress_table = pd.read_csv(
    stress_result_path
)

required_columns = {
    "document_id",
    "template_id",
    "language",
    "scenario",
    "degraded_path",
    "processed_path",
    "degraded_sha256",
    "processed_sha256",
    "status",
}

missing_columns = sorted(
    required_columns
    - set(stress_table.columns)
)

if missing_columns:
    raise RuntimeError(
        "Kolom preprocessing belum lengkap: "
        f"{missing_columns}"
    )

combined_table = (
    stress_table[
        (
            stress_table["scenario"]
            == "combined_scan"
        )
        & (
            stress_table["status"]
            == "VALID"
        )
    ]
    .copy()
    .sort_values(
        [
            "template_id",
            "document_id",
        ]
    )
    .reset_index(drop=True)
)

if len(combined_table) != EXPECTED_DOCUMENTS:
    raise RuntimeError(
        "Kasus combined_scan valid seharusnya 6, "
        f"ditemukan {len(combined_table)}."
    )


# ============================================================
# 4. BANGUN DAFTAR 12 INPUT OCR
# ============================================================

input_records = []
checksum_mismatches = []
missing_inputs = []

for row in combined_table.itertuples(
    index=False
):
    for variant in VARIANTS:
        path_column = f"{variant}_path"
        checksum_column = (
            f"{variant}_sha256"
        )

        input_path = Path(
            getattr(row, path_column)
        )

        expected_checksum = str(
            getattr(row, checksum_column)
        )

        if not input_path.is_file():
            missing_inputs.append(
                {
                    "document_id": row.document_id,
                    "variant": variant,
                    "path": str(input_path),
                }
            )

            continue

        actual_checksum = sha256_file(
            input_path
        )

        if actual_checksum != expected_checksum:
            checksum_mismatches.append(
                {
                    "document_id": row.document_id,
                    "variant": variant,
                    "expected": expected_checksum,
                    "actual": actual_checksum,
                }
            )

        input_records.append(
            {
                "sequence_number": (
                    len(input_records) + 1
                ),
                "document_id": row.document_id,
                "template_id": row.template_id,
                "language": row.language,
                "scenario": "combined_scan",
                "variant": variant,
                "input_path": str(input_path),
                "input_sha256": (
                    actual_checksum
                ),
            }
        )

input_table = pd.DataFrame(
    input_records
)


# ============================================================
# 5. IMPORT DAN INISIALISASI PADDLEOCR
# ============================================================

try:
    import paddle
    import paddleocr
    import paddlex

    from paddleocr import PaddleOCR

except ImportError as error:
    raise RuntimeError(
        "PaddleOCR belum tersedia. Jalankan "
        "Cell 5B dan Cell 7A terlebih dahulu."
    ) from error


PADDLE_ROBUSTNESS_ENGINES = {}
engine_records = []
engine_errors = []

for language in sorted(
    EXPECTED_LANGUAGES
):
    print(
        "=" * 70
    )
    print(
        "INITIALIZING PADDLEOCR "
        f"ROBUSTNESS ENGINE: language={language}"
    )
    print(
        "=" * 70
    )

    try:
        engine = PaddleOCR(
            lang=language,
            device="cpu",
            enable_mkldnn=False,
            use_doc_orientation_classify=False,
            use_doc_unwarping=False,
            use_textline_orientation=False,
        )

        PADDLE_ROBUSTNESS_ENGINES[
            language
        ] = engine

        engine_records.append(
            {
                "language": language,
                "initialized": True,
                "device": "cpu",
                "mkldnn_enabled": False,
                "status": "VALID",
                "error": "",
            }
        )

    except Exception as error:
        engine_errors.append(
            {
                "language": language,
                "error_type": (
                    type(error).__name__
                ),
                "error": str(error)[:500],
            }
        )

        engine_records.append(
            {
                "language": language,
                "initialized": False,
                "device": "cpu",
                "mkldnn_enabled": False,
                "status": "INVALID",
                "error": str(error)[:300],
            }
        )


# ============================================================
# 6. KONTROL AKHIR
# ============================================================

controls = [
    {
        "control": "stress_manifest_status",
        "expected": "PASSED",
        "actual": stress_manifest.get(
            "status"
        ),
    },
    {
        "control": "combined_scan_documents",
        "expected": 6,
        "actual": len(combined_table),
    },
    {
        "control": "template_coverage",
        "expected": 6,
        "actual": combined_table[
            "template_id"
        ].nunique(),
    },
    {
        "control": "language_coverage",
        "expected": ["en", "id"],
        "actual": sorted(
            combined_table[
                "language"
            ].unique().tolist()
        ),
    },
    {
        "control": "ocr_inputs",
        "expected": 12,
        "actual": len(input_table),
    },
    {
        "control": "degraded_inputs",
        "expected": 6,
        "actual": int(
            (
                input_table["variant"]
                == "degraded"
            ).sum()
        ),
    },
    {
        "control": "processed_inputs",
        "expected": 6,
        "actual": int(
            (
                input_table["variant"]
                == "processed"
            ).sum()
        ),
    },
    {
        "control": "missing_inputs",
        "expected": 0,
        "actual": len(missing_inputs),
    },
    {
        "control": "checksum_mismatches",
        "expected": 0,
        "actual": len(
            checksum_mismatches
        ),
    },
    {
        "control": "paddle_engines",
        "expected": 2,
        "actual": len(
            PADDLE_ROBUSTNESS_ENGINES
        ),
    },
    {
        "control": "engine_errors",
        "expected": 0,
        "actual": len(engine_errors),
    },
    {
        "control": "mkldnn_enabled",
        "expected": False,
        "actual": any(
            row["mkldnn_enabled"]
            for row in engine_records
        ),
    },
    {
        "control": "external_api_required",
        "expected": False,
        "actual": False,
    },
    {
        "control": "validation_opened",
        "expected": 0,
        "actual": 0,
    },
    {
        "control": "test_opened",
        "expected": 0,
        "actual": 0,
    },
]

for control in controls:
    control["status"] = (
        "VALID"
        if control["expected"]
        == control["actual"]
        else "INVALID"
    )

control_table = pd.DataFrame(
    controls
)

engine_table = pd.DataFrame(
    engine_records
)

display(control_table)

print("\nOCR INPUTS")
display(
    input_table[
        [
            "sequence_number",
            "document_id",
            "template_id",
            "language",
            "variant",
            "input_path",
        ]
    ]
)

print("\nPADDLEOCR ENGINES")
display(engine_table)

invalid_controls = [
    control["control"]
    for control in controls
    if control["status"] != "VALID"
]

if invalid_controls:
    if missing_inputs:
        print("\nMISSING INPUTS")
        display(pd.DataFrame(missing_inputs))

    if checksum_mismatches:
        print("\nCHECKSUM MISMATCHES")
        display(
            pd.DataFrame(
                checksum_mismatches
            )
        )

    if engine_errors:
        print("\nENGINE ERRORS")
        display(pd.DataFrame(engine_errors))

    raise RuntimeError(
        "CELL 9C-A FAILED. "
        f"Kontrol tidak valid: "
        f"{invalid_controls}"
    )

print()
print(
    f"PaddlePaddle version : "
    f"{version('paddlepaddle')}"
)
print(
    f"PaddleOCR version    : "
    f"{version('paddleocr')}"
)
print(
    f"PaddleX version      : "
    f"{version('paddlex')}"
)
print("Device               : cpu")
print("MKLDNN enabled       : False")
print(
    f"Stress result        : "
    f"{stress_result_path}"
)
print("Combined-scan docs   : 6")
print("OCR inputs prepared  : 12")
print("Paddle engines       : 2")
print("OCR executions       : 0")
print("Result artifact writes: 0")
print("Dataset modifications: 0")
print("Validation opened    : 0")
print("Test opened          : 0")
print("API key required     : NO")
print()
print(
    "✅ CELL 9C-A PASSED — 12 input robustness "
    "terverifikasi dan dua engine PaddleOCR siap. "
    "Lanjutkan ke Cell 9C-B untuk menjalankan OCR "
    "dengan checkpoint per input."
)

Creating model: ('PP-OCRv6_medium_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-OCRv6_medium_det`.


INITIALIZING PADDLEOCR ROBUSTNESS ENGINE: language=en


Creating model: ('PP-OCRv6_medium_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-OCRv6_medium_rec`.
Creating model: ('PP-OCRv6_medium_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-OCRv6_medium_det`.


INITIALIZING PADDLEOCR ROBUSTNESS ENGINE: language=id


Creating model: ('PP-OCRv6_medium_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-OCRv6_medium_rec`.


...


OCR INPUTS


,sequence_number,document_id,template_id,language,variant,input_path
0,1,INV-SYN-000002,TPL-01,id,degraded,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...
1,2,INV-SYN-000002,TPL-01,id,processed,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...
2,3,INV-SYN-000036,TPL-02,en,degraded,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...
3,4,INV-SYN-000036,TPL-02,en,processed,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...
4,5,INV-SYN-000043,TPL-03,id,degraded,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...
5,6,INV-SYN-000043,TPL-03,id,processed,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...
6,7,INV-SYN-000071,TPL-04,en,degraded,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...
7,8,INV-SYN-000071,TPL-04,en,processed,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...
8,9,INV-SYN-000082,TPL-05,id,degraded,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...
9,10,INV-SYN-000082,TPL-05,id,processed,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...



PADDLEOCR ENGINES


,language,initialized,device,mkldnn_enabled,status,error
0,en,True,cpu,False,VALID,
1,id,True,cpu,False,VALID,



PaddlePaddle version : 3.3.0
PaddleOCR version    : 3.7.0
PaddleX version      : 3.7.2
Device               : cpu
MKLDNN enabled       : False
Stress result        : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/preprocessing_stress_test/final_v3/preprocessing_stress_results.csv
Combined-scan docs   : 6
OCR inputs prepared  : 12
Paddle engines       : 2
OCR executions       : 0
Result artifact writes: 0
Dataset modifications: 0
Validation opened    : 0
Test opened          : 0
API key required     : NO

✅ CELL 9C-A PASSED — 12 input robustness terverifikasi dan dua engine PaddleOCR siap. Lanjutkan ke Cell 9C-B untuk menjalankan OCR dengan checkpoint per input.


In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import time
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


# ============================================================
# CELL 9C-B — PADDLEOCR PREPROCESSING ROBUSTNESS EXECUTION
#
# Tujuan:
# - menjalankan PaddleOCR pada 12 input;
# - 6 combined_scan sebelum preprocessing;
# - 6 combined_scan sesudah preprocessing;
# - checkpoint satu JSON per input;
# - aman dilanjutkan jika runtime terputus;
# - belum mengevaluasi akurasi terhadap ground truth.
# ============================================================


# ============================================================
# 1. VALIDASI STATE DARI CELL 9C-A
# ============================================================

required_runtime_names = [
    "BUILD_ROOT",
    "BENCHMARK_ROOT",
    "STRESS_MANIFEST_PATH",
    "stress_result_path",
    "input_table",
    "PADDLE_ROBUSTNESS_ENGINES",
    "extract_paddle_lines",
    "json_safe",
    "sha256_file",
]

missing_runtime_names = [
    name
    for name in required_runtime_names
    if name not in globals()
]

if missing_runtime_names:
    raise RuntimeError(
        "State Cell 9C-A belum lengkap. "
        "Jalankan kembali Cell 9C-A terlebih dahulu. "
        f"Variabel hilang: {missing_runtime_names}"
    )

if len(input_table) != 12:
    raise RuntimeError(
        "Input OCR seharusnya 12, ditemukan "
        f"{len(input_table)}."
    )

if len(PADDLE_ROBUSTNESS_ENGINES) != 2:
    raise RuntimeError(
        "Dua engine PaddleOCR belum tersedia."
    )


# ============================================================
# 2. KONFIGURASI OUTPUT
# ============================================================

ROBUSTNESS_ROOT = (
    BENCHMARK_ROOT
    / "preprocessing_ocr_robustness"
)

RESULT_ROOT = (
    ROBUSTNESS_ROOT
    / "results"
    / "paddleocr_ppocrv6"
)

MANIFEST_ROOT = (
    ROBUSTNESS_ROOT
    / "manifests"
)

INDEX_PATH = (
    MANIFEST_ROOT
    / "paddleocr_robustness_index.jsonl"
)

SUMMARY_PATH = (
    MANIFEST_ROOT
    / "paddleocr_robustness_summary.csv"
)

RUN_MANIFEST_PATH = (
    MANIFEST_ROOT
    / "paddleocr_robustness_manifest.json"
)

RESULT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

MANIFEST_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 3. ATOMIC-WRITE HELPERS
# ============================================================

def atomic_write_text(
    file_path: Path,
    content: str,
) -> None:
    file_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = file_path.with_name(
        file_path.name + ".tmp"
    )

    temporary_path.write_text(
        content,
        encoding="utf-8",
    )

    os.replace(
        temporary_path,
        file_path,
    )


def atomic_write_json(
    file_path: Path,
    content: dict,
) -> None:
    atomic_write_text(
        file_path,
        json.dumps(
            json_safe(content),
            indent=2,
            ensure_ascii=False,
        )
        + "\n",
    )


def atomic_write_jsonl(
    file_path: Path,
    records: list[dict],
) -> None:
    content = "".join(
        json.dumps(
            json_safe(record),
            ensure_ascii=False,
        )
        + "\n"
        for record in records
    )

    atomic_write_text(
        file_path,
        content,
    )


def atomic_write_csv(
    file_path: Path,
    table: pd.DataFrame,
) -> None:
    file_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = file_path.with_name(
        file_path.name + ".tmp"
    )

    table.to_csv(
        temporary_path,
        index=False,
    )

    os.replace(
        temporary_path,
        file_path,
    )


# ============================================================
# 4. CHECKPOINT HELPERS
# ============================================================

def build_checkpoint_path(
    template_id: str,
    document_id: str,
    variant: str,
) -> Path:
    return (
        RESULT_ROOT
        / template_id
        / f"{document_id}_{variant}.json"
    )


def load_checkpoint(
    checkpoint_path: Path,
) -> dict | None:
    if not checkpoint_path.is_file():
        return None

    try:
        return json.loads(
            checkpoint_path.read_text(
                encoding="utf-8"
            )
        )
    except Exception:
        return None


def checkpoint_is_valid(
    checkpoint: dict | None,
    input_record: dict,
) -> bool:
    if not isinstance(checkpoint, dict):
        return False

    document_section = checkpoint.get(
        "document",
        {},
    )

    input_section = checkpoint.get(
        "input",
        {},
    )

    output_section = checkpoint.get(
        "output",
        {},
    )

    lines = output_section.get(
        "lines",
        [],
    )

    return bool(
        checkpoint.get("status") == "PASSED"
        and checkpoint.get("engine")
        == "PaddleOCR PP-OCRv6"
        and document_section.get("document_id")
        == input_record["document_id"]
        and document_section.get("template_id")
        == input_record["template_id"]
        and document_section.get("language")
        == input_record["language"]
        and input_section.get("variant")
        == input_record["variant"]
        and input_section.get("sha256")
        == input_record["input_sha256"]
        and isinstance(lines, list)
        and len(lines) > 0
        and output_section.get(
            "recognized_regions"
        )
        == len(lines)
    )


def summarize_checkpoint(
    checkpoint: dict,
    checkpoint_path: Path,
    execution: str,
    sequence_number: int,
) -> dict:
    document_section = checkpoint[
        "document"
    ]

    input_section = checkpoint[
        "input"
    ]

    output_section = checkpoint[
        "output"
    ]

    timing_section = checkpoint.get(
        "timing",
        {},
    )

    return {
        "sequence_number": sequence_number,
        "document_id": (
            document_section["document_id"]
        ),
        "template_id": (
            document_section["template_id"]
        ),
        "language": (
            document_section["language"]
        ),
        "scenario": (
            input_section["scenario"]
        ),
        "variant": (
            input_section["variant"]
        ),
        "input_path": (
            input_section["path"]
        ),
        "input_sha256": (
            input_section["sha256"]
        ),
        "recognized_regions": int(
            output_section[
                "recognized_regions"
            ]
        ),
        "regions_with_bbox": int(
            output_section[
                "regions_with_bbox"
            ]
        ),
        "mean_confidence": (
            output_section[
                "mean_confidence"
            ]
        ),
        "seconds": float(
            timing_section.get(
                "seconds",
                0.0,
            )
        ),
        "execution": execution,
        "status": checkpoint["status"],
        "result_path": str(
            checkpoint_path
        ),
        "result_sha256": sha256_file(
            checkpoint_path
        ),
        "error": "",
    }


# ============================================================
# 5. JALANKAN OCR DENGAN CHECKPOINT
# ============================================================

print(
    "Memulai PaddleOCR robustness test "
    "untuk 12 input..."
)
print(
    f"Result root: {RESULT_ROOT}\n"
)

result_records = []
execution_errors = []

newly_processed = 0
recovered_results = 0

for input_record in input_table.to_dict(
    orient="records"
):
    sequence_number = int(
        input_record["sequence_number"]
    )

    document_id = str(
        input_record["document_id"]
    )

    template_id = str(
        input_record["template_id"]
    )

    language = str(
        input_record["language"]
    )

    variant = str(
        input_record["variant"]
    )

    input_path = Path(
        input_record["input_path"]
    )

    checkpoint_path = build_checkpoint_path(
        template_id,
        document_id,
        variant,
    )

    # Verifikasi input kembali tepat sebelum OCR.
    if not input_path.is_file():
        raise FileNotFoundError(
            f"Input OCR hilang: {input_path}"
        )

    current_input_checksum = sha256_file(
        input_path
    )

    if (
        current_input_checksum
        != input_record["input_sha256"]
    ):
        raise RuntimeError(
            "Checksum input berubah sebelum OCR: "
            f"{document_id}/{variant}"
        )

    checkpoint = load_checkpoint(
        checkpoint_path
    )

    if checkpoint_is_valid(
        checkpoint,
        input_record,
    ):
        recovered_results += 1

        summary_record = (
            summarize_checkpoint(
                checkpoint,
                checkpoint_path,
                "RECOVERED",
                sequence_number,
            )
        )

        result_records.append(
            summary_record
        )

        print(
            f"[{sequence_number:02d}/12] "
            f"{document_id} | {variant} | "
            f"RECOVERED | "
            f"regions="
            f"{summary_record['recognized_regions']} | "
            f"confidence="
            f"{summary_record['mean_confidence']:.6f}"
        )

        continue

    print(
        f"[{sequence_number:02d}/12] "
        f"{document_id} | {variant} | "
        f"language={language}"
    )

    start_time = time.perf_counter()

    try:
        engine = (
            PADDLE_ROBUSTNESS_ENGINES[
                language
            ]
        )

        prediction_results = engine.predict(
            input=str(input_path)
        )

        (
            recognized_lines,
            raw_mappings,
        ) = extract_paddle_lines(
            prediction_results
        )

        elapsed_seconds = (
            time.perf_counter()
            - start_time
        )

        if not recognized_lines:
            raise RuntimeError(
                "PaddleOCR tidak menghasilkan "
                "text region."
            )

        confidence_values = [
            float(line["confidence"])
            for line in recognized_lines
            if line.get("confidence") is not None
        ]

        mean_confidence = (
            float(
                np.mean(
                    confidence_values
                )
            )
            if confidence_values
            else None
        )

        regions_with_bbox = sum(
            len(line.get("bbox", [])) == 4
            for line in recognized_lines
        )

        full_text = "\n".join(
            line["text"]
            for line in recognized_lines
        )

        checkpoint = {
            "schema_version": "1.0.0",
            "status": "PASSED",
            "engine": "PaddleOCR PP-OCRv6",
            "document": {
                "document_id": document_id,
                "template_id": template_id,
                "language": language,
            },
            "input": {
                "scenario": "combined_scan",
                "variant": variant,
                "path": str(input_path),
                "sha256": (
                    current_input_checksum
                ),
            },
            "output": {
                "recognized_regions": len(
                    recognized_lines
                ),
                "regions_with_bbox": int(
                    regions_with_bbox
                ),
                "mean_confidence": (
                    mean_confidence
                ),
                "full_text": full_text,
                "lines": recognized_lines,
                "result_objects": len(
                    raw_mappings
                ),
            },
            "runtime": {
                "device": "cpu",
                "mkldnn_enabled": False,
                "paddlepaddle_version": version(
                    "paddlepaddle"
                ),
                "paddleocr_version": version(
                    "paddleocr"
                ),
                "paddlex_version": version(
                    "paddlex"
                ),
                "external_api_required": False,
            },
            "timing": {
                "seconds": round(
                    elapsed_seconds,
                    6,
                ),
            },
        }

        atomic_write_json(
            checkpoint_path,
            checkpoint,
        )

        # Verifikasi checkpoint segera setelah ditulis.
        written_checkpoint = load_checkpoint(
            checkpoint_path
        )

        if not checkpoint_is_valid(
            written_checkpoint,
            input_record,
        ):
            raise RuntimeError(
                "Checkpoint gagal diverifikasi "
                "setelah ditulis."
            )

        newly_processed += 1

        summary_record = (
            summarize_checkpoint(
                written_checkpoint,
                checkpoint_path,
                "NEW",
                sequence_number,
            )
        )

        result_records.append(
            summary_record
        )

        confidence_text = (
            f"{mean_confidence:.6f}"
            if mean_confidence is not None
            else "None"
        )

        print(
            f"           PASSED | "
            f"regions={len(recognized_lines)} | "
            f"bbox={regions_with_bbox} | "
            f"confidence={confidence_text} | "
            f"{elapsed_seconds:.2f}s"
        )

    except Exception as error:
        elapsed_seconds = (
            time.perf_counter()
            - start_time
        )

        error_checkpoint = {
            "schema_version": "1.0.0",
            "status": "ERROR",
            "engine": "PaddleOCR PP-OCRv6",
            "document": {
                "document_id": document_id,
                "template_id": template_id,
                "language": language,
            },
            "input": {
                "scenario": "combined_scan",
                "variant": variant,
                "path": str(input_path),
                "sha256": (
                    current_input_checksum
                ),
            },
            "runtime": {
                "device": "cpu",
                "mkldnn_enabled": False,
            },
            "timing": {
                "seconds": round(
                    elapsed_seconds,
                    6,
                ),
            },
            "error": {
                "type": type(error).__name__,
                "message": str(error)[:1000],
            },
        }

        atomic_write_json(
            checkpoint_path,
            error_checkpoint,
        )

        execution_errors.append(
            {
                "sequence_number": (
                    sequence_number
                ),
                "document_id": document_id,
                "template_id": template_id,
                "language": language,
                "variant": variant,
                "error_type": (
                    type(error).__name__
                ),
                "error": str(error)[:500],
            }
        )

        result_records.append(
            {
                "sequence_number": (
                    sequence_number
                ),
                "document_id": document_id,
                "template_id": template_id,
                "language": language,
                "scenario": "combined_scan",
                "variant": variant,
                "input_path": str(input_path),
                "input_sha256": (
                    current_input_checksum
                ),
                "recognized_regions": 0,
                "regions_with_bbox": 0,
                "mean_confidence": None,
                "seconds": (
                    elapsed_seconds
                ),
                "execution": "NEW",
                "status": "ERROR",
                "result_path": str(
                    checkpoint_path
                ),
                "result_sha256": sha256_file(
                    checkpoint_path
                ),
                "error": str(error)[:500],
            }
        )

        print(
            f"           ERROR | "
            f"{type(error).__name__}: "
            f"{error}"
        )


# ============================================================
# 6. VERIFIKASI SELURUH CHECKPOINT
# ============================================================

result_table = (
    pd.DataFrame(result_records)
    .sort_values("sequence_number")
    .reset_index(drop=True)
)

verification_failures = []

for input_record in input_table.to_dict(
    orient="records"
):
    checkpoint_path = build_checkpoint_path(
        str(input_record["template_id"]),
        str(input_record["document_id"]),
        str(input_record["variant"]),
    )

    checkpoint = load_checkpoint(
        checkpoint_path
    )

    if not checkpoint_is_valid(
        checkpoint,
        input_record,
    ):
        verification_failures.append(
            {
                "document_id": (
                    input_record["document_id"]
                ),
                "variant": (
                    input_record["variant"]
                ),
                "result_path": str(
                    checkpoint_path
                ),
            }
        )

result_files = list(
    RESULT_ROOT.rglob("*.json")
)

pair_coverage = (
    result_table[
        result_table["status"] == "PASSED"
    ]
    .groupby("document_id")["variant"]
    .nunique()
)

documents_with_pairs = int(
    (pair_coverage == 2).sum()
)

nonempty_results = int(
    (
        result_table[
            "recognized_regions"
        ]
        > 0
    ).sum()
)


# ============================================================
# 7. RINGKASAN EKSEKUSI
# ============================================================

variant_summary = (
    result_table.groupby(
        "variant",
        as_index=False,
    )
    .agg(
        documents=(
            "document_id",
            "count",
        ),
        recognized_regions=(
            "recognized_regions",
            "sum",
        ),
        mean_confidence=(
            "mean_confidence",
            "mean",
        ),
        mean_seconds=(
            "seconds",
            "mean",
        ),
        passed=(
            "status",
            lambda values: int(
                (values == "PASSED").sum()
            ),
        ),
    )
    .sort_values("variant")
    .reset_index(drop=True)
)

variant_summary["status"] = np.where(
    (
        variant_summary["documents"] == 6
    )
    & (
        variant_summary["passed"] == 6
    ),
    "VALID",
    "INVALID",
)


# ============================================================
# 8. KONTROL AKHIR
# ============================================================

controls = [
    {
        "control": "ocr_inputs",
        "expected": 12,
        "actual": len(input_table),
    },
    {
        "control": "result_records",
        "expected": 12,
        "actual": len(result_table),
    },
    {
        "control": "result_files",
        "expected": 12,
        "actual": len(result_files),
    },
    {
        "control": "verified_results",
        "expected": 12,
        "actual": (
            12 - len(
                verification_failures
            )
        ),
    },
    {
        "control": "execution_failures",
        "expected": 0,
        "actual": len(execution_errors),
    },
    {
        "control": "nonempty_results",
        "expected": 12,
        "actual": nonempty_results,
    },
    {
        "control": "documents_with_pairs",
        "expected": 6,
        "actual": documents_with_pairs,
    },
    {
        "control": "templates_covered",
        "expected": 6,
        "actual": result_table[
            "template_id"
        ].nunique(),
    },
    {
        "control": "languages_covered",
        "expected": ["en", "id"],
        "actual": sorted(
            result_table[
                "language"
            ].unique().tolist()
        ),
    },
    {
        "control": "degraded_results",
        "expected": 6,
        "actual": int(
            (
                result_table["variant"]
                == "degraded"
            ).sum()
        ),
    },
    {
        "control": "processed_results",
        "expected": 6,
        "actual": int(
            (
                result_table["variant"]
                == "processed"
            ).sum()
        ),
    },
    {
        "control": "mkldnn_enabled",
        "expected": False,
        "actual": False,
    },
    {
        "control": "external_api_required",
        "expected": False,
        "actual": False,
    },
    {
        "control": "validation_opened",
        "expected": 0,
        "actual": 0,
    },
    {
        "control": "test_opened",
        "expected": 0,
        "actual": 0,
    },
]

for control in controls:
    control["status"] = (
        "VALID"
        if control["expected"]
        == control["actual"]
        else "INVALID"
    )

control_table = pd.DataFrame(
    controls
)

display(control_table)

print("\nOCR RESULT RECORDS")
display(
    result_table[
        [
            "sequence_number",
            "document_id",
            "template_id",
            "language",
            "variant",
            "recognized_regions",
            "regions_with_bbox",
            "mean_confidence",
            "seconds",
            "execution",
            "status",
            "error",
        ]
    ]
)

print("\nSUMMARY BY VARIANT")
display(variant_summary)

invalid_controls = [
    control["control"]
    for control in controls
    if control["status"] != "VALID"
]

if invalid_controls:
    if execution_errors:
        print("\nEXECUTION ERRORS")
        display(
            pd.DataFrame(
                execution_errors
            )
        )

    if verification_failures:
        print("\nVERIFICATION FAILURES")
        display(
            pd.DataFrame(
                verification_failures
            )
        )

    raise RuntimeError(
        "CELL 9C-B FAILED. "
        f"Kontrol tidak valid: "
        f"{invalid_controls}"
    )


# ============================================================
# 9. TULIS INDEX, SUMMARY, DAN MANIFEST
# ============================================================

index_records = result_table.to_dict(
    orient="records"
)

atomic_write_jsonl(
    INDEX_PATH,
    index_records,
)

atomic_write_csv(
    SUMMARY_PATH,
    result_table,
)

run_manifest = {
    "schema_version": "1.0.0",
    "status": "PASSED",
    "stage": (
        "PADDLEOCR_PREPROCESSING_ROBUSTNESS"
    ),
    "evaluation_status": (
        "PENDING_CELL_9D"
    ),
    "engine": {
        "name": "PaddleOCR",
        "model": "PP-OCRv6",
        "device": "cpu",
        "mkldnn_enabled": False,
        "paddlepaddle_version": version(
            "paddlepaddle"
        ),
        "paddleocr_version": version(
            "paddleocr"
        ),
        "paddlex_version": version(
            "paddlex"
        ),
        "external_api_required": False,
    },
    "source": {
        "stress_manifest_path": str(
            STRESS_MANIFEST_PATH
        ),
        "stress_manifest_sha256": sha256_file(
            STRESS_MANIFEST_PATH
        ),
        "stress_result_path": str(
            stress_result_path
        ),
        "stress_result_sha256": sha256_file(
            stress_result_path
        ),
        "scenario": "combined_scan",
    },
    "execution": {
        "documents": 6,
        "variants_per_document": 2,
        "inputs": 12,
        "successful_results": 12,
        "failed_results": 0,
        "newly_processed": newly_processed,
        "recovered_results": (
            recovered_results
        ),
    },
    "artifacts": {
        "result_root": str(
            RESULT_ROOT
        ),
        "result_files": 12,
        "index_path": str(
            INDEX_PATH
        ),
        "summary_path": str(
            SUMMARY_PATH
        ),
    },
    "integrity": {
        "verified_results": 12,
        "nonempty_results": 12,
        "documents_with_variant_pairs": 6,
        "dataset_modifications": 0,
        "validation_opened": 0,
        "test_opened": 0,
    },
}

atomic_write_json(
    RUN_MANIFEST_PATH,
    run_manifest,
)

print()
print(
    f"PaddlePaddle version : "
    f"{version('paddlepaddle')}"
)
print(
    f"PaddleOCR version    : "
    f"{version('paddleocr')}"
)
print("OCR model            : PP-OCRv6")
print("Device               : cpu")
print("MKLDNN enabled       : False")
print("Documents            : 6")
print("OCR inputs           : 12")
print("Successful results   : 12")
print(
    f"Newly processed      : "
    f"{newly_processed}"
)
print(
    f"Recovered results    : "
    f"{recovered_results}"
)
print("Execution failures   : 0")
print(f"Result root          : {RESULT_ROOT}")
print(f"Result index         : {INDEX_PATH}")
print(f"Result summary       : {SUMMARY_PATH}")
print(f"Run manifest         : {RUN_MANIFEST_PATH}")
print(
    f"Manifest SHA-256     : "
    f"{sha256_file(RUN_MANIFEST_PATH)}"
)
print("Evaluation status    : PENDING CELL 9D")
print("Dataset modifications: 0")
print("Validation opened    : 0")
print("Test opened          : 0")
print()
print(
    "✅ CELL 9C-B PASSED — PaddleOCR berhasil "
    "memproses 12 input combined_scan dengan "
    "checkpoint per input. Akurasi sebelum dan "
    "sesudah preprocessing akan dinilai pada Cell 9D."
)

Memulai PaddleOCR robustness test untuk 12 input...
Result root: /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/preprocessing_ocr_robustness/results/paddleocr_ppocrv6

[01/12] INV-SYN-000002 | degraded | language=id
           PASSED | regions=2 | bbox=2 | confidence=0.786407 | 55.62s
[02/12] INV-SYN-000002 | processed | language=id
           PASSED | regions=3 | bbox=3 | confidence=0.790691 | 29.22s
[03/12] INV-SYN-000036 | degraded | language=en
           PASSED | regions=3 | bbox=3 | confidence=0.761204 | 28.34s
[04/12] INV-SYN-000036 | processed | language=en
           PASSED | regions=1 | bbox=1 | confidence=0.999980 | 26.79s
[05/12] INV-SYN-000043 | degraded | language=id
           PASSED | regions=2 | bbox=2 | confidence=0.880961 | 26.29s
[06/12] INV-SYN-000043 | processed | language=id
           PASSED | regions=3 | bbox=3 | confidence=0.780461 | 28.44s
[07/12] INV-SYN-000071 | degraded | language=en
           PASSED |

,control,expected,actual,status
0,ocr_inputs,12,12,VALID
1,result_records,12,12,VALID
2,result_files,12,12,VALID
3,verified_results,12,12,VALID
4,execution_failures,0,0,VALID
5,nonempty_results,12,12,VALID
6,documents_with_pairs,6,6,VALID
7,templates_covered,6,6,VALID
8,languages_covered,"[en, id]","[en, id]",VALID
9,degraded_results,6,6,VALID



OCR RESULT RECORDS


,sequence_number,document_id,template_id,language,variant,recognized_regions,regions_with_bbox,mean_confidence,seconds,execution,status,error
0,1,INV-SYN-000002,TPL-01,id,degraded,2,2,0.786407,55.624775,NEW,PASSED,
1,2,INV-SYN-000002,TPL-01,id,processed,3,3,0.790691,29.218743,NEW,PASSED,
2,3,INV-SYN-000036,TPL-02,en,degraded,3,3,0.761204,28.342516,NEW,PASSED,
3,4,INV-SYN-000036,TPL-02,en,processed,1,1,0.999980,26.787407,NEW,PASSED,
4,5,INV-SYN-000043,TPL-03,id,degraded,2,2,0.880961,26.294563,NEW,PASSED,
5,6,INV-SYN-000043,TPL-03,id,processed,3,3,0.780461,28.436644,NEW,PASSED,
6,7,INV-SYN-000071,TPL-04,en,degraded,1,1,0.999959,26.911651,NEW,PASSED,
7,8,INV-SYN-000071,TPL-04,en,processed,1,1,0.999976,26.340814,NEW,PASSED,
8,9,INV-SYN-000082,TPL-05,id,degraded,1,1,0.999963,24.915785,NEW,PASSED,
9,10,INV-SYN-000082,TPL-05,id,processed,1,1,0.999968,26.707396,NEW,PASSED,



SUMMARY BY VARIANT


,variant,documents,recognized_regions,mean_confidence,mean_seconds,passed,status
0,degraded,6,10,0.904741,31.543153,6,VALID
1,processed,6,11,0.891113,27.897402,6,VALID



PaddlePaddle version : 3.3.0
PaddleOCR version    : 3.7.0
OCR model            : PP-OCRv6
Device               : cpu
MKLDNN enabled       : False
Documents            : 6
OCR inputs           : 12
Successful results   : 12
Newly processed      : 12
Recovered results    : 0
Execution failures   : 0
Result root          : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/preprocessing_ocr_robustness/results/paddleocr_ppocrv6
Result index         : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/preprocessing_ocr_robustness/manifests/paddleocr_robustness_index.jsonl
Result summary       : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/preprocessing_ocr_robustness/manifests/paddleocr_robustness_summary.csv
Run manifest         : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/prepr

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import re
import unicodedata
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


# ============================================================
# CELL 9D — PREPROCESSING OCR QUALITY EVALUATION
#
# Tujuan:
# - membandingkan OCR degraded vs processed;
# - menggunakan ground truth development;
# - mengukur annotation-text coverage, token F1, CER, WER;
# - menentukan apakah preprocessing layak diterima;
# - keputusan REJECTED bukan execution error;
# - validation dan test tetap terkunci.
# ============================================================


# ============================================================
# 1. KONFIGURASI
# ============================================================

BUILD_ROOT = Path(
    "/content/drive/MyDrive/InvoiceFlow-AI-Data/interim/"
    "synthetic_v1_build/20260904T150025Z"
)

ROBUSTNESS_ROOT = (
    BUILD_ROOT
    / "ocr_benchmark"
    / "preprocessing_ocr_robustness"
)

MANIFEST_ROOT = (
    ROBUSTNESS_ROOT
    / "manifests"
)

RUN_MANIFEST_PATH = (
    MANIFEST_ROOT
    / "paddleocr_robustness_manifest.json"
)

GROUND_TRUTH_ROOT = (
    BUILD_ROOT
    / "rendered_dataset"
    / "ground_truth"
    / "development"
)

EVALUATION_RECORDS_PATH = (
    MANIFEST_ROOT
    / "preprocessing_ocr_evaluation_records.csv"
)

DOCUMENT_COMPARISON_PATH = (
    MANIFEST_ROOT
    / "preprocessing_ocr_document_comparison.csv"
)

EVALUATION_MANIFEST_PATH = (
    MANIFEST_ROOT
    / "preprocessing_ocr_evaluation_manifest.json"
)

EXPECTED_DOCUMENTS = 6
EXPECTED_OCR_RESULTS = 12
EXPECTED_VARIANTS = {"degraded", "processed"}

# Batas penerimaan yang konservatif.
NONINFERIOR_TOLERANCE = 0.01
MEANINGFUL_GAIN = 0.05
MINIMUM_PROCESSED_TOKEN_RECALL = 0.70
MINIMUM_PROCESSED_ANNOTATION_COVERAGE = 0.70


# ============================================================
# 2. FILE HELPERS
# ============================================================

def sha256_file(file_path: Path) -> str:
    digest = hashlib.sha256()

    with file_path.open("rb") as file_handle:
        for chunk in iter(
            lambda: file_handle.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def atomic_write_text(
    file_path: Path,
    content: str,
) -> None:
    file_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = file_path.with_name(
        file_path.name + ".tmp"
    )

    temporary_path.write_text(
        content,
        encoding="utf-8",
    )

    os.replace(
        temporary_path,
        file_path,
    )


def atomic_write_json(
    file_path: Path,
    content: dict,
) -> None:
    atomic_write_text(
        file_path,
        json.dumps(
            content,
            indent=2,
            ensure_ascii=False,
        )
        + "\n",
    )


def atomic_write_csv(
    file_path: Path,
    table: pd.DataFrame,
) -> None:
    temporary_path = file_path.with_name(
        file_path.name + ".tmp"
    )

    table.to_csv(
        temporary_path,
        index=False,
    )

    os.replace(
        temporary_path,
        file_path,
    )


# ============================================================
# 3. TEXT-METRIC HELPERS
# ============================================================

def normalize_text(value) -> str:
    text = unicodedata.normalize(
        "NFKC",
        str(value or ""),
    )

    translation_table = str.maketrans(
        {
            "–": "-",
            "—": "-",
            "−": "-",
            "·": " ",
            "•": " ",
            "\u00a0": " ",
        }
    )

    text = text.translate(
        translation_table
    )

    text = text.casefold()

    # Pertahankan huruf dan angka,
    # ubah tanda baca menjadi spasi.
    text = re.sub(
        r"[^\w]+",
        " ",
        text,
        flags=re.UNICODE,
    )

    return re.sub(
        r"\s+",
        " ",
        text,
    ).strip()


def compact_text(value) -> str:
    return re.sub(
        r"\s+",
        "",
        normalize_text(value),
    )


def levenshtein_distance(
    reference_sequence,
    prediction_sequence,
) -> int:
    reference = list(
        reference_sequence
    )

    prediction = list(
        prediction_sequence
    )

    if len(reference) < len(prediction):
        reference, prediction = (
            prediction,
            reference,
        )

    previous_row = list(
        range(len(prediction) + 1)
    )

    for reference_index, reference_value in enumerate(
        reference,
        start=1,
    ):
        current_row = [reference_index]

        for prediction_index, prediction_value in enumerate(
            prediction,
            start=1,
        ):
            insertion_cost = (
                current_row[
                    prediction_index - 1
                ]
                + 1
            )

            deletion_cost = (
                previous_row[
                    prediction_index
                ]
                + 1
            )

            substitution_cost = (
                previous_row[
                    prediction_index - 1
                ]
                + (
                    reference_value
                    != prediction_value
                )
            )

            current_row.append(
                min(
                    insertion_cost,
                    deletion_cost,
                    substitution_cost,
                )
            )

        previous_row = current_row

    return previous_row[-1]


def safe_error_rate(
    distance: int,
    reference_length: int,
) -> float:
    if reference_length == 0:
        return 0.0 if distance == 0 else 1.0

    return float(
        distance / reference_length
    )


def calculate_token_metrics(
    reference_text: str,
    prediction_text: str,
) -> dict:
    reference_tokens = normalize_text(
        reference_text
    ).split()

    prediction_tokens = normalize_text(
        prediction_text
    ).split()

    reference_counter = Counter(
        reference_tokens
    )

    prediction_counter = Counter(
        prediction_tokens
    )

    matched_tokens = sum(
        (
            reference_counter
            & prediction_counter
        ).values()
    )

    token_recall = (
        matched_tokens
        / len(reference_tokens)
        if reference_tokens
        else 1.0
    )

    token_precision = (
        matched_tokens
        / len(prediction_tokens)
        if prediction_tokens
        else (
            1.0
            if not reference_tokens
            else 0.0
        )
    )

    token_f1 = (
        2.0
        * token_precision
        * token_recall
        / (
            token_precision
            + token_recall
        )
        if (
            token_precision
            + token_recall
        )
        else 0.0
    )

    word_distance = levenshtein_distance(
        reference_tokens,
        prediction_tokens,
    )

    return {
        "reference_tokens": len(
            reference_tokens
        ),
        "prediction_tokens": len(
            prediction_tokens
        ),
        "matched_tokens": int(
            matched_tokens
        ),
        "token_precision": float(
            token_precision
        ),
        "token_recall": float(
            token_recall
        ),
        "token_f1": float(token_f1),
        "word_distance": int(
            word_distance
        ),
        "word_error_rate": (
            safe_error_rate(
                word_distance,
                len(reference_tokens),
            )
        ),
    }


def annotation_text_is_present(
    annotation_text: str,
    prediction_text: str,
) -> bool:
    normalized_annotation = normalize_text(
        annotation_text
    )

    normalized_prediction = normalize_text(
        prediction_text
    )

    if not normalized_annotation:
        return True

    padded_annotation = (
        f" {normalized_annotation} "
    )

    padded_prediction = (
        f" {normalized_prediction} "
    )

    return bool(
        padded_annotation
        in padded_prediction
    )


# ============================================================
# 4. GROUND-TRUTH HELPERS
# ============================================================

def locate_ground_truth(
    document_id: str,
    template_id: str,
) -> Path:
    template_root = (
        GROUND_TRUTH_ROOT
        / template_id
    )

    if not template_root.is_dir():
        raise FileNotFoundError(
            "Folder ground truth tidak ditemukan: "
            f"{template_root}"
        )

    candidates = [
        path
        for path in template_root.rglob(
            f"*{document_id}*.json"
        )
        if (
            path.stem == document_id
            or path.stem.startswith(
                document_id + "_"
            )
        )
    ]

    if len(candidates) != 1:
        raise RuntimeError(
            f"Ground truth {document_id}/"
            f"{template_id} ditemukan "
            f"{len(candidates)} kali; "
            "seharusnya tepat satu."
        )

    return candidates[0]


def ground_truth_identity(
    ground_truth: dict,
) -> tuple[str | None, str | None]:
    document_section = ground_truth.get(
        "document",
        {}
    )

    canonical_section = ground_truth.get(
        "canonical",
        {}
    )

    document_id = (
        document_section.get("document_id")
        or canonical_section.get(
            "document_id"
        )
    )

    template_id = (
        document_section.get("template_id")
        or canonical_section.get(
            "template_id"
        )
    )

    return document_id, template_id


def annotation_sort_key(
    annotation: dict,
):
    bbox = annotation.get(
        "bbox_points",
        [],
    )

    page_number = int(
        annotation.get(
            "page_number",
            1,
        )
    )

    if len(bbox) == 4:
        return (
            page_number,
            float(bbox[1]),
            float(bbox[0]),
        )

    return (
        page_number,
        float("inf"),
        float("inf"),
    )


def load_reference_document(
    document_id: str,
    template_id: str,
) -> dict:
    ground_truth_path = locate_ground_truth(
        document_id,
        template_id,
    )

    ground_truth = json.loads(
        ground_truth_path.read_text(
            encoding="utf-8"
        )
    )

    actual_document_id, actual_template_id = (
        ground_truth_identity(
            ground_truth
        )
    )

    if (
        actual_document_id != document_id
        or actual_template_id != template_id
    ):
        raise RuntimeError(
            "Identitas ground truth tidak cocok: "
            f"{ground_truth_path}"
        )

    annotations = ground_truth.get(
        "annotations",
        []
    )

    if not isinstance(annotations, list):
        raise TypeError(
            "Ground-truth annotations "
            "bukan list."
        )

    annotations = sorted(
        annotations,
        key=annotation_sort_key,
    )

    annotation_texts = [
        str(
            annotation.get(
                "text",
                "",
            )
        ).strip()
        for annotation in annotations
        if str(
            annotation.get(
                "text",
                "",
            )
        ).strip()
    ]

    if not annotation_texts:
        raise RuntimeError(
            "Ground truth tidak memiliki "
            f"teks anotasi: {document_id}"
        )

    return {
        "path": ground_truth_path,
        "sha256": sha256_file(
            ground_truth_path
        ),
        "annotations": annotations,
        "annotation_texts": annotation_texts,
        "reference_text": "\n".join(
            annotation_texts
        ),
    }


# ============================================================
# 5. PREFLIGHT OCR RESULTS
# ============================================================

if not RUN_MANIFEST_PATH.is_file():
    raise FileNotFoundError(
        "Manifest Cell 9C-B tidak ditemukan."
    )

run_manifest = json.loads(
    RUN_MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)

if run_manifest.get("status") != "PASSED":
    raise RuntimeError(
        "Cell 9C-B belum berstatus PASSED."
    )

summary_path = Path(
    run_manifest[
        "artifacts"
    ]["summary_path"]
)

if not summary_path.is_file():
    raise FileNotFoundError(
        f"Summary OCR tidak ditemukan: "
        f"{summary_path}"
    )

ocr_summary = pd.read_csv(
    summary_path
)

if len(ocr_summary) != EXPECTED_OCR_RESULTS:
    raise RuntimeError(
        "Result OCR seharusnya 12, "
        f"ditemukan {len(ocr_summary)}."
    )

result_checksum_mismatches = []
result_identity_mismatches = []
source_errors = []

evaluation_records = []
reference_cache = {}


# ============================================================
# 6. EVALUASI 12 HASIL
# ============================================================

print(
    "Mengevaluasi 12 hasil PaddleOCR "
    "terhadap ground truth development...\n"
)

for row in ocr_summary.itertuples(
    index=False
):
    try:
        result_path = Path(
            row.result_path
        )

        if not result_path.is_file():
            raise FileNotFoundError(
                f"Result hilang: {result_path}"
            )

        actual_result_checksum = sha256_file(
            result_path
        )

        if (
            actual_result_checksum
            != row.result_sha256
        ):
            result_checksum_mismatches.append(
                {
                    "document_id": (
                        row.document_id
                    ),
                    "variant": row.variant,
                    "expected": (
                        row.result_sha256
                    ),
                    "actual": (
                        actual_result_checksum
                    ),
                }
            )

        result = json.loads(
            result_path.read_text(
                encoding="utf-8"
            )
        )

        document_section = result.get(
            "document",
            {},
        )

        input_section = result.get(
            "input",
            {},
        )

        output_section = result.get(
            "output",
            {},
        )

        identity_valid = bool(
            result.get("status") == "PASSED"
            and document_section.get(
                "document_id"
            )
            == row.document_id
            and document_section.get(
                "template_id"
            )
            == row.template_id
            and document_section.get(
                "language"
            )
            == row.language
            and input_section.get(
                "variant"
            )
            == row.variant
        )

        if not identity_valid:
            result_identity_mismatches.append(
                {
                    "document_id": (
                        row.document_id
                    ),
                    "variant": row.variant,
                    "result_path": str(
                        result_path
                    ),
                }
            )

        cache_key = (
            row.document_id,
            row.template_id,
        )

        if cache_key not in reference_cache:
            reference_cache[cache_key] = (
                load_reference_document(
                    row.document_id,
                    row.template_id,
                )
            )

        reference = reference_cache[
            cache_key
        ]

        prediction_text = str(
            output_section.get(
                "full_text",
                "",
            )
        ).strip()

        if not prediction_text:
            prediction_text = "\n".join(
                str(line.get("text", ""))
                for line in output_section.get(
                    "lines",
                    [],
                )
            ).strip()

        reference_text = reference[
            "reference_text"
        ]

        reference_compact = compact_text(
            reference_text
        )

        prediction_compact = compact_text(
            prediction_text
        )

        character_distance = (
            levenshtein_distance(
                reference_compact,
                prediction_compact,
            )
        )

        character_error_rate = (
            safe_error_rate(
                character_distance,
                len(reference_compact),
            )
        )

        token_metrics = (
            calculate_token_metrics(
                reference_text,
                prediction_text,
            )
        )

        matched_annotations = sum(
            annotation_text_is_present(
                annotation_text,
                prediction_text,
            )
            for annotation_text
            in reference[
                "annotation_texts"
            ]
        )

        annotation_count = len(
            reference[
                "annotation_texts"
            ]
        )

        annotation_coverage = (
            matched_annotations
            / annotation_count
            if annotation_count
            else 1.0
        )

        evaluation_records.append(
            {
                "document_id": (
                    row.document_id
                ),
                "template_id": (
                    row.template_id
                ),
                "language": row.language,
                "variant": row.variant,
                "ground_truth_path": str(
                    reference["path"]
                ),
                "ground_truth_sha256": (
                    reference["sha256"]
                ),
                "result_path": str(
                    result_path
                ),
                "result_sha256": (
                    actual_result_checksum
                ),
                "annotation_count": (
                    annotation_count
                ),
                "matched_annotations": int(
                    matched_annotations
                ),
                "annotation_text_coverage": (
                    float(
                        annotation_coverage
                    )
                ),
                "reference_characters": len(
                    reference_compact
                ),
                "prediction_characters": len(
                    prediction_compact
                ),
                "character_distance": int(
                    character_distance
                ),
                "character_error_rate": (
                    float(
                        character_error_rate
                    )
                ),
                **token_metrics,
                "recognized_regions": int(
                    output_section.get(
                        "recognized_regions",
                        0,
                    )
                ),
                "mean_confidence": (
                    output_section.get(
                        "mean_confidence"
                    )
                ),
                "prediction_text": (
                    prediction_text
                ),
                "status": "VALID",
            }
        )

        print(
            f"{row.document_id} | "
            f"{row.variant:9s} | "
            f"regions="
            f"{output_section.get('recognized_regions', 0):2d} | "
            f"coverage="
            f"{annotation_coverage:.4f} | "
            f"token_recall="
            f"{token_metrics['token_recall']:.4f} | "
            f"CER={character_error_rate:.4f}"
        )

    except Exception as error:
        source_errors.append(
            {
                "document_id": (
                    row.document_id
                ),
                "template_id": (
                    row.template_id
                ),
                "variant": row.variant,
                "error_type": (
                    type(error).__name__
                ),
                "error": str(error)[:500],
            }
        )

        print(
            f"{row.document_id} | "
            f"{row.variant} | ERROR: "
            f"{type(error).__name__}: {error}"
        )


# ============================================================
# 7. DOCUMENT-LEVEL COMPARISON
# ============================================================

evaluation_table = pd.DataFrame(
    evaluation_records
)

if len(evaluation_table) != 12:
    raise RuntimeError(
        "Evaluation record tidak lengkap. "
        f"Ditemukan {len(evaluation_table)}/12."
    )

metric_columns = [
    "annotation_text_coverage",
    "token_precision",
    "token_recall",
    "token_f1",
    "character_error_rate",
    "word_error_rate",
    "recognized_regions",
    "mean_confidence",
]

degraded_table = (
    evaluation_table[
        evaluation_table["variant"]
        == "degraded"
    ][
        [
            "document_id",
            "template_id",
            "language",
            *metric_columns,
        ]
    ]
    .copy()
)

processed_table = (
    evaluation_table[
        evaluation_table["variant"]
        == "processed"
    ][
        [
            "document_id",
            *metric_columns,
        ]
    ]
    .copy()
)

degraded_table = degraded_table.rename(
    columns={
        column: f"degraded_{column}"
        for column in metric_columns
    }
)

processed_table = processed_table.rename(
    columns={
        column: f"processed_{column}"
        for column in metric_columns
    }
)

comparison_table = degraded_table.merge(
    processed_table,
    on="document_id",
    how="inner",
    validate="one_to_one",
)

comparison_table[
    "annotation_coverage_delta"
] = (
    comparison_table[
        "processed_annotation_text_coverage"
    ]
    - comparison_table[
        "degraded_annotation_text_coverage"
    ]
)

comparison_table[
    "token_recall_delta"
] = (
    comparison_table[
        "processed_token_recall"
    ]
    - comparison_table[
        "degraded_token_recall"
    ]
)

comparison_table[
    "token_f1_delta"
] = (
    comparison_table[
        "processed_token_f1"
    ]
    - comparison_table[
        "degraded_token_f1"
    ]
)

comparison_table[
    "cer_reduction"
] = (
    comparison_table[
        "degraded_character_error_rate"
    ]
    - comparison_table[
        "processed_character_error_rate"
    ]
)


def classify_document(row) -> str:
    meaningful_improvement = bool(
        row["annotation_coverage_delta"]
        >= MEANINGFUL_GAIN
        or row["token_recall_delta"]
        >= MEANINGFUL_GAIN
        or row["cer_reduction"]
        >= MEANINGFUL_GAIN
    )

    meaningful_worsening = bool(
        row["annotation_coverage_delta"]
        <= -MEANINGFUL_GAIN
        or row["token_recall_delta"]
        <= -MEANINGFUL_GAIN
        or row["cer_reduction"]
        <= -MEANINGFUL_GAIN
    )

    if (
        meaningful_improvement
        and not meaningful_worsening
    ):
        return "IMPROVED"

    if meaningful_worsening:
        return "WORSENED"

    return "NO_MEANINGFUL_CHANGE"


comparison_table[
    "preprocessing_effect"
] = comparison_table.apply(
    classify_document,
    axis=1,
)


# ============================================================
# 8. AGGREGATE METRICS
# ============================================================

aggregate_records = []

for variant in (
    "degraded",
    "processed",
):
    variant_table = evaluation_table[
        evaluation_table["variant"]
        == variant
    ]

    total_reference_characters = int(
        variant_table[
            "reference_characters"
        ].sum()
    )

    total_character_distance = int(
        variant_table[
            "character_distance"
        ].sum()
    )

    total_reference_tokens = int(
        variant_table[
            "reference_tokens"
        ].sum()
    )

    total_word_distance = int(
        variant_table[
            "word_distance"
        ].sum()
    )

    aggregate_records.append(
        {
            "variant": variant,
            "documents": len(
                variant_table
            ),
            "annotations": int(
                variant_table[
                    "annotation_count"
                ].sum()
            ),
            "matched_annotations": int(
                variant_table[
                    "matched_annotations"
                ].sum()
            ),
            "annotation_text_coverage": (
                variant_table[
                    "matched_annotations"
                ].sum()
                / variant_table[
                    "annotation_count"
                ].sum()
            ),
            "token_precision": float(
                variant_table[
                    "token_precision"
                ].mean()
            ),
            "token_recall": float(
                variant_table[
                    "token_recall"
                ].mean()
            ),
            "token_f1": float(
                variant_table[
                    "token_f1"
                ].mean()
            ),
            "micro_cer": (
                safe_error_rate(
                    total_character_distance,
                    total_reference_characters,
                )
            ),
            "micro_wer": (
                safe_error_rate(
                    total_word_distance,
                    total_reference_tokens,
                )
            ),
            "mean_recognized_regions": float(
                variant_table[
                    "recognized_regions"
                ].mean()
            ),
            "mean_confidence": float(
                variant_table[
                    "mean_confidence"
                ].mean()
            ),
        }
    )

aggregate_table = pd.DataFrame(
    aggregate_records
)

degraded_metrics = (
    aggregate_table[
        aggregate_table["variant"]
        == "degraded"
    ]
    .iloc[0]
)

processed_metrics = (
    aggregate_table[
        aggregate_table["variant"]
        == "processed"
    ]
    .iloc[0]
)

aggregate_annotation_gain = float(
    processed_metrics[
        "annotation_text_coverage"
    ]
    - degraded_metrics[
        "annotation_text_coverage"
    ]
)

aggregate_token_recall_gain = float(
    processed_metrics["token_recall"]
    - degraded_metrics["token_recall"]
)

aggregate_token_f1_gain = float(
    processed_metrics["token_f1"]
    - degraded_metrics["token_f1"]
)

aggregate_cer_reduction = float(
    degraded_metrics["micro_cer"]
    - processed_metrics["micro_cer"]
)

noninferior = bool(
    aggregate_annotation_gain
    >= -NONINFERIOR_TOLERANCE
    and aggregate_token_recall_gain
    >= -NONINFERIOR_TOLERANCE
    and aggregate_cer_reduction
    >= -NONINFERIOR_TOLERANCE
)

meaningful_improvement = bool(
    aggregate_annotation_gain
    >= MEANINGFUL_GAIN
    or aggregate_token_recall_gain
    >= MEANINGFUL_GAIN
    or aggregate_cer_reduction
    >= MEANINGFUL_GAIN
)

quality_floor_passed = bool(
    processed_metrics[
        "annotation_text_coverage"
    ]
    >= MINIMUM_PROCESSED_ANNOTATION_COVERAGE
    and processed_metrics[
        "token_recall"
    ]
    >= MINIMUM_PROCESSED_TOKEN_RECALL
)

worsened_documents = int(
    (
        comparison_table[
            "preprocessing_effect"
        ]
        == "WORSENED"
    ).sum()
)

preprocessing_accepted = bool(
    noninferior
    and meaningful_improvement
    and quality_floor_passed
    and worsened_documents <= 1
)

preprocessing_decision = (
    "ACCEPTED"
    if preprocessing_accepted
    else "REJECTED"
)


# ============================================================
# 9. TECHNICAL CONTROLS
# ============================================================

ground_truth_checksums_before = {
    cache_key: reference["sha256"]
    for cache_key, reference
    in reference_cache.items()
}

ground_truth_checksum_changes = []

for cache_key, checksum_before in (
    ground_truth_checksums_before.items()
):
    document_id, template_id = cache_key

    current_path = reference_cache[
        cache_key
    ]["path"]

    checksum_after = sha256_file(
        current_path
    )

    if checksum_before != checksum_after:
        ground_truth_checksum_changes.append(
            {
                "document_id": document_id,
                "template_id": template_id,
                "before": checksum_before,
                "after": checksum_after,
            }
        )

controls = [
    {
        "control": "ocr_run_status",
        "expected": "PASSED",
        "actual": run_manifest.get(
            "status"
        ),
    },
    {
        "control": "ocr_results",
        "expected": 12,
        "actual": len(ocr_summary),
    },
    {
        "control": "evaluation_records",
        "expected": 12,
        "actual": len(evaluation_table),
    },
    {
        "control": "ground_truth_documents",
        "expected": 6,
        "actual": len(reference_cache),
    },
    {
        "control": "document_comparisons",
        "expected": 6,
        "actual": len(comparison_table),
    },
    {
        "control": "variants",
        "expected": [
            "degraded",
            "processed",
        ],
        "actual": sorted(
            evaluation_table[
                "variant"
            ].unique().tolist()
        ),
    },
    {
        "control": "result_checksum_mismatches",
        "expected": 0,
        "actual": len(
            result_checksum_mismatches
        ),
    },
    {
        "control": "result_identity_mismatches",
        "expected": 0,
        "actual": len(
            result_identity_mismatches
        ),
    },
    {
        "control": "source_errors",
        "expected": 0,
        "actual": len(source_errors),
    },
    {
        "control": "ground_truth_checksum_changes",
        "expected": 0,
        "actual": len(
            ground_truth_checksum_changes
        ),
    },
    {
        "control": "validation_opened",
        "expected": 0,
        "actual": 0,
    },
    {
        "control": "test_opened",
        "expected": 0,
        "actual": 0,
    },
]

for control in controls:
    control["status"] = (
        "VALID"
        if control["expected"]
        == control["actual"]
        else "INVALID"
    )

control_table = pd.DataFrame(
    controls
)

display(control_table)

print("\nAGGREGATE OCR QUALITY")
display(aggregate_table)

print("\nDOCUMENT-LEVEL COMPARISON")
display(
    comparison_table[
        [
            "document_id",
            "template_id",
            "language",
            "degraded_annotation_text_coverage",
            "processed_annotation_text_coverage",
            "annotation_coverage_delta",
            "degraded_token_recall",
            "processed_token_recall",
            "token_recall_delta",
            "degraded_character_error_rate",
            "processed_character_error_rate",
            "cer_reduction",
            "preprocessing_effect",
        ]
    ]
)

invalid_controls = [
    control["control"]
    for control in controls
    if control["status"] != "VALID"
]

if invalid_controls:
    if source_errors:
        print("\nSOURCE ERRORS")
        display(pd.DataFrame(source_errors))

    if result_checksum_mismatches:
        print("\nRESULT CHECKSUM MISMATCHES")
        display(
            pd.DataFrame(
                result_checksum_mismatches
            )
        )

    if result_identity_mismatches:
        print("\nIDENTITY MISMATCHES")
        display(
            pd.DataFrame(
                result_identity_mismatches
            )
        )

    raise RuntimeError(
        "CELL 9D TECHNICAL EVALUATION FAILED. "
        f"Kontrol tidak valid: "
        f"{invalid_controls}"
    )


# ============================================================
# 10. SIMPAN EVALUATION ARTIFACT
# ============================================================

atomic_write_csv(
    EVALUATION_RECORDS_PATH,
    evaluation_table,
)

atomic_write_csv(
    DOCUMENT_COMPARISON_PATH,
    comparison_table,
)

evaluation_manifest = {
    "schema_version": "1.0.0",
    "status": "PASSED",
    "stage": (
        "PREPROCESSING_OCR_QUALITY_EVALUATION"
    ),
    "preprocessing_decision": (
        preprocessing_decision
    ),
    "decision_is_execution_error": False,
    "evaluation_scope": {
        "split": "development",
        "documents": 6,
        "templates": 6,
        "ocr_results": 12,
        "variants": [
            "degraded",
            "processed",
        ],
        "validation_opened": 0,
        "test_opened": 0,
    },
    "acceptance_policy": {
        "noninferior_tolerance": (
            NONINFERIOR_TOLERANCE
        ),
        "meaningful_gain": (
            MEANINGFUL_GAIN
        ),
        "minimum_processed_token_recall": (
            MINIMUM_PROCESSED_TOKEN_RECALL
        ),
        "minimum_processed_annotation_coverage": (
            MINIMUM_PROCESSED_ANNOTATION_COVERAGE
        ),
        "maximum_worsened_documents": 1,
    },
    "aggregate_metrics": (
        aggregate_table.to_dict(
            orient="records"
        )
    ),
    "decision_evidence": {
        "annotation_coverage_gain": (
            aggregate_annotation_gain
        ),
        "token_recall_gain": (
            aggregate_token_recall_gain
        ),
        "token_f1_gain": (
            aggregate_token_f1_gain
        ),
        "cer_reduction": (
            aggregate_cer_reduction
        ),
        "noninferior": noninferior,
        "meaningful_improvement": (
            meaningful_improvement
        ),
        "quality_floor_passed": (
            quality_floor_passed
        ),
        "worsened_documents": (
            worsened_documents
        ),
    },
    "artifacts": {
        "evaluation_records": str(
            EVALUATION_RECORDS_PATH
        ),
        "document_comparison": str(
            DOCUMENT_COMPARISON_PATH
        ),
        "source_run_manifest": str(
            RUN_MANIFEST_PATH
        ),
    },
    "integrity": {
        "result_checksum_mismatches": 0,
        "result_identity_mismatches": 0,
        "ground_truth_checksum_changes": 0,
        "dataset_modifications": 0,
    },
}

atomic_write_json(
    EVALUATION_MANIFEST_PATH,
    evaluation_manifest,
)

print()
print(f"Documents evaluated  : {len(comparison_table)}")
print(
    "Ground-truth annotations: "
    f"{int(degraded_metrics['annotations'])}"
)
print(
    "Degraded coverage   : "
    f"{degraded_metrics['annotation_text_coverage']:.6f}"
)
print(
    "Processed coverage  : "
    f"{processed_metrics['annotation_text_coverage']:.6f}"
)
print(
    "Degraded token recall: "
    f"{degraded_metrics['token_recall']:.6f}"
)
print(
    "Processed token recall: "
    f"{processed_metrics['token_recall']:.6f}"
)
print(
    "Degraded micro CER  : "
    f"{degraded_metrics['micro_cer']:.6f}"
)
print(
    "Processed micro CER : "
    f"{processed_metrics['micro_cer']:.6f}"
)
print(
    f"Worsened documents  : "
    f"{worsened_documents}"
)
print(
    f"Quality floor passed: "
    f"{quality_floor_passed}"
)
print(
    f"Preprocessing decision: "
    f"{preprocessing_decision}"
)
print(
    f"Evaluation records  : "
    f"{EVALUATION_RECORDS_PATH}"
)
print(
    f"Comparison table    : "
    f"{DOCUMENT_COMPARISON_PATH}"
)
print(
    f"Evaluation manifest : "
    f"{EVALUATION_MANIFEST_PATH}"
)
print(
    "Manifest SHA-256    : "
    f"{sha256_file(EVALUATION_MANIFEST_PATH)}"
)
print("Dataset modifications: 0")
print("Validation opened    : 0")
print("Test opened          : 0")
print()
print(
    "✅ CELL 9D PASSED — evaluasi kualitas "
    "preprocessing selesai. Keputusan eksperimen: "
    f"{preprocessing_decision}."
)

Mengevaluasi 12 hasil PaddleOCR terhadap ground truth development...

INV-SYN-000002 | degraded  | regions= 2 | coverage=0.0000 | token_recall=0.0000 | CER=0.9549
INV-SYN-000002 | processed | regions= 3 | coverage=0.0000 | token_recall=0.0085 | CER=0.9187
INV-SYN-000036 | degraded  | regions= 3 | coverage=0.0000 | token_recall=0.0000 | CER=0.9539
INV-SYN-000036 | processed | regions= 1 | coverage=0.0000 | token_recall=0.0000 | CER=0.9899
INV-SYN-000043 | degraded  | regions= 2 | coverage=0.0000 | token_recall=0.0168 | CER=0.9651
INV-SYN-000043 | processed | regions= 3 | coverage=0.0000 | token_recall=0.0000 | CER=0.9476
INV-SYN-000071 | degraded  | regions= 1 | coverage=0.0000 | token_recall=0.0000 | CER=0.9895
INV-SYN-000071 | processed | regions= 1 | coverage=0.0000 | token_recall=0.0000 | CER=0.9895
INV-SYN-000082 | degraded  | regions= 1 | coverage=0.0000 | token_recall=0.0000 | CER=0.9901
INV-SYN-000082 | processed | regions= 1 | coverage=0.0000 | token_recall=0.0000 | CER=0.9901


,control,expected,actual,status
0,ocr_run_status,PASSED,PASSED,VALID
1,ocr_results,12,12,VALID
2,evaluation_records,12,12,VALID
3,ground_truth_documents,6,6,VALID
4,document_comparisons,6,6,VALID
5,variants,"[degraded, processed]","[degraded, processed]",VALID
6,result_checksum_mismatches,0,0,VALID
7,result_identity_mismatches,0,0,VALID
8,source_errors,0,0,VALID
9,ground_truth_checksum_changes,0,0,VALID



AGGREGATE OCR QUALITY


,variant,documents,annotations,matched_annotations,annotation_text_coverage,token_precision,token_recall,token_f1,micro_cer,micro_wer,mean_recognized_regions,mean_confidence
0,degraded,6,250,0,0.0,0.083333,0.002801,0.005420,0.975696,0.997888,1.666667,0.904741
1,processed,6,250,0,0.0,0.060185,0.002338,0.004436,0.974058,0.997888,1.833333,0.891113



DOCUMENT-LEVEL COMPARISON


,document_id,template_id,language,degraded_annotation_text_coverage,processed_annotation_text_coverage,annotation_coverage_delta,degraded_token_recall,processed_token_recall,token_recall_delta,degraded_character_error_rate,processed_character_error_rate,cer_reduction,preprocessing_effect
0,INV-SYN-000002,TPL-01,id,0.0,0.0,0.0,0.000000,0.008475,0.008475,0.954853,0.918736,0.036117,NO_MEANINGFUL_CHANGE
1,INV-SYN-000036,TPL-02,en,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.953890,0.989914,-0.036023,NO_MEANINGFUL_CHANGE
2,INV-SYN-000043,TPL-03,id,0.0,0.0,0.0,0.016807,0.000000,-0.016807,0.965066,0.947598,0.017467,NO_MEANINGFUL_CHANGE
3,INV-SYN-000071,TPL-04,en,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.989489,0.989489,0.000000,NO_MEANINGFUL_CHANGE
4,INV-SYN-000082,TPL-05,id,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.990099,0.990099,0.000000,NO_MEANINGFUL_CHANGE
5,INV-SYN-000114,TPL-06,en,0.0,0.0,0.0,0.000000,0.005556,0.005556,0.989914,0.979827,0.010086,NO_MEANINGFUL_CHANGE



Documents evaluated  : 6
Ground-truth annotations: 250
Degraded coverage   : 0.000000
Processed coverage  : 0.000000
Degraded token recall: 0.002801
Processed token recall: 0.002338
Degraded micro CER  : 0.975696
Processed micro CER : 0.974058
Worsened documents  : 0
Quality floor passed: False
Preprocessing decision: REJECTED
Evaluation records  : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/preprocessing_ocr_robustness/manifests/preprocessing_ocr_evaluation_records.csv
Comparison table    : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/preprocessing_ocr_robustness/manifests/preprocessing_ocr_document_comparison.csv
Evaluation manifest : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/preprocessing_ocr_robustness/manifests/preprocessing_ocr_evaluation_manifest.json
Manifest SHA-256    : 30922d707f067e27ca8b5cd617705aefd9ab

**Cell 9E-A — Preprocessing Candidate V2**

In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import os
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


# ============================================================
# CELL 9E-A — PREPROCESSING CANDIDATE V2
#
# Perbaikan utama:
# 1. deskew;
# 2. denoise;
# 3. hitung ulang metrik;
# 4. normalisasi kontras;
# 5. hitung ulang ketajaman;
# 6. sharpening konservatif;
# 7. clean image harus tetap pixel-identical.
#
# Cell ini belum menjalankan OCR.
# ============================================================


# ============================================================
# 1. KONFIGURASI
# ============================================================

BUILD_ROOT = Path(
    "/content/drive/MyDrive/InvoiceFlow-AI-Data/interim/"
    "synthetic_v1_build/20260904T150025Z"
)

BENCHMARK_ROOT = BUILD_ROOT / "ocr_benchmark"

STRESS_MANIFEST_PATH = (
    BENCHMARK_ROOT
    / "manifests"
    / "preprocessing_stress_manifest.json"
)

REJECTED_EVALUATION_PATH = (
    BENCHMARK_ROOT
    / "preprocessing_ocr_robustness"
    / "manifests"
    / "preprocessing_ocr_evaluation_manifest.json"
)

CANDIDATE_ROOT = (
    BENCHMARK_ROOT
    / "preprocessing_candidate_v2"
)

CANDIDATE_IMAGE_ROOT = (
    CANDIDATE_ROOT
    / "processed"
)

CANDIDATE_RESULT_PATH = (
    CANDIDATE_ROOT
    / "preprocessing_candidate_v2_results.csv"
)

CANDIDATE_MANIFEST_PATH = (
    CANDIDATE_ROOT
    / "preprocessing_candidate_v2_manifest.json"
)

EXPECTED_DOCUMENTS = 6

# Tambahan engineering threshold khusus scan.
# Tidak menggantikan threshold hasil kalibrasi.
MINIMUM_SCAN_DYNAMIC_RANGE = 160.0
PERCENTILE_LOW = 1.0
PERCENTILE_HIGH = 99.0

CANDIDATE_IMAGE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 2. FILE HELPERS
# ============================================================

def sha256_file(file_path: Path) -> str:
    digest = hashlib.sha256()

    with file_path.open("rb") as file_handle:
        for chunk in iter(
            lambda: file_handle.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def atomic_write_text(
    file_path: Path,
    content: str,
) -> None:
    file_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = file_path.with_name(
        file_path.name + ".tmp"
    )

    temporary_path.write_text(
        content,
        encoding="utf-8",
    )

    os.replace(
        temporary_path,
        file_path,
    )


def atomic_write_json(
    file_path: Path,
    content: dict,
) -> None:
    atomic_write_text(
        file_path,
        json.dumps(
            content,
            indent=2,
            ensure_ascii=False,
        )
        + "\n",
    )


def atomic_write_csv(
    file_path: Path,
    table: pd.DataFrame,
) -> None:
    temporary_path = file_path.with_name(
        file_path.name + ".tmp"
    )

    table.to_csv(
        temporary_path,
        index=False,
    )

    os.replace(
        temporary_path,
        file_path,
    )


def atomic_write_png(
    file_path: Path,
    image: np.ndarray,
) -> None:
    file_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = file_path.with_name(
        file_path.stem + ".tmp.png"
    )

    if not cv2.imwrite(
        str(temporary_path),
        image,
    ):
        raise RuntimeError(
            f"Gagal menulis gambar: {temporary_path}"
        )

    os.replace(
        temporary_path,
        file_path,
    )


def read_grayscale(
    file_path: Path,
) -> np.ndarray:
    image = cv2.imread(
        str(file_path),
        cv2.IMREAD_GRAYSCALE,
    )

    if image is None:
        raise RuntimeError(
            f"Gagal membuka gambar: {file_path}"
        )

    return image


# ============================================================
# 3. THRESHOLD READER
# ============================================================

THRESHOLD_KEYS = {
    "minimum_contrast_std",
    "minimum_dynamic_range",
    "minimum_laplacian_variance",
    "minimum_brightness_mean",
    "maximum_brightness_mean",
    "maximum_noise_residual",
    "maximum_absolute_skew_degrees",
}


def locate_thresholds(value):
    if isinstance(value, dict):
        if THRESHOLD_KEYS.issubset(
            value.keys()
        ):
            return {
                key: float(value[key])
                for key in THRESHOLD_KEYS
            }

        for child in value.values():
            result = locate_thresholds(child)

            if result is not None:
                return result

    elif isinstance(value, list):
        for child in value:
            result = locate_thresholds(child)

            if result is not None:
                return result

    return None


# ============================================================
# 4. IMAGE METRICS
# ============================================================

def calculate_noise_residual(
    grayscale: np.ndarray,
) -> float:
    median_image = cv2.medianBlur(
        grayscale,
        3,
    )

    residual = (
        grayscale.astype(np.float32)
        - median_image.astype(np.float32)
    )

    gradient_x = cv2.Sobel(
        median_image,
        cv2.CV_32F,
        1,
        0,
        ksize=3,
    )

    gradient_y = cv2.Sobel(
        median_image,
        cv2.CV_32F,
        0,
        1,
        ksize=3,
    )

    gradient_magnitude = cv2.magnitude(
        gradient_x,
        gradient_y,
    )

    flat_mask = gradient_magnitude < 20.0
    residual_sample = residual[flat_mask]

    minimum_sample_size = max(
        1000,
        int(grayscale.size * 0.05),
    )

    if residual_sample.size < minimum_sample_size:
        residual_sample = residual.reshape(-1)

    return float(
        np.mean(
            np.abs(residual_sample)
        )
    )


def estimate_skew(
    grayscale: np.ndarray,
) -> float:
    reduced = grayscale

    maximum_dimension = max(
        grayscale.shape
    )

    if maximum_dimension > 1800:
        scale = 1800.0 / maximum_dimension

        reduced = cv2.resize(
            grayscale,
            None,
            fx=scale,
            fy=scale,
            interpolation=cv2.INTER_AREA,
        )

    reduced = cv2.medianBlur(
        reduced,
        3,
    )

    binary = cv2.threshold(
        reduced,
        0,
        255,
        cv2.THRESH_BINARY_INV
        + cv2.THRESH_OTSU,
    )[1]

    kernel_width = max(
        25,
        reduced.shape[1] // 35,
    )

    horizontal_kernel = (
        cv2.getStructuringElement(
            cv2.MORPH_RECT,
            (kernel_width, 1),
        )
    )

    horizontal_lines = cv2.morphologyEx(
        binary,
        cv2.MORPH_OPEN,
        horizontal_kernel,
    )

    lines = cv2.HoughLinesP(
        horizontal_lines,
        1,
        np.pi / 1800.0,
        threshold=max(
            40,
            reduced.shape[1] // 12,
        ),
        minLineLength=max(
            80,
            reduced.shape[1] // 7,
        ),
        maxLineGap=max(
            10,
            reduced.shape[1] // 100,
        ),
    )

    if lines is None:
        return 0.0

    angles = []
    weights = []

    for x1, y1, x2, y2 in lines[:, 0, :]:
        delta_x = int(x2) - int(x1)
        delta_y = int(y2) - int(y1)

        if delta_x == 0:
            continue

        angle = math.degrees(
            math.atan2(
                delta_y,
                delta_x,
            )
        )

        if abs(angle) <= 10.0:
            angles.append(angle)
            weights.append(
                math.hypot(
                    delta_x,
                    delta_y,
                )
            )

    if len(angles) < 2:
        return 0.0

    order = np.argsort(angles)

    sorted_angles = np.asarray(
        angles
    )[order]

    sorted_weights = np.asarray(
        weights
    )[order]

    cumulative_weights = np.cumsum(
        sorted_weights
    )

    middle_index = np.searchsorted(
        cumulative_weights,
        cumulative_weights[-1] / 2.0,
    )

    return float(
        sorted_angles[middle_index]
    )


def calculate_metrics(
    grayscale: np.ndarray,
) -> dict:
    percentile_1, percentile_99 = (
        np.percentile(
            grayscale,
            [1, 99],
        )
    )

    return {
        "brightness_mean": float(
            grayscale.mean()
        ),
        "contrast_std": float(
            grayscale.std()
        ),
        "dynamic_range": float(
            percentile_99 - percentile_1
        ),
        "laplacian_variance": float(
            cv2.Laplacian(
                grayscale,
                cv2.CV_64F,
            ).var()
        ),
        "noise_residual": float(
            calculate_noise_residual(
                grayscale
            )
        ),
        "skew_degrees": float(
            estimate_skew(
                grayscale
            )
        ),
    }


def detect_flags(
    metrics: dict,
    thresholds: dict,
) -> list[str]:
    flags = []

    if (
        metrics["contrast_std"]
        < thresholds[
            "minimum_contrast_std"
        ]
    ):
        flags.append("LOW_CONTRAST")

    if (
        metrics["dynamic_range"]
        < thresholds[
            "minimum_dynamic_range"
        ]
    ):
        flags.append("LOW_DYNAMIC_RANGE")

    if (
        metrics["laplacian_variance"]
        < thresholds[
            "minimum_laplacian_variance"
        ]
    ):
        flags.append("BLUR")

    if (
        metrics["brightness_mean"]
        < thresholds[
            "minimum_brightness_mean"
        ]
    ):
        flags.append("TOO_DARK")

    if (
        metrics["brightness_mean"]
        > thresholds[
            "maximum_brightness_mean"
        ]
    ):
        flags.append("TOO_BRIGHT")

    if (
        metrics["noise_residual"]
        > thresholds[
            "maximum_noise_residual"
        ]
    ):
        flags.append("NOISE")

    if (
        abs(metrics["skew_degrees"])
        > thresholds[
            "maximum_absolute_skew_degrees"
        ]
    ):
        flags.append("SKEW")

    return flags


# ============================================================
# 5. PREPROCESSING OPERATIONS
# ============================================================

def rotate_keep_size(
    image: np.ndarray,
    degrees: float,
) -> np.ndarray:
    height, width = image.shape[:2]

    rotation_matrix = cv2.getRotationMatrix2D(
        (width / 2.0, height / 2.0),
        degrees,
        1.0,
    )

    border_value = int(
        np.median(image)
    )

    return cv2.warpAffine(
        image,
        rotation_matrix,
        (width, height),
        flags=cv2.INTER_CUBIC,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=border_value,
    )


def select_best_deskew(
    image: np.ndarray,
    detected_angle: float,
) -> np.ndarray:
    candidates = [
        image,
        rotate_keep_size(
            image,
            detected_angle,
        ),
        rotate_keep_size(
            image,
            -detected_angle,
        ),
    ]

    return min(
        candidates,
        key=lambda candidate: abs(
            estimate_skew(candidate)
        ),
    )


def select_denoised_image(
    image: np.ndarray,
    thresholds: dict,
) -> tuple[np.ndarray, int]:
    target_residual = (
        thresholds[
            "maximum_noise_residual"
        ]
        * 0.85
    )

    denoise_strengths = (
        7,
        10,
        13,
        16,
        20,
    )

    best_image = image
    best_strength = 0
    best_residual = (
        calculate_noise_residual(image)
    )

    for strength in denoise_strengths:
        candidate = cv2.fastNlMeansDenoising(
            image,
            None,
            h=strength,
            templateWindowSize=7,
            searchWindowSize=21,
        )

        residual = calculate_noise_residual(
            candidate
        )

        if residual < best_residual:
            best_image = candidate
            best_strength = strength
            best_residual = residual

        if residual <= target_residual:
            return candidate, strength

    return best_image, best_strength


def percentile_contrast_normalization(
    image: np.ndarray,
) -> np.ndarray:
    low_value, high_value = np.percentile(
        image,
        [
            PERCENTILE_LOW,
            PERCENTILE_HIGH,
        ],
    )

    if high_value <= low_value + 1.0:
        return image.copy()

    normalized = (
        (
            image.astype(np.float32)
            - low_value
        )
        * (240.0 / (high_value - low_value))
        + 8.0
    )

    return np.clip(
        normalized,
        0,
        255,
    ).astype(np.uint8)


def select_sharpened_image(
    image: np.ndarray,
    thresholds: dict,
) -> tuple[np.ndarray, float]:
    baseline_metrics = calculate_metrics(
        image
    )

    if (
        baseline_metrics[
            "laplacian_variance"
        ]
        >= thresholds[
            "minimum_laplacian_variance"
        ]
    ):
        return image, 0.0

    blur = cv2.GaussianBlur(
        image,
        (0, 0),
        1.0,
    )

    candidate_amounts = (
        0.25,
        0.40,
        0.55,
        0.70,
    )

    best_image = image
    best_amount = 0.0
    best_sharpness = baseline_metrics[
        "laplacian_variance"
    ]

    for amount in candidate_amounts:
        candidate = cv2.addWeighted(
            image,
            1.0 + amount,
            blur,
            -amount,
            0,
        )

        metrics = calculate_metrics(
            candidate
        )

        noise_valid = (
            metrics["noise_residual"]
            <= thresholds[
                "maximum_noise_residual"
            ]
        )

        if (
            noise_valid
            and metrics[
                "laplacian_variance"
            ]
            > best_sharpness
        ):
            best_image = candidate
            best_amount = amount
            best_sharpness = metrics[
                "laplacian_variance"
            ]

        if (
            noise_valid
            and metrics[
                "laplacian_variance"
            ]
            >= thresholds[
                "minimum_laplacian_variance"
            ]
        ):
            return candidate, amount

    return best_image, best_amount


# ============================================================
# 6. CANDIDATE V2 PIPELINE
# ============================================================

def preprocess_candidate_v2(
    image: np.ndarray,
    thresholds: dict,
) -> tuple[np.ndarray, dict]:
    working = image.copy()

    initial_metrics = calculate_metrics(
        working
    )

    initial_flags = detect_flags(
        initial_metrics,
        thresholds,
    )

    applied_steps = []

    # Clean image tidak boleh dimodifikasi.
    if not initial_flags:
        return (
            working,
            {
                "initial_flags": [],
                "applied_steps": [],
                "initial_metrics": (
                    initial_metrics
                ),
                "after_deskew_metrics": (
                    initial_metrics
                ),
                "after_denoise_metrics": (
                    initial_metrics
                ),
                "after_contrast_metrics": (
                    initial_metrics
                ),
                "final_metrics": (
                    initial_metrics
                ),
            },
        )

    # Tahap 1: perbaiki geometri.
    if "SKEW" in initial_flags:
        working = select_best_deskew(
            working,
            initial_metrics[
                "skew_degrees"
            ],
        )

        applied_steps.append("DESKEW")

    after_deskew_metrics = (
        calculate_metrics(working)
    )

    # Tahap 2: hilangkan noise.
    if (
        "NOISE" in initial_flags
        or after_deskew_metrics[
            "noise_residual"
        ]
        > thresholds[
            "maximum_noise_residual"
        ]
    ):
        working, denoise_strength = (
            select_denoised_image(
                working,
                thresholds,
            )
        )

        applied_steps.append(
            f"DENOISE_H{denoise_strength}"
        )

    # Metrik wajib dihitung ulang setelah denoise.
    after_denoise_metrics = (
        calculate_metrics(working)
    )

    after_denoise_flags = detect_flags(
        after_denoise_metrics,
        thresholds,
    )

    # Tahap 3: tone/contrast recovery.
    scan_degradation_detected = bool(
        set(initial_flags)
        & {
            "NOISE",
            "LOW_CONTRAST",
            "LOW_DYNAMIC_RANGE",
            "TOO_DARK",
            "TOO_BRIGHT",
            "BLUR",
        }
    )

    contrast_recovery_required = bool(
        "LOW_CONTRAST"
        in after_denoise_flags
        or "LOW_DYNAMIC_RANGE"
        in after_denoise_flags
        or "TOO_DARK"
        in after_denoise_flags
        or "TOO_BRIGHT"
        in after_denoise_flags
        or (
            scan_degradation_detected
            and after_denoise_metrics[
                "dynamic_range"
            ]
            < MINIMUM_SCAN_DYNAMIC_RANGE
        )
    )

    if contrast_recovery_required:
        working = (
            percentile_contrast_normalization(
                working
            )
        )

        applied_steps.append(
            "PERCENTILE_CONTRAST"
        )

    after_contrast_metrics = (
        calculate_metrics(working)
    )

    # Tahap 4: sharpening setelah noise/contrast selesai.
    if (
        "BLUR" in initial_flags
        or after_contrast_metrics[
            "laplacian_variance"
        ]
        < thresholds[
            "minimum_laplacian_variance"
        ]
    ):
        working, sharpen_amount = (
            select_sharpened_image(
                working,
                thresholds,
            )
        )

        if sharpen_amount > 0:
            applied_steps.append(
                f"UNSHARP_{sharpen_amount:.2f}"
            )

    final_metrics = calculate_metrics(
        working
    )

    return (
        working,
        {
            "initial_flags": initial_flags,
            "applied_steps": applied_steps,
            "initial_metrics": (
                initial_metrics
            ),
            "after_deskew_metrics": (
                after_deskew_metrics
            ),
            "after_denoise_metrics": (
                after_denoise_metrics
            ),
            "after_contrast_metrics": (
                after_contrast_metrics
            ),
            "final_metrics": (
                final_metrics
            ),
        },
    )


# ============================================================
# 7. PREFLIGHT
# ============================================================

required_paths = [
    BUILD_ROOT,
    STRESS_MANIFEST_PATH,
    REJECTED_EVALUATION_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not path.exists()
]

if missing_paths:
    raise FileNotFoundError(
        "Checkpoint belum lengkap:\n"
        + "\n".join(missing_paths)
    )

stress_manifest = json.loads(
    STRESS_MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)

rejected_evaluation = json.loads(
    REJECTED_EVALUATION_PATH.read_text(
        encoding="utf-8"
    )
)

if stress_manifest.get("status") != "PASSED":
    raise RuntimeError(
        "Stress manifest belum PASSED."
    )

if (
    rejected_evaluation.get(
        "preprocessing_decision"
    )
    != "REJECTED"
):
    raise RuntimeError(
        "Eksperimen sebelumnya belum tercatat "
        "sebagai REJECTED."
    )

thresholds = locate_thresholds(
    stress_manifest
)

if thresholds is None:
    raise RuntimeError(
        "Threshold kalibrasi tidak ditemukan."
    )

stress_result_path = Path(
    stress_manifest[
        "results"
    ]["result_path"]
)

stress_table = pd.read_csv(
    stress_result_path
)

combined_table = (
    stress_table[
        (
            stress_table["scenario"]
            == "combined_scan"
        )
        & (
            stress_table["status"]
            == "VALID"
        )
    ]
    .copy()
    .sort_values(
        [
            "template_id",
            "document_id",
        ]
    )
    .reset_index(drop=True)
)

if len(combined_table) != EXPECTED_DOCUMENTS:
    raise RuntimeError(
        "Combined-scan documents seharusnya 6, "
        f"ditemukan {len(combined_table)}."
    )


# ============================================================
# 8. CLEAN INVARIANCE DAN CANDIDATE GENERATION
# ============================================================

records = []
errors = []

source_checksums_before = {}
degraded_checksums_before = {}

print(
    "Membangun preprocessing candidate V2...\n"
)

for sequence_number, row in enumerate(
    combined_table.itertuples(index=False),
    start=1,
):
    try:
        source_path = Path(row.source_path)
        degraded_path = Path(
            row.degraded_path
        )

        if not source_path.is_file():
            raise FileNotFoundError(
                f"Source hilang: {source_path}"
            )

        if not degraded_path.is_file():
            raise FileNotFoundError(
                f"Degraded input hilang: "
                f"{degraded_path}"
            )

        source_checksum = sha256_file(
            source_path
        )

        degraded_checksum = sha256_file(
            degraded_path
        )

        if (
            degraded_checksum
            != row.degraded_sha256
        ):
            raise RuntimeError(
                "Checksum degraded input tidak cocok."
            )

        source_checksums_before[
            row.document_id
        ] = source_checksum

        degraded_checksums_before[
            row.document_id
        ] = degraded_checksum

        clean_image = read_grayscale(
            source_path
        )

        degraded_image = read_grayscale(
            degraded_path
        )

        if (
            clean_image.shape
            != degraded_image.shape
        ):
            raise RuntimeError(
                "Dimensi clean dan degraded berbeda."
            )

        # Clean image wajib tidak berubah.
        (
            processed_clean,
            clean_decision,
        ) = preprocess_candidate_v2(
            clean_image,
            thresholds,
        )

        clean_pixel_identical = bool(
            np.array_equal(
                clean_image,
                processed_clean,
            )
        )

        clean_invariance_valid = bool(
            clean_pixel_identical
            and not clean_decision[
                "initial_flags"
            ]
            and not clean_decision[
                "applied_steps"
            ]
        )

        # Jalankan candidate pada combined scan.
        (
            candidate_image,
            decision,
        ) = preprocess_candidate_v2(
            degraded_image,
            thresholds,
        )

        candidate_path = (
            CANDIDATE_IMAGE_ROOT
            / row.template_id
            / (
                f"{row.document_id}"
                "_combined_scan_candidate_v2.png"
            )
        )

        atomic_write_png(
            candidate_path,
            candidate_image,
        )

        written_candidate = read_grayscale(
            candidate_path
        )

        pixel_write_valid = bool(
            np.array_equal(
                candidate_image,
                written_candidate,
            )
        )

        final_metrics = decision[
            "final_metrics"
        ]

        final_flags = detect_flags(
            final_metrics,
            thresholds,
        )

        dimensions_preserved = bool(
            candidate_image.shape
            == degraded_image.shape
        )

        candidate_changed = bool(
            not np.array_equal(
                degraded_image,
                candidate_image,
            )
        )

        candidate_nonblank = bool(
            candidate_image.std() > 1.0
            and candidate_image.min()
            < candidate_image.max()
        )

        geometry_valid = bool(
            abs(
                final_metrics[
                    "skew_degrees"
                ]
            )
            <= thresholds[
                "maximum_absolute_skew_degrees"
            ]
        )

        noise_valid = bool(
            final_metrics[
                "noise_residual"
            ]
            <= thresholds[
                "maximum_noise_residual"
            ]
        )

        contrast_valid = bool(
            final_metrics["contrast_std"]
            >= thresholds[
                "minimum_contrast_std"
            ]
            and final_metrics[
                "dynamic_range"
            ]
            >= thresholds[
                "minimum_dynamic_range"
            ]
        )

        brightness_valid = bool(
            thresholds[
                "minimum_brightness_mean"
            ]
            - 5.0
            <= final_metrics[
                "brightness_mean"
            ]
            <= thresholds[
                "maximum_brightness_mean"
            ]
            + 3.0
        )

        sharpness_valid = bool(
            final_metrics[
                "laplacian_variance"
            ]
            >= thresholds[
                "minimum_laplacian_variance"
            ]
        )

        candidate_valid = bool(
            clean_invariance_valid
            and pixel_write_valid
            and dimensions_preserved
            and candidate_changed
            and candidate_nonblank
            and geometry_valid
            and noise_valid
            and contrast_valid
            and brightness_valid
            and sharpness_valid
        )

        record = {
            "sequence_number": (
                sequence_number
            ),
            "document_id": row.document_id,
            "template_id": row.template_id,
            "language": row.language,
            "scenario": "combined_scan",
            "source_path": str(source_path),
            "degraded_path": str(
                degraded_path
            ),
            "candidate_path": str(
                candidate_path
            ),
            "source_sha256": (
                source_checksum
            ),
            "degraded_sha256": (
                degraded_checksum
            ),
            "candidate_sha256": sha256_file(
                candidate_path
            ),
            "clean_pixel_identical": (
                clean_pixel_identical
            ),
            "clean_invariance_valid": (
                clean_invariance_valid
            ),
            "initial_flags": "|".join(
                decision["initial_flags"]
            ),
            "applied_steps": "|".join(
                decision["applied_steps"]
            ),
            "final_flags": "|".join(
                final_flags
            ),
            "dimensions_preserved": (
                dimensions_preserved
            ),
            "candidate_changed": (
                candidate_changed
            ),
            "candidate_nonblank": (
                candidate_nonblank
            ),
            "geometry_valid": (
                geometry_valid
            ),
            "noise_valid": noise_valid,
            "contrast_valid": (
                contrast_valid
            ),
            "brightness_valid": (
                brightness_valid
            ),
            "sharpness_valid": (
                sharpness_valid
            ),
            "status": (
                "VALID"
                if candidate_valid
                else "INVALID"
            ),
        }

        for metric_name, metric_value in (
            decision[
                "initial_metrics"
            ].items()
        ):
            record[
                f"before_{metric_name}"
            ] = metric_value

        for metric_name, metric_value in (
            final_metrics.items()
        ):
            record[
                f"after_{metric_name}"
            ] = metric_value

        records.append(record)

        print(
            f"[{sequence_number}/6] "
            f"{row.document_id} | "
            f"flags={decision['initial_flags']} | "
            f"steps={decision['applied_steps']} | "
            f"final_flags={final_flags} | "
            f"status="
            f"{'VALID' if candidate_valid else 'INVALID'}"
        )

    except Exception as error:
        errors.append(
            {
                "sequence_number": (
                    sequence_number
                ),
                "document_id": row.document_id,
                "template_id": row.template_id,
                "error_type": (
                    type(error).__name__
                ),
                "error": str(error)[:500],
            }
        )

        print(
            f"[{sequence_number}/6] "
            f"{row.document_id} | ERROR: "
            f"{type(error).__name__}: {error}"
        )


# ============================================================
# 9. INTEGRITY DAN CONTROLS
# ============================================================

candidate_table = pd.DataFrame(
    records
)

source_checksum_changes = []
degraded_checksum_changes = []

for row in combined_table.itertuples(
    index=False
):
    source_path = Path(row.source_path)
    degraded_path = Path(row.degraded_path)

    if (
        row.document_id
        in source_checksums_before
        and sha256_file(source_path)
        != source_checksums_before[
            row.document_id
        ]
    ):
        source_checksum_changes.append(
            row.document_id
        )

    if (
        row.document_id
        in degraded_checksums_before
        and sha256_file(degraded_path)
        != degraded_checksums_before[
            row.document_id
        ]
    ):
        degraded_checksum_changes.append(
            row.document_id
        )

candidate_files = list(
    CANDIDATE_IMAGE_ROOT.rglob("*.png")
)

valid_candidates = (
    int(
        (
            candidate_table["status"]
            == "VALID"
        ).sum()
    )
    if not candidate_table.empty
    else 0
)

controls = [
    {
        "control": "previous_decision",
        "expected": "REJECTED",
        "actual": (
            rejected_evaluation.get(
                "preprocessing_decision"
            )
        ),
    },
    {
        "control": "candidate_documents",
        "expected": 6,
        "actual": len(candidate_table),
    },
    {
        "control": "template_coverage",
        "expected": 6,
        "actual": (
            candidate_table[
                "template_id"
            ].nunique()
            if not candidate_table.empty
            else 0
        ),
    },
    {
        "control": "clean_images_preserved",
        "expected": 6,
        "actual": (
            int(
                candidate_table[
                    "clean_invariance_valid"
                ].sum()
            )
            if not candidate_table.empty
            else 0
        ),
    },
    {
        "control": "valid_candidates",
        "expected": 6,
        "actual": valid_candidates,
    },
    {
        "control": "candidate_files",
        "expected": 6,
        "actual": len(candidate_files),
    },
    {
        "control": "unique_candidate_paths",
        "expected": 6,
        "actual": (
            candidate_table[
                "candidate_path"
            ].nunique()
            if not candidate_table.empty
            else 0
        ),
    },
    {
        "control": "processing_errors",
        "expected": 0,
        "actual": len(errors),
    },
    {
        "control": "source_checksum_changes",
        "expected": 0,
        "actual": len(
            source_checksum_changes
        ),
    },
    {
        "control": "degraded_checksum_changes",
        "expected": 0,
        "actual": len(
            degraded_checksum_changes
        ),
    },
    {
        "control": "validation_opened",
        "expected": 0,
        "actual": 0,
    },
    {
        "control": "test_opened",
        "expected": 0,
        "actual": 0,
    },
]

for control in controls:
    control["status"] = (
        "VALID"
        if control["expected"]
        == control["actual"]
        else "INVALID"
    )

control_table = pd.DataFrame(
    controls
)

display(control_table)

print("\nCANDIDATE V2 METRICS")

display(
    candidate_table[
        [
            "document_id",
            "template_id",
            "language",
            "initial_flags",
            "applied_steps",
            "before_dynamic_range",
            "after_dynamic_range",
            "before_noise_residual",
            "after_noise_residual",
            "before_skew_degrees",
            "after_skew_degrees",
            "before_laplacian_variance",
            "after_laplacian_variance",
            "status",
        ]
    ]
)


# ============================================================
# 10. VISUAL CONTACT SHEET
# ============================================================

if len(candidate_table) == 6:
    figure, axes = plt.subplots(
        nrows=6,
        ncols=3,
        figsize=(15, 32),
    )

    for row_index, row in enumerate(
        candidate_table.itertuples(
            index=False
        )
    ):
        clean_image = read_grayscale(
            Path(row.source_path)
        )

        degraded_image = read_grayscale(
            Path(row.degraded_path)
        )

        candidate_image = read_grayscale(
            Path(row.candidate_path)
        )

        images = [
            clean_image,
            degraded_image,
            candidate_image,
        ]

        titles = [
            (
                f"{row.document_id}\n"
                "CLEAN REFERENCE"
            ),
            "COMBINED SCAN — DEGRADED",
            "CANDIDATE V2",
        ]

        for column_index, (
            image,
            title,
        ) in enumerate(
            zip(images, titles)
        ):
            axes[
                row_index,
                column_index,
            ].imshow(
                image,
                cmap="gray",
                vmin=0,
                vmax=255,
            )

            axes[
                row_index,
                column_index,
            ].set_title(
                title,
                fontsize=10,
            )

            axes[
                row_index,
                column_index,
            ].axis("off")

    plt.tight_layout()
    plt.show()


# ============================================================
# 11. FINALIZE
# ============================================================

invalid_controls = [
    control["control"]
    for control in controls
    if control["status"] != "VALID"
]

if invalid_controls:
    if errors:
        print("\nPROCESSING ERRORS")
        display(pd.DataFrame(errors))

    print("\nINVALID CANDIDATES")
    display(
        candidate_table[
            candidate_table["status"]
            != "VALID"
        ]
    )

    raise RuntimeError(
        "CELL 9E-A FAILED. "
        f"Kontrol tidak valid: "
        f"{invalid_controls}"
    )

atomic_write_csv(
    CANDIDATE_RESULT_PATH,
    candidate_table,
)

candidate_manifest = {
    "schema_version": "2.0.0",
    "status": "READY_FOR_OCR",
    "candidate": (
        "ADAPTIVE_PREPROCESSING_V2"
    ),
    "previous_candidate_decision": (
        "REJECTED"
    ),
    "pipeline_order": [
        "quality_detection",
        "deskew",
        "metric_recalculation",
        "denoise",
        "metric_recalculation",
        "contrast_recovery",
        "metric_recalculation",
        "conservative_sharpening",
        "final_validation",
    ],
    "configuration": {
        "thresholds": thresholds,
        "minimum_scan_dynamic_range": (
            MINIMUM_SCAN_DYNAMIC_RANGE
        ),
        "contrast_percentiles": [
            PERCENTILE_LOW,
            PERCENTILE_HIGH,
        ],
    },
    "results": {
        "documents": 6,
        "templates": 6,
        "valid_candidates": 6,
        "invalid_candidates": 0,
        "processing_errors": 0,
        "candidate_result_path": str(
            CANDIDATE_RESULT_PATH
        ),
        "candidate_image_root": str(
            CANDIDATE_IMAGE_ROOT
        ),
    },
    "integrity": {
        "clean_images_preserved": 6,
        "source_checksum_changes": 0,
        "degraded_checksum_changes": 0,
        "dataset_modifications": 0,
        "validation_opened": 0,
        "test_opened": 0,
    },
    "next_stage": {
        "cell": "9E-B",
        "action": (
            "PADDLEOCR_CANDIDATE_V2_EXECUTION"
        ),
        "status": "PENDING",
    },
}

atomic_write_json(
    CANDIDATE_MANIFEST_PATH,
    candidate_manifest,
)

print()
print(
    f"Candidate root       : "
    f"{CANDIDATE_ROOT}"
)
print(
    f"Candidate images     : "
    f"{CANDIDATE_IMAGE_ROOT}"
)
print(
    f"Candidate results    : "
    f"{CANDIDATE_RESULT_PATH}"
)
print(
    f"Candidate manifest   : "
    f"{CANDIDATE_MANIFEST_PATH}"
)
print(
    f"Manifest SHA-256     : "
    f"{sha256_file(CANDIDATE_MANIFEST_PATH)}"
)
print("Candidate documents  : 6")
print("Valid candidates     : 6")
print("Clean images preserved: 6/6")
print("OCR executions       : 0")
print("Dataset modifications: 0")
print("Validation opened    : 0")
print("Test opened          : 0")
print("Candidate status     : READY_FOR_OCR")
print()
print(
    "✅ CELL 9E-A PASSED — preprocessing "
    "candidate V2 berhasil dibuat dan lolos "
    "validasi teknis. Lanjutkan ke Cell 9E-B "
    "untuk evaluasi PaddleOCR."
)

Output hidden; open in https://colab.research.google.com to view.

**Cell 9E-A2 — Candidate V2.1**

In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import os
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


# ============================================================
# CELL 9E-A2 — PREPROCESSING CANDIDATE V2.1
#
# Perbaikan:
# - tidak memakai percentile stretching;
# - deskew dilakukan terlebih dahulu;
# - denoise dipilih secara adaptif;
# - Otsu binarization digunakan untuk memulihkan
#   pemisahan teks dan latar;
# - kandidat dipilih dengan quality gate;
# - clean image wajib tetap pixel-identical;
# - belum menjalankan OCR.
# ============================================================


# ============================================================
# 1. KONFIGURASI
# ============================================================

BUILD_ROOT = Path(
    "/content/drive/MyDrive/InvoiceFlow-AI-Data/interim/"
    "synthetic_v1_build/20260904T150025Z"
)

BENCHMARK_ROOT = BUILD_ROOT / "ocr_benchmark"

STRESS_MANIFEST_PATH = (
    BENCHMARK_ROOT
    / "manifests"
    / "preprocessing_stress_manifest.json"
)

REJECTED_EVALUATION_PATH = (
    BENCHMARK_ROOT
    / "preprocessing_ocr_robustness"
    / "manifests"
    / "preprocessing_ocr_evaluation_manifest.json"
)

RUN_ID = "candidate_v2_1"

CANDIDATE_ROOT = (
    BENCHMARK_ROOT
    / "preprocessing_candidate_v2_1"
)

CANDIDATE_IMAGE_ROOT = (
    CANDIDATE_ROOT
    / "processed"
)

CANDIDATE_RESULT_PATH = (
    CANDIDATE_ROOT
    / "preprocessing_candidate_v2_1_results.csv"
)

CANDIDATE_MANIFEST_PATH = (
    CANDIDATE_ROOT
    / "preprocessing_candidate_v2_1_manifest.json"
)

EXPECTED_DOCUMENTS = 6

MINIMUM_FOREGROUND_RATIO = 0.01
MAXIMUM_FOREGROUND_RATIO = 0.40

DENOISE_STRENGTHS = (
    9,
    12,
    15,
    18,
    21,
)

CANDIDATE_IMAGE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 2. FILE HELPERS
# ============================================================

def sha256_file(file_path: Path) -> str:
    digest = hashlib.sha256()

    with file_path.open("rb") as file_handle:
        for chunk in iter(
            lambda: file_handle.read(
                1024 * 1024
            ),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def atomic_write_text(
    file_path: Path,
    content: str,
) -> None:
    file_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = file_path.with_name(
        file_path.name + ".tmp"
    )

    temporary_path.write_text(
        content,
        encoding="utf-8",
    )

    os.replace(
        temporary_path,
        file_path,
    )


def atomic_write_json(
    file_path: Path,
    content: dict,
) -> None:
    atomic_write_text(
        file_path,
        json.dumps(
            content,
            indent=2,
            ensure_ascii=False,
        )
        + "\n",
    )


def atomic_write_csv(
    file_path: Path,
    table: pd.DataFrame,
) -> None:
    temporary_path = file_path.with_name(
        file_path.name + ".tmp"
    )

    table.to_csv(
        temporary_path,
        index=False,
    )

    os.replace(
        temporary_path,
        file_path,
    )


def atomic_write_png(
    file_path: Path,
    image: np.ndarray,
) -> None:
    file_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = file_path.with_name(
        file_path.stem + ".tmp.png"
    )

    if not cv2.imwrite(
        str(temporary_path),
        image,
    ):
        raise RuntimeError(
            f"Gagal menulis PNG: {temporary_path}"
        )

    os.replace(
        temporary_path,
        file_path,
    )


def read_grayscale(
    file_path: Path,
) -> np.ndarray:
    image = cv2.imread(
        str(file_path),
        cv2.IMREAD_GRAYSCALE,
    )

    if image is None:
        raise RuntimeError(
            f"Gagal membuka gambar: {file_path}"
        )

    return image


# ============================================================
# 3. THRESHOLD READER
# ============================================================

THRESHOLD_KEYS = {
    "minimum_contrast_std",
    "minimum_dynamic_range",
    "minimum_laplacian_variance",
    "minimum_brightness_mean",
    "maximum_brightness_mean",
    "maximum_noise_residual",
    "maximum_absolute_skew_degrees",
}


def locate_thresholds(value):
    if isinstance(value, dict):
        if THRESHOLD_KEYS.issubset(
            value.keys()
        ):
            return {
                key: float(value[key])
                for key in THRESHOLD_KEYS
            }

        for child in value.values():
            result = locate_thresholds(child)

            if result is not None:
                return result

    elif isinstance(value, list):
        for child in value:
            result = locate_thresholds(child)

            if result is not None:
                return result

    return None


# ============================================================
# 4. IMAGE METRICS
# ============================================================

def calculate_noise_residual(
    grayscale: np.ndarray,
) -> float:
    median_image = cv2.medianBlur(
        grayscale,
        3,
    )

    residual = (
        grayscale.astype(np.float32)
        - median_image.astype(np.float32)
    )

    gradient_x = cv2.Sobel(
        median_image,
        cv2.CV_32F,
        1,
        0,
        ksize=3,
    )

    gradient_y = cv2.Sobel(
        median_image,
        cv2.CV_32F,
        0,
        1,
        ksize=3,
    )

    gradient_magnitude = cv2.magnitude(
        gradient_x,
        gradient_y,
    )

    flat_mask = gradient_magnitude < 20.0
    residual_sample = residual[flat_mask]

    minimum_sample_size = max(
        1000,
        int(grayscale.size * 0.05),
    )

    if residual_sample.size < minimum_sample_size:
        residual_sample = residual.reshape(-1)

    return float(
        np.mean(
            np.abs(residual_sample)
        )
    )


def estimate_skew(
    grayscale: np.ndarray,
) -> float:
    reduced = grayscale

    maximum_dimension = max(
        grayscale.shape
    )

    if maximum_dimension > 1800:
        scale = 1800.0 / maximum_dimension

        reduced = cv2.resize(
            grayscale,
            None,
            fx=scale,
            fy=scale,
            interpolation=cv2.INTER_AREA,
        )

    reduced = cv2.medianBlur(
        reduced,
        3,
    )

    binary = cv2.threshold(
        reduced,
        0,
        255,
        cv2.THRESH_BINARY_INV
        + cv2.THRESH_OTSU,
    )[1]

    kernel_width = max(
        25,
        reduced.shape[1] // 35,
    )

    horizontal_kernel = (
        cv2.getStructuringElement(
            cv2.MORPH_RECT,
            (kernel_width, 1),
        )
    )

    horizontal_lines = cv2.morphologyEx(
        binary,
        cv2.MORPH_OPEN,
        horizontal_kernel,
    )

    lines = cv2.HoughLinesP(
        horizontal_lines,
        1,
        np.pi / 1800.0,
        threshold=max(
            40,
            reduced.shape[1] // 12,
        ),
        minLineLength=max(
            80,
            reduced.shape[1] // 7,
        ),
        maxLineGap=max(
            10,
            reduced.shape[1] // 100,
        ),
    )

    if lines is None:
        return 0.0

    angles = []
    weights = []

    for x1, y1, x2, y2 in lines[:, 0, :]:
        delta_x = int(x2) - int(x1)
        delta_y = int(y2) - int(y1)

        if delta_x == 0:
            continue

        angle = math.degrees(
            math.atan2(
                delta_y,
                delta_x,
            )
        )

        if abs(angle) <= 10.0:
            angles.append(angle)

            weights.append(
                math.hypot(
                    delta_x,
                    delta_y,
                )
            )

    if len(angles) < 2:
        return 0.0

    order = np.argsort(angles)

    sorted_angles = np.asarray(
        angles
    )[order]

    sorted_weights = np.asarray(
        weights
    )[order]

    cumulative_weights = np.cumsum(
        sorted_weights
    )

    middle_index = np.searchsorted(
        cumulative_weights,
        cumulative_weights[-1] / 2.0,
    )

    return float(
        sorted_angles[middle_index]
    )


def calculate_metrics(
    grayscale: np.ndarray,
) -> dict:
    percentile_1, percentile_99 = (
        np.percentile(
            grayscale,
            [1, 99],
        )
    )

    return {
        "brightness_mean": float(
            grayscale.mean()
        ),
        "contrast_std": float(
            grayscale.std()
        ),
        "dynamic_range": float(
            percentile_99 - percentile_1
        ),
        "laplacian_variance": float(
            cv2.Laplacian(
                grayscale,
                cv2.CV_64F,
            ).var()
        ),
        "noise_residual": float(
            calculate_noise_residual(
                grayscale
            )
        ),
        "skew_degrees": float(
            estimate_skew(grayscale)
        ),
        "foreground_ratio": float(
            np.mean(grayscale < 128)
        ),
    }


def detect_flags(
    metrics: dict,
    thresholds: dict,
) -> list[str]:
    flags = []

    if (
        metrics["contrast_std"]
        < thresholds[
            "minimum_contrast_std"
        ]
    ):
        flags.append("LOW_CONTRAST")

    if (
        metrics["dynamic_range"]
        < thresholds[
            "minimum_dynamic_range"
        ]
    ):
        flags.append("LOW_DYNAMIC_RANGE")

    if (
        metrics["laplacian_variance"]
        < thresholds[
            "minimum_laplacian_variance"
        ]
    ):
        flags.append("BLUR")

    if (
        metrics["brightness_mean"]
        < thresholds[
            "minimum_brightness_mean"
        ]
    ):
        flags.append("TOO_DARK")

    if (
        metrics["brightness_mean"]
        > thresholds[
            "maximum_brightness_mean"
        ]
    ):
        flags.append("TOO_BRIGHT")

    if (
        metrics["noise_residual"]
        > thresholds[
            "maximum_noise_residual"
        ]
    ):
        flags.append("NOISE")

    if (
        abs(metrics["skew_degrees"])
        > thresholds[
            "maximum_absolute_skew_degrees"
        ]
    ):
        flags.append("SKEW")

    return flags


# ============================================================
# 5. CANDIDATE QUALITY GATE
# ============================================================

def evaluate_candidate(
    image: np.ndarray,
    metrics: dict,
    thresholds: dict,
) -> dict:
    return {
        "geometry_valid": bool(
            abs(metrics["skew_degrees"])
            <= thresholds[
                "maximum_absolute_skew_degrees"
            ]
        ),
        "noise_valid": bool(
            metrics["noise_residual"]
            <= thresholds[
                "maximum_noise_residual"
            ]
        ),
        "contrast_valid": bool(
            metrics["contrast_std"]
            >= thresholds[
                "minimum_contrast_std"
            ]
            and metrics["dynamic_range"]
            >= thresholds[
                "minimum_dynamic_range"
            ]
        ),
        "brightness_valid": bool(
            thresholds[
                "minimum_brightness_mean"
            ]
            <= metrics["brightness_mean"]
            <= thresholds[
                "maximum_brightness_mean"
            ]
        ),
        "sharpness_valid": bool(
            metrics["laplacian_variance"]
            >= thresholds[
                "minimum_laplacian_variance"
            ]
        ),
        "foreground_valid": bool(
            MINIMUM_FOREGROUND_RATIO
            <= metrics["foreground_ratio"]
            <= MAXIMUM_FOREGROUND_RATIO
        ),
        "nonblank": bool(
            image.std() > 1.0
            and image.min() < image.max()
        ),
    }


def all_quality_gates_passed(
    gates: dict,
) -> bool:
    return all(gates.values())


def candidate_penalty(
    metrics: dict,
    thresholds: dict,
) -> float:
    penalty = 0.0

    penalty += max(
        0.0,
        thresholds[
            "minimum_brightness_mean"
        ]
        - metrics["brightness_mean"],
    )

    penalty += max(
        0.0,
        metrics["brightness_mean"]
        - thresholds[
            "maximum_brightness_mean"
        ],
    )

    penalty += 5.0 * max(
        0.0,
        metrics["noise_residual"]
        - thresholds[
            "maximum_noise_residual"
        ],
    )

    penalty += 10.0 * max(
        0.0,
        abs(metrics["skew_degrees"])
        - thresholds[
            "maximum_absolute_skew_degrees"
        ],
    )

    if not (
        MINIMUM_FOREGROUND_RATIO
        <= metrics["foreground_ratio"]
        <= MAXIMUM_FOREGROUND_RATIO
    ):
        penalty += 100.0

    return float(penalty)


# ============================================================
# 6. PROCESSING OPERATIONS
# ============================================================

def rotate_keep_size(
    image: np.ndarray,
    degrees: float,
) -> np.ndarray:
    height, width = image.shape[:2]

    matrix = cv2.getRotationMatrix2D(
        (width / 2.0, height / 2.0),
        degrees,
        1.0,
    )

    return cv2.warpAffine(
        image,
        matrix,
        (width, height),
        flags=cv2.INTER_CUBIC,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=int(np.median(image)),
    )


def select_best_deskew(
    image: np.ndarray,
    detected_angle: float,
) -> np.ndarray:
    candidates = [
        image,
        rotate_keep_size(
            image,
            detected_angle,
        ),
        rotate_keep_size(
            image,
            -detected_angle,
        ),
    ]

    return min(
        candidates,
        key=lambda candidate: abs(
            estimate_skew(candidate)
        ),
    )


def build_binary_candidate(
    deskewed_image: np.ndarray,
    thresholds: dict,
) -> tuple[
    np.ndarray,
    int,
    dict,
    dict,
]:
    best_candidate = None
    best_strength = None
    best_metrics = None
    best_gates = None
    best_penalty = float("inf")

    for strength in DENOISE_STRENGTHS:
        denoised = cv2.fastNlMeansDenoising(
            deskewed_image,
            None,
            h=strength,
            templateWindowSize=7,
            searchWindowSize=21,
        )

        _, binary_candidate = cv2.threshold(
            denoised,
            0,
            255,
            cv2.THRESH_BINARY
            + cv2.THRESH_OTSU,
        )

        metrics = calculate_metrics(
            binary_candidate
        )

        gates = evaluate_candidate(
            binary_candidate,
            metrics,
            thresholds,
        )

        penalty = candidate_penalty(
            metrics,
            thresholds,
        )

        if penalty < best_penalty:
            best_candidate = (
                binary_candidate
            )
            best_strength = strength
            best_metrics = metrics
            best_gates = gates
            best_penalty = penalty

        # Pilih strength paling rendah yang lulus,
        # supaya detail karakter dipertahankan.
        if all_quality_gates_passed(gates):
            return (
                binary_candidate,
                strength,
                metrics,
                gates,
            )

    if best_candidate is None:
        raise RuntimeError(
            "Tidak ada binary candidate."
        )

    return (
        best_candidate,
        int(best_strength),
        best_metrics,
        best_gates,
    )


def preprocess_candidate_v2_1(
    image: np.ndarray,
    thresholds: dict,
) -> tuple[np.ndarray, dict]:
    initial_metrics = calculate_metrics(
        image
    )

    initial_flags = detect_flags(
        initial_metrics,
        thresholds,
    )

    # Clean image tidak boleh dimodifikasi.
    if not initial_flags:
        return (
            image.copy(),
            {
                "initial_flags": [],
                "applied_steps": [],
                "initial_metrics": (
                    initial_metrics
                ),
                "final_metrics": (
                    initial_metrics
                ),
                "quality_gates": (
                    evaluate_candidate(
                        image,
                        initial_metrics,
                        thresholds,
                    )
                ),
                "denoise_strength": 0,
            },
        )

    working = image.copy()
    applied_steps = []

    if "SKEW" in initial_flags:
        working = select_best_deskew(
            working,
            initial_metrics[
                "skew_degrees"
            ],
        )

        applied_steps.append("DESKEW")

    (
        working,
        denoise_strength,
        final_metrics,
        quality_gates,
    ) = build_binary_candidate(
        working,
        thresholds,
    )

    applied_steps.extend(
        [
            f"DENOISE_H{denoise_strength}",
            "OTSU_BINARIZATION",
        ]
    )

    return (
        working,
        {
            "initial_flags": initial_flags,
            "applied_steps": applied_steps,
            "initial_metrics": (
                initial_metrics
            ),
            "final_metrics": (
                final_metrics
            ),
            "quality_gates": quality_gates,
            "denoise_strength": (
                denoise_strength
            ),
        },
    )


# ============================================================
# 7. PREFLIGHT
# ============================================================

required_paths = [
    BUILD_ROOT,
    STRESS_MANIFEST_PATH,
    REJECTED_EVALUATION_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not path.exists()
]

if missing_paths:
    raise FileNotFoundError(
        "Checkpoint belum lengkap:\n"
        + "\n".join(missing_paths)
    )

stress_manifest = json.loads(
    STRESS_MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)

previous_evaluation = json.loads(
    REJECTED_EVALUATION_PATH.read_text(
        encoding="utf-8"
    )
)

if stress_manifest.get("status") != "PASSED":
    raise RuntimeError(
        "Stress manifest belum PASSED."
    )

if (
    previous_evaluation.get(
        "preprocessing_decision"
    )
    != "REJECTED"
):
    raise RuntimeError(
        "Candidate sebelumnya belum REJECTED."
    )

thresholds = locate_thresholds(
    stress_manifest
)

if thresholds is None:
    raise RuntimeError(
        "Threshold kalibrasi tidak ditemukan."
    )

stress_result_path = Path(
    stress_manifest[
        "results"
    ]["result_path"]
)

stress_table = pd.read_csv(
    stress_result_path
)

combined_table = (
    stress_table[
        (
            stress_table["scenario"]
            == "combined_scan"
        )
        & (
            stress_table["status"]
            == "VALID"
        )
    ]
    .copy()
    .sort_values(
        [
            "template_id",
            "document_id",
        ]
    )
    .reset_index(drop=True)
)

if len(combined_table) != EXPECTED_DOCUMENTS:
    raise RuntimeError(
        "Combined-scan documents harus 6, "
        f"ditemukan {len(combined_table)}."
    )


# ============================================================
# 8. GENERATE CANDIDATE V2.1
# ============================================================

records = []
errors = []

source_checksums_before = {}
degraded_checksums_before = {}

print(
    "Membangun preprocessing Candidate V2.1...\n"
)

for sequence_number, row in enumerate(
    combined_table.itertuples(index=False),
    start=1,
):
    try:
        source_path = Path(row.source_path)
        degraded_path = Path(
            row.degraded_path
        )

        source_checksum = sha256_file(
            source_path
        )

        degraded_checksum = sha256_file(
            degraded_path
        )

        if (
            degraded_checksum
            != row.degraded_sha256
        ):
            raise RuntimeError(
                "Checksum degraded tidak cocok."
            )

        source_checksums_before[
            row.document_id
        ] = source_checksum

        degraded_checksums_before[
            row.document_id
        ] = degraded_checksum

        clean_image = read_grayscale(
            source_path
        )

        degraded_image = read_grayscale(
            degraded_path
        )

        if (
            clean_image.shape
            != degraded_image.shape
        ):
            raise RuntimeError(
                "Dimensi input tidak konsisten."
            )

        # Uji clean-image invariance.
        (
            processed_clean,
            clean_decision,
        ) = preprocess_candidate_v2_1(
            clean_image,
            thresholds,
        )

        clean_invariance_valid = bool(
            np.array_equal(
                clean_image,
                processed_clean,
            )
            and not clean_decision[
                "initial_flags"
            ]
            and not clean_decision[
                "applied_steps"
            ]
        )

        # Proses combined scan.
        (
            candidate_image,
            decision,
        ) = preprocess_candidate_v2_1(
            degraded_image,
            thresholds,
        )

        candidate_path = (
            CANDIDATE_IMAGE_ROOT
            / row.template_id
            / (
                f"{row.document_id}"
                "_combined_scan_candidate_v2_1.png"
            )
        )

        atomic_write_png(
            candidate_path,
            candidate_image,
        )

        written_image = read_grayscale(
            candidate_path
        )

        pixel_write_valid = bool(
            np.array_equal(
                candidate_image,
                written_image,
            )
        )

        dimensions_preserved = bool(
            candidate_image.shape
            == degraded_image.shape
        )

        candidate_changed = bool(
            not np.array_equal(
                candidate_image,
                degraded_image,
            )
        )

        quality_gates = decision[
            "quality_gates"
        ]

        candidate_valid = bool(
            clean_invariance_valid
            and pixel_write_valid
            and dimensions_preserved
            and candidate_changed
            and all_quality_gates_passed(
                quality_gates
            )
        )

        initial_metrics = decision[
            "initial_metrics"
        ]

        final_metrics = decision[
            "final_metrics"
        ]

        record = {
            "sequence_number": (
                sequence_number
            ),
            "document_id": row.document_id,
            "template_id": row.template_id,
            "language": row.language,
            "scenario": "combined_scan",
            "candidate_version": "2.1",
            "source_path": str(source_path),
            "degraded_path": str(
                degraded_path
            ),
            "candidate_path": str(
                candidate_path
            ),
            "source_sha256": (
                source_checksum
            ),
            "degraded_sha256": (
                degraded_checksum
            ),
            "candidate_sha256": sha256_file(
                candidate_path
            ),
            "clean_invariance_valid": (
                clean_invariance_valid
            ),
            "pixel_write_valid": (
                pixel_write_valid
            ),
            "dimensions_preserved": (
                dimensions_preserved
            ),
            "candidate_changed": (
                candidate_changed
            ),
            "initial_flags": "|".join(
                decision["initial_flags"]
            ),
            "applied_steps": "|".join(
                decision["applied_steps"]
            ),
            "denoise_strength": (
                decision[
                    "denoise_strength"
                ]
            ),
            **quality_gates,
            "status": (
                "VALID"
                if candidate_valid
                else "INVALID"
            ),
        }

        for metric_name, value in (
            initial_metrics.items()
        ):
            record[
                f"before_{metric_name}"
            ] = value

        for metric_name, value in (
            final_metrics.items()
        ):
            record[
                f"after_{metric_name}"
            ] = value

        records.append(record)

        print(
            f"[{sequence_number}/6] "
            f"{row.document_id} | "
            f"steps={decision['applied_steps']} | "
            f"brightness="
            f"{final_metrics['brightness_mean']:.2f} | "
            f"noise="
            f"{final_metrics['noise_residual']:.4f} | "
            f"foreground="
            f"{final_metrics['foreground_ratio']:.4f} | "
            f"status="
            f"{'VALID' if candidate_valid else 'INVALID'}"
        )

    except Exception as error:
        errors.append(
            {
                "sequence_number": (
                    sequence_number
                ),
                "document_id": row.document_id,
                "template_id": row.template_id,
                "error_type": (
                    type(error).__name__
                ),
                "error": str(error)[:500],
            }
        )

        print(
            f"[{sequence_number}/6] "
            f"{row.document_id} | ERROR: "
            f"{type(error).__name__}: {error}"
        )


# ============================================================
# 9. INTEGRITY VALIDATION
# ============================================================

candidate_table = pd.DataFrame(
    records
)

source_changes = []
degraded_changes = []

for row in combined_table.itertuples(
    index=False
):
    if (
        row.document_id
        in source_checksums_before
        and sha256_file(
            Path(row.source_path)
        )
        != source_checksums_before[
            row.document_id
        ]
    ):
        source_changes.append(
            row.document_id
        )

    if (
        row.document_id
        in degraded_checksums_before
        and sha256_file(
            Path(row.degraded_path)
        )
        != degraded_checksums_before[
            row.document_id
        ]
    ):
        degraded_changes.append(
            row.document_id
        )

candidate_files = list(
    CANDIDATE_IMAGE_ROOT.rglob("*.png")
)

valid_candidates = (
    int(
        (
            candidate_table["status"]
            == "VALID"
        ).sum()
    )
    if not candidate_table.empty
    else 0
)

controls = [
    {
        "control": "previous_decision",
        "expected": "REJECTED",
        "actual": (
            previous_evaluation.get(
                "preprocessing_decision"
            )
        ),
    },
    {
        "control": "candidate_documents",
        "expected": 6,
        "actual": len(candidate_table),
    },
    {
        "control": "template_coverage",
        "expected": 6,
        "actual": (
            candidate_table[
                "template_id"
            ].nunique()
            if not candidate_table.empty
            else 0
        ),
    },
    {
        "control": "clean_images_preserved",
        "expected": 6,
        "actual": (
            int(
                candidate_table[
                    "clean_invariance_valid"
                ].sum()
            )
            if not candidate_table.empty
            else 0
        ),
    },
    {
        "control": "valid_candidates",
        "expected": 6,
        "actual": valid_candidates,
    },
    {
        "control": "candidate_files",
        "expected": 6,
        "actual": len(candidate_files),
    },
    {
        "control": "processing_errors",
        "expected": 0,
        "actual": len(errors),
    },
    {
        "control": "source_checksum_changes",
        "expected": 0,
        "actual": len(source_changes),
    },
    {
        "control": "degraded_checksum_changes",
        "expected": 0,
        "actual": len(degraded_changes),
    },
    {
        "control": "validation_opened",
        "expected": 0,
        "actual": 0,
    },
    {
        "control": "test_opened",
        "expected": 0,
        "actual": 0,
    },
]

for control in controls:
    control["status"] = (
        "VALID"
        if control["expected"]
        == control["actual"]
        else "INVALID"
    )

control_table = pd.DataFrame(
    controls
)

display(control_table)

print("\nCANDIDATE V2.1 METRICS")

display(
    candidate_table[
        [
            "document_id",
            "template_id",
            "language",
            "applied_steps",
            "denoise_strength",
            "before_brightness_mean",
            "after_brightness_mean",
            "before_noise_residual",
            "after_noise_residual",
            "before_skew_degrees",
            "after_skew_degrees",
            "after_foreground_ratio",
            "status",
        ]
    ]
)


# ============================================================
# 10. CONTACT SHEET
# ============================================================

if len(candidate_table) == 6:
    figure, axes = plt.subplots(
        6,
        3,
        figsize=(15, 32),
    )

    for row_index, row in enumerate(
        candidate_table.itertuples(
            index=False
        )
    ):
        images = [
            read_grayscale(
                Path(row.source_path)
            ),
            read_grayscale(
                Path(row.degraded_path)
            ),
            read_grayscale(
                Path(row.candidate_path)
            ),
        ]

        titles = [
            (
                f"{row.document_id}\n"
                "CLEAN REFERENCE"
            ),
            "COMBINED SCAN",
            "CANDIDATE V2.1",
        ]

        for column_index, (
            image,
            title,
        ) in enumerate(
            zip(images, titles)
        ):
            axes[
                row_index,
                column_index,
            ].imshow(
                image,
                cmap="gray",
                vmin=0,
                vmax=255,
            )

            axes[
                row_index,
                column_index,
            ].set_title(
                title,
                fontsize=10,
            )

            axes[
                row_index,
                column_index,
            ].axis("off")

    plt.tight_layout()
    plt.show()


# ============================================================
# 11. FINALIZE
# ============================================================

invalid_controls = [
    control["control"]
    for control in controls
    if control["status"] != "VALID"
]

if invalid_controls:
    if errors:
        print("\nPROCESSING ERRORS")
        display(pd.DataFrame(errors))

    print("\nINVALID CANDIDATES")
    display(
        candidate_table[
            candidate_table["status"]
            != "VALID"
        ]
    )

    raise RuntimeError(
        "CELL 9E-A2 FAILED. "
        f"Kontrol tidak valid: "
        f"{invalid_controls}"
    )

atomic_write_csv(
    CANDIDATE_RESULT_PATH,
    candidate_table,
)

candidate_manifest = {
    "schema_version": "2.1.0",
    "run_id": RUN_ID,
    "status": "READY_FOR_OCR",
    "previous_candidate": {
        "version": "2.0",
        "decision": "REJECTED",
    },
    "pipeline": [
        "quality_detection",
        "deskew",
        "adaptive_nlmeans_denoising",
        "otsu_binarization",
        "quality_gate",
    ],
    "selection_policy": {
        "denoise_strengths": list(
            DENOISE_STRENGTHS
        ),
        "selection": (
            "LOWEST_STRENGTH_PASSING_ALL_GATES"
        ),
        "foreground_ratio_range": [
            MINIMUM_FOREGROUND_RATIO,
            MAXIMUM_FOREGROUND_RATIO,
        ],
    },
    "thresholds": thresholds,
    "results": {
        "documents": 6,
        "templates": 6,
        "valid_candidates": 6,
        "invalid_candidates": 0,
        "candidate_result_path": str(
            CANDIDATE_RESULT_PATH
        ),
        "candidate_image_root": str(
            CANDIDATE_IMAGE_ROOT
        ),
    },
    "integrity": {
        "clean_images_preserved": 6,
        "source_checksum_changes": 0,
        "degraded_checksum_changes": 0,
        "dataset_modifications": 0,
        "validation_opened": 0,
        "test_opened": 0,
    },
    "next_stage": {
        "cell": "9E-B",
        "action": (
            "PADDLEOCR_CANDIDATE_V2_1"
        ),
        "status": "PENDING",
    },
}

atomic_write_json(
    CANDIDATE_MANIFEST_PATH,
    candidate_manifest,
)

print()
print(f"Run ID               : {RUN_ID}")
print(f"Candidate root       : {CANDIDATE_ROOT}")
print(
    f"Candidate results    : "
    f"{CANDIDATE_RESULT_PATH}"
)
print(
    f"Candidate manifest   : "
    f"{CANDIDATE_MANIFEST_PATH}"
)
print(
    f"Manifest SHA-256     : "
    f"{sha256_file(CANDIDATE_MANIFEST_PATH)}"
)
print("Candidate documents  : 6")
print("Valid candidates     : 6/6")
print("Clean images preserved: 6/6")
print("OCR executions       : 0")
print("Dataset modifications: 0")
print("Validation opened    : 0")
print("Test opened          : 0")
print("Candidate status     : READY_FOR_OCR")
print()
print(
    "✅ CELL 9E-A2 PASSED — Candidate V2.1 "
    "lulus quality gate dan siap diuji menggunakan "
    "PaddleOCR pada Cell 9E-B."
)

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import time
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


# ============================================================
# CELL 9E-B — PADDLEOCR CANDIDATE V2.1 EXECUTION
#
# - menjalankan OCR pada 6 Candidate V2.1;
# - checkpoint per dokumen;
# - aman jika runtime terputus;
# - belum mengambil keputusan kualitas akhir.
# ============================================================


# ============================================================
# 1. VALIDASI RUNTIME
# ============================================================

required_runtime_names = [
    "PADDLE_ROBUSTNESS_ENGINES",
    "extract_paddle_lines",
    "json_safe",
]

missing_runtime_names = [
    name
    for name in required_runtime_names
    if name not in globals()
]

if missing_runtime_names:
    raise RuntimeError(
        "Engine/helper PaddleOCR belum tersedia. "
        "Jalankan kembali Cell 9C-A terlebih dahulu. "
        f"State hilang: {missing_runtime_names}"
    )

if len(PADDLE_ROBUSTNESS_ENGINES) != 2:
    raise RuntimeError(
        "Dua engine PaddleOCR belum tersedia."
    )


# ============================================================
# 2. KONFIGURASI
# ============================================================

BUILD_ROOT = Path(
    "/content/drive/MyDrive/InvoiceFlow-AI-Data/interim/"
    "synthetic_v1_build/20260904T150025Z"
)

BENCHMARK_ROOT = BUILD_ROOT / "ocr_benchmark"

CANDIDATE_ROOT = (
    BENCHMARK_ROOT
    / "preprocessing_candidate_v2_1"
)

CANDIDATE_MANIFEST_PATH = (
    CANDIDATE_ROOT
    / "preprocessing_candidate_v2_1_manifest.json"
)

RESULT_ROOT = (
    CANDIDATE_ROOT
    / "ocr_results"
    / "paddleocr_ppocrv6"
)

OCR_INDEX_PATH = (
    CANDIDATE_ROOT
    / "paddleocr_candidate_v2_1_index.jsonl"
)

OCR_SUMMARY_PATH = (
    CANDIDATE_ROOT
    / "paddleocr_candidate_v2_1_summary.csv"
)

OCR_MANIFEST_PATH = (
    CANDIDATE_ROOT
    / "paddleocr_candidate_v2_1_manifest.json"
)

EXPECTED_DOCUMENTS = 6

RESULT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 3. FILE HELPERS
# ============================================================

def sha256_file(file_path: Path) -> str:
    digest = hashlib.sha256()

    with file_path.open("rb") as file_handle:
        for chunk in iter(
            lambda: file_handle.read(
                1024 * 1024
            ),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def atomic_write_text(
    file_path: Path,
    content: str,
) -> None:
    file_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = file_path.with_name(
        file_path.name + ".tmp"
    )

    temporary_path.write_text(
        content,
        encoding="utf-8",
    )

    os.replace(
        temporary_path,
        file_path,
    )


def atomic_write_json(
    file_path: Path,
    content: dict,
) -> None:
    atomic_write_text(
        file_path,
        json.dumps(
            json_safe(content),
            indent=2,
            ensure_ascii=False,
        )
        + "\n",
    )


def atomic_write_jsonl(
    file_path: Path,
    records: list[dict],
) -> None:
    content = "".join(
        json.dumps(
            json_safe(record),
            ensure_ascii=False,
        )
        + "\n"
        for record in records
    )

    atomic_write_text(
        file_path,
        content,
    )


def atomic_write_csv(
    file_path: Path,
    table: pd.DataFrame,
) -> None:
    temporary_path = file_path.with_name(
        file_path.name + ".tmp"
    )

    table.to_csv(
        temporary_path,
        index=False,
    )

    os.replace(
        temporary_path,
        file_path,
    )


# ============================================================
# 4. CHECKPOINT HELPERS
# ============================================================

def checkpoint_path_for(
    template_id: str,
    document_id: str,
) -> Path:
    return (
        RESULT_ROOT
        / template_id
        / (
            f"{document_id}"
            "_candidate_v2_1.json"
        )
    )


def load_checkpoint(
    checkpoint_path: Path,
):
    if not checkpoint_path.is_file():
        return None

    try:
        return json.loads(
            checkpoint_path.read_text(
                encoding="utf-8"
            )
        )
    except Exception:
        return None


def checkpoint_is_valid(
    checkpoint,
    input_record: dict,
) -> bool:
    if not isinstance(checkpoint, dict):
        return False

    document = checkpoint.get(
        "document",
        {},
    )

    input_section = checkpoint.get(
        "input",
        {},
    )

    output = checkpoint.get(
        "output",
        {},
    )

    lines = output.get("lines", [])

    return bool(
        checkpoint.get("status") == "PASSED"
        and checkpoint.get("engine")
        == "PaddleOCR PP-OCRv6"
        and document.get("document_id")
        == input_record["document_id"]
        and document.get("template_id")
        == input_record["template_id"]
        and document.get("language")
        == input_record["language"]
        and input_section.get(
            "candidate_version"
        )
        == "2.1"
        and input_section.get("sha256")
        == input_record["candidate_sha256"]
        and isinstance(lines, list)
        and len(lines) > 0
        and output.get(
            "recognized_regions"
        )
        == len(lines)
    )


def checkpoint_to_record(
    checkpoint: dict,
    checkpoint_path: Path,
    sequence_number: int,
    execution: str,
) -> dict:
    document = checkpoint["document"]
    input_section = checkpoint["input"]
    output = checkpoint["output"]
    timing = checkpoint["timing"]

    return {
        "sequence_number": (
            sequence_number
        ),
        "document_id": (
            document["document_id"]
        ),
        "template_id": (
            document["template_id"]
        ),
        "language": (
            document["language"]
        ),
        "candidate_version": "2.1",
        "candidate_path": (
            input_section["path"]
        ),
        "candidate_sha256": (
            input_section["sha256"]
        ),
        "recognized_regions": int(
            output["recognized_regions"]
        ),
        "regions_with_bbox": int(
            output["regions_with_bbox"]
        ),
        "mean_confidence": (
            output["mean_confidence"]
        ),
        "seconds": float(
            timing["seconds"]
        ),
        "execution": execution,
        "status": checkpoint["status"],
        "result_path": str(
            checkpoint_path
        ),
        "result_sha256": sha256_file(
            checkpoint_path
        ),
        "error": "",
    }


# ============================================================
# 5. PREFLIGHT
# ============================================================

if not CANDIDATE_MANIFEST_PATH.is_file():
    raise FileNotFoundError(
        "Manifest Candidate V2.1 tidak ditemukan. "
        "Jalankan Cell 9E-A2 terlebih dahulu."
    )

candidate_manifest = json.loads(
    CANDIDATE_MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)

if (
    candidate_manifest.get("status")
    != "READY_FOR_OCR"
):
    raise RuntimeError(
        "Candidate V2.1 belum READY_FOR_OCR."
    )

candidate_result_path = Path(
    candidate_manifest[
        "results"
    ]["candidate_result_path"]
)

if not candidate_result_path.is_file():
    raise FileNotFoundError(
        f"Candidate result tidak ditemukan: "
        f"{candidate_result_path}"
    )

candidate_table = pd.read_csv(
    candidate_result_path
)

candidate_table = (
    candidate_table[
        candidate_table["status"]
        == "VALID"
    ]
    .copy()
    .sort_values(
        [
            "template_id",
            "document_id",
        ]
    )
    .reset_index(drop=True)
)

if len(candidate_table) != EXPECTED_DOCUMENTS:
    raise RuntimeError(
        "Candidate valid harus 6, ditemukan "
        f"{len(candidate_table)}."
    )

input_checksum_mismatches = []

for row in candidate_table.itertuples(
    index=False
):
    candidate_path = Path(
        row.candidate_path
    )

    if not candidate_path.is_file():
        raise FileNotFoundError(
            f"Candidate image hilang: "
            f"{candidate_path}"
        )

    actual_checksum = sha256_file(
        candidate_path
    )

    if actual_checksum != row.candidate_sha256:
        input_checksum_mismatches.append(
            {
                "document_id": (
                    row.document_id
                ),
                "expected": (
                    row.candidate_sha256
                ),
                "actual": actual_checksum,
            }
        )

if input_checksum_mismatches:
    display(
        pd.DataFrame(
            input_checksum_mismatches
        )
    )

    raise RuntimeError(
        "Checksum Candidate V2.1 tidak cocok."
    )


# ============================================================
# 6. OCR EXECUTION
# ============================================================

print(
    "Memulai PaddleOCR pada 6 "
    "Candidate V2.1..."
)
print(
    f"Result root: {RESULT_ROOT}\n"
)

result_records = []
execution_errors = []

newly_processed = 0
recovered_results = 0

for sequence_number, row in enumerate(
    candidate_table.itertuples(
        index=False
    ),
    start=1,
):
    input_record = {
        "document_id": row.document_id,
        "template_id": row.template_id,
        "language": row.language,
        "candidate_path": (
            row.candidate_path
        ),
        "candidate_sha256": (
            row.candidate_sha256
        ),
    }

    candidate_path = Path(
        row.candidate_path
    )

    checkpoint_path = checkpoint_path_for(
        row.template_id,
        row.document_id,
    )

    checkpoint = load_checkpoint(
        checkpoint_path
    )

    if checkpoint_is_valid(
        checkpoint,
        input_record,
    ):
        recovered_results += 1

        result_record = (
            checkpoint_to_record(
                checkpoint,
                checkpoint_path,
                sequence_number,
                "RECOVERED",
            )
        )

        result_records.append(
            result_record
        )

        print(
            f"[{sequence_number}/6] "
            f"{row.document_id} | RECOVERED | "
            f"regions="
            f"{result_record['recognized_regions']} | "
            f"confidence="
            f"{result_record['mean_confidence']:.6f}"
        )

        continue

    print(
        f"[{sequence_number}/6] "
        f"{row.document_id} | "
        f"language={row.language}"
    )

    start_time = time.perf_counter()

    try:
        current_checksum = sha256_file(
            candidate_path
        )

        if (
            current_checksum
            != row.candidate_sha256
        ):
            raise RuntimeError(
                "Checksum input berubah "
                "sebelum OCR."
            )

        engine = (
            PADDLE_ROBUSTNESS_ENGINES[
                row.language
            ]
        )

        predictions = engine.predict(
            input=str(candidate_path)
        )

        (
            recognized_lines,
            raw_mappings,
        ) = extract_paddle_lines(
            predictions
        )

        elapsed_seconds = (
            time.perf_counter()
            - start_time
        )

        if not recognized_lines:
            raise RuntimeError(
                "PaddleOCR tidak menghasilkan "
                "text region."
            )

        confidence_values = [
            float(line["confidence"])
            for line in recognized_lines
            if line.get("confidence") is not None
        ]

        mean_confidence = (
            float(
                np.mean(
                    confidence_values
                )
            )
            if confidence_values
            else None
        )

        regions_with_bbox = int(
            sum(
                len(line.get("bbox", []))
                == 4
                for line in recognized_lines
            )
        )

        full_text = "\n".join(
            line["text"]
            for line in recognized_lines
        )

        checkpoint = {
            "schema_version": "1.0.0",
            "status": "PASSED",
            "engine": "PaddleOCR PP-OCRv6",
            "document": {
                "document_id": (
                    row.document_id
                ),
                "template_id": (
                    row.template_id
                ),
                "language": row.language,
            },
            "input": {
                "candidate_version": "2.1",
                "scenario": "combined_scan",
                "path": str(
                    candidate_path
                ),
                "sha256": (
                    current_checksum
                ),
            },
            "output": {
                "recognized_regions": len(
                    recognized_lines
                ),
                "regions_with_bbox": (
                    regions_with_bbox
                ),
                "mean_confidence": (
                    mean_confidence
                ),
                "full_text": full_text,
                "lines": recognized_lines,
                "result_objects": len(
                    raw_mappings
                ),
            },
            "runtime": {
                "device": "cpu",
                "mkldnn_enabled": False,
                "paddlepaddle_version": version(
                    "paddlepaddle"
                ),
                "paddleocr_version": version(
                    "paddleocr"
                ),
                "paddlex_version": version(
                    "paddlex"
                ),
                "external_api_required": False,
            },
            "timing": {
                "seconds": round(
                    elapsed_seconds,
                    6,
                ),
            },
        }

        atomic_write_json(
            checkpoint_path,
            checkpoint,
        )

        written_checkpoint = (
            load_checkpoint(
                checkpoint_path
            )
        )

        if not checkpoint_is_valid(
            written_checkpoint,
            input_record,
        ):
            raise RuntimeError(
                "Checkpoint gagal diverifikasi."
            )

        newly_processed += 1

        result_record = (
            checkpoint_to_record(
                written_checkpoint,
                checkpoint_path,
                sequence_number,
                "NEW",
            )
        )

        result_records.append(
            result_record
        )

        confidence_text = (
            f"{mean_confidence:.6f}"
            if mean_confidence is not None
            else "None"
        )

        print(
            f"         PASSED | "
            f"regions="
            f"{len(recognized_lines)} | "
            f"bbox={regions_with_bbox} | "
            f"confidence={confidence_text} | "
            f"{elapsed_seconds:.2f}s"
        )

    except Exception as error:
        elapsed_seconds = (
            time.perf_counter()
            - start_time
        )

        error_checkpoint = {
            "schema_version": "1.0.0",
            "status": "ERROR",
            "engine": "PaddleOCR PP-OCRv6",
            "document": {
                "document_id": (
                    row.document_id
                ),
                "template_id": (
                    row.template_id
                ),
                "language": row.language,
            },
            "input": {
                "candidate_version": "2.1",
                "scenario": "combined_scan",
                "path": str(
                    candidate_path
                ),
                "sha256": sha256_file(
                    candidate_path
                ),
            },
            "timing": {
                "seconds": round(
                    elapsed_seconds,
                    6,
                ),
            },
            "error": {
                "type": (
                    type(error).__name__
                ),
                "message": str(error)[:1000],
            },
        }

        atomic_write_json(
            checkpoint_path,
            error_checkpoint,
        )

        execution_errors.append(
            {
                "sequence_number": (
                    sequence_number
                ),
                "document_id": (
                    row.document_id
                ),
                "template_id": (
                    row.template_id
                ),
                "language": row.language,
                "error_type": (
                    type(error).__name__
                ),
                "error": str(error)[:500],
            }
        )

        print(
            f"         ERROR | "
            f"{type(error).__name__}: "
            f"{error}"
        )


# ============================================================
# 7. VERIFICATION
# ============================================================

result_table = pd.DataFrame(
    result_records
)

if not result_table.empty:
    result_table = (
        result_table
        .sort_values("sequence_number")
        .reset_index(drop=True)
    )

verification_failures = []

for row in candidate_table.itertuples(
    index=False
):
    checkpoint_path = checkpoint_path_for(
        row.template_id,
        row.document_id,
    )

    checkpoint = load_checkpoint(
        checkpoint_path
    )

    input_record = {
        "document_id": row.document_id,
        "template_id": row.template_id,
        "language": row.language,
        "candidate_sha256": (
            row.candidate_sha256
        ),
    }

    if not checkpoint_is_valid(
        checkpoint,
        input_record,
    ):
        verification_failures.append(
            {
                "document_id": (
                    row.document_id
                ),
                "result_path": str(
                    checkpoint_path
                ),
            }
        )

result_files = list(
    RESULT_ROOT.rglob("*.json")
)

nonempty_results = (
    int(
        (
            result_table[
                "recognized_regions"
            ]
            > 0
        ).sum()
    )
    if not result_table.empty
    else 0
)

controls = [
    {
        "control": "candidate_status",
        "expected": "READY_FOR_OCR",
        "actual": (
            candidate_manifest.get(
                "status"
            )
        ),
    },
    {
        "control": "candidate_documents",
        "expected": 6,
        "actual": len(candidate_table),
    },
    {
        "control": "result_records",
        "expected": 6,
        "actual": len(result_table),
    },
    {
        "control": "result_files",
        "expected": 6,
        "actual": len(result_files),
    },
    {
        "control": "verified_results",
        "expected": 6,
        "actual": (
            6 - len(
                verification_failures
            )
        ),
    },
    {
        "control": "nonempty_results",
        "expected": 6,
        "actual": nonempty_results,
    },
    {
        "control": "execution_errors",
        "expected": 0,
        "actual": len(execution_errors),
    },
    {
        "control": "template_coverage",
        "expected": 6,
        "actual": (
            result_table[
                "template_id"
            ].nunique()
            if not result_table.empty
            else 0
        ),
    },
    {
        "control": "language_coverage",
        "expected": ["en", "id"],
        "actual": (
            sorted(
                result_table[
                    "language"
                ].unique().tolist()
            )
            if not result_table.empty
            else []
        ),
    },
    {
        "control": "mkldnn_enabled",
        "expected": False,
        "actual": False,
    },
    {
        "control": "external_api_required",
        "expected": False,
        "actual": False,
    },
    {
        "control": "validation_opened",
        "expected": 0,
        "actual": 0,
    },
    {
        "control": "test_opened",
        "expected": 0,
        "actual": 0,
    },
]

for control in controls:
    control["status"] = (
        "VALID"
        if control["expected"]
        == control["actual"]
        else "INVALID"
    )

control_table = pd.DataFrame(
    controls
)

display(control_table)

print("\nCANDIDATE V2.1 OCR RESULTS")

if not result_table.empty:
    display(
        result_table[
            [
                "sequence_number",
                "document_id",
                "template_id",
                "language",
                "recognized_regions",
                "regions_with_bbox",
                "mean_confidence",
                "seconds",
                "execution",
                "status",
            ]
        ]
    )

invalid_controls = [
    control["control"]
    for control in controls
    if control["status"] != "VALID"
]

if invalid_controls:
    if execution_errors:
        print("\nEXECUTION ERRORS")
        display(
            pd.DataFrame(
                execution_errors
            )
        )

    if verification_failures:
        print("\nVERIFICATION FAILURES")
        display(
            pd.DataFrame(
                verification_failures
            )
        )

    raise RuntimeError(
        "CELL 9E-B FAILED. "
        f"Kontrol tidak valid: "
        f"{invalid_controls}"
    )


# ============================================================
# 8. WRITE INDEX, SUMMARY, MANIFEST
# ============================================================

atomic_write_jsonl(
    OCR_INDEX_PATH,
    result_table.to_dict(
        orient="records"
    ),
)

atomic_write_csv(
    OCR_SUMMARY_PATH,
    result_table,
)

ocr_manifest = {
    "schema_version": "1.0.0",
    "status": "PASSED",
    "stage": (
        "PADDLEOCR_CANDIDATE_V2_1"
    ),
    "evaluation_status": (
        "PENDING_CELL_9E_C"
    ),
    "candidate": {
        "version": "2.1",
        "manifest_path": str(
            CANDIDATE_MANIFEST_PATH
        ),
        "manifest_sha256": sha256_file(
            CANDIDATE_MANIFEST_PATH
        ),
        "result_path": str(
            candidate_result_path
        ),
        "result_sha256": sha256_file(
            candidate_result_path
        ),
    },
    "engine": {
        "name": "PaddleOCR",
        "model": "PP-OCRv6",
        "device": "cpu",
        "mkldnn_enabled": False,
        "paddlepaddle_version": version(
            "paddlepaddle"
        ),
        "paddleocr_version": version(
            "paddleocr"
        ),
        "paddlex_version": version(
            "paddlex"
        ),
        "external_api_required": False,
    },
    "execution": {
        "documents": 6,
        "successful_results": 6,
        "failed_results": 0,
        "newly_processed": (
            newly_processed
        ),
        "recovered_results": (
            recovered_results
        ),
    },
    "artifacts": {
        "result_root": str(
            RESULT_ROOT
        ),
        "result_files": 6,
        "index_path": str(
            OCR_INDEX_PATH
        ),
        "summary_path": str(
            OCR_SUMMARY_PATH
        ),
    },
    "integrity": {
        "verified_results": 6,
        "dataset_modifications": 0,
        "validation_opened": 0,
        "test_opened": 0,
    },
}

atomic_write_json(
    OCR_MANIFEST_PATH,
    ocr_manifest,
)

print()
print("Candidate version    : 2.1")
print("OCR model            : PP-OCRv6")
print("Device               : cpu")
print("MKLDNN enabled       : False")
print("Documents            : 6")
print("Successful results   : 6")
print(
    f"Newly processed      : "
    f"{newly_processed}"
)
print(
    f"Recovered results    : "
    f"{recovered_results}"
)
print("Execution errors     : 0")
print(f"Result root          : {RESULT_ROOT}")
print(f"Result index         : {OCR_INDEX_PATH}")
print(f"Result summary       : {OCR_SUMMARY_PATH}")
print(f"OCR manifest         : {OCR_MANIFEST_PATH}")
print(
    f"Manifest SHA-256     : "
    f"{sha256_file(OCR_MANIFEST_PATH)}"
)
print("Evaluation status    : PENDING CELL 9E-C")
print("Dataset modifications: 0")
print("Validation opened    : 0")
print("Test opened          : 0")
print()
print(
    "✅ CELL 9E-B PASSED — PaddleOCR selesai "
    "memproses enam Candidate V2.1. Lanjutkan "
    "ke Cell 9E-C untuk evaluasi ground truth "
    "dan keputusan akhir."
)

Memulai PaddleOCR pada 6 Candidate V2.1...
Result root: /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/preprocessing_candidate_v2_1/ocr_results/paddleocr_ppocrv6

[1/6] INV-SYN-000002 | language=id
         PASSED | regions=14 | bbox=14 | confidence=0.462315 | 68.56s
[2/6] INV-SYN-000036 | language=en
         PASSED | regions=2 | bbox=2 | confidence=0.883579 | 29.82s
[3/6] INV-SYN-000043 | language=id
         PASSED | regions=7 | bbox=7 | confidence=0.520952 | 44.03s
[4/6] INV-SYN-000071 | language=en
         PASSED | regions=3 | bbox=3 | confidence=0.567860 | 27.96s
[5/6] INV-SYN-000082 | language=id
         PASSED | regions=2 | bbox=2 | confidence=0.853030 | 27.53s
[6/6] INV-SYN-000114 | language=en
         PASSED | regions=17 | bbox=17 | confidence=0.434048 | 55.88s


,control,expected,actual,status
0,candidate_status,READY_FOR_OCR,READY_FOR_OCR,VALID
1,candidate_documents,6,6,VALID
2,result_records,6,6,VALID
3,result_files,6,6,VALID
4,verified_results,6,6,VALID
5,nonempty_results,6,6,VALID
6,execution_errors,0,0,VALID
7,template_coverage,6,6,VALID
8,language_coverage,"[en, id]","[en, id]",VALID
9,mkldnn_enabled,False,False,VALID



CANDIDATE V2.1 OCR RESULTS


,sequence_number,document_id,template_id,language,recognized_regions,regions_with_bbox,mean_confidence,seconds,execution,status
0,1,INV-SYN-000002,TPL-01,id,14,14,0.462315,68.563762,NEW,PASSED
1,2,INV-SYN-000036,TPL-02,en,2,2,0.883579,29.821409,NEW,PASSED
2,3,INV-SYN-000043,TPL-03,id,7,7,0.520952,44.026039,NEW,PASSED
3,4,INV-SYN-000071,TPL-04,en,3,3,0.567860,27.957675,NEW,PASSED
4,5,INV-SYN-000082,TPL-05,id,2,2,0.853030,27.525182,NEW,PASSED
5,6,INV-SYN-000114,TPL-06,en,17,17,0.434048,55.883166,NEW,PASSED



Candidate version    : 2.1
OCR model            : PP-OCRv6
Device               : cpu
MKLDNN enabled       : False
Documents            : 6
Successful results   : 6
Newly processed      : 6
Recovered results    : 0
Execution errors     : 0
Result root          : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/preprocessing_candidate_v2_1/ocr_results/paddleocr_ppocrv6
Result index         : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/preprocessing_candidate_v2_1/paddleocr_candidate_v2_1_index.jsonl
Result summary       : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/preprocessing_candidate_v2_1/paddleocr_candidate_v2_1_summary.csv
OCR manifest         : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/preprocessing_candidate_v2_1/paddleocr_candidate_v2_1_manifest.json
Mani

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import re
import unicodedata
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


# ============================================================
# CELL 9E-C — CANDIDATE V2.1 GROUND-TRUTH EVALUATION
#
# - mengevaluasi Candidate V2.1;
# - membandingkan dengan degraded dan Candidate lama;
# - menggunakan 250 anotasi development;
# - keputusan REJECTED bukan execution error;
# - tidak membuka validation/test.
# ============================================================


# ============================================================
# 1. KONFIGURASI
# ============================================================

BUILD_ROOT = Path(
    "/content/drive/MyDrive/InvoiceFlow-AI-Data/interim/"
    "synthetic_v1_build/20260904T150025Z"
)

BENCHMARK_ROOT = BUILD_ROOT / "ocr_benchmark"

OLD_EVALUATION_MANIFEST_PATH = (
    BENCHMARK_ROOT
    / "preprocessing_ocr_robustness"
    / "manifests"
    / "preprocessing_ocr_evaluation_manifest.json"
)

CANDIDATE_ROOT = (
    BENCHMARK_ROOT
    / "preprocessing_candidate_v2_1"
)

CANDIDATE_OCR_MANIFEST_PATH = (
    CANDIDATE_ROOT
    / "paddleocr_candidate_v2_1_manifest.json"
)

GROUND_TRUTH_ROOT = (
    BUILD_ROOT
    / "rendered_dataset"
    / "ground_truth"
    / "development"
)

EVALUATION_ROOT = (
    CANDIDATE_ROOT
    / "evaluation"
)

CANDIDATE_EVALUATION_PATH = (
    EVALUATION_ROOT
    / "candidate_v2_1_evaluation_records.csv"
)

COMBINED_EVALUATION_PATH = (
    EVALUATION_ROOT
    / "all_preprocessing_variants.csv"
)

COMPARISON_PATH = (
    EVALUATION_ROOT
    / "candidate_v2_1_document_comparison.csv"
)

EVALUATION_MANIFEST_PATH = (
    EVALUATION_ROOT
    / "candidate_v2_1_evaluation_manifest.json"
)

EXPECTED_DOCUMENTS = 6
EXPECTED_BASE_RECORDS = 12
EXPECTED_CANDIDATE_RECORDS = 6
EXPECTED_TOTAL_RECORDS = 18

NONINFERIOR_TOLERANCE = 0.01
MEANINGFUL_GAIN = 0.05
MINIMUM_ANNOTATION_COVERAGE = 0.70
MINIMUM_TOKEN_RECALL = 0.70
MAXIMUM_WORSENED_DOCUMENTS = 1

EVALUATION_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 2. FILE HELPERS
# ============================================================

def sha256_file(file_path: Path) -> str:
    digest = hashlib.sha256()

    with file_path.open("rb") as file_handle:
        for chunk in iter(
            lambda: file_handle.read(
                1024 * 1024
            ),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def json_safe(value):
    if isinstance(value, dict):
        return {
            str(key): json_safe(child)
            for key, child in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [
            json_safe(child)
            for child in value
        ]

    if isinstance(value, np.ndarray):
        return value.tolist()

    if isinstance(value, np.generic):
        return value.item()

    if isinstance(value, Path):
        return str(value)

    if pd.isna(value):
        return None

    return value


def atomic_write_text(
    file_path: Path,
    content: str,
) -> None:
    file_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = file_path.with_name(
        file_path.name + ".tmp"
    )

    temporary_path.write_text(
        content,
        encoding="utf-8",
    )

    os.replace(
        temporary_path,
        file_path,
    )


def atomic_write_json(
    file_path: Path,
    content: dict,
) -> None:
    atomic_write_text(
        file_path,
        json.dumps(
            json_safe(content),
            indent=2,
            ensure_ascii=False,
        )
        + "\n",
    )


def atomic_write_csv(
    file_path: Path,
    table: pd.DataFrame,
) -> None:
    temporary_path = file_path.with_name(
        file_path.name + ".tmp"
    )

    table.to_csv(
        temporary_path,
        index=False,
    )

    os.replace(
        temporary_path,
        file_path,
    )


# ============================================================
# 3. TEXT METRICS
# ============================================================

def normalize_text(value) -> str:
    text = unicodedata.normalize(
        "NFKC",
        str(value or ""),
    )

    text = text.translate(
        str.maketrans(
            {
                "–": "-",
                "—": "-",
                "−": "-",
                "·": " ",
                "•": " ",
                "\u00a0": " ",
            }
        )
    )

    text = text.casefold()

    text = re.sub(
        r"[^\w]+",
        " ",
        text,
        flags=re.UNICODE,
    )

    return re.sub(
        r"\s+",
        " ",
        text,
    ).strip()


def compact_text(value) -> str:
    return re.sub(
        r"\s+",
        "",
        normalize_text(value),
    )


def levenshtein_distance(
    reference_sequence,
    prediction_sequence,
) -> int:
    reference = list(reference_sequence)
    prediction = list(
        prediction_sequence
    )

    if len(reference) < len(prediction):
        reference, prediction = (
            prediction,
            reference,
        )

    previous_row = list(
        range(len(prediction) + 1)
    )

    for row_index, reference_value in enumerate(
        reference,
        start=1,
    ):
        current_row = [row_index]

        for column_index, prediction_value in enumerate(
            prediction,
            start=1,
        ):
            current_row.append(
                min(
                    current_row[
                        column_index - 1
                    ]
                    + 1,
                    previous_row[
                        column_index
                    ]
                    + 1,
                    previous_row[
                        column_index - 1
                    ]
                    + (
                        reference_value
                        != prediction_value
                    ),
                )
            )

        previous_row = current_row

    return previous_row[-1]


def safe_error_rate(
    distance: int,
    reference_length: int,
) -> float:
    if reference_length == 0:
        return (
            0.0
            if distance == 0
            else 1.0
        )

    return float(
        distance / reference_length
    )


def token_metrics(
    reference_text: str,
    prediction_text: str,
) -> dict:
    reference_tokens = normalize_text(
        reference_text
    ).split()

    prediction_tokens = normalize_text(
        prediction_text
    ).split()

    reference_counter = Counter(
        reference_tokens
    )

    prediction_counter = Counter(
        prediction_tokens
    )

    matched_tokens = sum(
        (
            reference_counter
            & prediction_counter
        ).values()
    )

    precision = (
        matched_tokens
        / len(prediction_tokens)
        if prediction_tokens
        else 0.0
    )

    recall = (
        matched_tokens
        / len(reference_tokens)
        if reference_tokens
        else 1.0
    )

    f1 = (
        2.0 * precision * recall
        / (precision + recall)
        if precision + recall
        else 0.0
    )

    word_distance = levenshtein_distance(
        reference_tokens,
        prediction_tokens,
    )

    return {
        "reference_tokens": len(
            reference_tokens
        ),
        "prediction_tokens": len(
            prediction_tokens
        ),
        "matched_tokens": int(
            matched_tokens
        ),
        "token_precision": float(
            precision
        ),
        "token_recall": float(recall),
        "token_f1": float(f1),
        "word_distance": int(
            word_distance
        ),
        "word_error_rate": (
            safe_error_rate(
                word_distance,
                len(reference_tokens),
            )
        ),
    }


def annotation_is_present(
    annotation_text: str,
    prediction_text: str,
) -> bool:
    normalized_annotation = normalize_text(
        annotation_text
    )

    normalized_prediction = normalize_text(
        prediction_text
    )

    if not normalized_annotation:
        return True

    return bool(
        f" {normalized_annotation} "
        in f" {normalized_prediction} "
    )


# ============================================================
# 4. GROUND TRUTH
# ============================================================

def locate_ground_truth(
    document_id: str,
    template_id: str,
) -> Path:
    template_root = (
        GROUND_TRUTH_ROOT
        / template_id
    )

    candidates = [
        path
        for path in template_root.rglob(
            f"*{document_id}*.json"
        )
        if (
            path.stem == document_id
            or path.stem.startswith(
                document_id + "_"
            )
        )
    ]

    if len(candidates) != 1:
        raise RuntimeError(
            f"Ground truth {document_id}/"
            f"{template_id} ditemukan "
            f"{len(candidates)} kali."
        )

    return candidates[0]


def ground_truth_identity(
    ground_truth: dict,
):
    document = ground_truth.get(
        "document",
        {},
    )

    canonical = ground_truth.get(
        "canonical",
        {},
    )

    return (
        document.get("document_id")
        or canonical.get("document_id"),
        document.get("template_id")
        or canonical.get("template_id"),
    )


def annotation_sort_key(
    annotation: dict,
):
    bbox = annotation.get(
        "bbox_points",
        [],
    )

    page_number = int(
        annotation.get(
            "page_number",
            1,
        )
    )

    if len(bbox) == 4:
        return (
            page_number,
            float(bbox[1]),
            float(bbox[0]),
        )

    return (
        page_number,
        float("inf"),
        float("inf"),
    )


def load_reference(
    document_id: str,
    template_id: str,
) -> dict:
    path = locate_ground_truth(
        document_id,
        template_id,
    )

    ground_truth = json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )

    actual_document_id, actual_template_id = (
        ground_truth_identity(
            ground_truth
        )
    )

    if (
        actual_document_id != document_id
        or actual_template_id != template_id
    ):
        raise RuntimeError(
            "Identitas ground truth tidak cocok."
        )

    annotations = sorted(
        ground_truth.get(
            "annotations",
            [],
        ),
        key=annotation_sort_key,
    )

    annotation_texts = [
        str(
            annotation.get("text", "")
        ).strip()
        for annotation in annotations
        if str(
            annotation.get("text", "")
        ).strip()
    ]

    if not annotation_texts:
        raise RuntimeError(
            "Ground truth tidak memiliki "
            f"teks: {document_id}"
        )

    return {
        "path": path,
        "sha256": sha256_file(path),
        "annotation_texts": (
            annotation_texts
        ),
        "reference_text": "\n".join(
            annotation_texts
        ),
    }


# ============================================================
# 5. PREFLIGHT
# ============================================================

required_paths = [
    OLD_EVALUATION_MANIFEST_PATH,
    CANDIDATE_OCR_MANIFEST_PATH,
    GROUND_TRUTH_ROOT,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not path.exists()
]

if missing_paths:
    raise FileNotFoundError(
        "Checkpoint belum lengkap:\n"
        + "\n".join(missing_paths)
    )

old_manifest = json.loads(
    OLD_EVALUATION_MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)

candidate_ocr_manifest = json.loads(
    CANDIDATE_OCR_MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)

if old_manifest.get("status") != "PASSED":
    raise RuntimeError(
        "Evaluasi lama belum PASSED."
    )

if (
    old_manifest.get(
        "preprocessing_decision"
    )
    != "REJECTED"
):
    raise RuntimeError(
        "Candidate lama belum REJECTED."
    )

if (
    candidate_ocr_manifest.get("status")
    != "PASSED"
):
    raise RuntimeError(
        "OCR Candidate V2.1 belum PASSED."
    )

old_records_path = Path(
    old_manifest[
        "artifacts"
    ]["evaluation_records"]
)

candidate_summary_path = Path(
    candidate_ocr_manifest[
        "artifacts"
    ]["summary_path"]
)

old_table = pd.read_csv(
    old_records_path
)

candidate_summary = pd.read_csv(
    candidate_summary_path
)

if len(old_table) != EXPECTED_BASE_RECORDS:
    raise RuntimeError(
        "Baseline evaluation harus 12 record."
    )

if len(candidate_summary) != EXPECTED_CANDIDATE_RECORDS:
    raise RuntimeError(
        "Candidate summary harus 6 record."
    )


# ============================================================
# 6. EVALUATE CANDIDATE V2.1
# ============================================================

reference_cache = {}
candidate_records = []

checksum_mismatches = []
identity_mismatches = []
evaluation_errors = []

print(
    "Mengevaluasi Candidate V2.1 terhadap "
    "ground truth development...\n"
)

for row in candidate_summary.itertuples(
    index=False
):
    try:
        result_path = Path(
            row.result_path
        )

        actual_checksum = sha256_file(
            result_path
        )

        if (
            actual_checksum
            != row.result_sha256
        ):
            checksum_mismatches.append(
                {
                    "document_id": (
                        row.document_id
                    ),
                    "expected": (
                        row.result_sha256
                    ),
                    "actual": (
                        actual_checksum
                    ),
                }
            )

        result = json.loads(
            result_path.read_text(
                encoding="utf-8"
            )
        )

        document = result.get(
            "document",
            {},
        )

        input_section = result.get(
            "input",
            {},
        )

        output = result.get(
            "output",
            {},
        )

        identity_valid = bool(
            result.get("status") == "PASSED"
            and document.get("document_id")
            == row.document_id
            and document.get("template_id")
            == row.template_id
            and document.get("language")
            == row.language
            and input_section.get(
                "candidate_version"
            )
            == "2.1"
        )

        if not identity_valid:
            identity_mismatches.append(
                {
                    "document_id": (
                        row.document_id
                    ),
                    "result_path": str(
                        result_path
                    ),
                }
            )

        cache_key = (
            row.document_id,
            row.template_id,
        )

        if cache_key not in reference_cache:
            reference_cache[cache_key] = (
                load_reference(
                    row.document_id,
                    row.template_id,
                )
            )

        reference = reference_cache[
            cache_key
        ]

        prediction_text = str(
            output.get(
                "full_text",
                "",
            )
        ).strip()

        if not prediction_text:
            prediction_text = "\n".join(
                str(line.get("text", ""))
                for line in output.get(
                    "lines",
                    [],
                )
            ).strip()

        reference_text = reference[
            "reference_text"
        ]

        reference_compact = compact_text(
            reference_text
        )

        prediction_compact = compact_text(
            prediction_text
        )

        character_distance = (
            levenshtein_distance(
                reference_compact,
                prediction_compact,
            )
        )

        character_error_rate = (
            safe_error_rate(
                character_distance,
                len(reference_compact),
            )
        )

        calculated_token_metrics = (
            token_metrics(
                reference_text,
                prediction_text,
            )
        )

        matched_annotations = sum(
            annotation_is_present(
                annotation_text,
                prediction_text,
            )
            for annotation_text
            in reference[
                "annotation_texts"
            ]
        )

        annotation_count = len(
            reference[
                "annotation_texts"
            ]
        )

        annotation_coverage = (
            matched_annotations
            / annotation_count
        )

        candidate_records.append(
            {
                "document_id": (
                    row.document_id
                ),
                "template_id": (
                    row.template_id
                ),
                "language": row.language,
                "variant": (
                    "candidate_v2_1"
                ),
                "ground_truth_path": str(
                    reference["path"]
                ),
                "ground_truth_sha256": (
                    reference["sha256"]
                ),
                "result_path": str(
                    result_path
                ),
                "result_sha256": (
                    actual_checksum
                ),
                "annotation_count": (
                    annotation_count
                ),
                "matched_annotations": int(
                    matched_annotations
                ),
                "annotation_text_coverage": (
                    float(
                        annotation_coverage
                    )
                ),
                "reference_characters": len(
                    reference_compact
                ),
                "prediction_characters": len(
                    prediction_compact
                ),
                "character_distance": int(
                    character_distance
                ),
                "character_error_rate": (
                    float(
                        character_error_rate
                    )
                ),
                **calculated_token_metrics,
                "recognized_regions": int(
                    output.get(
                        "recognized_regions",
                        0,
                    )
                ),
                "mean_confidence": (
                    output.get(
                        "mean_confidence"
                    )
                ),
                "prediction_text": (
                    prediction_text
                ),
                "status": "VALID",
            }
        )

        print(
            f"{row.document_id} | "
            f"regions="
            f"{output.get('recognized_regions', 0):2d} | "
            f"coverage="
            f"{annotation_coverage:.4f} | "
            f"token_recall="
            f"{calculated_token_metrics['token_recall']:.4f} | "
            f"CER={character_error_rate:.4f}"
        )

    except Exception as error:
        evaluation_errors.append(
            {
                "document_id": (
                    row.document_id
                ),
                "template_id": (
                    row.template_id
                ),
                "error_type": (
                    type(error).__name__
                ),
                "error": str(error)[:500],
            }
        )

        print(
            f"{row.document_id} | ERROR: "
            f"{type(error).__name__}: {error}"
        )


# ============================================================
# 7. COMBINE THREE VARIANTS
# ============================================================

candidate_table = pd.DataFrame(
    candidate_records
)

if len(candidate_table) != 6:
    raise RuntimeError(
        "Candidate evaluation tidak lengkap: "
        f"{len(candidate_table)}/6."
    )

all_variants_table = pd.concat(
    [
        old_table,
        candidate_table,
    ],
    ignore_index=True,
    sort=False,
)

all_variants_table = (
    all_variants_table
    .sort_values(
        [
            "document_id",
            "variant",
        ]
    )
    .reset_index(drop=True)
)


# ============================================================
# 8. AGGREGATE METRICS
# ============================================================

aggregate_records = []

for variant in (
    "degraded",
    "processed",
    "candidate_v2_1",
):
    variant_table = all_variants_table[
        all_variants_table["variant"]
        == variant
    ]

    total_annotations = int(
        variant_table[
            "annotation_count"
        ].sum()
    )

    total_matched_annotations = int(
        variant_table[
            "matched_annotations"
        ].sum()
    )

    total_reference_characters = int(
        variant_table[
            "reference_characters"
        ].sum()
    )

    total_character_distance = int(
        variant_table[
            "character_distance"
        ].sum()
    )

    total_reference_tokens = int(
        variant_table[
            "reference_tokens"
        ].sum()
    )

    total_word_distance = int(
        variant_table[
            "word_distance"
        ].sum()
    )

    aggregate_records.append(
        {
            "variant": variant,
            "documents": len(
                variant_table
            ),
            "annotations": (
                total_annotations
            ),
            "matched_annotations": (
                total_matched_annotations
            ),
            "annotation_text_coverage": (
                total_matched_annotations
                / total_annotations
            ),
            "token_precision": float(
                variant_table[
                    "token_precision"
                ].mean()
            ),
            "token_recall": float(
                variant_table[
                    "token_recall"
                ].mean()
            ),
            "token_f1": float(
                variant_table[
                    "token_f1"
                ].mean()
            ),
            "micro_cer": safe_error_rate(
                total_character_distance,
                total_reference_characters,
            ),
            "micro_wer": safe_error_rate(
                total_word_distance,
                total_reference_tokens,
            ),
            "mean_recognized_regions": float(
                variant_table[
                    "recognized_regions"
                ].mean()
            ),
            "mean_confidence": float(
                variant_table[
                    "mean_confidence"
                ].mean()
            ),
        }
    )

aggregate_table = pd.DataFrame(
    aggregate_records
)

degraded_metrics = (
    aggregate_table[
        aggregate_table["variant"]
        == "degraded"
    ]
    .iloc[0]
)

previous_metrics = (
    aggregate_table[
        aggregate_table["variant"]
        == "processed"
    ]
    .iloc[0]
)

candidate_metrics = (
    aggregate_table[
        aggregate_table["variant"]
        == "candidate_v2_1"
    ]
    .iloc[0]
)


# ============================================================
# 9. DOCUMENT COMPARISON
# ============================================================

comparison_columns = [
    "document_id",
    "template_id",
    "language",
    "annotation_text_coverage",
    "token_recall",
    "token_f1",
    "character_error_rate",
    "recognized_regions",
]

degraded_documents = (
    all_variants_table[
        all_variants_table["variant"]
        == "degraded"
    ][comparison_columns]
    .copy()
)

candidate_documents = (
    all_variants_table[
        all_variants_table["variant"]
        == "candidate_v2_1"
    ][
        [
            "document_id",
            "annotation_text_coverage",
            "token_recall",
            "token_f1",
            "character_error_rate",
            "recognized_regions",
        ]
    ]
    .copy()
)

degraded_documents = (
    degraded_documents.rename(
        columns={
            column: f"degraded_{column}"
            for column in comparison_columns
            if column not in {
                "document_id",
                "template_id",
                "language",
            }
        }
    )
)

candidate_documents = (
    candidate_documents.rename(
        columns={
            column: f"candidate_{column}"
            for column
            in candidate_documents.columns
            if column != "document_id"
        }
    )
)

comparison_table = (
    degraded_documents.merge(
        candidate_documents,
        on="document_id",
        how="inner",
        validate="one_to_one",
    )
)

comparison_table[
    "annotation_coverage_gain"
] = (
    comparison_table[
        "candidate_annotation_text_coverage"
    ]
    - comparison_table[
        "degraded_annotation_text_coverage"
    ]
)

comparison_table[
    "token_recall_gain"
] = (
    comparison_table[
        "candidate_token_recall"
    ]
    - comparison_table[
        "degraded_token_recall"
    ]
)

comparison_table[
    "token_f1_gain"
] = (
    comparison_table[
        "candidate_token_f1"
    ]
    - comparison_table[
        "degraded_token_f1"
    ]
)

comparison_table[
    "cer_reduction"
] = (
    comparison_table[
        "degraded_character_error_rate"
    ]
    - comparison_table[
        "candidate_character_error_rate"
    ]
)


def classify_effect(row) -> str:
    meaningful_improvement = bool(
        row["annotation_coverage_gain"]
        >= MEANINGFUL_GAIN
        or row["token_recall_gain"]
        >= MEANINGFUL_GAIN
        or row["cer_reduction"]
        >= MEANINGFUL_GAIN
    )

    meaningful_worsening = bool(
        row["annotation_coverage_gain"]
        <= -MEANINGFUL_GAIN
        or row["token_recall_gain"]
        <= -MEANINGFUL_GAIN
        or row["cer_reduction"]
        <= -MEANINGFUL_GAIN
    )

    if (
        meaningful_improvement
        and not meaningful_worsening
    ):
        return "IMPROVED"

    if meaningful_worsening:
        return "WORSENED"

    return "NO_MEANINGFUL_CHANGE"


comparison_table[
    "candidate_effect"
] = comparison_table.apply(
    classify_effect,
    axis=1,
)


# ============================================================
# 10. ACCEPTANCE DECISION
# ============================================================

annotation_gain = float(
    candidate_metrics[
        "annotation_text_coverage"
    ]
    - degraded_metrics[
        "annotation_text_coverage"
    ]
)

token_recall_gain = float(
    candidate_metrics["token_recall"]
    - degraded_metrics["token_recall"]
)

token_f1_gain = float(
    candidate_metrics["token_f1"]
    - degraded_metrics["token_f1"]
)

cer_reduction = float(
    degraded_metrics["micro_cer"]
    - candidate_metrics["micro_cer"]
)

noninferior = bool(
    annotation_gain
    >= -NONINFERIOR_TOLERANCE
    and token_recall_gain
    >= -NONINFERIOR_TOLERANCE
    and cer_reduction
    >= -NONINFERIOR_TOLERANCE
)

meaningful_improvement = bool(
    annotation_gain >= MEANINGFUL_GAIN
    or token_recall_gain
    >= MEANINGFUL_GAIN
    or cer_reduction
    >= MEANINGFUL_GAIN
)

quality_floor_passed = bool(
    candidate_metrics[
        "annotation_text_coverage"
    ]
    >= MINIMUM_ANNOTATION_COVERAGE
    and candidate_metrics[
        "token_recall"
    ]
    >= MINIMUM_TOKEN_RECALL
)

worsened_documents = int(
    (
        comparison_table[
            "candidate_effect"
        ]
        == "WORSENED"
    ).sum()
)

candidate_accepted = bool(
    noninferior
    and meaningful_improvement
    and quality_floor_passed
    and worsened_documents
    <= MAXIMUM_WORSENED_DOCUMENTS
)

candidate_decision = (
    "ACCEPTED"
    if candidate_accepted
    else "REJECTED"
)


# ============================================================
# 11. TECHNICAL CONTROLS
# ============================================================

ground_truth_changes = []

for reference in reference_cache.values():
    if (
        sha256_file(reference["path"])
        != reference["sha256"]
    ):
        ground_truth_changes.append(
            str(reference["path"])
        )

controls = [
    {
        "control": "old_evaluation_status",
        "expected": "PASSED",
        "actual": old_manifest.get(
            "status"
        ),
    },
    {
        "control": "previous_decision",
        "expected": "REJECTED",
        "actual": old_manifest.get(
            "preprocessing_decision"
        ),
    },
    {
        "control": "candidate_ocr_status",
        "expected": "PASSED",
        "actual": (
            candidate_ocr_manifest.get(
                "status"
            )
        ),
    },
    {
        "control": "baseline_records",
        "expected": 12,
        "actual": len(old_table),
    },
    {
        "control": "candidate_records",
        "expected": 6,
        "actual": len(candidate_table),
    },
    {
        "control": "total_variant_records",
        "expected": 18,
        "actual": len(
            all_variants_table
        ),
    },
    {
        "control": "ground_truth_documents",
        "expected": 6,
        "actual": len(reference_cache),
    },
    {
        "control": "document_comparisons",
        "expected": 6,
        "actual": len(comparison_table),
    },
    {
        "control": "checksum_mismatches",
        "expected": 0,
        "actual": len(
            checksum_mismatches
        ),
    },
    {
        "control": "identity_mismatches",
        "expected": 0,
        "actual": len(
            identity_mismatches
        ),
    },
    {
        "control": "evaluation_errors",
        "expected": 0,
        "actual": len(
            evaluation_errors
        ),
    },
    {
        "control": "ground_truth_changes",
        "expected": 0,
        "actual": len(
            ground_truth_changes
        ),
    },
    {
        "control": "validation_opened",
        "expected": 0,
        "actual": 0,
    },
    {
        "control": "test_opened",
        "expected": 0,
        "actual": 0,
    },
]

for control in controls:
    control["status"] = (
        "VALID"
        if control["expected"]
        == control["actual"]
        else "INVALID"
    )

control_table = pd.DataFrame(
    controls
)

display(control_table)

print("\nAGGREGATE VARIANT COMPARISON")
display(aggregate_table)

print("\nCANDIDATE V2.1 DOCUMENT COMPARISON")
display(
    comparison_table[
        [
            "document_id",
            "template_id",
            "language",
            "degraded_annotation_text_coverage",
            "candidate_annotation_text_coverage",
            "annotation_coverage_gain",
            "degraded_token_recall",
            "candidate_token_recall",
            "token_recall_gain",
            "degraded_character_error_rate",
            "candidate_character_error_rate",
            "cer_reduction",
            "candidate_effect",
        ]
    ]
)

invalid_controls = [
    control["control"]
    for control in controls
    if control["status"] != "VALID"
]

if invalid_controls:
    if evaluation_errors:
        print("\nEVALUATION ERRORS")
        display(
            pd.DataFrame(
                evaluation_errors
            )
        )

    raise RuntimeError(
        "CELL 9E-C TECHNICAL EVALUATION FAILED. "
        f"Kontrol tidak valid: "
        f"{invalid_controls}"
    )


# ============================================================
# 12. SAVE EVALUATION
# ============================================================

atomic_write_csv(
    CANDIDATE_EVALUATION_PATH,
    candidate_table,
)

atomic_write_csv(
    COMBINED_EVALUATION_PATH,
    all_variants_table,
)

atomic_write_csv(
    COMPARISON_PATH,
    comparison_table,
)

evaluation_manifest = {
    "schema_version": "2.1.0",
    "status": "PASSED",
    "stage": (
        "CANDIDATE_V2_1_GROUND_TRUTH_EVALUATION"
    ),
    "candidate_decision": (
        candidate_decision
    ),
    "decision_is_execution_error": False,
    "scope": {
        "split": "development",
        "documents": 6,
        "templates": 6,
        "ground_truth_annotations": int(
            candidate_metrics[
                "annotations"
            ]
        ),
        "variants": [
            "degraded",
            "processed",
            "candidate_v2_1",
        ],
        "validation_opened": 0,
        "test_opened": 0,
    },
    "acceptance_policy": {
        "noninferior_tolerance": (
            NONINFERIOR_TOLERANCE
        ),
        "meaningful_gain": (
            MEANINGFUL_GAIN
        ),
        "minimum_annotation_coverage": (
            MINIMUM_ANNOTATION_COVERAGE
        ),
        "minimum_token_recall": (
            MINIMUM_TOKEN_RECALL
        ),
        "maximum_worsened_documents": (
            MAXIMUM_WORSENED_DOCUMENTS
        ),
    },
    "aggregate_metrics": (
        aggregate_table.to_dict(
            orient="records"
        )
    ),
    "decision_evidence": {
        "annotation_coverage_gain": (
            annotation_gain
        ),
        "token_recall_gain": (
            token_recall_gain
        ),
        "token_f1_gain": token_f1_gain,
        "cer_reduction": cer_reduction,
        "noninferior": noninferior,
        "meaningful_improvement": (
            meaningful_improvement
        ),
        "quality_floor_passed": (
            quality_floor_passed
        ),
        "worsened_documents": (
            worsened_documents
        ),
    },
    "artifacts": {
        "candidate_evaluation": str(
            CANDIDATE_EVALUATION_PATH
        ),
        "combined_variants": str(
            COMBINED_EVALUATION_PATH
        ),
        "document_comparison": str(
            COMPARISON_PATH
        ),
        "candidate_ocr_manifest": str(
            CANDIDATE_OCR_MANIFEST_PATH
        ),
    },
    "integrity": {
        "checksum_mismatches": 0,
        "identity_mismatches": 0,
        "ground_truth_changes": 0,
        "dataset_modifications": 0,
    },
}

atomic_write_json(
    EVALUATION_MANIFEST_PATH,
    evaluation_manifest,
)

print()
print("Candidate version      : 2.1")
print(
    "Ground-truth annotations: "
    f"{int(candidate_metrics['annotations'])}"
)
print(
    "Degraded coverage     : "
    f"{degraded_metrics['annotation_text_coverage']:.6f}"
)
print(
    "Previous coverage     : "
    f"{previous_metrics['annotation_text_coverage']:.6f}"
)
print(
    "Candidate coverage    : "
    f"{candidate_metrics['annotation_text_coverage']:.6f}"
)
print(
    "Candidate token recall: "
    f"{candidate_metrics['token_recall']:.6f}"
)
print(
    "Candidate micro CER   : "
    f"{candidate_metrics['micro_cer']:.6f}"
)
print(
    f"Worsened documents    : "
    f"{worsened_documents}"
)
print(
    f"Meaningful improvement: "
    f"{meaningful_improvement}"
)
print(
    f"Quality floor passed  : "
    f"{quality_floor_passed}"
)
print(
    f"Candidate decision    : "
    f"{candidate_decision}"
)
print(
    f"Evaluation manifest   : "
    f"{EVALUATION_MANIFEST_PATH}"
)
print(
    "Manifest SHA-256      : "
    f"{sha256_file(EVALUATION_MANIFEST_PATH)}"
)
print("Dataset modifications : 0")
print("Validation opened      : 0")
print("Test opened            : 0")
print()
print(
    "✅ CELL 9E-C PASSED — Candidate V2.1 "
    "selesai dievaluasi. Keputusan kualitas: "
    f"{candidate_decision}."
)

Mengevaluasi Candidate V2.1 terhadap ground truth development...

INV-SYN-000002 | regions=14 | coverage=0.0741 | token_recall=0.0424 | CER=0.9368
INV-SYN-000036 | regions= 2 | coverage=0.0000 | token_recall=0.0000 | CER=0.9899
INV-SYN-000043 | regions= 7 | coverage=0.0000 | token_recall=0.0168 | CER=0.9672
INV-SYN-000071 | regions= 3 | coverage=0.0638 | token_recall=0.0060 | CER=0.9805
INV-SYN-000082 | regions= 2 | coverage=0.0000 | token_recall=0.0000 | CER=0.9901
INV-SYN-000114 | regions=17 | coverage=0.0784 | token_recall=0.0167 | CER=0.9769


,control,expected,actual,status
0,old_evaluation_status,PASSED,PASSED,VALID
1,previous_decision,REJECTED,REJECTED,VALID
2,candidate_ocr_status,PASSED,PASSED,VALID
3,baseline_records,12,12,VALID
4,candidate_records,6,6,VALID
5,total_variant_records,18,18,VALID
6,ground_truth_documents,6,6,VALID
7,document_comparisons,6,6,VALID
8,checksum_mismatches,0,0,VALID
9,identity_mismatches,0,0,VALID



AGGREGATE VARIANT COMPARISON


,variant,documents,annotations,matched_annotations,annotation_text_coverage,token_precision,token_recall,token_f1,micro_cer,micro_wer,mean_recognized_regions,mean_confidence
0,degraded,6,250,0,0.000,0.083333,0.002801,0.005420,0.975696,0.997888,1.666667,0.904741
1,processed,6,250,0,0.000,0.060185,0.002338,0.004436,0.974058,0.997888,1.833333,0.891113
2,candidate_v2_1,6,250,9,0.036,0.171024,0.013633,0.024579,0.976516,0.988384,7.500000,0.620297



CANDIDATE V2.1 DOCUMENT COMPARISON


,document_id,template_id,language,degraded_annotation_text_coverage,candidate_annotation_text_coverage,annotation_coverage_gain,degraded_token_recall,candidate_token_recall,token_recall_gain,degraded_character_error_rate,candidate_character_error_rate,cer_reduction,candidate_effect
0,INV-SYN-000002,TPL-01,id,0.0,0.074074,0.074074,0.000000,0.042373,4.237288e-02,0.954853,0.936795,1.805869e-02,IMPROVED
1,INV-SYN-000036,TPL-02,en,0.0,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.953890,0.989914,-3.602305e-02,NO_MEANINGFUL_CHANGE
2,INV-SYN-000043,TPL-03,id,0.0,0.000000,0.000000,0.016807,0.016807,3.122502e-17,0.965066,0.967249,-2.183406e-03,NO_MEANINGFUL_CHANGE
3,INV-SYN-000071,TPL-04,en,0.0,0.063830,0.063830,0.000000,0.005952,5.952381e-03,0.989489,0.980480,9.009009e-03,IMPROVED
4,INV-SYN-000082,TPL-05,id,0.0,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.990099,0.990099,-1.110223e-16,NO_MEANINGFUL_CHANGE
5,INV-SYN-000114,TPL-06,en,0.0,0.078431,0.078431,0.000000,0.016667,1.666667e-02,0.989914,0.976945,1.296830e-02,IMPROVED



Candidate version      : 2.1
Ground-truth annotations: 250
Degraded coverage     : 0.000000
Previous coverage     : 0.000000
Candidate coverage    : 0.036000
Candidate token recall: 0.013633
Candidate micro CER   : 0.976516
Worsened documents    : 0
Meaningful improvement: False
Quality floor passed  : False
Candidate decision    : REJECTED
Evaluation manifest   : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/preprocessing_candidate_v2_1/evaluation/candidate_v2_1_evaluation_manifest.json
Manifest SHA-256      : 069e8f661646989a203693ae4f156510ac0773bd8551f0d3c27434e27e134000
Dataset modifications : 0
Validation opened      : 0
Test opened            : 0

✅ CELL 9E-C PASSED — Candidate V2.1 selesai dievaluasi. Keputusan kualitas: REJECTED.


In [ ]:
from __future__ import annotations

import hashlib
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


# ============================================================
# CELL 9F — FREEZE PREPROCESSING DECISION AND ROUTING POLICY
# ============================================================

BUILD_ROOT = Path(
    "/content/drive/MyDrive/InvoiceFlow-AI-Data/interim/"
    "synthetic_v1_build/20260904T150025Z"
)

BENCHMARK_ROOT = BUILD_ROOT / "ocr_benchmark"
CANDIDATE_ROOT = (
    BENCHMARK_ROOT
    / "preprocessing_candidate_v2_1"
)

INPUT_PATHS = {
    "ocr_evaluation": (
        BENCHMARK_ROOT
        / "manifests"
        / "ocr_evaluation_manifest.json"
    ),
    "ocr_routing_decision": (
        BENCHMARK_ROOT
        / "manifests"
        / "ocr_routing_decision.json"
    ),
    "preprocessing_calibration": (
        BENCHMARK_ROOT
        / "manifests"
        / "preprocessing_calibration.json"
    ),
    "old_evaluation": (
        BENCHMARK_ROOT
        / "preprocessing_ocr_robustness"
        / "manifests"
        / "preprocessing_ocr_evaluation_manifest.json"
    ),
    "candidate_preprocessing": (
        CANDIDATE_ROOT
        / "preprocessing_candidate_v2_1_manifest.json"
    ),
    "candidate_ocr": (
        CANDIDATE_ROOT
        / "paddleocr_candidate_v2_1_manifest.json"
    ),
    "candidate_evaluation": (
        CANDIDATE_ROOT
        / "evaluation"
        / "candidate_v2_1_evaluation_manifest.json"
    ),
}

POLICY_PATH = (
    BENCHMARK_ROOT
    / "manifests"
    / "preprocessing_routing_policy_v1.json"
)

EXPECTED_DOCUMENTS = 6
EXPECTED_ANNOTATIONS = 250

EXPECTED_VARIANTS = {
    "degraded",
    "processed",
    "candidate_v2_1",
}


# ============================================================
# HELPERS
# ============================================================

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def load_json(path: Path) -> dict:
    with path.open(
        "r",
        encoding="utf-8",
    ) as handle:
        value = json.load(handle)

    if not isinstance(value, dict):
        raise TypeError(
            f"JSON harus berupa object: {path}"
        )

    return value


def json_safe(value):
    if isinstance(value, dict):
        return {
            str(key): json_safe(child)
            for key, child in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [
            json_safe(child)
            for child in value
        ]

    if isinstance(value, np.ndarray):
        return value.tolist()

    if isinstance(value, np.generic):
        return value.item()

    if isinstance(value, Path):
        return str(value)

    return value


def canonical_json(value: dict) -> str:
    return json.dumps(
        json_safe(value),
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )


def atomic_write_json(
    path: Path,
    value: dict,
) -> None:
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            json_safe(value),
            indent=2,
            ensure_ascii=False,
        )
        + "\n",
        encoding="utf-8",
    )

    os.replace(
        temporary_path,
        path,
    )


def get_variant_metrics(
    manifest: dict,
    variant: str,
) -> dict:
    matches = [
        row
        for row in manifest.get(
            "aggregate_metrics",
            [],
        )
        if row.get("variant") == variant
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Metric variant {variant!r} "
            f"ditemukan {len(matches)} kali."
        )

    return matches[0]


# ============================================================
# PREFLIGHT DAN PEMBACAAN BUKTI
# ============================================================

missing_paths = [
    str(path)
    for path in INPUT_PATHS.values()
    if not path.is_file()
]

if missing_paths:
    raise FileNotFoundError(
        "Checkpoint eksperimen belum lengkap:\n"
        + "\n".join(missing_paths)
    )

checksums_before = {
    name: sha256_file(path)
    for name, path in INPUT_PATHS.items()
}

manifests = {
    name: load_json(path)
    for name, path in INPUT_PATHS.items()
}

old_evaluation = manifests[
    "old_evaluation"
]

candidate_preprocessing = manifests[
    "candidate_preprocessing"
]

candidate_ocr = manifests[
    "candidate_ocr"
]

candidate_evaluation = manifests[
    "candidate_evaluation"
]

scope = candidate_evaluation.get(
    "scope",
    {},
)

acceptance_policy = (
    candidate_evaluation.get(
        "acceptance_policy",
        {},
    )
)

decision_evidence = (
    candidate_evaluation.get(
        "decision_evidence",
        {},
    )
)

degraded_metrics = get_variant_metrics(
    candidate_evaluation,
    "degraded",
)

previous_metrics = get_variant_metrics(
    candidate_evaluation,
    "processed",
)

candidate_metrics = get_variant_metrics(
    candidate_evaluation,
    "candidate_v2_1",
)

variant_names = {
    row.get("variant")
    for row in candidate_evaluation.get(
        "aggregate_metrics",
        [],
    )
}

candidate_coverage = float(
    candidate_metrics.get(
        "annotation_text_coverage",
        0.0,
    )
)

candidate_token_recall = float(
    candidate_metrics.get(
        "token_recall",
        0.0,
    )
)

candidate_micro_cer = float(
    candidate_metrics.get(
        "micro_cer",
        1.0,
    )
)

minimum_coverage = float(
    acceptance_policy.get(
        "minimum_annotation_coverage",
        0.70,
    )
)

minimum_token_recall = float(
    acceptance_policy.get(
        "minimum_token_recall",
        0.70,
    )
)

independent_quality_floor = bool(
    candidate_coverage >= minimum_coverage
    and candidate_token_recall
    >= minimum_token_recall
)


# ============================================================
# ROUTING YANG DIIZINKAN
# ============================================================

routing_records = [
    {
        "route_id": "NATIVE_PDF_TEXT",
        "condition": (
            "PDF has a usable native text layer"
        ),
        "engine": "PyMuPDF",
        "preprocessing": "NONE",
        "action": "DIRECT_TEXT_EXTRACTION",
        "development_status": "APPROVED",
        "production_status": (
            "REQUIRES_FIELD_EXTRACTION_VALIDATION"
        ),
    },
    {
        "route_id": (
            "CLEAN_OR_ACCEPTABLE_IMAGE"
        ),
        "condition": (
            "No usable text layer and image "
            "passes calibrated quality checks"
        ),
        "engine": (
            "PaddleOCR PP-OCRv6 CPU"
        ),
        "preprocessing": "NONE",
        "action": "DIRECT_OCR",
        "development_status": "APPROVED",
        "production_status": (
            "REQUIRES_REAL_SCAN_VALIDATION"
        ),
    },
    {
        "route_id": (
            "SEVERELY_DEGRADED_IMAGE"
        ),
        "condition": (
            "Image fails quality checks or OCR "
            "output is empty, sparse, or unreliable"
        ),
        "engine": "NONE_AUTOMATICALLY",
        "preprocessing": (
            "CANDIDATE_V2_1_NOT_PROMOTED"
        ),
        "action": (
            "REQUEST_RESCAN_OR_MANUAL_REVIEW"
        ),
        "development_status": (
            "SAFETY_ROUTE"
        ),
        "production_status": (
            "BLOCK_AUTOMATIC_EXPORT"
        ),
    },
]

approved_routes = sum(
    row["development_status"]
    == "APPROVED"
    for row in routing_records
)

safety_routes = sum(
    row["development_status"]
    == "SAFETY_ROUTE"
    for row in routing_records
)


# ============================================================
# PRE-FREEZE AUDIT
# ============================================================

controls = [
    (
        "old_evaluation_status",
        "PASSED",
        old_evaluation.get("status"),
    ),
    (
        "old_candidate_decision",
        "REJECTED",
        old_evaluation.get(
            "preprocessing_decision"
        ),
    ),
    (
        "candidate_preprocessing_status",
        "READY_FOR_OCR",
        candidate_preprocessing.get(
            "status"
        ),
    ),
    (
        "candidate_ocr_status",
        "PASSED",
        candidate_ocr.get("status"),
    ),
    (
        "candidate_evaluation_status",
        "PASSED",
        candidate_evaluation.get(
            "status"
        ),
    ),
    (
        "candidate_decision",
        "REJECTED",
        candidate_evaluation.get(
            "candidate_decision"
        ),
    ),
    (
        "quality_floor_passed",
        False,
        independent_quality_floor,
    ),
    (
        "manifest_quality_floor_passed",
        False,
        decision_evidence.get(
            "quality_floor_passed"
        ),
    ),
    (
        "evaluation_documents",
        EXPECTED_DOCUMENTS,
        int(
            scope.get(
                "documents",
                -1,
            )
        ),
    ),
    (
        "ground_truth_annotations",
        EXPECTED_ANNOTATIONS,
        int(
            scope.get(
                "ground_truth_annotations",
                -1,
            )
        ),
    ),
    (
        "evaluated_variants",
        sorted(EXPECTED_VARIANTS),
        sorted(variant_names),
    ),
    (
        "approved_processing_routes",
        2,
        approved_routes,
    ),
    (
        "safety_routes",
        1,
        safety_routes,
    ),
    (
        "validation_opened",
        0,
        int(
            scope.get(
                "validation_opened",
                -1,
            )
        ),
    ),
    (
        "test_opened",
        0,
        int(
            scope.get(
                "test_opened",
                -1,
            )
        ),
    ),
]

control_records = [
    {
        "control": name,
        "expected": expected,
        "actual": actual,
        "status": (
            "VALID"
            if expected == actual
            else "INVALID"
        ),
    }
    for name, expected, actual
    in controls
]

control_table = pd.DataFrame(
    control_records
)

routing_table = pd.DataFrame(
    routing_records
)

display(control_table)
display(routing_table)

invalid_controls = [
    row["control"]
    for row in control_records
    if row["status"] != "VALID"
]

if invalid_controls:
    raise RuntimeError(
        "CELL 9F PRE-FREEZE AUDIT FAILED. "
        f"Kontrol tidak valid: "
        f"{invalid_controls}"
    )


# ============================================================
# POLICY VERSIONED DAN DETERMINISTIK
# ============================================================

policy_manifest = {
    "schema_version": "1.0.0",
    "policy_id": (
        "OCR-PREPROCESSING-ROUTING-V1"
    ),
    "status": "FROZEN",
    "scope": {
        "dataset_split": "development",
        "evaluated_documents": (
            EXPECTED_DOCUMENTS
        ),
        "evaluated_annotations": (
            EXPECTED_ANNOTATIONS
        ),
        "validation_opened": 0,
        "test_opened": 0,
    },
    "experiment_closure": {
        "candidate": "2.1",
        "candidate_decision": "REJECTED",
        "candidate_promoted": False,
        "automatic_extreme_scan_recovery": (
            False
        ),
        "reason": (
            "Candidate V2.1 failed the "
            "pre-registered annotation-coverage "
            "and token-recall quality floors on "
            "development ground truth."
        ),
        "failed_experiments_retained": True,
    },
    "quality_evidence": {
        "degraded_annotation_coverage": float(
            degraded_metrics[
                "annotation_text_coverage"
            ]
        ),
        "previous_annotation_coverage": float(
            previous_metrics[
                "annotation_text_coverage"
            ]
        ),
        "minimum_annotation_coverage": (
            minimum_coverage
        ),
        "candidate_annotation_coverage": (
            candidate_coverage
        ),
        "minimum_token_recall": (
            minimum_token_recall
        ),
        "candidate_token_recall": (
            candidate_token_recall
        ),
        "candidate_micro_cer": (
            candidate_micro_cer
        ),
        "quality_floor_passed": False,
        "decision_is_execution_error": False,
    },
    "routing_policy": routing_records,
    "deployment_guardrails": {
        "production_ready": False,
        "synthetic_development_baseline_ready": (
            True
        ),
        "automatic_export_from_severely_degraded_input": (
            False
        ),
        "external_api_required": False,
        "real_scan_validation_required": True,
        "realistic_degradation_protocol_required": (
            True
        ),
        "manual_review_or_rescan_supported": (
            True
        ),
    },
    "next_stage": {
        "cell": "CELL 10",
        "action": (
            "BUILD_FIELD_EXTRACTION_BASELINE_"
            "ON_DEVELOPMENT_SPLIT"
        ),
        "allowed_inputs": [
            "native PDF text",
            (
                "clean or quality-gate-approved "
                "image OCR"
            ),
        ],
        "blocked_inputs": [
            "severely degraded combined_scan",
            "validation split",
            "test split",
        ],
    },
    "input_artifacts": {
        name: {
            "path": str(path),
            "sha256": checksums_before[
                name
            ],
        }
        for name, path
        in INPUT_PATHS.items()
    },
    "integrity": {
        "dataset_modifications": 0,
        "ground_truth_modifications": 0,
        "ocr_result_modifications": 0,
        "input_manifest_modifications": 0,
    },
}


# ============================================================
# IDEMPOTENT WRITE
# ============================================================

if POLICY_PATH.exists():
    existing_policy = load_json(
        POLICY_PATH
    )

    if (
        canonical_json(existing_policy)
        != canonical_json(policy_manifest)
    ):
        raise RuntimeError(
            "Policy v1 sudah ada tetapi "
            "isinya berbeda. File berstatus "
            "FROZEN tidak ditimpa otomatis."
        )

    checkpoint_action = "RECOVERED"

else:
    atomic_write_json(
        POLICY_PATH,
        policy_manifest,
    )

    checkpoint_action = "CREATED"


# ============================================================
# POST-WRITE VERIFICATION
# ============================================================

saved_policy = load_json(
    POLICY_PATH
)

if (
    canonical_json(saved_policy)
    != canonical_json(policy_manifest)
):
    raise RuntimeError(
        "Policy tersimpan tidak cocok "
        "dengan policy di memori."
    )

checksums_after = {
    name: sha256_file(path)
    for name, path in INPUT_PATHS.items()
}

changed_inputs = [
    name
    for name in INPUT_PATHS
    if (
        checksums_before[name]
        != checksums_after[name]
    )
]

if changed_inputs:
    raise RuntimeError(
        "Input artifact berubah saat freeze: "
        f"{changed_inputs}"
    )


# ============================================================
# OUTPUT
# ============================================================

print()
print(
    f"Policy ID              : "
    f"{policy_manifest['policy_id']}"
)
print(
    f"Policy status          : "
    f"{policy_manifest['status']}"
)
print(
    f"Checkpoint action      : "
    f"{checkpoint_action}"
)
print("Candidate version      : 2.1")
print("Candidate decision     : REJECTED")
print("Candidate promoted     : False")
print(
    "Annotation coverage   : "
    f"{candidate_coverage:.6f} "
    f"(minimum {minimum_coverage:.2f})"
)
print(
    "Token recall          : "
    f"{candidate_token_recall:.6f} "
    f"(minimum {minimum_token_recall:.2f})"
)
print(
    f"Micro CER             : "
    f"{candidate_micro_cer:.6f}"
)
print(
    f"Approved dev routes   : "
    f"{approved_routes}"
)
print(
    "Extreme scan handling: "
    "RESCAN_OR_MANUAL_REVIEW"
)
print("Production ready      : False")
print("Validation opened     : 0")
print("Test opened           : 0")
print("Input artifact changes: 0")
print(
    f"Policy manifest       : "
    f"{POLICY_PATH}"
)
print(
    f"Manifest SHA-256      : "
    f"{sha256_file(POLICY_PATH)}"
)
print()
print(
    "✅ CELL 9F PASSED — Candidate V2.1 "
    "dibekukan sebagai REJECTED. Routing "
    "pengembangan aman dan siap dipakai "
    "untuk baseline ekstraksi field pada "
    "Cell 10."
)

,control,expected,actual,status
0,old_evaluation_status,PASSED,PASSED,VALID
1,old_candidate_decision,REJECTED,REJECTED,VALID
2,candidate_preprocessing_status,READY_FOR_OCR,READY_FOR_OCR,VALID
3,candidate_ocr_status,PASSED,PASSED,VALID
4,candidate_evaluation_status,PASSED,PASSED,VALID
5,candidate_decision,REJECTED,REJECTED,VALID
6,quality_floor_passed,False,False,VALID
7,manifest_quality_floor_passed,False,False,VALID
8,evaluation_documents,6,6,VALID
9,ground_truth_annotations,250,250,VALID


,route_id,condition,engine,preprocessing,action,development_status,production_status
0,NATIVE_PDF_TEXT,PDF has a usable native text layer,PyMuPDF,NONE,DIRECT_TEXT_EXTRACTION,APPROVED,REQUIRES_FIELD_EXTRACTION_VALIDATION
1,CLEAN_OR_ACCEPTABLE_IMAGE,No usable text layer and image passes calibrat...,PaddleOCR PP-OCRv6 CPU,NONE,DIRECT_OCR,APPROVED,REQUIRES_REAL_SCAN_VALIDATION
2,SEVERELY_DEGRADED_IMAGE,Image fails quality checks or OCR output is em...,NONE_AUTOMATICALLY,CANDIDATE_V2_1_NOT_PROMOTED,REQUEST_RESCAN_OR_MANUAL_REVIEW,SAFETY_ROUTE,BLOCK_AUTOMATIC_EXPORT



Policy ID              : OCR-PREPROCESSING-ROUTING-V1
Policy status          : FROZEN
Checkpoint action      : CREATED
Candidate version      : 2.1
Candidate decision     : REJECTED
Candidate promoted     : False
Annotation coverage   : 0.036000 (minimum 0.70)
Token recall          : 0.013633 (minimum 0.70)
Micro CER             : 0.976516
Approved dev routes   : 2
Extreme scan handling: RESCAN_OR_MANUAL_REVIEW
Production ready      : False
Validation opened     : 0
Test opened           : 0
Input artifact changes: 0
Policy manifest       : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/manifests/preprocessing_routing_policy_v1.json
Manifest SHA-256      : 8ed191e85e0846640e6f307e76297d5900ba3c2accf88f41475898368e1e5436

✅ CELL 9F PASSED — Candidate V2.1 dibekukan sebagai REJECTED. Routing pengembangan aman dan siap dipakai untuk baseline ekstraksi field pada Cell 10.


In [ ]:
from __future__ import annotations

import hashlib
import json
import os
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
from IPython.display import display


# ============================================================
# CELL 10A — FIELD EXTRACTION CONTRACT AND BENCHMARK FREEZE
# ============================================================

BUILD_ROOT = Path(
    "/content/drive/MyDrive/InvoiceFlow-AI-Data/interim/"
    "synthetic_v1_build/20260904T150025Z"
)
BENCHMARK_ROOT = BUILD_ROOT / "ocr_benchmark"

ROUTING_POLICY_PATH = (
    BENCHMARK_ROOT
    / "manifests/preprocessing_routing_policy_v1.json"
)
OCR_SELECTION_PATH = (
    BENCHMARK_ROOT / "benchmark_selection.json"
)
OUTPUT_ROOT = (
    BENCHMARK_ROOT / "field_extraction"
)
CONTRACT_PATH = (
    OUTPUT_ROOT / "field_extraction_contract_v1.json"
)

EXPECTED_DOCUMENTS = 18
EXPECTED_TEMPLATES = {
    "TPL-01",
    "TPL-02",
    "TPL-03",
    "TPL-04",
    "TPL-05",
    "TPL-06",
}
EXPECTED_LANGUAGES = {"en", "id"}
EXPECTED_DOCUMENTS_PER_TEMPLATE = 3


# ============================================================
# HELPERS
# ============================================================

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def load_json(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as handle:
        value = json.load(handle)
    if not isinstance(value, dict):
        raise TypeError(f"JSON harus berupa object: {path}")
    return value


def canonical_json(value) -> str:
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    )


def atomic_write_json(path: Path, value: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(path.name + ".tmp")
    temporary_path.write_text(
        json.dumps(value, indent=2, ensure_ascii=False) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary_path, path)


def walk_dicts(value):
    if isinstance(value, dict):
        yield value
        for child in value.values():
            yield from walk_dicts(child)
    elif isinstance(value, list):
        for child in value:
            yield from walk_dicts(child)


def value_from(record: dict, *keys, default=None):
    for key in keys:
        value = record.get(key)
        if value is not None and str(value).strip():
            return value
    return default


def record_richness(record: dict) -> int:
    useful_keys = {
        "document_id",
        "template_id",
        "language",
        "currency",
        "item_count",
        "annotation_count",
        "selection_reason",
        "canonical_invoice_id",
    }
    return sum(key in record for key in useful_keys)


# ============================================================
# TARGET FIELD CONTRACT
# ============================================================

field_definitions = [
    {
        "field": "invoice_number",
        "group": "metadata",
        "cardinality": "ONE",
        "data_type": "STRING",
        "required": True,
        "normalization": "TRIM_AND_CASE_PRESERVE",
        "critical": True,
    },
    {
        "field": "invoice_date",
        "group": "metadata",
        "cardinality": "ONE",
        "data_type": "DATE",
        "required": True,
        "normalization": "ISO_8601_DATE",
        "critical": True,
    },
    {
        "field": "due_date",
        "group": "metadata",
        "cardinality": "ONE",
        "data_type": "DATE",
        "required": True,
        "normalization": "ISO_8601_DATE",
        "critical": True,
    },
    {
        "field": "currency",
        "group": "metadata",
        "cardinality": "ONE",
        "data_type": "CURRENCY_CODE",
        "required": True,
        "normalization": "ISO_4217_UPPERCASE",
        "critical": True,
    },
    {
        "field": "vendor.name",
        "group": "vendor",
        "cardinality": "ONE",
        "data_type": "STRING",
        "required": True,
        "normalization": "UNICODE_NFKC_AND_WHITESPACE",
        "critical": True,
    },
    {
        "field": "vendor.tax_identifier",
        "group": "vendor",
        "cardinality": "ZERO_OR_ONE",
        "data_type": "IDENTIFIER",
        "required": False,
        "normalization": "UPPERCASE_REMOVE_LABEL",
        "critical": False,
    },
    {
        "field": "buyer.name",
        "group": "buyer",
        "cardinality": "ONE",
        "data_type": "STRING",
        "required": True,
        "normalization": "UNICODE_NFKC_AND_WHITESPACE",
        "critical": True,
    },
    {
        "field": "buyer.tax_identifier",
        "group": "buyer",
        "cardinality": "ZERO_OR_ONE",
        "data_type": "IDENTIFIER",
        "required": False,
        "normalization": "UPPERCASE_REMOVE_LABEL",
        "critical": False,
    },
    {
        "field": "financials.subtotal",
        "group": "financials",
        "cardinality": "ONE",
        "data_type": "DECIMAL_MONEY",
        "required": True,
        "normalization": "DECIMAL_WITH_CURRENCY_CONTEXT",
        "critical": True,
    },
    {
        "field": "financials.tax",
        "group": "financials",
        "cardinality": "ONE",
        "data_type": "DECIMAL_MONEY",
        "required": True,
        "normalization": "DECIMAL_WITH_CURRENCY_CONTEXT",
        "critical": True,
    },
    {
        "field": "financials.discount",
        "group": "financials",
        "cardinality": "ONE",
        "data_type": "DECIMAL_MONEY",
        "required": True,
        "normalization": "DECIMAL_WITH_CURRENCY_CONTEXT",
        "critical": True,
    },
    {
        "field": "financials.total",
        "group": "financials",
        "cardinality": "ONE",
        "data_type": "DECIMAL_MONEY",
        "required": True,
        "normalization": "DECIMAL_WITH_CURRENCY_CONTEXT",
        "critical": True,
    },
    {
        "field": "items[].description",
        "group": "items",
        "cardinality": "REPEATING",
        "data_type": "STRING",
        "required": True,
        "normalization": "UNICODE_NFKC_AND_WHITESPACE",
        "critical": True,
    },
    {
        "field": "items[].quantity",
        "group": "items",
        "cardinality": "REPEATING",
        "data_type": "DECIMAL_QUANTITY",
        "required": True,
        "normalization": "DECIMAL",
        "critical": True,
    },
    {
        "field": "items[].unit_price",
        "group": "items",
        "cardinality": "REPEATING",
        "data_type": "DECIMAL_MONEY",
        "required": True,
        "normalization": "DECIMAL_WITH_CURRENCY_CONTEXT",
        "critical": True,
    },
    {
        "field": "items[].line_total",
        "group": "items",
        "cardinality": "REPEATING",
        "data_type": "DECIMAL_MONEY",
        "required": True,
        "normalization": "DECIMAL_WITH_CURRENCY_CONTEXT",
        "critical": True,
    },
]


# ============================================================
# PREFLIGHT
# ============================================================

required_paths = [
    ROUTING_POLICY_PATH,
    OCR_SELECTION_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.is_file()]
if missing_paths:
    raise FileNotFoundError(
        "Checkpoint yang diperlukan belum tersedia:\n"
        + "\n".join(missing_paths)
    )

input_checksums_before = {
    "routing_policy": sha256_file(ROUTING_POLICY_PATH),
    "ocr_selection": sha256_file(OCR_SELECTION_PATH),
}

routing_policy = load_json(ROUTING_POLICY_PATH)
selection_manifest = load_json(OCR_SELECTION_PATH)


# ============================================================
# RECOVER 18 FROZEN DEVELOPMENT DOCUMENTS
# ============================================================

records_by_document = {}

for record in walk_dicts(selection_manifest):
    document_id = value_from(record, "document_id")
    template_id = value_from(record, "template_id")

    if not document_id or not template_id:
        continue

    candidate = dict(record)
    current = records_by_document.get(str(document_id))

    if current is None or record_richness(candidate) > record_richness(current):
        records_by_document[str(document_id)] = candidate

benchmark_records = []

for document_id, source_record in records_by_document.items():
    template_id = str(source_record["template_id"])

    if template_id not in EXPECTED_TEMPLATES:
        continue

    language = str(
        value_from(source_record, "language", "locale", default="")
    ).lower()

    benchmark_records.append(
        {
            "document_id": document_id,
            "canonical_invoice_id": value_from(
                source_record,
                "canonical_invoice_id",
                "canonical_id",
            ),
            "template_id": template_id,
            "split": "development",
            "language": language,
            "currency": value_from(source_record, "currency"),
            "item_count": value_from(source_record, "item_count"),
            "annotation_count": value_from(
                source_record,
                "annotation_count",
            ),
            "selection_reason": value_from(
                source_record,
                "selection_reason",
                default="FROZEN_OCR_BENCHMARK",
            ),
            "source_record_sha256": hashlib.sha256(
                canonical_json(source_record).encode("utf-8")
            ).hexdigest(),
        }
    )

benchmark_records = sorted(
    benchmark_records,
    key=lambda row: (
        row["template_id"],
        row["document_id"],
    ),
)

for sequence_number, record in enumerate(benchmark_records, start=1):
    record["sequence_number"] = sequence_number

document_ids = [row["document_id"] for row in benchmark_records]
template_counts = Counter(row["template_id"] for row in benchmark_records)
languages_by_template = defaultdict(set)

for row in benchmark_records:
    languages_by_template[row["template_id"]].add(row["language"])

language_coverage_valid = all(
    languages_by_template[template_id] == EXPECTED_LANGUAGES
    for template_id in EXPECTED_TEMPLATES
)


# ============================================================
# EVALUATION POLICY — FROZEN BEFORE PARSER DEVELOPMENT
# ============================================================

evaluation_policy = {
    "prediction_inputs": {
        "allowed": [
            "native PDF text and geometry",
            "approved PaddleOCR text and geometry",
        ],
        "forbidden": [
            "ground-truth values",
            "canonical payload values",
            "validation labels during development",
            "test labels before final evaluation",
        ],
    },
    "scalar_matching": {
        "primary": "TYPE_AWARE_NORMALIZED_EXACT_MATCH",
        "secondary": ["CER", "WER"],
    },
    "line_item_matching": {
        "strategy": "MAXIMUM_WEIGHT_BIPARTITE_ROW_MATCHING",
        "matching_signals": ["description", "line_total"],
        "reported_metrics": [
            "row_precision",
            "row_recall",
            "row_f1",
            "field_exact_match",
        ],
    },
    "financial_validation": {
        "formula": "subtotal + tax - discount = total",
        "decimal_arithmetic": True,
        "floating_point_for_money": False,
    },
    "development_acceptance_targets": {
        "critical_scalar_exact_match_minimum": 0.98,
        "all_scalar_exact_match_minimum": 0.95,
        "line_item_field_f1_minimum": 0.90,
        "financial_consistency_minimum": 0.99,
        "document_identity_leakage_allowed": False,
    },
    "split_protocol": {
        "development": "ITERATIVE_ENGINEERING_ALLOWED",
        "validation": "LOCKED_UNTIL_RULES_FROZEN",
        "test": "ONE_FINAL_EVALUATION_ONLY",
    },
}


# ============================================================
# CONTROLS
# ============================================================

scalar_fields = [
    row for row in field_definitions if row["cardinality"] != "REPEATING"
]
repeating_fields = [
    row for row in field_definitions if row["cardinality"] == "REPEATING"
]

controls = [
    ("routing_policy_status", "FROZEN", routing_policy.get("status")),
    (
        "candidate_promoted",
        False,
        routing_policy.get("experiment_closure", {}).get("candidate_promoted"),
    ),
    ("benchmark_documents", EXPECTED_DOCUMENTS, len(benchmark_records)),
    ("unique_document_ids", EXPECTED_DOCUMENTS, len(set(document_ids))),
    ("development_templates", 6, len(template_counts)),
    (
        "documents_per_template",
        True,
        all(
            template_counts[template_id] == EXPECTED_DOCUMENTS_PER_TEMPLATE
            for template_id in EXPECTED_TEMPLATES
        ),
    ),
    ("languages_per_template", True, language_coverage_valid),
    ("target_fields", 16, len(field_definitions)),
    ("scalar_fields", 12, len(scalar_fields)),
    ("repeating_fields", 4, len(repeating_fields)),
    ("validation_opened", 0, 0),
    ("test_opened", 0, 0),
]

control_records = [
    {
        "control": name,
        "expected": expected,
        "actual": actual,
        "status": "VALID" if expected == actual else "INVALID",
    }
    for name, expected, actual in controls
]

display(pd.DataFrame(control_records))
display(pd.DataFrame(benchmark_records))
display(pd.DataFrame(field_definitions))

invalid_controls = [
    row["control"] for row in control_records if row["status"] != "VALID"
]
if invalid_controls:
    raise RuntimeError(
        "CELL 10A PRE-FREEZE AUDIT FAILED. "
        f"Kontrol tidak valid: {invalid_controls}"
    )


# ============================================================
# CONTRACT MANIFEST — DETERMINISTIC AND IDEMPOTENT
# ============================================================

contract_manifest = {
    "schema_version": "1.0.0",
    "contract_id": "INVOICE-FIELD-EXTRACTION-CONTRACT-V1",
    "status": "FROZEN_FOR_DEVELOPMENT_BASELINE",
    "scope": {
        "split": "development",
        "documents": EXPECTED_DOCUMENTS,
        "templates": sorted(EXPECTED_TEMPLATES),
        "languages": sorted(EXPECTED_LANGUAGES),
        "validation_opened": 0,
        "test_opened": 0,
    },
    "field_definitions": field_definitions,
    "evaluation_policy": evaluation_policy,
    "benchmark_records": benchmark_records,
    "input_artifacts": {
        "routing_policy": {
            "path": str(ROUTING_POLICY_PATH),
            "sha256": input_checksums_before["routing_policy"],
        },
        "ocr_selection": {
            "path": str(OCR_SELECTION_PATH),
            "sha256": input_checksums_before["ocr_selection"],
        },
    },
    "integrity": {
        "prediction_generation_executed": False,
        "ground_truth_values_loaded": False,
        "dataset_modifications": 0,
        "validation_opened": 0,
        "test_opened": 0,
    },
    "next_stage": {
        "cell": "CELL 10B",
        "action": "BUILD_UNIFIED_DOCUMENT_TEXT_LAYER",
        "primary_route": "PyMuPDF native text and geometry",
        "fallback_route": "PaddleOCR PP-OCRv6",
    },
}

if CONTRACT_PATH.exists():
    existing_contract = load_json(CONTRACT_PATH)
    if canonical_json(existing_contract) != canonical_json(contract_manifest):
        raise RuntimeError(
            "Contract v1 sudah ada tetapi berbeda. "
            "Contract FROZEN tidak ditimpa otomatis."
        )
    checkpoint_action = "RECOVERED"
else:
    atomic_write_json(CONTRACT_PATH, contract_manifest)
    checkpoint_action = "CREATED"


# ============================================================
# POST-WRITE INTEGRITY
# ============================================================

saved_contract = load_json(CONTRACT_PATH)
if canonical_json(saved_contract) != canonical_json(contract_manifest):
    raise RuntimeError("Contract tersimpan tidak cocok dengan contract di memori.")

input_checksums_after = {
    "routing_policy": sha256_file(ROUTING_POLICY_PATH),
    "ocr_selection": sha256_file(OCR_SELECTION_PATH),
}
changed_inputs = [
    name
    for name in input_checksums_before
    if input_checksums_before[name] != input_checksums_after[name]
]
if changed_inputs:
    raise RuntimeError(f"Input artifact berubah: {changed_inputs}")

print()
print(f"Contract ID          : {contract_manifest['contract_id']}")
print(f"Contract status      : {contract_manifest['status']}")
print(f"Checkpoint action    : {checkpoint_action}")
print(f"Benchmark documents : {len(benchmark_records)}")
print(f"Templates           : {len(template_counts)}")
print(f"Target fields       : {len(field_definitions)}")
print(f"Scalar fields       : {len(scalar_fields)}")
print(f"Repeating fields    : {len(repeating_fields)}")
print("Ground-truth loaded : False")
print("Predictions created : False")
print("Validation opened   : 0")
print("Test opened         : 0")
print("Input changes       : 0")
print(f"Contract manifest   : {CONTRACT_PATH}")
print(f"Manifest SHA-256    : {sha256_file(CONTRACT_PATH)}")
print()
print(
    "✅ CELL 10A PASSED — kontrak 16 field dan benchmark "
    "18 dokumen development telah dibekukan tanpa membuka "
    "ground truth, validation, atau test."
)


,control,expected,actual,status
0,routing_policy_status,FROZEN,FROZEN,VALID
1,candidate_promoted,False,False,VALID
2,benchmark_documents,18,18,VALID
3,unique_document_ids,18,18,VALID
4,development_templates,6,6,VALID
5,documents_per_template,True,True,VALID
6,languages_per_template,True,True,VALID
7,target_fields,16,16,VALID
8,scalar_fields,12,12,VALID
9,repeating_fields,4,4,VALID


,document_id,canonical_invoice_id,template_id,split,language,currency,item_count,annotation_count,selection_reason,source_record_sha256,sequence_number
0,INV-SYN-000002,CANON-000002,TPL-01,development,id,USD,2,27,minimum_items,9d693c9643a5a94a9a94e392dd004945f771deea0b3df2...,1
1,INV-SYN-000013,CANON-000013,TPL-01,development,en,USD,8,51,highest_text_risk,adb5fd59797040513b81e79221e10a1de83cdc098edbd7...,2
2,INV-SYN-000015,CANON-000015,TPL-01,development,en,USD,8,51,maximum_items,c146b01d5e6351071aaae7c3ac2640c639911d8f4147eb...,3
3,INV-SYN-000025,CANON-000025,TPL-02,development,id,IDR,2,27,minimum_items,abb019f25b4bfbdfbc407631c77995a1e25ba4c67af469...,4
4,INV-SYN-000036,CANON-000036,TPL-02,development,en,EUR,7,47,maximum_items,a2b9ad7dbb9a914003ac39448bbbe7b92f8f6430481127...,5
5,INV-SYN-000037,CANON-000037,TPL-02,development,en,USD,7,47,highest_text_risk,dec39c92d88baf09b2c297b1f0c12f8c67c1f8af7777d4...,6
6,INV-SYN-000043,CANON-000043,TPL-03,development,id,IDR,2,27,minimum_items,6e0203a5a15820379c6d8e49ddb6c11538cea744a0f2d5...,7
7,INV-SYN-000052,CANON-000052,TPL-03,development,en,GBP,7,47,highest_text_risk,8447a844fc1dd82192d581cec4cff4726db58b976e6497...,8
8,INV-SYN-000060,CANON-000060,TPL-03,development,en,USD,7,47,maximum_items,431c86cfc37c0eeac0c64e5e58c6d06761586ff2a019fe...,9
9,INV-SYN-000064,CANON-000064,TPL-04,development,id,USD,8,51,maximum_items,cfbf21801d76fa0981239addaf1d9f83579aa4134000d3...,10


,field,group,cardinality,data_type,required,normalization,critical
0,invoice_number,metadata,ONE,STRING,True,TRIM_AND_CASE_PRESERVE,True
1,invoice_date,metadata,ONE,DATE,True,ISO_8601_DATE,True
2,due_date,metadata,ONE,DATE,True,ISO_8601_DATE,True
3,currency,metadata,ONE,CURRENCY_CODE,True,ISO_4217_UPPERCASE,True
4,vendor.name,vendor,ONE,STRING,True,UNICODE_NFKC_AND_WHITESPACE,True
5,vendor.tax_identifier,vendor,ZERO_OR_ONE,IDENTIFIER,False,UPPERCASE_REMOVE_LABEL,False
6,buyer.name,buyer,ONE,STRING,True,UNICODE_NFKC_AND_WHITESPACE,True
7,buyer.tax_identifier,buyer,ZERO_OR_ONE,IDENTIFIER,False,UPPERCASE_REMOVE_LABEL,False
8,financials.subtotal,financials,ONE,DECIMAL_MONEY,True,DECIMAL_WITH_CURRENCY_CONTEXT,True
9,financials.tax,financials,ONE,DECIMAL_MONEY,True,DECIMAL_WITH_CURRENCY_CONTEXT,True



Contract ID          : INVOICE-FIELD-EXTRACTION-CONTRACT-V1
Contract status      : FROZEN_FOR_DEVELOPMENT_BASELINE
Checkpoint action    : CREATED
Benchmark documents : 18
Templates           : 6
Target fields       : 16
Scalar fields       : 12
Repeating fields    : 4
Ground-truth loaded : False
Predictions created : False
Validation opened   : 0
Test opened         : 0
Input changes       : 0
Contract manifest   : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/field_extraction/field_extraction_contract_v1.json
Manifest SHA-256    : 3f03ba3a0d09af38a2ef6d9128cb80fa8c1baaee61824ab0f5402167481562ec

✅ CELL 10A PASSED — kontrak 16 field dan benchmark 18 dokumen development telah dibekukan tanpa membuka ground truth, validation, atau test.


In [ ]:
from __future__ import annotations

import hashlib
import json
import os
from collections import Counter, defaultdict
from pathlib import Path

import fitz
import pandas as pd
from IPython.display import display


# ============================================================
# CELL 10B — UNIFIED DOCUMENT TEXT LAYER
# ============================================================

BUILD_ROOT = Path(
    "/content/drive/MyDrive/InvoiceFlow-AI-Data/interim/"
    "synthetic_v1_build/20260904T150025Z"
)
BENCHMARK_ROOT = BUILD_ROOT / "ocr_benchmark"
FIELD_ROOT = BENCHMARK_ROOT / "field_extraction"

CONTRACT_PATH = FIELD_ROOT / "field_extraction_contract_v1.json"
ROUTING_POLICY_PATH = (
    BENCHMARK_ROOT / "manifests/preprocessing_routing_policy_v1.json"
)
BATCH_INDEX_PATH = BUILD_ROOT / "manifests/batch_render_index.jsonl"
PADDLE_MANIFEST_PATH = (
    BENCHMARK_ROOT / "manifests/paddleocr_run_manifest.json"
)
PADDLE_RESULTS_ROOT = BENCHMARK_ROOT / "results/paddleocr"

TEXT_LAYER_ROOT = FIELD_ROOT / "text_layers"
SUMMARY_PATH = FIELD_ROOT / "unified_text_layer_summary.csv"
MANIFEST_PATH = FIELD_ROOT / "unified_text_layer_manifest.json"

EXPECTED_DOCUMENTS = 18
EXPECTED_TEMPLATES = 6
MINIMUM_NATIVE_WORDS = 10
MINIMUM_NATIVE_CHARACTERS = 50


# ============================================================
# GENERAL HELPERS
# ============================================================

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def load_json(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as handle:
        value = json.load(handle)
    if not isinstance(value, dict):
        raise TypeError(f"JSON harus berupa object: {path}")
    return value


def load_jsonl(path: Path) -> list[dict]:
    records = []
    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue
            try:
                value = json.loads(line)
            except json.JSONDecodeError as error:
                raise RuntimeError(
                    f"JSONL tidak valid pada baris {line_number}: {path}"
                ) from error
            if not isinstance(value, dict):
                raise TypeError(
                    f"Record JSONL baris {line_number} bukan object."
                )
            records.append(value)
    return records


def canonical_json(value) -> str:
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    )


def atomic_write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(path.name + ".tmp")
    temporary_path.write_text(text, encoding="utf-8")
    os.replace(temporary_path, path)


def atomic_write_json(path: Path, value: dict) -> None:
    atomic_write_text(
        path,
        json.dumps(value, indent=2, ensure_ascii=False, default=str) + "\n",
    )


def walk_strings(value):
    if isinstance(value, str):
        yield value
    elif isinstance(value, dict):
        for child in value.values():
            yield from walk_strings(child)
    elif isinstance(value, list):
        for child in value:
            yield from walk_strings(child)


def walk_dicts(value):
    if isinstance(value, dict):
        yield value
        for child in value.values():
            yield from walk_dicts(child)
    elif isinstance(value, list):
        for child in value:
            yield from walk_dicts(child)


def nested_identity(record: dict) -> tuple[str, str]:
    document_id = ""
    template_id = ""

    for candidate in walk_dicts(record):
        if not document_id and candidate.get("document_id"):
            document_id = str(candidate["document_id"])
        if not template_id and candidate.get("template_id"):
            template_id = str(candidate["template_id"])
        if document_id and template_id:
            break

    return document_id, template_id


def resolve_possible_path(value: str, parent: Path) -> list[Path]:
    candidate = Path(value)
    if candidate.is_absolute():
        return [candidate]
    return [parent / candidate, BUILD_ROOT / candidate]


def round_float(value: float) -> float:
    return round(float(value), 6)


# ============================================================
# PDF DISCOVERY
# ============================================================

def build_index_record_map(records: list[dict]) -> dict[str, dict]:
    mapped = {}
    for record in records:
        document_id, _ = nested_identity(record)
        if not document_id:
            continue
        if document_id in mapped:
            raise RuntimeError(
                f"Document ID duplikat pada batch index: {document_id}"
            )
        mapped[document_id] = record
    return mapped


def locate_pdf(
    document_id: str,
    template_id: str,
    index_record: dict,
) -> Path:
    discovered = []

    for path_value in walk_strings(index_record):
        if not path_value.lower().endswith(".pdf"):
            continue

        for candidate in resolve_possible_path(
            path_value,
            BATCH_INDEX_PATH.parent,
        ):
            candidate_text = candidate.as_posix().lower()
            if (
                candidate.is_file()
                and document_id.lower() in candidate_text
                and template_id.lower() in candidate_text
            ):
                discovered.append(candidate.resolve())

    if not discovered:
        rendered_root = BUILD_ROOT / "rendered_dataset"
        for candidate in rendered_root.rglob(f"*{document_id}*.pdf"):
            candidate_text = candidate.as_posix().lower()
            if template_id.lower() in candidate_text:
                discovered.append(candidate.resolve())

    unique_paths = sorted(set(discovered), key=lambda path: path.as_posix())

    if len(unique_paths) != 1:
        raise RuntimeError(
            f"PDF {document_id}/{template_id} ditemukan "
            f"{len(unique_paths)} kali; seharusnya tepat satu."
        )

    return unique_paths[0]


# ============================================================
# NATIVE PDF TEXT EXTRACTION
# ============================================================

def extract_native_layer(pdf_path: Path) -> dict:
    tokens = []
    lines = []
    page_records = []
    global_token_index = 0
    global_line_index = 0

    with fitz.open(str(pdf_path)) as document:
        for page_index, page in enumerate(document):
            width = float(page.rect.width)
            height = float(page.rect.height)
            page_number = page_index + 1

            page_records.append(
                {
                    "page_number": page_number,
                    "width_points": round_float(width),
                    "height_points": round_float(height),
                    "rotation": int(page.rotation),
                }
            )

            word_rows = page.get_text("words", sort=True)
            grouped_lines = defaultdict(list)

            for word_row in word_rows:
                if len(word_row) < 8:
                    continue

                x0, y0, x1, y1 = map(float, word_row[:4])
                text = str(word_row[4]).strip()
                block_number = int(word_row[5])
                line_number = int(word_row[6])
                word_number = int(word_row[7])

                if not text:
                    continue

                token = {
                    "token_index": global_token_index,
                    "page_number": page_number,
                    "block_number": block_number,
                    "line_number": line_number,
                    "word_number": word_number,
                    "text": text,
                    "bbox_points": [
                        round_float(x0),
                        round_float(y0),
                        round_float(x1),
                        round_float(y1),
                    ],
                    "bbox_normalized": [
                        round_float(x0 / width),
                        round_float(y0 / height),
                        round_float(x1 / width),
                        round_float(y1 / height),
                    ],
                    "confidence": 1.0,
                    "source": "PYMUPDF_NATIVE_TEXT",
                }

                tokens.append(token)
                grouped_lines[(block_number, line_number)].append(token)
                global_token_index += 1

            sorted_line_groups = sorted(
                grouped_lines.items(),
                key=lambda item: (
                    min(token["bbox_points"][1] for token in item[1]),
                    min(token["bbox_points"][0] for token in item[1]),
                ),
            )

            for (block_number, line_number), line_tokens in sorted_line_groups:
                line_tokens = sorted(
                    line_tokens,
                    key=lambda token: (
                        token["word_number"],
                        token["bbox_points"][0],
                    ),
                )

                x0 = min(token["bbox_points"][0] for token in line_tokens)
                y0 = min(token["bbox_points"][1] for token in line_tokens)
                x1 = max(token["bbox_points"][2] for token in line_tokens)
                y1 = max(token["bbox_points"][3] for token in line_tokens)

                lines.append(
                    {
                        "line_index": global_line_index,
                        "page_number": page_number,
                        "block_number": block_number,
                        "line_number": line_number,
                        "text": " ".join(
                            token["text"] for token in line_tokens
                        ),
                        "token_indexes": [
                            token["token_index"] for token in line_tokens
                        ],
                        "bbox_points": [x0, y0, x1, y1],
                        "bbox_normalized": [
                            round_float(x0 / width),
                            round_float(y0 / height),
                            round_float(x1 / width),
                            round_float(y1 / height),
                        ],
                        "confidence": 1.0,
                        "source": "PYMUPDF_NATIVE_TEXT",
                    }
                )

                global_line_index += 1

    full_text = "\n".join(line["text"] for line in lines).strip()
    nonspace_characters = sum(not character.isspace() for character in full_text)
    usable = bool(
        len(tokens) >= MINIMUM_NATIVE_WORDS
        and nonspace_characters >= MINIMUM_NATIVE_CHARACTERS
    )

    return {
        "usable": usable,
        "pages": page_records,
        "tokens": tokens,
        "lines": lines,
        "full_text": full_text,
        "metrics": {
            "page_count": len(page_records),
            "token_count": len(tokens),
            "line_count": len(lines),
            "nonspace_character_count": nonspace_characters,
        },
    }


# ============================================================
# EXISTING PADDLEOCR FALLBACK LOADER
# ============================================================

def locate_paddle_result(document_id: str) -> Path:
    candidates = [
        path.resolve()
        for path in PADDLE_RESULTS_ROOT.rglob(f"*{document_id}*.json")
        if path.is_file()
    ]
    candidates = sorted(set(candidates), key=lambda path: path.as_posix())

    if len(candidates) != 1:
        raise RuntimeError(
            f"PaddleOCR result {document_id} ditemukan "
            f"{len(candidates)} kali."
        )

    return candidates[0]


def bbox_from_value(value) -> list[float] | None:
    if not isinstance(value, (list, tuple)):
        return None

    if len(value) == 4 and all(isinstance(item, (int, float)) for item in value):
        x0, y0, x1, y1 = map(float, value)
        return [x0, y0, x1, y1]

    points = []
    for point in value:
        if (
            isinstance(point, (list, tuple))
            and len(point) >= 2
            and isinstance(point[0], (int, float))
            and isinstance(point[1], (int, float))
        ):
            points.append((float(point[0]), float(point[1])))

    if points:
        xs = [point[0] for point in points]
        ys = [point[1] for point in points]
        return [min(xs), min(ys), max(xs), max(ys)]

    return None


def load_paddle_layer(document_id: str) -> dict:
    result_path = locate_paddle_result(document_id)
    result = load_json(result_path)

    output = result.get("output", {})
    source_lines = output.get("lines") or output.get("regions") or []
    if not isinstance(source_lines, list):
        source_lines = []

    lines = []
    tokens = []

    for index, source_line in enumerate(source_lines):
        if not isinstance(source_line, dict):
            continue

        text = str(source_line.get("text", "")).strip()
        if not text:
            continue

        bbox = None
        for key in ("bbox_points", "bbox", "box", "polygon", "points"):
            bbox = bbox_from_value(source_line.get(key))
            if bbox is not None:
                break

        confidence = source_line.get("confidence", source_line.get("score"))
        confidence = float(confidence) if confidence is not None else None

        token = {
            "token_index": len(tokens),
            "page_number": int(source_line.get("page_number", 1)),
            "text": text,
            "bbox_pixels": [round_float(value) for value in bbox] if bbox else None,
            "confidence": confidence,
            "source": "PADDLEOCR_PP_OCRV6",
        }
        tokens.append(token)
        lines.append(
            {
                "line_index": len(lines),
                "page_number": token["page_number"],
                "text": text,
                "token_indexes": [token["token_index"]],
                "bbox_pixels": token["bbox_pixels"],
                "confidence": confidence,
                "source": "PADDLEOCR_PP_OCRV6",
            }
        )

    full_text = "\n".join(line["text"] for line in lines).strip()

    if not full_text:
        full_text = str(output.get("full_text", "")).strip()

    if not full_text:
        raise RuntimeError(f"PaddleOCR fallback kosong: {document_id}")

    return {
        "result_path": result_path,
        "result_sha256": sha256_file(result_path),
        "pages": result.get("pages", []),
        "tokens": tokens,
        "lines": lines,
        "full_text": full_text,
        "metrics": {
            "page_count": int(output.get("page_count", 1)),
            "token_count": len(tokens),
            "line_count": len(lines),
            "nonspace_character_count": sum(
                not character.isspace() for character in full_text
            ),
        },
    }


# ============================================================
# PREFLIGHT
# ============================================================

required_paths = [
    CONTRACT_PATH,
    ROUTING_POLICY_PATH,
    BATCH_INDEX_PATH,
    PADDLE_MANIFEST_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.is_file()]
if missing_paths:
    raise FileNotFoundError(
        "Checkpoint belum lengkap:\n" + "\n".join(missing_paths)
    )

contract = load_json(CONTRACT_PATH)
routing_policy = load_json(ROUTING_POLICY_PATH)
paddle_manifest = load_json(PADDLE_MANIFEST_PATH)
batch_index_records = load_jsonl(BATCH_INDEX_PATH)
index_record_map = build_index_record_map(batch_index_records)

benchmark_records = contract.get("benchmark_records", [])

preflight_controls = [
    (
        "contract_status",
        "FROZEN_FOR_DEVELOPMENT_BASELINE",
        contract.get("status"),
    ),
    ("routing_policy_status", "FROZEN", routing_policy.get("status")),
    ("paddle_manifest_status", "PASSED", paddle_manifest.get("status")),
    ("benchmark_documents", EXPECTED_DOCUMENTS, len(benchmark_records)),
]

invalid_preflight = [
    name for name, expected, actual in preflight_controls if expected != actual
]
if invalid_preflight:
    raise RuntimeError(
        "CELL 10B PREFLIGHT FAILED. "
        f"Kontrol tidak valid: {invalid_preflight}"
    )


# ============================================================
# RESOLVE AND FREEZE SOURCE PDF CHECKSUMS
# ============================================================

source_records = []
for record in benchmark_records:
    document_id = str(record["document_id"])
    template_id = str(record["template_id"])

    index_record = index_record_map.get(document_id)
    if index_record is None:
        raise RuntimeError(f"Document tidak ada pada batch index: {document_id}")

    indexed_document_id, indexed_template_id = nested_identity(index_record)
    if indexed_document_id != document_id or indexed_template_id != template_id:
        raise RuntimeError(f"Identitas batch index tidak cocok: {document_id}")

    pdf_path = locate_pdf(document_id, template_id, index_record)
    source_records.append(
        {
            **record,
            "pdf_path": str(pdf_path),
            "pdf_sha256": sha256_file(pdf_path),
        }
    )

pdf_checksums_before = {
    row["document_id"]: row["pdf_sha256"] for row in source_records
}
source_manifest_checksums_before = {
    "contract": sha256_file(CONTRACT_PATH),
    "routing_policy": sha256_file(ROUTING_POLICY_PATH),
    "batch_index": sha256_file(BATCH_INDEX_PATH),
    "paddle_manifest": sha256_file(PADDLE_MANIFEST_PATH),
}


# ============================================================
# BUILD OR RECOVER TEXT LAYERS
# ============================================================

runtime_records = []
manifest_records = []
errors = []
newly_created = 0
recovered = 0

print(f"Membangun unified text layer untuk {len(source_records)} dokumen...\n")

for sequence_number, source in enumerate(source_records, start=1):
    document_id = source["document_id"]
    template_id = source["template_id"]
    pdf_path = Path(source["pdf_path"])
    result_path = TEXT_LAYER_ROOT / template_id / f"{document_id}_text_layer.json"

    try:
        native_layer = extract_native_layer(pdf_path)

        if native_layer["usable"]:
            route_id = "NATIVE_PDF_TEXT"
            engine = "PyMuPDF"
            layer = native_layer
            fallback_artifact = None
        else:
            route_id = "CLEAN_OR_ACCEPTABLE_IMAGE"
            engine = "PaddleOCR PP-OCRv6 CPU"
            layer = load_paddle_layer(document_id)
            fallback_artifact = {
                "path": str(layer.pop("result_path")),
                "sha256": layer.pop("result_sha256"),
            }

        text_layer = {
            "schema_version": "1.0.0",
            "status": "PASSED",
            "document": {
                "canonical_invoice_id": source.get("canonical_invoice_id"),
                "document_id": document_id,
                "template_id": template_id,
                "split": "development",
                "language": source.get("language"),
                "currency": source.get("currency"),
            },
            "source": {
                "pdf_path": str(pdf_path),
                "pdf_sha256": source["pdf_sha256"],
                "fallback_artifact": fallback_artifact,
            },
            "routing": {
                "route_id": route_id,
                "engine": engine,
                "preprocessing": "NONE",
                "ocr_executed_in_this_cell": False,
            },
            "pages": layer["pages"],
            "tokens": layer["tokens"],
            "lines": layer["lines"],
            "full_text": layer["full_text"],
            "metrics": layer["metrics"],
            "integrity": {
                "ground_truth_loaded": False,
                "canonical_payload_loaded": False,
                "validation_opened": 0,
                "test_opened": 0,
                "dataset_modifications": 0,
            },
        }

        if result_path.exists():
            existing = load_json(result_path)
            if canonical_json(existing) != canonical_json(text_layer):
                raise RuntimeError(
                    "Text layer checkpoint berbeda dengan hasil deterministik."
                )
            execution = "RECOVERED"
            recovered += 1
        else:
            atomic_write_json(result_path, text_layer)
            execution = "NEW"
            newly_created += 1

        result_sha256 = sha256_file(result_path)
        metrics = text_layer["metrics"]

        manifest_record = {
            "sequence_number": sequence_number,
            "document_id": document_id,
            "template_id": template_id,
            "language": source.get("language"),
            "route_id": route_id,
            "engine": engine,
            "page_count": int(metrics["page_count"]),
            "token_count": int(metrics["token_count"]),
            "line_count": int(metrics["line_count"]),
            "nonspace_character_count": int(
                metrics["nonspace_character_count"]
            ),
            "pdf_path": str(pdf_path),
            "pdf_sha256": source["pdf_sha256"],
            "text_layer_path": str(result_path),
            "text_layer_sha256": result_sha256,
            "status": "PASSED",
        }
        manifest_records.append(manifest_record)
        runtime_records.append({**manifest_record, "execution": execution})

        print(
            f"[{sequence_number:02d}/{len(source_records):02d}] "
            f"{document_id} | route={route_id} | "
            f"tokens={metrics['token_count']} | "
            f"lines={metrics['line_count']} | {execution}"
        )

    except Exception as error:
        errors.append(
            {
                "sequence_number": sequence_number,
                "document_id": document_id,
                "template_id": template_id,
                "error_type": type(error).__name__,
                "error": str(error)[:500],
            }
        )
        print(
            f"[{sequence_number:02d}/{len(source_records):02d}] "
            f"{document_id} | ERROR: {type(error).__name__}: {error}"
        )


# ============================================================
# AUDIT RESULTS
# ============================================================

runtime_table = pd.DataFrame(runtime_records)
manifest_table = pd.DataFrame(manifest_records)

if errors:
    display(pd.DataFrame(errors))

route_counts = Counter(
    row["route_id"] for row in manifest_records
)

controls = [
    ("result_records", EXPECTED_DOCUMENTS, len(manifest_records)),
    ("result_files", EXPECTED_DOCUMENTS, sum(
        Path(row["text_layer_path"]).is_file() for row in manifest_records
    )),
    ("unique_document_ids", EXPECTED_DOCUMENTS, len({
        row["document_id"] for row in manifest_records
    })),
    ("template_count", EXPECTED_TEMPLATES, len({
        row["template_id"] for row in manifest_records
    })),
    ("native_text_routes", EXPECTED_DOCUMENTS, route_counts.get(
        "NATIVE_PDF_TEXT", 0
    )),
    ("ocr_fallback_routes", 0, route_counts.get(
        "CLEAN_OR_ACCEPTABLE_IMAGE", 0
    )),
    ("single_page_documents", EXPECTED_DOCUMENTS, sum(
        row["page_count"] == 1 for row in manifest_records
    )),
    ("nonempty_token_layers", EXPECTED_DOCUMENTS, sum(
        row["token_count"] > 0 for row in manifest_records
    )),
    ("processing_errors", 0, len(errors)),
    ("ground_truth_opened", 0, 0),
    ("validation_opened", 0, 0),
    ("test_opened", 0, 0),
]

control_records = [
    {
        "control": name,
        "expected": expected,
        "actual": actual,
        "status": "VALID" if expected == actual else "INVALID",
    }
    for name, expected, actual in controls
]

display(pd.DataFrame(control_records))
display(runtime_table)

invalid_controls = [
    row["control"] for row in control_records if row["status"] != "VALID"
]
if invalid_controls:
    raise RuntimeError(
        "CELL 10B TEXT LAYER BUILD FAILED. "
        f"Kontrol tidak valid: {invalid_controls}"
    )


# ============================================================
# SAVE SUMMARY AND MANIFEST
# ============================================================

summary_columns = [
    "sequence_number",
    "document_id",
    "template_id",
    "language",
    "route_id",
    "engine",
    "page_count",
    "token_count",
    "line_count",
    "nonspace_character_count",
    "pdf_path",
    "pdf_sha256",
    "text_layer_path",
    "text_layer_sha256",
    "status",
]

summary_text = manifest_table[summary_columns].to_csv(
    index=False,
    lineterminator="\n",
)

if SUMMARY_PATH.exists():
    if SUMMARY_PATH.read_text(encoding="utf-8") != summary_text:
        raise RuntimeError(
            "Summary checkpoint sudah ada tetapi isinya berbeda."
        )
    summary_action = "RECOVERED"
else:
    atomic_write_text(SUMMARY_PATH, summary_text)
    summary_action = "CREATED"

text_layer_manifest = {
    "schema_version": "1.0.0",
    "status": "PASSED",
    "stage": "UNIFIED_DOCUMENT_TEXT_LAYER",
    "scope": {
        "split": "development",
        "documents": EXPECTED_DOCUMENTS,
        "templates": EXPECTED_TEMPLATES,
        "validation_opened": 0,
        "test_opened": 0,
    },
    "routing": {
        "native_pdf_engine": "PyMuPDF",
        "ocr_fallback_engine": "PaddleOCR PP-OCRv6 CPU",
        "minimum_native_words": MINIMUM_NATIVE_WORDS,
        "minimum_native_characters": MINIMUM_NATIVE_CHARACTERS,
        "route_distribution": dict(sorted(route_counts.items())),
        "ocr_executions_in_this_cell": 0,
    },
    "records": manifest_records,
    "artifacts": {
        "summary_path": str(SUMMARY_PATH),
        "summary_sha256": sha256_file(SUMMARY_PATH),
        "text_layer_root": str(TEXT_LAYER_ROOT),
    },
    "input_artifacts": {
        name: {
            "path": str({
                "contract": CONTRACT_PATH,
                "routing_policy": ROUTING_POLICY_PATH,
                "batch_index": BATCH_INDEX_PATH,
                "paddle_manifest": PADDLE_MANIFEST_PATH,
            }[name]),
            "sha256": checksum,
        }
        for name, checksum in source_manifest_checksums_before.items()
    },
    "integrity": {
        "ground_truth_loaded": False,
        "canonical_payload_loaded": False,
        "dataset_modifications": 0,
        "source_pdf_modifications": 0,
        "validation_opened": 0,
        "test_opened": 0,
    },
    "next_stage": {
        "cell": "CELL 10C",
        "action": "BUILD_RULE_BASED_FIELD_EXTRACTION_BASELINE",
    },
}

if MANIFEST_PATH.exists():
    existing_manifest = load_json(MANIFEST_PATH)
    if canonical_json(existing_manifest) != canonical_json(text_layer_manifest):
        raise RuntimeError(
            "Text-layer manifest sudah ada tetapi isinya berbeda."
        )
    manifest_action = "RECOVERED"
else:
    atomic_write_json(MANIFEST_PATH, text_layer_manifest)
    manifest_action = "CREATED"


# ============================================================
# FINAL INTEGRITY VERIFICATION
# ============================================================

source_manifest_checksums_after = {
    "contract": sha256_file(CONTRACT_PATH),
    "routing_policy": sha256_file(ROUTING_POLICY_PATH),
    "batch_index": sha256_file(BATCH_INDEX_PATH),
    "paddle_manifest": sha256_file(PADDLE_MANIFEST_PATH),
}
changed_manifests = [
    name
    for name in source_manifest_checksums_before
    if source_manifest_checksums_before[name]
    != source_manifest_checksums_after[name]
]

changed_pdfs = []
for source in source_records:
    current_checksum = sha256_file(Path(source["pdf_path"]))
    if current_checksum != pdf_checksums_before[source["document_id"]]:
        changed_pdfs.append(source["document_id"])

if changed_manifests or changed_pdfs:
    raise RuntimeError(
        "Source berubah selama text-layer build: "
        f"manifests={changed_manifests}, pdfs={changed_pdfs}"
    )

print()
print(f"Documents              : {len(manifest_records)}")
print(f"Templates              : {len(set(row['template_id'] for row in manifest_records))}")
print(f"Native text routes     : {route_counts.get('NATIVE_PDF_TEXT', 0)}")
print(f"OCR fallback routes    : {route_counts.get('CLEAN_OR_ACCEPTABLE_IMAGE', 0)}")
print(f"New text layers        : {newly_created}")
print(f"Recovered text layers  : {recovered}")
print(f"Summary action         : {summary_action}")
print(f"Manifest action        : {manifest_action}")
print(f"Text-layer root        : {TEXT_LAYER_ROOT}")
print(f"Summary                : {SUMMARY_PATH}")
print(f"Manifest               : {MANIFEST_PATH}")
print(f"Manifest SHA-256       : {sha256_file(MANIFEST_PATH)}")
print("Ground truth opened    : 0")
print("OCR executions         : 0")
print("Validation opened      : 0")
print("Test opened            : 0")
print("Dataset modifications  : 0")
print("Source modifications   : 0")
print()
print(
    "✅ CELL 10B PASSED — unified text layer untuk 18 dokumen "
    "development berhasil dibuat dan siap digunakan oleh parser "
    "field pada Cell 10C."
)


Membangun unified text layer untuk 18 dokumen...

[01/18] INV-SYN-000002 | route=NATIVE_PDF_TEXT | tokens=114 | lines=52 | NEW
[02/18] INV-SYN-000013 | route=NATIVE_PDF_TEXT | tokens=163 | lines=82 | NEW
[03/18] INV-SYN-000015 | route=NATIVE_PDF_TEXT | tokens=164 | lines=82 | NEW
[04/18] INV-SYN-000025 | route=NATIVE_PDF_TEXT | tokens=115 | lines=52 | NEW
[05/18] INV-SYN-000036 | route=NATIVE_PDF_TEXT | tokens=156 | lines=77 | NEW
[06/18] INV-SYN-000037 | route=NATIVE_PDF_TEXT | tokens=155 | lines=77 | NEW
[07/18] INV-SYN-000043 | route=NATIVE_PDF_TEXT | tokens=111 | lines=51 | NEW
[08/18] INV-SYN-000052 | route=NATIVE_PDF_TEXT | tokens=152 | lines=76 | NEW
[09/18] INV-SYN-000060 | route=NATIVE_PDF_TEXT | tokens=153 | lines=76 | NEW
[10/18] INV-SYN-000064 | route=NATIVE_PDF_TEXT | tokens=171 | lines=85 | NEW
[11/18] INV-SYN-000070 | route=NATIVE_PDF_TEXT | tokens=117 | lines=55 | NEW
[12/18] INV-SYN-000071 | route=NATIVE_PDF_TEXT | tokens=158 | lines=81 | NEW
[13/18] INV-SYN-000082 | r

,control,expected,actual,status
0,result_records,18,18,VALID
1,result_files,18,18,VALID
2,unique_document_ids,18,18,VALID
3,template_count,6,6,VALID
4,native_text_routes,18,18,VALID
5,ocr_fallback_routes,0,0,VALID
6,single_page_documents,18,18,VALID
7,nonempty_token_layers,18,18,VALID
8,processing_errors,0,0,VALID
9,ground_truth_opened,0,0,VALID


,sequence_number,document_id,template_id,language,route_id,engine,page_count,token_count,line_count,nonspace_character_count,pdf_path,pdf_sha256,text_layer_path,text_layer_sha256,status,execution
0,1,INV-SYN-000002,TPL-01,id,NATIVE_PDF_TEXT,PyMuPDF,1,114,52,696,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,2d94f6a203343f317520f8b62c571b03d3b38e9e3363c2...,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,06c34a34b677a775bf56459d39d872fe4002f31f5301ab...,PASSED,NEW
1,2,INV-SYN-000013,TPL-01,en,NATIVE_PDF_TEXT,PyMuPDF,1,163,82,970,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,6118f643ca2d00af39d799619d3fe23089dabb6295148b...,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,da43ff9c99e60c095b700eaa87ec08eda5cf1d5347061d...,PASSED,NEW
2,3,INV-SYN-000015,TPL-01,en,NATIVE_PDF_TEXT,PyMuPDF,1,164,82,994,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,b4932f54159386024f759a4b91527af68e03d8e0d50109...,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,325f33213da8649999e96e2dba65ba5b524a8bec3166de...,PASSED,NEW
3,4,INV-SYN-000025,TPL-02,id,NATIVE_PDF_TEXT,PyMuPDF,1,115,52,721,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,e525e4c43fdc350f49d81d52a1593fccadee66cbdb931f...,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,9bec918e85cbbc4f92407f5d290b64a05ee0313ab7ff3b...,PASSED,NEW
4,5,INV-SYN-000036,TPL-02,en,NATIVE_PDF_TEXT,PyMuPDF,1,156,77,952,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,3b152eef37b928726ec9dd20cacd34f0ce3a5573b31cf0...,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,cc05ee225a45d447c09659a030f63d97560c2388c6a0ae...,PASSED,NEW
5,6,INV-SYN-000037,TPL-02,en,NATIVE_PDF_TEXT,PyMuPDF,1,155,77,936,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,b3bc0ad9a2e8caca193b1ddb26e40d957e943ebbff9423...,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,b467d55f51a6dd45210d4d6019e4b31cb6bdcffba37ac8...,PASSED,NEW
6,7,INV-SYN-000043,TPL-03,id,NATIVE_PDF_TEXT,PyMuPDF,1,111,51,687,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,a6c2a1c39a394b554dd47f39062aab61eee51cf22f7cd8...,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,fed54055294530789eab4624a05d778ace05e9e4c54d2e...,PASSED,NEW
7,8,INV-SYN-000052,TPL-03,en,NATIVE_PDF_TEXT,PyMuPDF,1,152,76,921,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,578985a577c99634e11279d84395246d331ecfe5f8e6b5...,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,f6cb289cee4b78ecac9bb304029d6068dc17f36d883691...,PASSED,NEW
8,9,INV-SYN-000060,TPL-03,en,NATIVE_PDF_TEXT,PyMuPDF,1,153,76,934,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,d7e2eef2686e7bd111409bc461b0b772ca94662f418213...,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,c2c2ab0c7ba811db5d361a7287684aaa6e2fc165509e9c...,PASSED,NEW
9,10,INV-SYN-000064,TPL-04,id,NATIVE_PDF_TEXT,PyMuPDF,1,171,85,997,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,9135e17d97f342e0ee4a540bf91d7dc156c5dc07b75c6a...,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,2ab247be3d0c30850de61d147ee106397af75b804aa9bf...,PASSED,NEW



Documents              : 18
Templates              : 6
Native text routes     : 18
OCR fallback routes    : 0
New text layers        : 18
Recovered text layers  : 0
Summary action         : CREATED
Manifest action        : CREATED
Text-layer root        : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/field_extraction/text_layers
Summary                : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/field_extraction/unified_text_layer_summary.csv
Manifest               : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/field_extraction/unified_text_layer_manifest.json
Manifest SHA-256       : 75e034dbd1a20d4178a12b75f851399035dff023ac284756e3c18308ef96ed8d
Ground truth opened    : 0
OCR executions         : 0
Validation opened      : 0
Test opened            : 0
Dataset modifications  : 0
Source modifications   : 0

✅ CELL 10B

In [ ]:
from __future__ import annotations

import hashlib
import json
from pathlib import Path

import pandas as pd
from IPython.display import display


# ============================================================
# CELL 10B.1 — READ-ONLY TEXT-LAYER STRUCTURE INSPECTION
# ============================================================

BUILD_ROOT = Path(
    "/content/drive/MyDrive/InvoiceFlow-AI-Data/interim/"
    "synthetic_v1_build/20260904T150025Z"
)

FIELD_ROOT = BUILD_ROOT / "ocr_benchmark" / "field_extraction"

CONTRACT_PATH = (
    FIELD_ROOT / "field_extraction_contract_v1.json"
)

TEXT_LAYER_MANIFEST_PATH = (
    FIELD_ROOT / "unified_text_layer_manifest.json"
)

EXPECTED_TEMPLATES = {
    "TPL-01",
    "TPL-02",
    "TPL-03",
    "TPL-04",
    "TPL-05",
    "TPL-06",
}


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def load_json(path: Path) -> dict:
    with path.open(
        "r",
        encoding="utf-8",
    ) as handle:
        value = json.load(handle)

    if not isinstance(value, dict):
        raise TypeError(
            f"JSON harus berupa object: {path}"
        )

    return value


required_paths = [
    CONTRACT_PATH,
    TEXT_LAYER_MANIFEST_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not path.is_file()
]

if missing_paths:
    raise FileNotFoundError(
        "Checkpoint belum lengkap:\n"
        + "\n".join(missing_paths)
    )

contract = load_json(CONTRACT_PATH)
text_layer_manifest = load_json(
    TEXT_LAYER_MANIFEST_PATH
)

contract_records = contract.get(
    "benchmark_records",
    [],
)

text_layer_records = {
    str(record["document_id"]): record
    for record in text_layer_manifest.get(
        "records",
        [],
    )
}

# Pilih dokumen minimum-items untuk setiap template.
selected_records = []

for template_id in sorted(EXPECTED_TEMPLATES):
    candidates = [
        record
        for record in contract_records
        if (
            record.get("template_id")
            == template_id
            and record.get("selection_reason")
            == "minimum_items"
        )
    ]

    if len(candidates) != 1:
        raise RuntimeError(
            f"Sampel minimum_items {template_id} "
            f"ditemukan {len(candidates)} kali."
        )

    selected_records.append(candidates[0])


inspection_records = []
errors = []

for selected in selected_records:
    document_id = str(
        selected["document_id"]
    )

    template_id = str(
        selected["template_id"]
    )

    manifest_record = text_layer_records.get(
        document_id
    )

    if manifest_record is None:
        errors.append(
            {
                "document_id": document_id,
                "error": (
                    "Tidak ditemukan pada "
                    "text-layer manifest"
                ),
            }
        )
        continue

    text_layer_path = Path(
        manifest_record["text_layer_path"]
    )

    if not text_layer_path.is_file():
        errors.append(
            {
                "document_id": document_id,
                "error": "Text-layer file tidak ada",
            }
        )
        continue

    actual_checksum = sha256_file(
        text_layer_path
    )

    if (
        actual_checksum
        != manifest_record["text_layer_sha256"]
    ):
        errors.append(
            {
                "document_id": document_id,
                "error": "Checksum tidak cocok",
            }
        )
        continue

    layer = load_json(text_layer_path)

    identity = layer.get("document", {})

    if (
        layer.get("status") != "PASSED"
        or identity.get("document_id")
        != document_id
        or identity.get("template_id")
        != template_id
    ):
        errors.append(
            {
                "document_id": document_id,
                "error": "Identitas layer tidak valid",
            }
        )
        continue

    print()
    print("=" * 100)
    print(
        f"{template_id} | {document_id} | "
        f"language={selected.get('language')} | "
        f"items={selected.get('item_count')}"
    )
    print("=" * 100)

    lines = sorted(
        layer.get("lines", []),
        key=lambda row: (
            int(row.get("page_number", 1)),
            float(
                row.get(
                    "bbox_points",
                    [0, 0, 0, 0],
                )[1]
            ),
            float(
                row.get(
                    "bbox_points",
                    [0, 0, 0, 0],
                )[0]
            ),
        ),
    )

    for line in lines:
        bbox = line.get(
            "bbox_points",
            [0, 0, 0, 0],
        )

        print(
            f"[{int(line.get('line_index', -1)):03d}] "
            f"p={int(line.get('page_number', 1))} "
            f"x0={float(bbox[0]):7.2f} "
            f"y0={float(bbox[1]):7.2f} "
            f"x1={float(bbox[2]):7.2f} "
            f"y1={float(bbox[3]):7.2f} | "
            f"{line.get('text', '')}"
        )

    inspection_records.append(
        {
            "template_id": template_id,
            "document_id": document_id,
            "language": selected.get(
                "language"
            ),
            "item_count": selected.get(
                "item_count"
            ),
            "line_count": len(lines),
            "token_count": len(
                layer.get("tokens", [])
            ),
            "checksum_valid": True,
            "status": "VALID",
        }
    )


controls = [
    {
        "control": "contract_status",
        "expected": (
            "FROZEN_FOR_DEVELOPMENT_BASELINE"
        ),
        "actual": contract.get("status"),
    },
    {
        "control": "text_layer_status",
        "expected": "PASSED",
        "actual": text_layer_manifest.get(
            "status"
        ),
    },
    {
        "control": "templates_inspected",
        "expected": 6,
        "actual": len(inspection_records),
    },
    {
        "control": "inspection_errors",
        "expected": 0,
        "actual": len(errors),
    },
    {
        "control": "ground_truth_opened",
        "expected": 0,
        "actual": 0,
    },
    {
        "control": "artifact_writes",
        "expected": 0,
        "actual": 0,
    },
]

for control in controls:
    control["status"] = (
        "VALID"
        if control["expected"]
        == control["actual"]
        else "INVALID"
    )

print()
print("INSPECTION SUMMARY")
display(pd.DataFrame(inspection_records))
display(pd.DataFrame(controls))

if errors:
    display(pd.DataFrame(errors))

invalid_controls = [
    control["control"]
    for control in controls
    if control["status"] != "VALID"
]

if invalid_controls:
    raise RuntimeError(
        "CELL 10B.1 INSPECTION FAILED. "
        f"Kontrol tidak valid: "
        f"{invalid_controls}"
    )

print()
print("Templates inspected  : 6")
print("Ground truth opened  : 0")
print("Artifact writes      : 0")
print("Validation opened    : 0")
print("Test opened          : 0")
print()
print(
    "✅ CELL 10B.1 PASSED — struktur line dan "
    "bounding box enam template berhasil ditampilkan "
    "secara read-only."
)



TPL-01 | INV-SYN-000002 | language=id | items=2
[000] p=1 x0= 458.70 y0=  29.00 x1= 553.28 y1=  60.67 | INVOICE
[001] p=1 x0=  42.00 y0=  34.00 x1= 185.55 y1=  45.02 | SYNTHETIC COMMERCE DOCUMENT
[002] p=1 x0=  42.00 y0=  58.00 x1= 122.50 y1=  67.34 | INV-SYN-000002 · TPL-01
[003] p=1 x0=  52.00 y0= 122.00 x1=  68.37 y1= 131.23 | DARI
[004] p=1 x0= 320.00 y0= 122.00 x1= 392.21 y1= 131.23 | DITAGIHKAN KEPADA
[005] p=1 x0=  52.00 y0= 143.00 x1= 152.03 y1= 154.02 | PT Swaranusa Logistik Uji
[006] p=1 x0= 320.00 y0= 143.00 x1= 430.27 y1= 154.02 | CV Terasena Kreasi Simulasi
[007] p=1 x0=  52.00 y0= 166.00 x1= 115.87 y1= 175.34 | Koridor Fiktif No. 157
[008] p=1 x0= 320.00 y0= 166.00 x1= 401.63 y1= 175.34 | Kompleks Data Uji No. 985
[009] p=1 x0=  52.00 y0= 176.04 x1= 108.31 y1= 185.39 | Blok L-83, Kota Uji
[010] p=1 x0= 320.00 y0= 176.04 x1= 377.82 y1= 185.39 | Blok O-61, Kota Uji
[011] p=1 x0=  52.00 y0= 186.09 x1= 102.28 y1= 195.43 | Kode Pos 90002
[012] p=1 x0= 320.00 y0= 186.09 x1= 37

,template_id,document_id,language,item_count,line_count,token_count,checksum_valid,status
0,TPL-01,INV-SYN-000002,id,2,52,114,True,VALID
1,TPL-02,INV-SYN-000025,id,2,52,115,True,VALID
2,TPL-03,INV-SYN-000043,id,2,51,111,True,VALID
3,TPL-04,INV-SYN-000070,id,2,55,117,True,VALID
4,TPL-05,INV-SYN-000088,id,2,52,113,True,VALID
5,TPL-06,INV-SYN-000107,id,2,54,118,True,VALID


,control,expected,actual,status
0,contract_status,FROZEN_FOR_DEVELOPMENT_BASELINE,FROZEN_FOR_DEVELOPMENT_BASELINE,VALID
1,text_layer_status,PASSED,PASSED,VALID
2,templates_inspected,6,6,VALID
3,inspection_errors,0,0,VALID
4,ground_truth_opened,0,0,VALID
5,artifact_writes,0,0,VALID



Templates inspected  : 6
Ground truth opened  : 0
Artifact writes      : 0
Validation opened    : 0
Test opened          : 0

✅ CELL 10B.1 PASSED — struktur line dan bounding box enam template berhasil ditampilkan secara read-only.


In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import re
import unicodedata
from collections import Counter
from datetime import date
from decimal import Decimal, InvalidOperation
from pathlib import Path

import pandas as pd
from IPython.display import display


# ============================================================
# CELL 10C — DETERMINISTIC RULE-BASED FIELD PARSER BASELINE
# ============================================================

BUILD_ROOT = Path(
    "/content/drive/MyDrive/InvoiceFlow-AI-Data/interim/"
    "synthetic_v1_build/20260904T150025Z"
)
BENCHMARK_ROOT = BUILD_ROOT / "ocr_benchmark"
FIELD_ROOT = BENCHMARK_ROOT / "field_extraction"

CONTRACT_PATH = FIELD_ROOT / "field_extraction_contract_v1.json"
TEXT_LAYER_MANIFEST_PATH = (
    FIELD_ROOT / "unified_text_layer_manifest.json"
)

PREDICTION_ROOT = (
    FIELD_ROOT / "predictions" / "rule_based_baseline_v1_0_1"
)
SUMMARY_PATH = (
    FIELD_ROOT / "rule_based_baseline_v1_0_1_summary.csv"
)
MANIFEST_PATH = (
    FIELD_ROOT / "rule_based_baseline_v1_0_1_manifest.json"
)

PARSER_ID = "RULE-BASED-INVOICE-PARSER-V1"
PARSER_VERSION = "1.0.1"
EXPECTED_DOCUMENTS = 18
EXPECTED_TEMPLATES = 6
EXPECTED_TARGET_FIELDS = 16
EXPECTED_SCALAR_FIELDS = 12
EXPECTED_REPEATING_FIELDS = 4

SUPPORTED_CURRENCIES = {"IDR", "USD", "EUR", "GBP"}


# ============================================================
# LABEL DICTIONARY — LANGUAGE-AWARE, TEMPLATE-INDEPENDENT
# ============================================================

LABEL_ALIASES = {
    "invoice_number": {
        "nomor invoice",
        "invoice number",
        "invoice no",
        "invoice #",
    },
    "invoice_date": {
        "tanggal invoice",
        "invoice date",
        "date of invoice",
    },
    "due_date": {
        "jatuh tempo",
        "due date",
        "payment due",
    },
    "currency": {
        "mata uang",
        "currency",
    },
    "vendor_anchor": {
        "dari",
        "from",
        "vendor",
        "seller",
    },
    "buyer_anchor": {
        "ditagihkan kepada",
        "tagihan kepada",
        "bill to",
        "billed to",
        "buyer",
        "customer",
    },
    "tax_identifier": {
        "id pajak",
        "nomor pajak",
        "tax id",
        "tax identifier",
        "vat id",
        "vat number",
    },
    "description": {
        "deskripsi",
        "description",
        "item description",
    },
    "quantity": {
        "kuantitas",
        "qty",
        "quantity",
    },
    "unit_price": {
        "harga satuan",
        "unit price",
        "price",
    },
    "line_total": {
        "jumlah",
        "line total",
        "amount",
    },
    "subtotal": {
        "subtotal",
        "sub total",
    },
    "tax": {
        "pajak",
        "tax",
        "vat",
    },
    "discount": {
        "diskon",
        "discount",
    },
    "total": {
        "total",
        "grand total",
        "total due",
        "amount due",
    },
}

MONTHS = {
    "januari": 1,
    "january": 1,
    "jan": 1,
    "februari": 2,
    "february": 2,
    "feb": 2,
    "maret": 3,
    "march": 3,
    "mar": 3,
    "april": 4,
    "apr": 4,
    "mei": 5,
    "may": 5,
    "juni": 6,
    "june": 6,
    "jun": 6,
    "juli": 7,
    "july": 7,
    "jul": 7,
    "agustus": 8,
    "august": 8,
    "aug": 8,
    "september": 9,
    "sep": 9,
    "sept": 9,
    "oktober": 10,
    "october": 10,
    "oct": 10,
    "november": 11,
    "nov": 11,
    "desember": 12,
    "december": 12,
    "dec": 12,
}

ADDRESS_TERMS = {
    "jalan",
    "jl",
    "street",
    "road",
    "avenue",
    "kompleks",
    "kawasan",
    "koridor",
    "blok",
    "block",
    "kota",
    "city",
    "kode pos",
    "postal code",
    "postcode",
    "zip code",
}


# ============================================================
# FILE AND SERIALIZATION HELPERS
# ============================================================

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def load_json(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as handle:
        value = json.load(handle)
    if not isinstance(value, dict):
        raise TypeError(f"JSON harus berupa object: {path}")
    return value


def canonical_json(value) -> str:
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    )


def atomic_write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(path.name + ".tmp")
    temporary_path.write_text(text, encoding="utf-8")
    os.replace(temporary_path, path)


def atomic_write_json(path: Path, value: dict) -> None:
    atomic_write_text(
        path,
        json.dumps(
            value,
            indent=2,
            ensure_ascii=False,
            default=str,
        )
        + "\n",
    )


def require_file(path: Path, label: str) -> None:
    if not path.is_file():
        raise FileNotFoundError(f"{label} tidak ditemukan: {path}")


# ============================================================
# TEXT NORMALIZATION AND GEOMETRY
# ============================================================

def normalize_spaces(value: str) -> str:
    value = unicodedata.normalize("NFKC", str(value))
    return re.sub(r"\s+", " ", value).strip()


def normalize_for_match(value: str) -> str:
    value = normalize_spaces(value).casefold()
    value = value.replace("&", " and ")
    value = re.sub(r"[^a-z0-9]+", " ", value)
    return re.sub(r"\s+", " ", value).strip()


def normalized_aliases(key: str) -> set[str]:
    return {
        normalize_for_match(alias)
        for alias in LABEL_ALIASES[key]
    }


NORMALIZED_LABEL_ALIASES = {
    key: normalized_aliases(key)
    for key in LABEL_ALIASES
}


def line_matches_alias(
    line: dict,
    alias_key: str,
    *,
    allow_suffix: bool = False,
) -> bool:
    normalized = line["normalized_text"]
    for alias in NORMALIZED_LABEL_ALIASES[alias_key]:
        if normalized == alias:
            return True
        if allow_suffix and normalized.startswith(alias + " "):
            return True
    return False


def prepare_lines(text_layer: dict) -> list[dict]:
    prepared = []
    source_lines = text_layer.get("lines", [])
    if not isinstance(source_lines, list):
        raise TypeError("text_layer.lines harus berupa list.")

    for fallback_index, source_line in enumerate(source_lines):
        if not isinstance(source_line, dict):
            continue

        text = normalize_spaces(source_line.get("text", ""))
        bbox = source_line.get("bbox_points", [])
        if not text or not isinstance(bbox, list) or len(bbox) != 4:
            continue

        x0, y0, x1, y1 = map(float, bbox)
        if x1 <= x0 or y1 <= y0:
            continue

        prepared.append(
            {
                "line_index": int(
                    source_line.get("line_index", fallback_index)
                ),
                "page_number": int(
                    source_line.get("page_number", 1)
                ),
                "text": text,
                "normalized_text": normalize_for_match(text),
                "x0": x0,
                "y0": y0,
                "x1": x1,
                "y1": y1,
                "cx": (x0 + x1) / 2.0,
                "cy": (y0 + y1) / 2.0,
                "width": x1 - x0,
                "height": y1 - y0,
                "bbox_points": [x0, y0, x1, y1],
                "confidence": float(source_line.get("confidence", 1.0)),
                "source": str(source_line.get("source", "UNKNOWN")),
            }
        )

    return sorted(
        prepared,
        key=lambda line: (
            line["page_number"],
            line["y0"],
            line["x0"],
            line["line_index"],
        ),
    )


def evidence_from_lines(lines: list[dict]) -> list[dict]:
    return [
        {
            "line_index": line["line_index"],
            "page_number": line["page_number"],
            "text": line["text"],
            "bbox_points": [round(value, 4) for value in line["bbox_points"]],
        }
        for line in lines
    ]


def find_label_lines(
    lines: list[dict],
    alias_key: str,
    *,
    minimum_y: float | None = None,
    maximum_y: float | None = None,
    allow_suffix: bool = False,
) -> list[dict]:
    matches = []
    for line in lines:
        if minimum_y is not None and line["y0"] < minimum_y:
            continue
        if maximum_y is not None and line["y0"] > maximum_y:
            continue
        if line_matches_alias(
            line,
            alias_key,
            allow_suffix=allow_suffix,
        ):
            matches.append(line)
    return matches


def choose_first_label(
    lines: list[dict],
    alias_key: str,
    *,
    minimum_y: float | None = None,
    maximum_y: float | None = None,
    allow_suffix: bool = False,
) -> dict | None:
    matches = find_label_lines(
        lines,
        alias_key,
        minimum_y=minimum_y,
        maximum_y=maximum_y,
        allow_suffix=allow_suffix,
    )
    if not matches:
        return None
    return min(matches, key=lambda line: (line["y0"], line["x0"]))


# ============================================================
# VALUE NORMALIZERS
# ============================================================

def decimal_to_string(value: Decimal) -> str:
    if value == value.to_integral():
        return str(value.quantize(Decimal("1")))
    normalized = format(value.normalize(), "f")
    return normalized.rstrip("0").rstrip(".")


def parse_decimal_number(raw_value: str) -> Decimal | None:
    text = normalize_spaces(raw_value)
    text = re.sub(r"[^0-9,\.\-]", "", text)
    if not text or text in {"-", ".", ","}:
        return None

    negative = text.startswith("-")
    text = text.lstrip("-")

    if "," in text and "." in text:
        decimal_separator = (
            "," if text.rfind(",") > text.rfind(".") else "."
        )
        thousands_separator = "." if decimal_separator == "," else ","
        text = text.replace(thousands_separator, "")
        text = text.replace(decimal_separator, ".")
    elif "," in text or "." in text:
        separator = "," if "," in text else "."
        pieces = text.split(separator)

        if len(pieces) == 2 and len(pieces[-1]) in {1, 2}:
            text = pieces[0] + "." + pieces[1]
        elif len(pieces) > 2 and len(pieces[-1]) in {1, 2}:
            text = "".join(pieces[:-1]) + "." + pieces[-1]
        else:
            text = "".join(pieces)

    if negative:
        text = "-" + text

    try:
        return Decimal(text)
    except InvalidOperation:
        return None


def normalize_money(raw_value: str) -> str | None:
    value = parse_decimal_number(raw_value)
    if value is None:
        return None
    return decimal_to_string(value)


def normalize_quantity(raw_value: str) -> str | None:
    text = normalize_spaces(raw_value)
    if not re.fullmatch(r"[-+]?\d+(?:[\.,]\d+)?", text):
        return None
    value = parse_decimal_number(text)
    if value is None:
        return None
    return decimal_to_string(value)


def normalize_currency(raw_value: str) -> str | None:
    match = re.fullmatch(
        r"\s*(IDR|USD|EUR|GBP)\s*",
        normalize_spaces(raw_value),
        flags=re.IGNORECASE,
    )
    if not match:
        return None
    currency = match.group(1).upper()
    return currency if currency in SUPPORTED_CURRENCIES else None


def normalize_date(raw_value: str) -> str | None:
    text = normalize_for_match(raw_value)

    iso_match = re.fullmatch(
        r"(\d{4})[\-/](\d{1,2})[\-/](\d{1,2})",
        text,
    )
    if iso_match:
        year, month, day = map(int, iso_match.groups())
        try:
            return date(year, month, day).isoformat()
        except ValueError:
            return None

    numeric_match = re.fullmatch(
        r"(\d{1,2})[\-/](\d{1,2})[\-/](\d{4})",
        text,
    )
    if numeric_match:
        day, month, year = map(int, numeric_match.groups())
        try:
            return date(year, month, day).isoformat()
        except ValueError:
            return None

    day_first = re.fullmatch(
        r"(\d{1,2}) ([a-z]+) (\d{4})",
        text,
    )
    month_first = re.fullmatch(
        r"([a-z]+) (\d{1,2}) (\d{4})",
        text,
    )

    if day_first:
        day = int(day_first.group(1))
        month = MONTHS.get(day_first.group(2))
        year = int(day_first.group(3))
    elif month_first:
        month = MONTHS.get(month_first.group(1))
        day = int(month_first.group(2))
        year = int(month_first.group(3))
    else:
        return None

    if month is None:
        return None

    try:
        return date(year, month, day).isoformat()
    except ValueError:
        return None


def normalize_identifier(raw_value: str) -> str | None:
    text = normalize_spaces(raw_value)
    normalized = normalize_for_match(text)

    for alias in sorted(
        NORMALIZED_LABEL_ALIASES["tax_identifier"],
        key=len,
        reverse=True,
    ):
        if normalized.startswith(alias):
            original_pattern = re.compile(
                r"^\s*"
                + r"[\W_]*".join(
                    re.escape(part)
                    for part in alias.split()
                )
                + r"\s*[:#\-]?\s*",
                flags=re.IGNORECASE,
            )
            text = original_pattern.sub("", text, count=1)
            break

    text = normalize_spaces(text).upper()
    return text or None


def normalize_string(raw_value: str) -> str | None:
    text = normalize_spaces(raw_value)
    return text or None


def normalize_invoice_number(raw_value: str) -> str | None:
    text = normalize_spaces(raw_value).upper()
    if " " in text:
        return None
    if not re.search(r"[A-Z]", text) or not re.search(r"\d", text):
        return None
    if not re.fullmatch(r"[A-Z]{2,12}[-/][A-Z0-9][A-Z0-9./-]{4,}", text):
        return None
    return text


def looks_like_money(raw_value: str) -> bool:
    text = normalize_spaces(raw_value)
    has_digit = bool(re.search(r"\d", text))
    has_currency = bool(
        re.search(r"\b(?:IDR|USD|EUR|GBP)\b", text, re.IGNORECASE)
    )
    return has_digit and has_currency and normalize_money(text) is not None


def normalize_currency_money(raw_value: str) -> str | None:
    """Normalize only values that visibly carry a currency code."""
    if not looks_like_money(raw_value):
        return None
    return normalize_money(raw_value)


def looks_like_date(raw_value: str) -> bool:
    return normalize_date(raw_value) is not None


def looks_like_currency(raw_value: str) -> bool:
    return normalize_currency(raw_value) is not None


# ============================================================
# GENERIC LABEL-TO-VALUE ASSOCIATION
# ============================================================

def select_labeled_value(
    lines: list[dict],
    anchor: dict | None,
    validator,
    *,
    minimum_y: float | None = None,
    maximum_vertical_gap: float = 52.0,
) -> tuple[dict | None, str | None]:
    if anchor is None:
        return None, None

    candidates = []
    same_row_tolerance = max(4.0, anchor["height"] * 1.25)

    for line in lines:
        if line["line_index"] == anchor["line_index"]:
            continue
        if line["page_number"] != anchor["page_number"]:
            continue
        if minimum_y is not None and line["y0"] < minimum_y:
            continue

        normalized_value = validator(line["text"])
        if normalized_value is None:
            continue

        y_distance = abs(line["cy"] - anchor["cy"])
        is_same_row = (
            y_distance <= same_row_tolerance
            and line["x0"] >= anchor["x1"] - 4.0
        )

        vertical_gap = line["y0"] - anchor["y1"]
        horizontal_center_gap = abs(line["cx"] - anchor["cx"])
        below_width_limit = max(
            105.0,
            anchor["width"] * 2.75,
        )
        is_below = (
            -2.0 <= vertical_gap <= maximum_vertical_gap
            and horizontal_center_gap <= below_width_limit
        )

        if not is_same_row and not is_below:
            continue

        if is_same_row:
            score = (
                y_distance * 6.0
                + max(0.0, line["x0"] - anchor["x1"]) * 0.08
            )
            relation = "SAME_ROW_RIGHT"
        else:
            score = (
                max(0.0, vertical_gap)
                + horizontal_center_gap * 0.08
            )
            relation = "BELOW_ALIGNED"

        candidates.append(
            (score, line["y0"], line["x0"], line, normalized_value, relation)
        )

    if not candidates:
        return None, None

    _, _, _, best_line, normalized_value, relation = min(candidates)
    best_line = {**best_line, "relation_to_label": relation}
    return best_line, normalized_value


def field_prediction(
    raw_value: str | None,
    normalized_value: str | None,
    method: str,
    evidence_lines: list[dict],
    *,
    rule_score: float,
) -> dict:
    return {
        "raw_value": raw_value,
        "normalized_value": normalized_value,
        "method": method,
        "rule_score": round(float(rule_score), 4),
        "rule_score_is_calibrated_probability": False,
        "evidence": evidence_from_lines(evidence_lines),
    }


def empty_prediction(method: str) -> dict:
    return field_prediction(
        None,
        None,
        method,
        [],
        rule_score=0.0,
    )


def extract_labeled_scalar(
    lines: list[dict],
    alias_key: str,
    validator,
    *,
    method: str,
    rule_score: float,
    minimum_y: float | None = None,
    allow_label_suffix: bool = False,
    maximum_vertical_gap: float = 52.0,
) -> dict:
    anchor = choose_first_label(
        lines,
        alias_key,
        minimum_y=minimum_y,
        allow_suffix=allow_label_suffix,
    )
    value_line, normalized_value = select_labeled_value(
        lines,
        anchor,
        validator,
        minimum_y=minimum_y,
        maximum_vertical_gap=maximum_vertical_gap,
    )

    if anchor is None or value_line is None:
        return empty_prediction(method)

    return field_prediction(
        value_line["text"],
        normalized_value,
        method + ":" + value_line["relation_to_label"],
        [anchor, value_line],
        rule_score=rule_score,
    )


# ============================================================
# METADATA PARSER
# ============================================================

INVOICE_NUMBER_PATTERN = re.compile(
    r"\b(?:FTR|INV)-\d{4}-\d{2}-\d{6}\b",
    flags=re.IGNORECASE,
)


def extract_invoice_number(lines: list[dict]) -> dict:
    matches = []
    for line in lines:
        match = INVOICE_NUMBER_PATTERN.search(line["text"])
        if match:
            matches.append((line, match.group(0).upper()))

    if matches:
        line, normalized_value = min(
            matches,
            key=lambda item: (item[0]["y0"], item[0]["x0"]),
        )
        return field_prediction(
            normalized_value,
            normalized_value,
            "REGEX:BILINGUAL_INVOICE_NUMBER",
            [line],
            rule_score=0.99,
        )

    labeled_prediction = extract_labeled_scalar(
        lines,
        "invoice_number",
        normalize_invoice_number,
        method="LABEL_GEOMETRY:INVOICE_NUMBER",
        rule_score=0.95,
    )
    if labeled_prediction["normalized_value"] is not None:
        return labeled_prediction

    return empty_prediction("REGEX_AND_LABEL_GEOMETRY:INVOICE_NUMBER")


def parse_metadata(lines: list[dict]) -> dict[str, dict]:
    return {
        "invoice_number": extract_invoice_number(lines),
        "invoice_date": extract_labeled_scalar(
            lines,
            "invoice_date",
            normalize_date,
            method="LABEL_GEOMETRY:INVOICE_DATE",
            rule_score=0.96,
        ),
        "due_date": extract_labeled_scalar(
            lines,
            "due_date",
            normalize_date,
            method="LABEL_GEOMETRY:DUE_DATE",
            rule_score=0.96,
        ),
        "currency": extract_labeled_scalar(
            lines,
            "currency",
            normalize_currency,
            method="LABEL_GEOMETRY:CURRENCY",
            rule_score=0.98,
        ),
    }


# ============================================================
# PARTY PARSER
# ============================================================

def is_any_known_label(line: dict) -> bool:
    return any(
        line_matches_alias(line, key, allow_suffix=True)
        for key in NORMALIZED_LABEL_ALIASES
    )


def looks_like_party_name(line: dict) -> bool:
    text = line["text"]
    normalized = line["normalized_text"]

    if not re.search(r"[A-Za-z]", text):
        return False
    if "@" in text or re.search(r"\bhttps?://", text, re.IGNORECASE):
        return False
    if re.search(r"\+?\d[\d\-() ]{7,}", text):
        return False
    if INVOICE_NUMBER_PATTERN.search(text):
        return False
    if re.search(r"\bINV-SYN-\d+\b", text, re.IGNORECASE):
        return False
    if looks_like_date(text) or looks_like_money(text):
        return False
    if is_any_known_label(line):
        return False
    if any(term in normalized for term in ADDRESS_TERMS):
        return False
    if normalized.startswith("synthetic") or normalized.startswith("data sintetis"):
        return False
    if len(normalized.split()) < 2:
        return False

    return True


def side_boundary(
    first_anchor: dict | None,
    second_anchor: dict | None,
) -> float | None:
    if first_anchor is None or second_anchor is None:
        return None
    if abs(first_anchor["cy"] - second_anchor["cy"]) > 32.0:
        return None
    return (first_anchor["cx"] + second_anchor["cx"]) / 2.0


def extract_party_name(
    lines: list[dict],
    anchor: dict | None,
    other_anchor: dict | None,
    party: str,
) -> dict:
    method = f"PARTY_ANCHOR_GEOMETRY:{party.upper()}_NAME"
    if anchor is None:
        return empty_prediction(method)

    boundary = side_boundary(anchor, other_anchor)
    candidates = []

    for line in lines:
        if line["page_number"] != anchor["page_number"]:
            continue
        if line["line_index"] == anchor["line_index"]:
            continue

        vertical_gap = line["y0"] - anchor["y1"]
        if not 0.0 <= vertical_gap <= 82.0:
            continue
        if not looks_like_party_name(line):
            continue

        if boundary is not None:
            if party == "vendor" and line["cx"] >= boundary:
                continue
            if party == "buyer" and line["cx"] < boundary:
                continue
        elif (
            other_anchor is not None
            and other_anchor["cy"] > anchor["cy"] + 32.0
            and line["cy"] >= other_anchor["cy"] - 2.0
        ):
            continue

        horizontal_gap = abs(line["x0"] - anchor["x0"])
        score = vertical_gap * 5.0 + horizontal_gap * 0.35
        candidates.append((score, line["y0"], line["x0"], line))

    if not candidates:
        return empty_prediction(method)

    _, _, _, name_line = min(candidates)
    return field_prediction(
        name_line["text"],
        normalize_string(name_line["text"]),
        method,
        [anchor, name_line],
        rule_score=0.91,
    )


def extract_party_tax_identifier(
    lines: list[dict],
    anchor: dict | None,
    other_anchor: dict | None,
    party: str,
) -> dict:
    method = f"PARTY_ANCHOR_GEOMETRY:{party.upper()}_TAX_IDENTIFIER"
    if anchor is None:
        return empty_prediction(method)

    tax_lines = find_label_lines(
        lines,
        "tax_identifier",
        allow_suffix=True,
    )
    candidates = []

    for line in tax_lines:
        if line["page_number"] != anchor["page_number"]:
            continue

        anchors_are_vertically_stacked = bool(
            other_anchor is not None
            and abs(anchor["cy"] - other_anchor["cy"]) > 32.0
            and abs(anchor["x0"] - other_anchor["x0"]) < 40.0
        )
        if anchors_are_vertically_stacked:
            if (
                anchor["cy"] < other_anchor["cy"]
                and line["cy"] >= other_anchor["cy"]
            ):
                continue
            if (
                anchor["cy"] > other_anchor["cy"]
                and line["cy"] <= anchor["cy"]
            ):
                continue

        normalized_value = normalize_identifier(line["text"])
        if not normalized_value:
            continue

        score = (
            abs(line["x0"] - anchor["x0"])
            + abs(line["cy"] - anchor["cy"]) * 0.25
        )
        candidates.append((score, line["y0"], line["x0"], line, normalized_value))

    if not candidates:
        return empty_prediction(method)

    _, _, _, tax_line, normalized_value = min(candidates)
    return field_prediction(
        tax_line["text"],
        normalized_value,
        method,
        [anchor, tax_line],
        rule_score=0.93,
    )


def parse_parties(lines: list[dict]) -> dict[str, dict]:
    vendor_anchor = choose_first_label(lines, "vendor_anchor")
    buyer_anchor = choose_first_label(lines, "buyer_anchor")

    return {
        "vendor.name": extract_party_name(
            lines,
            vendor_anchor,
            buyer_anchor,
            "vendor",
        ),
        "vendor.tax_identifier": extract_party_tax_identifier(
            lines,
            vendor_anchor,
            buyer_anchor,
            "vendor",
        ),
        "buyer.name": extract_party_name(
            lines,
            buyer_anchor,
            vendor_anchor,
            "buyer",
        ),
        "buyer.tax_identifier": extract_party_tax_identifier(
            lines,
            buyer_anchor,
            vendor_anchor,
            "buyer",
        ),
    }


# ============================================================
# ITEM TABLE PARSER
# ============================================================

def locate_table_header(lines: list[dict]) -> dict | None:
    descriptions = find_label_lines(lines, "description")
    quantities = find_label_lines(lines, "quantity")
    unit_prices = find_label_lines(lines, "unit_price")
    line_totals = find_label_lines(lines, "line_total")

    header_candidates = []
    for description in descriptions:
        for quantity in quantities:
            if quantity["page_number"] != description["page_number"]:
                continue
            if abs(quantity["cy"] - description["cy"]) > 4.0:
                continue
            if quantity["cx"] <= description["cx"]:
                continue

            compatible_unit_prices = [
                line
                for line in unit_prices
                if line["page_number"] == description["page_number"]
                and abs(line["cy"] - description["cy"]) <= 4.0
                and line["cx"] > quantity["cx"]
            ]
            compatible_line_totals = [
                line
                for line in line_totals
                if line["page_number"] == description["page_number"]
                and abs(line["cy"] - description["cy"]) <= 4.0
                and line["cx"] > quantity["cx"]
            ]

            for unit_price in compatible_unit_prices:
                for line_total in compatible_line_totals:
                    if line_total["cx"] <= unit_price["cx"]:
                        continue
                    vertical_spread = max(
                        description["cy"],
                        quantity["cy"],
                        unit_price["cy"],
                        line_total["cy"],
                    ) - min(
                        description["cy"],
                        quantity["cy"],
                        unit_price["cy"],
                        line_total["cy"],
                    )
                    header_candidates.append(
                        (
                            vertical_spread,
                            description["y0"],
                            {
                                "description": description,
                                "quantity": quantity,
                                "unit_price": unit_price,
                                "line_total": line_total,
                            },
                        )
                    )

    if not header_candidates:
        return None

    _, _, header = min(header_candidates, key=lambda item: item[:2])
    return header


def cluster_lines_by_row(
    lines: list[dict],
    tolerance: float = 2.8,
) -> list[list[dict]]:
    clusters = []

    for line in sorted(lines, key=lambda value: (value["cy"], value["x0"])):
        if not clusters:
            clusters.append([line])
            continue

        last_cluster = clusters[-1]
        cluster_center = sum(item["cy"] for item in last_cluster) / len(last_cluster)
        if abs(line["cy"] - cluster_center) <= tolerance:
            last_cluster.append(line)
        else:
            clusters.append([line])

    return clusters


def concatenate_cell_lines(lines: list[dict]) -> str:
    return normalize_spaces(
        " ".join(line["text"] for line in sorted(lines, key=lambda item: item["x0"]))
    )


def find_table_stop_y(lines: list[dict], header_y: float) -> float:
    subtotal_labels = find_label_lines(
        lines,
        "subtotal",
        minimum_y=header_y + 5.0,
    )
    if subtotal_labels:
        return min(line["y0"] for line in subtotal_labels)

    page_lines = [line for line in lines if line["page_number"] == 1]
    return max((line["y1"] for line in page_lines), default=10000.0)


def parse_item_table(lines: list[dict]) -> tuple[list[dict], dict]:
    header = locate_table_header(lines)
    if header is None:
        return [], {
            "table_detected": False,
            "header_evidence": [],
            "table_stop_y": None,
            "candidate_row_count": 0,
        }

    header_lines = list(header.values())
    header_y = sum(line["cy"] for line in header_lines) / len(header_lines)
    table_stop_y = find_table_stop_y(lines, header_y)

    centers = {
        key: header[key]["cx"]
        for key in ("description", "quantity", "unit_price", "line_total")
    }
    description_quantity_boundary = (
        centers["description"] + centers["quantity"]
    ) / 2.0
    quantity_unit_boundary = (
        centers["quantity"] + centers["unit_price"]
    ) / 2.0
    unit_total_boundary = (
        centers["unit_price"] + centers["line_total"]
    ) / 2.0

    row_candidates = [
        line
        for line in lines
        if line["page_number"] == header["description"]["page_number"]
        and line["y0"] > max(item["y1"] for item in header_lines) + 3.0
        and line["y1"] < table_stop_y - 1.0
    ]

    row_clusters = cluster_lines_by_row(row_candidates)
    parsed_items = []

    for cluster in row_clusters:
        cell_lines = {
            "description": [],
            "quantity": [],
            "unit_price": [],
            "line_total": [],
        }

        for line in cluster:
            if line["x1"] < header["description"]["x0"] - 2.0:
                continue

            if line["cx"] < description_quantity_boundary:
                column = "description"
            elif line["cx"] < quantity_unit_boundary:
                column = "quantity"
            elif line["cx"] < unit_total_boundary:
                column = "unit_price"
            else:
                column = "line_total"
            cell_lines[column].append(line)

        raw_description = concatenate_cell_lines(cell_lines["description"])
        raw_quantity = concatenate_cell_lines(cell_lines["quantity"])
        raw_unit_price = concatenate_cell_lines(cell_lines["unit_price"])
        raw_line_total = concatenate_cell_lines(cell_lines["line_total"])

        normalized_description = normalize_string(raw_description)
        normalized_quantity = normalize_quantity(raw_quantity)
        normalized_unit_price = normalize_money(raw_unit_price)
        normalized_line_total = normalize_money(raw_line_total)

        if (
            not normalized_description
            or not re.search(r"[A-Za-z]", normalized_description)
            or normalized_quantity is None
            or normalized_unit_price is None
            or normalized_line_total is None
            or not looks_like_money(raw_unit_price)
            or not looks_like_money(raw_line_total)
        ):
            continue

        parsed_items.append(
            {
                "row_number": len(parsed_items) + 1,
                "description": field_prediction(
                    raw_description,
                    normalized_description,
                    "TABLE_GEOMETRY:DESCRIPTION_COLUMN",
                    cell_lines["description"],
                    rule_score=0.94,
                ),
                "quantity": field_prediction(
                    raw_quantity,
                    normalized_quantity,
                    "TABLE_GEOMETRY:QUANTITY_COLUMN",
                    cell_lines["quantity"],
                    rule_score=0.94,
                ),
                "unit_price": field_prediction(
                    raw_unit_price,
                    normalized_unit_price,
                    "TABLE_GEOMETRY:UNIT_PRICE_COLUMN",
                    cell_lines["unit_price"],
                    rule_score=0.94,
                ),
                "line_total": field_prediction(
                    raw_line_total,
                    normalized_line_total,
                    "TABLE_GEOMETRY:LINE_TOTAL_COLUMN",
                    cell_lines["line_total"],
                    rule_score=0.94,
                ),
            }
        )

    diagnostics = {
        "table_detected": True,
        "header_evidence": evidence_from_lines(header_lines),
        "header_centers": {
            key: round(value, 4) for key, value in centers.items()
        },
        "table_stop_y": round(table_stop_y, 4),
        "candidate_row_count": len(row_clusters),
        "parsed_item_count": len(parsed_items),
    }
    return parsed_items, diagnostics


# ============================================================
# FINANCIAL SUMMARY PARSER
# ============================================================

def extract_financial_field(
    lines: list[dict],
    alias_key: str,
    table_stop_y: float | None,
) -> dict:
    minimum_y = None if table_stop_y is None else table_stop_y - 2.0
    return extract_labeled_scalar(
        lines,
        alias_key,
        normalize_currency_money,
        method=f"SUMMARY_LABEL_GEOMETRY:{alias_key.upper()}",
        rule_score=0.96,
        minimum_y=minimum_y,
        allow_label_suffix=alias_key == "tax",
        maximum_vertical_gap=46.0,
    )


def parse_financials(
    lines: list[dict],
    table_stop_y: float | None,
) -> dict[str, dict]:
    return {
        "financials.subtotal": extract_financial_field(
            lines,
            "subtotal",
            table_stop_y,
        ),
        "financials.tax": extract_financial_field(
            lines,
            "tax",
            table_stop_y,
        ),
        "financials.discount": extract_financial_field(
            lines,
            "discount",
            table_stop_y,
        ),
        "financials.total": extract_financial_field(
            lines,
            "total",
            table_stop_y,
        ),
    }


def financial_equation_check(scalars: dict[str, dict]) -> dict:
    keys = {
        "subtotal": "financials.subtotal",
        "tax": "financials.tax",
        "discount": "financials.discount",
        "total": "financials.total",
    }
    values = {}

    for short_name, field_name in keys.items():
        raw_value = scalars[field_name]["normalized_value"]
        if raw_value is None:
            return {
                "executed": False,
                "passed": None,
                "reason": f"MISSING_{short_name.upper()}",
            }
        try:
            values[short_name] = Decimal(raw_value)
        except InvalidOperation:
            return {
                "executed": False,
                "passed": None,
                "reason": f"INVALID_{short_name.upper()}",
            }

    expected_total = (
        values["subtotal"]
        + values["tax"]
        - values["discount"]
    )
    delta = values["total"] - expected_total

    return {
        "executed": True,
        "passed": delta == Decimal("0"),
        "formula": "subtotal + tax - discount = total",
        "expected_total": decimal_to_string(expected_total),
        "observed_total": decimal_to_string(values["total"]),
        "delta": decimal_to_string(delta),
    }


# ============================================================
# DOCUMENT PARSER
# ============================================================

SCALAR_FIELD_NAMES = [
    "invoice_number",
    "invoice_date",
    "due_date",
    "currency",
    "vendor.name",
    "vendor.tax_identifier",
    "buyer.name",
    "buyer.tax_identifier",
    "financials.subtotal",
    "financials.tax",
    "financials.discount",
    "financials.total",
]

REQUIRED_SCALAR_FIELD_NAMES = [
    field_name
    for field_name in SCALAR_FIELD_NAMES
    if field_name
    not in {"vendor.tax_identifier", "buyer.tax_identifier"}
]


def parse_document(text_layer: dict) -> dict:
    lines = prepare_lines(text_layer)
    metadata = parse_metadata(lines)
    parties = parse_parties(lines)
    items, table_diagnostics = parse_item_table(lines)
    financials = parse_financials(
        lines,
        table_diagnostics.get("table_stop_y"),
    )

    scalars = {**metadata, **parties, **financials}
    missing_required_scalars = [
        field_name
        for field_name in REQUIRED_SCALAR_FIELD_NAMES
        if scalars[field_name]["normalized_value"] is None
    ]

    return {
        "scalar_fields": scalars,
        "items": items,
        "diagnostics": {
            "prepared_line_count": len(lines),
            "populated_scalar_field_count": sum(
                prediction["normalized_value"] is not None
                for prediction in scalars.values()
            ),
            "missing_required_scalar_fields": missing_required_scalars,
            "table": table_diagnostics,
            "financial_equation": financial_equation_check(scalars),
        },
    }


# ============================================================
# INPUT GATES — NO GROUND TRUTH IS OPENED IN THIS CELL
# ============================================================

require_file(CONTRACT_PATH, "Field extraction contract")
require_file(TEXT_LAYER_MANIFEST_PATH, "Unified text-layer manifest")

contract = load_json(CONTRACT_PATH)
text_layer_manifest = load_json(TEXT_LAYER_MANIFEST_PATH)

contract_fields = contract.get("field_definitions", [])
scalar_contract_fields = [
    field
    for field in contract_fields
    if field.get("cardinality") != "REPEATING"
]
repeating_contract_fields = [
    field
    for field in contract_fields
    if field.get("cardinality") == "REPEATING"
]

input_gate_values = [
    (
        "contract_status",
        "FROZEN_FOR_DEVELOPMENT_BASELINE",
        contract.get("status"),
    ),
    (
        "text_layer_status",
        "PASSED",
        text_layer_manifest.get("status"),
    ),
    (
        "target_fields",
        EXPECTED_TARGET_FIELDS,
        len(contract_fields),
    ),
    (
        "scalar_fields",
        EXPECTED_SCALAR_FIELDS,
        len(scalar_contract_fields),
    ),
    (
        "repeating_fields",
        EXPECTED_REPEATING_FIELDS,
        len(repeating_contract_fields),
    ),
    (
        "text_layer_records",
        EXPECTED_DOCUMENTS,
        len(text_layer_manifest.get("records", [])),
    ),
]

input_gate_records = [
    {
        "control": name,
        "expected": expected,
        "actual": actual,
        "status": "VALID" if expected == actual else "INVALID",
    }
    for name, expected, actual in input_gate_values
]

invalid_input_gates = [
    row["control"]
    for row in input_gate_records
    if row["status"] != "VALID"
]
if invalid_input_gates:
    display(pd.DataFrame(input_gate_records))
    raise RuntimeError(
        "CELL 10C INPUT GATE FAILED. "
        f"Kontrol tidak valid: {invalid_input_gates}"
    )

source_records = sorted(
    text_layer_manifest["records"],
    key=lambda row: (
        int(row.get("sequence_number", 0)),
        str(row.get("document_id", "")),
    ),
)

source_checksums_before = {
    "contract": sha256_file(CONTRACT_PATH),
    "text_layer_manifest": sha256_file(TEXT_LAYER_MANIFEST_PATH),
}

for source_record in source_records:
    source_path = Path(source_record["text_layer_path"])
    require_file(source_path, "Text-layer checkpoint")
    actual_checksum = sha256_file(source_path)
    if actual_checksum != source_record["text_layer_sha256"]:
        raise RuntimeError(
            "Checksum text layer tidak cocok: "
            f"{source_record['document_id']}"
        )
    source_checksums_before[
        f"text_layer:{source_record['document_id']}"
    ] = actual_checksum


# ============================================================
# PARSER SIGNATURE
# ============================================================

parser_configuration = {
    "parser_id": PARSER_ID,
    "parser_version": PARSER_VERSION,
    "supported_currencies": sorted(SUPPORTED_CURRENCIES),
    "label_aliases": {
        key: sorted(value)
        for key, value in sorted(LABEL_ALIASES.items())
    },
    "month_dictionary": dict(sorted(MONTHS.items())),
    "table_row_tolerance_points": 2.8,
    "ground_truth_as_prediction_input": False,
    "template_specific_branching": False,
}
parser_signature = hashlib.sha256(
    canonical_json(parser_configuration).encode("utf-8")
).hexdigest()


# ============================================================
# EXECUTE OR RECOVER PER-DOCUMENT CHECKPOINTS
# ============================================================

runtime_records = []
manifest_records = []
errors = []
newly_created = 0
recovered = 0

print("=" * 88)
print(f"CELL 10C — {PARSER_ID} — VERSION {PARSER_VERSION}")
print(f"Prediction root: {PREDICTION_ROOT}")
print("=" * 88)
print(f"Menjalankan parser baseline untuk {len(source_records)} dokumen...\n")

for sequence_number, source_record in enumerate(source_records, start=1):
    document_id = str(source_record["document_id"])
    template_id = str(source_record["template_id"])
    text_layer_path = Path(source_record["text_layer_path"])
    prediction_path = (
        PREDICTION_ROOT
        / template_id
        / f"{document_id}_prediction.json"
    )

    try:
        text_layer = load_json(text_layer_path)
        document = text_layer.get("document", {})

        if document.get("document_id") != document_id:
            raise RuntimeError("Document ID text layer tidak cocok.")
        if document.get("template_id") != template_id:
            raise RuntimeError("Template ID text layer tidak cocok.")
        if document.get("split") != "development":
            raise RuntimeError("Parser hanya boleh membuka development split.")

        parsed = parse_document(text_layer)
        scalars = parsed["scalar_fields"]
        items = parsed["items"]
        diagnostics = parsed["diagnostics"]

        prediction_artifact = {
            "schema_version": "1.0.0",
            "status": "EXECUTED",
            "quality_status": "PENDING_GROUND_TRUTH_EVALUATION",
            "parser": {
                "parser_id": PARSER_ID,
                "parser_version": PARSER_VERSION,
                "parser_signature_sha256": parser_signature,
                "approach": "DETERMINISTIC_LABEL_AND_GEOMETRY_RULES",
                "template_specific_branching": False,
                "ground_truth_used_as_prediction_input": False,
            },
            "document": {
                "canonical_invoice_id": document.get(
                    "canonical_invoice_id"
                ),
                "document_id": document_id,
                "template_id": template_id,
                "split": "development",
                "language": document.get("language"),
            },
            "predictions": {
                "scalar_fields": scalars,
                "items": items,
            },
            "diagnostics": diagnostics,
            "source": {
                "text_layer_path": str(text_layer_path),
                "text_layer_sha256": source_record[
                    "text_layer_sha256"
                ],
                "route_id": source_record.get("route_id"),
                "engine": source_record.get("engine"),
            },
            "integrity": {
                "ground_truth_loaded": False,
                "canonical_payload_loaded": False,
                "validation_opened": 0,
                "test_opened": 0,
                "dataset_modifications": 0,
                "source_modifications": 0,
            },
        }

        if prediction_path.exists():
            existing = load_json(prediction_path)
            if canonical_json(existing) != canonical_json(prediction_artifact):
                raise RuntimeError(
                    "Prediction checkpoint sudah ada tetapi berbeda. "
                    "Hapus hanya checkpoint parser v1 ini jika memang "
                    "ingin membangun ulang dengan aturan baru."
                )
            execution = "RECOVERED"
            recovered += 1
        else:
            atomic_write_json(prediction_path, prediction_artifact)
            execution = "NEW"
            newly_created += 1

        prediction_sha256 = sha256_file(prediction_path)
        populated_scalar_count = sum(
            prediction["normalized_value"] is not None
            for prediction in scalars.values()
        )
        missing_required = diagnostics[
            "missing_required_scalar_fields"
        ]
        financial_check = diagnostics["financial_equation"]

        manifest_record = {
            "sequence_number": sequence_number,
            "document_id": document_id,
            "template_id": template_id,
            "language": document.get("language"),
            "route_id": source_record.get("route_id"),
            "populated_scalar_fields": populated_scalar_count,
            "missing_required_scalar_fields": missing_required,
            "parsed_item_count": len(items),
            "table_detected": diagnostics["table"]["table_detected"],
            "financial_equation_executed": financial_check["executed"],
            "financial_equation_passed": financial_check["passed"],
            "prediction_path": str(prediction_path),
            "prediction_sha256": prediction_sha256,
            "text_layer_path": str(text_layer_path),
            "text_layer_sha256": source_record["text_layer_sha256"],
            "status": "EXECUTED",
            "quality_status": "PENDING_GROUND_TRUTH_EVALUATION",
        }
        manifest_records.append(manifest_record)
        runtime_records.append({**manifest_record, "execution": execution})

        print(
            f"[{sequence_number:02d}/{len(source_records):02d}] "
            f"{document_id} | scalars={populated_scalar_count}/12 | "
            f"items={len(items)} | financial="
            f"{financial_check['passed']} | {execution}"
        )

    except Exception as error:
        errors.append(
            {
                "sequence_number": sequence_number,
                "document_id": document_id,
                "template_id": template_id,
                "error_type": type(error).__name__,
                "error": str(error)[:700],
            }
        )
        print(
            f"[{sequence_number:02d}/{len(source_records):02d}] "
            f"{document_id} | ERROR: {type(error).__name__}: {error}"
        )


# ============================================================
# TECHNICAL AUDIT — QUALITY REMAINS PENDING UNTIL CELL 10D
# ============================================================

runtime_table = pd.DataFrame(runtime_records)
manifest_table = pd.DataFrame(manifest_records)

if errors:
    display(pd.DataFrame(errors))

template_counts = Counter(
    row["template_id"] for row in manifest_records
)
missing_field_counts = Counter()
for row in manifest_records:
    missing_field_counts.update(row["missing_required_scalar_fields"])

technical_controls = [
    ("prediction_records", EXPECTED_DOCUMENTS, len(manifest_records)),
    ("prediction_files", EXPECTED_DOCUMENTS, sum(
        Path(row["prediction_path"]).is_file()
        for row in manifest_records
    )),
    ("unique_document_ids", EXPECTED_DOCUMENTS, len({
        row["document_id"] for row in manifest_records
    })),
    ("template_count", EXPECTED_TEMPLATES, len(template_counts)),
    ("documents_with_complete_required_scalars", EXPECTED_DOCUMENTS, sum(
        not row["missing_required_scalar_fields"]
        for row in manifest_records
    )),
    ("documents_with_table_detected", EXPECTED_DOCUMENTS, sum(
        row["table_detected"] is True
        for row in manifest_records
    )),
    ("documents_with_items", EXPECTED_DOCUMENTS, sum(
        row["parsed_item_count"] > 0
        for row in manifest_records
    )),
    ("financial_equations_executed", EXPECTED_DOCUMENTS, sum(
        row["financial_equation_executed"] is True
        for row in manifest_records
    )),
    ("financial_equations_passed", EXPECTED_DOCUMENTS, sum(
        row["financial_equation_passed"] is True
        for row in manifest_records
    )),
    ("processing_errors", 0, len(errors)),
    ("ground_truth_opened", 0, 0),
    ("validation_opened", 0, 0),
    ("test_opened", 0, 0),
]

control_records = [
    {
        "control": name,
        "expected": expected,
        "actual": actual,
        "status": "VALID" if expected == actual else "INVALID",
    }
    for name, expected, actual in technical_controls
]

display(pd.DataFrame(control_records))

if not runtime_table.empty:
    display(
        runtime_table[
            [
                "sequence_number",
                "document_id",
                "template_id",
                "language",
                "populated_scalar_fields",
                "parsed_item_count",
                "table_detected",
                "financial_equation_passed",
                "execution",
                "status",
                "quality_status",
            ]
        ]
    )

if missing_field_counts:
    display(
        pd.DataFrame(
            [
                {"field": field, "missing_documents": count}
                for field, count in sorted(missing_field_counts.items())
            ]
        )
    )

invalid_controls = [
    row["control"]
    for row in control_records
    if row["status"] != "VALID"
]
if invalid_controls:
    raise RuntimeError(
        "CELL 10C PARSER TECHNICAL AUDIT FAILED. "
        f"Kontrol tidak valid: {invalid_controls}. "
        "Jangan lanjut ke Cell 10D; kirim seluruh output ini."
    )


# ============================================================
# SAVE DETERMINISTIC SUMMARY AND MANIFEST
# ============================================================

summary_columns = [
    "sequence_number",
    "document_id",
    "template_id",
    "language",
    "route_id",
    "populated_scalar_fields",
    "parsed_item_count",
    "table_detected",
    "financial_equation_executed",
    "financial_equation_passed",
    "prediction_path",
    "prediction_sha256",
    "text_layer_path",
    "text_layer_sha256",
    "status",
    "quality_status",
]

summary_text = manifest_table[summary_columns].to_csv(
    index=False,
    lineterminator="\n",
)

if SUMMARY_PATH.exists():
    if SUMMARY_PATH.read_text(encoding="utf-8") != summary_text:
        raise RuntimeError(
            "Summary parser v1 sudah ada tetapi berbeda."
        )
    summary_action = "RECOVERED"
else:
    atomic_write_text(SUMMARY_PATH, summary_text)
    summary_action = "CREATED"

baseline_manifest = {
    "schema_version": "1.0.0",
    "status": "EXECUTED",
    "quality_status": "PENDING_GROUND_TRUTH_EVALUATION",
    "stage": "RULE_BASED_FIELD_EXTRACTION_BASELINE",
    "parser": {
        **parser_configuration,
        "parser_signature_sha256": parser_signature,
    },
    "scope": {
        "split": "development",
        "documents": EXPECTED_DOCUMENTS,
        "templates": EXPECTED_TEMPLATES,
        "target_fields": EXPECTED_TARGET_FIELDS,
        "validation_opened": 0,
        "test_opened": 0,
    },
    "records": manifest_records,
    "technical_controls": control_records,
    "artifacts": {
        "prediction_root": str(PREDICTION_ROOT),
        "summary_path": str(SUMMARY_PATH),
        "summary_sha256": sha256_file(SUMMARY_PATH),
    },
    "input_artifacts": {
        "contract": {
            "path": str(CONTRACT_PATH),
            "sha256": source_checksums_before["contract"],
        },
        "text_layer_manifest": {
            "path": str(TEXT_LAYER_MANIFEST_PATH),
            "sha256": source_checksums_before[
                "text_layer_manifest"
            ],
        },
    },
    "integrity": {
        "ground_truth_loaded": False,
        "canonical_payload_loaded": False,
        "ground_truth_used_as_prediction_input": False,
        "dataset_modifications": 0,
        "source_modifications": 0,
        "validation_opened": 0,
        "test_opened": 0,
    },
    "next_stage": {
        "cell": "CELL 10D",
        "action": "DEVELOPMENT_GROUND_TRUTH_EVALUATION",
        "warning": (
            "Status kualitas parser belum PASSED. Ground truth baru "
            "boleh dibuka pada evaluasi Cell 10D setelah seluruh "
            "prediction checkpoint dibekukan."
        ),
    },
}

if MANIFEST_PATH.exists():
    existing_manifest = load_json(MANIFEST_PATH)
    if canonical_json(existing_manifest) != canonical_json(baseline_manifest):
        raise RuntimeError(
            "Manifest parser v1 sudah ada tetapi berbeda."
        )
    manifest_action = "RECOVERED"
else:
    atomic_write_json(MANIFEST_PATH, baseline_manifest)
    manifest_action = "CREATED"


# ============================================================
# POST-WRITE INPUT IMMUTABILITY CHECK
# ============================================================

source_checksums_after = {
    "contract": sha256_file(CONTRACT_PATH),
    "text_layer_manifest": sha256_file(TEXT_LAYER_MANIFEST_PATH),
}
for source_record in source_records:
    source_checksums_after[
        f"text_layer:{source_record['document_id']}"
    ] = sha256_file(Path(source_record["text_layer_path"]))

changed_sources = [
    name
    for name, checksum in source_checksums_before.items()
    if source_checksums_after.get(name) != checksum
]
if changed_sources:
    raise RuntimeError(
        f"Source artifact berubah selama parsing: {changed_sources}"
    )


print()
print(f"Parser ID              : {PARSER_ID}")
print(f"Parser version         : {PARSER_VERSION}")
print(f"Parser SHA-256         : {parser_signature}")
print(f"Documents              : {len(manifest_records)}")
print(f"Templates              : {len(template_counts)}")
print(f"New predictions        : {newly_created}")
print(f"Recovered predictions  : {recovered}")
print(f"Summary action         : {summary_action}")
print(f"Manifest action        : {manifest_action}")
print(f"Prediction root        : {PREDICTION_ROOT}")
print(f"Summary                : {SUMMARY_PATH}")
print(f"Manifest               : {MANIFEST_PATH}")
print(f"Manifest SHA-256       : {sha256_file(MANIFEST_PATH)}")
print("Ground truth opened    : 0")
print("Validation opened      : 0")
print("Test opened            : 0")
print("Dataset modifications  : 0")
print("Source modifications   : 0")
print("Quality status         : PENDING_GROUND_TRUTH_EVALUATION")
print()
print(
    "✅ CELL 10C PASSED — parser rule-based baseline telah "
    "dijalankan dan prediction checkpoint 18 dokumen development "
    "telah dibekukan. Kualitas belum dinyatakan PASSED; lanjutkan "
    "ke Cell 10D untuk evaluasi terhadap ground truth development."
)


CELL 10C — RULE-BASED-INVOICE-PARSER-V1 — VERSION 1.0.1
Prediction root: /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/field_extraction/predictions/rule_based_baseline_v1_0_1
Menjalankan parser baseline untuk 18 dokumen...

[01/18] INV-SYN-000002 | scalars=12/12 | items=2 | financial=True | NEW
[02/18] INV-SYN-000013 | scalars=12/12 | items=8 | financial=True | NEW
[03/18] INV-SYN-000015 | scalars=12/12 | items=8 | financial=True | NEW
[04/18] INV-SYN-000025 | scalars=12/12 | items=2 | financial=True | NEW
[05/18] INV-SYN-000036 | scalars=12/12 | items=7 | financial=True | NEW
[06/18] INV-SYN-000037 | scalars=12/12 | items=7 | financial=True | NEW
[07/18] INV-SYN-000043 | scalars=12/12 | items=2 | financial=True | NEW
[08/18] INV-SYN-000052 | scalars=12/12 | items=7 | financial=True | NEW
[09/18] INV-SYN-000060 | scalars=12/12 | items=7 | financial=True | NEW
[10/18] INV-SYN-000064 | scalars=12/12 | items=8 | financial=True | NEW
[

,control,expected,actual,status
0,prediction_records,18,18,VALID
1,prediction_files,18,18,VALID
2,unique_document_ids,18,18,VALID
3,template_count,6,6,VALID
4,documents_with_complete_required_scalars,18,18,VALID
5,documents_with_table_detected,18,18,VALID
6,documents_with_items,18,18,VALID
7,financial_equations_executed,18,18,VALID
8,financial_equations_passed,18,18,VALID
9,processing_errors,0,0,VALID


,sequence_number,document_id,template_id,language,populated_scalar_fields,parsed_item_count,table_detected,financial_equation_passed,execution,status,quality_status
0,1,INV-SYN-000002,TPL-01,id,12,2,True,True,NEW,EXECUTED,PENDING_GROUND_TRUTH_EVALUATION
1,2,INV-SYN-000013,TPL-01,en,12,8,True,True,NEW,EXECUTED,PENDING_GROUND_TRUTH_EVALUATION
2,3,INV-SYN-000015,TPL-01,en,12,8,True,True,NEW,EXECUTED,PENDING_GROUND_TRUTH_EVALUATION
3,4,INV-SYN-000025,TPL-02,id,12,2,True,True,NEW,EXECUTED,PENDING_GROUND_TRUTH_EVALUATION
4,5,INV-SYN-000036,TPL-02,en,12,7,True,True,NEW,EXECUTED,PENDING_GROUND_TRUTH_EVALUATION
5,6,INV-SYN-000037,TPL-02,en,12,7,True,True,NEW,EXECUTED,PENDING_GROUND_TRUTH_EVALUATION
6,7,INV-SYN-000043,TPL-03,id,12,2,True,True,NEW,EXECUTED,PENDING_GROUND_TRUTH_EVALUATION
7,8,INV-SYN-000052,TPL-03,en,12,7,True,True,NEW,EXECUTED,PENDING_GROUND_TRUTH_EVALUATION
8,9,INV-SYN-000060,TPL-03,en,12,7,True,True,NEW,EXECUTED,PENDING_GROUND_TRUTH_EVALUATION
9,10,INV-SYN-000064,TPL-04,id,12,8,True,True,NEW,EXECUTED,PENDING_GROUND_TRUTH_EVALUATION



Parser ID              : RULE-BASED-INVOICE-PARSER-V1
Parser version         : 1.0.1
Parser SHA-256         : fd8389c1b7f7bada3a786810953bfc39ea619735ae39cd2332cf3cc9b627cf85
Documents              : 18
Templates              : 6
New predictions        : 18
Recovered predictions  : 0
Summary action         : CREATED
Manifest action        : CREATED
Prediction root        : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/field_extraction/predictions/rule_based_baseline_v1_0_1
Summary                : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/field_extraction/rule_based_baseline_v1_0_1_summary.csv
Manifest               : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/field_extraction/rule_based_baseline_v1_0_1_manifest.json
Manifest SHA-256       : d633cc2d79c15337464d79cc0637d837aa631010f25d7d39c3734f92135627dd
Ground tru

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import re
import unicodedata
from collections import Counter, defaultdict
from decimal import Decimal, InvalidOperation
from functools import lru_cache
from pathlib import Path

import pandas as pd
from IPython.display import display


# ============================================================
# CELL 10D — DEVELOPMENT GROUND-TRUTH EVALUATION
# ============================================================

BUILD_ROOT = Path(
    "/content/drive/MyDrive/InvoiceFlow-AI-Data/interim/"
    "synthetic_v1_build/20260904T150025Z"
)
BENCHMARK_ROOT = BUILD_ROOT / "ocr_benchmark"
FIELD_ROOT = BENCHMARK_ROOT / "field_extraction"
GROUND_TRUTH_ROOT = BUILD_ROOT / "rendered_dataset" / "ground_truth"

CONTRACT_PATH = FIELD_ROOT / "field_extraction_contract_v1.json"
PREDICTION_MANIFEST_PATH = (
    FIELD_ROOT / "rule_based_baseline_v1_0_1_manifest.json"
)

EVALUATION_ROOT = (
    FIELD_ROOT
    / "evaluations"
    / "rule_based_baseline_v1_0_1_development_eval_v1_0_1"
)
DOCUMENT_EVALUATION_ROOT = EVALUATION_ROOT / "documents"
DOCUMENT_SUMMARY_PATH = EVALUATION_ROOT / "document_summary.csv"
FIELD_SUMMARY_PATH = EVALUATION_ROOT / "field_summary.csv"
TEMPLATE_SUMMARY_PATH = EVALUATION_ROOT / "template_summary.csv"
MISMATCH_DETAIL_PATH = EVALUATION_ROOT / "mismatch_details.csv"
EVALUATION_MANIFEST_PATH = (
    EVALUATION_ROOT / "development_evaluation_manifest.json"
)

EVALUATOR_ID = "INVOICE-FIELD-EVALUATOR-V1"
EVALUATOR_VERSION = "1.0.1"
EXPECTED_PARSER_ID = "RULE-BASED-INVOICE-PARSER-V1"
EXPECTED_PARSER_VERSION = "1.0.1"
EXPECTED_DOCUMENTS = 18
EXPECTED_TEMPLATES = 6
EXPECTED_SCALAR_FIELDS = 12
EXPECTED_ITEM_FIELDS = 4

ITEM_FIELD_NAMES = [
    "description",
    "quantity",
    "unit_price",
    "line_total",
]

SCALAR_PATHS = {
    "invoice_number": ("invoice_number",),
    "invoice_date": ("invoice_date",),
    "due_date": ("due_date",),
    "currency": ("currency",),
    "vendor.name": ("vendor", "name"),
    "vendor.tax_identifier": ("vendor", "tax_identifier"),
    "buyer.name": ("buyer", "name"),
    "buyer.tax_identifier": ("buyer", "tax_identifier"),
    "financials.subtotal": ("financials", "subtotal"),
    "financials.tax": ("financials", "tax"),
    "financials.discount": ("financials", "discount"),
    "financials.total": ("financials", "total"),
}

MONEY_FIELDS = {
    "financials.subtotal",
    "financials.tax",
    "financials.discount",
    "financials.total",
    "items[].unit_price",
    "items[].line_total",
}

IDENTIFIER_FIELDS = {
    "invoice_number",
    "currency",
    "vendor.tax_identifier",
    "buyer.tax_identifier",
}


# ============================================================
# FILE AND SERIALIZATION HELPERS
# ============================================================

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def load_json(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as handle:
        value = json.load(handle)
    if not isinstance(value, dict):
        raise TypeError(f"JSON harus berupa object: {path}")
    return value


def canonical_json(value) -> str:
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    )


def atomic_write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(path.name + ".tmp")
    temporary_path.write_text(text, encoding="utf-8")
    os.replace(temporary_path, path)


def atomic_write_json(path: Path, value: dict) -> None:
    atomic_write_text(
        path,
        json.dumps(
            value,
            indent=2,
            ensure_ascii=False,
            default=str,
        )
        + "\n",
    )


def require_file(path: Path, label: str) -> None:
    if not path.is_file():
        raise FileNotFoundError(f"{label} tidak ditemukan: {path}")


def require_directory(path: Path, label: str) -> None:
    if not path.is_dir():
        raise FileNotFoundError(f"{label} tidak ditemukan: {path}")


def nested_value(record: dict, path: tuple[str, ...]):
    value = record
    for key in path:
        if not isinstance(value, dict) or key not in value:
            return None
        value = value[key]
    return value


# ============================================================
# TYPE-AWARE NORMALIZATION
# ============================================================

def normalize_spaces(value) -> str:
    if value is None:
        return ""
    text = unicodedata.normalize("NFKC", str(value))
    return re.sub(r"\s+", " ", text).strip()


def decimal_to_string(value: Decimal) -> str:
    if value == value.to_integral():
        return str(value.quantize(Decimal("1")))
    text = format(value.normalize(), "f")
    return text.rstrip("0").rstrip(".")


def normalize_decimal(value) -> str:
    text = normalize_spaces(value)
    if not text:
        return ""
    try:
        return decimal_to_string(Decimal(text))
    except InvalidOperation:
        return text


def normalize_field_value(field_name: str, value) -> str:
    text = normalize_spaces(value)

    if field_name in MONEY_FIELDS or field_name == "items[].quantity":
        return normalize_decimal(text)
    if field_name in IDENTIFIER_FIELDS:
        return text.upper()
    return text


# ============================================================
# EDIT-DISTANCE METRICS
# ============================================================

def levenshtein_distance(reference, hypothesis) -> int:
    reference = list(reference)
    hypothesis = list(hypothesis)

    if len(reference) < len(hypothesis):
        reference, hypothesis = hypothesis, reference

    previous = list(range(len(hypothesis) + 1))
    for reference_index, reference_value in enumerate(reference, start=1):
        current = [reference_index]
        for hypothesis_index, hypothesis_value in enumerate(
            hypothesis,
            start=1,
        ):
            insertion = current[hypothesis_index - 1] + 1
            deletion = previous[hypothesis_index] + 1
            substitution = (
                previous[hypothesis_index - 1]
                + int(reference_value != hypothesis_value)
            )
            current.append(min(insertion, deletion, substitution))
        previous = current
    return previous[-1]


def safe_ratio(numerator: int | float, denominator: int | float) -> float:
    if denominator == 0:
        return 1.0 if numerator == 0 else 0.0
    return float(numerator) / float(denominator)


def safe_error_rate(
    errors: int | float,
    reference_units: int | float,
) -> float:
    if reference_units == 0:
        return 0.0 if errors == 0 else 1.0
    return float(errors) / float(reference_units)


def harmonic_mean(precision: float, recall: float) -> float:
    if precision + recall == 0:
        return 0.0
    return 2.0 * precision * recall / (precision + recall)


def string_similarity(first: str, second: str) -> float:
    first = normalize_spaces(first).casefold()
    second = normalize_spaces(second).casefold()
    denominator = max(len(first), len(second), 1)
    return 1.0 - levenshtein_distance(first, second) / denominator


# ============================================================
# GROUND-TRUTH DISCOVERY — ONLY FROZEN DEVELOPMENT IDS
# ============================================================

def locate_ground_truth(document_id: str, template_id: str) -> Path:
    candidates = []

    for candidate in GROUND_TRUTH_ROOT.rglob(f"*{document_id}*.json"):
        if candidate.is_file():
            candidates.append(candidate.resolve())

    candidates = sorted(set(candidates), key=lambda path: path.as_posix())

    if len(candidates) == 1:
        return candidates[0]

    identity_matches = []
    for candidate in candidates:
        candidate_payload = load_json(candidate)
        candidate_document = candidate_payload.get("document", {})
        if (
            isinstance(candidate_document, dict)
            and candidate_document.get("document_id") == document_id
            and candidate_document.get("template_id") == template_id
            and candidate_document.get("split") == "development"
        ):
            identity_matches.append(candidate)

    if len(identity_matches) != 1:
        raise RuntimeError(
            f"Ground truth {document_id}/{template_id} ditemukan "
            f"{len(identity_matches)} kali setelah verifikasi identitas; "
            f"kandidat berdasarkan document_id={len(candidates)}. "
            "Seharusnya tepat satu."
        )
    return identity_matches[0]


# ============================================================
# CANONICAL GROUND-TRUTH ADAPTER
# ============================================================

def canonical_from_ground_truth(
    ground_truth: dict,
    document_id: str,
    template_id: str,
) -> dict:
    document = ground_truth.get("document", {})
    canonical = ground_truth.get("canonical")

    if not isinstance(document, dict):
        raise TypeError("ground_truth.document harus berupa object.")
    if not isinstance(canonical, dict):
        raise TypeError("ground_truth.canonical harus berupa object.")
    if document.get("document_id") != document_id:
        raise RuntimeError("Document ID ground truth tidak cocok.")
    if document.get("template_id") != template_id:
        raise RuntimeError("Template ID ground truth tidak cocok.")
    if document.get("split") != "development":
        raise RuntimeError(
            "Evaluasi Cell 10D hanya boleh membuka development split."
        )
    if canonical.get("document_id") != document_id:
        raise RuntimeError("Document ID canonical tidak cocok.")
    if canonical.get("template_id") != template_id:
        raise RuntimeError("Template ID canonical tidak cocok.")
    if canonical.get("split") != "development":
        raise RuntimeError("Canonical payload bukan development split.")

    items = canonical.get("items")
    if not isinstance(items, list) or not items:
        raise RuntimeError("Canonical items kosong atau tidak valid.")
    if not all(isinstance(item, dict) for item in items):
        raise TypeError("Setiap canonical item harus berupa object.")

    return canonical


def expected_scalars(canonical: dict) -> dict[str, str]:
    return {
        field_name: normalize_field_value(
            field_name,
            nested_value(canonical, path),
        )
        for field_name, path in SCALAR_PATHS.items()
    }


def expected_items(canonical: dict) -> list[dict[str, str]]:
    normalized_items = []
    for item in canonical["items"]:
        normalized_items.append(
            {
                field_name: normalize_field_value(
                    f"items[].{field_name}",
                    item.get(field_name),
                )
                for field_name in ITEM_FIELD_NAMES
            }
        )
    return normalized_items


def predicted_scalars(prediction: dict) -> dict[str, str]:
    scalar_predictions = (
        prediction.get("predictions", {}).get("scalar_fields", {})
    )
    if not isinstance(scalar_predictions, dict):
        raise TypeError("predictions.scalar_fields harus berupa object.")

    values = {}
    for field_name in SCALAR_PATHS:
        field_prediction = scalar_predictions.get(field_name, {})
        if not isinstance(field_prediction, dict):
            field_prediction = {}
        values[field_name] = normalize_field_value(
            field_name,
            field_prediction.get("normalized_value"),
        )
    return values


def predicted_items(prediction: dict) -> list[dict[str, str]]:
    item_predictions = prediction.get("predictions", {}).get("items", [])
    if not isinstance(item_predictions, list):
        raise TypeError("predictions.items harus berupa list.")

    normalized_items = []
    for item in item_predictions:
        if not isinstance(item, dict):
            raise TypeError("Setiap prediction item harus berupa object.")

        normalized_item = {}
        for field_name in ITEM_FIELD_NAMES:
            field_prediction = item.get(field_name, {})
            if not isinstance(field_prediction, dict):
                field_prediction = {}
            normalized_item[field_name] = normalize_field_value(
                f"items[].{field_name}",
                field_prediction.get("normalized_value"),
            )
        normalized_items.append(normalized_item)
    return normalized_items


# ============================================================
# MAXIMUM-WEIGHT BIPARTITE ITEM ALIGNMENT
# ============================================================

def item_pair_weight(predicted: dict, expected: dict) -> int:
    description_similarity = string_similarity(
        predicted.get("description", ""),
        expected.get("description", ""),
    )
    line_total_exact = (
        predicted.get("line_total", "")
        == expected.get("line_total", "")
    )
    quantity_exact = (
        predicted.get("quantity", "")
        == expected.get("quantity", "")
    )
    unit_price_exact = (
        predicted.get("unit_price", "")
        == expected.get("unit_price", "")
    )

    return int(round(description_similarity * 10000)) + (
        10000 if line_total_exact else 0
    ) + (100 if quantity_exact else 0) + (100 if unit_price_exact else 0)


def maximum_weight_item_alignment(
    predictions: list[dict],
    references: list[dict],
) -> list[tuple[int | None, int | None]]:
    size = max(len(predictions), len(references))
    if size == 0:
        return []

    weights = []
    for prediction_index in range(size):
        row = []
        for reference_index in range(size):
            if (
                prediction_index < len(predictions)
                and reference_index < len(references)
            ):
                row.append(
                    item_pair_weight(
                        predictions[prediction_index],
                        references[reference_index],
                    )
                )
            else:
                row.append(0)
        weights.append(row)

    @lru_cache(maxsize=None)
    def solve(
        prediction_index: int,
        used_reference_mask: int,
    ) -> tuple[int, tuple[int, ...]]:
        if prediction_index == size:
            return 0, ()

        best_score = -1
        best_assignment = ()
        for reference_index in range(size):
            bit = 1 << reference_index
            if used_reference_mask & bit:
                continue

            remaining_score, remaining_assignment = solve(
                prediction_index + 1,
                used_reference_mask | bit,
            )
            score = weights[prediction_index][reference_index] + remaining_score
            assignment = (reference_index,) + remaining_assignment

            if score > best_score or (
                score == best_score and assignment < best_assignment
            ):
                best_score = score
                best_assignment = assignment

        return best_score, best_assignment

    _, assignment = solve(0, 0)
    aligned_pairs = []
    for prediction_index, reference_index in enumerate(assignment):
        real_prediction = (
            prediction_index if prediction_index < len(predictions) else None
        )
        real_reference = (
            reference_index if reference_index < len(references) else None
        )
        if real_prediction is not None or real_reference is not None:
            aligned_pairs.append((real_prediction, real_reference))
    return aligned_pairs


# ============================================================
# DOCUMENT EVALUATION
# ============================================================

def evaluate_document(
    prediction: dict,
    canonical: dict,
    contract_field_map: dict[str, dict],
) -> dict:
    document = prediction.get("document", {})
    document_id = str(document["document_id"])
    template_id = str(document["template_id"])
    language = str(document.get("language", ""))

    scalar_expected = expected_scalars(canonical)
    scalar_predicted = predicted_scalars(prediction)
    scalar_records = []

    for field_name in SCALAR_PATHS:
        expected_value = scalar_expected[field_name]
        predicted_value = scalar_predicted[field_name]
        exact_match = predicted_value == expected_value
        character_errors = levenshtein_distance(
            expected_value,
            predicted_value,
        )
        word_errors = levenshtein_distance(
            expected_value.split(),
            predicted_value.split(),
        )

        scalar_records.append(
            {
                "document_id": document_id,
                "template_id": template_id,
                "language": language,
                "group": "scalar",
                "field": field_name,
                "critical": bool(
                    contract_field_map[field_name].get("critical")
                ),
                "expected": expected_value,
                "predicted": predicted_value,
                "exact_match": exact_match,
                "character_errors": character_errors,
                "reference_characters": len(expected_value),
                "word_errors": word_errors,
                "reference_words": len(expected_value.split()),
            }
        )

    item_expected = expected_items(canonical)
    item_predicted = predicted_items(prediction)
    alignment = maximum_weight_item_alignment(
        item_predicted,
        item_expected,
    )

    item_field_records = []
    item_pair_records = []

    for pair_number, (prediction_index, reference_index) in enumerate(
        alignment,
        start=1,
    ):
        predicted_item = (
            item_predicted[prediction_index]
            if prediction_index is not None
            else None
        )
        expected_item = (
            item_expected[reference_index]
            if reference_index is not None
            else None
        )

        description_exact = bool(
            predicted_item is not None
            and expected_item is not None
            and predicted_item["description"] == expected_item["description"]
        )
        line_total_exact = bool(
            predicted_item is not None
            and expected_item is not None
            and predicted_item["line_total"] == expected_item["line_total"]
        )
        accepted_row_match = description_exact and line_total_exact

        item_pair_records.append(
            {
                "pair_number": pair_number,
                "prediction_row": (
                    prediction_index + 1
                    if prediction_index is not None
                    else None
                ),
                "reference_row": (
                    reference_index + 1
                    if reference_index is not None
                    else None
                ),
                "description_exact": description_exact,
                "line_total_exact": line_total_exact,
                "accepted_row_match": accepted_row_match,
            }
        )

        for field_name in ITEM_FIELD_NAMES:
            full_field_name = f"items[].{field_name}"
            expected_value = (
                expected_item[field_name]
                if expected_item is not None
                else ""
            )
            predicted_value = (
                predicted_item[field_name]
                if predicted_item is not None
                else ""
            )
            exact_match = bool(
                predicted_item is not None
                and expected_item is not None
                and predicted_value == expected_value
            )

            item_field_records.append(
                {
                    "document_id": document_id,
                    "template_id": template_id,
                    "language": language,
                    "group": "item",
                    "field": full_field_name,
                    "critical": bool(
                        contract_field_map[full_field_name].get("critical")
                    ),
                    "prediction_row": (
                        prediction_index + 1
                        if prediction_index is not None
                        else None
                    ),
                    "reference_row": (
                        reference_index + 1
                        if reference_index is not None
                        else None
                    ),
                    "expected": expected_value,
                    "predicted": predicted_value,
                    "exact_match": exact_match,
                    "character_errors": levenshtein_distance(
                        expected_value,
                        predicted_value,
                    ),
                    "reference_characters": len(expected_value),
                    "word_errors": levenshtein_distance(
                        expected_value.split(),
                        predicted_value.split(),
                    ),
                    "reference_words": len(expected_value.split()),
                }
            )

    row_true_positives = sum(
        record["accepted_row_match"] for record in item_pair_records
    )
    row_precision = safe_ratio(row_true_positives, len(item_predicted))
    row_recall = safe_ratio(row_true_positives, len(item_expected))
    row_f1 = harmonic_mean(row_precision, row_recall)

    item_exact_fields = sum(
        record["exact_match"] for record in item_field_records
    )
    predicted_item_fields = len(item_predicted) * EXPECTED_ITEM_FIELDS
    reference_item_fields = len(item_expected) * EXPECTED_ITEM_FIELDS
    item_field_precision = safe_ratio(
        item_exact_fields,
        predicted_item_fields,
    )
    item_field_recall = safe_ratio(
        item_exact_fields,
        reference_item_fields,
    )
    item_field_f1 = harmonic_mean(
        item_field_precision,
        item_field_recall,
    )

    scalar_exact_count = sum(
        record["exact_match"] for record in scalar_records
    )
    document_exact = bool(
        scalar_exact_count == EXPECTED_SCALAR_FIELDS
        and len(item_predicted) == len(item_expected)
        and item_exact_fields == reference_item_fields
    )

    financial_check = prediction.get("diagnostics", {}).get(
        "financial_equation",
        {},
    )
    financial_consistent = bool(
        financial_check.get("executed") is True
        and financial_check.get("passed") is True
    )

    return {
        "document_id": document_id,
        "template_id": template_id,
        "language": language,
        "scalar_records": scalar_records,
        "item_field_records": item_field_records,
        "item_alignment": item_pair_records,
        "metrics": {
            "scalar_exact_count": scalar_exact_count,
            "scalar_field_count": EXPECTED_SCALAR_FIELDS,
            "scalar_exact_match": safe_ratio(
                scalar_exact_count,
                EXPECTED_SCALAR_FIELDS,
            ),
            "predicted_item_count": len(item_predicted),
            "reference_item_count": len(item_expected),
            "item_count_exact": len(item_predicted) == len(item_expected),
            "row_true_positives": row_true_positives,
            "row_precision": row_precision,
            "row_recall": row_recall,
            "row_f1": row_f1,
            "item_exact_fields": item_exact_fields,
            "predicted_item_fields": predicted_item_fields,
            "reference_item_fields": reference_item_fields,
            "item_field_precision": item_field_precision,
            "item_field_recall": item_field_recall,
            "item_field_f1": item_field_f1,
            "financial_consistent": financial_consistent,
            "document_exact_match": document_exact,
        },
    }


# ============================================================
# PREFLIGHT AND FROZEN-PREDICTION GATE
# ============================================================

require_file(CONTRACT_PATH, "Field extraction contract")
require_file(PREDICTION_MANIFEST_PATH, "Prediction manifest v1.0.1")
require_directory(GROUND_TRUTH_ROOT, "Ground-truth root")

contract = load_json(CONTRACT_PATH)
prediction_manifest = load_json(PREDICTION_MANIFEST_PATH)

field_definitions = contract.get("field_definitions", [])
contract_field_map = {
    field["field"]: field
    for field in field_definitions
    if isinstance(field, dict) and field.get("field")
}
critical_scalar_fields = {
    field_name
    for field_name in SCALAR_PATHS
    if contract_field_map.get(field_name, {}).get("critical") is True
}

prediction_parser = prediction_manifest.get("parser", {})
prediction_records = prediction_manifest.get("records", [])

preflight_values = [
    (
        "contract_status",
        "FROZEN_FOR_DEVELOPMENT_BASELINE",
        contract.get("status"),
    ),
    (
        "prediction_manifest_status",
        "EXECUTED",
        prediction_manifest.get("status"),
    ),
    (
        "prediction_quality_before_evaluation",
        "PENDING_GROUND_TRUTH_EVALUATION",
        prediction_manifest.get("quality_status"),
    ),
    (
        "parser_id",
        EXPECTED_PARSER_ID,
        prediction_parser.get("parser_id"),
    ),
    (
        "parser_version",
        EXPECTED_PARSER_VERSION,
        prediction_parser.get("parser_version"),
    ),
    ("prediction_records", EXPECTED_DOCUMENTS, len(prediction_records)),
    ("target_fields", 16, len(contract_field_map)),
    ("scalar_fields", EXPECTED_SCALAR_FIELDS, len(SCALAR_PATHS)),
    ("item_fields", EXPECTED_ITEM_FIELDS, len(ITEM_FIELD_NAMES)),
    (
        "prediction_document_ids_unique",
        EXPECTED_DOCUMENTS,
        len({record.get("document_id") for record in prediction_records}),
    ),
]

preflight_records = [
    {
        "control": name,
        "expected": expected,
        "actual": actual,
        "status": "VALID" if expected == actual else "INVALID",
    }
    for name, expected, actual in preflight_values
]

invalid_preflight = [
    record["control"]
    for record in preflight_records
    if record["status"] != "VALID"
]
if invalid_preflight:
    display(pd.DataFrame(preflight_records))
    raise RuntimeError(
        "CELL 10D PREFLIGHT FAILED. "
        f"Kontrol tidak valid: {invalid_preflight}"
    )

expected_document_ids = {
    record["document_id"]
    for record in contract.get("benchmark_records", [])
}
prediction_document_ids = {
    record["document_id"] for record in prediction_records
}
if expected_document_ids != prediction_document_ids:
    raise RuntimeError(
        "Document ID prediction tidak sama dengan benchmark yang dibekukan."
    )

input_checksums_before = {
    "contract": sha256_file(CONTRACT_PATH),
    "prediction_manifest": sha256_file(PREDICTION_MANIFEST_PATH),
}

for record in prediction_records:
    prediction_path = Path(record["prediction_path"])
    require_file(prediction_path, "Prediction checkpoint")
    checksum = sha256_file(prediction_path)
    if checksum != record["prediction_sha256"]:
        raise RuntimeError(
            f"Checksum prediction tidak cocok: {record['document_id']}"
        )
    input_checksums_before[
        f"prediction:{record['document_id']}"
    ] = checksum


# ============================================================
# OPEN ONLY 18 DEVELOPMENT GROUND-TRUTH FILES AND EVALUATE
# ============================================================

runtime_records = []
document_manifest_records = []
all_scalar_records = []
all_item_field_records = []
errors = []
ground_truth_paths = {}
ground_truth_checksums = {}
new_evaluations = 0
recovered_evaluations = 0

print("=" * 88)
print(
    f"CELL 10D — {EVALUATOR_ID} — VERSION {EVALUATOR_VERSION}"
)
print(f"Evaluation root: {EVALUATION_ROOT}")
print("Scope: 18 frozen DEVELOPMENT documents only")
print("Validation opened: 0 | Test opened: 0")
print("=" * 88)
print(f"Mengevaluasi {len(prediction_records)} dokumen...\n")

for sequence_number, prediction_record in enumerate(
    sorted(
        prediction_records,
        key=lambda row: (
            int(row.get("sequence_number", 0)),
            str(row.get("document_id", "")),
        ),
    ),
    start=1,
):
    document_id = str(prediction_record["document_id"])
    template_id = str(prediction_record["template_id"])
    prediction_path = Path(prediction_record["prediction_path"])
    evaluation_path = (
        DOCUMENT_EVALUATION_ROOT
        / template_id
        / f"{document_id}_evaluation.json"
    )

    try:
        prediction = load_json(prediction_path)
        prediction_document = prediction.get("document", {})
        if prediction_document.get("document_id") != document_id:
            raise RuntimeError("Prediction document ID tidak cocok.")
        if prediction_document.get("template_id") != template_id:
            raise RuntimeError("Prediction template ID tidak cocok.")
        if prediction_document.get("split") != "development":
            raise RuntimeError("Prediction bukan development split.")

        ground_truth_path = locate_ground_truth(document_id, template_id)
        ground_truth = load_json(ground_truth_path)
        canonical = canonical_from_ground_truth(
            ground_truth,
            document_id,
            template_id,
        )
        ground_truth_checksum = sha256_file(ground_truth_path)
        ground_truth_paths[document_id] = ground_truth_path
        ground_truth_checksums[document_id] = ground_truth_checksum

        result = evaluate_document(
            prediction,
            canonical,
            contract_field_map,
        )
        metrics = result["metrics"]

        evaluation_artifact = {
            "schema_version": "1.0.0",
            "status": "EVALUATED",
            "evaluator": {
                "evaluator_id": EVALUATOR_ID,
                "evaluator_version": EVALUATOR_VERSION,
                "matching_policy": contract["evaluation_policy"],
            },
            "document": {
                "document_id": document_id,
                "template_id": template_id,
                "split": "development",
                "language": result["language"],
            },
            "metrics": metrics,
            "scalar_comparisons": result["scalar_records"],
            "item_alignment": result["item_alignment"],
            "item_field_comparisons": result["item_field_records"],
            "inputs": {
                "prediction_path": str(prediction_path),
                "prediction_sha256": prediction_record[
                    "prediction_sha256"
                ],
                "ground_truth_path": str(ground_truth_path),
                "ground_truth_sha256": ground_truth_checksum,
            },
            "integrity": {
                "ground_truth_split": "development",
                "validation_opened": 0,
                "test_opened": 0,
                "prediction_modified": False,
                "ground_truth_modified": False,
            },
        }

        if evaluation_path.exists():
            existing = load_json(evaluation_path)
            if canonical_json(existing) != canonical_json(evaluation_artifact):
                raise RuntimeError(
                    "Evaluation checkpoint sudah ada tetapi berbeda."
                )
            execution = "RECOVERED"
            recovered_evaluations += 1
        else:
            atomic_write_json(evaluation_path, evaluation_artifact)
            execution = "NEW"
            new_evaluations += 1

        evaluation_sha256 = sha256_file(evaluation_path)
        record = {
            "sequence_number": sequence_number,
            "document_id": document_id,
            "template_id": template_id,
            "language": result["language"],
            "scalar_exact_match": metrics["scalar_exact_match"],
            "predicted_item_count": metrics["predicted_item_count"],
            "reference_item_count": metrics["reference_item_count"],
            "item_count_exact": metrics["item_count_exact"],
            "row_precision": metrics["row_precision"],
            "row_recall": metrics["row_recall"],
            "row_f1": metrics["row_f1"],
            "item_field_precision": metrics["item_field_precision"],
            "item_field_recall": metrics["item_field_recall"],
            "item_field_f1": metrics["item_field_f1"],
            "financial_consistent": metrics["financial_consistent"],
            "document_exact_match": metrics["document_exact_match"],
            "evaluation_path": str(evaluation_path),
            "evaluation_sha256": evaluation_sha256,
            "prediction_path": str(prediction_path),
            "prediction_sha256": prediction_record[
                "prediction_sha256"
            ],
            "ground_truth_path": str(ground_truth_path),
            "ground_truth_sha256": ground_truth_checksum,
            "status": "EVALUATED",
        }
        document_manifest_records.append(record)
        runtime_records.append({**record, "execution": execution})
        all_scalar_records.extend(result["scalar_records"])
        all_item_field_records.extend(result["item_field_records"])

        print(
            f"[{sequence_number:02d}/{len(prediction_records):02d}] "
            f"{document_id} | scalar={metrics['scalar_exact_match']:.4f} | "
            f"items={metrics['predicted_item_count']}/"
            f"{metrics['reference_item_count']} | "
            f"item_f1={metrics['item_field_f1']:.4f} | "
            f"exact={metrics['document_exact_match']} | {execution}"
        )

    except Exception as error:
        errors.append(
            {
                "sequence_number": sequence_number,
                "document_id": document_id,
                "template_id": template_id,
                "error_type": type(error).__name__,
                "error": str(error)[:700],
            }
        )
        print(
            f"[{sequence_number:02d}/{len(prediction_records):02d}] "
            f"{document_id} | ERROR: {type(error).__name__}: {error}"
        )


# ============================================================
# TECHNICAL COMPLETENESS GATE
# ============================================================

if errors:
    display(pd.DataFrame(errors))

technical_values = [
    ("evaluated_documents", EXPECTED_DOCUMENTS, len(document_manifest_records)),
    ("evaluation_files", EXPECTED_DOCUMENTS, sum(
        Path(record["evaluation_path"]).is_file()
        for record in document_manifest_records
    )),
    ("unique_document_ids", EXPECTED_DOCUMENTS, len({
        record["document_id"] for record in document_manifest_records
    })),
    ("templates_evaluated", EXPECTED_TEMPLATES, len({
        record["template_id"] for record in document_manifest_records
    })),
    ("development_ground_truth_opened", EXPECTED_DOCUMENTS, len(
        ground_truth_paths
    )),
    ("validation_opened", 0, 0),
    ("test_opened", 0, 0),
    ("evaluation_errors", 0, len(errors)),
]

technical_control_records = [
    {
        "control": name,
        "expected": expected,
        "actual": actual,
        "status": "VALID" if expected == actual else "INVALID",
    }
    for name, expected, actual in technical_values
]

display(pd.DataFrame(technical_control_records))

invalid_technical_controls = [
    record["control"]
    for record in technical_control_records
    if record["status"] != "VALID"
]
if invalid_technical_controls:
    raise RuntimeError(
        "CELL 10D TECHNICAL EVALUATION FAILED. "
        f"Kontrol tidak valid: {invalid_technical_controls}"
    )


# ============================================================
# AGGREGATE METRICS
# ============================================================

scalar_total = len(all_scalar_records)
scalar_exact = sum(record["exact_match"] for record in all_scalar_records)
critical_scalar_records = [
    record for record in all_scalar_records if record["critical"]
]
critical_scalar_exact = sum(
    record["exact_match"] for record in critical_scalar_records
)

all_scalar_exact_match = safe_ratio(scalar_exact, scalar_total)
critical_scalar_exact_match = safe_ratio(
    critical_scalar_exact,
    len(critical_scalar_records),
)

scalar_character_errors = sum(
    record["character_errors"] for record in all_scalar_records
)
scalar_reference_characters = sum(
    record["reference_characters"] for record in all_scalar_records
)
scalar_word_errors = sum(
    record["word_errors"] for record in all_scalar_records
)
scalar_reference_words = sum(
    record["reference_words"] for record in all_scalar_records
)
scalar_micro_cer = safe_error_rate(
    scalar_character_errors,
    scalar_reference_characters,
)
scalar_micro_wer = safe_error_rate(
    scalar_word_errors,
    scalar_reference_words,
)

item_exact_fields = sum(
    record["exact_match"] for record in all_item_field_records
)
predicted_item_field_count = sum(
    record["predicted_item_count"] * EXPECTED_ITEM_FIELDS
    for record in document_manifest_records
)
reference_item_field_count = sum(
    record["reference_item_count"] * EXPECTED_ITEM_FIELDS
    for record in document_manifest_records
)
line_item_field_precision = safe_ratio(
    item_exact_fields,
    predicted_item_field_count,
)
line_item_field_recall = safe_ratio(
    item_exact_fields,
    reference_item_field_count,
)
line_item_field_f1 = harmonic_mean(
    line_item_field_precision,
    line_item_field_recall,
)

row_true_positives = 0
predicted_rows = 0
reference_rows = 0
for record in document_manifest_records:
    evaluation = load_json(Path(record["evaluation_path"]))
    metrics = evaluation["metrics"]
    row_true_positives += int(metrics["row_true_positives"])
    predicted_rows += int(metrics["predicted_item_count"])
    reference_rows += int(metrics["reference_item_count"])

row_precision = safe_ratio(row_true_positives, predicted_rows)
row_recall = safe_ratio(row_true_positives, reference_rows)
row_f1 = harmonic_mean(row_precision, row_recall)

financial_consistency = safe_ratio(
    sum(
        record["financial_consistent"]
        for record in document_manifest_records
    ),
    len(document_manifest_records),
)
document_exact_match = safe_ratio(
    sum(
        record["document_exact_match"]
        for record in document_manifest_records
    ),
    len(document_manifest_records),
)
item_count_exact_match = safe_ratio(
    sum(record["item_count_exact"] for record in document_manifest_records),
    len(document_manifest_records),
)

overall_metrics = {
    "documents": len(document_manifest_records),
    "templates": EXPECTED_TEMPLATES,
    "scalar_field_observations": scalar_total,
    "scalar_field_exact": scalar_exact,
    "all_scalar_exact_match": all_scalar_exact_match,
    "critical_scalar_observations": len(critical_scalar_records),
    "critical_scalar_exact": critical_scalar_exact,
    "critical_scalar_exact_match": critical_scalar_exact_match,
    "scalar_micro_cer": scalar_micro_cer,
    "scalar_micro_wer": scalar_micro_wer,
    "predicted_item_rows": predicted_rows,
    "reference_item_rows": reference_rows,
    "matched_item_rows": row_true_positives,
    "row_precision": row_precision,
    "row_recall": row_recall,
    "row_f1": row_f1,
    "item_field_exact": item_exact_fields,
    "predicted_item_fields": predicted_item_field_count,
    "reference_item_fields": reference_item_field_count,
    "line_item_field_precision": line_item_field_precision,
    "line_item_field_recall": line_item_field_recall,
    "line_item_field_f1": line_item_field_f1,
    "item_count_exact_match": item_count_exact_match,
    "financial_consistency": financial_consistency,
    "document_exact_match": document_exact_match,
}


# ============================================================
# FIELD SUMMARY
# ============================================================

combined_field_records = all_scalar_records + all_item_field_records
field_summary_records = []

for field_name in sorted({record["field"] for record in combined_field_records}):
    records = [
        record
        for record in combined_field_records
        if record["field"] == field_name
    ]
    exact_count = sum(record["exact_match"] for record in records)
    character_errors = sum(record["character_errors"] for record in records)
    reference_characters = sum(
        record["reference_characters"] for record in records
    )
    word_errors = sum(record["word_errors"] for record in records)
    reference_words = sum(record["reference_words"] for record in records)

    field_summary_records.append(
        {
            "field": field_name,
            "group": records[0]["group"],
            "critical": records[0]["critical"],
            "observations": len(records),
            "exact_matches": exact_count,
            "exact_match_rate": safe_ratio(exact_count, len(records)),
            "character_errors": character_errors,
            "reference_characters": reference_characters,
            "cer": safe_error_rate(
                character_errors,
                reference_characters,
            ),
            "word_errors": word_errors,
            "reference_words": reference_words,
            "wer": safe_error_rate(word_errors, reference_words),
            "status": (
                "PERFECT" if exact_count == len(records) else "HAS_ERRORS"
            ),
        }
    )


# ============================================================
# TEMPLATE SUMMARY
# ============================================================

template_summary_records = []

for template_id in sorted({
    record["template_id"] for record in document_manifest_records
}):
    documents = [
        record
        for record in document_manifest_records
        if record["template_id"] == template_id
    ]
    scalar_records = [
        record
        for record in all_scalar_records
        if record["template_id"] == template_id
    ]
    item_records = [
        record
        for record in all_item_field_records
        if record["template_id"] == template_id
    ]

    scalar_correct = sum(record["exact_match"] for record in scalar_records)
    item_correct = sum(record["exact_match"] for record in item_records)
    template_predicted_item_fields = sum(
        record["predicted_item_count"] * EXPECTED_ITEM_FIELDS
        for record in documents
    )
    template_reference_item_fields = sum(
        record["reference_item_count"] * EXPECTED_ITEM_FIELDS
        for record in documents
    )
    item_precision = safe_ratio(
        item_correct,
        template_predicted_item_fields,
    )
    item_recall = safe_ratio(
        item_correct,
        template_reference_item_fields,
    )

    template_summary_records.append(
        {
            "template_id": template_id,
            "documents": len(documents),
            "scalar_exact_match": safe_ratio(
                scalar_correct,
                len(scalar_records),
            ),
            "line_item_field_f1": harmonic_mean(
                item_precision,
                item_recall,
            ),
            "item_count_exact_match": safe_ratio(
                sum(record["item_count_exact"] for record in documents),
                len(documents),
            ),
            "financial_consistency": safe_ratio(
                sum(record["financial_consistent"] for record in documents),
                len(documents),
            ),
            "document_exact_match": safe_ratio(
                sum(record["document_exact_match"] for record in documents),
                len(documents),
            ),
        }
    )


# ============================================================
# ACCEPTANCE POLICY
# ============================================================

acceptance_targets = contract["evaluation_policy"][
    "development_acceptance_targets"
]

acceptance_checks = [
    {
        "metric": "critical_scalar_exact_match",
        "minimum": float(
            acceptance_targets["critical_scalar_exact_match_minimum"]
        ),
        "actual": critical_scalar_exact_match,
    },
    {
        "metric": "all_scalar_exact_match",
        "minimum": float(
            acceptance_targets["all_scalar_exact_match_minimum"]
        ),
        "actual": all_scalar_exact_match,
    },
    {
        "metric": "line_item_field_f1",
        "minimum": float(
            acceptance_targets["line_item_field_f1_minimum"]
        ),
        "actual": line_item_field_f1,
    },
    {
        "metric": "financial_consistency",
        "minimum": float(
            acceptance_targets["financial_consistency_minimum"]
        ),
        "actual": financial_consistency,
    },
]

for check in acceptance_checks:
    check["status"] = (
        "PASSED" if check["actual"] >= check["minimum"] else "FAILED"
    )

acceptance_status = (
    "PASSED"
    if all(check["status"] == "PASSED" for check in acceptance_checks)
    else "FAILED"
)


# ============================================================
# MISMATCH DETAILS
# ============================================================

mismatch_records = []
for record in combined_field_records:
    if record["exact_match"]:
        continue
    mismatch_records.append(
        {
            "document_id": record["document_id"],
            "template_id": record["template_id"],
            "language": record["language"],
            "group": record["group"],
            "field": record["field"],
            "prediction_row": record.get("prediction_row"),
            "reference_row": record.get("reference_row"),
            "expected": record["expected"],
            "predicted": record["predicted"],
            "character_errors": record["character_errors"],
            "word_errors": record["word_errors"],
        }
    )


# ============================================================
# DISPLAY RESULTS
# ============================================================

document_table = pd.DataFrame(runtime_records)
field_table = pd.DataFrame(field_summary_records)
template_table = pd.DataFrame(template_summary_records)
acceptance_table = pd.DataFrame(acceptance_checks)

display(acceptance_table)
display(field_table)
display(template_table)
display(
    document_table[
        [
            "sequence_number",
            "document_id",
            "template_id",
            "language",
            "scalar_exact_match",
            "predicted_item_count",
            "reference_item_count",
            "row_f1",
            "item_field_f1",
            "financial_consistent",
            "document_exact_match",
            "execution",
        ]
    ]
)

if mismatch_records:
    print("\nMISMATCH DETAILS")
    display(pd.DataFrame(mismatch_records))


# ============================================================
# SAVE SUMMARIES AND EVALUATION MANIFEST
# ============================================================

def save_csv_checkpoint(
    path: Path,
    table: pd.DataFrame,
) -> str:
    text = table.to_csv(index=False, lineterminator="\n")
    if path.exists():
        if path.read_text(encoding="utf-8") != text:
            raise RuntimeError(
                f"CSV checkpoint sudah ada tetapi berbeda: {path}"
            )
        return "RECOVERED"
    atomic_write_text(path, text)
    return "CREATED"


document_summary_action = save_csv_checkpoint(
    DOCUMENT_SUMMARY_PATH,
    pd.DataFrame(document_manifest_records),
)
field_summary_action = save_csv_checkpoint(
    FIELD_SUMMARY_PATH,
    field_table,
)
template_summary_action = save_csv_checkpoint(
    TEMPLATE_SUMMARY_PATH,
    template_table,
)

mismatch_columns = [
    "document_id",
    "template_id",
    "language",
    "group",
    "field",
    "prediction_row",
    "reference_row",
    "expected",
    "predicted",
    "character_errors",
    "word_errors",
]
mismatch_table = pd.DataFrame(mismatch_records, columns=mismatch_columns)
mismatch_action = save_csv_checkpoint(
    MISMATCH_DETAIL_PATH,
    mismatch_table,
)

evaluation_manifest = {
    "schema_version": "1.0.0",
    "status": acceptance_status,
    "stage": "DEVELOPMENT_GROUND_TRUTH_EVALUATION",
    "evaluator": {
        "evaluator_id": EVALUATOR_ID,
        "evaluator_version": EVALUATOR_VERSION,
        "scalar_matching": "TYPE_AWARE_NORMALIZED_EXACT_MATCH",
        "item_alignment": "MAXIMUM_WEIGHT_BIPARTITE_ROW_MATCHING",
    },
    "scope": {
        "split": "development",
        "documents": EXPECTED_DOCUMENTS,
        "templates": EXPECTED_TEMPLATES,
        "development_ground_truth_opened": len(ground_truth_paths),
        "validation_opened": 0,
        "test_opened": 0,
    },
    "overall_metrics": overall_metrics,
    "acceptance_checks": acceptance_checks,
    "technical_controls": technical_control_records,
    "field_summary": field_summary_records,
    "template_summary": template_summary_records,
    "records": document_manifest_records,
    "artifacts": {
        "evaluation_root": str(EVALUATION_ROOT),
        "document_summary": {
            "path": str(DOCUMENT_SUMMARY_PATH),
            "sha256": sha256_file(DOCUMENT_SUMMARY_PATH),
        },
        "field_summary": {
            "path": str(FIELD_SUMMARY_PATH),
            "sha256": sha256_file(FIELD_SUMMARY_PATH),
        },
        "template_summary": {
            "path": str(TEMPLATE_SUMMARY_PATH),
            "sha256": sha256_file(TEMPLATE_SUMMARY_PATH),
        },
        "mismatch_details": {
            "path": str(MISMATCH_DETAIL_PATH),
            "sha256": sha256_file(MISMATCH_DETAIL_PATH),
            "records": len(mismatch_records),
        },
    },
    "inputs": {
        "contract": {
            "path": str(CONTRACT_PATH),
            "sha256": input_checksums_before["contract"],
        },
        "prediction_manifest": {
            "path": str(PREDICTION_MANIFEST_PATH),
            "sha256": input_checksums_before["prediction_manifest"],
        },
        "prediction_files": {
            document_id: checksum
            for document_id, checksum in sorted(
                (
                    key.split(":", 1)[1],
                    checksum,
                )
                for key, checksum in input_checksums_before.items()
                if key.startswith("prediction:")
            )
        },
        "development_ground_truth_files": {
            document_id: {
                "path": str(ground_truth_paths[document_id]),
                "sha256": ground_truth_checksums[document_id],
            }
            for document_id in sorted(ground_truth_paths)
        },
    },
    "integrity": {
        "prediction_frozen_before_ground_truth_open": True,
        "ground_truth_used_for_evaluation_only": True,
        "ground_truth_used_as_prediction_input": False,
        "validation_opened": 0,
        "test_opened": 0,
        "prediction_modifications": 0,
        "ground_truth_modifications": 0,
        "dataset_modifications": 0,
    },
    "next_stage": {
        "action_if_passed": "FREEZE_BASELINE_AND_PREPARE_VALIDATION_GATE",
        "action_if_failed": "ANALYZE_DEVELOPMENT_MISMATCHES_ONLY",
        "validation_remains_locked": True,
        "test_remains_locked": True,
    },
}

if EVALUATION_MANIFEST_PATH.exists():
    existing_manifest = load_json(EVALUATION_MANIFEST_PATH)
    if canonical_json(existing_manifest) != canonical_json(evaluation_manifest):
        raise RuntimeError(
            "Evaluation manifest sudah ada tetapi berbeda."
        )
    manifest_action = "RECOVERED"
else:
    atomic_write_json(EVALUATION_MANIFEST_PATH, evaluation_manifest)
    manifest_action = "CREATED"


# ============================================================
# INPUT IMMUTABILITY CHECK
# ============================================================

input_checksums_after = {
    "contract": sha256_file(CONTRACT_PATH),
    "prediction_manifest": sha256_file(PREDICTION_MANIFEST_PATH),
}
for record in prediction_records:
    input_checksums_after[
        f"prediction:{record['document_id']}"
    ] = sha256_file(Path(record["prediction_path"]))

changed_inputs = [
    name
    for name, checksum in input_checksums_before.items()
    if input_checksums_after.get(name) != checksum
]
changed_ground_truth = [
    document_id
    for document_id, checksum in ground_truth_checksums.items()
    if sha256_file(ground_truth_paths[document_id]) != checksum
]

if changed_inputs or changed_ground_truth:
    raise RuntimeError(
        "Input evaluation berubah selama eksekusi: "
        f"inputs={changed_inputs}, ground_truth={changed_ground_truth}"
    )


print()
print(f"Evaluation status        : {acceptance_status}")
print(f"Documents evaluated      : {len(document_manifest_records)}")
print(f"Templates                : {EXPECTED_TEMPLATES}")
print(f"All scalar exact match   : {all_scalar_exact_match:.6f}")
print(f"Critical scalar exact    : {critical_scalar_exact_match:.6f}")
print(f"Scalar micro CER         : {scalar_micro_cer:.6f}")
print(f"Scalar micro WER         : {scalar_micro_wer:.6f}")
print(f"Row precision            : {row_precision:.6f}")
print(f"Row recall               : {row_recall:.6f}")
print(f"Row F1                   : {row_f1:.6f}")
print(f"Line-item field F1       : {line_item_field_f1:.6f}")
print(f"Item-count exact match   : {item_count_exact_match:.6f}")
print(f"Financial consistency    : {financial_consistency:.6f}")
print(f"Document exact match     : {document_exact_match:.6f}")
print(f"Mismatch records         : {len(mismatch_records)}")
print(f"New evaluations          : {new_evaluations}")
print(f"Recovered evaluations    : {recovered_evaluations}")
print(f"Document summary action  : {document_summary_action}")
print(f"Field summary action     : {field_summary_action}")
print(f"Template summary action  : {template_summary_action}")
print(f"Mismatch file action     : {mismatch_action}")
print(f"Manifest action          : {manifest_action}")
print(f"Evaluation root          : {EVALUATION_ROOT}")
print(f"Evaluation manifest      : {EVALUATION_MANIFEST_PATH}")
print(
    f"Manifest SHA-256         : "
    f"{sha256_file(EVALUATION_MANIFEST_PATH)}"
)
print(f"Development GT opened    : {len(ground_truth_paths)}")
print("Validation opened        : 0")
print("Test opened              : 0")
print("Prediction modifications : 0")
print("Ground-truth modifications: 0")
print("Dataset modifications    : 0")

if acceptance_status != "PASSED":
    failed_metrics = [
        check["metric"]
        for check in acceptance_checks
        if check["status"] != "PASSED"
    ]
    raise RuntimeError(
        "DEVELOPMENT BASELINE BELOW ACCEPTANCE TARGET. "
        f"Metrik gagal: {failed_metrics}. "
        "Validation dan test tetap terkunci."
    )

print()
print(
    "✅ CELL 10D PASSED — parser baseline memenuhi seluruh target "
    "development yang dibekukan. Ground truth hanya dibuka untuk "
    "18 dokumen development; validation dan test tetap terkunci."
)


CELL 10D — INVOICE-FIELD-EVALUATOR-V1 — VERSION 1.0.1
Evaluation root: /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/field_extraction/evaluations/rule_based_baseline_v1_0_1_development_eval_v1_0_1
Scope: 18 frozen DEVELOPMENT documents only
Validation opened: 0 | Test opened: 0
Mengevaluasi 18 dokumen...

[01/18] INV-SYN-000002 | scalar=1.0000 | items=2/2 | item_f1=1.0000 | exact=True | NEW
[02/18] INV-SYN-000013 | scalar=1.0000 | items=8/8 | item_f1=1.0000 | exact=True | NEW
[03/18] INV-SYN-000015 | scalar=1.0000 | items=8/8 | item_f1=1.0000 | exact=True | NEW
[04/18] INV-SYN-000025 | scalar=1.0000 | items=2/2 | item_f1=1.0000 | exact=True | NEW
[05/18] INV-SYN-000036 | scalar=1.0000 | items=7/7 | item_f1=1.0000 | exact=True | NEW
[06/18] INV-SYN-000037 | scalar=1.0000 | items=7/7 | item_f1=1.0000 | exact=True | NEW
[07/18] INV-SYN-000043 | scalar=1.0000 | items=2/2 | item_f1=1.0000 | exact=True | NEW
[08/18] INV-SYN-000052 | scal

,control,expected,actual,status
0,evaluated_documents,18,18,VALID
1,evaluation_files,18,18,VALID
2,unique_document_ids,18,18,VALID
3,templates_evaluated,6,6,VALID
4,development_ground_truth_opened,18,18,VALID
5,validation_opened,0,0,VALID
6,test_opened,0,0,VALID
7,evaluation_errors,0,0,VALID


,metric,minimum,actual,status
0,critical_scalar_exact_match,0.98,0.994444,PASSED
1,all_scalar_exact_match,0.95,0.995370,PASSED
2,line_item_field_f1,0.90,1.000000,PASSED
3,financial_consistency,0.99,1.000000,PASSED


,field,group,critical,observations,exact_matches,exact_match_rate,character_errors,reference_characters,cer,word_errors,reference_words,wer,status
0,buyer.name,scalar,True,18,18,1.000000,0,534,0.000000,0,72,0.000000,PERFECT
1,buyer.tax_identifier,scalar,False,18,18,1.000000,0,306,0.000000,0,18,0.000000,PERFECT
2,currency,scalar,True,18,18,1.000000,0,54,0.000000,0,18,0.000000,PERFECT
3,due_date,scalar,True,18,18,1.000000,0,180,0.000000,0,18,0.000000,PERFECT
4,financials.discount,scalar,True,18,18,1.000000,0,77,0.000000,0,18,0.000000,PERFECT
5,financials.subtotal,scalar,True,18,18,1.000000,0,144,0.000000,0,18,0.000000,PERFECT
6,financials.tax,scalar,True,18,18,1.000000,0,112,0.000000,0,18,0.000000,PERFECT
7,financials.total,scalar,True,18,18,1.000000,0,149,0.000000,0,18,0.000000,PERFECT
8,invoice_date,scalar,True,18,18,1.000000,0,180,0.000000,0,18,0.000000,PERFECT
9,invoice_number,scalar,True,18,18,1.000000,0,324,0.000000,0,18,0.000000,PERFECT


,template_id,documents,scalar_exact_match,line_item_field_f1,item_count_exact_match,financial_consistency,document_exact_match
0,TPL-01,3,1.000000,1.0,1.0,1.0,1.000000
1,TPL-02,3,1.000000,1.0,1.0,1.0,1.000000
2,TPL-03,3,1.000000,1.0,1.0,1.0,1.000000
3,TPL-04,3,0.972222,1.0,1.0,1.0,0.666667
4,TPL-05,3,1.000000,1.0,1.0,1.0,1.000000
5,TPL-06,3,1.000000,1.0,1.0,1.0,1.000000


,sequence_number,document_id,template_id,language,scalar_exact_match,predicted_item_count,reference_item_count,row_f1,item_field_f1,financial_consistent,document_exact_match,execution
0,1,INV-SYN-000002,TPL-01,id,1.000000,2,2,1.0,1.0,True,True,NEW
1,2,INV-SYN-000013,TPL-01,en,1.000000,8,8,1.0,1.0,True,True,NEW
2,3,INV-SYN-000015,TPL-01,en,1.000000,8,8,1.0,1.0,True,True,NEW
3,4,INV-SYN-000025,TPL-02,id,1.000000,2,2,1.0,1.0,True,True,NEW
4,5,INV-SYN-000036,TPL-02,en,1.000000,7,7,1.0,1.0,True,True,NEW
5,6,INV-SYN-000037,TPL-02,en,1.000000,7,7,1.0,1.0,True,True,NEW
6,7,INV-SYN-000043,TPL-03,id,1.000000,2,2,1.0,1.0,True,True,NEW
7,8,INV-SYN-000052,TPL-03,en,1.000000,7,7,1.0,1.0,True,True,NEW
8,9,INV-SYN-000060,TPL-03,en,1.000000,7,7,1.0,1.0,True,True,NEW
9,10,INV-SYN-000064,TPL-04,id,1.000000,8,8,1.0,1.0,True,True,NEW



MISMATCH DETAILS


,document_id,template_id,language,group,field,prediction_row,reference_row,expected,predicted,character_errors,word_errors
0,INV-SYN-000071,TPL-04,en,scalar,vendor.name,None,None,Brightmere Example Analytics Inc.,Brightmere Example Analytics,5,1



Evaluation status        : PASSED
Documents evaluated      : 18
Templates                : 6
All scalar exact match   : 0.995370
Critical scalar exact    : 0.994444
Scalar micro CER         : 0.001726
Scalar micro WER         : 0.003086
Row precision            : 1.000000
Row recall               : 1.000000
Row F1                   : 1.000000
Line-item field F1       : 1.000000
Item-count exact match   : 1.000000
Financial consistency    : 1.000000
Document exact match     : 0.944444
Mismatch records         : 1
New evaluations          : 18
Recovered evaluations    : 0
Document summary action  : CREATED
Field summary action     : CREATED
Template summary action  : CREATED
Mismatch file action     : CREATED
Manifest action          : CREATED
Evaluation root          : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/field_extraction/evaluations/rule_based_baseline_v1_0_1_development_eval_v1_0_1
Evaluation manifest      : /content/dri

In [ ]:
from __future__ import annotations

import hashlib
import json
import re
import unicodedata
from pathlib import Path

import pandas as pd
from IPython.display import display


# ============================================================
# CELL 10D.1 — READ-ONLY PARTY-NAME MISMATCH DIAGNOSTIC
# ============================================================

BUILD_ROOT = Path(
    "/content/drive/MyDrive/InvoiceFlow-AI-Data/interim/"
    "synthetic_v1_build/20260904T150025Z"
)
FIELD_ROOT = BUILD_ROOT / "ocr_benchmark" / "field_extraction"

TEXT_LAYER_MANIFEST_PATH = (
    FIELD_ROOT / "unified_text_layer_manifest.json"
)
PREDICTION_MANIFEST_PATH = (
    FIELD_ROOT / "rule_based_baseline_v1_0_1_manifest.json"
)
EVALUATION_MANIFEST_PATH = (
    FIELD_ROOT
    / "evaluations"
    / "rule_based_baseline_v1_0_1_development_eval_v1_0_1"
    / "development_evaluation_manifest.json"
)

TARGET_DOCUMENT_ID = "INV-SYN-000071"
TARGET_TEMPLATE_ID = "TPL-04"
TARGET_FIELD = "vendor.name"


# ============================================================
# HELPERS
# ============================================================

def require_file(path: Path, label: str) -> None:
    if not path.is_file():
        raise FileNotFoundError(f"{label} tidak ditemukan: {path}")
    if path.stat().st_size <= 0:
        raise RuntimeError(f"{label} kosong: {path}")


def load_json(path: Path) -> dict:
    require_file(path, "JSON artifact")
    with path.open("r", encoding="utf-8") as file_handle:
        value = json.load(file_handle)
    if not isinstance(value, dict):
        raise TypeError(f"Root JSON bukan object: {path}")
    return value


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as file_handle:
        for chunk in iter(lambda: file_handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def normalize_text(value: object) -> str:
    text = unicodedata.normalize("NFKC", str(value or ""))
    text = text.casefold()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return " ".join(text.split())


def unique_record(
    records: list[dict],
    *,
    document_id: str,
    label: str,
) -> dict:
    matches = [
        record
        for record in records
        if record.get("document_id") == document_id
    ]
    if len(matches) != 1:
        raise RuntimeError(
            f"{label} untuk {document_id} ditemukan {len(matches)} kali; "
            "seharusnya tepat satu."
        )
    return matches[0]


def bbox_from_line(line: dict) -> tuple[float, float, float, float]:
    bbox = line.get("bbox_points")
    if not isinstance(bbox, list) or len(bbox) != 4:
        raise ValueError(
            f"bbox_points tidak valid pada line_index="
            f"{line.get('line_index')}"
        )
    return tuple(float(value) for value in bbox)


def scalar_prediction(prediction: dict, field_name: str) -> dict:
    scalar_fields = (
        prediction.get("predictions", {}).get("scalar_fields", {})
    )
    field_prediction = scalar_fields.get(field_name)
    if not isinstance(field_prediction, dict):
        raise RuntimeError(f"Prediction field tidak ditemukan: {field_name}")
    return field_prediction


# ============================================================
# INPUT GATES
# ============================================================

for artifact_path, artifact_label in (
    (TEXT_LAYER_MANIFEST_PATH, "Text-layer manifest"),
    (PREDICTION_MANIFEST_PATH, "Prediction manifest"),
    (EVALUATION_MANIFEST_PATH, "Evaluation manifest"),
):
    require_file(artifact_path, artifact_label)

text_layer_manifest = load_json(TEXT_LAYER_MANIFEST_PATH)
prediction_manifest = load_json(PREDICTION_MANIFEST_PATH)
evaluation_manifest = load_json(EVALUATION_MANIFEST_PATH)

text_record = unique_record(
    text_layer_manifest.get("records", []),
    document_id=TARGET_DOCUMENT_ID,
    label="Text-layer record",
)
prediction_record = unique_record(
    prediction_manifest.get("records", []),
    document_id=TARGET_DOCUMENT_ID,
    label="Prediction record",
)
evaluation_record = unique_record(
    evaluation_manifest.get("records", []),
    document_id=TARGET_DOCUMENT_ID,
    label="Evaluation record",
)

for record_name, record in (
    ("text layer", text_record),
    ("prediction", prediction_record),
    ("evaluation", evaluation_record),
):
    if record.get("template_id") != TARGET_TEMPLATE_ID:
        raise RuntimeError(
            f"Template {record_name} tidak cocok: "
            f"{record.get('template_id')}"
        )

text_layer_path = Path(text_record["text_layer_path"])
prediction_path = Path(prediction_record["prediction_path"])
evaluation_path = Path(evaluation_record["evaluation_path"])

for artifact_path, expected_sha256, artifact_label in (
    (
        text_layer_path,
        text_record["text_layer_sha256"],
        "Text-layer checkpoint",
    ),
    (
        prediction_path,
        prediction_record["prediction_sha256"],
        "Prediction checkpoint",
    ),
    (
        evaluation_path,
        evaluation_record["evaluation_sha256"],
        "Evaluation checkpoint",
    ),
):
    require_file(artifact_path, artifact_label)
    actual_sha256 = sha256_file(artifact_path)
    if actual_sha256 != expected_sha256:
        raise RuntimeError(
            f"Checksum {artifact_label} tidak cocok. "
            f"Expected={expected_sha256}, actual={actual_sha256}"
        )

text_layer = load_json(text_layer_path)
prediction = load_json(prediction_path)
evaluation = load_json(evaluation_path)

for artifact_name, artifact in (
    ("text layer", text_layer),
    ("prediction", prediction),
    ("evaluation", evaluation),
):
    document = artifact.get("document", {})
    if document.get("document_id") != TARGET_DOCUMENT_ID:
        raise RuntimeError(f"Document ID {artifact_name} tidak cocok.")
    if document.get("template_id") != TARGET_TEMPLATE_ID:
        raise RuntimeError(f"Template ID {artifact_name} tidak cocok.")
    if document.get("split") != "development":
        raise RuntimeError(f"Artifact {artifact_name} bukan development.")


# ============================================================
# LOCATE THE SINGLE KNOWN MISMATCH
# ============================================================

scalar_comparisons = evaluation.get("scalar_comparisons", [])
target_comparisons = [
    row
    for row in scalar_comparisons
    if row.get("field") == TARGET_FIELD
]
if len(target_comparisons) != 1:
    raise RuntimeError(
        f"Comparison {TARGET_FIELD} ditemukan "
        f"{len(target_comparisons)} kali; seharusnya tepat satu."
    )

target_comparison = target_comparisons[0]
if target_comparison.get("exact_match") is not False:
    raise RuntimeError(
        "Mismatch target sudah tidak tersedia. Jangan gunakan cell "
        "diagnostik lama terhadap artifact yang telah berubah."
    )

vendor_prediction = scalar_prediction(prediction, TARGET_FIELD)
evidence = vendor_prediction.get("evidence", [])
evidence_indexes = {
    int(row["line_index"])
    for row in evidence
    if row.get("line_index") is not None
}

lines = text_layer.get("lines", [])
if not isinstance(lines, list) or not lines:
    raise RuntimeError("Text layer tidak memiliki lines.")

prepared_lines = []
for line in lines:
    x0, y0, x1, y1 = bbox_from_line(line)
    prepared_lines.append(
        {
            **line,
            "x0": x0,
            "y0": y0,
            "x1": x1,
            "y1": y1,
            "cx": (x0 + x1) / 2.0,
            "cy": (y0 + y1) / 2.0,
            "normalized_text": normalize_text(line.get("text")),
        }
    )

evidence_lines = [
    line
    for line in prepared_lines
    if int(line.get("line_index", -1)) in evidence_indexes
]
if not evidence_lines:
    raise RuntimeError("Evidence vendor.name tidak ditemukan pada text layer.")

name_evidence_candidates = [
    line
    for line in evidence_lines
    if normalize_text(line.get("text"))
    == normalize_text(vendor_prediction.get("raw_value"))
]
if len(name_evidence_candidates) != 1:
    raise RuntimeError(
        "Baris evidence nilai vendor.name tidak ditemukan secara unik."
    )

name_line = name_evidence_candidates[0]


# ============================================================
# GEOMETRIC NEIGHBORHOOD AND CONTINUATION CANDIDATES
# ============================================================

# Window sengaja cukup lebar untuk menangkap suffix yang terpecah oleh
# PDF text extraction, tetapi tetap terbatas pada area party vendor.
nearby_lines = [
    line
    for line in prepared_lines
    if line["page_number"] == name_line["page_number"]
    and name_line["y0"] - 16.0 <= line["y0"] <= name_line["y1"] + 44.0
    and line["x0"] <= max(name_line["x1"] + 90.0, name_line["x0"] + 160.0)
]
nearby_lines.sort(
    key=lambda line: (
        line["page_number"],
        line["y0"],
        line["x0"],
        line["line_index"],
    )
)

LEGAL_SUFFIX_PATTERN = re.compile(
    r"^(?:pt|cv|tbk|ud|inc|incorporated|llc|ltd|limited|corp|"
    r"corporation|co|company|gmbh|plc|pte|sdn|bhd)\.?$",
    flags=re.IGNORECASE,
)

continuation_records = []
for line in nearby_lines:
    if line["line_index"] == name_line["line_index"]:
        continue

    text = str(line.get("text", "")).strip()
    same_visual_row = abs(line["cy"] - name_line["cy"]) <= 4.0
    next_visual_line = 0.0 < line["y0"] - name_line["y1"] <= 16.0
    horizontally_related = (
        line["x0"] >= name_line["x0"] - 4.0
        and line["x0"] <= name_line["x1"] + 22.0
    )
    is_legal_suffix = bool(LEGAL_SUFFIX_PATTERN.fullmatch(text))

    if (
        is_legal_suffix
        and horizontally_related
        and (same_visual_row or next_visual_line)
    ):
        continuation_records.append(
            {
                "line_index": line["line_index"],
                "text": text,
                "same_visual_row": same_visual_row,
                "next_visual_line": next_visual_line,
                "horizontal_gap": round(line["x0"] - name_line["x1"], 3),
                "vertical_gap": round(line["y0"] - name_line["y1"], 3),
                "bbox_points": [
                    round(line["x0"], 3),
                    round(line["y0"], 3),
                    round(line["x1"], 3),
                    round(line["y1"], 3),
                ],
            }
        )

display_rows = []
for line in nearby_lines:
    display_rows.append(
        {
            "line_index": line["line_index"],
            "evidence": line["line_index"] in evidence_indexes,
            "text": line.get("text"),
            "x0": round(line["x0"], 3),
            "y0": round(line["y0"], 3),
            "x1": round(line["x1"], 3),
            "y1": round(line["y1"], 3),
        }
    )


# ============================================================
# READ-ONLY CONTROLS AND REPORT
# ============================================================

expected_value = target_comparison.get("expected")
predicted_value = target_comparison.get("predicted")

control_values = [
    ("target_document", TARGET_DOCUMENT_ID, TARGET_DOCUMENT_ID),
    ("target_template", TARGET_TEMPLATE_ID, TARGET_TEMPLATE_ID),
    ("target_split", "development", evaluation["document"].get("split")),
    ("target_field", TARGET_FIELD, target_comparison.get("field")),
    ("exact_match", False, target_comparison.get("exact_match")),
    ("text_layer_checksum", True, sha256_file(text_layer_path) == text_record["text_layer_sha256"]),
    ("prediction_checksum", True, sha256_file(prediction_path) == prediction_record["prediction_sha256"]),
    ("evaluation_checksum", True, sha256_file(evaluation_path) == evaluation_record["evaluation_sha256"]),
    ("validation_opened", 0, 0),
    ("test_opened", 0, 0),
    ("artifact_writes", 0, 0),
]

control_records = [
    {
        "control": name,
        "expected": expected,
        "actual": actual,
        "status": "VALID" if expected == actual else "INVALID",
    }
    for name, expected, actual in control_values
]

display(pd.DataFrame(control_records))

print("\nTARGET MISMATCH")
display(
    pd.DataFrame(
        [
            {
                "document_id": TARGET_DOCUMENT_ID,
                "template_id": TARGET_TEMPLATE_ID,
                "language": evaluation["document"].get("language"),
                "field": TARGET_FIELD,
                "expected": expected_value,
                "predicted": predicted_value,
                "method": vendor_prediction.get("method"),
                "rule_score": vendor_prediction.get("rule_score"),
            }
        ]
    )
)

print("\nVENDOR-NAME GEOMETRIC NEIGHBORHOOD")
display(pd.DataFrame(display_rows))

print("\nLEGAL-SUFFIX CONTINUATION CANDIDATES")
if continuation_records:
    display(pd.DataFrame(continuation_records))
else:
    print("Tidak ada kandidat suffix yang memenuhi aturan geometri konservatif.")

invalid_controls = [
    row["control"]
    for row in control_records
    if row["status"] != "VALID"
]
if invalid_controls:
    raise RuntimeError(
        "CELL 10D.1 DIAGNOSTIC FAILED. "
        f"Kontrol tidak valid: {invalid_controls}"
    )

print()
print(f"Text layer       : {text_layer_path}")
print(f"Prediction       : {prediction_path}")
print(f"Evaluation       : {evaluation_path}")
print(f"Expected         : {expected_value!r}")
print(f"Predicted        : {predicted_value!r}")
print(f"Evidence indexes : {sorted(evidence_indexes)}")
print(f"Suffix candidates: {len(continuation_records)}")
print("Ground truth opened: 1 development evaluation artifact only")
print("Validation opened  : 0")
print("Test opened        : 0")
print("Artifact writes    : 0")
print()
print(
    "✅ CELL 10D.1 PASSED — mismatch vendor.name telah dipetakan "
    "secara read-only. Kirim seluruh output sebelum parser diubah."
)


,control,expected,actual,status
0,target_document,INV-SYN-000071,INV-SYN-000071,VALID
1,target_template,TPL-04,TPL-04,VALID
2,target_split,development,development,VALID
3,target_field,vendor.name,vendor.name,VALID
4,exact_match,False,False,VALID
5,text_layer_checksum,True,True,VALID
6,prediction_checksum,True,True,VALID
7,evaluation_checksum,True,True,VALID
8,validation_opened,0,0,VALID
9,test_opened,0,0,VALID



TARGET MISMATCH


,document_id,template_id,language,field,expected,predicted,method,rule_score
0,INV-SYN-000071,TPL-04,en,vendor.name,Brightmere Example Analytics Inc.,Brightmere Example Analytics,PARTY_ANCHOR_GEOMETRY:VENDOR_NAME,0.91



VENDOR-NAME GEOMETRIC NEIGHBORHOOD


,line_index,evidence,text,x0,y0,x1,y1
0,4,True,Brightmere Example Analytics,28.0,72.000,146.482,83.291
1,7,False,Inc.,28.0,84.082,42.129,95.373
2,8,False,LEFT SIDEBAR · SYNTHETIC,200.0,91.000,289.563,99.951
3,11,False,8984 Example Avenue,28.0,118.000,97.421,127.481



LEGAL-SUFFIX CONTINUATION CANDIDATES


,line_index,text,same_visual_row,next_visual_line,horizontal_gap,vertical_gap,bbox_points
0,7,Inc.,False,True,-118.482,0.79,"[28.0, 84.082, 42.129, 95.373]"



Text layer       : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/field_extraction/text_layers/TPL-04/INV-SYN-000071_text_layer.json
Prediction       : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/field_extraction/predictions/rule_based_baseline_v1_0_1/TPL-04/INV-SYN-000071_prediction.json
Evaluation       : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/field_extraction/evaluations/rule_based_baseline_v1_0_1_development_eval_v1_0_1/documents/TPL-04/INV-SYN-000071_evaluation.json
Expected         : 'Brightmere Example Analytics Inc.'
Predicted        : 'Brightmere Example Analytics'
Evidence indexes : [1, 4]
Suffix candidates: 1
Ground truth opened: 1 development evaluation artifact only
Validation opened  : 0
Test opened        : 0
Artifact writes    : 0

✅ CELL 10D.1 PASSED — mismatch vendor.name telah dipetakan secara re

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import re
import unicodedata
from collections import Counter
from datetime import date
from decimal import Decimal, InvalidOperation
from pathlib import Path

import pandas as pd
from IPython.display import display


# ============================================================
# CELL 10C — DETERMINISTIC RULE-BASED FIELD PARSER BASELINE
# ============================================================

BUILD_ROOT = Path(
    "/content/drive/MyDrive/InvoiceFlow-AI-Data/interim/"
    "synthetic_v1_build/20260904T150025Z"
)
BENCHMARK_ROOT = BUILD_ROOT / "ocr_benchmark"
FIELD_ROOT = BENCHMARK_ROOT / "field_extraction"

CONTRACT_PATH = FIELD_ROOT / "field_extraction_contract_v1.json"
TEXT_LAYER_MANIFEST_PATH = (
    FIELD_ROOT / "unified_text_layer_manifest.json"
)

PREDICTION_ROOT = (
    FIELD_ROOT / "predictions" / "rule_based_baseline_v1_0_2"
)
SUMMARY_PATH = (
    FIELD_ROOT / "rule_based_baseline_v1_0_2_summary.csv"
)
MANIFEST_PATH = (
    FIELD_ROOT / "rule_based_baseline_v1_0_2_manifest.json"
)

PARSER_ID = "RULE-BASED-INVOICE-PARSER-V1"
PARSER_VERSION = "1.0.2"
EXPECTED_DOCUMENTS = 18
EXPECTED_TEMPLATES = 6
EXPECTED_TARGET_FIELDS = 16
EXPECTED_SCALAR_FIELDS = 12
EXPECTED_REPEATING_FIELDS = 4

SUPPORTED_CURRENCIES = {"IDR", "USD", "EUR", "GBP"}

# Legal-entity suffixes may be emitted by a PDF text extractor as a
# separate visual line when a company name wraps inside a narrow box.
# Values are stored without punctuation after normalize_text().
LEGAL_ENTITY_SUFFIXES = {
    "bhd",
    "co",
    "company",
    "corp",
    "corporation",
    "gmbh",
    "inc",
    "incorporated",
    "limited",
    "llc",
    "ltd",
    "plc",
    "pte",
    "sdn",
    "tbk",
}

PARTY_SUFFIX_SAME_ROW_TOLERANCE_POINTS = 4.0
PARTY_SUFFIX_NEXT_LINE_GAP_POINTS = 14.0
PARTY_SUFFIX_LEFT_ALIGNMENT_TOLERANCE_POINTS = 8.0
PARTY_SUFFIX_SAME_ROW_GAP_POINTS = 18.0


# ============================================================
# LABEL DICTIONARY — LANGUAGE-AWARE, TEMPLATE-INDEPENDENT
# ============================================================

LABEL_ALIASES = {
    "invoice_number": {
        "nomor invoice",
        "invoice number",
        "invoice no",
        "invoice #",
    },
    "invoice_date": {
        "tanggal invoice",
        "invoice date",
        "date of invoice",
    },
    "due_date": {
        "jatuh tempo",
        "due date",
        "payment due",
    },
    "currency": {
        "mata uang",
        "currency",
    },
    "vendor_anchor": {
        "dari",
        "from",
        "vendor",
        "seller",
    },
    "buyer_anchor": {
        "ditagihkan kepada",
        "tagihan kepada",
        "bill to",
        "billed to",
        "buyer",
        "customer",
    },
    "tax_identifier": {
        "id pajak",
        "nomor pajak",
        "tax id",
        "tax identifier",
        "vat id",
        "vat number",
    },
    "description": {
        "deskripsi",
        "description",
        "item description",
    },
    "quantity": {
        "kuantitas",
        "qty",
        "quantity",
    },
    "unit_price": {
        "harga satuan",
        "unit price",
        "price",
    },
    "line_total": {
        "jumlah",
        "line total",
        "amount",
    },
    "subtotal": {
        "subtotal",
        "sub total",
    },
    "tax": {
        "pajak",
        "tax",
        "vat",
    },
    "discount": {
        "diskon",
        "discount",
    },
    "total": {
        "total",
        "grand total",
        "total due",
        "amount due",
    },
}

MONTHS = {
    "januari": 1,
    "january": 1,
    "jan": 1,
    "februari": 2,
    "february": 2,
    "feb": 2,
    "maret": 3,
    "march": 3,
    "mar": 3,
    "april": 4,
    "apr": 4,
    "mei": 5,
    "may": 5,
    "juni": 6,
    "june": 6,
    "jun": 6,
    "juli": 7,
    "july": 7,
    "jul": 7,
    "agustus": 8,
    "august": 8,
    "aug": 8,
    "september": 9,
    "sep": 9,
    "sept": 9,
    "oktober": 10,
    "october": 10,
    "oct": 10,
    "november": 11,
    "nov": 11,
    "desember": 12,
    "december": 12,
    "dec": 12,
}

ADDRESS_TERMS = {
    "jalan",
    "jl",
    "street",
    "road",
    "avenue",
    "kompleks",
    "kawasan",
    "koridor",
    "blok",
    "block",
    "kota",
    "city",
    "kode pos",
    "postal code",
    "postcode",
    "zip code",
}


# ============================================================
# FILE AND SERIALIZATION HELPERS
# ============================================================

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def load_json(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as handle:
        value = json.load(handle)
    if not isinstance(value, dict):
        raise TypeError(f"JSON harus berupa object: {path}")
    return value


def canonical_json(value) -> str:
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    )


def atomic_write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(path.name + ".tmp")
    temporary_path.write_text(text, encoding="utf-8")
    os.replace(temporary_path, path)


def atomic_write_json(path: Path, value: dict) -> None:
    atomic_write_text(
        path,
        json.dumps(
            value,
            indent=2,
            ensure_ascii=False,
            default=str,
        )
        + "\n",
    )


def require_file(path: Path, label: str) -> None:
    if not path.is_file():
        raise FileNotFoundError(f"{label} tidak ditemukan: {path}")


# ============================================================
# TEXT NORMALIZATION AND GEOMETRY
# ============================================================

def normalize_spaces(value: str) -> str:
    value = unicodedata.normalize("NFKC", str(value))
    return re.sub(r"\s+", " ", value).strip()


def normalize_for_match(value: str) -> str:
    value = normalize_spaces(value).casefold()
    value = value.replace("&", " and ")
    value = re.sub(r"[^a-z0-9]+", " ", value)
    return re.sub(r"\s+", " ", value).strip()


def normalized_aliases(key: str) -> set[str]:
    return {
        normalize_for_match(alias)
        for alias in LABEL_ALIASES[key]
    }


NORMALIZED_LABEL_ALIASES = {
    key: normalized_aliases(key)
    for key in LABEL_ALIASES
}


def line_matches_alias(
    line: dict,
    alias_key: str,
    *,
    allow_suffix: bool = False,
) -> bool:
    normalized = line["normalized_text"]
    for alias in NORMALIZED_LABEL_ALIASES[alias_key]:
        if normalized == alias:
            return True
        if allow_suffix and normalized.startswith(alias + " "):
            return True
    return False


def prepare_lines(text_layer: dict) -> list[dict]:
    prepared = []
    source_lines = text_layer.get("lines", [])
    if not isinstance(source_lines, list):
        raise TypeError("text_layer.lines harus berupa list.")

    for fallback_index, source_line in enumerate(source_lines):
        if not isinstance(source_line, dict):
            continue

        text = normalize_spaces(source_line.get("text", ""))
        bbox = source_line.get("bbox_points", [])
        if not text or not isinstance(bbox, list) or len(bbox) != 4:
            continue

        x0, y0, x1, y1 = map(float, bbox)
        if x1 <= x0 or y1 <= y0:
            continue

        prepared.append(
            {
                "line_index": int(
                    source_line.get("line_index", fallback_index)
                ),
                "page_number": int(
                    source_line.get("page_number", 1)
                ),
                "text": text,
                "normalized_text": normalize_for_match(text),
                "x0": x0,
                "y0": y0,
                "x1": x1,
                "y1": y1,
                "cx": (x0 + x1) / 2.0,
                "cy": (y0 + y1) / 2.0,
                "width": x1 - x0,
                "height": y1 - y0,
                "bbox_points": [x0, y0, x1, y1],
                "confidence": float(source_line.get("confidence", 1.0)),
                "source": str(source_line.get("source", "UNKNOWN")),
            }
        )

    return sorted(
        prepared,
        key=lambda line: (
            line["page_number"],
            line["y0"],
            line["x0"],
            line["line_index"],
        ),
    )


def evidence_from_lines(lines: list[dict]) -> list[dict]:
    return [
        {
            "line_index": line["line_index"],
            "page_number": line["page_number"],
            "text": line["text"],
            "bbox_points": [round(value, 4) for value in line["bbox_points"]],
        }
        for line in lines
    ]


def find_label_lines(
    lines: list[dict],
    alias_key: str,
    *,
    minimum_y: float | None = None,
    maximum_y: float | None = None,
    allow_suffix: bool = False,
) -> list[dict]:
    matches = []
    for line in lines:
        if minimum_y is not None and line["y0"] < minimum_y:
            continue
        if maximum_y is not None and line["y0"] > maximum_y:
            continue
        if line_matches_alias(
            line,
            alias_key,
            allow_suffix=allow_suffix,
        ):
            matches.append(line)
    return matches


def choose_first_label(
    lines: list[dict],
    alias_key: str,
    *,
    minimum_y: float | None = None,
    maximum_y: float | None = None,
    allow_suffix: bool = False,
) -> dict | None:
    matches = find_label_lines(
        lines,
        alias_key,
        minimum_y=minimum_y,
        maximum_y=maximum_y,
        allow_suffix=allow_suffix,
    )
    if not matches:
        return None
    return min(matches, key=lambda line: (line["y0"], line["x0"]))


# ============================================================
# VALUE NORMALIZERS
# ============================================================

def decimal_to_string(value: Decimal) -> str:
    if value == value.to_integral():
        return str(value.quantize(Decimal("1")))
    normalized = format(value.normalize(), "f")
    return normalized.rstrip("0").rstrip(".")


def parse_decimal_number(raw_value: str) -> Decimal | None:
    text = normalize_spaces(raw_value)
    text = re.sub(r"[^0-9,\.\-]", "", text)
    if not text or text in {"-", ".", ","}:
        return None

    negative = text.startswith("-")
    text = text.lstrip("-")

    if "," in text and "." in text:
        decimal_separator = (
            "," if text.rfind(",") > text.rfind(".") else "."
        )
        thousands_separator = "." if decimal_separator == "," else ","
        text = text.replace(thousands_separator, "")
        text = text.replace(decimal_separator, ".")
    elif "," in text or "." in text:
        separator = "," if "," in text else "."
        pieces = text.split(separator)

        if len(pieces) == 2 and len(pieces[-1]) in {1, 2}:
            text = pieces[0] + "." + pieces[1]
        elif len(pieces) > 2 and len(pieces[-1]) in {1, 2}:
            text = "".join(pieces[:-1]) + "." + pieces[-1]
        else:
            text = "".join(pieces)

    if negative:
        text = "-" + text

    try:
        return Decimal(text)
    except InvalidOperation:
        return None


def normalize_money(raw_value: str) -> str | None:
    value = parse_decimal_number(raw_value)
    if value is None:
        return None
    return decimal_to_string(value)


def normalize_quantity(raw_value: str) -> str | None:
    text = normalize_spaces(raw_value)
    if not re.fullmatch(r"[-+]?\d+(?:[\.,]\d+)?", text):
        return None
    value = parse_decimal_number(text)
    if value is None:
        return None
    return decimal_to_string(value)


def normalize_currency(raw_value: str) -> str | None:
    match = re.fullmatch(
        r"\s*(IDR|USD|EUR|GBP)\s*",
        normalize_spaces(raw_value),
        flags=re.IGNORECASE,
    )
    if not match:
        return None
    currency = match.group(1).upper()
    return currency if currency in SUPPORTED_CURRENCIES else None


def normalize_date(raw_value: str) -> str | None:
    text = normalize_for_match(raw_value)

    iso_match = re.fullmatch(
        r"(\d{4})[\-/](\d{1,2})[\-/](\d{1,2})",
        text,
    )
    if iso_match:
        year, month, day = map(int, iso_match.groups())
        try:
            return date(year, month, day).isoformat()
        except ValueError:
            return None

    numeric_match = re.fullmatch(
        r"(\d{1,2})[\-/](\d{1,2})[\-/](\d{4})",
        text,
    )
    if numeric_match:
        day, month, year = map(int, numeric_match.groups())
        try:
            return date(year, month, day).isoformat()
        except ValueError:
            return None

    day_first = re.fullmatch(
        r"(\d{1,2}) ([a-z]+) (\d{4})",
        text,
    )
    month_first = re.fullmatch(
        r"([a-z]+) (\d{1,2}) (\d{4})",
        text,
    )

    if day_first:
        day = int(day_first.group(1))
        month = MONTHS.get(day_first.group(2))
        year = int(day_first.group(3))
    elif month_first:
        month = MONTHS.get(month_first.group(1))
        day = int(month_first.group(2))
        year = int(month_first.group(3))
    else:
        return None

    if month is None:
        return None

    try:
        return date(year, month, day).isoformat()
    except ValueError:
        return None


def normalize_identifier(raw_value: str) -> str | None:
    text = normalize_spaces(raw_value)
    normalized = normalize_for_match(text)

    for alias in sorted(
        NORMALIZED_LABEL_ALIASES["tax_identifier"],
        key=len,
        reverse=True,
    ):
        if normalized.startswith(alias):
            original_pattern = re.compile(
                r"^\s*"
                + r"[\W_]*".join(
                    re.escape(part)
                    for part in alias.split()
                )
                + r"\s*[:#\-]?\s*",
                flags=re.IGNORECASE,
            )
            text = original_pattern.sub("", text, count=1)
            break

    text = normalize_spaces(text).upper()
    return text or None


def normalize_string(raw_value: str) -> str | None:
    text = normalize_spaces(raw_value)
    return text or None


def normalize_invoice_number(raw_value: str) -> str | None:
    text = normalize_spaces(raw_value).upper()
    if " " in text:
        return None
    if not re.search(r"[A-Z]", text) or not re.search(r"\d", text):
        return None
    if not re.fullmatch(r"[A-Z]{2,12}[-/][A-Z0-9][A-Z0-9./-]{4,}", text):
        return None
    return text


def looks_like_money(raw_value: str) -> bool:
    text = normalize_spaces(raw_value)
    has_digit = bool(re.search(r"\d", text))
    has_currency = bool(
        re.search(r"\b(?:IDR|USD|EUR|GBP)\b", text, re.IGNORECASE)
    )
    return has_digit and has_currency and normalize_money(text) is not None


def normalize_currency_money(raw_value: str) -> str | None:
    """Normalize only values that visibly carry a currency code."""
    if not looks_like_money(raw_value):
        return None
    return normalize_money(raw_value)


def looks_like_date(raw_value: str) -> bool:
    return normalize_date(raw_value) is not None


def looks_like_currency(raw_value: str) -> bool:
    return normalize_currency(raw_value) is not None


# ============================================================
# GENERIC LABEL-TO-VALUE ASSOCIATION
# ============================================================

def select_labeled_value(
    lines: list[dict],
    anchor: dict | None,
    validator,
    *,
    minimum_y: float | None = None,
    maximum_vertical_gap: float = 52.0,
) -> tuple[dict | None, str | None]:
    if anchor is None:
        return None, None

    candidates = []
    same_row_tolerance = max(4.0, anchor["height"] * 1.25)

    for line in lines:
        if line["line_index"] == anchor["line_index"]:
            continue
        if line["page_number"] != anchor["page_number"]:
            continue
        if minimum_y is not None and line["y0"] < minimum_y:
            continue

        normalized_value = validator(line["text"])
        if normalized_value is None:
            continue

        y_distance = abs(line["cy"] - anchor["cy"])
        is_same_row = (
            y_distance <= same_row_tolerance
            and line["x0"] >= anchor["x1"] - 4.0
        )

        vertical_gap = line["y0"] - anchor["y1"]
        horizontal_center_gap = abs(line["cx"] - anchor["cx"])
        below_width_limit = max(
            105.0,
            anchor["width"] * 2.75,
        )
        is_below = (
            -2.0 <= vertical_gap <= maximum_vertical_gap
            and horizontal_center_gap <= below_width_limit
        )

        if not is_same_row and not is_below:
            continue

        if is_same_row:
            score = (
                y_distance * 6.0
                + max(0.0, line["x0"] - anchor["x1"]) * 0.08
            )
            relation = "SAME_ROW_RIGHT"
        else:
            score = (
                max(0.0, vertical_gap)
                + horizontal_center_gap * 0.08
            )
            relation = "BELOW_ALIGNED"

        candidates.append(
            (score, line["y0"], line["x0"], line, normalized_value, relation)
        )

    if not candidates:
        return None, None

    _, _, _, best_line, normalized_value, relation = min(candidates)
    best_line = {**best_line, "relation_to_label": relation}
    return best_line, normalized_value


def field_prediction(
    raw_value: str | None,
    normalized_value: str | None,
    method: str,
    evidence_lines: list[dict],
    *,
    rule_score: float,
) -> dict:
    return {
        "raw_value": raw_value,
        "normalized_value": normalized_value,
        "method": method,
        "rule_score": round(float(rule_score), 4),
        "rule_score_is_calibrated_probability": False,
        "evidence": evidence_from_lines(evidence_lines),
    }


def empty_prediction(method: str) -> dict:
    return field_prediction(
        None,
        None,
        method,
        [],
        rule_score=0.0,
    )


def extract_labeled_scalar(
    lines: list[dict],
    alias_key: str,
    validator,
    *,
    method: str,
    rule_score: float,
    minimum_y: float | None = None,
    allow_label_suffix: bool = False,
    maximum_vertical_gap: float = 52.0,
) -> dict:
    anchor = choose_first_label(
        lines,
        alias_key,
        minimum_y=minimum_y,
        allow_suffix=allow_label_suffix,
    )
    value_line, normalized_value = select_labeled_value(
        lines,
        anchor,
        validator,
        minimum_y=minimum_y,
        maximum_vertical_gap=maximum_vertical_gap,
    )

    if anchor is None or value_line is None:
        return empty_prediction(method)

    return field_prediction(
        value_line["text"],
        normalized_value,
        method + ":" + value_line["relation_to_label"],
        [anchor, value_line],
        rule_score=rule_score,
    )


# ============================================================
# METADATA PARSER
# ============================================================

INVOICE_NUMBER_PATTERN = re.compile(
    r"\b(?:FTR|INV)-\d{4}-\d{2}-\d{6}\b",
    flags=re.IGNORECASE,
)


def extract_invoice_number(lines: list[dict]) -> dict:
    matches = []
    for line in lines:
        match = INVOICE_NUMBER_PATTERN.search(line["text"])
        if match:
            matches.append((line, match.group(0).upper()))

    if matches:
        line, normalized_value = min(
            matches,
            key=lambda item: (item[0]["y0"], item[0]["x0"]),
        )
        return field_prediction(
            normalized_value,
            normalized_value,
            "REGEX:BILINGUAL_INVOICE_NUMBER",
            [line],
            rule_score=0.99,
        )

    labeled_prediction = extract_labeled_scalar(
        lines,
        "invoice_number",
        normalize_invoice_number,
        method="LABEL_GEOMETRY:INVOICE_NUMBER",
        rule_score=0.95,
    )
    if labeled_prediction["normalized_value"] is not None:
        return labeled_prediction

    return empty_prediction("REGEX_AND_LABEL_GEOMETRY:INVOICE_NUMBER")


def parse_metadata(lines: list[dict]) -> dict[str, dict]:
    return {
        "invoice_number": extract_invoice_number(lines),
        "invoice_date": extract_labeled_scalar(
            lines,
            "invoice_date",
            normalize_date,
            method="LABEL_GEOMETRY:INVOICE_DATE",
            rule_score=0.96,
        ),
        "due_date": extract_labeled_scalar(
            lines,
            "due_date",
            normalize_date,
            method="LABEL_GEOMETRY:DUE_DATE",
            rule_score=0.96,
        ),
        "currency": extract_labeled_scalar(
            lines,
            "currency",
            normalize_currency,
            method="LABEL_GEOMETRY:CURRENCY",
            rule_score=0.98,
        ),
    }


# ============================================================
# PARTY PARSER
# ============================================================

def is_any_known_label(line: dict) -> bool:
    return any(
        line_matches_alias(line, key, allow_suffix=True)
        for key in NORMALIZED_LABEL_ALIASES
    )


def looks_like_party_name(line: dict) -> bool:
    text = line["text"]
    normalized = line["normalized_text"]

    if not re.search(r"[A-Za-z]", text):
        return False
    if "@" in text or re.search(r"\bhttps?://", text, re.IGNORECASE):
        return False
    if re.search(r"\+?\d[\d\-() ]{7,}", text):
        return False
    if INVOICE_NUMBER_PATTERN.search(text):
        return False
    if re.search(r"\bINV-SYN-\d+\b", text, re.IGNORECASE):
        return False
    if looks_like_date(text) or looks_like_money(text):
        return False
    if is_any_known_label(line):
        return False
    if any(term in normalized for term in ADDRESS_TERMS):
        return False
    if normalized.startswith("synthetic") or normalized.startswith("data sintetis"):
        return False
    if len(normalized.split()) < 2:
        return False

    return True


def side_boundary(
    first_anchor: dict | None,
    second_anchor: dict | None,
) -> float | None:
    if first_anchor is None or second_anchor is None:
        return None
    if abs(first_anchor["cy"] - second_anchor["cy"]) > 32.0:
        return None
    return (first_anchor["cx"] + second_anchor["cx"]) / 2.0


def legal_suffix_continuation(
    lines: list[dict],
    name_line: dict,
    anchor: dict,
    other_anchor: dict | None,
    party: str,
) -> dict | None:
    """Return one conservative wrapped legal-suffix continuation.

    This rule is template-independent. It accepts only a known legal suffix
    on the same page, either immediately to the right on the same visual row
    or immediately below with the same left alignment. Party boundaries are
    applied before ranking candidates, preventing a suffix from crossing
    into the opposite party region.
    """

    boundary = side_boundary(anchor, other_anchor)
    candidates = []

    for line in lines:
        if line["page_number"] != name_line["page_number"]:
            continue
        if line["line_index"] == name_line["line_index"]:
            continue
        if line["normalized_text"] not in LEGAL_ENTITY_SUFFIXES:
            continue

        if boundary is not None:
            if party == "vendor" and line["cx"] >= boundary:
                continue
            if party == "buyer" and line["cx"] < boundary:
                continue
        elif (
            other_anchor is not None
            and other_anchor["cy"] > anchor["cy"] + 32.0
            and line["cy"] >= other_anchor["cy"] - 2.0
        ):
            continue

        same_visual_row = (
            abs(line["cy"] - name_line["cy"])
            <= PARTY_SUFFIX_SAME_ROW_TOLERANCE_POINTS
        )
        same_row_gap = line["x0"] - name_line["x1"]
        same_row_candidate = bool(
            same_visual_row
            and -2.0 <= same_row_gap <= PARTY_SUFFIX_SAME_ROW_GAP_POINTS
        )

        next_line_gap = line["y0"] - name_line["y1"]
        left_alignment_delta = abs(line["x0"] - name_line["x0"])
        next_line_candidate = bool(
            0.0 <= next_line_gap <= PARTY_SUFFIX_NEXT_LINE_GAP_POINTS
            and left_alignment_delta
            <= PARTY_SUFFIX_LEFT_ALIGNMENT_TOLERANCE_POINTS
        )

        if not (same_row_candidate or next_line_candidate):
            continue

        relation_priority = 0 if same_row_candidate else 1
        geometric_distance = (
            abs(line["cy"] - name_line["cy"])
            if same_row_candidate
            else next_line_gap + left_alignment_delta
        )
        candidates.append(
            (
                relation_priority,
                geometric_distance,
                line["y0"],
                line["x0"],
                line,
            )
        )

    if not candidates:
        return None

    return min(
        candidates,
        key=lambda candidate: candidate[:4],
    )[-1]


def extract_party_name(
    lines: list[dict],
    anchor: dict | None,
    other_anchor: dict | None,
    party: str,
) -> dict:
    method = f"PARTY_ANCHOR_GEOMETRY:{party.upper()}_NAME"
    if anchor is None:
        return empty_prediction(method)

    boundary = side_boundary(anchor, other_anchor)
    candidates = []

    for line in lines:
        if line["page_number"] != anchor["page_number"]:
            continue
        if line["line_index"] == anchor["line_index"]:
            continue

        vertical_gap = line["y0"] - anchor["y1"]
        if not 0.0 <= vertical_gap <= 82.0:
            continue
        if not looks_like_party_name(line):
            continue

        if boundary is not None:
            if party == "vendor" and line["cx"] >= boundary:
                continue
            if party == "buyer" and line["cx"] < boundary:
                continue
        elif (
            other_anchor is not None
            and other_anchor["cy"] > anchor["cy"] + 32.0
            and line["cy"] >= other_anchor["cy"] - 2.0
        ):
            continue

        horizontal_gap = abs(line["x0"] - anchor["x0"])
        score = vertical_gap * 5.0 + horizontal_gap * 0.35
        candidates.append((score, line["y0"], line["x0"], line))

    if not candidates:
        return empty_prediction(method)

    _, _, _, name_line = min(candidates)
    suffix_line = legal_suffix_continuation(
        lines,
        name_line,
        anchor,
        other_anchor,
        party,
    )

    if suffix_line is None:
        raw_value = name_line["text"]
        evidence_lines = [anchor, name_line]
        final_method = method
        rule_score = 0.91
    else:
        raw_value = f"{name_line['text'].rstrip()} {suffix_line['text'].strip()}"
        evidence_lines = [anchor, name_line, suffix_line]
        final_method = f"{method}:LEGAL_SUFFIX_CONTINUATION"
        rule_score = 0.93

    return field_prediction(
        raw_value,
        normalize_string(raw_value),
        final_method,
        evidence_lines,
        rule_score=rule_score,
    )


def extract_party_tax_identifier(
    lines: list[dict],
    anchor: dict | None,
    other_anchor: dict | None,
    party: str,
) -> dict:
    method = f"PARTY_ANCHOR_GEOMETRY:{party.upper()}_TAX_IDENTIFIER"
    if anchor is None:
        return empty_prediction(method)

    tax_lines = find_label_lines(
        lines,
        "tax_identifier",
        allow_suffix=True,
    )
    candidates = []

    for line in tax_lines:
        if line["page_number"] != anchor["page_number"]:
            continue

        anchors_are_vertically_stacked = bool(
            other_anchor is not None
            and abs(anchor["cy"] - other_anchor["cy"]) > 32.0
            and abs(anchor["x0"] - other_anchor["x0"]) < 40.0
        )
        if anchors_are_vertically_stacked:
            if (
                anchor["cy"] < other_anchor["cy"]
                and line["cy"] >= other_anchor["cy"]
            ):
                continue
            if (
                anchor["cy"] > other_anchor["cy"]
                and line["cy"] <= anchor["cy"]
            ):
                continue

        normalized_value = normalize_identifier(line["text"])
        if not normalized_value:
            continue

        score = (
            abs(line["x0"] - anchor["x0"])
            + abs(line["cy"] - anchor["cy"]) * 0.25
        )
        candidates.append((score, line["y0"], line["x0"], line, normalized_value))

    if not candidates:
        return empty_prediction(method)

    _, _, _, tax_line, normalized_value = min(candidates)
    return field_prediction(
        tax_line["text"],
        normalized_value,
        method,
        [anchor, tax_line],
        rule_score=0.93,
    )


def parse_parties(lines: list[dict]) -> dict[str, dict]:
    vendor_anchor = choose_first_label(lines, "vendor_anchor")
    buyer_anchor = choose_first_label(lines, "buyer_anchor")

    return {
        "vendor.name": extract_party_name(
            lines,
            vendor_anchor,
            buyer_anchor,
            "vendor",
        ),
        "vendor.tax_identifier": extract_party_tax_identifier(
            lines,
            vendor_anchor,
            buyer_anchor,
            "vendor",
        ),
        "buyer.name": extract_party_name(
            lines,
            buyer_anchor,
            vendor_anchor,
            "buyer",
        ),
        "buyer.tax_identifier": extract_party_tax_identifier(
            lines,
            buyer_anchor,
            vendor_anchor,
            "buyer",
        ),
    }


# ============================================================
# ITEM TABLE PARSER
# ============================================================

def locate_table_header(lines: list[dict]) -> dict | None:
    descriptions = find_label_lines(lines, "description")
    quantities = find_label_lines(lines, "quantity")
    unit_prices = find_label_lines(lines, "unit_price")
    line_totals = find_label_lines(lines, "line_total")

    header_candidates = []
    for description in descriptions:
        for quantity in quantities:
            if quantity["page_number"] != description["page_number"]:
                continue
            if abs(quantity["cy"] - description["cy"]) > 4.0:
                continue
            if quantity["cx"] <= description["cx"]:
                continue

            compatible_unit_prices = [
                line
                for line in unit_prices
                if line["page_number"] == description["page_number"]
                and abs(line["cy"] - description["cy"]) <= 4.0
                and line["cx"] > quantity["cx"]
            ]
            compatible_line_totals = [
                line
                for line in line_totals
                if line["page_number"] == description["page_number"]
                and abs(line["cy"] - description["cy"]) <= 4.0
                and line["cx"] > quantity["cx"]
            ]

            for unit_price in compatible_unit_prices:
                for line_total in compatible_line_totals:
                    if line_total["cx"] <= unit_price["cx"]:
                        continue
                    vertical_spread = max(
                        description["cy"],
                        quantity["cy"],
                        unit_price["cy"],
                        line_total["cy"],
                    ) - min(
                        description["cy"],
                        quantity["cy"],
                        unit_price["cy"],
                        line_total["cy"],
                    )
                    header_candidates.append(
                        (
                            vertical_spread,
                            description["y0"],
                            {
                                "description": description,
                                "quantity": quantity,
                                "unit_price": unit_price,
                                "line_total": line_total,
                            },
                        )
                    )

    if not header_candidates:
        return None

    _, _, header = min(header_candidates, key=lambda item: item[:2])
    return header


def cluster_lines_by_row(
    lines: list[dict],
    tolerance: float = 2.8,
) -> list[list[dict]]:
    clusters = []

    for line in sorted(lines, key=lambda value: (value["cy"], value["x0"])):
        if not clusters:
            clusters.append([line])
            continue

        last_cluster = clusters[-1]
        cluster_center = sum(item["cy"] for item in last_cluster) / len(last_cluster)
        if abs(line["cy"] - cluster_center) <= tolerance:
            last_cluster.append(line)
        else:
            clusters.append([line])

    return clusters


def concatenate_cell_lines(lines: list[dict]) -> str:
    return normalize_spaces(
        " ".join(line["text"] for line in sorted(lines, key=lambda item: item["x0"]))
    )


def find_table_stop_y(lines: list[dict], header_y: float) -> float:
    subtotal_labels = find_label_lines(
        lines,
        "subtotal",
        minimum_y=header_y + 5.0,
    )
    if subtotal_labels:
        return min(line["y0"] for line in subtotal_labels)

    page_lines = [line for line in lines if line["page_number"] == 1]
    return max((line["y1"] for line in page_lines), default=10000.0)


def parse_item_table(lines: list[dict]) -> tuple[list[dict], dict]:
    header = locate_table_header(lines)
    if header is None:
        return [], {
            "table_detected": False,
            "header_evidence": [],
            "table_stop_y": None,
            "candidate_row_count": 0,
        }

    header_lines = list(header.values())
    header_y = sum(line["cy"] for line in header_lines) / len(header_lines)
    table_stop_y = find_table_stop_y(lines, header_y)

    centers = {
        key: header[key]["cx"]
        for key in ("description", "quantity", "unit_price", "line_total")
    }
    description_quantity_boundary = (
        centers["description"] + centers["quantity"]
    ) / 2.0
    quantity_unit_boundary = (
        centers["quantity"] + centers["unit_price"]
    ) / 2.0
    unit_total_boundary = (
        centers["unit_price"] + centers["line_total"]
    ) / 2.0

    row_candidates = [
        line
        for line in lines
        if line["page_number"] == header["description"]["page_number"]
        and line["y0"] > max(item["y1"] for item in header_lines) + 3.0
        and line["y1"] < table_stop_y - 1.0
    ]

    row_clusters = cluster_lines_by_row(row_candidates)
    parsed_items = []

    for cluster in row_clusters:
        cell_lines = {
            "description": [],
            "quantity": [],
            "unit_price": [],
            "line_total": [],
        }

        for line in cluster:
            if line["x1"] < header["description"]["x0"] - 2.0:
                continue

            if line["cx"] < description_quantity_boundary:
                column = "description"
            elif line["cx"] < quantity_unit_boundary:
                column = "quantity"
            elif line["cx"] < unit_total_boundary:
                column = "unit_price"
            else:
                column = "line_total"
            cell_lines[column].append(line)

        raw_description = concatenate_cell_lines(cell_lines["description"])
        raw_quantity = concatenate_cell_lines(cell_lines["quantity"])
        raw_unit_price = concatenate_cell_lines(cell_lines["unit_price"])
        raw_line_total = concatenate_cell_lines(cell_lines["line_total"])

        normalized_description = normalize_string(raw_description)
        normalized_quantity = normalize_quantity(raw_quantity)
        normalized_unit_price = normalize_money(raw_unit_price)
        normalized_line_total = normalize_money(raw_line_total)

        if (
            not normalized_description
            or not re.search(r"[A-Za-z]", normalized_description)
            or normalized_quantity is None
            or normalized_unit_price is None
            or normalized_line_total is None
            or not looks_like_money(raw_unit_price)
            or not looks_like_money(raw_line_total)
        ):
            continue

        parsed_items.append(
            {
                "row_number": len(parsed_items) + 1,
                "description": field_prediction(
                    raw_description,
                    normalized_description,
                    "TABLE_GEOMETRY:DESCRIPTION_COLUMN",
                    cell_lines["description"],
                    rule_score=0.94,
                ),
                "quantity": field_prediction(
                    raw_quantity,
                    normalized_quantity,
                    "TABLE_GEOMETRY:QUANTITY_COLUMN",
                    cell_lines["quantity"],
                    rule_score=0.94,
                ),
                "unit_price": field_prediction(
                    raw_unit_price,
                    normalized_unit_price,
                    "TABLE_GEOMETRY:UNIT_PRICE_COLUMN",
                    cell_lines["unit_price"],
                    rule_score=0.94,
                ),
                "line_total": field_prediction(
                    raw_line_total,
                    normalized_line_total,
                    "TABLE_GEOMETRY:LINE_TOTAL_COLUMN",
                    cell_lines["line_total"],
                    rule_score=0.94,
                ),
            }
        )

    diagnostics = {
        "table_detected": True,
        "header_evidence": evidence_from_lines(header_lines),
        "header_centers": {
            key: round(value, 4) for key, value in centers.items()
        },
        "table_stop_y": round(table_stop_y, 4),
        "candidate_row_count": len(row_clusters),
        "parsed_item_count": len(parsed_items),
    }
    return parsed_items, diagnostics


# ============================================================
# FINANCIAL SUMMARY PARSER
# ============================================================

def extract_financial_field(
    lines: list[dict],
    alias_key: str,
    table_stop_y: float | None,
) -> dict:
    minimum_y = None if table_stop_y is None else table_stop_y - 2.0
    return extract_labeled_scalar(
        lines,
        alias_key,
        normalize_currency_money,
        method=f"SUMMARY_LABEL_GEOMETRY:{alias_key.upper()}",
        rule_score=0.96,
        minimum_y=minimum_y,
        allow_label_suffix=alias_key == "tax",
        maximum_vertical_gap=46.0,
    )


def parse_financials(
    lines: list[dict],
    table_stop_y: float | None,
) -> dict[str, dict]:
    return {
        "financials.subtotal": extract_financial_field(
            lines,
            "subtotal",
            table_stop_y,
        ),
        "financials.tax": extract_financial_field(
            lines,
            "tax",
            table_stop_y,
        ),
        "financials.discount": extract_financial_field(
            lines,
            "discount",
            table_stop_y,
        ),
        "financials.total": extract_financial_field(
            lines,
            "total",
            table_stop_y,
        ),
    }


def financial_equation_check(scalars: dict[str, dict]) -> dict:
    keys = {
        "subtotal": "financials.subtotal",
        "tax": "financials.tax",
        "discount": "financials.discount",
        "total": "financials.total",
    }
    values = {}

    for short_name, field_name in keys.items():
        raw_value = scalars[field_name]["normalized_value"]
        if raw_value is None:
            return {
                "executed": False,
                "passed": None,
                "reason": f"MISSING_{short_name.upper()}",
            }
        try:
            values[short_name] = Decimal(raw_value)
        except InvalidOperation:
            return {
                "executed": False,
                "passed": None,
                "reason": f"INVALID_{short_name.upper()}",
            }

    expected_total = (
        values["subtotal"]
        + values["tax"]
        - values["discount"]
    )
    delta = values["total"] - expected_total

    return {
        "executed": True,
        "passed": delta == Decimal("0"),
        "formula": "subtotal + tax - discount = total",
        "expected_total": decimal_to_string(expected_total),
        "observed_total": decimal_to_string(values["total"]),
        "delta": decimal_to_string(delta),
    }


# ============================================================
# DOCUMENT PARSER
# ============================================================

SCALAR_FIELD_NAMES = [
    "invoice_number",
    "invoice_date",
    "due_date",
    "currency",
    "vendor.name",
    "vendor.tax_identifier",
    "buyer.name",
    "buyer.tax_identifier",
    "financials.subtotal",
    "financials.tax",
    "financials.discount",
    "financials.total",
]

REQUIRED_SCALAR_FIELD_NAMES = [
    field_name
    for field_name in SCALAR_FIELD_NAMES
    if field_name
    not in {"vendor.tax_identifier", "buyer.tax_identifier"}
]


def parse_document(text_layer: dict) -> dict:
    lines = prepare_lines(text_layer)
    metadata = parse_metadata(lines)
    parties = parse_parties(lines)
    items, table_diagnostics = parse_item_table(lines)
    financials = parse_financials(
        lines,
        table_diagnostics.get("table_stop_y"),
    )

    scalars = {**metadata, **parties, **financials}
    missing_required_scalars = [
        field_name
        for field_name in REQUIRED_SCALAR_FIELD_NAMES
        if scalars[field_name]["normalized_value"] is None
    ]

    return {
        "scalar_fields": scalars,
        "items": items,
        "diagnostics": {
            "prepared_line_count": len(lines),
            "populated_scalar_field_count": sum(
                prediction["normalized_value"] is not None
                for prediction in scalars.values()
            ),
            "missing_required_scalar_fields": missing_required_scalars,
            "table": table_diagnostics,
            "financial_equation": financial_equation_check(scalars),
        },
    }


# ============================================================
# INPUT GATES — NO GROUND TRUTH IS OPENED IN THIS CELL
# ============================================================

require_file(CONTRACT_PATH, "Field extraction contract")
require_file(TEXT_LAYER_MANIFEST_PATH, "Unified text-layer manifest")

contract = load_json(CONTRACT_PATH)
text_layer_manifest = load_json(TEXT_LAYER_MANIFEST_PATH)

contract_fields = contract.get("field_definitions", [])
scalar_contract_fields = [
    field
    for field in contract_fields
    if field.get("cardinality") != "REPEATING"
]
repeating_contract_fields = [
    field
    for field in contract_fields
    if field.get("cardinality") == "REPEATING"
]

input_gate_values = [
    (
        "contract_status",
        "FROZEN_FOR_DEVELOPMENT_BASELINE",
        contract.get("status"),
    ),
    (
        "text_layer_status",
        "PASSED",
        text_layer_manifest.get("status"),
    ),
    (
        "target_fields",
        EXPECTED_TARGET_FIELDS,
        len(contract_fields),
    ),
    (
        "scalar_fields",
        EXPECTED_SCALAR_FIELDS,
        len(scalar_contract_fields),
    ),
    (
        "repeating_fields",
        EXPECTED_REPEATING_FIELDS,
        len(repeating_contract_fields),
    ),
    (
        "text_layer_records",
        EXPECTED_DOCUMENTS,
        len(text_layer_manifest.get("records", [])),
    ),
]

input_gate_records = [
    {
        "control": name,
        "expected": expected,
        "actual": actual,
        "status": "VALID" if expected == actual else "INVALID",
    }
    for name, expected, actual in input_gate_values
]

invalid_input_gates = [
    row["control"]
    for row in input_gate_records
    if row["status"] != "VALID"
]
if invalid_input_gates:
    display(pd.DataFrame(input_gate_records))
    raise RuntimeError(
        "CELL 10C INPUT GATE FAILED. "
        f"Kontrol tidak valid: {invalid_input_gates}"
    )

source_records = sorted(
    text_layer_manifest["records"],
    key=lambda row: (
        int(row.get("sequence_number", 0)),
        str(row.get("document_id", "")),
    ),
)

source_checksums_before = {
    "contract": sha256_file(CONTRACT_PATH),
    "text_layer_manifest": sha256_file(TEXT_LAYER_MANIFEST_PATH),
}

for source_record in source_records:
    source_path = Path(source_record["text_layer_path"])
    require_file(source_path, "Text-layer checkpoint")
    actual_checksum = sha256_file(source_path)
    if actual_checksum != source_record["text_layer_sha256"]:
        raise RuntimeError(
            "Checksum text layer tidak cocok: "
            f"{source_record['document_id']}"
        )
    source_checksums_before[
        f"text_layer:{source_record['document_id']}"
    ] = actual_checksum


# ============================================================
# PARSER SIGNATURE
# ============================================================

parser_configuration = {
    "parser_id": PARSER_ID,
    "parser_version": PARSER_VERSION,
    "supported_currencies": sorted(SUPPORTED_CURRENCIES),
    "label_aliases": {
        key: sorted(value)
        for key, value in sorted(LABEL_ALIASES.items())
    },
    "month_dictionary": dict(sorted(MONTHS.items())),
    "table_row_tolerance_points": 2.8,
    "party_legal_suffixes": sorted(LEGAL_ENTITY_SUFFIXES),
    "party_suffix_geometry": {
        "same_row_tolerance_points": (
            PARTY_SUFFIX_SAME_ROW_TOLERANCE_POINTS
        ),
        "next_line_gap_points": PARTY_SUFFIX_NEXT_LINE_GAP_POINTS,
        "left_alignment_tolerance_points": (
            PARTY_SUFFIX_LEFT_ALIGNMENT_TOLERANCE_POINTS
        ),
        "same_row_gap_points": PARTY_SUFFIX_SAME_ROW_GAP_POINTS,
    },
    "ground_truth_as_prediction_input": False,
    "template_specific_branching": False,
}
parser_signature = hashlib.sha256(
    canonical_json(parser_configuration).encode("utf-8")
).hexdigest()


# ============================================================
# EXECUTE OR RECOVER PER-DOCUMENT CHECKPOINTS
# ============================================================

runtime_records = []
manifest_records = []
errors = []
newly_created = 0
recovered = 0

print("=" * 88)
print(f"CELL 10C — {PARSER_ID} — VERSION {PARSER_VERSION}")
print(f"Prediction root: {PREDICTION_ROOT}")
print("=" * 88)
print(f"Menjalankan parser baseline untuk {len(source_records)} dokumen...\n")

for sequence_number, source_record in enumerate(source_records, start=1):
    document_id = str(source_record["document_id"])
    template_id = str(source_record["template_id"])
    text_layer_path = Path(source_record["text_layer_path"])
    prediction_path = (
        PREDICTION_ROOT
        / template_id
        / f"{document_id}_prediction.json"
    )

    try:
        text_layer = load_json(text_layer_path)
        document = text_layer.get("document", {})

        if document.get("document_id") != document_id:
            raise RuntimeError("Document ID text layer tidak cocok.")
        if document.get("template_id") != template_id:
            raise RuntimeError("Template ID text layer tidak cocok.")
        if document.get("split") != "development":
            raise RuntimeError("Parser hanya boleh membuka development split.")

        parsed = parse_document(text_layer)
        scalars = parsed["scalar_fields"]
        items = parsed["items"]
        diagnostics = parsed["diagnostics"]

        prediction_artifact = {
            "schema_version": "1.0.0",
            "status": "EXECUTED",
            "quality_status": "PENDING_GROUND_TRUTH_EVALUATION",
            "parser": {
                "parser_id": PARSER_ID,
                "parser_version": PARSER_VERSION,
                "parser_signature_sha256": parser_signature,
                "approach": "DETERMINISTIC_LABEL_AND_GEOMETRY_RULES",
                "template_specific_branching": False,
                "ground_truth_used_as_prediction_input": False,
            },
            "document": {
                "canonical_invoice_id": document.get(
                    "canonical_invoice_id"
                ),
                "document_id": document_id,
                "template_id": template_id,
                "split": "development",
                "language": document.get("language"),
            },
            "predictions": {
                "scalar_fields": scalars,
                "items": items,
            },
            "diagnostics": diagnostics,
            "source": {
                "text_layer_path": str(text_layer_path),
                "text_layer_sha256": source_record[
                    "text_layer_sha256"
                ],
                "route_id": source_record.get("route_id"),
                "engine": source_record.get("engine"),
            },
            "integrity": {
                "ground_truth_loaded": False,
                "canonical_payload_loaded": False,
                "validation_opened": 0,
                "test_opened": 0,
                "dataset_modifications": 0,
                "source_modifications": 0,
            },
        }

        if prediction_path.exists():
            existing = load_json(prediction_path)
            if canonical_json(existing) != canonical_json(prediction_artifact):
                raise RuntimeError(
                    "Prediction checkpoint sudah ada tetapi berbeda. "
                    "Hapus hanya checkpoint parser v1 ini jika memang "
                    "ingin membangun ulang dengan aturan baru."
                )
            execution = "RECOVERED"
            recovered += 1
        else:
            atomic_write_json(prediction_path, prediction_artifact)
            execution = "NEW"
            newly_created += 1

        prediction_sha256 = sha256_file(prediction_path)
        populated_scalar_count = sum(
            prediction["normalized_value"] is not None
            for prediction in scalars.values()
        )
        party_suffix_fields = sorted(
            field_name
            for field_name, field_value in scalars.items()
            if str(field_value.get("method", "")).endswith(
                ":LEGAL_SUFFIX_CONTINUATION"
            )
        )
        missing_required = diagnostics[
            "missing_required_scalar_fields"
        ]
        financial_check = diagnostics["financial_equation"]

        manifest_record = {
            "sequence_number": sequence_number,
            "document_id": document_id,
            "template_id": template_id,
            "language": document.get("language"),
            "route_id": source_record.get("route_id"),
            "populated_scalar_fields": populated_scalar_count,
            "party_suffix_continuation_fields": party_suffix_fields,
            "party_suffix_continuation_count": len(party_suffix_fields),
            "missing_required_scalar_fields": missing_required,
            "parsed_item_count": len(items),
            "table_detected": diagnostics["table"]["table_detected"],
            "financial_equation_executed": financial_check["executed"],
            "financial_equation_passed": financial_check["passed"],
            "prediction_path": str(prediction_path),
            "prediction_sha256": prediction_sha256,
            "text_layer_path": str(text_layer_path),
            "text_layer_sha256": source_record["text_layer_sha256"],
            "status": "EXECUTED",
            "quality_status": "PENDING_GROUND_TRUTH_EVALUATION",
        }
        manifest_records.append(manifest_record)
        runtime_records.append({**manifest_record, "execution": execution})

        print(
            f"[{sequence_number:02d}/{len(source_records):02d}] "
            f"{document_id} | scalars={populated_scalar_count}/12 | "
            f"items={len(items)} | financial="
            f"{financial_check['passed']} | "
            f"suffix_merge={len(party_suffix_fields)} | {execution}"
        )

    except Exception as error:
        errors.append(
            {
                "sequence_number": sequence_number,
                "document_id": document_id,
                "template_id": template_id,
                "error_type": type(error).__name__,
                "error": str(error)[:700],
            }
        )
        print(
            f"[{sequence_number:02d}/{len(source_records):02d}] "
            f"{document_id} | ERROR: {type(error).__name__}: {error}"
        )


# ============================================================
# TECHNICAL AUDIT — QUALITY REMAINS PENDING UNTIL CELL 10D
# ============================================================

runtime_table = pd.DataFrame(runtime_records)
manifest_table = pd.DataFrame(manifest_records)

if errors:
    display(pd.DataFrame(errors))

template_counts = Counter(
    row["template_id"] for row in manifest_records
)
missing_field_counts = Counter()
for row in manifest_records:
    missing_field_counts.update(row["missing_required_scalar_fields"])

technical_controls = [
    ("prediction_records", EXPECTED_DOCUMENTS, len(manifest_records)),
    ("prediction_files", EXPECTED_DOCUMENTS, sum(
        Path(row["prediction_path"]).is_file()
        for row in manifest_records
    )),
    ("unique_document_ids", EXPECTED_DOCUMENTS, len({
        row["document_id"] for row in manifest_records
    })),
    ("template_count", EXPECTED_TEMPLATES, len(template_counts)),
    ("documents_with_complete_required_scalars", EXPECTED_DOCUMENTS, sum(
        not row["missing_required_scalar_fields"]
        for row in manifest_records
    )),
    ("documents_with_table_detected", EXPECTED_DOCUMENTS, sum(
        row["table_detected"] is True
        for row in manifest_records
    )),
    ("documents_with_items", EXPECTED_DOCUMENTS, sum(
        row["parsed_item_count"] > 0
        for row in manifest_records
    )),
    ("financial_equations_executed", EXPECTED_DOCUMENTS, sum(
        row["financial_equation_executed"] is True
        for row in manifest_records
    )),
    ("financial_equations_passed", EXPECTED_DOCUMENTS, sum(
        row["financial_equation_passed"] is True
        for row in manifest_records
    )),
    ("processing_errors", 0, len(errors)),
    ("ground_truth_opened", 0, 0),
    ("validation_opened", 0, 0),
    ("test_opened", 0, 0),
]

control_records = [
    {
        "control": name,
        "expected": expected,
        "actual": actual,
        "status": "VALID" if expected == actual else "INVALID",
    }
    for name, expected, actual in technical_controls
]

display(pd.DataFrame(control_records))

if not runtime_table.empty:
    display(
        runtime_table[
            [
                "sequence_number",
                "document_id",
                "template_id",
                "language",
                "populated_scalar_fields",
                "party_suffix_continuation_count",
                "parsed_item_count",
                "table_detected",
                "financial_equation_passed",
                "execution",
                "status",
                "quality_status",
            ]
        ]
    )

    suffix_runtime_table = runtime_table[
        runtime_table["party_suffix_continuation_count"] > 0
    ][
        [
            "document_id",
            "template_id",
            "language",
            "party_suffix_continuation_fields",
            "party_suffix_continuation_count",
        ]
    ]
    if not suffix_runtime_table.empty:
        print("\nPARTY LEGAL-SUFFIX CONTINUATIONS")
        display(suffix_runtime_table)

if missing_field_counts:
    display(
        pd.DataFrame(
            [
                {"field": field, "missing_documents": count}
                for field, count in sorted(missing_field_counts.items())
            ]
        )
    )

invalid_controls = [
    row["control"]
    for row in control_records
    if row["status"] != "VALID"
]
if invalid_controls:
    raise RuntimeError(
        "CELL 10C PARSER TECHNICAL AUDIT FAILED. "
        f"Kontrol tidak valid: {invalid_controls}. "
        "Jangan lanjut ke Cell 10D; kirim seluruh output ini."
    )


# ============================================================
# SAVE DETERMINISTIC SUMMARY AND MANIFEST
# ============================================================

summary_columns = [
    "sequence_number",
    "document_id",
    "template_id",
    "language",
    "route_id",
    "populated_scalar_fields",
    "party_suffix_continuation_count",
    "parsed_item_count",
    "table_detected",
    "financial_equation_executed",
    "financial_equation_passed",
    "prediction_path",
    "prediction_sha256",
    "text_layer_path",
    "text_layer_sha256",
    "status",
    "quality_status",
]

summary_text = manifest_table[summary_columns].to_csv(
    index=False,
    lineterminator="\n",
)

if SUMMARY_PATH.exists():
    if SUMMARY_PATH.read_text(encoding="utf-8") != summary_text:
        raise RuntimeError(
            "Summary parser v1 sudah ada tetapi berbeda."
        )
    summary_action = "RECOVERED"
else:
    atomic_write_text(SUMMARY_PATH, summary_text)
    summary_action = "CREATED"

baseline_manifest = {
    "schema_version": "1.0.0",
    "status": "EXECUTED",
    "quality_status": "PENDING_GROUND_TRUTH_EVALUATION",
    "stage": "RULE_BASED_FIELD_EXTRACTION_BASELINE",
    "parser": {
        **parser_configuration,
        "parser_signature_sha256": parser_signature,
    },
    "scope": {
        "split": "development",
        "documents": EXPECTED_DOCUMENTS,
        "templates": EXPECTED_TEMPLATES,
        "target_fields": EXPECTED_TARGET_FIELDS,
        "validation_opened": 0,
        "test_opened": 0,
    },
    "records": manifest_records,
    "technical_controls": control_records,
    "artifacts": {
        "prediction_root": str(PREDICTION_ROOT),
        "summary_path": str(SUMMARY_PATH),
        "summary_sha256": sha256_file(SUMMARY_PATH),
    },
    "input_artifacts": {
        "contract": {
            "path": str(CONTRACT_PATH),
            "sha256": source_checksums_before["contract"],
        },
        "text_layer_manifest": {
            "path": str(TEXT_LAYER_MANIFEST_PATH),
            "sha256": source_checksums_before[
                "text_layer_manifest"
            ],
        },
    },
    "integrity": {
        "ground_truth_loaded": False,
        "canonical_payload_loaded": False,
        "ground_truth_used_as_prediction_input": False,
        "dataset_modifications": 0,
        "source_modifications": 0,
        "validation_opened": 0,
        "test_opened": 0,
    },
    "next_stage": {
        "cell": "CELL 10D",
        "action": "DEVELOPMENT_GROUND_TRUTH_EVALUATION",
        "warning": (
            "Status kualitas parser belum PASSED. Ground truth baru "
            "boleh dibuka pada evaluasi Cell 10D setelah seluruh "
            "prediction checkpoint dibekukan."
        ),
    },
}

if MANIFEST_PATH.exists():
    existing_manifest = load_json(MANIFEST_PATH)
    if canonical_json(existing_manifest) != canonical_json(baseline_manifest):
        raise RuntimeError(
            "Manifest parser v1 sudah ada tetapi berbeda."
        )
    manifest_action = "RECOVERED"
else:
    atomic_write_json(MANIFEST_PATH, baseline_manifest)
    manifest_action = "CREATED"


# ============================================================
# POST-WRITE INPUT IMMUTABILITY CHECK
# ============================================================

source_checksums_after = {
    "contract": sha256_file(CONTRACT_PATH),
    "text_layer_manifest": sha256_file(TEXT_LAYER_MANIFEST_PATH),
}
for source_record in source_records:
    source_checksums_after[
        f"text_layer:{source_record['document_id']}"
    ] = sha256_file(Path(source_record["text_layer_path"]))

changed_sources = [
    name
    for name, checksum in source_checksums_before.items()
    if source_checksums_after.get(name) != checksum
]
if changed_sources:
    raise RuntimeError(
        f"Source artifact berubah selama parsing: {changed_sources}"
    )


print()
print(f"Parser ID              : {PARSER_ID}")
print(f"Parser version         : {PARSER_VERSION}")
print(f"Parser SHA-256         : {parser_signature}")
print(f"Documents              : {len(manifest_records)}")
print(f"Templates              : {len(template_counts)}")
print(
    "Party suffix merges    : "
    f"{sum(row['party_suffix_continuation_count'] for row in manifest_records)}"
)
print(f"New predictions        : {newly_created}")
print(f"Recovered predictions  : {recovered}")
print(f"Summary action         : {summary_action}")
print(f"Manifest action        : {manifest_action}")
print(f"Prediction root        : {PREDICTION_ROOT}")
print(f"Summary                : {SUMMARY_PATH}")
print(f"Manifest               : {MANIFEST_PATH}")
print(f"Manifest SHA-256       : {sha256_file(MANIFEST_PATH)}")
print("Ground truth opened    : 0")
print("Validation opened      : 0")
print("Test opened            : 0")
print("Dataset modifications  : 0")
print("Source modifications   : 0")
print("Quality status         : PENDING_GROUND_TRUTH_EVALUATION")
print()
print(
    "✅ CELL 10C PASSED — parser rule-based baseline telah "
    "dijalankan dan prediction checkpoint 18 dokumen development "
    "telah dibekukan. Kualitas belum dinyatakan PASSED; lanjutkan "
    "ke Cell 10D untuk evaluasi terhadap ground truth development."
)


CELL 10C — RULE-BASED-INVOICE-PARSER-V1 — VERSION 1.0.2
Prediction root: /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/field_extraction/predictions/rule_based_baseline_v1_0_2
Menjalankan parser baseline untuk 18 dokumen...

[01/18] INV-SYN-000002 | scalars=12/12 | items=2 | financial=True | suffix_merge=0 | NEW
[02/18] INV-SYN-000013 | scalars=12/12 | items=8 | financial=True | suffix_merge=0 | NEW
[03/18] INV-SYN-000015 | scalars=12/12 | items=8 | financial=True | suffix_merge=0 | NEW
[04/18] INV-SYN-000025 | scalars=12/12 | items=2 | financial=True | suffix_merge=0 | NEW
[05/18] INV-SYN-000036 | scalars=12/12 | items=7 | financial=True | suffix_merge=0 | NEW
[06/18] INV-SYN-000037 | scalars=12/12 | items=7 | financial=True | suffix_merge=0 | NEW
[07/18] INV-SYN-000043 | scalars=12/12 | items=2 | financial=True | suffix_merge=0 | NEW
[08/18] INV-SYN-000052 | scalars=12/12 | items=7 | financial=True | suffix_merge=0 | NEW
[09/18] I

,control,expected,actual,status
0,prediction_records,18,18,VALID
1,prediction_files,18,18,VALID
2,unique_document_ids,18,18,VALID
3,template_count,6,6,VALID
4,documents_with_complete_required_scalars,18,18,VALID
5,documents_with_table_detected,18,18,VALID
6,documents_with_items,18,18,VALID
7,financial_equations_executed,18,18,VALID
8,financial_equations_passed,18,18,VALID
9,processing_errors,0,0,VALID


,sequence_number,document_id,template_id,language,populated_scalar_fields,party_suffix_continuation_count,parsed_item_count,table_detected,financial_equation_passed,execution,status,quality_status
0,1,INV-SYN-000002,TPL-01,id,12,0,2,True,True,NEW,EXECUTED,PENDING_GROUND_TRUTH_EVALUATION
1,2,INV-SYN-000013,TPL-01,en,12,0,8,True,True,NEW,EXECUTED,PENDING_GROUND_TRUTH_EVALUATION
2,3,INV-SYN-000015,TPL-01,en,12,0,8,True,True,NEW,EXECUTED,PENDING_GROUND_TRUTH_EVALUATION
3,4,INV-SYN-000025,TPL-02,id,12,0,2,True,True,NEW,EXECUTED,PENDING_GROUND_TRUTH_EVALUATION
4,5,INV-SYN-000036,TPL-02,en,12,0,7,True,True,NEW,EXECUTED,PENDING_GROUND_TRUTH_EVALUATION
5,6,INV-SYN-000037,TPL-02,en,12,0,7,True,True,NEW,EXECUTED,PENDING_GROUND_TRUTH_EVALUATION
6,7,INV-SYN-000043,TPL-03,id,12,0,2,True,True,NEW,EXECUTED,PENDING_GROUND_TRUTH_EVALUATION
7,8,INV-SYN-000052,TPL-03,en,12,0,7,True,True,NEW,EXECUTED,PENDING_GROUND_TRUTH_EVALUATION
8,9,INV-SYN-000060,TPL-03,en,12,0,7,True,True,NEW,EXECUTED,PENDING_GROUND_TRUTH_EVALUATION
9,10,INV-SYN-000064,TPL-04,id,12,0,8,True,True,NEW,EXECUTED,PENDING_GROUND_TRUTH_EVALUATION



PARTY LEGAL-SUFFIX CONTINUATIONS


,document_id,template_id,language,party_suffix_continuation_fields,party_suffix_continuation_count
11,INV-SYN-000071,TPL-04,en,[vendor.name],1



Parser ID              : RULE-BASED-INVOICE-PARSER-V1
Parser version         : 1.0.2
Parser SHA-256         : ffe04f258fb3b1a6a55661698fa6c7dda29ddd7b714c2f210b3e6fc1fcd464b3
Documents              : 18
Templates              : 6
Party suffix merges    : 1
New predictions        : 18
Recovered predictions  : 0
Summary action         : CREATED
Manifest action        : CREATED
Prediction root        : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/field_extraction/predictions/rule_based_baseline_v1_0_2
Summary                : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/field_extraction/rule_based_baseline_v1_0_2_summary.csv
Manifest               : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/field_extraction/rule_based_baseline_v1_0_2_manifest.json
Manifest SHA-256       : aaf0ab1102abcee57b1e5735264a46cc5baaef0395daf4a9

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import re
import unicodedata
from collections import Counter, defaultdict
from decimal import Decimal, InvalidOperation
from functools import lru_cache
from pathlib import Path

import pandas as pd
from IPython.display import display


# ============================================================
# CELL 10D — DEVELOPMENT GROUND-TRUTH EVALUATION
# ============================================================

BUILD_ROOT = Path(
    "/content/drive/MyDrive/InvoiceFlow-AI-Data/interim/"
    "synthetic_v1_build/20260904T150025Z"
)
BENCHMARK_ROOT = BUILD_ROOT / "ocr_benchmark"
FIELD_ROOT = BENCHMARK_ROOT / "field_extraction"
GROUND_TRUTH_ROOT = BUILD_ROOT / "rendered_dataset" / "ground_truth"

CONTRACT_PATH = FIELD_ROOT / "field_extraction_contract_v1.json"
PREDICTION_MANIFEST_PATH = (
    FIELD_ROOT / "rule_based_baseline_v1_0_2_manifest.json"
)

EVALUATION_ROOT = (
    FIELD_ROOT
    / "evaluations"
    / "rule_based_baseline_v1_0_2_development_eval_v1_0_2"
)
DOCUMENT_EVALUATION_ROOT = EVALUATION_ROOT / "documents"
DOCUMENT_SUMMARY_PATH = EVALUATION_ROOT / "document_summary.csv"
FIELD_SUMMARY_PATH = EVALUATION_ROOT / "field_summary.csv"
TEMPLATE_SUMMARY_PATH = EVALUATION_ROOT / "template_summary.csv"
MISMATCH_DETAIL_PATH = EVALUATION_ROOT / "mismatch_details.csv"
EVALUATION_MANIFEST_PATH = (
    EVALUATION_ROOT / "development_evaluation_manifest.json"
)

EVALUATOR_ID = "INVOICE-FIELD-EVALUATOR-V1"
EVALUATOR_VERSION = "1.0.2"
EXPECTED_PARSER_ID = "RULE-BASED-INVOICE-PARSER-V1"
EXPECTED_PARSER_VERSION = "1.0.2"
EXPECTED_DOCUMENTS = 18
EXPECTED_TEMPLATES = 6
EXPECTED_SCALAR_FIELDS = 12
EXPECTED_ITEM_FIELDS = 4

ITEM_FIELD_NAMES = [
    "description",
    "quantity",
    "unit_price",
    "line_total",
]

SCALAR_PATHS = {
    "invoice_number": ("invoice_number",),
    "invoice_date": ("invoice_date",),
    "due_date": ("due_date",),
    "currency": ("currency",),
    "vendor.name": ("vendor", "name"),
    "vendor.tax_identifier": ("vendor", "tax_identifier"),
    "buyer.name": ("buyer", "name"),
    "buyer.tax_identifier": ("buyer", "tax_identifier"),
    "financials.subtotal": ("financials", "subtotal"),
    "financials.tax": ("financials", "tax"),
    "financials.discount": ("financials", "discount"),
    "financials.total": ("financials", "total"),
}

MONEY_FIELDS = {
    "financials.subtotal",
    "financials.tax",
    "financials.discount",
    "financials.total",
    "items[].unit_price",
    "items[].line_total",
}

IDENTIFIER_FIELDS = {
    "invoice_number",
    "currency",
    "vendor.tax_identifier",
    "buyer.tax_identifier",
}


# ============================================================
# FILE AND SERIALIZATION HELPERS
# ============================================================

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def load_json(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as handle:
        value = json.load(handle)
    if not isinstance(value, dict):
        raise TypeError(f"JSON harus berupa object: {path}")
    return value


def canonical_json(value) -> str:
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    )


def atomic_write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(path.name + ".tmp")
    temporary_path.write_text(text, encoding="utf-8")
    os.replace(temporary_path, path)


def atomic_write_json(path: Path, value: dict) -> None:
    atomic_write_text(
        path,
        json.dumps(
            value,
            indent=2,
            ensure_ascii=False,
            default=str,
        )
        + "\n",
    )


def require_file(path: Path, label: str) -> None:
    if not path.is_file():
        raise FileNotFoundError(f"{label} tidak ditemukan: {path}")


def require_directory(path: Path, label: str) -> None:
    if not path.is_dir():
        raise FileNotFoundError(f"{label} tidak ditemukan: {path}")


def nested_value(record: dict, path: tuple[str, ...]):
    value = record
    for key in path:
        if not isinstance(value, dict) or key not in value:
            return None
        value = value[key]
    return value


# ============================================================
# TYPE-AWARE NORMALIZATION
# ============================================================

def normalize_spaces(value) -> str:
    if value is None:
        return ""
    text = unicodedata.normalize("NFKC", str(value))
    return re.sub(r"\s+", " ", text).strip()


def decimal_to_string(value: Decimal) -> str:
    if value == value.to_integral():
        return str(value.quantize(Decimal("1")))
    text = format(value.normalize(), "f")
    return text.rstrip("0").rstrip(".")


def normalize_decimal(value) -> str:
    text = normalize_spaces(value)
    if not text:
        return ""
    try:
        return decimal_to_string(Decimal(text))
    except InvalidOperation:
        return text


def normalize_field_value(field_name: str, value) -> str:
    text = normalize_spaces(value)

    if field_name in MONEY_FIELDS or field_name == "items[].quantity":
        return normalize_decimal(text)
    if field_name in IDENTIFIER_FIELDS:
        return text.upper()
    return text


# ============================================================
# EDIT-DISTANCE METRICS
# ============================================================

def levenshtein_distance(reference, hypothesis) -> int:
    reference = list(reference)
    hypothesis = list(hypothesis)

    if len(reference) < len(hypothesis):
        reference, hypothesis = hypothesis, reference

    previous = list(range(len(hypothesis) + 1))
    for reference_index, reference_value in enumerate(reference, start=1):
        current = [reference_index]
        for hypothesis_index, hypothesis_value in enumerate(
            hypothesis,
            start=1,
        ):
            insertion = current[hypothesis_index - 1] + 1
            deletion = previous[hypothesis_index] + 1
            substitution = (
                previous[hypothesis_index - 1]
                + int(reference_value != hypothesis_value)
            )
            current.append(min(insertion, deletion, substitution))
        previous = current
    return previous[-1]


def safe_ratio(numerator: int | float, denominator: int | float) -> float:
    if denominator == 0:
        return 1.0 if numerator == 0 else 0.0
    return float(numerator) / float(denominator)


def safe_error_rate(
    errors: int | float,
    reference_units: int | float,
) -> float:
    if reference_units == 0:
        return 0.0 if errors == 0 else 1.0
    return float(errors) / float(reference_units)


def harmonic_mean(precision: float, recall: float) -> float:
    if precision + recall == 0:
        return 0.0
    return 2.0 * precision * recall / (precision + recall)


def string_similarity(first: str, second: str) -> float:
    first = normalize_spaces(first).casefold()
    second = normalize_spaces(second).casefold()
    denominator = max(len(first), len(second), 1)
    return 1.0 - levenshtein_distance(first, second) / denominator


# ============================================================
# GROUND-TRUTH DISCOVERY — ONLY FROZEN DEVELOPMENT IDS
# ============================================================

def locate_ground_truth(document_id: str, template_id: str) -> Path:
    candidates = []

    for candidate in GROUND_TRUTH_ROOT.rglob(f"*{document_id}*.json"):
        if candidate.is_file():
            candidates.append(candidate.resolve())

    candidates = sorted(set(candidates), key=lambda path: path.as_posix())

    if len(candidates) == 1:
        return candidates[0]

    identity_matches = []
    for candidate in candidates:
        candidate_payload = load_json(candidate)
        candidate_document = candidate_payload.get("document", {})
        if (
            isinstance(candidate_document, dict)
            and candidate_document.get("document_id") == document_id
            and candidate_document.get("template_id") == template_id
            and candidate_document.get("split") == "development"
        ):
            identity_matches.append(candidate)

    if len(identity_matches) != 1:
        raise RuntimeError(
            f"Ground truth {document_id}/{template_id} ditemukan "
            f"{len(identity_matches)} kali setelah verifikasi identitas; "
            f"kandidat berdasarkan document_id={len(candidates)}. "
            "Seharusnya tepat satu."
        )
    return identity_matches[0]


# ============================================================
# CANONICAL GROUND-TRUTH ADAPTER
# ============================================================

def canonical_from_ground_truth(
    ground_truth: dict,
    document_id: str,
    template_id: str,
) -> dict:
    document = ground_truth.get("document", {})
    canonical = ground_truth.get("canonical")

    if not isinstance(document, dict):
        raise TypeError("ground_truth.document harus berupa object.")
    if not isinstance(canonical, dict):
        raise TypeError("ground_truth.canonical harus berupa object.")
    if document.get("document_id") != document_id:
        raise RuntimeError("Document ID ground truth tidak cocok.")
    if document.get("template_id") != template_id:
        raise RuntimeError("Template ID ground truth tidak cocok.")
    if document.get("split") != "development":
        raise RuntimeError(
            "Evaluasi Cell 10D hanya boleh membuka development split."
        )
    if canonical.get("document_id") != document_id:
        raise RuntimeError("Document ID canonical tidak cocok.")
    if canonical.get("template_id") != template_id:
        raise RuntimeError("Template ID canonical tidak cocok.")
    if canonical.get("split") != "development":
        raise RuntimeError("Canonical payload bukan development split.")

    items = canonical.get("items")
    if not isinstance(items, list) or not items:
        raise RuntimeError("Canonical items kosong atau tidak valid.")
    if not all(isinstance(item, dict) for item in items):
        raise TypeError("Setiap canonical item harus berupa object.")

    return canonical


def expected_scalars(canonical: dict) -> dict[str, str]:
    return {
        field_name: normalize_field_value(
            field_name,
            nested_value(canonical, path),
        )
        for field_name, path in SCALAR_PATHS.items()
    }


def expected_items(canonical: dict) -> list[dict[str, str]]:
    normalized_items = []
    for item in canonical["items"]:
        normalized_items.append(
            {
                field_name: normalize_field_value(
                    f"items[].{field_name}",
                    item.get(field_name),
                )
                for field_name in ITEM_FIELD_NAMES
            }
        )
    return normalized_items


def predicted_scalars(prediction: dict) -> dict[str, str]:
    scalar_predictions = (
        prediction.get("predictions", {}).get("scalar_fields", {})
    )
    if not isinstance(scalar_predictions, dict):
        raise TypeError("predictions.scalar_fields harus berupa object.")

    values = {}
    for field_name in SCALAR_PATHS:
        field_prediction = scalar_predictions.get(field_name, {})
        if not isinstance(field_prediction, dict):
            field_prediction = {}
        values[field_name] = normalize_field_value(
            field_name,
            field_prediction.get("normalized_value"),
        )
    return values


def predicted_items(prediction: dict) -> list[dict[str, str]]:
    item_predictions = prediction.get("predictions", {}).get("items", [])
    if not isinstance(item_predictions, list):
        raise TypeError("predictions.items harus berupa list.")

    normalized_items = []
    for item in item_predictions:
        if not isinstance(item, dict):
            raise TypeError("Setiap prediction item harus berupa object.")

        normalized_item = {}
        for field_name in ITEM_FIELD_NAMES:
            field_prediction = item.get(field_name, {})
            if not isinstance(field_prediction, dict):
                field_prediction = {}
            normalized_item[field_name] = normalize_field_value(
                f"items[].{field_name}",
                field_prediction.get("normalized_value"),
            )
        normalized_items.append(normalized_item)
    return normalized_items


# ============================================================
# MAXIMUM-WEIGHT BIPARTITE ITEM ALIGNMENT
# ============================================================

def item_pair_weight(predicted: dict, expected: dict) -> int:
    description_similarity = string_similarity(
        predicted.get("description", ""),
        expected.get("description", ""),
    )
    line_total_exact = (
        predicted.get("line_total", "")
        == expected.get("line_total", "")
    )
    quantity_exact = (
        predicted.get("quantity", "")
        == expected.get("quantity", "")
    )
    unit_price_exact = (
        predicted.get("unit_price", "")
        == expected.get("unit_price", "")
    )

    return int(round(description_similarity * 10000)) + (
        10000 if line_total_exact else 0
    ) + (100 if quantity_exact else 0) + (100 if unit_price_exact else 0)


def maximum_weight_item_alignment(
    predictions: list[dict],
    references: list[dict],
) -> list[tuple[int | None, int | None]]:
    size = max(len(predictions), len(references))
    if size == 0:
        return []

    weights = []
    for prediction_index in range(size):
        row = []
        for reference_index in range(size):
            if (
                prediction_index < len(predictions)
                and reference_index < len(references)
            ):
                row.append(
                    item_pair_weight(
                        predictions[prediction_index],
                        references[reference_index],
                    )
                )
            else:
                row.append(0)
        weights.append(row)

    @lru_cache(maxsize=None)
    def solve(
        prediction_index: int,
        used_reference_mask: int,
    ) -> tuple[int, tuple[int, ...]]:
        if prediction_index == size:
            return 0, ()

        best_score = -1
        best_assignment = ()
        for reference_index in range(size):
            bit = 1 << reference_index
            if used_reference_mask & bit:
                continue

            remaining_score, remaining_assignment = solve(
                prediction_index + 1,
                used_reference_mask | bit,
            )
            score = weights[prediction_index][reference_index] + remaining_score
            assignment = (reference_index,) + remaining_assignment

            if score > best_score or (
                score == best_score and assignment < best_assignment
            ):
                best_score = score
                best_assignment = assignment

        return best_score, best_assignment

    _, assignment = solve(0, 0)
    aligned_pairs = []
    for prediction_index, reference_index in enumerate(assignment):
        real_prediction = (
            prediction_index if prediction_index < len(predictions) else None
        )
        real_reference = (
            reference_index if reference_index < len(references) else None
        )
        if real_prediction is not None or real_reference is not None:
            aligned_pairs.append((real_prediction, real_reference))
    return aligned_pairs


# ============================================================
# DOCUMENT EVALUATION
# ============================================================

def evaluate_document(
    prediction: dict,
    canonical: dict,
    contract_field_map: dict[str, dict],
) -> dict:
    document = prediction.get("document", {})
    document_id = str(document["document_id"])
    template_id = str(document["template_id"])
    language = str(document.get("language", ""))

    scalar_expected = expected_scalars(canonical)
    scalar_predicted = predicted_scalars(prediction)
    scalar_records = []

    for field_name in SCALAR_PATHS:
        expected_value = scalar_expected[field_name]
        predicted_value = scalar_predicted[field_name]
        exact_match = predicted_value == expected_value
        character_errors = levenshtein_distance(
            expected_value,
            predicted_value,
        )
        word_errors = levenshtein_distance(
            expected_value.split(),
            predicted_value.split(),
        )

        scalar_records.append(
            {
                "document_id": document_id,
                "template_id": template_id,
                "language": language,
                "group": "scalar",
                "field": field_name,
                "critical": bool(
                    contract_field_map[field_name].get("critical")
                ),
                "expected": expected_value,
                "predicted": predicted_value,
                "exact_match": exact_match,
                "character_errors": character_errors,
                "reference_characters": len(expected_value),
                "word_errors": word_errors,
                "reference_words": len(expected_value.split()),
            }
        )

    item_expected = expected_items(canonical)
    item_predicted = predicted_items(prediction)
    alignment = maximum_weight_item_alignment(
        item_predicted,
        item_expected,
    )

    item_field_records = []
    item_pair_records = []

    for pair_number, (prediction_index, reference_index) in enumerate(
        alignment,
        start=1,
    ):
        predicted_item = (
            item_predicted[prediction_index]
            if prediction_index is not None
            else None
        )
        expected_item = (
            item_expected[reference_index]
            if reference_index is not None
            else None
        )

        description_exact = bool(
            predicted_item is not None
            and expected_item is not None
            and predicted_item["description"] == expected_item["description"]
        )
        line_total_exact = bool(
            predicted_item is not None
            and expected_item is not None
            and predicted_item["line_total"] == expected_item["line_total"]
        )
        accepted_row_match = description_exact and line_total_exact

        item_pair_records.append(
            {
                "pair_number": pair_number,
                "prediction_row": (
                    prediction_index + 1
                    if prediction_index is not None
                    else None
                ),
                "reference_row": (
                    reference_index + 1
                    if reference_index is not None
                    else None
                ),
                "description_exact": description_exact,
                "line_total_exact": line_total_exact,
                "accepted_row_match": accepted_row_match,
            }
        )

        for field_name in ITEM_FIELD_NAMES:
            full_field_name = f"items[].{field_name}"
            expected_value = (
                expected_item[field_name]
                if expected_item is not None
                else ""
            )
            predicted_value = (
                predicted_item[field_name]
                if predicted_item is not None
                else ""
            )
            exact_match = bool(
                predicted_item is not None
                and expected_item is not None
                and predicted_value == expected_value
            )

            item_field_records.append(
                {
                    "document_id": document_id,
                    "template_id": template_id,
                    "language": language,
                    "group": "item",
                    "field": full_field_name,
                    "critical": bool(
                        contract_field_map[full_field_name].get("critical")
                    ),
                    "prediction_row": (
                        prediction_index + 1
                        if prediction_index is not None
                        else None
                    ),
                    "reference_row": (
                        reference_index + 1
                        if reference_index is not None
                        else None
                    ),
                    "expected": expected_value,
                    "predicted": predicted_value,
                    "exact_match": exact_match,
                    "character_errors": levenshtein_distance(
                        expected_value,
                        predicted_value,
                    ),
                    "reference_characters": len(expected_value),
                    "word_errors": levenshtein_distance(
                        expected_value.split(),
                        predicted_value.split(),
                    ),
                    "reference_words": len(expected_value.split()),
                }
            )

    row_true_positives = sum(
        record["accepted_row_match"] for record in item_pair_records
    )
    row_precision = safe_ratio(row_true_positives, len(item_predicted))
    row_recall = safe_ratio(row_true_positives, len(item_expected))
    row_f1 = harmonic_mean(row_precision, row_recall)

    item_exact_fields = sum(
        record["exact_match"] for record in item_field_records
    )
    predicted_item_fields = len(item_predicted) * EXPECTED_ITEM_FIELDS
    reference_item_fields = len(item_expected) * EXPECTED_ITEM_FIELDS
    item_field_precision = safe_ratio(
        item_exact_fields,
        predicted_item_fields,
    )
    item_field_recall = safe_ratio(
        item_exact_fields,
        reference_item_fields,
    )
    item_field_f1 = harmonic_mean(
        item_field_precision,
        item_field_recall,
    )

    scalar_exact_count = sum(
        record["exact_match"] for record in scalar_records
    )
    document_exact = bool(
        scalar_exact_count == EXPECTED_SCALAR_FIELDS
        and len(item_predicted) == len(item_expected)
        and item_exact_fields == reference_item_fields
    )

    financial_check = prediction.get("diagnostics", {}).get(
        "financial_equation",
        {},
    )
    financial_consistent = bool(
        financial_check.get("executed") is True
        and financial_check.get("passed") is True
    )

    return {
        "document_id": document_id,
        "template_id": template_id,
        "language": language,
        "scalar_records": scalar_records,
        "item_field_records": item_field_records,
        "item_alignment": item_pair_records,
        "metrics": {
            "scalar_exact_count": scalar_exact_count,
            "scalar_field_count": EXPECTED_SCALAR_FIELDS,
            "scalar_exact_match": safe_ratio(
                scalar_exact_count,
                EXPECTED_SCALAR_FIELDS,
            ),
            "predicted_item_count": len(item_predicted),
            "reference_item_count": len(item_expected),
            "item_count_exact": len(item_predicted) == len(item_expected),
            "row_true_positives": row_true_positives,
            "row_precision": row_precision,
            "row_recall": row_recall,
            "row_f1": row_f1,
            "item_exact_fields": item_exact_fields,
            "predicted_item_fields": predicted_item_fields,
            "reference_item_fields": reference_item_fields,
            "item_field_precision": item_field_precision,
            "item_field_recall": item_field_recall,
            "item_field_f1": item_field_f1,
            "financial_consistent": financial_consistent,
            "document_exact_match": document_exact,
        },
    }


# ============================================================
# PREFLIGHT AND FROZEN-PREDICTION GATE
# ============================================================

require_file(CONTRACT_PATH, "Field extraction contract")
require_file(PREDICTION_MANIFEST_PATH, "Prediction manifest v1.0.2")
require_directory(GROUND_TRUTH_ROOT, "Ground-truth root")

contract = load_json(CONTRACT_PATH)
prediction_manifest = load_json(PREDICTION_MANIFEST_PATH)

field_definitions = contract.get("field_definitions", [])
contract_field_map = {
    field["field"]: field
    for field in field_definitions
    if isinstance(field, dict) and field.get("field")
}
critical_scalar_fields = {
    field_name
    for field_name in SCALAR_PATHS
    if contract_field_map.get(field_name, {}).get("critical") is True
}

prediction_parser = prediction_manifest.get("parser", {})
prediction_records = prediction_manifest.get("records", [])

preflight_values = [
    (
        "contract_status",
        "FROZEN_FOR_DEVELOPMENT_BASELINE",
        contract.get("status"),
    ),
    (
        "prediction_manifest_status",
        "EXECUTED",
        prediction_manifest.get("status"),
    ),
    (
        "prediction_quality_before_evaluation",
        "PENDING_GROUND_TRUTH_EVALUATION",
        prediction_manifest.get("quality_status"),
    ),
    (
        "parser_id",
        EXPECTED_PARSER_ID,
        prediction_parser.get("parser_id"),
    ),
    (
        "parser_version",
        EXPECTED_PARSER_VERSION,
        prediction_parser.get("parser_version"),
    ),
    ("prediction_records", EXPECTED_DOCUMENTS, len(prediction_records)),
    ("target_fields", 16, len(contract_field_map)),
    ("scalar_fields", EXPECTED_SCALAR_FIELDS, len(SCALAR_PATHS)),
    ("item_fields", EXPECTED_ITEM_FIELDS, len(ITEM_FIELD_NAMES)),
    (
        "prediction_document_ids_unique",
        EXPECTED_DOCUMENTS,
        len({record.get("document_id") for record in prediction_records}),
    ),
]

preflight_records = [
    {
        "control": name,
        "expected": expected,
        "actual": actual,
        "status": "VALID" if expected == actual else "INVALID",
    }
    for name, expected, actual in preflight_values
]

invalid_preflight = [
    record["control"]
    for record in preflight_records
    if record["status"] != "VALID"
]
if invalid_preflight:
    display(pd.DataFrame(preflight_records))
    raise RuntimeError(
        "CELL 10D PREFLIGHT FAILED. "
        f"Kontrol tidak valid: {invalid_preflight}"
    )

expected_document_ids = {
    record["document_id"]
    for record in contract.get("benchmark_records", [])
}
prediction_document_ids = {
    record["document_id"] for record in prediction_records
}
if expected_document_ids != prediction_document_ids:
    raise RuntimeError(
        "Document ID prediction tidak sama dengan benchmark yang dibekukan."
    )

input_checksums_before = {
    "contract": sha256_file(CONTRACT_PATH),
    "prediction_manifest": sha256_file(PREDICTION_MANIFEST_PATH),
}

for record in prediction_records:
    prediction_path = Path(record["prediction_path"])
    require_file(prediction_path, "Prediction checkpoint")
    checksum = sha256_file(prediction_path)
    if checksum != record["prediction_sha256"]:
        raise RuntimeError(
            f"Checksum prediction tidak cocok: {record['document_id']}"
        )
    input_checksums_before[
        f"prediction:{record['document_id']}"
    ] = checksum


# ============================================================
# OPEN ONLY 18 DEVELOPMENT GROUND-TRUTH FILES AND EVALUATE
# ============================================================

runtime_records = []
document_manifest_records = []
all_scalar_records = []
all_item_field_records = []
errors = []
ground_truth_paths = {}
ground_truth_checksums = {}
new_evaluations = 0
recovered_evaluations = 0

print("=" * 88)
print(
    f"CELL 10D — {EVALUATOR_ID} — VERSION {EVALUATOR_VERSION}"
)
print(f"Evaluation root: {EVALUATION_ROOT}")
print("Scope: 18 frozen DEVELOPMENT documents only")
print("Validation opened: 0 | Test opened: 0")
print("=" * 88)
print(f"Mengevaluasi {len(prediction_records)} dokumen...\n")

for sequence_number, prediction_record in enumerate(
    sorted(
        prediction_records,
        key=lambda row: (
            int(row.get("sequence_number", 0)),
            str(row.get("document_id", "")),
        ),
    ),
    start=1,
):
    document_id = str(prediction_record["document_id"])
    template_id = str(prediction_record["template_id"])
    prediction_path = Path(prediction_record["prediction_path"])
    evaluation_path = (
        DOCUMENT_EVALUATION_ROOT
        / template_id
        / f"{document_id}_evaluation.json"
    )

    try:
        prediction = load_json(prediction_path)
        prediction_document = prediction.get("document", {})
        if prediction_document.get("document_id") != document_id:
            raise RuntimeError("Prediction document ID tidak cocok.")
        if prediction_document.get("template_id") != template_id:
            raise RuntimeError("Prediction template ID tidak cocok.")
        if prediction_document.get("split") != "development":
            raise RuntimeError("Prediction bukan development split.")

        ground_truth_path = locate_ground_truth(document_id, template_id)
        ground_truth = load_json(ground_truth_path)
        canonical = canonical_from_ground_truth(
            ground_truth,
            document_id,
            template_id,
        )
        ground_truth_checksum = sha256_file(ground_truth_path)
        ground_truth_paths[document_id] = ground_truth_path
        ground_truth_checksums[document_id] = ground_truth_checksum

        result = evaluate_document(
            prediction,
            canonical,
            contract_field_map,
        )
        metrics = result["metrics"]

        evaluation_artifact = {
            "schema_version": "1.0.0",
            "status": "EVALUATED",
            "evaluator": {
                "evaluator_id": EVALUATOR_ID,
                "evaluator_version": EVALUATOR_VERSION,
                "matching_policy": contract["evaluation_policy"],
            },
            "document": {
                "document_id": document_id,
                "template_id": template_id,
                "split": "development",
                "language": result["language"],
            },
            "metrics": metrics,
            "scalar_comparisons": result["scalar_records"],
            "item_alignment": result["item_alignment"],
            "item_field_comparisons": result["item_field_records"],
            "inputs": {
                "prediction_path": str(prediction_path),
                "prediction_sha256": prediction_record[
                    "prediction_sha256"
                ],
                "ground_truth_path": str(ground_truth_path),
                "ground_truth_sha256": ground_truth_checksum,
            },
            "integrity": {
                "ground_truth_split": "development",
                "validation_opened": 0,
                "test_opened": 0,
                "prediction_modified": False,
                "ground_truth_modified": False,
            },
        }

        if evaluation_path.exists():
            existing = load_json(evaluation_path)
            if canonical_json(existing) != canonical_json(evaluation_artifact):
                raise RuntimeError(
                    "Evaluation checkpoint sudah ada tetapi berbeda."
                )
            execution = "RECOVERED"
            recovered_evaluations += 1
        else:
            atomic_write_json(evaluation_path, evaluation_artifact)
            execution = "NEW"
            new_evaluations += 1

        evaluation_sha256 = sha256_file(evaluation_path)
        record = {
            "sequence_number": sequence_number,
            "document_id": document_id,
            "template_id": template_id,
            "language": result["language"],
            "scalar_exact_match": metrics["scalar_exact_match"],
            "predicted_item_count": metrics["predicted_item_count"],
            "reference_item_count": metrics["reference_item_count"],
            "item_count_exact": metrics["item_count_exact"],
            "row_precision": metrics["row_precision"],
            "row_recall": metrics["row_recall"],
            "row_f1": metrics["row_f1"],
            "item_field_precision": metrics["item_field_precision"],
            "item_field_recall": metrics["item_field_recall"],
            "item_field_f1": metrics["item_field_f1"],
            "financial_consistent": metrics["financial_consistent"],
            "document_exact_match": metrics["document_exact_match"],
            "evaluation_path": str(evaluation_path),
            "evaluation_sha256": evaluation_sha256,
            "prediction_path": str(prediction_path),
            "prediction_sha256": prediction_record[
                "prediction_sha256"
            ],
            "ground_truth_path": str(ground_truth_path),
            "ground_truth_sha256": ground_truth_checksum,
            "status": "EVALUATED",
        }
        document_manifest_records.append(record)
        runtime_records.append({**record, "execution": execution})
        all_scalar_records.extend(result["scalar_records"])
        all_item_field_records.extend(result["item_field_records"])

        print(
            f"[{sequence_number:02d}/{len(prediction_records):02d}] "
            f"{document_id} | scalar={metrics['scalar_exact_match']:.4f} | "
            f"items={metrics['predicted_item_count']}/"
            f"{metrics['reference_item_count']} | "
            f"item_f1={metrics['item_field_f1']:.4f} | "
            f"exact={metrics['document_exact_match']} | {execution}"
        )

    except Exception as error:
        errors.append(
            {
                "sequence_number": sequence_number,
                "document_id": document_id,
                "template_id": template_id,
                "error_type": type(error).__name__,
                "error": str(error)[:700],
            }
        )
        print(
            f"[{sequence_number:02d}/{len(prediction_records):02d}] "
            f"{document_id} | ERROR: {type(error).__name__}: {error}"
        )


# ============================================================
# TECHNICAL COMPLETENESS GATE
# ============================================================

if errors:
    display(pd.DataFrame(errors))

technical_values = [
    ("evaluated_documents", EXPECTED_DOCUMENTS, len(document_manifest_records)),
    ("evaluation_files", EXPECTED_DOCUMENTS, sum(
        Path(record["evaluation_path"]).is_file()
        for record in document_manifest_records
    )),
    ("unique_document_ids", EXPECTED_DOCUMENTS, len({
        record["document_id"] for record in document_manifest_records
    })),
    ("templates_evaluated", EXPECTED_TEMPLATES, len({
        record["template_id"] for record in document_manifest_records
    })),
    ("development_ground_truth_opened", EXPECTED_DOCUMENTS, len(
        ground_truth_paths
    )),
    ("validation_opened", 0, 0),
    ("test_opened", 0, 0),
    ("evaluation_errors", 0, len(errors)),
]

technical_control_records = [
    {
        "control": name,
        "expected": expected,
        "actual": actual,
        "status": "VALID" if expected == actual else "INVALID",
    }
    for name, expected, actual in technical_values
]

display(pd.DataFrame(technical_control_records))

invalid_technical_controls = [
    record["control"]
    for record in technical_control_records
    if record["status"] != "VALID"
]
if invalid_technical_controls:
    raise RuntimeError(
        "CELL 10D TECHNICAL EVALUATION FAILED. "
        f"Kontrol tidak valid: {invalid_technical_controls}"
    )


# ============================================================
# AGGREGATE METRICS
# ============================================================

scalar_total = len(all_scalar_records)
scalar_exact = sum(record["exact_match"] for record in all_scalar_records)
critical_scalar_records = [
    record for record in all_scalar_records if record["critical"]
]
critical_scalar_exact = sum(
    record["exact_match"] for record in critical_scalar_records
)

all_scalar_exact_match = safe_ratio(scalar_exact, scalar_total)
critical_scalar_exact_match = safe_ratio(
    critical_scalar_exact,
    len(critical_scalar_records),
)

scalar_character_errors = sum(
    record["character_errors"] for record in all_scalar_records
)
scalar_reference_characters = sum(
    record["reference_characters"] for record in all_scalar_records
)
scalar_word_errors = sum(
    record["word_errors"] for record in all_scalar_records
)
scalar_reference_words = sum(
    record["reference_words"] for record in all_scalar_records
)
scalar_micro_cer = safe_error_rate(
    scalar_character_errors,
    scalar_reference_characters,
)
scalar_micro_wer = safe_error_rate(
    scalar_word_errors,
    scalar_reference_words,
)

item_exact_fields = sum(
    record["exact_match"] for record in all_item_field_records
)
predicted_item_field_count = sum(
    record["predicted_item_count"] * EXPECTED_ITEM_FIELDS
    for record in document_manifest_records
)
reference_item_field_count = sum(
    record["reference_item_count"] * EXPECTED_ITEM_FIELDS
    for record in document_manifest_records
)
line_item_field_precision = safe_ratio(
    item_exact_fields,
    predicted_item_field_count,
)
line_item_field_recall = safe_ratio(
    item_exact_fields,
    reference_item_field_count,
)
line_item_field_f1 = harmonic_mean(
    line_item_field_precision,
    line_item_field_recall,
)

row_true_positives = 0
predicted_rows = 0
reference_rows = 0
for record in document_manifest_records:
    evaluation = load_json(Path(record["evaluation_path"]))
    metrics = evaluation["metrics"]
    row_true_positives += int(metrics["row_true_positives"])
    predicted_rows += int(metrics["predicted_item_count"])
    reference_rows += int(metrics["reference_item_count"])

row_precision = safe_ratio(row_true_positives, predicted_rows)
row_recall = safe_ratio(row_true_positives, reference_rows)
row_f1 = harmonic_mean(row_precision, row_recall)

financial_consistency = safe_ratio(
    sum(
        record["financial_consistent"]
        for record in document_manifest_records
    ),
    len(document_manifest_records),
)
document_exact_match = safe_ratio(
    sum(
        record["document_exact_match"]
        for record in document_manifest_records
    ),
    len(document_manifest_records),
)
item_count_exact_match = safe_ratio(
    sum(record["item_count_exact"] for record in document_manifest_records),
    len(document_manifest_records),
)

overall_metrics = {
    "documents": len(document_manifest_records),
    "templates": EXPECTED_TEMPLATES,
    "scalar_field_observations": scalar_total,
    "scalar_field_exact": scalar_exact,
    "all_scalar_exact_match": all_scalar_exact_match,
    "critical_scalar_observations": len(critical_scalar_records),
    "critical_scalar_exact": critical_scalar_exact,
    "critical_scalar_exact_match": critical_scalar_exact_match,
    "scalar_micro_cer": scalar_micro_cer,
    "scalar_micro_wer": scalar_micro_wer,
    "predicted_item_rows": predicted_rows,
    "reference_item_rows": reference_rows,
    "matched_item_rows": row_true_positives,
    "row_precision": row_precision,
    "row_recall": row_recall,
    "row_f1": row_f1,
    "item_field_exact": item_exact_fields,
    "predicted_item_fields": predicted_item_field_count,
    "reference_item_fields": reference_item_field_count,
    "line_item_field_precision": line_item_field_precision,
    "line_item_field_recall": line_item_field_recall,
    "line_item_field_f1": line_item_field_f1,
    "item_count_exact_match": item_count_exact_match,
    "financial_consistency": financial_consistency,
    "document_exact_match": document_exact_match,
}


# ============================================================
# FIELD SUMMARY
# ============================================================

combined_field_records = all_scalar_records + all_item_field_records
field_summary_records = []

for field_name in sorted({record["field"] for record in combined_field_records}):
    records = [
        record
        for record in combined_field_records
        if record["field"] == field_name
    ]
    exact_count = sum(record["exact_match"] for record in records)
    character_errors = sum(record["character_errors"] for record in records)
    reference_characters = sum(
        record["reference_characters"] for record in records
    )
    word_errors = sum(record["word_errors"] for record in records)
    reference_words = sum(record["reference_words"] for record in records)

    field_summary_records.append(
        {
            "field": field_name,
            "group": records[0]["group"],
            "critical": records[0]["critical"],
            "observations": len(records),
            "exact_matches": exact_count,
            "exact_match_rate": safe_ratio(exact_count, len(records)),
            "character_errors": character_errors,
            "reference_characters": reference_characters,
            "cer": safe_error_rate(
                character_errors,
                reference_characters,
            ),
            "word_errors": word_errors,
            "reference_words": reference_words,
            "wer": safe_error_rate(word_errors, reference_words),
            "status": (
                "PERFECT" if exact_count == len(records) else "HAS_ERRORS"
            ),
        }
    )


# ============================================================
# TEMPLATE SUMMARY
# ============================================================

template_summary_records = []

for template_id in sorted({
    record["template_id"] for record in document_manifest_records
}):
    documents = [
        record
        for record in document_manifest_records
        if record["template_id"] == template_id
    ]
    scalar_records = [
        record
        for record in all_scalar_records
        if record["template_id"] == template_id
    ]
    item_records = [
        record
        for record in all_item_field_records
        if record["template_id"] == template_id
    ]

    scalar_correct = sum(record["exact_match"] for record in scalar_records)
    item_correct = sum(record["exact_match"] for record in item_records)
    template_predicted_item_fields = sum(
        record["predicted_item_count"] * EXPECTED_ITEM_FIELDS
        for record in documents
    )
    template_reference_item_fields = sum(
        record["reference_item_count"] * EXPECTED_ITEM_FIELDS
        for record in documents
    )
    item_precision = safe_ratio(
        item_correct,
        template_predicted_item_fields,
    )
    item_recall = safe_ratio(
        item_correct,
        template_reference_item_fields,
    )

    template_summary_records.append(
        {
            "template_id": template_id,
            "documents": len(documents),
            "scalar_exact_match": safe_ratio(
                scalar_correct,
                len(scalar_records),
            ),
            "line_item_field_f1": harmonic_mean(
                item_precision,
                item_recall,
            ),
            "item_count_exact_match": safe_ratio(
                sum(record["item_count_exact"] for record in documents),
                len(documents),
            ),
            "financial_consistency": safe_ratio(
                sum(record["financial_consistent"] for record in documents),
                len(documents),
            ),
            "document_exact_match": safe_ratio(
                sum(record["document_exact_match"] for record in documents),
                len(documents),
            ),
        }
    )


# ============================================================
# ACCEPTANCE POLICY
# ============================================================

acceptance_targets = contract["evaluation_policy"][
    "development_acceptance_targets"
]

acceptance_checks = [
    {
        "metric": "critical_scalar_exact_match",
        "minimum": float(
            acceptance_targets["critical_scalar_exact_match_minimum"]
        ),
        "actual": critical_scalar_exact_match,
    },
    {
        "metric": "all_scalar_exact_match",
        "minimum": float(
            acceptance_targets["all_scalar_exact_match_minimum"]
        ),
        "actual": all_scalar_exact_match,
    },
    {
        "metric": "line_item_field_f1",
        "minimum": float(
            acceptance_targets["line_item_field_f1_minimum"]
        ),
        "actual": line_item_field_f1,
    },
    {
        "metric": "financial_consistency",
        "minimum": float(
            acceptance_targets["financial_consistency_minimum"]
        ),
        "actual": financial_consistency,
    },
]

for check in acceptance_checks:
    check["status"] = (
        "PASSED" if check["actual"] >= check["minimum"] else "FAILED"
    )

acceptance_status = (
    "PASSED"
    if all(check["status"] == "PASSED" for check in acceptance_checks)
    else "FAILED"
)


# ============================================================
# MISMATCH DETAILS
# ============================================================

mismatch_records = []
for record in combined_field_records:
    if record["exact_match"]:
        continue
    mismatch_records.append(
        {
            "document_id": record["document_id"],
            "template_id": record["template_id"],
            "language": record["language"],
            "group": record["group"],
            "field": record["field"],
            "prediction_row": record.get("prediction_row"),
            "reference_row": record.get("reference_row"),
            "expected": record["expected"],
            "predicted": record["predicted"],
            "character_errors": record["character_errors"],
            "word_errors": record["word_errors"],
        }
    )


# ============================================================
# DISPLAY RESULTS
# ============================================================

document_table = pd.DataFrame(runtime_records)
field_table = pd.DataFrame(field_summary_records)
template_table = pd.DataFrame(template_summary_records)
acceptance_table = pd.DataFrame(acceptance_checks)

display(acceptance_table)
display(field_table)
display(template_table)
display(
    document_table[
        [
            "sequence_number",
            "document_id",
            "template_id",
            "language",
            "scalar_exact_match",
            "predicted_item_count",
            "reference_item_count",
            "row_f1",
            "item_field_f1",
            "financial_consistent",
            "document_exact_match",
            "execution",
        ]
    ]
)

if mismatch_records:
    print("\nMISMATCH DETAILS")
    display(pd.DataFrame(mismatch_records))


# ============================================================
# SAVE SUMMARIES AND EVALUATION MANIFEST
# ============================================================

def save_csv_checkpoint(
    path: Path,
    table: pd.DataFrame,
) -> str:
    text = table.to_csv(index=False, lineterminator="\n")
    if path.exists():
        if path.read_text(encoding="utf-8") != text:
            raise RuntimeError(
                f"CSV checkpoint sudah ada tetapi berbeda: {path}"
            )
        return "RECOVERED"
    atomic_write_text(path, text)
    return "CREATED"


document_summary_action = save_csv_checkpoint(
    DOCUMENT_SUMMARY_PATH,
    pd.DataFrame(document_manifest_records),
)
field_summary_action = save_csv_checkpoint(
    FIELD_SUMMARY_PATH,
    field_table,
)
template_summary_action = save_csv_checkpoint(
    TEMPLATE_SUMMARY_PATH,
    template_table,
)

mismatch_columns = [
    "document_id",
    "template_id",
    "language",
    "group",
    "field",
    "prediction_row",
    "reference_row",
    "expected",
    "predicted",
    "character_errors",
    "word_errors",
]
mismatch_table = pd.DataFrame(mismatch_records, columns=mismatch_columns)
mismatch_action = save_csv_checkpoint(
    MISMATCH_DETAIL_PATH,
    mismatch_table,
)

evaluation_manifest = {
    "schema_version": "1.0.0",
    "status": acceptance_status,
    "stage": "DEVELOPMENT_GROUND_TRUTH_EVALUATION",
    "evaluator": {
        "evaluator_id": EVALUATOR_ID,
        "evaluator_version": EVALUATOR_VERSION,
        "scalar_matching": "TYPE_AWARE_NORMALIZED_EXACT_MATCH",
        "item_alignment": "MAXIMUM_WEIGHT_BIPARTITE_ROW_MATCHING",
    },
    "scope": {
        "split": "development",
        "documents": EXPECTED_DOCUMENTS,
        "templates": EXPECTED_TEMPLATES,
        "development_ground_truth_opened": len(ground_truth_paths),
        "validation_opened": 0,
        "test_opened": 0,
    },
    "overall_metrics": overall_metrics,
    "acceptance_checks": acceptance_checks,
    "technical_controls": technical_control_records,
    "field_summary": field_summary_records,
    "template_summary": template_summary_records,
    "records": document_manifest_records,
    "artifacts": {
        "evaluation_root": str(EVALUATION_ROOT),
        "document_summary": {
            "path": str(DOCUMENT_SUMMARY_PATH),
            "sha256": sha256_file(DOCUMENT_SUMMARY_PATH),
        },
        "field_summary": {
            "path": str(FIELD_SUMMARY_PATH),
            "sha256": sha256_file(FIELD_SUMMARY_PATH),
        },
        "template_summary": {
            "path": str(TEMPLATE_SUMMARY_PATH),
            "sha256": sha256_file(TEMPLATE_SUMMARY_PATH),
        },
        "mismatch_details": {
            "path": str(MISMATCH_DETAIL_PATH),
            "sha256": sha256_file(MISMATCH_DETAIL_PATH),
            "records": len(mismatch_records),
        },
    },
    "inputs": {
        "contract": {
            "path": str(CONTRACT_PATH),
            "sha256": input_checksums_before["contract"],
        },
        "prediction_manifest": {
            "path": str(PREDICTION_MANIFEST_PATH),
            "sha256": input_checksums_before["prediction_manifest"],
        },
        "prediction_files": {
            document_id: checksum
            for document_id, checksum in sorted(
                (
                    key.split(":", 1)[1],
                    checksum,
                )
                for key, checksum in input_checksums_before.items()
                if key.startswith("prediction:")
            )
        },
        "development_ground_truth_files": {
            document_id: {
                "path": str(ground_truth_paths[document_id]),
                "sha256": ground_truth_checksums[document_id],
            }
            for document_id in sorted(ground_truth_paths)
        },
    },
    "integrity": {
        "prediction_frozen_before_ground_truth_open": True,
        "ground_truth_used_for_evaluation_only": True,
        "ground_truth_used_as_prediction_input": False,
        "validation_opened": 0,
        "test_opened": 0,
        "prediction_modifications": 0,
        "ground_truth_modifications": 0,
        "dataset_modifications": 0,
    },
    "next_stage": {
        "action_if_passed": "FREEZE_BASELINE_AND_PREPARE_VALIDATION_GATE",
        "action_if_failed": "ANALYZE_DEVELOPMENT_MISMATCHES_ONLY",
        "validation_remains_locked": True,
        "test_remains_locked": True,
    },
}

if EVALUATION_MANIFEST_PATH.exists():
    existing_manifest = load_json(EVALUATION_MANIFEST_PATH)
    if canonical_json(existing_manifest) != canonical_json(evaluation_manifest):
        raise RuntimeError(
            "Evaluation manifest sudah ada tetapi berbeda."
        )
    manifest_action = "RECOVERED"
else:
    atomic_write_json(EVALUATION_MANIFEST_PATH, evaluation_manifest)
    manifest_action = "CREATED"


# ============================================================
# INPUT IMMUTABILITY CHECK
# ============================================================

input_checksums_after = {
    "contract": sha256_file(CONTRACT_PATH),
    "prediction_manifest": sha256_file(PREDICTION_MANIFEST_PATH),
}
for record in prediction_records:
    input_checksums_after[
        f"prediction:{record['document_id']}"
    ] = sha256_file(Path(record["prediction_path"]))

changed_inputs = [
    name
    for name, checksum in input_checksums_before.items()
    if input_checksums_after.get(name) != checksum
]
changed_ground_truth = [
    document_id
    for document_id, checksum in ground_truth_checksums.items()
    if sha256_file(ground_truth_paths[document_id]) != checksum
]

if changed_inputs or changed_ground_truth:
    raise RuntimeError(
        "Input evaluation berubah selama eksekusi: "
        f"inputs={changed_inputs}, ground_truth={changed_ground_truth}"
    )


print()
print(f"Evaluation status        : {acceptance_status}")
print(f"Documents evaluated      : {len(document_manifest_records)}")
print(f"Templates                : {EXPECTED_TEMPLATES}")
print(f"All scalar exact match   : {all_scalar_exact_match:.6f}")
print(f"Critical scalar exact    : {critical_scalar_exact_match:.6f}")
print(f"Scalar micro CER         : {scalar_micro_cer:.6f}")
print(f"Scalar micro WER         : {scalar_micro_wer:.6f}")
print(f"Row precision            : {row_precision:.6f}")
print(f"Row recall               : {row_recall:.6f}")
print(f"Row F1                   : {row_f1:.6f}")
print(f"Line-item field F1       : {line_item_field_f1:.6f}")
print(f"Item-count exact match   : {item_count_exact_match:.6f}")
print(f"Financial consistency    : {financial_consistency:.6f}")
print(f"Document exact match     : {document_exact_match:.6f}")
print(f"Mismatch records         : {len(mismatch_records)}")
print(f"New evaluations          : {new_evaluations}")
print(f"Recovered evaluations    : {recovered_evaluations}")
print(f"Document summary action  : {document_summary_action}")
print(f"Field summary action     : {field_summary_action}")
print(f"Template summary action  : {template_summary_action}")
print(f"Mismatch file action     : {mismatch_action}")
print(f"Manifest action          : {manifest_action}")
print(f"Evaluation root          : {EVALUATION_ROOT}")
print(f"Evaluation manifest      : {EVALUATION_MANIFEST_PATH}")
print(
    f"Manifest SHA-256         : "
    f"{sha256_file(EVALUATION_MANIFEST_PATH)}"
)
print(f"Development GT opened    : {len(ground_truth_paths)}")
print("Validation opened        : 0")
print("Test opened              : 0")
print("Prediction modifications : 0")
print("Ground-truth modifications: 0")
print("Dataset modifications    : 0")

if acceptance_status != "PASSED":
    failed_metrics = [
        check["metric"]
        for check in acceptance_checks
        if check["status"] != "PASSED"
    ]
    raise RuntimeError(
        "DEVELOPMENT BASELINE BELOW ACCEPTANCE TARGET. "
        f"Metrik gagal: {failed_metrics}. "
        "Validation dan test tetap terkunci."
    )

print()
print(
    "✅ CELL 10D PASSED — parser baseline memenuhi seluruh target "
    "development yang dibekukan. Ground truth hanya dibuka untuk "
    "18 dokumen development; validation dan test tetap terkunci."
)


CELL 10D — INVOICE-FIELD-EVALUATOR-V1 — VERSION 1.0.2
Evaluation root: /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/field_extraction/evaluations/rule_based_baseline_v1_0_2_development_eval_v1_0_2
Scope: 18 frozen DEVELOPMENT documents only
Validation opened: 0 | Test opened: 0
Mengevaluasi 18 dokumen...

[01/18] INV-SYN-000002 | scalar=1.0000 | items=2/2 | item_f1=1.0000 | exact=True | NEW
[02/18] INV-SYN-000013 | scalar=1.0000 | items=8/8 | item_f1=1.0000 | exact=True | NEW
[03/18] INV-SYN-000015 | scalar=1.0000 | items=8/8 | item_f1=1.0000 | exact=True | NEW
[04/18] INV-SYN-000025 | scalar=1.0000 | items=2/2 | item_f1=1.0000 | exact=True | NEW
[05/18] INV-SYN-000036 | scalar=1.0000 | items=7/7 | item_f1=1.0000 | exact=True | NEW
[06/18] INV-SYN-000037 | scalar=1.0000 | items=7/7 | item_f1=1.0000 | exact=True | NEW
[07/18] INV-SYN-000043 | scalar=1.0000 | items=2/2 | item_f1=1.0000 | exact=True | NEW
[08/18] INV-SYN-000052 | scal

,control,expected,actual,status
0,evaluated_documents,18,18,VALID
1,evaluation_files,18,18,VALID
2,unique_document_ids,18,18,VALID
3,templates_evaluated,6,6,VALID
4,development_ground_truth_opened,18,18,VALID
5,validation_opened,0,0,VALID
6,test_opened,0,0,VALID
7,evaluation_errors,0,0,VALID


,metric,minimum,actual,status
0,critical_scalar_exact_match,0.98,1.0,PASSED
1,all_scalar_exact_match,0.95,1.0,PASSED
2,line_item_field_f1,0.90,1.0,PASSED
3,financial_consistency,0.99,1.0,PASSED


,field,group,critical,observations,exact_matches,exact_match_rate,character_errors,reference_characters,cer,word_errors,reference_words,wer,status
0,buyer.name,scalar,True,18,18,1.0,0,534,0.0,0,72,0.0,PERFECT
1,buyer.tax_identifier,scalar,False,18,18,1.0,0,306,0.0,0,18,0.0,PERFECT
2,currency,scalar,True,18,18,1.0,0,54,0.0,0,18,0.0,PERFECT
3,due_date,scalar,True,18,18,1.0,0,180,0.0,0,18,0.0,PERFECT
4,financials.discount,scalar,True,18,18,1.0,0,77,0.0,0,18,0.0,PERFECT
5,financials.subtotal,scalar,True,18,18,1.0,0,144,0.0,0,18,0.0,PERFECT
6,financials.tax,scalar,True,18,18,1.0,0,112,0.0,0,18,0.0,PERFECT
7,financials.total,scalar,True,18,18,1.0,0,149,0.0,0,18,0.0,PERFECT
8,invoice_date,scalar,True,18,18,1.0,0,180,0.0,0,18,0.0,PERFECT
9,invoice_number,scalar,True,18,18,1.0,0,324,0.0,0,18,0.0,PERFECT


,template_id,documents,scalar_exact_match,line_item_field_f1,item_count_exact_match,financial_consistency,document_exact_match
0,TPL-01,3,1.0,1.0,1.0,1.0,1.0
1,TPL-02,3,1.0,1.0,1.0,1.0,1.0
2,TPL-03,3,1.0,1.0,1.0,1.0,1.0
3,TPL-04,3,1.0,1.0,1.0,1.0,1.0
4,TPL-05,3,1.0,1.0,1.0,1.0,1.0
5,TPL-06,3,1.0,1.0,1.0,1.0,1.0


,sequence_number,document_id,template_id,language,scalar_exact_match,predicted_item_count,reference_item_count,row_f1,item_field_f1,financial_consistent,document_exact_match,execution
0,1,INV-SYN-000002,TPL-01,id,1.0,2,2,1.0,1.0,True,True,NEW
1,2,INV-SYN-000013,TPL-01,en,1.0,8,8,1.0,1.0,True,True,NEW
2,3,INV-SYN-000015,TPL-01,en,1.0,8,8,1.0,1.0,True,True,NEW
3,4,INV-SYN-000025,TPL-02,id,1.0,2,2,1.0,1.0,True,True,NEW
4,5,INV-SYN-000036,TPL-02,en,1.0,7,7,1.0,1.0,True,True,NEW
5,6,INV-SYN-000037,TPL-02,en,1.0,7,7,1.0,1.0,True,True,NEW
6,7,INV-SYN-000043,TPL-03,id,1.0,2,2,1.0,1.0,True,True,NEW
7,8,INV-SYN-000052,TPL-03,en,1.0,7,7,1.0,1.0,True,True,NEW
8,9,INV-SYN-000060,TPL-03,en,1.0,7,7,1.0,1.0,True,True,NEW
9,10,INV-SYN-000064,TPL-04,id,1.0,8,8,1.0,1.0,True,True,NEW



Evaluation status        : PASSED
Documents evaluated      : 18
Templates                : 6
All scalar exact match   : 1.000000
Critical scalar exact    : 1.000000
Scalar micro CER         : 0.000000
Scalar micro WER         : 0.000000
Row precision            : 1.000000
Row recall               : 1.000000
Row F1                   : 1.000000
Line-item field F1       : 1.000000
Item-count exact match   : 1.000000
Financial consistency    : 1.000000
Document exact match     : 1.000000
Mismatch records         : 0
New evaluations          : 18
Recovered evaluations    : 0
Document summary action  : CREATED
Field summary action     : CREATED
Template summary action  : CREATED
Mismatch file action     : CREATED
Manifest action          : CREATED
Evaluation root          : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/field_extraction/evaluations/rule_based_baseline_v1_0_2_development_eval_v1_0_2
Evaluation manifest      : /content/dri

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from IPython import get_ipython
from IPython.display import display


# ============================================================
# CELL 10E — FREEZE DEVELOPMENT BASELINE FOR VALIDATION
# ============================================================

BUILD_ROOT = Path(
    "/content/drive/MyDrive/InvoiceFlow-AI-Data/interim/"
    "synthetic_v1_build/20260904T150025Z"
)
FIELD_ROOT = BUILD_ROOT / "ocr_benchmark" / "field_extraction"

PARSER_MANIFEST_PATH = (
    FIELD_ROOT / "rule_based_baseline_v1_0_2_manifest.json"
)
DEVELOPMENT_EVALUATION_MANIFEST_PATH = (
    FIELD_ROOT
    / "evaluations"
    / "rule_based_baseline_v1_0_2_development_eval_v1_0_2"
    / "development_evaluation_manifest.json"
)
PREVIOUS_EVALUATION_MANIFEST_PATH = (
    FIELD_ROOT
    / "evaluations"
    / "rule_based_baseline_v1_0_1_development_eval_v1_0_1"
    / "development_evaluation_manifest.json"
)

FREEZE_ROOT = (
    FIELD_ROOT
    / "frozen_baselines"
    / "rule_based_baseline_v1_0_2"
)
PARSER_SOURCE_PATH = (
    FREEZE_ROOT / "rule_based_invoice_parser_v1_0_2.py"
)
FREEZE_MANIFEST_PATH = (
    FREEZE_ROOT / "development_freeze_manifest.json"
)
VALIDATION_BASELINE_POINTER_PATH = (
    FIELD_ROOT / "validation_baseline_pointer.json"
)

EXPECTED_PARSER_ID = "RULE-BASED-INVOICE-PARSER-V1"
EXPECTED_PARSER_VERSION = "1.0.2"
EXPECTED_DOCUMENTS = 18
EXPECTED_TEMPLATES = 6
EXPECTED_TARGET_FIELDS = 16
EXPECTED_MISMATCHES = 0


# ============================================================
# HELPERS
# ============================================================

def require_file(path: Path, label: str) -> None:
    if not path.is_file():
        raise FileNotFoundError(f"{label} tidak ditemukan: {path}")
    if path.stat().st_size <= 0:
        raise RuntimeError(f"{label} kosong: {path}")


def load_json(path: Path) -> dict:
    require_file(path, "JSON artifact")
    with path.open("r", encoding="utf-8") as file_handle:
        value = json.load(file_handle)
    if not isinstance(value, dict):
        raise TypeError(f"Root JSON bukan object: {path}")
    return value


def canonical_json(value: object) -> str:
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )


def sha256_bytes(value: bytes) -> str:
    return hashlib.sha256(value).hexdigest()


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as file_handle:
        for chunk in iter(lambda: file_handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def atomic_write_text(path: Path, content: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(
        f".{path.name}.{os.getpid()}.tmp"
    )
    try:
        temporary_path.write_text(content, encoding="utf-8")
        os.replace(temporary_path, path)
    finally:
        if temporary_path.exists():
            temporary_path.unlink()


def atomic_write_json(path: Path, value: dict) -> None:
    atomic_write_text(
        path,
        json.dumps(value, indent=2, ensure_ascii=False) + "\n",
    )


def save_immutable_text(path: Path, content: str) -> str:
    if path.exists():
        existing = path.read_text(encoding="utf-8")
        if existing != content:
            raise RuntimeError(
                f"Immutable artifact sudah ada tetapi berbeda: {path}"
            )
        return "RECOVERED"
    atomic_write_text(path, content)
    return "CREATED"


def save_immutable_json(path: Path, value: dict) -> str:
    if path.exists():
        existing = load_json(path)
        if canonical_json(existing) != canonical_json(value):
            raise RuntimeError(
                f"Immutable artifact sudah ada tetapi berbeda: {path}"
            )
        return "RECOVERED"
    atomic_write_json(path, value)
    return "CREATED"


def verify_reference(
    path_value: str,
    checksum_value: str,
    label: str,
) -> dict:
    path = Path(path_value)
    require_file(path, label)
    actual_checksum = sha256_file(path)
    if actual_checksum != checksum_value:
        raise RuntimeError(
            f"Checksum {label} tidak cocok: {path}. "
            f"Expected={checksum_value}, actual={actual_checksum}"
        )
    return {
        "label": label,
        "path": str(path),
        "sha256": actual_checksum,
        "status": "VALID",
    }


def capture_parser_source() -> tuple[str, str]:
    shell = get_ipython()
    if shell is None or not hasattr(shell, "history_manager"):
        raise RuntimeError(
            "Riwayat cell IPython tidak tersedia. Jalankan kembali Cell "
            "10C v1.0.2 pada runtime ini sebelum Cell 10E."
        )

    version_marker = "\n" + "PARSER_VERSION" + ' = "1.0.2"\n'
    suffix_helper_marker = "def legal_suffix_" + "continuation("
    parser_entry_marker = "def parse_" + "document("
    parser_id_marker = "RULE-BASED-" + "INVOICE-PARSER-V1"

    source_candidates = []
    for source in shell.history_manager.input_hist_raw:
        source_text = str(source or "")
        if (
            version_marker in "\n" + source_text
            and suffix_helper_marker in source_text
            and parser_entry_marker in source_text
            and parser_id_marker in source_text
        ):
            normalized_source = source_text.rstrip() + "\n"
            source_candidates.append(normalized_source)

    unique_sources = {
        sha256_bytes(source.encode("utf-8")): source
        for source in source_candidates
    }
    if len(unique_sources) != 1:
        raise RuntimeError(
            "Source parser v1.0.2 tidak ditemukan secara unik dalam "
            f"riwayat runtime. Ditemukan {len(unique_sources)} versi unik. "
            "Jangan lanjut ke validation."
        )

    source_sha256, source_text = next(iter(unique_sources.items()))
    return source_text, source_sha256


# ============================================================
# LOAD AND VALIDATE MANIFESTS
# ============================================================

require_file(PARSER_MANIFEST_PATH, "Parser manifest v1.0.2")
require_file(
    DEVELOPMENT_EVALUATION_MANIFEST_PATH,
    "Development evaluation manifest v1.0.2",
)

parser_manifest = load_json(PARSER_MANIFEST_PATH)
evaluation_manifest = load_json(
    DEVELOPMENT_EVALUATION_MANIFEST_PATH
)

parser = parser_manifest.get("parser", {})
parser_scope = parser_manifest.get("scope", {})
evaluation_scope = evaluation_manifest.get("scope", {})
metrics = evaluation_manifest.get("overall_metrics", {})
acceptance_checks = evaluation_manifest.get("acceptance_checks", [])
actual_mismatch_records = (
    evaluation_manifest.get("artifacts", {})
    .get("mismatch_details", {})
    .get("records")
)

runtime_parser_id = globals().get("PARSER_ID")
runtime_parser_version = globals().get("PARSER_VERSION")
runtime_parser_signature = globals().get("parser_signature")

preflight_values = [
    (
        "parser_manifest_status",
        "EXECUTED",
        parser_manifest.get("status"),
    ),
    (
        "parser_manifest_quality_before_freeze",
        "PENDING_GROUND_TRUTH_EVALUATION",
        parser_manifest.get("quality_status"),
    ),
    ("parser_id", EXPECTED_PARSER_ID, parser.get("parser_id")),
    (
        "parser_version",
        EXPECTED_PARSER_VERSION,
        parser.get("parser_version"),
    ),
    (
        "runtime_parser_id",
        EXPECTED_PARSER_ID,
        runtime_parser_id,
    ),
    (
        "runtime_parser_version",
        EXPECTED_PARSER_VERSION,
        runtime_parser_version,
    ),
    (
        "runtime_parser_signature",
        parser.get("parser_signature_sha256"),
        runtime_parser_signature,
    ),
    (
        "parser_records",
        EXPECTED_DOCUMENTS,
        len(parser_manifest.get("records", [])),
    ),
    (
        "parser_templates",
        EXPECTED_TEMPLATES,
        parser_scope.get("templates"),
    ),
    (
        "target_fields",
        EXPECTED_TARGET_FIELDS,
        parser_scope.get("target_fields"),
    ),
    (
        "development_evaluation_status",
        "PASSED",
        evaluation_manifest.get("status"),
    ),
    (
        "evaluated_documents",
        EXPECTED_DOCUMENTS,
        metrics.get("documents"),
    ),
    (
        "evaluated_templates",
        EXPECTED_TEMPLATES,
        metrics.get("templates"),
    ),
    (
        "all_scalar_exact_match",
        1.0,
        metrics.get("all_scalar_exact_match"),
    ),
    (
        "critical_scalar_exact_match",
        1.0,
        metrics.get("critical_scalar_exact_match"),
    ),
    (
        "line_item_field_f1",
        1.0,
        metrics.get("line_item_field_f1"),
    ),
    (
        "financial_consistency",
        1.0,
        metrics.get("financial_consistency"),
    ),
    (
        "document_exact_match",
        1.0,
        metrics.get("document_exact_match"),
    ),
    ("scalar_micro_cer", 0.0, metrics.get("scalar_micro_cer")),
    ("scalar_micro_wer", 0.0, metrics.get("scalar_micro_wer")),
    (
        "mismatch_records",
        EXPECTED_MISMATCHES,
        actual_mismatch_records,
    ),
    (
        "acceptance_checks_passed",
        True,
        bool(acceptance_checks)
        and all(row.get("status") == "PASSED" for row in acceptance_checks),
    ),
    (
        "parser_validation_opened",
        0,
        parser_scope.get("validation_opened"),
    ),
    (
        "parser_test_opened",
        0,
        parser_scope.get("test_opened"),
    ),
    (
        "evaluation_validation_opened",
        0,
        evaluation_scope.get("validation_opened"),
    ),
    (
        "evaluation_test_opened",
        0,
        evaluation_scope.get("test_opened"),
    ),
]

preflight_records = [
    {
        "control": name,
        "expected": expected,
        "actual": actual,
        "status": "VALID" if expected == actual else "INVALID",
    }
    for name, expected, actual in preflight_values
]

invalid_preflight = [
    row["control"]
    for row in preflight_records
    if row["status"] != "VALID"
]

display(pd.DataFrame(preflight_records))
if invalid_preflight:
    raise RuntimeError(
        "CELL 10E FREEZE PREFLIGHT FAILED. "
        f"Kontrol tidak valid: {invalid_preflight}"
    )


# ============================================================
# VERIFY ALL NON-GROUND-TRUTH INPUT AND OUTPUT REFERENCES
# ============================================================

verified_references = []

for record in parser_manifest.get("records", []):
    verified_references.append(
        verify_reference(
            record["prediction_path"],
            record["prediction_sha256"],
            f"Prediction {record['document_id']}",
        )
    )
    verified_references.append(
        verify_reference(
            record["text_layer_path"],
            record["text_layer_sha256"],
            f"Text layer {record['document_id']}",
        )
    )

for record in evaluation_manifest.get("records", []):
    verified_references.append(
        verify_reference(
            record["evaluation_path"],
            record["evaluation_sha256"],
            f"Evaluation {record['document_id']}",
        )
    )

for artifact_name, artifact in evaluation_manifest.get(
    "artifacts", {}
).items():
    if not isinstance(artifact, dict):
        continue
    if "path" not in artifact or "sha256" not in artifact:
        continue
    verified_references.append(
        verify_reference(
            artifact["path"],
            artifact["sha256"],
            f"Evaluation artifact {artifact_name}",
        )
    )

for input_name, input_artifact in parser_manifest.get(
    "input_artifacts", {}
).items():
    verified_references.append(
        verify_reference(
            input_artifact["path"],
            input_artifact["sha256"],
            f"Parser input {input_name}",
        )
    )

reference_path_counts = {}
for record in verified_references:
    reference_path_counts[record["path"]] = (
        reference_path_counts.get(record["path"], 0) + 1
    )


# ============================================================
# CAPTURE THE EXACT PARSER CELL SOURCE
# ============================================================

parser_source, parser_source_sha256 = capture_parser_source()
if parser_source_sha256 == sha256_file(PARSER_MANIFEST_PATH):
    raise RuntimeError(
        "Source parser dan manifest tidak boleh memiliki checksum sama."
    )

parser_source_action = save_immutable_text(
    PARSER_SOURCE_PATH,
    parser_source,
)

if sha256_file(PARSER_SOURCE_PATH) != parser_source_sha256:
    raise RuntimeError("Checksum source parser setelah penulisan tidak cocok.")


# ============================================================
# OPTIONAL TRACEABILITY TO THE PREVIOUS ITERATION
# ============================================================

previous_iteration = None
comparison_records = []

if PREVIOUS_EVALUATION_MANIFEST_PATH.is_file():
    previous_evaluation = load_json(
        PREVIOUS_EVALUATION_MANIFEST_PATH
    )
    previous_metrics = previous_evaluation.get("overall_metrics", {})
    previous_mismatches = (
        previous_evaluation.get("artifacts", {})
        .get("mismatch_details", {})
        .get("records")
    )

    for metric_name in (
        "all_scalar_exact_match",
        "critical_scalar_exact_match",
        "line_item_field_f1",
        "document_exact_match",
    ):
        comparison_records.append(
            {
                "metric": metric_name,
                "v1.0.1": previous_metrics.get(metric_name),
                "v1.0.2": metrics.get(metric_name),
                "delta": round(
                    float(metrics.get(metric_name, 0.0))
                    - float(previous_metrics.get(metric_name, 0.0)),
                    6,
                ),
            }
        )

    comparison_records.append(
        {
            "metric": "mismatch_records",
            "v1.0.1": previous_mismatches,
            "v1.0.2": EXPECTED_MISMATCHES,
            "delta": EXPECTED_MISMATCHES - int(previous_mismatches),
        }
    )

    previous_iteration = {
        "manifest_path": str(PREVIOUS_EVALUATION_MANIFEST_PATH),
        "manifest_sha256": sha256_file(
            PREVIOUS_EVALUATION_MANIFEST_PATH
        ),
        "status": previous_evaluation.get("status"),
        "overall_metrics": previous_metrics,
        "mismatch_records": previous_mismatches,
    }

if comparison_records:
    print("\nDEVELOPMENT ITERATION COMPARISON")
    display(pd.DataFrame(comparison_records))


# ============================================================
# CREATE IMMUTABLE FREEZE MANIFEST AND RECOVERY POINTER
# ============================================================

FREEZE_ROOT.mkdir(parents=True, exist_ok=True)

if FREEZE_MANIFEST_PATH.exists():
    existing_freeze = load_json(FREEZE_MANIFEST_PATH)
    frozen_at_utc = existing_freeze.get("frozen_at_utc")
    if not frozen_at_utc:
        raise RuntimeError("Freeze manifest lama tidak memiliki timestamp.")
else:
    frozen_at_utc = datetime.now(timezone.utc).isoformat(
        timespec="seconds"
    )

freeze_manifest = {
    "schema_version": "1.0.0",
    "status": "FROZEN_FOR_VALIDATION",
    "baseline_id": (
        "RULE-BASED-INVOICE-PARSER-V1@1.0.2"
    ),
    "frozen_at_utc": frozen_at_utc,
    "parser": {
        "parser_id": parser.get("parser_id"),
        "parser_version": parser.get("parser_version"),
        "parser_signature_sha256": parser.get(
            "parser_signature_sha256"
        ),
        "source_path": str(PARSER_SOURCE_PATH),
        "source_sha256": parser_source_sha256,
        "template_specific_branching": parser.get(
            "template_specific_branching"
        ),
        "ground_truth_as_prediction_input": parser.get(
            "ground_truth_as_prediction_input"
        ),
    },
    "development_evidence": {
        "documents": EXPECTED_DOCUMENTS,
        "templates": EXPECTED_TEMPLATES,
        "target_fields": EXPECTED_TARGET_FIELDS,
        "overall_metrics": metrics,
        "mismatch_records": actual_mismatch_records,
        "acceptance_checks": acceptance_checks,
    },
    "immutable_inputs": {
        "parser_manifest": {
            "path": str(PARSER_MANIFEST_PATH),
            "sha256": sha256_file(PARSER_MANIFEST_PATH),
        },
        "development_evaluation_manifest": {
            "path": str(DEVELOPMENT_EVALUATION_MANIFEST_PATH),
            "sha256": sha256_file(
                DEVELOPMENT_EVALUATION_MANIFEST_PATH
            ),
        },
        "previous_iteration": previous_iteration,
    },
    "verified_artifacts": {
        "reference_checks": len(verified_references),
        "unique_paths": len(reference_path_counts),
        "all_checksums_valid": True,
    },
    "split_policy": {
        "development_tuning_complete": True,
        "next_allowed_split": "validation",
        "validation_opened_during_freeze": 0,
        "test_opened_during_freeze": 0,
        "test_remains_locked": True,
        "parser_changes_after_validation_require_new_version": True,
        "validation_results_must_not_trigger_silent_v1_0_2_changes": True,
    },
    "integrity": {
        "development_ground_truth_opened_in_this_cell": 0,
        "validation_opened": 0,
        "test_opened": 0,
        "prediction_modifications": 0,
        "evaluation_modifications": 0,
        "dataset_modifications": 0,
        "source_modifications": 0,
    },
    "next_stage": {
        "cell": "CELL 11A",
        "action": "VALIDATION_COHORT_PREFLIGHT",
        "allowed_templates": ["TPL-07", "TPL-08"],
        "expected_documents": 40,
        "ground_truth_must_remain_closed": True,
        "test_remains_locked": True,
    },
}

freeze_action = save_immutable_json(
    FREEZE_MANIFEST_PATH,
    freeze_manifest,
)
freeze_manifest_sha256 = sha256_file(FREEZE_MANIFEST_PATH)

pointer = {
    "schema_version": "1.0.0",
    "status": "ACTIVE_FOR_VALIDATION",
    "baseline_id": freeze_manifest["baseline_id"],
    "parser_id": parser.get("parser_id"),
    "parser_version": parser.get("parser_version"),
    "freeze_manifest_path": str(FREEZE_MANIFEST_PATH),
    "freeze_manifest_sha256": freeze_manifest_sha256,
    "parser_source_path": str(PARSER_SOURCE_PATH),
    "parser_source_sha256": parser_source_sha256,
    "next_allowed_split": "validation",
    "test_remains_locked": True,
}

pointer_action = save_immutable_json(
    VALIDATION_BASELINE_POINTER_PATH,
    pointer,
)

recovered_pointer = load_json(VALIDATION_BASELINE_POINTER_PATH)
if canonical_json(recovered_pointer) != canonical_json(pointer):
    raise RuntimeError("Recovery pointer berbeda setelah penulisan.")
if sha256_file(FREEZE_MANIFEST_PATH) != pointer[
    "freeze_manifest_sha256"
]:
    raise RuntimeError("Pointer tidak cocok dengan freeze manifest.")


# ============================================================
# FINAL REPORT
# ============================================================

final_controls = [
    {
        "control": "freeze_status",
        "expected": "FROZEN_FOR_VALIDATION",
        "actual": freeze_manifest.get("status"),
    },
    {
        "control": "parser_source_checksum",
        "expected": parser_source_sha256,
        "actual": sha256_file(PARSER_SOURCE_PATH),
    },
    {
        "control": "freeze_pointer_checksum",
        "expected": freeze_manifest_sha256,
        "actual": recovered_pointer.get("freeze_manifest_sha256"),
    },
    {
        "control": "development_exact_match",
        "expected": 1.0,
        "actual": metrics.get("document_exact_match"),
    },
    {
        "control": "mismatch_records",
        "expected": 0,
        "actual": actual_mismatch_records,
    },
    {
        "control": "validation_opened",
        "expected": 0,
        "actual": 0,
    },
    {
        "control": "test_opened",
        "expected": 0,
        "actual": 0,
    },
]

for control in final_controls:
    control["status"] = (
        "VALID"
        if control["expected"] == control["actual"]
        else "INVALID"
    )

display(pd.DataFrame(final_controls))

invalid_final = [
    row["control"]
    for row in final_controls
    if row["status"] != "VALID"
]
if invalid_final:
    raise RuntimeError(
        "CELL 10E FINAL GATE FAILED. "
        f"Kontrol tidak valid: {invalid_final}"
    )

print()
print(f"Baseline ID          : {freeze_manifest['baseline_id']}")
print(f"Freeze status        : {freeze_manifest['status']}")
print(f"Parser source action : {parser_source_action}")
print(f"Parser source        : {PARSER_SOURCE_PATH}")
print(f"Parser source SHA-256: {parser_source_sha256}")
print(f"Freeze action        : {freeze_action}")
print(f"Freeze manifest      : {FREEZE_MANIFEST_PATH}")
print(f"Freeze SHA-256       : {freeze_manifest_sha256}")
print(f"Pointer action       : {pointer_action}")
print(f"Recovery pointer     : {VALIDATION_BASELINE_POINTER_PATH}")
print(f"Verified references  : {len(verified_references)}")
print(f"Unique artifact paths: {len(reference_path_counts)}")
print("Development exact    : 1.000000")
print("Mismatch records     : 0")
print("Ground truth opened  : 0 in this cell")
print("Validation opened    : 0")
print("Test opened          : 0")
print("Dataset modifications: 0")
print()
print(
    "✅ CELL 10E PASSED — parser v1.0.2 dan bukti development "
    "telah dibekukan secara immutable. Validation kini boleh "
    "memasuki tahap preflight; test tetap terkunci."
)


,control,expected,actual,status
0,parser_manifest_status,EXECUTED,EXECUTED,VALID
1,parser_manifest_quality_before_freeze,PENDING_GROUND_TRUTH_EVALUATION,PENDING_GROUND_TRUTH_EVALUATION,VALID
2,parser_id,RULE-BASED-INVOICE-PARSER-V1,RULE-BASED-INVOICE-PARSER-V1,VALID
3,parser_version,1.0.2,1.0.2,VALID
4,runtime_parser_id,RULE-BASED-INVOICE-PARSER-V1,RULE-BASED-INVOICE-PARSER-V1,VALID
5,runtime_parser_version,1.0.2,1.0.2,VALID
6,runtime_parser_signature,ffe04f258fb3b1a6a55661698fa6c7dda29ddd7b714c2f...,ffe04f258fb3b1a6a55661698fa6c7dda29ddd7b714c2f...,VALID
7,parser_records,18,18,VALID
8,parser_templates,6,6,VALID
9,target_fields,16,16,VALID



DEVELOPMENT ITERATION COMPARISON


,metric,v1.0.1,v1.0.2,delta
0,all_scalar_exact_match,0.995370,1.0,0.004630
1,critical_scalar_exact_match,0.994444,1.0,0.005556
2,line_item_field_f1,1.000000,1.0,0.000000
3,document_exact_match,0.944444,1.0,0.055556
4,mismatch_records,1.000000,0.0,-1.000000


,control,expected,actual,status
0,freeze_status,FROZEN_FOR_VALIDATION,FROZEN_FOR_VALIDATION,VALID
1,parser_source_checksum,0a737f57a86df7d3e16eef6749e3c3a23e82fb51227d09...,0a737f57a86df7d3e16eef6749e3c3a23e82fb51227d09...,VALID
2,freeze_pointer_checksum,9f234c5c8cdc10c277e9c26a1f0831d1e08f3dc6fb9503...,9f234c5c8cdc10c277e9c26a1f0831d1e08f3dc6fb9503...,VALID
3,development_exact_match,1.0,1.0,VALID
4,mismatch_records,0,0,VALID
5,validation_opened,0,0,VALID
6,test_opened,0,0,VALID



Baseline ID          : RULE-BASED-INVOICE-PARSER-V1@1.0.2
Freeze status        : FROZEN_FOR_VALIDATION
Parser source action : CREATED
Parser source        : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/field_extraction/frozen_baselines/rule_based_baseline_v1_0_2/rule_based_invoice_parser_v1_0_2.py
Parser source SHA-256: 0a737f57a86df7d3e16eef6749e3c3a23e82fb51227d099845908e89c8be642c
Freeze action        : CREATED
Freeze manifest      : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/field_extraction/frozen_baselines/rule_based_baseline_v1_0_2/development_freeze_manifest.json
Freeze SHA-256       : 9f234c5c8cdc10c277e9c26a1f0831d1e08f3dc6fb9503628f425ebda6c635b4
Pointer action       : CREATED
Recovery pointer     : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/field_extraction/validation_baseline_pointer.json
Verified refer

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import re
from collections import Counter
from pathlib import Path

import pandas as pd
from IPython.display import display


# ============================================================
# CELL 11A — VALIDATION COHORT PREFLIGHT (GROUND TRUTH CLOSED)
# ============================================================

CELL_VERSION = "1.1.0"

DATA_ROOT = Path(
    "/content/drive/MyDrive/InvoiceFlow-AI-Data"
)
BUILD_ROOT = (
    DATA_ROOT
    / "interim"
    / "synthetic_v1_build"
    / "20260904T150025Z"
)
RELEASE_ROOT = (
    DATA_ROOT
    / "releases"
    / "SYNTHETIC-INVOICE-V1"
    / "1.0.0"
)
FIELD_ROOT = BUILD_ROOT / "ocr_benchmark" / "field_extraction"
RENDERED_DATASET_ROOT = BUILD_ROOT / "rendered_dataset"

CURRENT_RELEASE_POINTER_PATH = (
    DATA_ROOT / "releases" / "current_release.json"
)
RELEASE_INDEX_PATH = RELEASE_ROOT / "release_index.jsonl"
RELEASE_MANIFEST_PATH = RELEASE_ROOT / "release_manifest.json"
RENDER_INDEX_PATH = (
    BUILD_ROOT / "manifests" / "batch_render_index.jsonl"
)

VALIDATION_BASELINE_POINTER_PATH = (
    FIELD_ROOT / "validation_baseline_pointer.json"
)
VALIDATION_ROOT = (
    FIELD_ROOT / "validation" / "rule_based_baseline_v1_0_2"
)
VALIDATION_SELECTION_PATH = (
    VALIDATION_ROOT / "validation_cohort_manifest.json"
)

EXPECTED_BASELINE_ID = "RULE-BASED-INVOICE-PARSER-V1@1.0.2"
EXPECTED_PARSER_VERSION = "1.0.2"
EXPECTED_RELEASE_RECORDS = 200
EXPECTED_VALIDATION_DOCUMENTS = 40
EXPECTED_VALIDATION_TEMPLATES = {"TPL-07", "TPL-08"}
EXPECTED_DOCUMENTS_PER_TEMPLATE = 20
EXPECTED_LANGUAGES = {"en", "id"}
SUPPORTED_CURRENCIES = {"EUR", "GBP", "IDR", "USD"}

SHA256_PATTERN = re.compile(r"^[0-9a-f]{64}$", re.IGNORECASE)


# ============================================================
# FILE AND SERIALIZATION HELPERS
# ============================================================

def require_file(path: Path, label: str) -> None:
    if not path.is_file():
        raise FileNotFoundError(f"{label} tidak ditemukan: {path}")
    if path.stat().st_size <= 0:
        raise RuntimeError(f"{label} kosong: {path}")


def load_json(path: Path) -> dict:
    require_file(path, "JSON artifact")
    with path.open("r", encoding="utf-8") as file_handle:
        value = json.load(file_handle)
    if not isinstance(value, dict):
        raise TypeError(f"Root JSON bukan object: {path}")
    return value


def load_jsonl(path: Path) -> list[dict]:
    require_file(path, "JSONL artifact")
    records = []
    with path.open("r", encoding="utf-8") as file_handle:
        for line_number, line in enumerate(file_handle, start=1):
            if not line.strip():
                continue
            value = json.loads(line)
            if not isinstance(value, dict):
                raise TypeError(
                    f"Record JSONL baris {line_number} bukan object: {path}"
                )
            records.append(value)
    return records


def canonical_json(value: object) -> str:
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as file_handle:
        for chunk in iter(lambda: file_handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def sha256_object(value: object) -> str:
    return hashlib.sha256(
        canonical_json(value).encode("utf-8")
    ).hexdigest()


def atomic_write_json(path: Path, value: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(
        f".{path.name}.{os.getpid()}.tmp"
    )
    try:
        temporary_path.write_text(
            json.dumps(value, indent=2, ensure_ascii=False) + "\n",
            encoding="utf-8",
        )
        os.replace(temporary_path, path)
    finally:
        if temporary_path.exists():
            temporary_path.unlink()


def save_immutable_json(path: Path, value: dict) -> str:
    if path.exists():
        existing = load_json(path)
        if canonical_json(existing) != canonical_json(value):
            raise RuntimeError(
                f"Checkpoint sudah ada tetapi berbeda: {path}"
            )
        return "RECOVERED"
    atomic_write_json(path, value)
    return "CREATED"


# ============================================================
# GENERIC MANIFEST HELPERS
# ============================================================

def walk_values(value: object):
    if isinstance(value, dict):
        for nested in value.values():
            yield from walk_values(nested)
    elif isinstance(value, list):
        for nested in value:
            yield from walk_values(nested)
    else:
        yield value


def deep_key_values(value: object, keys: set[str]) -> list[object]:
    matches = []
    if isinstance(value, dict):
        for key, nested in value.items():
            if str(key).casefold() in keys:
                matches.append(nested)
            matches.extend(deep_key_values(nested, keys))
    elif isinstance(value, list):
        for nested in value:
            matches.extend(deep_key_values(nested, keys))
    return matches


def first_scalar(
    record: dict,
    aliases: tuple[str, ...],
    label: str,
) -> object:
    alias_set = {alias.casefold() for alias in aliases}
    matches = [
        value
        for value in deep_key_values(record, alias_set)
        if isinstance(value, (str, int, float, bool))
    ]
    unique = []
    for value in matches:
        if value not in unique:
            unique.append(value)
    if len(unique) != 1:
        raise RuntimeError(
            f"{label} tidak ditemukan secara unik. Kandidat={unique}"
        )
    return unique[0]


def normalize_split(value: object) -> str:
    normalized = str(value or "").strip().casefold()
    aliases = {
        "dev": "development",
        "train": "development",
        "development": "development",
        "val": "validation",
        "valid": "validation",
        "validation": "validation",
        "test": "test",
    }
    return aliases.get(normalized, normalized)


def resolve_release_status(manifest: dict) -> str | None:
    """Read the release status without accepting unrelated nested statuses."""

    for key in ("release_status", "status"):
        value = manifest.get(key)
        if isinstance(value, str) and value.strip():
            return value.strip().upper()

    release_section = manifest.get("release")
    if isinstance(release_section, dict):
        for key in ("release_status", "status"):
            value = release_section.get(key)
            if isinstance(value, str) and value.strip():
                return value.strip().upper()

    explicit_nested_values = [
        value
        for value in deep_key_values(manifest, {"release_status"})
        if isinstance(value, str) and value.strip()
    ]
    normalized_values = sorted(
        {value.strip().upper() for value in explicit_nested_values}
    )
    if len(normalized_values) == 1:
        return normalized_values[0]

    return None


def locate_pdf_and_checksum(
    render_record: dict,
    document_id: str,
    template_id: str,
) -> tuple[Path, str]:
    artifacts = render_record.get("artifacts")
    if not isinstance(artifacts, dict):
        raise RuntimeError(
            f"artifacts tidak valid pada render record {document_id}."
        )

    pdf_artifact = artifacts.get("pdf")
    if not isinstance(pdf_artifact, dict):
        raise RuntimeError(
            f"artifacts.pdf tidak valid pada render record {document_id}."
        )

    relative_path_value = pdf_artifact.get("relative_path")
    expected_size = pdf_artifact.get("size_bytes")
    expected_checksum = pdf_artifact.get("sha256")

    if not isinstance(relative_path_value, str) or not relative_path_value:
        raise RuntimeError(f"PDF relative_path tidak valid: {document_id}")
    if not isinstance(expected_size, int) or expected_size <= 0:
        raise RuntimeError(f"PDF size_bytes tidak valid: {document_id}")
    if (
        not isinstance(expected_checksum, str)
        or not SHA256_PATTERN.fullmatch(expected_checksum)
    ):
        raise RuntimeError(f"PDF sha256 tidak valid: {document_id}")

    relative_path = Path(relative_path_value)
    if relative_path.is_absolute() or ".." in relative_path.parts:
        raise RuntimeError(
            f"PDF relative_path tidak aman: {relative_path_value}"
        )

    expected_relative_path = Path(
        "pdf",
        "validation",
        template_id,
        f"{document_id}.pdf",
    )
    if relative_path != expected_relative_path:
        raise RuntimeError(
            f"Struktur path PDF tidak cocok untuk {document_id}. "
            f"Expected={expected_relative_path}, actual={relative_path}"
        )

    rendered_root_resolved = RENDERED_DATASET_ROOT.resolve()
    pdf_path = (RENDERED_DATASET_ROOT / relative_path).resolve()
    if not pdf_path.is_relative_to(rendered_root_resolved):
        raise RuntimeError(
            f"PDF keluar dari rendered_dataset root: {pdf_path}"
        )
    if not pdf_path.is_file():
        raise FileNotFoundError(
            f"PDF validation tidak ditemukan: {pdf_path}"
        )
    if pdf_path.stat().st_size != expected_size:
        raise RuntimeError(
            f"Ukuran PDF {document_id} tidak cocok. "
            f"Expected={expected_size}, actual={pdf_path.stat().st_size}"
        )

    actual_checksum = sha256_file(pdf_path)
    if actual_checksum.casefold() != expected_checksum.casefold():
        raise RuntimeError(
            f"Checksum PDF {document_id} tidak cocok. "
            f"Expected={expected_checksum}, actual={actual_checksum}"
        )

    return pdf_path, actual_checksum


def unique_record_by_document(
    records: list[dict],
    document_id: str,
    label: str,
) -> dict:
    matches = []
    for record in records:
        try:
            record_document_id = str(
                first_scalar(
                    record,
                    ("document_id",),
                    f"{label} document_id",
                )
            )
        except RuntimeError:
            continue
        if record_document_id == document_id:
            matches.append(record)
    if len(matches) != 1:
        raise RuntimeError(
            f"{label} {document_id} ditemukan {len(matches)} kali; "
            "seharusnya tepat satu."
        )
    return matches[0]


# ============================================================
# SOURCE PREFLIGHT
# ============================================================

for path, label in (
    (CURRENT_RELEASE_POINTER_PATH, "Current release pointer"),
    (RELEASE_INDEX_PATH, "Release index"),
    (RELEASE_MANIFEST_PATH, "Release manifest"),
    (RENDER_INDEX_PATH, "Batch render index"),
    (VALIDATION_BASELINE_POINTER_PATH, "Validation baseline pointer"),
):
    require_file(path, label)

current_release_pointer = load_json(CURRENT_RELEASE_POINTER_PATH)
release_manifest = load_json(RELEASE_MANIFEST_PATH)
release_records = load_jsonl(RELEASE_INDEX_PATH)
render_records = load_jsonl(RENDER_INDEX_PATH)
baseline_pointer = load_json(VALIDATION_BASELINE_POINTER_PATH)
release_status = resolve_release_status(release_manifest)

freeze_manifest_path = Path(baseline_pointer["freeze_manifest_path"])
require_file(freeze_manifest_path, "Frozen baseline manifest")
freeze_manifest = load_json(freeze_manifest_path)

source_checksums_before = {
    "current_release_pointer": sha256_file(CURRENT_RELEASE_POINTER_PATH),
    "release_index": sha256_file(RELEASE_INDEX_PATH),
    "release_manifest": sha256_file(RELEASE_MANIFEST_PATH),
    "render_index": sha256_file(RENDER_INDEX_PATH),
    "baseline_pointer": sha256_file(VALIDATION_BASELINE_POINTER_PATH),
    "freeze_manifest": sha256_file(freeze_manifest_path),
}

release_manifest_sha256 = source_checksums_before["release_manifest"]
pointer_strings = {
    str(value)
    for value in walk_values(current_release_pointer)
    if isinstance(value, str)
}

source_controls = [
    (
        "release_manifest_status",
        "FROZEN",
        release_status,
    ),
    (
        "release_pointer_matches_manifest",
        True,
        release_manifest_sha256 in pointer_strings,
    ),
    (
        "release_records",
        EXPECTED_RELEASE_RECORDS,
        len(release_records),
    ),
    (
        "render_index_records",
        EXPECTED_RELEASE_RECORDS,
        len(render_records),
    ),
    (
        "baseline_pointer_status",
        "ACTIVE_FOR_VALIDATION",
        baseline_pointer.get("status"),
    ),
    (
        "baseline_id",
        EXPECTED_BASELINE_ID,
        baseline_pointer.get("baseline_id"),
    ),
    (
        "parser_version",
        EXPECTED_PARSER_VERSION,
        baseline_pointer.get("parser_version"),
    ),
    (
        "freeze_status",
        "FROZEN_FOR_VALIDATION",
        freeze_manifest.get("status"),
    ),
    (
        "freeze_checksum",
        baseline_pointer.get("freeze_manifest_sha256"),
        source_checksums_before["freeze_manifest"],
    ),
    (
        "next_allowed_split",
        "validation",
        baseline_pointer.get("next_allowed_split"),
    ),
    (
        "test_locked",
        True,
        baseline_pointer.get("test_remains_locked"),
    ),
]

source_control_records = [
    {
        "control": name,
        "expected": expected,
        "actual": actual,
        "status": "VALID" if expected == actual else "INVALID",
    }
    for name, expected, actual in source_controls
]
display(pd.DataFrame(source_control_records))

invalid_source_controls = [
    row["control"]
    for row in source_control_records
    if row["status"] != "VALID"
]
if invalid_source_controls:
    raise RuntimeError(
        "CELL 11A SOURCE PREFLIGHT FAILED. "
        f"Kontrol tidak valid: {invalid_source_controls}"
    )


# ============================================================
# BUILD THE COMPLETE VALIDATION COHORT FROM RELEASE METADATA
# ============================================================

validation_release_records = []
for release_record in release_records:
    split = normalize_split(
        first_scalar(
            release_record,
            ("split", "dataset_split"),
            "release split",
        )
    )
    if split == "validation":
        validation_release_records.append(release_record)

cohort_records = []
errors = []

for release_record in validation_release_records:
    try:
        document_id = str(
            first_scalar(
                release_record,
                ("document_id",),
                "document_id",
            )
        )
        canonical_invoice_id = str(
            first_scalar(
                release_record,
                ("canonical_invoice_id", "canonical_id"),
                "canonical_invoice_id",
            )
        )
        template_id = str(
            first_scalar(
                release_record,
                ("template_id",),
                "template_id",
            )
        )
        language = str(
            first_scalar(
                release_record,
                ("language", "locale"),
                "language",
            )
        ).casefold()
        currency = str(
            first_scalar(
                release_record,
                ("currency", "currency_code"),
                "currency",
            )
        ).upper()
        item_count = int(
            first_scalar(
                release_record,
                ("item_count", "line_item_count"),
                "item_count",
            )
        )

        render_record = unique_record_by_document(
            render_records,
            document_id,
            "Batch render record",
        )
        render_template_id = str(
            first_scalar(
                render_record,
                ("template_id",),
                "render template_id",
            )
        )
        if render_template_id != template_id:
            raise RuntimeError(
                f"Template release/render berbeda untuk {document_id}: "
                f"{template_id} != {render_template_id}"
            )

        pdf_path, pdf_sha256 = locate_pdf_and_checksum(
            render_record,
            document_id,
            template_id,
        )

        cohort_records.append(
            {
                "sequence_number": 0,
                "canonical_invoice_id": canonical_invoice_id,
                "document_id": document_id,
                "template_id": template_id,
                "split": "validation",
                "language": language,
                "currency": currency,
                "item_count": item_count,
                "pdf_path": str(pdf_path),
                "pdf_sha256": pdf_sha256,
                "release_record_sha256": sha256_object(release_record),
                "render_record_sha256": sha256_object(render_record),
            }
        )
    except Exception as error:
        errors.append(
            {
                "error_type": type(error).__name__,
                "error": str(error)[:700],
                "record_preview": str(release_record)[:300],
            }
        )

cohort_records.sort(
    key=lambda row: (
        row["template_id"],
        row["document_id"],
    )
)
for sequence_number, record in enumerate(cohort_records, start=1):
    record["sequence_number"] = sequence_number

if errors:
    display(pd.DataFrame(errors))
    raise RuntimeError(
        f"Gagal memetakan {len(errors)} validation records."
    )


# ============================================================
# COHORT GATES
# ============================================================

template_counts = Counter(
    row["template_id"] for row in cohort_records
)
language_counts = Counter(
    row["language"] for row in cohort_records
)
currency_counts = Counter(
    row["currency"] for row in cohort_records
)
template_languages = {
    template_id: sorted(
        {
            row["language"]
            for row in cohort_records
            if row["template_id"] == template_id
        }
    )
    for template_id in sorted(template_counts)
}

cohort_values = [
    ("validation_documents", EXPECTED_VALIDATION_DOCUMENTS, len(cohort_records)),
    (
        "unique_document_ids",
        EXPECTED_VALIDATION_DOCUMENTS,
        len({row["document_id"] for row in cohort_records}),
    ),
    (
        "unique_canonical_ids",
        EXPECTED_VALIDATION_DOCUMENTS,
        len({row["canonical_invoice_id"] for row in cohort_records}),
    ),
    (
        "validation_templates",
        sorted(EXPECTED_VALIDATION_TEMPLATES),
        sorted(template_counts),
    ),
    (
        "documents_per_template",
        True,
        all(
            template_counts.get(template_id) == EXPECTED_DOCUMENTS_PER_TEMPLATE
            for template_id in EXPECTED_VALIDATION_TEMPLATES
        ),
    ),
    (
        "languages_per_template",
        True,
        all(
            set(template_languages.get(template_id, []))
            == EXPECTED_LANGUAGES
            for template_id in EXPECTED_VALIDATION_TEMPLATES
        ),
    ),
    (
        "supported_currencies",
        True,
        set(currency_counts).issubset(SUPPORTED_CURRENCIES),
    ),
    (
        "all_records_validation",
        True,
        all(row["split"] == "validation" for row in cohort_records),
    ),
    (
        "item_capacity_valid",
        True,
        all(2 <= row["item_count"] <= 8 for row in cohort_records),
    ),
    (
        "pdf_files",
        EXPECTED_VALIDATION_DOCUMENTS,
        sum(Path(row["pdf_path"]).is_file() for row in cohort_records),
    ),
    (
        "pdf_checksums_unique",
        EXPECTED_VALIDATION_DOCUMENTS,
        len({row["pdf_sha256"] for row in cohort_records}),
    ),
    ("ground_truth_opened", 0, 0),
    ("test_opened", 0, 0),
]

cohort_control_records = [
    {
        "control": name,
        "expected": expected,
        "actual": actual,
        "status": "VALID" if expected == actual else "INVALID",
    }
    for name, expected, actual in cohort_values
]

display(pd.DataFrame(cohort_control_records))
display(pd.DataFrame(cohort_records))

invalid_cohort_controls = [
    row["control"]
    for row in cohort_control_records
    if row["status"] != "VALID"
]
if invalid_cohort_controls:
    raise RuntimeError(
        "CELL 11A VALIDATION COHORT GATE FAILED. "
        f"Kontrol tidak valid: {invalid_cohort_controls}"
    )


# ============================================================
# IMMUTABLE VALIDATION SELECTION CHECKPOINT
# ============================================================

selection_manifest = {
    "schema_version": "1.0.0",
    "status": "READY",
    "stage": "VALIDATION_COHORT_PREFLIGHT",
    "baseline": {
        "baseline_id": baseline_pointer["baseline_id"],
        "parser_id": baseline_pointer["parser_id"],
        "parser_version": baseline_pointer["parser_version"],
        "freeze_manifest_path": str(freeze_manifest_path),
        "freeze_manifest_sha256": source_checksums_before[
            "freeze_manifest"
        ],
        "parser_source_path": baseline_pointer["parser_source_path"],
        "parser_source_sha256": baseline_pointer[
            "parser_source_sha256"
        ],
    },
    "scope": {
        "split": "validation",
        "documents": EXPECTED_VALIDATION_DOCUMENTS,
        "templates": sorted(EXPECTED_VALIDATION_TEMPLATES),
        "documents_per_template": dict(sorted(template_counts.items())),
        "language_distribution": dict(sorted(language_counts.items())),
        "currency_distribution": dict(sorted(currency_counts.items())),
        "template_languages": template_languages,
        "selection_policy": "ALL_FROZEN_VALIDATION_RECORDS",
    },
    "records": cohort_records,
    "source_artifacts": {
        "current_release_pointer": {
            "path": str(CURRENT_RELEASE_POINTER_PATH),
            "sha256": source_checksums_before["current_release_pointer"],
        },
        "release_index": {
            "path": str(RELEASE_INDEX_PATH),
            "sha256": source_checksums_before["release_index"],
        },
        "release_manifest": {
            "path": str(RELEASE_MANIFEST_PATH),
            "sha256": source_checksums_before["release_manifest"],
        },
        "render_index": {
            "path": str(RENDER_INDEX_PATH),
            "sha256": source_checksums_before["render_index"],
        },
        "validation_baseline_pointer": {
            "path": str(VALIDATION_BASELINE_POINTER_PATH),
            "sha256": source_checksums_before["baseline_pointer"],
        },
    },
    "integrity": {
        "ground_truth_loaded": False,
        "canonical_payload_loaded": False,
        "validation_ground_truth_opened": 0,
        "test_opened": 0,
        "dataset_modifications": 0,
        "source_modifications": 0,
    },
    "next_stage": {
        "cell": "CELL 11B",
        "action": "BUILD_VALIDATION_UNIFIED_TEXT_LAYER",
        "ground_truth_must_remain_closed": True,
        "test_remains_locked": True,
    },
}

selection_action = save_immutable_json(
    VALIDATION_SELECTION_PATH,
    selection_manifest,
)

selection_sha256 = sha256_file(VALIDATION_SELECTION_PATH)
recovered_selection = load_json(VALIDATION_SELECTION_PATH)
if canonical_json(recovered_selection) != canonical_json(selection_manifest):
    raise RuntimeError("Validation selection berbeda setelah penulisan.")


# ============================================================
# SOURCE IMMUTABILITY CHECK
# ============================================================

source_checksums_after = {
    "current_release_pointer": sha256_file(CURRENT_RELEASE_POINTER_PATH),
    "release_index": sha256_file(RELEASE_INDEX_PATH),
    "release_manifest": sha256_file(RELEASE_MANIFEST_PATH),
    "render_index": sha256_file(RENDER_INDEX_PATH),
    "baseline_pointer": sha256_file(VALIDATION_BASELINE_POINTER_PATH),
    "freeze_manifest": sha256_file(freeze_manifest_path),
}

changed_sources = [
    name
    for name, checksum in source_checksums_before.items()
    if source_checksums_after.get(name) != checksum
]
if changed_sources:
    raise RuntimeError(
        f"Source berubah selama Cell 11A: {changed_sources}"
    )


print()
print(f"Cell version         : {CELL_VERSION}")
print(f"Baseline ID          : {baseline_pointer['baseline_id']}")
print(f"Parser version       : {baseline_pointer['parser_version']}")
print(f"Freeze status        : {freeze_manifest['status']}")
print(f"Validation documents : {len(cohort_records)}")
print(f"Templates            : {dict(sorted(template_counts.items()))}")
print(f"Languages            : {dict(sorted(language_counts.items()))}")
print(f"Currencies           : {dict(sorted(currency_counts.items()))}")
print(f"Selection action     : {selection_action}")
print(f"Selection manifest   : {VALIDATION_SELECTION_PATH}")
print(f"Selection SHA-256    : {selection_sha256}")
print("Ground truth opened  : 0")
print("Validation GT opened : 0")
print("Test opened          : 0")
print("OCR executions       : 0")
print("Dataset modifications: 0")
print("Source modifications : 0")
print()
print(
    "✅ CELL 11A PASSED — seluruh 40 dokumen validation TPL-07 "
    "dan TPL-08 telah dipetakan serta diverifikasi tanpa membuka "
    "ground truth. Lanjutkan ke Cell 11B."
)


,control,expected,actual,status
0,release_manifest_status,FROZEN,FROZEN,VALID
1,release_pointer_matches_manifest,True,True,VALID
2,release_records,200,200,VALID
3,render_index_records,200,200,VALID
4,baseline_pointer_status,ACTIVE_FOR_VALIDATION,ACTIVE_FOR_VALIDATION,VALID
5,baseline_id,RULE-BASED-INVOICE-PARSER-V1@1.0.2,RULE-BASED-INVOICE-PARSER-V1@1.0.2,VALID
6,parser_version,1.0.2,1.0.2,VALID
7,freeze_status,FROZEN_FOR_VALIDATION,FROZEN_FOR_VALIDATION,VALID
8,freeze_checksum,9f234c5c8cdc10c277e9c26a1f0831d1e08f3dc6fb9503...,9f234c5c8cdc10c277e9c26a1f0831d1e08f3dc6fb9503...,VALID
9,next_allowed_split,validation,validation,VALID


,control,expected,actual,status
0,validation_documents,40,40,VALID
1,unique_document_ids,40,40,VALID
2,unique_canonical_ids,40,40,VALID
3,validation_templates,"[TPL-07, TPL-08]","[TPL-07, TPL-08]",VALID
4,documents_per_template,True,True,VALID
5,languages_per_template,True,True,VALID
6,supported_currencies,True,True,VALID
7,all_records_validation,True,True,VALID
8,item_capacity_valid,True,True,VALID
9,pdf_files,40,40,VALID


,sequence_number,canonical_invoice_id,document_id,template_id,split,language,currency,item_count,pdf_path,pdf_sha256,release_record_sha256,render_record_sha256
0,1,CANON-000121,INV-SYN-000121,TPL-07,validation,id,IDR,7,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,4f9c017d08ff54340c3d935a1c7fae9f870e818ce98a18...,a34db5a703e37ca12a93b6fc6cae8ef258d4a9b6b9b611...,b555fa658dc80341390da5808e6f7a7cf90e989fc1087e...
1,2,CANON-000122,INV-SYN-000122,TPL-07,validation,id,USD,7,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,795cf4ddafa79837038cd7100258efb328d98df335b028...,abea076352893231ff5be7aef2bb113fe7c6288828122c...,30d768015b43ad15876880513f820d8a3f366ab65eb006...
2,3,CANON-000123,INV-SYN-000123,TPL-07,validation,id,IDR,3,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,c1e2352a30dbc47ea05f84ed964de5cb2e02c4ab167cd6...,83d87ce26dc5621cbc80537ba024ea725ed6840a6fcbc7...,34f500d2f92a0dfda3c6e8149c7f1031455d250c217f60...
3,4,CANON-000124,INV-SYN-000124,TPL-07,validation,id,USD,4,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,53c96910164a02a80174df1b57ec543bc16c5e1c2b9a4e...,15414fc0283614a6399d44c5d9f8bf54adc217684e70c0...,c4e5ac4633ca5abb304080f79e03262f2cf590517b9bcb...
4,5,CANON-000125,INV-SYN-000125,TPL-07,validation,id,IDR,4,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,8e6bfdcd5f8fe53c08f1b3260dc199d5d4b70c894dfef1...,f23b67517a6ad663e7b8f1f70961116751331d404f25a5...,7b9e6bc0cddaf2623e1c0895eec892b79cb56200062468...
5,6,CANON-000126,INV-SYN-000126,TPL-07,validation,id,IDR,2,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,28d5b04eca6230a052602aab81b2823f4734f1778f99ee...,514e8d8bcb2dd0c0535542dab48b9119a30fc53c0fcc7b...,2860e702ab47e93970dc0c6c60f9100144604f4fd8dd01...
6,7,CANON-000127,INV-SYN-000127,TPL-07,validation,id,IDR,3,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,740924b9285b7537d6dee56a9da66311f54397cc3854bf...,08473b8dfe8db4c7bfa2b3132d283e58e10a0514f767ae...,418a8fd95a2a06dc22e8693517326204164226c7d7e8d0...
7,8,CANON-000128,INV-SYN-000128,TPL-07,validation,id,IDR,4,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,584923462159cf083fb062f0303cbd3ba352a67efe682f...,8a17af70bf6eebb41588c9d3ecdf3396e463116ffdf15d...,29978054d13bcab95697cf6f1a28c312aaee8cde58bc96...
8,9,CANON-000129,INV-SYN-000129,TPL-07,validation,id,IDR,6,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,5241eae01d900eb94c67115c955f75f67b457651a7f237...,7fbf2909f37c202543a9248c87acd7b78d596b9ac091d2...,bb69c261567a62a45d6959e7cdbe8df44a80b5b2a129a6...
9,10,CANON-000130,INV-SYN-000130,TPL-07,validation,id,IDR,5,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,3380045ab49e30e7f6acb2a971400eb86f55c6f8bf6431...,fbfc715211ef43a89b606de24a350d06d44be0c08995f9...,23a10067d8c09c07d7971a885d557f28c558e917252c09...



Cell version         : 1.1.0
Baseline ID          : RULE-BASED-INVOICE-PARSER-V1@1.0.2
Parser version       : 1.0.2
Freeze status        : FROZEN_FOR_VALIDATION
Validation documents : 40
Templates            : {'TPL-07': 20, 'TPL-08': 20}
Languages            : {'en': 20, 'id': 20}
Currencies           : {'EUR': 6, 'GBP': 4, 'IDR': 16, 'USD': 14}
Selection action     : CREATED
Selection manifest   : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/field_extraction/validation/rule_based_baseline_v1_0_2/validation_cohort_manifest.json
Selection SHA-256    : 7c49d4bb1d97de5fea8e24c45591473fbbe0f47b09d27be3f023b09febba6194
Ground truth opened  : 0
Validation GT opened : 0
Test opened          : 0
OCR executions       : 0
Dataset modifications: 0
Source modifications : 0

✅ CELL 11A PASSED — seluruh 40 dokumen validation TPL-07 dan TPL-08 telah dipetakan serta diverifikasi tanpa membuka ground truth. Lanjutkan ke Cell 11B.


In [ ]:
from __future__ import annotations

import hashlib
import json
import os
from collections import Counter, defaultdict
from pathlib import Path

import fitz
import pandas as pd
from IPython.display import display


# ============================================================
# CELL 11B — VALIDATION UNIFIED DOCUMENT TEXT LAYER
#             (GROUND TRUTH CLOSED, TEST LOCKED)
# ============================================================

CELL_VERSION = "1.0.0"

DATA_ROOT = Path(
    "/content/drive/MyDrive/InvoiceFlow-AI-Data"
)
BUILD_ROOT = (
    DATA_ROOT
    / "interim"
    / "synthetic_v1_build"
    / "20260904T150025Z"
)
FIELD_ROOT = BUILD_ROOT / "ocr_benchmark" / "field_extraction"

VALIDATION_ROOT = (
    FIELD_ROOT / "validation" / "rule_based_baseline_v1_0_2"
)
VALIDATION_SELECTION_PATH = (
    VALIDATION_ROOT / "validation_cohort_manifest.json"
)
VALIDATION_BASELINE_POINTER_PATH = (
    FIELD_ROOT / "validation_baseline_pointer.json"
)

TEXT_LAYER_ROOT = VALIDATION_ROOT / "text_layers"
SUMMARY_PATH = (
    VALIDATION_ROOT / "validation_unified_text_layer_summary.csv"
)
MANIFEST_PATH = (
    VALIDATION_ROOT / "validation_unified_text_layer_manifest.json"
)

EXPECTED_BASELINE_ID = "RULE-BASED-INVOICE-PARSER-V1@1.0.2"
EXPECTED_PARSER_VERSION = "1.0.2"
EXPECTED_DOCUMENTS = 40
EXPECTED_TEMPLATES = {"TPL-07", "TPL-08"}
EXPECTED_LANGUAGES = {"en", "id"}
MINIMUM_NATIVE_WORDS = 10
MINIMUM_NATIVE_CHARACTERS = 50


# ============================================================
# FILE, HASH, AND IMMUTABLE-WRITE HELPERS
# ============================================================

def require_file(path: Path, label: str) -> None:
    if not path.is_file():
        raise FileNotFoundError(f"{label} tidak ditemukan: {path}")
    if path.stat().st_size <= 0:
        raise RuntimeError(f"{label} kosong: {path}")


def load_json(path: Path) -> dict:
    require_file(path, "JSON artifact")
    with path.open("r", encoding="utf-8") as file_handle:
        value = json.load(file_handle)
    if not isinstance(value, dict):
        raise TypeError(f"Root JSON bukan object: {path}")
    return value


def canonical_json(value: object) -> str:
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as file_handle:
        for chunk in iter(lambda: file_handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def atomic_write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(
        f".{path.name}.{os.getpid()}.tmp"
    )
    try:
        temporary_path.write_text(text, encoding="utf-8")
        os.replace(temporary_path, path)
    finally:
        if temporary_path.exists():
            temporary_path.unlink()


def atomic_write_json(path: Path, value: dict) -> None:
    atomic_write_text(
        path,
        json.dumps(value, indent=2, ensure_ascii=False) + "\n",
    )


def save_immutable_json(path: Path, value: dict) -> str:
    if path.exists():
        existing = load_json(path)
        if canonical_json(existing) != canonical_json(value):
            raise RuntimeError(
                f"Checkpoint sudah ada tetapi berbeda: {path}"
            )
        return "RECOVERED"

    atomic_write_json(path, value)
    return "CREATED"


def save_immutable_text(path: Path, text: str) -> str:
    if path.exists():
        existing = path.read_text(encoding="utf-8")
        if existing != text:
            raise RuntimeError(
                f"Checkpoint sudah ada tetapi berbeda: {path}"
            )
        return "RECOVERED"

    atomic_write_text(path, text)
    return "CREATED"


def round_float(value: float) -> float:
    return round(float(value), 6)


# ============================================================
# NATIVE PDF TEXT EXTRACTION
# Keep this representation identical to the development layer.
# ============================================================

def extract_native_layer(pdf_path: Path) -> dict:
    tokens = []
    lines = []
    page_records = []
    global_token_index = 0
    global_line_index = 0

    with fitz.open(str(pdf_path)) as document:
        if document.needs_pass:
            raise RuntimeError(f"PDF terenkripsi: {pdf_path}")

        for page_index, page in enumerate(document):
            width = float(page.rect.width)
            height = float(page.rect.height)
            page_number = page_index + 1

            if width <= 0 or height <= 0:
                raise RuntimeError(
                    f"Dimensi halaman tidak valid: {pdf_path}, "
                    f"page={page_number}"
                )

            page_records.append(
                {
                    "page_number": page_number,
                    "width_points": round_float(width),
                    "height_points": round_float(height),
                    "rotation": int(page.rotation),
                }
            )

            word_rows = page.get_text("words", sort=True)
            grouped_lines = defaultdict(list)

            for word_row in word_rows:
                if len(word_row) < 8:
                    continue

                x0, y0, x1, y1 = map(float, word_row[:4])
                text = str(word_row[4]).strip()
                block_number = int(word_row[5])
                line_number = int(word_row[6])
                word_number = int(word_row[7])

                if not text:
                    continue

                token = {
                    "token_index": global_token_index,
                    "page_number": page_number,
                    "block_number": block_number,
                    "line_number": line_number,
                    "word_number": word_number,
                    "text": text,
                    "bbox_points": [
                        round_float(x0),
                        round_float(y0),
                        round_float(x1),
                        round_float(y1),
                    ],
                    "bbox_normalized": [
                        round_float(x0 / width),
                        round_float(y0 / height),
                        round_float(x1 / width),
                        round_float(y1 / height),
                    ],
                    "confidence": 1.0,
                    "source": "PYMUPDF_NATIVE_TEXT",
                }

                tokens.append(token)
                grouped_lines[(block_number, line_number)].append(token)
                global_token_index += 1

            sorted_line_groups = sorted(
                grouped_lines.items(),
                key=lambda item: (
                    min(token["bbox_points"][1] for token in item[1]),
                    min(token["bbox_points"][0] for token in item[1]),
                ),
            )

            for (block_number, line_number), line_tokens in sorted_line_groups:
                line_tokens = sorted(
                    line_tokens,
                    key=lambda token: (
                        token["word_number"],
                        token["bbox_points"][0],
                    ),
                )

                x0 = min(token["bbox_points"][0] for token in line_tokens)
                y0 = min(token["bbox_points"][1] for token in line_tokens)
                x1 = max(token["bbox_points"][2] for token in line_tokens)
                y1 = max(token["bbox_points"][3] for token in line_tokens)

                lines.append(
                    {
                        "line_index": global_line_index,
                        "page_number": page_number,
                        "block_number": block_number,
                        "line_number": line_number,
                        "text": " ".join(
                            token["text"] for token in line_tokens
                        ),
                        "token_indexes": [
                            token["token_index"] for token in line_tokens
                        ],
                        "bbox_points": [x0, y0, x1, y1],
                        "bbox_normalized": [
                            round_float(x0 / width),
                            round_float(y0 / height),
                            round_float(x1 / width),
                            round_float(y1 / height),
                        ],
                        "confidence": 1.0,
                        "source": "PYMUPDF_NATIVE_TEXT",
                    }
                )
                global_line_index += 1

    full_text = "\n".join(line["text"] for line in lines).strip()
    nonspace_characters = sum(
        not character.isspace() for character in full_text
    )
    usable = bool(
        len(tokens) >= MINIMUM_NATIVE_WORDS
        and nonspace_characters >= MINIMUM_NATIVE_CHARACTERS
    )

    return {
        "usable": usable,
        "pages": page_records,
        "tokens": tokens,
        "lines": lines,
        "full_text": full_text,
        "metrics": {
            "page_count": len(page_records),
            "token_count": len(tokens),
            "line_count": len(lines),
            "nonspace_character_count": nonspace_characters,
        },
    }


# ============================================================
# PREFLIGHT — ONLY FROZEN METADATA; NO GROUND TRUTH
# ============================================================

for required_path, label in (
    (VALIDATION_SELECTION_PATH, "Validation cohort manifest"),
    (VALIDATION_BASELINE_POINTER_PATH, "Validation baseline pointer"),
):
    require_file(required_path, label)

selection_manifest = load_json(VALIDATION_SELECTION_PATH)
baseline_pointer = load_json(VALIDATION_BASELINE_POINTER_PATH)

freeze_manifest_path = Path(
    str(baseline_pointer.get("freeze_manifest_path", ""))
)
require_file(freeze_manifest_path, "Frozen baseline manifest")
freeze_manifest = load_json(freeze_manifest_path)

selection_records = selection_manifest.get("records")
if not isinstance(selection_records, list):
    raise TypeError("records pada validation cohort manifest bukan list.")

source_checksums_before = {
    "validation_selection": sha256_file(VALIDATION_SELECTION_PATH),
    "validation_baseline_pointer": sha256_file(
        VALIDATION_BASELINE_POINTER_PATH
    ),
    "freeze_manifest": sha256_file(freeze_manifest_path),
}

selection_scope = selection_manifest.get("scope", {})
selection_integrity = selection_manifest.get("integrity", {})

preflight_values = [
    ("selection_status", "READY", selection_manifest.get("status")),
    (
        "selection_stage",
        "VALIDATION_COHORT_PREFLIGHT",
        selection_manifest.get("stage"),
    ),
    ("selection_split", "validation", selection_scope.get("split")),
    (
        "selection_documents",
        EXPECTED_DOCUMENTS,
        selection_scope.get("documents"),
    ),
    (
        "selection_templates",
        sorted(EXPECTED_TEMPLATES),
        sorted(selection_scope.get("templates", [])),
    ),
    (
        "selection_baseline_id",
        EXPECTED_BASELINE_ID,
        selection_manifest.get("baseline", {}).get("baseline_id"),
    ),
    (
        "baseline_pointer_status",
        "ACTIVE_FOR_VALIDATION",
        baseline_pointer.get("status"),
    ),
    (
        "baseline_pointer_id",
        EXPECTED_BASELINE_ID,
        baseline_pointer.get("baseline_id"),
    ),
    (
        "parser_version",
        EXPECTED_PARSER_VERSION,
        baseline_pointer.get("parser_version"),
    ),
    (
        "freeze_status",
        "FROZEN_FOR_VALIDATION",
        freeze_manifest.get("status"),
    ),
    (
        "freeze_checksum",
        baseline_pointer.get("freeze_manifest_sha256"),
        source_checksums_before["freeze_manifest"],
    ),
    (
        "ground_truth_loaded",
        False,
        selection_integrity.get("ground_truth_loaded"),
    ),
    (
        "canonical_payload_loaded",
        False,
        selection_integrity.get("canonical_payload_loaded"),
    ),
    (
        "validation_ground_truth_opened",
        0,
        selection_integrity.get("validation_ground_truth_opened"),
    ),
    ("test_opened", 0, selection_integrity.get("test_opened")),
    (
        "test_locked",
        True,
        baseline_pointer.get("test_remains_locked"),
    ),
]

preflight_controls = [
    {
        "control": name,
        "expected": expected,
        "actual": actual,
        "status": "VALID" if expected == actual else "INVALID",
    }
    for name, expected, actual in preflight_values
]
display(pd.DataFrame(preflight_controls))

invalid_preflight = [
    row["control"]
    for row in preflight_controls
    if row["status"] != "VALID"
]
if invalid_preflight:
    raise RuntimeError(
        "CELL 11B PREFLIGHT FAILED. "
        f"Kontrol tidak valid: {invalid_preflight}"
    )


# ============================================================
# VERIFY THE 40 FROZEN VALIDATION SOURCES
# ============================================================

source_records = []
source_errors = []

for record_index, source_record in enumerate(selection_records, start=1):
    try:
        if not isinstance(source_record, dict):
            raise TypeError("Cohort record bukan object.")

        document_id = str(source_record.get("document_id", "")).strip()
        template_id = str(source_record.get("template_id", "")).strip()
        split = str(source_record.get("split", "")).strip().casefold()
        language = str(source_record.get("language", "")).strip().casefold()
        pdf_path = Path(str(source_record.get("pdf_path", "")))
        expected_pdf_sha256 = str(
            source_record.get("pdf_sha256", "")
        ).casefold()

        if not document_id:
            raise RuntimeError("document_id kosong.")
        if template_id not in EXPECTED_TEMPLATES:
            raise RuntimeError(
                f"Template di luar validation cohort: {template_id}"
            )
        if split != "validation":
            raise RuntimeError(f"Split bukan validation: {split}")
        if language not in EXPECTED_LANGUAGES:
            raise RuntimeError(f"Language tidak didukung: {language}")

        require_file(pdf_path, f"Validation PDF {document_id}")
        actual_pdf_sha256 = sha256_file(pdf_path)
        if actual_pdf_sha256.casefold() != expected_pdf_sha256:
            raise RuntimeError(
                f"Checksum PDF berubah untuk {document_id}: "
                f"expected={expected_pdf_sha256}, "
                f"actual={actual_pdf_sha256}"
            )

        source_records.append(
            {
                **source_record,
                "sequence_number": int(
                    source_record.get("sequence_number", record_index)
                ),
                "document_id": document_id,
                "template_id": template_id,
                "split": split,
                "language": language,
                "pdf_path": str(pdf_path),
                "pdf_sha256": actual_pdf_sha256,
            }
        )
    except Exception as error:
        source_errors.append(
            {
                "record_index": record_index,
                "error_type": type(error).__name__,
                "error": str(error)[:700],
            }
        )

if source_errors:
    display(pd.DataFrame(source_errors))
    raise RuntimeError(
        f"CELL 11B SOURCE VERIFICATION FAILED: "
        f"{len(source_errors)} source tidak valid."
    )

source_records.sort(key=lambda row: int(row["sequence_number"]))

source_gate_values = [
    ("source_records", EXPECTED_DOCUMENTS, len(source_records)),
    (
        "unique_document_ids",
        EXPECTED_DOCUMENTS,
        len({row["document_id"] for row in source_records}),
    ),
    (
        "source_templates",
        sorted(EXPECTED_TEMPLATES),
        sorted({row["template_id"] for row in source_records}),
    ),
    (
        "source_languages",
        sorted(EXPECTED_LANGUAGES),
        sorted({row["language"] for row in source_records}),
    ),
    (
        "sequence_numbers",
        list(range(1, EXPECTED_DOCUMENTS + 1)),
        [int(row["sequence_number"]) for row in source_records],
    ),
    (
        "pdf_files",
        EXPECTED_DOCUMENTS,
        sum(Path(row["pdf_path"]).is_file() for row in source_records),
    ),
    (
        "unique_pdf_checksums",
        EXPECTED_DOCUMENTS,
        len({row["pdf_sha256"] for row in source_records}),
    ),
]

source_gate_controls = [
    {
        "control": name,
        "expected": expected,
        "actual": actual,
        "status": "VALID" if expected == actual else "INVALID",
    }
    for name, expected, actual in source_gate_values
]
display(pd.DataFrame(source_gate_controls))

invalid_source_gates = [
    row["control"]
    for row in source_gate_controls
    if row["status"] != "VALID"
]
if invalid_source_gates:
    raise RuntimeError(
        "CELL 11B SOURCE GATE FAILED. "
        f"Kontrol tidak valid: {invalid_source_gates}"
    )

pdf_checksums_before = {
    row["document_id"]: row["pdf_sha256"] for row in source_records
}


# ============================================================
# BUILD OR RECOVER VALIDATION TEXT LAYERS
# ============================================================

runtime_records = []
manifest_records = []
processing_errors = []
newly_created = 0
recovered = 0

print()
print("=" * 88)
print(
    f"CELL 11B — VALIDATION UNIFIED TEXT LAYER — "
    f"VERSION {CELL_VERSION}"
)
print(f"Text-layer root: {TEXT_LAYER_ROOT}")
print("Ground truth: CLOSED | Test: LOCKED | OCR executions: 0")
print("=" * 88)
print(f"Membangun text layer untuk {len(source_records)} dokumen validation...\n")

for position, source in enumerate(source_records, start=1):
    document_id = source["document_id"]
    template_id = source["template_id"]
    pdf_path = Path(source["pdf_path"])
    result_path = (
        TEXT_LAYER_ROOT
        / template_id
        / f"{document_id}_text_layer.json"
    )

    try:
        native_layer = extract_native_layer(pdf_path)

        # The frozen validation PDFs are digital PDFs. A non-usable native
        # layer is therefore a source/routing exception and must not be hidden
        # by changing engines during the blind validation run.
        if not native_layer["usable"]:
            metrics = native_layer["metrics"]
            raise RuntimeError(
                "OCR_FALLBACK_REQUIRED: native text berada di bawah "
                f"threshold untuk {document_id}; "
                f"tokens={metrics['token_count']}, "
                f"nonspace_characters="
                f"{metrics['nonspace_character_count']}. "
                "Hentikan validation dan audit routing; jangan membuka "
                "ground truth."
            )

        route_id = "NATIVE_PDF_TEXT"
        engine = "PyMuPDF"

        text_layer = {
            "schema_version": "1.0.0",
            "status": "PASSED",
            "document": {
                "canonical_invoice_id": source.get(
                    "canonical_invoice_id"
                ),
                "document_id": document_id,
                "template_id": template_id,
                "split": "validation",
                "language": source.get("language"),
                "currency": source.get("currency"),
                "item_count": source.get("item_count"),
            },
            "source": {
                "pdf_path": str(pdf_path),
                "pdf_sha256": source["pdf_sha256"],
                "validation_selection_path": str(
                    VALIDATION_SELECTION_PATH
                ),
                "validation_selection_sha256": source_checksums_before[
                    "validation_selection"
                ],
                "fallback_artifact": None,
            },
            "routing": {
                "route_id": route_id,
                "engine": engine,
                "preprocessing": "NONE",
                "ocr_executed_in_this_cell": False,
            },
            "pages": native_layer["pages"],
            "tokens": native_layer["tokens"],
            "lines": native_layer["lines"],
            "full_text": native_layer["full_text"],
            "metrics": native_layer["metrics"],
            "integrity": {
                "ground_truth_loaded": False,
                "canonical_payload_loaded": False,
                "validation_ground_truth_opened": 0,
                "test_opened": 0,
                "dataset_modifications": 0,
                "source_pdf_modifications": 0,
            },
        }

        checkpoint_action = save_immutable_json(result_path, text_layer)
        if checkpoint_action == "CREATED":
            newly_created += 1
            execution = "NEW"
        else:
            recovered += 1
            execution = "RECOVERED"

        persisted = load_json(result_path)
        if canonical_json(persisted) != canonical_json(text_layer):
            raise RuntimeError(
                f"Text layer berbeda setelah penulisan: {document_id}"
            )

        metrics = text_layer["metrics"]
        manifest_record = {
            "sequence_number": int(source["sequence_number"]),
            "document_id": document_id,
            "template_id": template_id,
            "language": source.get("language"),
            "route_id": route_id,
            "engine": engine,
            "page_count": int(metrics["page_count"]),
            "token_count": int(metrics["token_count"]),
            "line_count": int(metrics["line_count"]),
            "nonspace_character_count": int(
                metrics["nonspace_character_count"]
            ),
            "pdf_path": str(pdf_path),
            "pdf_sha256": source["pdf_sha256"],
            "text_layer_path": str(result_path),
            "text_layer_sha256": sha256_file(result_path),
            "status": "PASSED",
        }
        manifest_records.append(manifest_record)
        runtime_records.append(
            {**manifest_record, "execution": execution}
        )

        print(
            f"[{position:02d}/{len(source_records):02d}] "
            f"{document_id} | route={route_id} | "
            f"tokens={metrics['token_count']} | "
            f"lines={metrics['line_count']} | {execution}"
        )

    except Exception as error:
        processing_errors.append(
            {
                "sequence_number": source.get("sequence_number"),
                "document_id": document_id,
                "template_id": template_id,
                "error_type": type(error).__name__,
                "error": str(error)[:900],
            }
        )
        print(
            f"[{position:02d}/{len(source_records):02d}] "
            f"{document_id} | ERROR: "
            f"{type(error).__name__}: {error}"
        )


# ============================================================
# RESULT GATES
# ============================================================

if processing_errors:
    print("\nPROCESSING ERRORS")
    display(pd.DataFrame(processing_errors))

runtime_table = pd.DataFrame(runtime_records)
manifest_table = pd.DataFrame(manifest_records)
route_counts = Counter(
    row["route_id"] for row in manifest_records
)

result_values = [
    ("result_records", EXPECTED_DOCUMENTS, len(manifest_records)),
    (
        "result_files",
        EXPECTED_DOCUMENTS,
        sum(
            Path(row["text_layer_path"]).is_file()
            for row in manifest_records
        ),
    ),
    (
        "unique_document_ids",
        EXPECTED_DOCUMENTS,
        len({row["document_id"] for row in manifest_records}),
    ),
    (
        "templates",
        sorted(EXPECTED_TEMPLATES),
        sorted({row["template_id"] for row in manifest_records}),
    ),
    (
        "native_text_routes",
        EXPECTED_DOCUMENTS,
        route_counts.get("NATIVE_PDF_TEXT", 0),
    ),
    (
        "ocr_fallback_routes",
        0,
        sum(
            row["route_id"] != "NATIVE_PDF_TEXT"
            for row in manifest_records
        ),
    ),
    (
        "single_page_documents",
        EXPECTED_DOCUMENTS,
        sum(row["page_count"] == 1 for row in manifest_records),
    ),
    (
        "usable_native_layers",
        EXPECTED_DOCUMENTS,
        sum(
            row["token_count"] >= MINIMUM_NATIVE_WORDS
            and row["nonspace_character_count"]
            >= MINIMUM_NATIVE_CHARACTERS
            for row in manifest_records
        ),
    ),
    ("processing_errors", 0, len(processing_errors)),
    ("ground_truth_opened", 0, 0),
    ("validation_ground_truth_opened", 0, 0),
    ("test_opened", 0, 0),
    ("ocr_executions", 0, 0),
]

result_controls = [
    {
        "control": name,
        "expected": expected,
        "actual": actual,
        "status": "VALID" if expected == actual else "INVALID",
    }
    for name, expected, actual in result_values
]

print("\nRESULT CONTROLS")
display(pd.DataFrame(result_controls))
if not runtime_table.empty:
    display(runtime_table)

invalid_result_controls = [
    row["control"]
    for row in result_controls
    if row["status"] != "VALID"
]
if invalid_result_controls:
    raise RuntimeError(
        "CELL 11B TEXT-LAYER BUILD FAILED. "
        f"Kontrol tidak valid: {invalid_result_controls}"
    )


# ============================================================
# IMMUTABLE SUMMARY AND MANIFEST
# ============================================================

summary_columns = [
    "sequence_number",
    "document_id",
    "template_id",
    "language",
    "route_id",
    "engine",
    "page_count",
    "token_count",
    "line_count",
    "nonspace_character_count",
    "pdf_path",
    "pdf_sha256",
    "text_layer_path",
    "text_layer_sha256",
    "status",
]

summary_text = manifest_table[summary_columns].to_csv(
    index=False,
    lineterminator="\n",
)
summary_action = save_immutable_text(SUMMARY_PATH, summary_text)

text_layer_manifest = {
    "schema_version": "1.0.0",
    "cell_version": CELL_VERSION,
    "status": "PASSED",
    "stage": "VALIDATION_UNIFIED_DOCUMENT_TEXT_LAYER",
    "baseline": {
        "baseline_id": baseline_pointer["baseline_id"],
        "parser_id": baseline_pointer["parser_id"],
        "parser_version": baseline_pointer["parser_version"],
        "freeze_manifest_path": str(freeze_manifest_path),
        "freeze_manifest_sha256": source_checksums_before[
            "freeze_manifest"
        ],
    },
    "scope": {
        "split": "validation",
        "documents": EXPECTED_DOCUMENTS,
        "templates": sorted(EXPECTED_TEMPLATES),
        "languages": sorted(EXPECTED_LANGUAGES),
        "ground_truth_opened": 0,
        "validation_ground_truth_opened": 0,
        "test_opened": 0,
    },
    "routing": {
        "native_pdf_engine": "PyMuPDF",
        "minimum_native_words": MINIMUM_NATIVE_WORDS,
        "minimum_native_characters": MINIMUM_NATIVE_CHARACTERS,
        "route_distribution": dict(sorted(route_counts.items())),
        "ocr_fallback_policy": "STOP_AND_AUDIT",
        "ocr_executions_in_this_cell": 0,
    },
    "records": manifest_records,
    "artifacts": {
        "summary_path": str(SUMMARY_PATH),
        "summary_sha256": sha256_file(SUMMARY_PATH),
        "text_layer_root": str(TEXT_LAYER_ROOT),
    },
    "input_artifacts": {
        "validation_selection": {
            "path": str(VALIDATION_SELECTION_PATH),
            "sha256": source_checksums_before[
                "validation_selection"
            ],
        },
        "validation_baseline_pointer": {
            "path": str(VALIDATION_BASELINE_POINTER_PATH),
            "sha256": source_checksums_before[
                "validation_baseline_pointer"
            ],
        },
        "freeze_manifest": {
            "path": str(freeze_manifest_path),
            "sha256": source_checksums_before["freeze_manifest"],
        },
    },
    "integrity": {
        "ground_truth_loaded": False,
        "canonical_payload_loaded": False,
        "validation_ground_truth_opened": 0,
        "test_opened": 0,
        "ocr_executions": 0,
        "dataset_modifications": 0,
        "source_pdf_modifications": 0,
        "source_manifest_modifications": 0,
    },
    "next_stage": {
        "cell": "CELL 11C",
        "action": "RUN_FROZEN_PARSER_ON_VALIDATION_TEXT_LAYERS",
        "parser_mutation_allowed": False,
        "ground_truth_must_remain_closed": True,
        "test_remains_locked": True,
    },
}

manifest_action = save_immutable_json(
    MANIFEST_PATH,
    text_layer_manifest,
)

persisted_manifest = load_json(MANIFEST_PATH)
if canonical_json(persisted_manifest) != canonical_json(text_layer_manifest):
    raise RuntimeError("Manifest Cell 11B berbeda setelah penulisan.")


# ============================================================
# FINAL SOURCE IMMUTABILITY VERIFICATION
# ============================================================

source_checksums_after = {
    "validation_selection": sha256_file(VALIDATION_SELECTION_PATH),
    "validation_baseline_pointer": sha256_file(
        VALIDATION_BASELINE_POINTER_PATH
    ),
    "freeze_manifest": sha256_file(freeze_manifest_path),
}

changed_source_manifests = [
    name
    for name, checksum in source_checksums_before.items()
    if source_checksums_after.get(name) != checksum
]

changed_pdfs = []
for source in source_records:
    current_checksum = sha256_file(Path(source["pdf_path"]))
    if current_checksum != pdf_checksums_before[source["document_id"]]:
        changed_pdfs.append(source["document_id"])

if changed_source_manifests or changed_pdfs:
    raise RuntimeError(
        "Source berubah selama Cell 11B: "
        f"manifests={changed_source_manifests}, "
        f"pdfs={changed_pdfs}"
    )


print()
print(f"Cell version          : {CELL_VERSION}")
print(f"Baseline ID           : {baseline_pointer['baseline_id']}")
print(f"Parser version        : {baseline_pointer['parser_version']}")
print(f"Validation documents  : {len(manifest_records)}")
print(
    "Templates             : "
    f"{dict(sorted(Counter(row['template_id'] for row in manifest_records).items()))}"
)
print(
    "Languages             : "
    f"{dict(sorted(Counter(row['language'] for row in manifest_records).items()))}"
)
print(
    "Native text routes    : "
    f"{route_counts.get('NATIVE_PDF_TEXT', 0)}"
)
print("OCR fallback routes   : 0")
print(f"New text layers       : {newly_created}")
print(f"Recovered text layers : {recovered}")
print(f"Summary action        : {summary_action}")
print(f"Manifest action       : {manifest_action}")
print(f"Text-layer root       : {TEXT_LAYER_ROOT}")
print(f"Summary               : {SUMMARY_PATH}")
print(f"Manifest              : {MANIFEST_PATH}")
print(f"Manifest SHA-256      : {sha256_file(MANIFEST_PATH)}")
print("Ground truth opened   : 0")
print("Validation GT opened  : 0")
print("Test opened           : 0")
print("OCR executions        : 0")
print("Dataset modifications : 0")
print("Source modifications  : 0")
print()
print(
    "✅ CELL 11B PASSED — unified text layer untuk 40 dokumen "
    "validation berhasil dibuat secara immutable tanpa membuka "
    "ground truth atau test. Lanjutkan ke Cell 11C."
)


,control,expected,actual,status
0,selection_status,READY,READY,VALID
1,selection_stage,VALIDATION_COHORT_PREFLIGHT,VALIDATION_COHORT_PREFLIGHT,VALID
2,selection_split,validation,validation,VALID
3,selection_documents,40,40,VALID
4,selection_templates,"[TPL-07, TPL-08]","[TPL-07, TPL-08]",VALID
5,selection_baseline_id,RULE-BASED-INVOICE-PARSER-V1@1.0.2,RULE-BASED-INVOICE-PARSER-V1@1.0.2,VALID
6,baseline_pointer_status,ACTIVE_FOR_VALIDATION,ACTIVE_FOR_VALIDATION,VALID
7,baseline_pointer_id,RULE-BASED-INVOICE-PARSER-V1@1.0.2,RULE-BASED-INVOICE-PARSER-V1@1.0.2,VALID
8,parser_version,1.0.2,1.0.2,VALID
9,freeze_status,FROZEN_FOR_VALIDATION,FROZEN_FOR_VALIDATION,VALID


,control,expected,actual,status
0,source_records,40,40,VALID
1,unique_document_ids,40,40,VALID
2,source_templates,"[TPL-07, TPL-08]","[TPL-07, TPL-08]",VALID
3,source_languages,"[en, id]","[en, id]",VALID
4,sequence_numbers,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",VALID
5,pdf_files,40,40,VALID
6,unique_pdf_checksums,40,40,VALID



CELL 11B — VALIDATION UNIFIED TEXT LAYER — VERSION 1.0.0
Text-layer root: /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/field_extraction/validation/rule_based_baseline_v1_0_2/text_layers
Ground truth: CLOSED | Test: LOCKED | OCR executions: 0
Membangun text layer untuk 40 dokumen validation...

[01/40] INV-SYN-000121 | route=NATIVE_PDF_TEXT | tokens=159 | lines=78 | NEW
[02/40] INV-SYN-000122 | route=NATIVE_PDF_TEXT | tokens=160 | lines=78 | NEW
[03/40] INV-SYN-000123 | route=NATIVE_PDF_TEXT | tokens=125 | lines=58 | NEW
[04/40] INV-SYN-000124 | route=NATIVE_PDF_TEXT | tokens=131 | lines=63 | NEW
[05/40] INV-SYN-000125 | route=NATIVE_PDF_TEXT | tokens=133 | lines=63 | NEW
[06/40] INV-SYN-000126 | route=NATIVE_PDF_TEXT | tokens=117 | lines=53 | NEW
[07/40] INV-SYN-000127 | route=NATIVE_PDF_TEXT | tokens=124 | lines=58 | NEW
[08/40] INV-SYN-000128 | route=NATIVE_PDF_TEXT | tokens=133 | lines=63 | NEW
[09/40] INV-SYN-000129 | route=N

,control,expected,actual,status
0,result_records,40,40,VALID
1,result_files,40,40,VALID
2,unique_document_ids,40,40,VALID
3,templates,"[TPL-07, TPL-08]","[TPL-07, TPL-08]",VALID
4,native_text_routes,40,40,VALID
5,ocr_fallback_routes,0,0,VALID
6,single_page_documents,40,40,VALID
7,usable_native_layers,40,40,VALID
8,processing_errors,0,0,VALID
9,ground_truth_opened,0,0,VALID


,sequence_number,document_id,template_id,language,route_id,engine,page_count,token_count,line_count,nonspace_character_count,pdf_path,pdf_sha256,text_layer_path,text_layer_sha256,status,execution
0,1,INV-SYN-000121,TPL-07,id,NATIVE_PDF_TEXT,PyMuPDF,1,159,78,957,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,4f9c017d08ff54340c3d935a1c7fae9f870e818ce98a18...,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,a4025c852e28bd2e2f846ca65964cceaa4ce4eb2e19330...,PASSED,NEW
1,2,INV-SYN-000122,TPL-07,id,NATIVE_PDF_TEXT,PyMuPDF,1,160,78,952,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,795cf4ddafa79837038cd7100258efb328d98df335b028...,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,ea789afc0feabbd4624ffb2630ac416bbaa8845efa7c41...,PASSED,NEW
2,3,INV-SYN-000123,TPL-07,id,NATIVE_PDF_TEXT,PyMuPDF,1,125,58,765,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,c1e2352a30dbc47ea05f84ed964de5cb2e02c4ab167cd6...,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,6646d11e5ea90fcb0a1da19aabf792097d4d8daf43da4b...,PASSED,NEW
3,4,INV-SYN-000124,TPL-07,id,NATIVE_PDF_TEXT,PyMuPDF,1,131,63,788,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,53c96910164a02a80174df1b57ec543bc16c5e1c2b9a4e...,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,7d7172b81636e02e7a1b953b5e0922f16216995c2a7d30...,PASSED,NEW
4,5,INV-SYN-000125,TPL-07,id,NATIVE_PDF_TEXT,PyMuPDF,1,133,63,822,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,8e6bfdcd5f8fe53c08f1b3260dc199d5d4b70c894dfef1...,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,3033151f97d591cb23ad04619f39a9180d156e3aa2d74b...,PASSED,NEW
5,6,INV-SYN-000126,TPL-07,id,NATIVE_PDF_TEXT,PyMuPDF,1,117,53,706,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,28d5b04eca6230a052602aab81b2823f4734f1778f99ee...,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,25c8778f213fd92eb4187a7c0236a7ab93a9e905105ce7...,PASSED,NEW
6,7,INV-SYN-000127,TPL-07,id,NATIVE_PDF_TEXT,PyMuPDF,1,124,58,743,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,740924b9285b7537d6dee56a9da66311f54397cc3854bf...,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,2a0a88b750bd61f922c02538ebd6c585c620bd43bd8e7c...,PASSED,NEW
7,8,INV-SYN-000128,TPL-07,id,NATIVE_PDF_TEXT,PyMuPDF,1,133,63,791,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,584923462159cf083fb062f0303cbd3ba352a67efe682f...,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,50faee8b31244e1dd93f0a6c12ebdf398e52931102f952...,PASSED,NEW
8,9,INV-SYN-000129,TPL-07,id,NATIVE_PDF_TEXT,PyMuPDF,1,150,73,904,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,5241eae01d900eb94c67115c955f75f67b457651a7f237...,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,342c0b2ff706f29fe71586ed8b202c447ac60aac64bf31...,PASSED,NEW
9,10,INV-SYN-000130,TPL-07,id,NATIVE_PDF_TEXT,PyMuPDF,1,143,68,873,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,3380045ab49e30e7f6acb2a971400eb86f55c6f8bf6431...,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,bdd79a19621a9c395d16883961a6f0e4d62d5da7fe13df...,PASSED,NEW



Cell version          : 1.0.0
Baseline ID           : RULE-BASED-INVOICE-PARSER-V1@1.0.2
Parser version        : 1.0.2
Validation documents  : 40
Templates             : {'TPL-07': 20, 'TPL-08': 20}
Languages             : {'en': 20, 'id': 20}
Native text routes    : 40
OCR fallback routes   : 0
New text layers       : 40
Recovered text layers : 0
Summary action        : CREATED
Manifest action       : CREATED
Text-layer root       : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/field_extraction/validation/rule_based_baseline_v1_0_2/text_layers
Summary               : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/field_extraction/validation/rule_based_baseline_v1_0_2/validation_unified_text_layer_summary.csv
Manifest              : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/field_extraction/validation/rule_based_baselin

In [ ]:
from __future__ import annotations

import ast
import hashlib
import json
import os
from collections import Counter
from decimal import Decimal
from pathlib import Path

import pandas as pd
from IPython.display import display


# ============================================================
# CELL 11C — RUN FROZEN PARSER ON VALIDATION
#             (BLIND PREDICTION; GROUND TRUTH CLOSED)
# ============================================================

CELL_VERSION = "1.0.0"

DATA_ROOT = Path(
    "/content/drive/MyDrive/InvoiceFlow-AI-Data"
)
BUILD_ROOT = (
    DATA_ROOT
    / "interim"
    / "synthetic_v1_build"
    / "20260904T150025Z"
)
FIELD_ROOT = BUILD_ROOT / "ocr_benchmark" / "field_extraction"

VALIDATION_ROOT = (
    FIELD_ROOT / "validation" / "rule_based_baseline_v1_0_2"
)
VALIDATION_BASELINE_POINTER_PATH = (
    FIELD_ROOT / "validation_baseline_pointer.json"
)
TEXT_LAYER_MANIFEST_PATH = (
    VALIDATION_ROOT / "validation_unified_text_layer_manifest.json"
)

PREDICTION_ROOT = VALIDATION_ROOT / "predictions"
SUMMARY_PATH = VALIDATION_ROOT / "validation_prediction_summary.csv"
MANIFEST_PATH = VALIDATION_ROOT / "validation_prediction_manifest.json"

EXPECTED_BASELINE_ID = "RULE-BASED-INVOICE-PARSER-V1@1.0.2"
EXPECTED_PARSER_ID = "RULE-BASED-INVOICE-PARSER-V1"
EXPECTED_PARSER_VERSION = "1.0.2"
EXPECTED_DOCUMENTS = 40
EXPECTED_TEMPLATES = {"TPL-07", "TPL-08"}
EXPECTED_SCALAR_FIELDS = 12


# ============================================================
# FILE, HASH, AND IMMUTABLE-WRITE HELPERS
# ============================================================

def require_file(path: Path, label: str) -> None:
    if not path.is_file():
        raise FileNotFoundError(f"{label} tidak ditemukan: {path}")
    if path.stat().st_size <= 0:
        raise RuntimeError(f"{label} kosong: {path}")


def load_json(path: Path) -> dict:
    require_file(path, "JSON artifact")
    with path.open("r", encoding="utf-8") as file_handle:
        value = json.load(file_handle)
    if not isinstance(value, dict):
        raise TypeError(f"Root JSON bukan object: {path}")
    return value


def canonical_json(value: object) -> str:
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    )


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as file_handle:
        for chunk in iter(lambda: file_handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def atomic_write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(
        f".{path.name}.{os.getpid()}.tmp"
    )
    try:
        temporary_path.write_text(text, encoding="utf-8")
        os.replace(temporary_path, path)
    finally:
        if temporary_path.exists():
            temporary_path.unlink()


def atomic_write_json(path: Path, value: dict) -> None:
    atomic_write_text(
        path,
        json.dumps(
            value,
            indent=2,
            ensure_ascii=False,
            default=str,
        )
        + "\n",
    )


def save_immutable_json(path: Path, value: dict) -> str:
    if path.exists():
        existing = load_json(path)
        if canonical_json(existing) != canonical_json(value):
            raise RuntimeError(
                f"Checkpoint sudah ada tetapi berbeda: {path}"
            )
        return "RECOVERED"

    atomic_write_json(path, value)
    return "CREATED"


def save_immutable_text(path: Path, text: str) -> str:
    if path.exists():
        existing = path.read_text(encoding="utf-8")
        if existing != text:
            raise RuntimeError(
                f"Checkpoint sudah ada tetapi berbeda: {path}"
            )
        return "RECOVERED"

    atomic_write_text(path, text)
    return "CREATED"


# ============================================================
# LOAD ONLY THE FROZEN PARSER DEFINITION PREFIX
#
# The frozen source is the complete historical Cell 10C. Importing that file
# directly would also execute its old development runner. Instead, this loader
# verifies the complete frozen file checksum, parses it, and executes exactly
# the source prefix through parse_document(). No parser rule is copied,
# rewritten, patched, or selected by validation template.
# ============================================================

def load_frozen_parser(
    parser_source_path: Path,
    expected_source_sha256: str,
) -> tuple[dict, dict]:
    require_file(parser_source_path, "Frozen parser source")

    actual_source_sha256 = sha256_file(parser_source_path)
    if actual_source_sha256 != expected_source_sha256:
        raise RuntimeError(
            "Checksum frozen parser source tidak cocok. "
            f"Expected={expected_source_sha256}, "
            f"actual={actual_source_sha256}"
        )

    source_text = parser_source_path.read_text(encoding="utf-8")
    source_tree = ast.parse(
        source_text,
        filename=str(parser_source_path),
        mode="exec",
    )

    entrypoints = [
        node
        for node in source_tree.body
        if isinstance(node, ast.FunctionDef)
        and node.name == "parse_document"
    ]
    if len(entrypoints) != 1:
        raise RuntimeError(
            "Entrypoint parse_document pada frozen source tidak unik: "
            f"ditemukan {len(entrypoints)}."
        )

    entrypoint = entrypoints[0]
    entrypoint_end_line = int(entrypoint.end_lineno or entrypoint.lineno)
    prefix_nodes = [
        node
        for node in source_tree.body
        if int(node.end_lineno or node.lineno) <= entrypoint_end_line
    ]

    allowed_top_level_types = (
        ast.Import,
        ast.ImportFrom,
        ast.Assign,
        ast.AnnAssign,
        ast.FunctionDef,
    )
    disallowed_nodes = [
        {
            "node_type": type(node).__name__,
            "line": int(node.lineno),
        }
        for node in prefix_nodes
        if not isinstance(node, allowed_top_level_types)
    ]
    if disallowed_nodes:
        raise RuntimeError(
            "Frozen parser definition prefix mengandung top-level statement "
            f"yang tidak diizinkan: {disallowed_nodes}"
        )

    prefix_tree = ast.Module(
        body=prefix_nodes,
        type_ignores=source_tree.type_ignores,
    )
    ast.fix_missing_locations(prefix_tree)

    parser_namespace = {
        "__name__": "frozen_invoice_parser_v1_0_2",
        "__file__": str(parser_source_path),
    }
    compiled_prefix = compile(
        prefix_tree,
        filename=str(parser_source_path),
        mode="exec",
    )
    exec(compiled_prefix, parser_namespace)

    parse_document = parser_namespace.get("parse_document")
    if not callable(parse_document):
        raise RuntimeError(
            "Frozen parser entrypoint parse_document tidak callable."
        )

    loader_audit = {
        "method": "AST_PREFIX_THROUGH_PARSE_DOCUMENT",
        "complete_source_verified_before_load": True,
        "complete_source_sha256": actual_source_sha256,
        "entrypoint": "parse_document",
        "entrypoint_end_line": entrypoint_end_line,
        "loaded_top_level_nodes": len(prefix_nodes),
        "development_runner_executed": False,
        "parser_rule_modifications": 0,
    }
    return parser_namespace, loader_audit


def parser_configuration_from_namespace(namespace: dict) -> dict:
    return {
        "parser_id": namespace["PARSER_ID"],
        "parser_version": namespace["PARSER_VERSION"],
        "supported_currencies": sorted(
            namespace["SUPPORTED_CURRENCIES"]
        ),
        "label_aliases": {
            key: sorted(value)
            for key, value in sorted(
                namespace["LABEL_ALIASES"].items()
            )
        },
        "month_dictionary": dict(
            sorted(namespace["MONTHS"].items())
        ),
        "table_row_tolerance_points": 2.8,
        "party_legal_suffixes": sorted(
            namespace["LEGAL_ENTITY_SUFFIXES"]
        ),
        "party_suffix_geometry": {
            "same_row_tolerance_points": namespace[
                "PARTY_SUFFIX_SAME_ROW_TOLERANCE_POINTS"
            ],
            "next_line_gap_points": namespace[
                "PARTY_SUFFIX_NEXT_LINE_GAP_POINTS"
            ],
            "left_alignment_tolerance_points": namespace[
                "PARTY_SUFFIX_LEFT_ALIGNMENT_TOLERANCE_POINTS"
            ],
            "same_row_gap_points": namespace[
                "PARTY_SUFFIX_SAME_ROW_GAP_POINTS"
            ],
        },
        "ground_truth_as_prediction_input": False,
        "template_specific_branching": False,
    }


# ============================================================
# PREFLIGHT — VERIFY THE FROZEN BASELINE AND TEXT LAYERS
# ============================================================

for required_path, label in (
    (VALIDATION_BASELINE_POINTER_PATH, "Validation baseline pointer"),
    (TEXT_LAYER_MANIFEST_PATH, "Validation text-layer manifest"),
):
    require_file(required_path, label)

baseline_pointer = load_json(VALIDATION_BASELINE_POINTER_PATH)
text_layer_manifest = load_json(TEXT_LAYER_MANIFEST_PATH)

freeze_manifest_path = Path(
    str(baseline_pointer.get("freeze_manifest_path", ""))
)
parser_source_path = Path(
    str(baseline_pointer.get("parser_source_path", ""))
)

require_file(freeze_manifest_path, "Frozen baseline manifest")
require_file(parser_source_path, "Frozen parser source")

freeze_manifest = load_json(freeze_manifest_path)
freeze_parser = freeze_manifest.get("parser", {})
text_scope = text_layer_manifest.get("scope", {})
text_integrity = text_layer_manifest.get("integrity", {})

source_checksums_before = {
    "validation_baseline_pointer": sha256_file(
        VALIDATION_BASELINE_POINTER_PATH
    ),
    "freeze_manifest": sha256_file(freeze_manifest_path),
    "parser_source": sha256_file(parser_source_path),
    "text_layer_manifest": sha256_file(TEXT_LAYER_MANIFEST_PATH),
}

preflight_values = [
    (
        "baseline_pointer_status",
        "ACTIVE_FOR_VALIDATION",
        baseline_pointer.get("status"),
    ),
    (
        "baseline_id",
        EXPECTED_BASELINE_ID,
        baseline_pointer.get("baseline_id"),
    ),
    (
        "parser_id",
        EXPECTED_PARSER_ID,
        baseline_pointer.get("parser_id"),
    ),
    (
        "parser_version",
        EXPECTED_PARSER_VERSION,
        baseline_pointer.get("parser_version"),
    ),
    (
        "next_allowed_split",
        "validation",
        baseline_pointer.get("next_allowed_split"),
    ),
    (
        "test_locked",
        True,
        baseline_pointer.get("test_remains_locked"),
    ),
    (
        "freeze_status",
        "FROZEN_FOR_VALIDATION",
        freeze_manifest.get("status"),
    ),
    (
        "freeze_checksum",
        baseline_pointer.get("freeze_manifest_sha256"),
        source_checksums_before["freeze_manifest"],
    ),
    (
        "parser_source_path",
        str(parser_source_path),
        str(freeze_parser.get("source_path")),
    ),
    (
        "parser_source_checksum_pointer",
        baseline_pointer.get("parser_source_sha256"),
        source_checksums_before["parser_source"],
    ),
    (
        "parser_source_checksum_freeze",
        freeze_parser.get("source_sha256"),
        source_checksums_before["parser_source"],
    ),
    (
        "template_specific_branching",
        False,
        freeze_parser.get("template_specific_branching"),
    ),
    (
        "ground_truth_as_prediction_input",
        False,
        freeze_parser.get("ground_truth_as_prediction_input"),
    ),
    (
        "text_layer_status",
        "PASSED",
        text_layer_manifest.get("status"),
    ),
    (
        "text_layer_stage",
        "VALIDATION_UNIFIED_DOCUMENT_TEXT_LAYER",
        text_layer_manifest.get("stage"),
    ),
    ("text_layer_split", "validation", text_scope.get("split")),
    (
        "text_layer_documents",
        EXPECTED_DOCUMENTS,
        text_scope.get("documents"),
    ),
    (
        "text_layer_templates",
        sorted(EXPECTED_TEMPLATES),
        sorted(text_scope.get("templates", [])),
    ),
    (
        "ground_truth_loaded",
        False,
        text_integrity.get("ground_truth_loaded"),
    ),
    (
        "canonical_payload_loaded",
        False,
        text_integrity.get("canonical_payload_loaded"),
    ),
    (
        "validation_ground_truth_opened",
        0,
        text_integrity.get("validation_ground_truth_opened"),
    ),
    ("test_opened", 0, text_integrity.get("test_opened")),
]

preflight_controls = [
    {
        "control": name,
        "expected": expected,
        "actual": actual,
        "status": "VALID" if expected == actual else "INVALID",
    }
    for name, expected, actual in preflight_values
]
display(pd.DataFrame(preflight_controls))

invalid_preflight = [
    row["control"]
    for row in preflight_controls
    if row["status"] != "VALID"
]
if invalid_preflight:
    raise RuntimeError(
        "CELL 11C PREFLIGHT FAILED. "
        f"Kontrol tidak valid: {invalid_preflight}"
    )


# ============================================================
# LOAD AND VERIFY THE FROZEN PARSER ENTRYPOINT
# ============================================================

parser_namespace, parser_loader_audit = load_frozen_parser(
    parser_source_path,
    baseline_pointer["parser_source_sha256"],
)
parse_document = parser_namespace["parse_document"]

parser_configuration = parser_configuration_from_namespace(
    parser_namespace
)
parser_signature = hashlib.sha256(
    canonical_json(parser_configuration).encode("utf-8")
).hexdigest()
expected_parser_signature = str(
    freeze_parser.get("parser_signature_sha256", "")
)

parser_load_values = [
    (
        "runtime_parser_id",
        EXPECTED_PARSER_ID,
        parser_namespace.get("PARSER_ID"),
    ),
    (
        "runtime_parser_version",
        EXPECTED_PARSER_VERSION,
        parser_namespace.get("PARSER_VERSION"),
    ),
    (
        "runtime_parser_signature",
        expected_parser_signature,
        parser_signature,
    ),
    (
        "parse_document_callable",
        True,
        callable(parse_document),
    ),
    (
        "development_runner_executed",
        False,
        parser_loader_audit["development_runner_executed"],
    ),
    (
        "parser_rule_modifications",
        0,
        parser_loader_audit["parser_rule_modifications"],
    ),
]

parser_load_controls = [
    {
        "control": name,
        "expected": expected,
        "actual": actual,
        "status": "VALID" if expected == actual else "INVALID",
    }
    for name, expected, actual in parser_load_values
]
print("\nFROZEN PARSER LOAD CONTROLS")
display(pd.DataFrame(parser_load_controls))

invalid_parser_load = [
    row["control"]
    for row in parser_load_controls
    if row["status"] != "VALID"
]
if invalid_parser_load:
    raise RuntimeError(
        "CELL 11C FROZEN PARSER LOAD FAILED. "
        f"Kontrol tidak valid: {invalid_parser_load}"
    )


# ============================================================
# VERIFY ALL 40 IMMUTABLE VALIDATION TEXT LAYERS
# ============================================================

text_layer_records = text_layer_manifest.get("records")
if not isinstance(text_layer_records, list):
    raise TypeError("records pada text-layer manifest bukan list.")

source_records = sorted(
    text_layer_records,
    key=lambda row: (
        int(row.get("sequence_number", 0)),
        str(row.get("document_id", "")),
    ),
)

source_errors = []
for record_index, source_record in enumerate(source_records, start=1):
    try:
        if not isinstance(source_record, dict):
            raise TypeError("Text-layer record bukan object.")

        document_id = str(source_record.get("document_id", ""))
        template_id = str(source_record.get("template_id", ""))
        text_layer_path = Path(
            str(source_record.get("text_layer_path", ""))
        )
        expected_checksum = str(
            source_record.get("text_layer_sha256", "")
        )

        if not document_id:
            raise RuntimeError("document_id kosong.")
        if template_id not in EXPECTED_TEMPLATES:
            raise RuntimeError(
                f"Template di luar validation: {template_id}"
            )

        require_file(text_layer_path, f"Text layer {document_id}")
        actual_checksum = sha256_file(text_layer_path)
        if actual_checksum != expected_checksum:
            raise RuntimeError(
                f"Checksum text layer berubah: {document_id}"
            )

        text_layer = load_json(text_layer_path)
        document = text_layer.get("document", {})
        integrity = text_layer.get("integrity", {})

        if text_layer.get("status") != "PASSED":
            raise RuntimeError("Status text layer bukan PASSED.")
        if document.get("document_id") != document_id:
            raise RuntimeError("Document ID text layer tidak cocok.")
        if document.get("template_id") != template_id:
            raise RuntimeError("Template ID text layer tidak cocok.")
        if document.get("split") != "validation":
            raise RuntimeError("Text layer bukan validation split.")
        if integrity.get("ground_truth_loaded") is not False:
            raise RuntimeError("Text layer menandai ground truth telah dibuka.")
        if integrity.get("validation_ground_truth_opened") != 0:
            raise RuntimeError("Validation ground truth telah dibuka.")
        if integrity.get("test_opened") != 0:
            raise RuntimeError("Test telah dibuka.")

        source_checksums_before[
            f"text_layer:{document_id}"
        ] = actual_checksum

    except Exception as error:
        source_errors.append(
            {
                "record_index": record_index,
                "document_id": source_record.get("document_id")
                if isinstance(source_record, dict)
                else None,
                "error_type": type(error).__name__,
                "error": str(error)[:700],
            }
        )

source_gate_values = [
    ("text_layer_records", EXPECTED_DOCUMENTS, len(source_records)),
    (
        "unique_document_ids",
        EXPECTED_DOCUMENTS,
        len({row.get("document_id") for row in source_records}),
    ),
    (
        "validation_templates",
        sorted(EXPECTED_TEMPLATES),
        sorted({row.get("template_id") for row in source_records}),
    ),
    ("source_errors", 0, len(source_errors)),
]

source_gate_controls = [
    {
        "control": name,
        "expected": expected,
        "actual": actual,
        "status": "VALID" if expected == actual else "INVALID",
    }
    for name, expected, actual in source_gate_values
]

print("\nTEXT-LAYER SOURCE CONTROLS")
display(pd.DataFrame(source_gate_controls))
if source_errors:
    display(pd.DataFrame(source_errors))

invalid_source_gates = [
    row["control"]
    for row in source_gate_controls
    if row["status"] != "VALID"
]
if invalid_source_gates:
    raise RuntimeError(
        "CELL 11C TEXT-LAYER SOURCE GATE FAILED. "
        f"Kontrol tidak valid: {invalid_source_gates}"
    )


# ============================================================
# BLIND VALIDATION PREDICTION
# ============================================================

runtime_records = []
manifest_records = []
processing_errors = []
newly_created = 0
recovered = 0

print()
print("=" * 92)
print(
    f"CELL 11C — {EXPECTED_PARSER_ID} — VERSION "
    f"{EXPECTED_PARSER_VERSION} — BLIND VALIDATION"
)
print(f"Prediction root: {PREDICTION_ROOT}")
print("Parser: FROZEN | Ground truth: CLOSED | Test: LOCKED")
print("=" * 92)
print(f"Menjalankan frozen parser untuk {len(source_records)} dokumen...\n")

for position, source_record in enumerate(source_records, start=1):
    document_id = str(source_record["document_id"])
    template_id = str(source_record["template_id"])
    text_layer_path = Path(source_record["text_layer_path"])
    prediction_path = (
        PREDICTION_ROOT
        / template_id
        / f"{document_id}_prediction.json"
    )

    try:
        text_layer = load_json(text_layer_path)
        document = text_layer["document"]

        # Only extracted lines enter the parser. Cohort metadata such as
        # template, expected item count, canonical payload, and ground truth
        # is not available to the parser entrypoint.
        parser_input = {
            "lines": text_layer.get("lines", []),
        }
        parsed = parse_document(parser_input)

        if not isinstance(parsed, dict):
            raise TypeError("Frozen parser output bukan object.")

        scalars = parsed.get("scalar_fields")
        items = parsed.get("items")
        diagnostics = parsed.get("diagnostics")

        if not isinstance(scalars, dict):
            raise TypeError("scalar_fields bukan object.")
        if not isinstance(items, list):
            raise TypeError("items bukan list.")
        if not isinstance(diagnostics, dict):
            raise TypeError("diagnostics bukan object.")
        if len(scalars) != EXPECTED_SCALAR_FIELDS:
            raise RuntimeError(
                f"Jumlah scalar field tidak valid: {len(scalars)}"
            )

        missing_required = list(
            diagnostics.get("missing_required_scalar_fields", [])
        )
        table_diagnostics = diagnostics.get("table", {})
        financial_check = diagnostics.get("financial_equation", {})

        populated_scalar_count = sum(
            isinstance(prediction, dict)
            and prediction.get("normalized_value") is not None
            for prediction in scalars.values()
        )
        party_suffix_fields = sorted(
            field_name
            for field_name, field_value in scalars.items()
            if isinstance(field_value, dict)
            and str(field_value.get("method", "")).endswith(
                ":LEGAL_SUFFIX_CONTINUATION"
            )
        )

        prediction_artifact = {
            "schema_version": "1.0.0",
            "status": "EXECUTED",
            "quality_status": (
                "PENDING_VALIDATION_GROUND_TRUTH_EVALUATION"
            ),
            "parser": {
                "baseline_id": EXPECTED_BASELINE_ID,
                "parser_id": EXPECTED_PARSER_ID,
                "parser_version": EXPECTED_PARSER_VERSION,
                "parser_signature_sha256": parser_signature,
                "parser_source_path": str(parser_source_path),
                "parser_source_sha256": source_checksums_before[
                    "parser_source"
                ],
                "freeze_status": freeze_manifest["status"],
                "approach": "DETERMINISTIC_LABEL_AND_GEOMETRY_RULES",
                "template_specific_branching": False,
                "ground_truth_used_as_prediction_input": False,
                "parser_input_fields": ["lines"],
            },
            "document": {
                "canonical_invoice_id": document.get(
                    "canonical_invoice_id"
                ),
                "document_id": document_id,
                "template_id": template_id,
                "split": "validation",
                "language": document.get("language"),
            },
            "predictions": {
                "scalar_fields": scalars,
                "items": items,
            },
            "diagnostics": diagnostics,
            "source": {
                "text_layer_path": str(text_layer_path),
                "text_layer_sha256": source_record[
                    "text_layer_sha256"
                ],
                "route_id": source_record.get("route_id"),
                "engine": source_record.get("engine"),
            },
            "integrity": {
                "ground_truth_loaded": False,
                "canonical_payload_loaded": False,
                "ground_truth_used_as_prediction_input": False,
                "validation_ground_truth_opened": 0,
                "test_opened": 0,
                "parser_modifications": 0,
                "dataset_modifications": 0,
                "source_modifications": 0,
            },
        }

        checkpoint_action = save_immutable_json(
            prediction_path,
            prediction_artifact,
        )
        if checkpoint_action == "CREATED":
            newly_created += 1
            execution = "NEW"
        else:
            recovered += 1
            execution = "RECOVERED"

        persisted = load_json(prediction_path)
        if canonical_json(persisted) != canonical_json(
            prediction_artifact
        ):
            raise RuntimeError(
                f"Prediction berbeda setelah penulisan: {document_id}"
            )

        manifest_record = {
            "sequence_number": int(
                source_record.get("sequence_number", position)
            ),
            "document_id": document_id,
            "template_id": template_id,
            "language": document.get("language"),
            "route_id": source_record.get("route_id"),
            "populated_scalar_fields": populated_scalar_count,
            "missing_required_scalar_fields": missing_required,
            "party_suffix_continuation_fields": party_suffix_fields,
            "party_suffix_continuation_count": len(
                party_suffix_fields
            ),
            "parsed_item_count": len(items),
            "table_detected": bool(
                table_diagnostics.get("table_detected", False)
            ),
            "financial_equation_executed": bool(
                financial_check.get("executed", False)
            ),
            "financial_equation_passed": bool(
                financial_check.get("passed", False)
            ),
            "prediction_path": str(prediction_path),
            "prediction_sha256": sha256_file(prediction_path),
            "text_layer_path": str(text_layer_path),
            "text_layer_sha256": source_record[
                "text_layer_sha256"
            ],
            "status": "EXECUTED",
            "quality_status": (
                "PENDING_VALIDATION_GROUND_TRUTH_EVALUATION"
            ),
        }
        manifest_records.append(manifest_record)
        runtime_records.append(
            {**manifest_record, "execution": execution}
        )

        print(
            f"[{position:02d}/{len(source_records):02d}] "
            f"{document_id} | scalars={populated_scalar_count}/12 | "
            f"items={len(items)} | "
            f"table={manifest_record['table_detected']} | "
            f"financial={manifest_record['financial_equation_passed']} | "
            f"{execution}"
        )

    except Exception as error:
        processing_errors.append(
            {
                "sequence_number": source_record.get(
                    "sequence_number", position
                ),
                "document_id": document_id,
                "template_id": template_id,
                "error_type": type(error).__name__,
                "error": str(error)[:900],
            }
        )
        print(
            f"[{position:02d}/{len(source_records):02d}] "
            f"{document_id} | ERROR: "
            f"{type(error).__name__}: {error}"
        )


# ============================================================
# EXECUTION GATES — DO NOT TURN DIAGNOSTICS INTO QUALITY CLAIMS
# ============================================================

runtime_table = pd.DataFrame(runtime_records)
manifest_table = pd.DataFrame(manifest_records)

if processing_errors:
    print("\nPROCESSING ERRORS")
    display(pd.DataFrame(processing_errors))

execution_values = [
    ("prediction_records", EXPECTED_DOCUMENTS, len(manifest_records)),
    (
        "prediction_files",
        EXPECTED_DOCUMENTS,
        sum(
            Path(row["prediction_path"]).is_file()
            for row in manifest_records
        ),
    ),
    (
        "unique_document_ids",
        EXPECTED_DOCUMENTS,
        len({row["document_id"] for row in manifest_records}),
    ),
    (
        "validation_templates",
        sorted(EXPECTED_TEMPLATES),
        sorted({row["template_id"] for row in manifest_records}),
    ),
    ("processing_errors", 0, len(processing_errors)),
    (
        "parser_source_checksum",
        baseline_pointer["parser_source_sha256"],
        sha256_file(parser_source_path),
    ),
    ("parser_modifications", 0, 0),
    ("ground_truth_opened", 0, 0),
    ("validation_ground_truth_opened", 0, 0),
    ("test_opened", 0, 0),
]

execution_controls = [
    {
        "control": name,
        "expected": expected,
        "actual": actual,
        "status": "VALID" if expected == actual else "INVALID",
    }
    for name, expected, actual in execution_values
]

print("\nEXECUTION CONTROLS")
display(pd.DataFrame(execution_controls))

if not runtime_table.empty:
    display(
        runtime_table[
            [
                "sequence_number",
                "document_id",
                "template_id",
                "language",
                "populated_scalar_fields",
                "parsed_item_count",
                "table_detected",
                "financial_equation_executed",
                "financial_equation_passed",
                "party_suffix_continuation_count",
                "execution",
                "quality_status",
            ]
        ]
    )

invalid_execution_controls = [
    row["control"]
    for row in execution_controls
    if row["status"] != "VALID"
]
if invalid_execution_controls:
    raise RuntimeError(
        "CELL 11C EXECUTION FAILED. "
        f"Kontrol tidak valid: {invalid_execution_controls}"
    )


# These are observations, not acceptance gates. Their correctness can only be
# measured against the frozen validation ground truth in Cell 11D.
technical_observations = [
    {
        "observation": "complete_required_scalars",
        "documents": sum(
            not row["missing_required_scalar_fields"]
            for row in manifest_records
        ),
        "status": "OBSERVED_NOT_EVALUATED",
    },
    {
        "observation": "table_detected",
        "documents": sum(
            row["table_detected"] for row in manifest_records
        ),
        "status": "OBSERVED_NOT_EVALUATED",
    },
    {
        "observation": "nonempty_items",
        "documents": sum(
            row["parsed_item_count"] > 0 for row in manifest_records
        ),
        "status": "OBSERVED_NOT_EVALUATED",
    },
    {
        "observation": "financial_equation_executed",
        "documents": sum(
            row["financial_equation_executed"]
            for row in manifest_records
        ),
        "status": "OBSERVED_NOT_EVALUATED",
    },
    {
        "observation": "financial_equation_passed",
        "documents": sum(
            row["financial_equation_passed"]
            for row in manifest_records
        ),
        "status": "OBSERVED_NOT_EVALUATED",
    },
]

print("\nTECHNICAL OBSERVATIONS — NOT QUALITY METRICS")
display(pd.DataFrame(technical_observations))


# ============================================================
# IMMUTABLE SUMMARY AND PREDICTION MANIFEST
# ============================================================

summary_columns = [
    "sequence_number",
    "document_id",
    "template_id",
    "language",
    "route_id",
    "populated_scalar_fields",
    "missing_required_scalar_fields",
    "party_suffix_continuation_count",
    "parsed_item_count",
    "table_detected",
    "financial_equation_executed",
    "financial_equation_passed",
    "prediction_path",
    "prediction_sha256",
    "text_layer_path",
    "text_layer_sha256",
    "status",
    "quality_status",
]

summary_text = manifest_table[summary_columns].to_csv(
    index=False,
    lineterminator="\n",
)
summary_action = save_immutable_text(SUMMARY_PATH, summary_text)

prediction_manifest = {
    "schema_version": "1.0.0",
    "cell_version": CELL_VERSION,
    "status": "EXECUTED",
    "quality_status": (
        "PENDING_VALIDATION_GROUND_TRUTH_EVALUATION"
    ),
    "stage": "FROZEN_PARSER_BLIND_VALIDATION_PREDICTION",
    "parser": {
        "baseline_id": EXPECTED_BASELINE_ID,
        **parser_configuration,
        "parser_signature_sha256": parser_signature,
        "parser_source_path": str(parser_source_path),
        "parser_source_sha256": source_checksums_before[
            "parser_source"
        ],
        "freeze_manifest_path": str(freeze_manifest_path),
        "freeze_manifest_sha256": source_checksums_before[
            "freeze_manifest"
        ],
        "loader_audit": parser_loader_audit,
    },
    "scope": {
        "split": "validation",
        "documents": EXPECTED_DOCUMENTS,
        "templates": sorted(EXPECTED_TEMPLATES),
        "ground_truth_opened": 0,
        "validation_ground_truth_opened": 0,
        "test_opened": 0,
    },
    "records": manifest_records,
    "execution_controls": execution_controls,
    "technical_observations": technical_observations,
    "artifacts": {
        "prediction_root": str(PREDICTION_ROOT),
        "summary_path": str(SUMMARY_PATH),
        "summary_sha256": sha256_file(SUMMARY_PATH),
    },
    "input_artifacts": {
        "validation_baseline_pointer": {
            "path": str(VALIDATION_BASELINE_POINTER_PATH),
            "sha256": source_checksums_before[
                "validation_baseline_pointer"
            ],
        },
        "freeze_manifest": {
            "path": str(freeze_manifest_path),
            "sha256": source_checksums_before["freeze_manifest"],
        },
        "parser_source": {
            "path": str(parser_source_path),
            "sha256": source_checksums_before["parser_source"],
        },
        "text_layer_manifest": {
            "path": str(TEXT_LAYER_MANIFEST_PATH),
            "sha256": source_checksums_before[
                "text_layer_manifest"
            ],
        },
    },
    "integrity": {
        "ground_truth_loaded": False,
        "canonical_payload_loaded": False,
        "ground_truth_used_as_prediction_input": False,
        "validation_ground_truth_opened": 0,
        "test_opened": 0,
        "parser_modifications": 0,
        "dataset_modifications": 0,
        "source_modifications": 0,
    },
    "next_stage": {
        "cell": "CELL 11D",
        "action": "VALIDATION_GROUND_TRUTH_EVALUATION",
        "predictions_must_be_frozen_before_evaluation": True,
        "parser_changes_allowed_after_results": False,
        "test_remains_locked": True,
    },
}

manifest_action = save_immutable_json(
    MANIFEST_PATH,
    prediction_manifest,
)

persisted_manifest = load_json(MANIFEST_PATH)
if canonical_json(persisted_manifest) != canonical_json(
    prediction_manifest
):
    raise RuntimeError("Manifest Cell 11C berbeda setelah penulisan.")


# ============================================================
# FINAL INPUT IMMUTABILITY VERIFICATION
# ============================================================

source_checksums_after = {
    "validation_baseline_pointer": sha256_file(
        VALIDATION_BASELINE_POINTER_PATH
    ),
    "freeze_manifest": sha256_file(freeze_manifest_path),
    "parser_source": sha256_file(parser_source_path),
    "text_layer_manifest": sha256_file(TEXT_LAYER_MANIFEST_PATH),
}
for source_record in source_records:
    source_checksums_after[
        f"text_layer:{source_record['document_id']}"
    ] = sha256_file(Path(source_record["text_layer_path"]))

changed_sources = [
    name
    for name, checksum in source_checksums_before.items()
    if source_checksums_after.get(name) != checksum
]
if changed_sources:
    raise RuntimeError(
        f"Input berubah selama Cell 11C: {changed_sources}"
    )


template_counts = Counter(
    row["template_id"] for row in manifest_records
)
language_counts = Counter(
    row["language"] for row in manifest_records
)

print()
print(f"Cell version           : {CELL_VERSION}")
print(f"Baseline ID            : {EXPECTED_BASELINE_ID}")
print(f"Parser ID              : {EXPECTED_PARSER_ID}")
print(f"Parser version         : {EXPECTED_PARSER_VERSION}")
print(f"Parser signature       : {parser_signature}")
print(
    "Parser source SHA-256: "
    f"{source_checksums_before['parser_source']}"
)
print(f"Validation documents   : {len(manifest_records)}")
print(f"Templates              : {dict(sorted(template_counts.items()))}")
print(f"Languages              : {dict(sorted(language_counts.items()))}")
print(f"New predictions        : {newly_created}")
print(f"Recovered predictions  : {recovered}")
print(f"Summary action         : {summary_action}")
print(f"Manifest action        : {manifest_action}")
print(f"Prediction root        : {PREDICTION_ROOT}")
print(f"Summary                : {SUMMARY_PATH}")
print(f"Manifest               : {MANIFEST_PATH}")
print(f"Manifest SHA-256       : {sha256_file(MANIFEST_PATH)}")
print("Ground truth opened    : 0")
print("Validation GT opened   : 0")
print("Test opened            : 0")
print("Parser modifications   : 0")
print("Dataset modifications  : 0")
print("Source modifications   : 0")
print(
    "Quality status        : "
    "PENDING_VALIDATION_GROUND_TRUTH_EVALUATION"
)
print()
print(
    "✅ CELL 11C PASSED — frozen parser v1.0.2 telah menghasilkan "
    "prediction checkpoint untuk 40 dokumen validation tanpa "
    "membuka ground truth atau test. Lanjutkan ke Cell 11D untuk "
    "evaluasi validation satu kali."
)


,control,expected,actual,status
0,baseline_pointer_status,ACTIVE_FOR_VALIDATION,ACTIVE_FOR_VALIDATION,VALID
1,baseline_id,RULE-BASED-INVOICE-PARSER-V1@1.0.2,RULE-BASED-INVOICE-PARSER-V1@1.0.2,VALID
2,parser_id,RULE-BASED-INVOICE-PARSER-V1,RULE-BASED-INVOICE-PARSER-V1,VALID
3,parser_version,1.0.2,1.0.2,VALID
4,next_allowed_split,validation,validation,VALID
5,test_locked,True,True,VALID
6,freeze_status,FROZEN_FOR_VALIDATION,FROZEN_FOR_VALIDATION,VALID
7,freeze_checksum,9f234c5c8cdc10c277e9c26a1f0831d1e08f3dc6fb9503...,9f234c5c8cdc10c277e9c26a1f0831d1e08f3dc6fb9503...,VALID
8,parser_source_path,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,VALID
9,parser_source_checksum_pointer,0a737f57a86df7d3e16eef6749e3c3a23e82fb51227d09...,0a737f57a86df7d3e16eef6749e3c3a23e82fb51227d09...,VALID



FROZEN PARSER LOAD CONTROLS


,control,expected,actual,status
0,runtime_parser_id,RULE-BASED-INVOICE-PARSER-V1,RULE-BASED-INVOICE-PARSER-V1,VALID
1,runtime_parser_version,1.0.2,1.0.2,VALID
2,runtime_parser_signature,ffe04f258fb3b1a6a55661698fa6c7dda29ddd7b714c2f...,ffe04f258fb3b1a6a55661698fa6c7dda29ddd7b714c2f...,VALID
3,parse_document_callable,True,True,VALID
4,development_runner_executed,False,False,VALID
5,parser_rule_modifications,0,0,VALID



TEXT-LAYER SOURCE CONTROLS


,control,expected,actual,status
0,text_layer_records,40,40,VALID
1,unique_document_ids,40,40,VALID
2,validation_templates,"[TPL-07, TPL-08]","[TPL-07, TPL-08]",VALID
3,source_errors,0,0,VALID


...

,control,expected,actual,status
0,prediction_records,40,40,VALID
1,prediction_files,40,40,VALID
2,unique_document_ids,40,40,VALID
3,validation_templates,"[TPL-07, TPL-08]","[TPL-07, TPL-08]",VALID
4,processing_errors,0,0,VALID
5,parser_source_checksum,0a737f57a86df7d3e16eef6749e3c3a23e82fb51227d09...,0a737f57a86df7d3e16eef6749e3c3a23e82fb51227d09...,VALID
6,parser_modifications,0,0,VALID
7,ground_truth_opened,0,0,VALID
8,validation_ground_truth_opened,0,0,VALID
9,test_opened,0,0,VALID


,sequence_number,document_id,template_id,language,populated_scalar_fields,parsed_item_count,table_detected,financial_equation_executed,financial_equation_passed,party_suffix_continuation_count,execution,quality_status
0,1,INV-SYN-000121,TPL-07,id,12,7,True,True,True,0,NEW,PENDING_VALIDATION_GROUND_TRUTH_EVALUATION
1,2,INV-SYN-000122,TPL-07,id,12,7,True,True,True,0,NEW,PENDING_VALIDATION_GROUND_TRUTH_EVALUATION
2,3,INV-SYN-000123,TPL-07,id,12,3,True,True,True,0,NEW,PENDING_VALIDATION_GROUND_TRUTH_EVALUATION
3,4,INV-SYN-000124,TPL-07,id,12,4,True,True,True,0,NEW,PENDING_VALIDATION_GROUND_TRUTH_EVALUATION
4,5,INV-SYN-000125,TPL-07,id,12,4,True,True,True,0,NEW,PENDING_VALIDATION_GROUND_TRUTH_EVALUATION
5,6,INV-SYN-000126,TPL-07,id,12,2,True,True,True,0,NEW,PENDING_VALIDATION_GROUND_TRUTH_EVALUATION
6,7,INV-SYN-000127,TPL-07,id,12,3,True,True,True,0,NEW,PENDING_VALIDATION_GROUND_TRUTH_EVALUATION
7,8,INV-SYN-000128,TPL-07,id,12,4,True,True,True,0,NEW,PENDING_VALIDATION_GROUND_TRUTH_EVALUATION
8,9,INV-SYN-000129,TPL-07,id,12,6,True,True,True,0,NEW,PENDING_VALIDATION_GROUND_TRUTH_EVALUATION
9,10,INV-SYN-000130,TPL-07,id,12,5,True,True,True,0,NEW,PENDING_VALIDATION_GROUND_TRUTH_EVALUATION



TECHNICAL OBSERVATIONS — NOT QUALITY METRICS


,observation,documents,status
0,complete_required_scalars,40,OBSERVED_NOT_EVALUATED
1,table_detected,40,OBSERVED_NOT_EVALUATED
2,nonempty_items,40,OBSERVED_NOT_EVALUATED
3,financial_equation_executed,40,OBSERVED_NOT_EVALUATED
4,financial_equation_passed,40,OBSERVED_NOT_EVALUATED



Cell version           : 1.0.0
Baseline ID            : RULE-BASED-INVOICE-PARSER-V1@1.0.2
Parser ID              : RULE-BASED-INVOICE-PARSER-V1
Parser version         : 1.0.2
Parser signature       : ffe04f258fb3b1a6a55661698fa6c7dda29ddd7b714c2f210b3e6fc1fcd464b3
Parser source SHA-256: 0a737f57a86df7d3e16eef6749e3c3a23e82fb51227d099845908e89c8be642c
Validation documents   : 40
Templates              : {'TPL-07': 20, 'TPL-08': 20}
Languages              : {'en': 20, 'id': 20}
New predictions        : 40
Recovered predictions  : 0
Summary action         : CREATED
Manifest action        : CREATED
Prediction root        : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/field_extraction/validation/rule_based_baseline_v1_0_2/predictions
Summary                : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/field_extraction/validation/rule_based_baseline_v1_0_2/validation_prediction_

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import re
import unicodedata
from decimal import Decimal, InvalidOperation
from functools import lru_cache
from pathlib import Path

import pandas as pd
from IPython.display import display


# ============================================================
# CELL 11D — ONE-TIME VALIDATION GROUND-TRUTH EVALUATION
#             (FROZEN PREDICTIONS; TEST REMAINS LOCKED)
# ============================================================

CELL_VERSION = "1.0.0"
EVALUATOR_ID = "INVOICE-FIELD-EVALUATOR-V1"
EVALUATOR_VERSION = "1.0.2"

DATA_ROOT = Path("/content/drive/MyDrive/InvoiceFlow-AI-Data")
BUILD_ROOT = (
    DATA_ROOT
    / "interim"
    / "synthetic_v1_build"
    / "20260904T150025Z"
)
FIELD_ROOT = BUILD_ROOT / "ocr_benchmark" / "field_extraction"
RENDERED_DATASET_ROOT = BUILD_ROOT / "rendered_dataset"
VALIDATION_GROUND_TRUTH_ROOT = (
    RENDERED_DATASET_ROOT / "ground_truth" / "validation"
)
RENDER_INDEX_PATH = BUILD_ROOT / "manifests" / "batch_render_index.jsonl"
CONTRACT_PATH = FIELD_ROOT / "field_extraction_contract_v1.json"

VALIDATION_ROOT = (
    FIELD_ROOT / "validation" / "rule_based_baseline_v1_0_2"
)
BASELINE_POINTER_PATH = FIELD_ROOT / "validation_baseline_pointer.json"
PREDICTION_MANIFEST_PATH = (
    VALIDATION_ROOT / "validation_prediction_manifest.json"
)

EVALUATION_ROOT = (
    VALIDATION_ROOT / "evaluations" / "validation_eval_v1_0_2"
)
DOCUMENT_EVALUATION_ROOT = EVALUATION_ROOT / "documents"
DOCUMENT_SUMMARY_PATH = EVALUATION_ROOT / "document_summary.csv"
FIELD_SUMMARY_PATH = EVALUATION_ROOT / "field_summary.csv"
TEMPLATE_SUMMARY_PATH = EVALUATION_ROOT / "template_summary.csv"
MISMATCH_DETAIL_PATH = EVALUATION_ROOT / "mismatch_details.csv"
EVALUATION_MANIFEST_PATH = (
    EVALUATION_ROOT / "validation_evaluation_manifest.json"
)

EXPECTED_BASELINE_ID = "RULE-BASED-INVOICE-PARSER-V1@1.0.2"
EXPECTED_PARSER_ID = "RULE-BASED-INVOICE-PARSER-V1"
EXPECTED_PARSER_VERSION = "1.0.2"
EXPECTED_PARSER_SIGNATURE = (
    "ffe04f258fb3b1a6a55661698fa6c7dda29ddd7b714c2f210b3e6fc1fcd464b3"
)
EXPECTED_DOCUMENTS = 40
EXPECTED_RENDER_RECORDS = 200
EXPECTED_TEMPLATES = {"TPL-07", "TPL-08"}
EXPECTED_SCALAR_FIELDS = 12
EXPECTED_ITEM_FIELDS = 4

ITEM_FIELD_NAMES = [
    "description",
    "quantity",
    "unit_price",
    "line_total",
]

SCALAR_PATHS = {
    "invoice_number": ("invoice_number",),
    "invoice_date": ("invoice_date",),
    "due_date": ("due_date",),
    "currency": ("currency",),
    "vendor.name": ("vendor", "name"),
    "vendor.tax_identifier": ("vendor", "tax_identifier"),
    "buyer.name": ("buyer", "name"),
    "buyer.tax_identifier": ("buyer", "tax_identifier"),
    "financials.subtotal": ("financials", "subtotal"),
    "financials.tax": ("financials", "tax"),
    "financials.discount": ("financials", "discount"),
    "financials.total": ("financials", "total"),
}

MONEY_FIELDS = {
    "financials.subtotal",
    "financials.tax",
    "financials.discount",
    "financials.total",
    "items[].unit_price",
    "items[].line_total",
}

IDENTIFIER_FIELDS = {
    "invoice_number",
    "currency",
    "vendor.tax_identifier",
    "buyer.tax_identifier",
}

SHA256_PATTERN = re.compile(r"^[0-9a-f]{64}$", re.IGNORECASE)


# ============================================================
# FILE, HASH, AND IMMUTABLE-WRITE HELPERS
# ============================================================

def require_file(path: Path, label: str) -> None:
    if not path.is_file():
        raise FileNotFoundError(f"{label} tidak ditemukan: {path}")
    if path.stat().st_size <= 0:
        raise RuntimeError(f"{label} kosong: {path}")


def require_directory(path: Path, label: str) -> None:
    if not path.is_dir():
        raise FileNotFoundError(f"{label} tidak ditemukan: {path}")


def load_json(path: Path) -> dict:
    require_file(path, "JSON artifact")
    with path.open("r", encoding="utf-8") as file_handle:
        value = json.load(file_handle)
    if not isinstance(value, dict):
        raise TypeError(f"Root JSON bukan object: {path}")
    return value


def load_jsonl(path: Path) -> list[dict]:
    require_file(path, "JSONL artifact")
    records = []
    with path.open("r", encoding="utf-8") as file_handle:
        for line_number, line in enumerate(file_handle, start=1):
            if not line.strip():
                continue
            value = json.loads(line)
            if not isinstance(value, dict):
                raise TypeError(
                    f"Record JSONL baris {line_number} bukan object: {path}"
                )
            records.append(value)
    return records


def canonical_json(value: object) -> str:
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    )


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as file_handle:
        for chunk in iter(lambda: file_handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def atomic_write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(
        f".{path.name}.{os.getpid()}.tmp"
    )
    try:
        temporary_path.write_text(text, encoding="utf-8")
        os.replace(temporary_path, path)
    finally:
        if temporary_path.exists():
            temporary_path.unlink()


def atomic_write_json(path: Path, value: dict) -> None:
    atomic_write_text(
        path,
        json.dumps(
            value,
            indent=2,
            ensure_ascii=False,
            default=str,
        )
        + "\n",
    )


def save_immutable_json(path: Path, value: dict) -> str:
    if path.exists():
        existing = load_json(path)
        if canonical_json(existing) != canonical_json(value):
            raise RuntimeError(
                f"Checkpoint sudah ada tetapi berbeda: {path}"
            )
        return "RECOVERED"
    atomic_write_json(path, value)
    return "CREATED"


def save_csv_checkpoint(path: Path, table: pd.DataFrame) -> str:
    text = table.to_csv(index=False, lineterminator="\n")
    if path.exists():
        if path.read_text(encoding="utf-8") != text:
            raise RuntimeError(
                f"CSV checkpoint sudah ada tetapi berbeda: {path}"
            )
        return "RECOVERED"
    atomic_write_text(path, text)
    return "CREATED"


def nested_value(record: dict, path: tuple[str, ...]):
    value = record
    for key in path:
        if not isinstance(value, dict) or key not in value:
            return None
        value = value[key]
    return value


# ============================================================
# TYPE-AWARE NORMALIZATION — IDENTICAL TO CELL 10D
# ============================================================

def normalize_spaces(value) -> str:
    if value is None:
        return ""
    text = unicodedata.normalize("NFKC", str(value))
    return re.sub(r"\s+", " ", text).strip()


def decimal_to_string(value: Decimal) -> str:
    if value == value.to_integral():
        return str(value.quantize(Decimal("1")))
    text = format(value.normalize(), "f")
    return text.rstrip("0").rstrip(".")


def normalize_decimal(value) -> str:
    text = normalize_spaces(value)
    if not text:
        return ""
    try:
        return decimal_to_string(Decimal(text))
    except InvalidOperation:
        return text


def normalize_field_value(field_name: str, value) -> str:
    text = normalize_spaces(value)
    if field_name in MONEY_FIELDS or field_name == "items[].quantity":
        return normalize_decimal(text)
    if field_name in IDENTIFIER_FIELDS:
        return text.upper()
    return text


# ============================================================
# EDIT-DISTANCE METRICS — IDENTICAL TO CELL 10D
# ============================================================

def levenshtein_distance(reference, hypothesis) -> int:
    reference = list(reference)
    hypothesis = list(hypothesis)

    if len(reference) < len(hypothesis):
        reference, hypothesis = hypothesis, reference

    previous = list(range(len(hypothesis) + 1))
    for reference_index, reference_value in enumerate(reference, start=1):
        current = [reference_index]
        for hypothesis_index, hypothesis_value in enumerate(
            hypothesis,
            start=1,
        ):
            insertion = current[hypothesis_index - 1] + 1
            deletion = previous[hypothesis_index] + 1
            substitution = (
                previous[hypothesis_index - 1]
                + int(reference_value != hypothesis_value)
            )
            current.append(min(insertion, deletion, substitution))
        previous = current
    return previous[-1]


def safe_ratio(numerator: int | float, denominator: int | float) -> float:
    if denominator == 0:
        return 1.0 if numerator == 0 else 0.0
    return float(numerator) / float(denominator)


def safe_error_rate(
    errors: int | float,
    reference_units: int | float,
) -> float:
    if reference_units == 0:
        return 0.0 if errors == 0 else 1.0
    return float(errors) / float(reference_units)


def harmonic_mean(precision: float, recall: float) -> float:
    if precision + recall == 0:
        return 0.0
    return 2.0 * precision * recall / (precision + recall)


def string_similarity(first: str, second: str) -> float:
    first = normalize_spaces(first).casefold()
    second = normalize_spaces(second).casefold()
    denominator = max(len(first), len(second), 1)
    return 1.0 - levenshtein_distance(first, second) / denominator


# ============================================================
# GROUND-TRUTH ACCESS PLAN FROM THE FROZEN RENDER INDEX
# This section resolves paths but does not load JSON payloads.
# ============================================================

def build_render_record_map(records: list[dict]) -> dict[str, dict]:
    mapped = {}
    for record in records:
        document_id = str(record.get("document_id", ""))
        if not document_id:
            continue
        if document_id in mapped:
            raise RuntimeError(
                f"Document ID duplikat pada render index: {document_id}"
            )
        mapped[document_id] = record
    return mapped


def ground_truth_access_plan(
    prediction_records: list[dict],
    render_record_map: dict[str, dict],
) -> dict[str, dict]:
    validation_root = VALIDATION_GROUND_TRUTH_ROOT.resolve()
    plan = {}

    for prediction_record in prediction_records:
        document_id = str(prediction_record["document_id"])
        template_id = str(prediction_record["template_id"])
        render_record = render_record_map.get(document_id)
        if render_record is None:
            raise RuntimeError(
                f"Render record tidak ditemukan: {document_id}"
            )
        if render_record.get("template_id") != template_id:
            raise RuntimeError(
                f"Template render/prediction berbeda: {document_id}"
            )
        if str(render_record.get("split", "")).casefold() != "validation":
            raise RuntimeError(
                f"Render record bukan validation: {document_id}"
            )

        artifact = (
            render_record.get("artifacts", {}).get("ground_truth", {})
        )
        if not isinstance(artifact, dict):
            raise TypeError(
                f"Ground-truth artifact metadata tidak valid: {document_id}"
            )

        relative_path = Path(str(artifact.get("relative_path", "")))
        expected_sha256 = str(artifact.get("sha256", "")).casefold()
        expected_size = int(artifact.get("size_bytes", -1))

        if relative_path.is_absolute() or relative_path.suffix != ".json":
            raise RuntimeError(
                f"Ground-truth relative path tidak valid: {document_id}"
            )
        if not SHA256_PATTERN.fullmatch(expected_sha256):
            raise RuntimeError(
                f"Ground-truth checksum metadata tidak valid: {document_id}"
            )

        ground_truth_path = (
            RENDERED_DATASET_ROOT / relative_path
        ).resolve()
        try:
            ground_truth_path.relative_to(validation_root)
        except ValueError as error:
            raise RuntimeError(
                f"Ground truth di luar validation root: {document_id}"
            ) from error

        if "test" in {
            part.casefold() for part in ground_truth_path.parts
        }:
            raise RuntimeError(
                f"Path test dilarang pada Cell 11D: {ground_truth_path}"
            )

        require_file(ground_truth_path, f"Validation GT {document_id}")
        if ground_truth_path.stat().st_size != expected_size:
            raise RuntimeError(
                f"Ukuran validation GT tidak cocok: {document_id}"
            )

        plan[document_id] = {
            "document_id": document_id,
            "template_id": template_id,
            "path": str(ground_truth_path),
            "expected_sha256": expected_sha256,
            "expected_size_bytes": expected_size,
        }

    return plan


# ============================================================
# CANONICAL GROUND-TRUTH AND PREDICTION ADAPTERS
# ============================================================

def canonical_from_ground_truth(
    ground_truth: dict,
    document_id: str,
    template_id: str,
) -> dict:
    document = ground_truth.get("document", {})
    canonical = ground_truth.get("canonical")

    if not isinstance(document, dict):
        raise TypeError("ground_truth.document harus berupa object.")
    if not isinstance(canonical, dict):
        raise TypeError("ground_truth.canonical harus berupa object.")
    if document.get("document_id") != document_id:
        raise RuntimeError("Document ID ground truth tidak cocok.")
    if document.get("template_id") != template_id:
        raise RuntimeError("Template ID ground truth tidak cocok.")
    if str(document.get("split", "")).casefold() != "validation":
        raise RuntimeError("Ground truth bukan validation split.")
    if canonical.get("document_id") != document_id:
        raise RuntimeError("Document ID canonical tidak cocok.")
    if canonical.get("template_id") != template_id:
        raise RuntimeError("Template ID canonical tidak cocok.")
    if str(canonical.get("split", "")).casefold() != "validation":
        raise RuntimeError("Canonical payload bukan validation split.")

    items = canonical.get("items")
    if not isinstance(items, list) or not items:
        raise RuntimeError("Canonical items kosong atau tidak valid.")
    if not all(isinstance(item, dict) for item in items):
        raise TypeError("Setiap canonical item harus berupa object.")
    return canonical


def expected_scalars(canonical: dict) -> dict[str, str]:
    return {
        field_name: normalize_field_value(
            field_name,
            nested_value(canonical, path),
        )
        for field_name, path in SCALAR_PATHS.items()
    }


def expected_items(canonical: dict) -> list[dict[str, str]]:
    normalized_items = []
    for item in canonical["items"]:
        normalized_items.append(
            {
                field_name: normalize_field_value(
                    f"items[].{field_name}",
                    item.get(field_name),
                )
                for field_name in ITEM_FIELD_NAMES
            }
        )
    return normalized_items


def predicted_scalars(prediction: dict) -> dict[str, str]:
    scalar_predictions = (
        prediction.get("predictions", {}).get("scalar_fields", {})
    )
    if not isinstance(scalar_predictions, dict):
        raise TypeError("predictions.scalar_fields harus berupa object.")

    values = {}
    for field_name in SCALAR_PATHS:
        field_prediction = scalar_predictions.get(field_name, {})
        if not isinstance(field_prediction, dict):
            field_prediction = {}
        values[field_name] = normalize_field_value(
            field_name,
            field_prediction.get("normalized_value"),
        )
    return values


def predicted_items(prediction: dict) -> list[dict[str, str]]:
    item_predictions = prediction.get("predictions", {}).get("items", [])
    if not isinstance(item_predictions, list):
        raise TypeError("predictions.items harus berupa list.")

    normalized_items = []
    for item in item_predictions:
        if not isinstance(item, dict):
            raise TypeError("Setiap prediction item harus berupa object.")
        normalized_item = {}
        for field_name in ITEM_FIELD_NAMES:
            field_prediction = item.get(field_name, {})
            if not isinstance(field_prediction, dict):
                field_prediction = {}
            normalized_item[field_name] = normalize_field_value(
                f"items[].{field_name}",
                field_prediction.get("normalized_value"),
            )
        normalized_items.append(normalized_item)
    return normalized_items


# ============================================================
# MAXIMUM-WEIGHT BIPARTITE ITEM ALIGNMENT — IDENTICAL TO 10D
# ============================================================

def item_pair_weight(predicted: dict, expected: dict) -> int:
    description_similarity = string_similarity(
        predicted.get("description", ""),
        expected.get("description", ""),
    )
    line_total_exact = (
        predicted.get("line_total", "")
        == expected.get("line_total", "")
    )
    quantity_exact = (
        predicted.get("quantity", "")
        == expected.get("quantity", "")
    )
    unit_price_exact = (
        predicted.get("unit_price", "")
        == expected.get("unit_price", "")
    )
    return int(round(description_similarity * 10000)) + (
        10000 if line_total_exact else 0
    ) + (100 if quantity_exact else 0) + (100 if unit_price_exact else 0)


def maximum_weight_item_alignment(
    predictions: list[dict],
    references: list[dict],
) -> list[tuple[int | None, int | None]]:
    size = max(len(predictions), len(references))
    if size == 0:
        return []

    weights = []
    for prediction_index in range(size):
        row = []
        for reference_index in range(size):
            if (
                prediction_index < len(predictions)
                and reference_index < len(references)
            ):
                row.append(
                    item_pair_weight(
                        predictions[prediction_index],
                        references[reference_index],
                    )
                )
            else:
                row.append(0)
        weights.append(row)

    @lru_cache(maxsize=None)
    def solve(
        prediction_index: int,
        used_reference_mask: int,
    ) -> tuple[int, tuple[int, ...]]:
        if prediction_index == size:
            return 0, ()
        best_score = -1
        best_assignment = ()
        for reference_index in range(size):
            bit = 1 << reference_index
            if used_reference_mask & bit:
                continue
            remaining_score, remaining_assignment = solve(
                prediction_index + 1,
                used_reference_mask | bit,
            )
            score = weights[prediction_index][reference_index] + remaining_score
            assignment = (reference_index,) + remaining_assignment
            if score > best_score or (
                score == best_score and assignment < best_assignment
            ):
                best_score = score
                best_assignment = assignment
        return best_score, best_assignment

    _, assignment = solve(0, 0)
    aligned_pairs = []
    for prediction_index, reference_index in enumerate(assignment):
        real_prediction = (
            prediction_index if prediction_index < len(predictions) else None
        )
        real_reference = (
            reference_index if reference_index < len(references) else None
        )
        if real_prediction is not None or real_reference is not None:
            aligned_pairs.append((real_prediction, real_reference))
    return aligned_pairs


# ============================================================
# DOCUMENT EVALUATION — IDENTICAL METRIC DEFINITION TO 10D
# ============================================================

def evaluate_document(
    prediction: dict,
    canonical: dict,
    contract_field_map: dict[str, dict],
) -> dict:
    document = prediction.get("document", {})
    document_id = str(document["document_id"])
    template_id = str(document["template_id"])
    language = str(document.get("language", ""))

    scalar_expected = expected_scalars(canonical)
    scalar_predicted = predicted_scalars(prediction)
    scalar_records = []

    for field_name in SCALAR_PATHS:
        expected_value = scalar_expected[field_name]
        predicted_value = scalar_predicted[field_name]
        exact_match = predicted_value == expected_value
        character_errors = levenshtein_distance(
            expected_value,
            predicted_value,
        )
        word_errors = levenshtein_distance(
            expected_value.split(),
            predicted_value.split(),
        )
        scalar_records.append(
            {
                "document_id": document_id,
                "template_id": template_id,
                "language": language,
                "group": "scalar",
                "field": field_name,
                "critical": bool(
                    contract_field_map[field_name].get("critical")
                ),
                "expected": expected_value,
                "predicted": predicted_value,
                "exact_match": exact_match,
                "character_errors": character_errors,
                "reference_characters": len(expected_value),
                "word_errors": word_errors,
                "reference_words": len(expected_value.split()),
            }
        )

    item_expected = expected_items(canonical)
    item_predicted = predicted_items(prediction)
    alignment = maximum_weight_item_alignment(
        item_predicted,
        item_expected,
    )

    item_field_records = []
    item_pair_records = []
    for pair_number, (prediction_index, reference_index) in enumerate(
        alignment,
        start=1,
    ):
        predicted_item = (
            item_predicted[prediction_index]
            if prediction_index is not None
            else None
        )
        expected_item = (
            item_expected[reference_index]
            if reference_index is not None
            else None
        )

        description_exact = bool(
            predicted_item is not None
            and expected_item is not None
            and predicted_item["description"] == expected_item["description"]
        )
        line_total_exact = bool(
            predicted_item is not None
            and expected_item is not None
            and predicted_item["line_total"] == expected_item["line_total"]
        )
        accepted_row_match = description_exact and line_total_exact
        item_pair_records.append(
            {
                "pair_number": pair_number,
                "prediction_row": (
                    prediction_index + 1
                    if prediction_index is not None
                    else None
                ),
                "reference_row": (
                    reference_index + 1
                    if reference_index is not None
                    else None
                ),
                "description_exact": description_exact,
                "line_total_exact": line_total_exact,
                "accepted_row_match": accepted_row_match,
            }
        )

        for field_name in ITEM_FIELD_NAMES:
            full_field_name = f"items[].{field_name}"
            expected_value = (
                expected_item[field_name] if expected_item is not None else ""
            )
            predicted_value = (
                predicted_item[field_name]
                if predicted_item is not None
                else ""
            )
            exact_match = bool(
                predicted_item is not None
                and expected_item is not None
                and predicted_value == expected_value
            )
            item_field_records.append(
                {
                    "document_id": document_id,
                    "template_id": template_id,
                    "language": language,
                    "group": "item",
                    "field": full_field_name,
                    "critical": bool(
                        contract_field_map[full_field_name].get("critical")
                    ),
                    "prediction_row": (
                        prediction_index + 1
                        if prediction_index is not None
                        else None
                    ),
                    "reference_row": (
                        reference_index + 1
                        if reference_index is not None
                        else None
                    ),
                    "expected": expected_value,
                    "predicted": predicted_value,
                    "exact_match": exact_match,
                    "character_errors": levenshtein_distance(
                        expected_value,
                        predicted_value,
                    ),
                    "reference_characters": len(expected_value),
                    "word_errors": levenshtein_distance(
                        expected_value.split(),
                        predicted_value.split(),
                    ),
                    "reference_words": len(expected_value.split()),
                }
            )

    row_true_positives = sum(
        record["accepted_row_match"] for record in item_pair_records
    )
    row_precision = safe_ratio(row_true_positives, len(item_predicted))
    row_recall = safe_ratio(row_true_positives, len(item_expected))
    row_f1 = harmonic_mean(row_precision, row_recall)

    item_exact_fields = sum(
        record["exact_match"] for record in item_field_records
    )
    predicted_item_fields = len(item_predicted) * EXPECTED_ITEM_FIELDS
    reference_item_fields = len(item_expected) * EXPECTED_ITEM_FIELDS
    item_field_precision = safe_ratio(
        item_exact_fields,
        predicted_item_fields,
    )
    item_field_recall = safe_ratio(
        item_exact_fields,
        reference_item_fields,
    )
    item_field_f1 = harmonic_mean(
        item_field_precision,
        item_field_recall,
    )

    scalar_exact_count = sum(
        record["exact_match"] for record in scalar_records
    )
    document_exact = bool(
        scalar_exact_count == EXPECTED_SCALAR_FIELDS
        and len(item_predicted) == len(item_expected)
        and item_exact_fields == reference_item_fields
    )
    financial_check = prediction.get("diagnostics", {}).get(
        "financial_equation",
        {},
    )
    financial_consistent = bool(
        financial_check.get("executed") is True
        and financial_check.get("passed") is True
    )

    return {
        "document_id": document_id,
        "template_id": template_id,
        "language": language,
        "scalar_records": scalar_records,
        "item_field_records": item_field_records,
        "item_alignment": item_pair_records,
        "metrics": {
            "scalar_exact_count": scalar_exact_count,
            "scalar_field_count": EXPECTED_SCALAR_FIELDS,
            "scalar_exact_match": safe_ratio(
                scalar_exact_count,
                EXPECTED_SCALAR_FIELDS,
            ),
            "predicted_item_count": len(item_predicted),
            "reference_item_count": len(item_expected),
            "item_count_exact": len(item_predicted) == len(item_expected),
            "row_true_positives": row_true_positives,
            "row_precision": row_precision,
            "row_recall": row_recall,
            "row_f1": row_f1,
            "item_exact_fields": item_exact_fields,
            "predicted_item_fields": predicted_item_fields,
            "reference_item_fields": reference_item_fields,
            "item_field_precision": item_field_precision,
            "item_field_recall": item_field_recall,
            "item_field_f1": item_field_f1,
            "financial_consistent": financial_consistent,
            "document_exact_match": document_exact,
        },
    }


# ============================================================
# PREFLIGHT — FREEZE ALL PREDICTIONS BEFORE OPENING VALIDATION GT
# ============================================================

for required_path, label in (
    (CONTRACT_PATH, "Field extraction contract"),
    (BASELINE_POINTER_PATH, "Validation baseline pointer"),
    (PREDICTION_MANIFEST_PATH, "Validation prediction manifest"),
    (RENDER_INDEX_PATH, "Batch render index"),
):
    require_file(required_path, label)
require_directory(
    VALIDATION_GROUND_TRUTH_ROOT,
    "Validation ground-truth root",
)

contract = load_json(CONTRACT_PATH)
baseline_pointer = load_json(BASELINE_POINTER_PATH)
prediction_manifest = load_json(PREDICTION_MANIFEST_PATH)
render_records = load_jsonl(RENDER_INDEX_PATH)

freeze_manifest_path = Path(
    str(baseline_pointer.get("freeze_manifest_path", ""))
)
parser_source_path = Path(
    str(baseline_pointer.get("parser_source_path", ""))
)
require_file(freeze_manifest_path, "Frozen baseline manifest")
require_file(parser_source_path, "Frozen parser source")
freeze_manifest = load_json(freeze_manifest_path)

development_evaluation_reference = (
    freeze_manifest.get("immutable_inputs", {})
    .get("development_evaluation_manifest", {})
)
development_evaluation_path = Path(
    str(development_evaluation_reference.get("path", ""))
)
require_file(
    development_evaluation_path,
    "Frozen development evaluation manifest",
)
development_evaluation = load_json(development_evaluation_path)

field_definitions = contract.get("field_definitions", [])
contract_field_map = {
    field["field"]: field
    for field in field_definitions
    if isinstance(field, dict) and field.get("field")
}
prediction_parser = prediction_manifest.get("parser", {})
prediction_scope = prediction_manifest.get("scope", {})
prediction_integrity = prediction_manifest.get("integrity", {})
freeze_parser = freeze_manifest.get("parser", {})
freeze_split_policy = freeze_manifest.get("split_policy", {})
prediction_records = prediction_manifest.get("records", [])
if not isinstance(prediction_records, list):
    raise TypeError("records pada prediction manifest bukan list.")

input_checksums_before = {
    "contract": sha256_file(CONTRACT_PATH),
    "baseline_pointer": sha256_file(BASELINE_POINTER_PATH),
    "freeze_manifest": sha256_file(freeze_manifest_path),
    "parser_source": sha256_file(parser_source_path),
    "prediction_manifest": sha256_file(PREDICTION_MANIFEST_PATH),
    "render_index": sha256_file(RENDER_INDEX_PATH),
    "development_evaluation": sha256_file(
        development_evaluation_path
    ),
}

frozen_acceptance_checks = (
    freeze_manifest.get("development_evidence", {})
    .get("acceptance_checks", [])
)
frozen_acceptance_minima = {
    str(check.get("metric")): float(check.get("minimum"))
    for check in frozen_acceptance_checks
    if isinstance(check, dict)
    and check.get("metric")
    and check.get("minimum") is not None
}
required_acceptance_metrics = {
    "critical_scalar_exact_match",
    "all_scalar_exact_match",
    "line_item_field_f1",
    "financial_consistency",
}

preflight_values = [
    (
        "contract_status",
        "FROZEN_FOR_DEVELOPMENT_BASELINE",
        contract.get("status"),
    ),
    ("target_fields", 16, len(contract_field_map)),
    (
        "target_field_names",
        sorted(
            set(SCALAR_PATHS)
            | {f"items[].{name}" for name in ITEM_FIELD_NAMES}
        ),
        sorted(contract_field_map),
    ),
    ("render_index_records", EXPECTED_RENDER_RECORDS, len(render_records)),
    (
        "baseline_pointer_status",
        "ACTIVE_FOR_VALIDATION",
        baseline_pointer.get("status"),
    ),
    (
        "baseline_id",
        EXPECTED_BASELINE_ID,
        baseline_pointer.get("baseline_id"),
    ),
    (
        "baseline_pointer_parser_id",
        EXPECTED_PARSER_ID,
        baseline_pointer.get("parser_id"),
    ),
    (
        "baseline_pointer_parser_version",
        EXPECTED_PARSER_VERSION,
        baseline_pointer.get("parser_version"),
    ),
    (
        "next_allowed_split",
        "validation",
        baseline_pointer.get("next_allowed_split"),
    ),
    (
        "test_locked_by_pointer",
        True,
        baseline_pointer.get("test_remains_locked"),
    ),
    (
        "freeze_status",
        "FROZEN_FOR_VALIDATION",
        freeze_manifest.get("status"),
    ),
    (
        "freeze_baseline_id",
        EXPECTED_BASELINE_ID,
        freeze_manifest.get("baseline_id"),
    ),
    (
        "freeze_parser_id",
        EXPECTED_PARSER_ID,
        freeze_parser.get("parser_id"),
    ),
    (
        "freeze_parser_version",
        EXPECTED_PARSER_VERSION,
        freeze_parser.get("parser_version"),
    ),
    (
        "freeze_parser_signature",
        EXPECTED_PARSER_SIGNATURE,
        freeze_parser.get("parser_signature_sha256"),
    ),
    (
        "freeze_checksum",
        baseline_pointer.get("freeze_manifest_sha256"),
        input_checksums_before["freeze_manifest"],
    ),
    (
        "parser_source_checksum",
        baseline_pointer.get("parser_source_sha256"),
        input_checksums_before["parser_source"],
    ),
    (
        "parser_source_matches_freeze_path",
        True,
        Path(str(freeze_parser.get("source_path", ""))).resolve()
        == parser_source_path.resolve(),
    ),
    (
        "parser_source_matches_freeze_checksum",
        input_checksums_before["parser_source"],
        freeze_parser.get("source_sha256"),
    ),
    (
        "freeze_next_allowed_split",
        "validation",
        freeze_split_policy.get("next_allowed_split"),
    ),
    (
        "test_locked_by_freeze",
        True,
        freeze_split_policy.get("test_remains_locked"),
    ),
    (
        "prediction_manifest_status",
        "EXECUTED",
        prediction_manifest.get("status"),
    ),
    (
        "prediction_quality_before_evaluation",
        "PENDING_VALIDATION_GROUND_TRUTH_EVALUATION",
        prediction_manifest.get("quality_status"),
    ),
    (
        "prediction_stage",
        "FROZEN_PARSER_BLIND_VALIDATION_PREDICTION",
        prediction_manifest.get("stage"),
    ),
    (
        "prediction_parser_id",
        EXPECTED_PARSER_ID,
        prediction_parser.get("parser_id"),
    ),
    (
        "prediction_parser_version",
        EXPECTED_PARSER_VERSION,
        prediction_parser.get("parser_version"),
    ),
    (
        "prediction_parser_signature",
        EXPECTED_PARSER_SIGNATURE,
        prediction_parser.get("parser_signature_sha256"),
    ),
    (
        "prediction_parser_source_path",
        True,
        Path(str(prediction_parser.get("parser_source_path", ""))).resolve()
        == parser_source_path.resolve(),
    ),
    (
        "prediction_parser_source_checksum",
        input_checksums_before["parser_source"],
        prediction_parser.get("parser_source_sha256"),
    ),
    (
        "prediction_freeze_checksum",
        input_checksums_before["freeze_manifest"],
        prediction_parser.get("freeze_manifest_sha256"),
    ),
    ("prediction_split", "validation", prediction_scope.get("split")),
    (
        "prediction_documents",
        EXPECTED_DOCUMENTS,
        prediction_scope.get("documents"),
    ),
    (
        "prediction_templates",
        sorted(EXPECTED_TEMPLATES),
        sorted(prediction_scope.get("templates", [])),
    ),
    (
        "predictions_frozen_before_gt",
        True,
        len(prediction_records) == EXPECTED_DOCUMENTS,
    ),
    (
        "ground_truth_loaded_before_evaluation",
        False,
        prediction_integrity.get("ground_truth_loaded"),
    ),
    (
        "ground_truth_used_as_prediction_input",
        False,
        prediction_integrity.get("ground_truth_used_as_prediction_input"),
    ),
    (
        "validation_gt_opened_before_evaluation",
        0,
        prediction_integrity.get("validation_ground_truth_opened"),
    ),
    ("test_opened", 0, prediction_integrity.get("test_opened")),
    (
        "frozen_acceptance_metrics",
        sorted(required_acceptance_metrics),
        sorted(frozen_acceptance_minima),
    ),
    (
        "frozen_acceptance_checks_passed",
        True,
        bool(frozen_acceptance_checks)
        and all(
            check.get("status") == "PASSED"
            for check in frozen_acceptance_checks
            if isinstance(check, dict)
        ),
    ),
    (
        "development_evaluation_status",
        "PASSED",
        development_evaluation.get("status"),
    ),
    (
        "development_evaluator_id",
        EVALUATOR_ID,
        development_evaluation.get("evaluator", {}).get("evaluator_id"),
    ),
    (
        "development_evaluator_version",
        EVALUATOR_VERSION,
        development_evaluation.get("evaluator", {}).get(
            "evaluator_version"
        ),
    ),
    (
        "development_evaluation_checksum",
        development_evaluation_reference.get("sha256"),
        input_checksums_before["development_evaluation"],
    ),
]

preflight_controls = [
    {
        "control": name,
        "expected": expected,
        "actual": actual,
        "status": "VALID" if expected == actual else "INVALID",
    }
    for name, expected, actual in preflight_values
]

print("=" * 96)
print(
    f"CELL 11D — {EVALUATOR_ID} — VERSION {EVALUATOR_VERSION} — "
    "VALIDATION EVALUATION"
)
print(f"Evaluation root: {EVALUATION_ROOT}")
print("Predictions: FROZEN | Validation GT: AUTHORIZED | Test: LOCKED")
print("=" * 96)
print("PREFLIGHT CONTROLS — BEFORE VALIDATION GROUND TRUTH IS OPENED")
display(pd.DataFrame(preflight_controls))

invalid_preflight = [
    row["control"]
    for row in preflight_controls
    if row["status"] != "VALID"
]
if invalid_preflight:
    raise RuntimeError(
        "CELL 11D PREFLIGHT FAILED. "
        f"Kontrol tidak valid: {invalid_preflight}. "
        "Validation ground truth belum dibuka dan test tetap terkunci."
    )


# ============================================================
# VERIFY ALL FROZEN PREDICTIONS BEFORE BUILDING THE GT PLAN
# ============================================================

prediction_records = sorted(
    prediction_records,
    key=lambda row: (
        int(row.get("sequence_number", 0)),
        str(row.get("document_id", "")),
    ),
)
prediction_source_errors = []

for record_index, prediction_record in enumerate(
    prediction_records,
    start=1,
):
    try:
        if not isinstance(prediction_record, dict):
            raise TypeError("Prediction manifest record bukan object.")

        document_id = str(prediction_record.get("document_id", ""))
        template_id = str(prediction_record.get("template_id", ""))
        prediction_path = Path(
            str(prediction_record.get("prediction_path", ""))
        )

        if not document_id:
            raise RuntimeError("document_id kosong.")
        if template_id not in EXPECTED_TEMPLATES:
            raise RuntimeError(
                f"Template prediction di luar validation: {template_id}"
            )

        require_file(prediction_path, f"Prediction {document_id}")
        actual_checksum = sha256_file(prediction_path)
        if actual_checksum != prediction_record.get("prediction_sha256"):
            raise RuntimeError(
                f"Checksum prediction tidak cocok: {document_id}"
            )

        prediction = load_json(prediction_path)
        document = prediction.get("document", {})
        parser = prediction.get("parser", {})
        integrity = prediction.get("integrity", {})

        if prediction.get("status") != "EXECUTED":
            raise RuntimeError("Status prediction bukan EXECUTED.")
        if prediction.get("quality_status") != (
            "PENDING_VALIDATION_GROUND_TRUTH_EVALUATION"
        ):
            raise RuntimeError("Quality status prediction tidak valid.")
        if document.get("document_id") != document_id:
            raise RuntimeError("Prediction document ID tidak cocok.")
        if document.get("template_id") != template_id:
            raise RuntimeError("Prediction template ID tidak cocok.")
        if document.get("split") != "validation":
            raise RuntimeError("Prediction bukan validation split.")
        if parser.get("parser_id") != EXPECTED_PARSER_ID:
            raise RuntimeError("Prediction parser ID tidak cocok.")
        if parser.get("parser_version") != EXPECTED_PARSER_VERSION:
            raise RuntimeError("Prediction parser version tidak cocok.")
        if parser.get("parser_signature_sha256") != (
            EXPECTED_PARSER_SIGNATURE
        ):
            raise RuntimeError("Prediction parser signature tidak cocok.")
        if integrity.get("ground_truth_loaded") is not False:
            raise RuntimeError("Prediction menandai ground truth telah dibuka.")
        if integrity.get("ground_truth_used_as_prediction_input") is not False:
            raise RuntimeError("Ground truth terindikasi sebagai input parser.")
        if integrity.get("validation_ground_truth_opened") != 0:
            raise RuntimeError("Prediction dibuat setelah validation GT dibuka.")
        if integrity.get("test_opened") != 0:
            raise RuntimeError("Prediction menandai test telah dibuka.")

        input_checksums_before[
            f"prediction:{document_id}"
        ] = actual_checksum

    except Exception as error:
        prediction_source_errors.append(
            {
                "record_index": record_index,
                "document_id": (
                    prediction_record.get("document_id")
                    if isinstance(prediction_record, dict)
                    else None
                ),
                "error_type": type(error).__name__,
                "error": str(error)[:700],
            }
        )

prediction_gate_values = [
    ("prediction_records", EXPECTED_DOCUMENTS, len(prediction_records)),
    (
        "unique_prediction_ids",
        EXPECTED_DOCUMENTS,
        len({row.get("document_id") for row in prediction_records}),
    ),
    (
        "prediction_templates",
        sorted(EXPECTED_TEMPLATES),
        sorted({row.get("template_id") for row in prediction_records}),
    ),
    (
        "prediction_sequence_numbers",
        list(range(1, EXPECTED_DOCUMENTS + 1)),
        [int(row.get("sequence_number", 0)) for row in prediction_records],
    ),
    ("prediction_source_errors", 0, len(prediction_source_errors)),
]

prediction_gate_controls = [
    {
        "control": name,
        "expected": expected,
        "actual": actual,
        "status": "VALID" if expected == actual else "INVALID",
    }
    for name, expected, actual in prediction_gate_values
]

print("\nFROZEN PREDICTION CONTROLS")
display(pd.DataFrame(prediction_gate_controls))
if prediction_source_errors:
    display(pd.DataFrame(prediction_source_errors))

invalid_prediction_gates = [
    row["control"]
    for row in prediction_gate_controls
    if row["status"] != "VALID"
]
if invalid_prediction_gates:
    raise RuntimeError(
        "CELL 11D PREDICTION GATE FAILED. "
        f"Kontrol tidak valid: {invalid_prediction_gates}. "
        "Validation ground truth belum dibuka."
    )


# Build the access plan using metadata only. No GT JSON is loaded here.
render_record_map = build_render_record_map(render_records)
gt_plan = ground_truth_access_plan(
    prediction_records,
    render_record_map,
)

gt_plan_values = [
    ("validation_gt_plan_records", EXPECTED_DOCUMENTS, len(gt_plan)),
    (
        "unique_validation_gt_paths",
        EXPECTED_DOCUMENTS,
        len({record["path"] for record in gt_plan.values()}),
    ),
    (
        "validation_gt_templates",
        sorted(EXPECTED_TEMPLATES),
        sorted({record["template_id"] for record in gt_plan.values()}),
    ),
    (
        "all_paths_inside_validation_root",
        True,
        all(
            Path(record["path"]).resolve().is_relative_to(
                VALIDATION_GROUND_TRUTH_ROOT.resolve()
            )
            for record in gt_plan.values()
        ),
    ),
    ("validation_gt_payloads_opened_so_far", 0, 0),
    ("test_payloads_opened", 0, 0),
]

gt_plan_controls = [
    {
        "control": name,
        "expected": expected,
        "actual": actual,
        "status": "VALID" if expected == actual else "INVALID",
    }
    for name, expected, actual in gt_plan_values
]
print("\nVALIDATION GROUND-TRUTH ACCESS PLAN")
display(pd.DataFrame(gt_plan_controls))

invalid_gt_plan = [
    row["control"]
    for row in gt_plan_controls
    if row["status"] != "VALID"
]
if invalid_gt_plan:
    raise RuntimeError(
        "CELL 11D VALIDATION GT PLAN FAILED. "
        f"Kontrol tidak valid: {invalid_gt_plan}."
    )

print("\nFROZEN ACCEPTANCE THRESHOLDS")
display(
    pd.DataFrame(
        [
            {
                "metric": metric,
                "minimum": frozen_acceptance_minima[metric],
                "source": "FROZEN_BEFORE_VALIDATION",
            }
            for metric in sorted(frozen_acceptance_minima)
        ]
    )
)


# ============================================================
# OPEN EXACTLY 40 VALIDATION GT FILES AND EVALUATE
# ============================================================

runtime_records = []
document_manifest_records = []
all_scalar_records = []
all_item_field_records = []
evaluation_errors = []
ground_truth_paths = {}
ground_truth_checksums = {}
new_evaluations = 0
recovered_evaluations = 0

print(
    f"\nMengevaluasi {len(prediction_records)} frozen validation "
    "predictions...\n"
)

for sequence_number, prediction_record in enumerate(
    prediction_records,
    start=1,
):
    document_id = str(prediction_record["document_id"])
    template_id = str(prediction_record["template_id"])
    prediction_path = Path(prediction_record["prediction_path"])
    evaluation_path = (
        DOCUMENT_EVALUATION_ROOT
        / template_id
        / f"{document_id}_evaluation.json"
    )

    try:
        prediction = load_json(prediction_path)
        gt_record = gt_plan[document_id]
        ground_truth_path = Path(gt_record["path"])

        # This is the first point at which validation GT bytes and JSON are
        # opened. Only the exact path planned above is permitted.
        actual_gt_checksum = sha256_file(ground_truth_path)
        if actual_gt_checksum != gt_record["expected_sha256"]:
            raise RuntimeError(
                f"Checksum validation GT tidak cocok: {document_id}"
            )
        ground_truth = load_json(ground_truth_path)
        canonical = canonical_from_ground_truth(
            ground_truth,
            document_id,
            template_id,
        )

        ground_truth_paths[document_id] = ground_truth_path
        ground_truth_checksums[document_id] = actual_gt_checksum

        result = evaluate_document(
            prediction,
            canonical,
            contract_field_map,
        )
        metrics = result["metrics"]

        evaluation_artifact = {
            "schema_version": "1.0.0",
            "status": "EVALUATED",
            "evaluator": {
                "evaluator_id": EVALUATOR_ID,
                "evaluator_version": EVALUATOR_VERSION,
                "scalar_matching": (
                    "TYPE_AWARE_NORMALIZED_EXACT_MATCH"
                ),
                "item_alignment": (
                    "MAXIMUM_WEIGHT_BIPARTITE_ROW_MATCHING"
                ),
                "acceptance_threshold_source": (
                    "FROZEN_DEVELOPMENT_FREEZE_MANIFEST"
                ),
            },
            "document": {
                "document_id": document_id,
                "template_id": template_id,
                "split": "validation",
                "language": result["language"],
            },
            "metrics": metrics,
            "scalar_comparisons": result["scalar_records"],
            "item_alignment": result["item_alignment"],
            "item_field_comparisons": result["item_field_records"],
            "inputs": {
                "prediction_path": str(prediction_path),
                "prediction_sha256": prediction_record[
                    "prediction_sha256"
                ],
                "ground_truth_path": str(ground_truth_path),
                "ground_truth_sha256": actual_gt_checksum,
            },
            "integrity": {
                "predictions_frozen_before_ground_truth_open": True,
                "ground_truth_split": "validation",
                "ground_truth_used_for_evaluation_only": True,
                "ground_truth_used_as_prediction_input": False,
                "test_opened": 0,
                "prediction_modified": False,
                "ground_truth_modified": False,
            },
        }

        checkpoint_action = save_immutable_json(
            evaluation_path,
            evaluation_artifact,
        )
        if checkpoint_action == "CREATED":
            new_evaluations += 1
            execution = "NEW"
        else:
            recovered_evaluations += 1
            execution = "RECOVERED"

        persisted = load_json(evaluation_path)
        if canonical_json(persisted) != canonical_json(
            evaluation_artifact
        ):
            raise RuntimeError(
                f"Evaluation berbeda setelah penulisan: {document_id}"
            )

        record = {
            "sequence_number": sequence_number,
            "document_id": document_id,
            "template_id": template_id,
            "language": result["language"],
            "scalar_exact_match": metrics["scalar_exact_match"],
            "predicted_item_count": metrics["predicted_item_count"],
            "reference_item_count": metrics["reference_item_count"],
            "item_count_exact": metrics["item_count_exact"],
            "row_precision": metrics["row_precision"],
            "row_recall": metrics["row_recall"],
            "row_f1": metrics["row_f1"],
            "item_field_precision": metrics["item_field_precision"],
            "item_field_recall": metrics["item_field_recall"],
            "item_field_f1": metrics["item_field_f1"],
            "financial_consistent": metrics["financial_consistent"],
            "document_exact_match": metrics["document_exact_match"],
            "evaluation_path": str(evaluation_path),
            "evaluation_sha256": sha256_file(evaluation_path),
            "prediction_path": str(prediction_path),
            "prediction_sha256": prediction_record[
                "prediction_sha256"
            ],
            "ground_truth_path": str(ground_truth_path),
            "ground_truth_sha256": actual_gt_checksum,
            "status": "EVALUATED",
        }
        document_manifest_records.append(record)
        runtime_records.append({**record, "execution": execution})
        all_scalar_records.extend(result["scalar_records"])
        all_item_field_records.extend(result["item_field_records"])

        print(
            f"[{sequence_number:02d}/{len(prediction_records):02d}] "
            f"{document_id} | "
            f"scalar={metrics['scalar_exact_match']:.4f} | "
            f"items={metrics['predicted_item_count']}/"
            f"{metrics['reference_item_count']} | "
            f"item_f1={metrics['item_field_f1']:.4f} | "
            f"exact={metrics['document_exact_match']} | {execution}"
        )

    except Exception as error:
        evaluation_errors.append(
            {
                "sequence_number": sequence_number,
                "document_id": document_id,
                "template_id": template_id,
                "error_type": type(error).__name__,
                "error": str(error)[:900],
            }
        )
        print(
            f"[{sequence_number:02d}/{len(prediction_records):02d}] "
            f"{document_id} | ERROR: "
            f"{type(error).__name__}: {error}"
        )


# ============================================================
# TECHNICAL COMPLETENESS GATE
# ============================================================

if evaluation_errors:
    print("\nEVALUATION ERRORS")
    display(pd.DataFrame(evaluation_errors))

technical_values = [
    (
        "evaluated_documents",
        EXPECTED_DOCUMENTS,
        len(document_manifest_records),
    ),
    (
        "evaluation_files",
        EXPECTED_DOCUMENTS,
        sum(
            Path(record["evaluation_path"]).is_file()
            for record in document_manifest_records
        ),
    ),
    (
        "unique_document_ids",
        EXPECTED_DOCUMENTS,
        len({record["document_id"] for record in document_manifest_records}),
    ),
    (
        "templates_evaluated",
        sorted(EXPECTED_TEMPLATES),
        sorted(
            {record["template_id"] for record in document_manifest_records}
        ),
    ),
    (
        "validation_ground_truth_opened",
        EXPECTED_DOCUMENTS,
        len(ground_truth_paths),
    ),
    ("test_opened", 0, 0),
    ("evaluation_errors", 0, len(evaluation_errors)),
]

technical_control_records = [
    {
        "control": name,
        "expected": expected,
        "actual": actual,
        "status": "VALID" if expected == actual else "INVALID",
    }
    for name, expected, actual in technical_values
]

print("\nTECHNICAL EVALUATION CONTROLS")
display(pd.DataFrame(technical_control_records))

invalid_technical_controls = [
    row["control"]
    for row in technical_control_records
    if row["status"] != "VALID"
]
if invalid_technical_controls:
    raise RuntimeError(
        "CELL 11D TECHNICAL EVALUATION FAILED. "
        f"Kontrol tidak valid: {invalid_technical_controls}. "
        "Test tetap terkunci."
    )


# ============================================================
# AGGREGATE VALIDATION METRICS
# ============================================================

scalar_total = len(all_scalar_records)
scalar_exact = sum(
    record["exact_match"] for record in all_scalar_records
)
critical_scalar_records = [
    record for record in all_scalar_records if record["critical"]
]
critical_scalar_exact = sum(
    record["exact_match"] for record in critical_scalar_records
)

all_scalar_exact_match = safe_ratio(scalar_exact, scalar_total)
critical_scalar_exact_match = safe_ratio(
    critical_scalar_exact,
    len(critical_scalar_records),
)

scalar_character_errors = sum(
    record["character_errors"] for record in all_scalar_records
)
scalar_reference_characters = sum(
    record["reference_characters"] for record in all_scalar_records
)
scalar_word_errors = sum(
    record["word_errors"] for record in all_scalar_records
)
scalar_reference_words = sum(
    record["reference_words"] for record in all_scalar_records
)
scalar_micro_cer = safe_error_rate(
    scalar_character_errors,
    scalar_reference_characters,
)
scalar_micro_wer = safe_error_rate(
    scalar_word_errors,
    scalar_reference_words,
)

item_exact_fields = sum(
    record["exact_match"] for record in all_item_field_records
)
predicted_item_field_count = sum(
    record["predicted_item_count"] * EXPECTED_ITEM_FIELDS
    for record in document_manifest_records
)
reference_item_field_count = sum(
    record["reference_item_count"] * EXPECTED_ITEM_FIELDS
    for record in document_manifest_records
)
line_item_field_precision = safe_ratio(
    item_exact_fields,
    predicted_item_field_count,
)
line_item_field_recall = safe_ratio(
    item_exact_fields,
    reference_item_field_count,
)
line_item_field_f1 = harmonic_mean(
    line_item_field_precision,
    line_item_field_recall,
)

row_true_positives = 0
predicted_rows = 0
reference_rows = 0
for record in document_manifest_records:
    evaluation = load_json(Path(record["evaluation_path"]))
    metrics = evaluation["metrics"]
    row_true_positives += int(metrics["row_true_positives"])
    predicted_rows += int(metrics["predicted_item_count"])
    reference_rows += int(metrics["reference_item_count"])

row_precision = safe_ratio(row_true_positives, predicted_rows)
row_recall = safe_ratio(row_true_positives, reference_rows)
row_f1 = harmonic_mean(row_precision, row_recall)

financial_consistency = safe_ratio(
    sum(
        record["financial_consistent"]
        for record in document_manifest_records
    ),
    len(document_manifest_records),
)
document_exact_match = safe_ratio(
    sum(
        record["document_exact_match"]
        for record in document_manifest_records
    ),
    len(document_manifest_records),
)
item_count_exact_match = safe_ratio(
    sum(
        record["item_count_exact"]
        for record in document_manifest_records
    ),
    len(document_manifest_records),
)

overall_metrics = {
    "documents": len(document_manifest_records),
    "templates": len(EXPECTED_TEMPLATES),
    "scalar_field_observations": scalar_total,
    "scalar_field_exact": scalar_exact,
    "all_scalar_exact_match": all_scalar_exact_match,
    "critical_scalar_observations": len(critical_scalar_records),
    "critical_scalar_exact": critical_scalar_exact,
    "critical_scalar_exact_match": critical_scalar_exact_match,
    "scalar_micro_cer": scalar_micro_cer,
    "scalar_micro_wer": scalar_micro_wer,
    "predicted_item_rows": predicted_rows,
    "reference_item_rows": reference_rows,
    "matched_item_rows": row_true_positives,
    "row_precision": row_precision,
    "row_recall": row_recall,
    "row_f1": row_f1,
    "item_field_exact": item_exact_fields,
    "predicted_item_fields": predicted_item_field_count,
    "reference_item_fields": reference_item_field_count,
    "line_item_field_precision": line_item_field_precision,
    "line_item_field_recall": line_item_field_recall,
    "line_item_field_f1": line_item_field_f1,
    "item_count_exact_match": item_count_exact_match,
    "financial_consistency": financial_consistency,
    "document_exact_match": document_exact_match,
}


# ============================================================
# FIELD AND TEMPLATE SUMMARIES
# ============================================================

combined_field_records = all_scalar_records + all_item_field_records
field_summary_records = []

for field_name in sorted(
    {record["field"] for record in combined_field_records}
):
    records = [
        record
        for record in combined_field_records
        if record["field"] == field_name
    ]
    exact_count = sum(record["exact_match"] for record in records)
    character_errors = sum(
        record["character_errors"] for record in records
    )
    reference_characters = sum(
        record["reference_characters"] for record in records
    )
    word_errors = sum(record["word_errors"] for record in records)
    reference_words = sum(
        record["reference_words"] for record in records
    )

    field_summary_records.append(
        {
            "field": field_name,
            "group": records[0]["group"],
            "critical": records[0]["critical"],
            "observations": len(records),
            "exact_matches": exact_count,
            "exact_match_rate": safe_ratio(exact_count, len(records)),
            "character_errors": character_errors,
            "reference_characters": reference_characters,
            "cer": safe_error_rate(
                character_errors,
                reference_characters,
            ),
            "word_errors": word_errors,
            "reference_words": reference_words,
            "wer": safe_error_rate(word_errors, reference_words),
            "status": (
                "PERFECT" if exact_count == len(records) else "HAS_ERRORS"
            ),
        }
    )

template_summary_records = []
for template_id in sorted(EXPECTED_TEMPLATES):
    documents = [
        record
        for record in document_manifest_records
        if record["template_id"] == template_id
    ]
    scalar_records = [
        record
        for record in all_scalar_records
        if record["template_id"] == template_id
    ]
    item_records = [
        record
        for record in all_item_field_records
        if record["template_id"] == template_id
    ]

    scalar_correct = sum(
        record["exact_match"] for record in scalar_records
    )
    item_correct = sum(record["exact_match"] for record in item_records)
    template_predicted_item_fields = sum(
        record["predicted_item_count"] * EXPECTED_ITEM_FIELDS
        for record in documents
    )
    template_reference_item_fields = sum(
        record["reference_item_count"] * EXPECTED_ITEM_FIELDS
        for record in documents
    )
    item_precision = safe_ratio(
        item_correct,
        template_predicted_item_fields,
    )
    item_recall = safe_ratio(
        item_correct,
        template_reference_item_fields,
    )

    template_summary_records.append(
        {
            "template_id": template_id,
            "documents": len(documents),
            "scalar_exact_match": safe_ratio(
                scalar_correct,
                len(scalar_records),
            ),
            "line_item_field_f1": harmonic_mean(
                item_precision,
                item_recall,
            ),
            "item_count_exact_match": safe_ratio(
                sum(record["item_count_exact"] for record in documents),
                len(documents),
            ),
            "financial_consistency": safe_ratio(
                sum(record["financial_consistent"] for record in documents),
                len(documents),
            ),
            "document_exact_match": safe_ratio(
                sum(record["document_exact_match"] for record in documents),
                len(documents),
            ),
        }
    )


# ============================================================
# APPLY PRE-FROZEN ACCEPTANCE THRESHOLDS
# ============================================================

metric_values = {
    "critical_scalar_exact_match": critical_scalar_exact_match,
    "all_scalar_exact_match": all_scalar_exact_match,
    "line_item_field_f1": line_item_field_f1,
    "financial_consistency": financial_consistency,
}

acceptance_checks = []
for metric_name in sorted(required_acceptance_metrics):
    minimum = frozen_acceptance_minima[metric_name]
    actual = metric_values[metric_name]
    acceptance_checks.append(
        {
            "metric": metric_name,
            "minimum": minimum,
            "actual": actual,
            "threshold_source": "FROZEN_BEFORE_VALIDATION",
            "status": "PASSED" if actual >= minimum else "FAILED",
        }
    )

acceptance_status = (
    "PASSED"
    if all(check["status"] == "PASSED" for check in acceptance_checks)
    else "FAILED"
)


# ============================================================
# MISMATCH DETAILS AND RESULT DISPLAY
# ============================================================

mismatch_records = []
for record in combined_field_records:
    if record["exact_match"]:
        continue
    mismatch_records.append(
        {
            "document_id": record["document_id"],
            "template_id": record["template_id"],
            "language": record["language"],
            "group": record["group"],
            "field": record["field"],
            "prediction_row": record.get("prediction_row"),
            "reference_row": record.get("reference_row"),
            "expected": record["expected"],
            "predicted": record["predicted"],
            "character_errors": record["character_errors"],
            "word_errors": record["word_errors"],
        }
    )

document_table = pd.DataFrame(runtime_records)
field_table = pd.DataFrame(field_summary_records)
template_table = pd.DataFrame(template_summary_records)
acceptance_table = pd.DataFrame(acceptance_checks)

print("\nVALIDATION ACCEPTANCE RESULTS")
display(acceptance_table)
display(field_table)
display(template_table)
display(
    document_table[
        [
            "sequence_number",
            "document_id",
            "template_id",
            "language",
            "scalar_exact_match",
            "predicted_item_count",
            "reference_item_count",
            "row_f1",
            "item_field_f1",
            "financial_consistent",
            "document_exact_match",
            "execution",
        ]
    ]
)

if mismatch_records:
    print("\nVALIDATION MISMATCH DETAILS")
    display(pd.DataFrame(mismatch_records))


# ============================================================
# IMMUTABLE SUMMARIES AND VALIDATION EVALUATION MANIFEST
# ============================================================

document_summary_action = save_csv_checkpoint(
    DOCUMENT_SUMMARY_PATH,
    pd.DataFrame(document_manifest_records),
)
field_summary_action = save_csv_checkpoint(
    FIELD_SUMMARY_PATH,
    field_table,
)
template_summary_action = save_csv_checkpoint(
    TEMPLATE_SUMMARY_PATH,
    template_table,
)

mismatch_columns = [
    "document_id",
    "template_id",
    "language",
    "group",
    "field",
    "prediction_row",
    "reference_row",
    "expected",
    "predicted",
    "character_errors",
    "word_errors",
]
mismatch_table = pd.DataFrame(
    mismatch_records,
    columns=mismatch_columns,
)
mismatch_action = save_csv_checkpoint(
    MISMATCH_DETAIL_PATH,
    mismatch_table,
)

evaluation_protocol = {
    "evaluator_id": EVALUATOR_ID,
    "evaluator_version": EVALUATOR_VERSION,
    "scalar_matching": "TYPE_AWARE_NORMALIZED_EXACT_MATCH",
    "item_alignment": "MAXIMUM_WEIGHT_BIPARTITE_ROW_MATCHING",
    "row_match": "DESCRIPTION_AND_LINE_TOTAL_EXACT",
    "acceptance_threshold_source": (
        "DEVELOPMENT_FREEZE_MANIFEST_FROZEN_BEFORE_VALIDATION"
    ),
    "acceptance_minima": dict(sorted(frozen_acceptance_minima.items())),
}
evaluation_protocol_sha256 = hashlib.sha256(
    canonical_json(evaluation_protocol).encode("utf-8")
).hexdigest()

evaluation_manifest = {
    "schema_version": "1.0.0",
    "cell_version": CELL_VERSION,
    "status": acceptance_status,
    "stage": "VALIDATION_GROUND_TRUTH_EVALUATION",
    "baseline": {
        "baseline_id": EXPECTED_BASELINE_ID,
        "parser_id": EXPECTED_PARSER_ID,
        "parser_version": EXPECTED_PARSER_VERSION,
        "parser_signature_sha256": EXPECTED_PARSER_SIGNATURE,
        "freeze_manifest_path": str(freeze_manifest_path),
        "freeze_manifest_sha256": input_checksums_before[
            "freeze_manifest"
        ],
    },
    "evaluator": {
        **evaluation_protocol,
        "evaluation_protocol_sha256": evaluation_protocol_sha256,
        "development_evaluator_manifest_path": str(
            development_evaluation_path
        ),
        "development_evaluator_manifest_sha256": (
            input_checksums_before["development_evaluation"]
        ),
    },
    "scope": {
        "split": "validation",
        "documents": EXPECTED_DOCUMENTS,
        "templates": sorted(EXPECTED_TEMPLATES),
        "validation_ground_truth_opened": len(ground_truth_paths),
        "test_opened": 0,
    },
    "overall_metrics": overall_metrics,
    "acceptance_checks": acceptance_checks,
    "technical_controls": technical_control_records,
    "field_summary": field_summary_records,
    "template_summary": template_summary_records,
    "records": document_manifest_records,
    "artifacts": {
        "evaluation_root": str(EVALUATION_ROOT),
        "document_summary": {
            "path": str(DOCUMENT_SUMMARY_PATH),
            "sha256": sha256_file(DOCUMENT_SUMMARY_PATH),
        },
        "field_summary": {
            "path": str(FIELD_SUMMARY_PATH),
            "sha256": sha256_file(FIELD_SUMMARY_PATH),
        },
        "template_summary": {
            "path": str(TEMPLATE_SUMMARY_PATH),
            "sha256": sha256_file(TEMPLATE_SUMMARY_PATH),
        },
        "mismatch_details": {
            "path": str(MISMATCH_DETAIL_PATH),
            "sha256": sha256_file(MISMATCH_DETAIL_PATH),
            "records": len(mismatch_records),
        },
    },
    "inputs": {
        "contract": {
            "path": str(CONTRACT_PATH),
            "sha256": input_checksums_before["contract"],
        },
        "baseline_pointer": {
            "path": str(BASELINE_POINTER_PATH),
            "sha256": input_checksums_before["baseline_pointer"],
        },
        "prediction_manifest": {
            "path": str(PREDICTION_MANIFEST_PATH),
            "sha256": input_checksums_before["prediction_manifest"],
        },
        "render_index": {
            "path": str(RENDER_INDEX_PATH),
            "sha256": input_checksums_before["render_index"],
        },
        "prediction_files": {
            record["document_id"]: {
                "path": record["prediction_path"],
                "sha256": record["prediction_sha256"],
            }
            for record in document_manifest_records
        },
        "validation_ground_truth_files": {
            document_id: {
                "path": str(ground_truth_paths[document_id]),
                "sha256": ground_truth_checksums[document_id],
            }
            for document_id in sorted(ground_truth_paths)
        },
    },
    "integrity": {
        "predictions_frozen_before_ground_truth_open": True,
        "ground_truth_used_for_evaluation_only": True,
        "ground_truth_used_as_prediction_input": False,
        "validation_ground_truth_opened": len(ground_truth_paths),
        "test_opened": 0,
        "parser_modifications": 0,
        "prediction_modifications": 0,
        "ground_truth_modifications": 0,
        "dataset_modifications": 0,
        "source_modifications": 0,
    },
    "next_stage": {
        "cell": "CELL 11E",
        "action_if_passed": (
            "FREEZE_VALIDATION_DECISION_AND_PREPARE_TEST_PREFLIGHT"
        ),
        "action_if_failed": (
            "FREEZE_FAILED_VALIDATION_RESULT_AND_KEEP_TEST_LOCKED"
        ),
        "silent_parser_changes_allowed": False,
        "test_remains_locked_until_cell_11e": True,
    },
}

manifest_action = save_immutable_json(
    EVALUATION_MANIFEST_PATH,
    evaluation_manifest,
)

persisted_manifest = load_json(EVALUATION_MANIFEST_PATH)
if canonical_json(persisted_manifest) != canonical_json(
    evaluation_manifest
):
    raise RuntimeError("Manifest Cell 11D berbeda setelah penulisan.")


# ============================================================
# FINAL INPUT AND GROUND-TRUTH IMMUTABILITY CHECK
# ============================================================

input_checksums_after = {
    "contract": sha256_file(CONTRACT_PATH),
    "baseline_pointer": sha256_file(BASELINE_POINTER_PATH),
    "freeze_manifest": sha256_file(freeze_manifest_path),
    "parser_source": sha256_file(parser_source_path),
    "prediction_manifest": sha256_file(PREDICTION_MANIFEST_PATH),
    "render_index": sha256_file(RENDER_INDEX_PATH),
    "development_evaluation": sha256_file(
        development_evaluation_path
    ),
}
for prediction_record in prediction_records:
    input_checksums_after[
        f"prediction:{prediction_record['document_id']}"
    ] = sha256_file(Path(prediction_record["prediction_path"]))

changed_inputs = [
    name
    for name, checksum in input_checksums_before.items()
    if input_checksums_after.get(name) != checksum
]
changed_ground_truth = [
    document_id
    for document_id, checksum in ground_truth_checksums.items()
    if sha256_file(ground_truth_paths[document_id]) != checksum
]

if changed_inputs or changed_ground_truth:
    raise RuntimeError(
        "Input evaluation berubah selama Cell 11D: "
        f"inputs={changed_inputs}, "
        f"ground_truth={changed_ground_truth}"
    )


print()
print(f"Cell version             : {CELL_VERSION}")
print(f"Evaluation status        : {acceptance_status}")
print(f"Documents evaluated      : {len(document_manifest_records)}")
print(f"Templates                : {len(EXPECTED_TEMPLATES)}")
print(f"All scalar exact match   : {all_scalar_exact_match:.6f}")
print(f"Critical scalar exact    : {critical_scalar_exact_match:.6f}")
print(f"Scalar micro CER         : {scalar_micro_cer:.6f}")
print(f"Scalar micro WER         : {scalar_micro_wer:.6f}")
print(f"Row precision            : {row_precision:.6f}")
print(f"Row recall               : {row_recall:.6f}")
print(f"Row F1                   : {row_f1:.6f}")
print(f"Line-item field F1       : {line_item_field_f1:.6f}")
print(f"Item-count exact match   : {item_count_exact_match:.6f}")
print(f"Financial consistency    : {financial_consistency:.6f}")
print(f"Document exact match     : {document_exact_match:.6f}")
print(f"Mismatch records         : {len(mismatch_records)}")
print(f"New evaluations          : {new_evaluations}")
print(f"Recovered evaluations    : {recovered_evaluations}")
print(f"Document summary action  : {document_summary_action}")
print(f"Field summary action     : {field_summary_action}")
print(f"Template summary action  : {template_summary_action}")
print(f"Mismatch file action     : {mismatch_action}")
print(f"Manifest action          : {manifest_action}")
print(f"Evaluation root          : {EVALUATION_ROOT}")
print(f"Evaluation manifest      : {EVALUATION_MANIFEST_PATH}")
print(f"Manifest SHA-256         : {sha256_file(EVALUATION_MANIFEST_PATH)}")
print(f"Validation GT opened     : {len(ground_truth_paths)}")
print("Test opened              : 0")
print("Parser modifications     : 0")
print("Prediction modifications : 0")
print("Ground-truth modifications: 0")
print("Dataset modifications    : 0")
print("Source modifications     : 0")

if acceptance_status != "PASSED":
    failed_metrics = [
        check["metric"]
        for check in acceptance_checks
        if check["status"] != "PASSED"
    ]
    raise RuntimeError(
        "VALIDATION BASELINE BELOW PRE-FROZEN ACCEPTANCE TARGET. "
        f"Metrik gagal: {failed_metrics}. "
        "Hasil FAILED sudah disimpan secara immutable; parser v1.0.2 "
        "tidak boleh diubah diam-diam dan test tetap terkunci."
    )

print()
print(
    "✅ CELL 11D PASSED — frozen parser v1.0.2 memenuhi seluruh "
    "pre-frozen acceptance target pada 40 dokumen validation. "
    "Validation ground truth hanya digunakan untuk evaluasi; test "
    "tetap terkunci sampai keputusan Cell 11E dibekukan."
)


CELL 11D — INVOICE-FIELD-EVALUATOR-V1 — VERSION 1.0.2 — VALIDATION EVALUATION
Evaluation root: /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/field_extraction/validation/rule_based_baseline_v1_0_2/evaluations/validation_eval_v1_0_2
Predictions: FROZEN | Validation GT: AUTHORIZED | Test: LOCKED
PREFLIGHT CONTROLS — BEFORE VALIDATION GROUND TRUTH IS OPENED


,control,expected,actual,status
0,contract_status,FROZEN_FOR_DEVELOPMENT_BASELINE,FROZEN_FOR_DEVELOPMENT_BASELINE,VALID
1,target_fields,16,16,VALID
2,target_field_names,"[buyer.name, buyer.tax_identifier, currency, d...","[buyer.name, buyer.tax_identifier, currency, d...",VALID
3,render_index_records,200,200,VALID
4,baseline_pointer_status,ACTIVE_FOR_VALIDATION,ACTIVE_FOR_VALIDATION,VALID
5,baseline_id,RULE-BASED-INVOICE-PARSER-V1@1.0.2,RULE-BASED-INVOICE-PARSER-V1@1.0.2,VALID
6,baseline_pointer_parser_id,RULE-BASED-INVOICE-PARSER-V1,RULE-BASED-INVOICE-PARSER-V1,VALID
7,baseline_pointer_parser_version,1.0.2,1.0.2,VALID
8,next_allowed_split,validation,validation,VALID
9,test_locked_by_pointer,True,True,VALID



FROZEN PREDICTION CONTROLS


,control,expected,actual,status
0,prediction_records,40,40,VALID
1,unique_prediction_ids,40,40,VALID
2,prediction_templates,"[TPL-07, TPL-08]","[TPL-07, TPL-08]",VALID
3,prediction_sequence_numbers,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",VALID
4,prediction_source_errors,0,0,VALID



VALIDATION GROUND-TRUTH ACCESS PLAN


...

,metric,minimum,actual,threshold_source,status
0,all_scalar_exact_match,0.95,1.0,FROZEN_BEFORE_VALIDATION,PASSED
1,critical_scalar_exact_match,0.98,1.0,FROZEN_BEFORE_VALIDATION,PASSED
2,financial_consistency,0.99,1.0,FROZEN_BEFORE_VALIDATION,PASSED
3,line_item_field_f1,0.90,1.0,FROZEN_BEFORE_VALIDATION,PASSED


,field,group,critical,observations,exact_matches,exact_match_rate,character_errors,reference_characters,cer,word_errors,reference_words,wer,status
0,buyer.name,scalar,True,40,40,1.0,0,1185,0.0,0,160,0.0,PERFECT
1,buyer.tax_identifier,scalar,False,40,40,1.0,0,680,0.0,0,40,0.0,PERFECT
2,currency,scalar,True,40,40,1.0,0,120,0.0,0,40,0.0,PERFECT
3,due_date,scalar,True,40,40,1.0,0,400,0.0,0,40,0.0,PERFECT
4,financials.discount,scalar,True,40,40,1.0,0,147,0.0,0,40,0.0,PERFECT
5,financials.subtotal,scalar,True,40,40,1.0,0,331,0.0,0,40,0.0,PERFECT
6,financials.tax,scalar,True,40,40,1.0,0,244,0.0,0,40,0.0,PERFECT
7,financials.total,scalar,True,40,40,1.0,0,331,0.0,0,40,0.0,PERFECT
8,invoice_date,scalar,True,40,40,1.0,0,400,0.0,0,40,0.0,PERFECT
9,invoice_number,scalar,True,40,40,1.0,0,720,0.0,0,40,0.0,PERFECT


,template_id,documents,scalar_exact_match,line_item_field_f1,item_count_exact_match,financial_consistency,document_exact_match
0,TPL-07,20,1.0,1.0,1.0,1.0,1.0
1,TPL-08,20,1.0,1.0,1.0,1.0,1.0


,sequence_number,document_id,template_id,language,scalar_exact_match,predicted_item_count,reference_item_count,row_f1,item_field_f1,financial_consistent,document_exact_match,execution
0,1,INV-SYN-000121,TPL-07,id,1.0,7,7,1.0,1.0,True,True,NEW
1,2,INV-SYN-000122,TPL-07,id,1.0,7,7,1.0,1.0,True,True,NEW
2,3,INV-SYN-000123,TPL-07,id,1.0,3,3,1.0,1.0,True,True,NEW
3,4,INV-SYN-000124,TPL-07,id,1.0,4,4,1.0,1.0,True,True,NEW
4,5,INV-SYN-000125,TPL-07,id,1.0,4,4,1.0,1.0,True,True,NEW
5,6,INV-SYN-000126,TPL-07,id,1.0,2,2,1.0,1.0,True,True,NEW
6,7,INV-SYN-000127,TPL-07,id,1.0,3,3,1.0,1.0,True,True,NEW
7,8,INV-SYN-000128,TPL-07,id,1.0,4,4,1.0,1.0,True,True,NEW
8,9,INV-SYN-000129,TPL-07,id,1.0,6,6,1.0,1.0,True,True,NEW
9,10,INV-SYN-000130,TPL-07,id,1.0,5,5,1.0,1.0,True,True,NEW



Cell version             : 1.0.0
Evaluation status        : PASSED
Documents evaluated      : 40
Templates                : 2
All scalar exact match   : 1.000000
Critical scalar exact    : 1.000000
Scalar micro CER         : 0.000000
Scalar micro WER         : 0.000000
Row precision            : 1.000000
Row recall               : 1.000000
Row F1                   : 1.000000
Line-item field F1       : 1.000000
Item-count exact match   : 1.000000
Financial consistency    : 1.000000
Document exact match     : 1.000000
Mismatch records         : 0
New evaluations          : 40
Recovered evaluations    : 0
Document summary action  : CREATED
Field summary action     : CREATED
Template summary action  : CREATED
Mismatch file action     : CREATED
Manifest action          : CREATED
Evaluation root          : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/field_extraction/validation/rule_based_baseline_v1_0_2/evaluations/validation_eval_v1_

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from IPython.display import display


# ============================================================
# CELL 11E — FREEZE VALIDATION DECISION AND AUTHORIZE
#             TEST COHORT PREFLIGHT (TEST GT REMAINS LOCKED)
# ============================================================

CELL_VERSION = "1.0.0"

DATA_ROOT = Path("/content/drive/MyDrive/InvoiceFlow-AI-Data")
BUILD_ROOT = (
    DATA_ROOT
    / "interim"
    / "synthetic_v1_build"
    / "20260904T150025Z"
)
FIELD_ROOT = BUILD_ROOT / "ocr_benchmark" / "field_extraction"

VALIDATION_ROOT = (
    FIELD_ROOT / "validation" / "rule_based_baseline_v1_0_2"
)
BASELINE_POINTER_PATH = FIELD_ROOT / "validation_baseline_pointer.json"
PREDICTION_MANIFEST_PATH = (
    VALIDATION_ROOT / "validation_prediction_manifest.json"
)
EVALUATION_ROOT = (
    VALIDATION_ROOT / "evaluations" / "validation_eval_v1_0_2"
)
EVALUATION_MANIFEST_PATH = (
    EVALUATION_ROOT / "validation_evaluation_manifest.json"
)

DECISION_ROOT = (
    FIELD_ROOT
    / "frozen_validation_decisions"
    / "rule_based_baseline_v1_0_2"
)
DECISION_MANIFEST_PATH = (
    DECISION_ROOT / "validation_decision_manifest.json"
)
TEST_BASELINE_POINTER_PATH = FIELD_ROOT / "test_baseline_pointer.json"

EXPECTED_BASELINE_ID = "RULE-BASED-INVOICE-PARSER-V1@1.0.2"
EXPECTED_PARSER_ID = "RULE-BASED-INVOICE-PARSER-V1"
EXPECTED_PARSER_VERSION = "1.0.2"
EXPECTED_PARSER_SIGNATURE = (
    "ffe04f258fb3b1a6a55661698fa6c7dda29ddd7b714c2f210b3e6fc1fcd464b3"
)
EXPECTED_EVALUATOR_ID = "INVOICE-FIELD-EVALUATOR-V1"
EXPECTED_EVALUATOR_VERSION = "1.0.2"
EXPECTED_VALIDATION_DOCUMENTS = 40
EXPECTED_VALIDATION_TEMPLATES = {"TPL-07", "TPL-08"}
EXPECTED_TARGET_FIELDS = 16
EXPECTED_ACCEPTANCE_METRICS = {
    "critical_scalar_exact_match",
    "all_scalar_exact_match",
    "line_item_field_f1",
    "financial_consistency",
}

EXPECTED_TEST_DOCUMENTS = 40
EXPECTED_TEST_TEMPLATES = {"TPL-09", "TPL-10"}


# ============================================================
# FILE, HASH, AND IMMUTABLE-WRITE HELPERS
# ============================================================

def require_file(path: Path, label: str) -> None:
    if not path.is_file():
        raise FileNotFoundError(f"{label} tidak ditemukan: {path}")
    if path.stat().st_size <= 0:
        raise RuntimeError(f"{label} kosong: {path}")


def load_json(path: Path) -> dict:
    require_file(path, "JSON artifact")
    with path.open("r", encoding="utf-8") as file_handle:
        value = json.load(file_handle)
    if not isinstance(value, dict):
        raise TypeError(f"Root JSON bukan object: {path}")
    return value


def canonical_json(value: object) -> str:
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    )


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as file_handle:
        for chunk in iter(lambda: file_handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def sha256_object(value: object) -> str:
    return hashlib.sha256(
        canonical_json(value).encode("utf-8")
    ).hexdigest()


def atomic_write_json(path: Path, value: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(
        f".{path.name}.{os.getpid()}.tmp"
    )
    try:
        temporary_path.write_text(
            json.dumps(value, indent=2, ensure_ascii=False) + "\n",
            encoding="utf-8",
        )
        os.replace(temporary_path, path)
    finally:
        if temporary_path.exists():
            temporary_path.unlink()


def save_immutable_json(path: Path, value: dict) -> str:
    if path.exists():
        existing = load_json(path)
        if canonical_json(existing) != canonical_json(value):
            raise RuntimeError(
                f"Checkpoint sudah ada tetapi berbeda: {path}"
            )
        return "RECOVERED"

    atomic_write_json(path, value)
    return "CREATED"


def path_is_inside(path: Path, root: Path) -> bool:
    try:
        path.resolve().relative_to(root.resolve())
        return True
    except ValueError:
        return False


def paths_equal(first: object, second: Path) -> bool:
    try:
        return Path(str(first)).resolve() == second.resolve()
    except (OSError, RuntimeError, ValueError):
        return False


def close_enough(first: object, second: object) -> bool:
    try:
        return abs(float(first) - float(second)) <= 1e-12
    except (TypeError, ValueError):
        return False


def safe_ratio(numerator: int | float, denominator: int | float) -> float:
    if denominator == 0:
        return 1.0 if numerator == 0 else 0.0
    return float(numerator) / float(denominator)


def safe_error_rate(
    errors: int | float,
    reference_units: int | float,
) -> float:
    if reference_units == 0:
        return 0.0 if errors == 0 else 1.0
    return float(errors) / float(reference_units)


def harmonic_mean(precision: float, recall: float) -> float:
    if precision + recall == 0:
        return 0.0
    return 2.0 * precision * recall / (precision + recall)


# ============================================================
# PREFLIGHT — VALIDATION EVIDENCE ONLY; TEST DATA IS NOT OPENED
# ============================================================

for required_path, label in (
    (BASELINE_POINTER_PATH, "Validation baseline pointer"),
    (PREDICTION_MANIFEST_PATH, "Validation prediction manifest"),
    (EVALUATION_MANIFEST_PATH, "Validation evaluation manifest"),
):
    require_file(required_path, label)

baseline_pointer = load_json(BASELINE_POINTER_PATH)
prediction_manifest = load_json(PREDICTION_MANIFEST_PATH)
evaluation_manifest = load_json(EVALUATION_MANIFEST_PATH)

freeze_manifest_path = Path(
    str(baseline_pointer.get("freeze_manifest_path", ""))
)
parser_source_path = Path(
    str(baseline_pointer.get("parser_source_path", ""))
)
require_file(freeze_manifest_path, "Frozen development baseline manifest")
require_file(parser_source_path, "Frozen parser source")
freeze_manifest = load_json(freeze_manifest_path)

source_checksums_before = {
    "baseline_pointer": sha256_file(BASELINE_POINTER_PATH),
    "freeze_manifest": sha256_file(freeze_manifest_path),
    "parser_source": sha256_file(parser_source_path),
    "prediction_manifest": sha256_file(PREDICTION_MANIFEST_PATH),
    "evaluation_manifest": sha256_file(EVALUATION_MANIFEST_PATH),
}

freeze_parser = freeze_manifest.get("parser", {})
freeze_split_policy = freeze_manifest.get("split_policy", {})
prediction_parser = prediction_manifest.get("parser", {})
prediction_scope = prediction_manifest.get("scope", {})
prediction_integrity = prediction_manifest.get("integrity", {})
evaluation_baseline = evaluation_manifest.get("baseline", {})
evaluation_scope = evaluation_manifest.get("scope", {})
evaluation_integrity = evaluation_manifest.get("integrity", {})
evaluation_evaluator = evaluation_manifest.get("evaluator", {})
overall_metrics = evaluation_manifest.get("overall_metrics", {})
acceptance_checks = evaluation_manifest.get("acceptance_checks", [])
technical_controls = evaluation_manifest.get("technical_controls", [])
evaluation_records = evaluation_manifest.get("records", [])
prediction_records = prediction_manifest.get("records", [])

if not isinstance(acceptance_checks, list):
    raise TypeError("acceptance_checks pada evaluation manifest bukan list.")
if not isinstance(technical_controls, list):
    raise TypeError("technical_controls pada evaluation manifest bukan list.")
if not isinstance(evaluation_records, list):
    raise TypeError("records pada evaluation manifest bukan list.")
if not isinstance(prediction_records, list):
    raise TypeError("records pada prediction manifest bukan list.")

frozen_checks = (
    freeze_manifest.get("development_evidence", {})
    .get("acceptance_checks", [])
)
if not isinstance(frozen_checks, list):
    raise TypeError("Frozen acceptance checks bukan list.")

frozen_minima = {
    str(record.get("metric")): float(record.get("minimum"))
    for record in frozen_checks
    if isinstance(record, dict)
    and record.get("metric")
    and record.get("minimum") is not None
}
validation_check_map = {
    str(record.get("metric")): record
    for record in acceptance_checks
    if isinstance(record, dict) and record.get("metric")
}

acceptance_consistent = bool(
    set(frozen_minima) == EXPECTED_ACCEPTANCE_METRICS
    and set(validation_check_map) == EXPECTED_ACCEPTANCE_METRICS
)
if acceptance_consistent:
    for metric_name in sorted(EXPECTED_ACCEPTANCE_METRICS):
        check = validation_check_map[metric_name]
        minimum = frozen_minima[metric_name]
        actual = overall_metrics.get(metric_name)
        acceptance_consistent = bool(
            acceptance_consistent
            and close_enough(check.get("minimum"), minimum)
            and close_enough(check.get("actual"), actual)
            and float(actual) >= minimum
            and check.get("status") == "PASSED"
            and check.get("threshold_source")
            == "FROZEN_BEFORE_VALIDATION"
        )

technical_controls_passed = bool(
    technical_controls
    and all(
        isinstance(record, dict)
        and record.get("status") == "VALID"
        and record.get("expected") == record.get("actual")
        for record in technical_controls
    )
)

preflight_values = [
    (
        "baseline_pointer_status",
        "ACTIVE_FOR_VALIDATION",
        baseline_pointer.get("status"),
    ),
    (
        "baseline_id",
        EXPECTED_BASELINE_ID,
        baseline_pointer.get("baseline_id"),
    ),
    (
        "baseline_parser_id",
        EXPECTED_PARSER_ID,
        baseline_pointer.get("parser_id"),
    ),
    (
        "baseline_parser_version",
        EXPECTED_PARSER_VERSION,
        baseline_pointer.get("parser_version"),
    ),
    (
        "baseline_freeze_checksum",
        baseline_pointer.get("freeze_manifest_sha256"),
        source_checksums_before["freeze_manifest"],
    ),
    (
        "baseline_parser_source_checksum",
        baseline_pointer.get("parser_source_sha256"),
        source_checksums_before["parser_source"],
    ),
    (
        "baseline_next_allowed_split",
        "validation",
        baseline_pointer.get("next_allowed_split"),
    ),
    (
        "test_locked_by_baseline_pointer",
        True,
        baseline_pointer.get("test_remains_locked"),
    ),
    (
        "freeze_status",
        "FROZEN_FOR_VALIDATION",
        freeze_manifest.get("status"),
    ),
    (
        "freeze_baseline_id",
        EXPECTED_BASELINE_ID,
        freeze_manifest.get("baseline_id"),
    ),
    (
        "freeze_parser_id",
        EXPECTED_PARSER_ID,
        freeze_parser.get("parser_id"),
    ),
    (
        "freeze_parser_version",
        EXPECTED_PARSER_VERSION,
        freeze_parser.get("parser_version"),
    ),
    (
        "freeze_parser_signature",
        EXPECTED_PARSER_SIGNATURE,
        freeze_parser.get("parser_signature_sha256"),
    ),
    (
        "freeze_parser_source_path",
        True,
        paths_equal(freeze_parser.get("source_path"), parser_source_path),
    ),
    (
        "freeze_parser_source_checksum",
        source_checksums_before["parser_source"],
        freeze_parser.get("source_sha256"),
    ),
    (
        "freeze_next_allowed_split",
        "validation",
        freeze_split_policy.get("next_allowed_split"),
    ),
    (
        "test_locked_by_freeze",
        True,
        freeze_split_policy.get("test_remains_locked"),
    ),
    (
        "prediction_manifest_status",
        "EXECUTED",
        prediction_manifest.get("status"),
    ),
    (
        "prediction_quality_status",
        "PENDING_VALIDATION_GROUND_TRUTH_EVALUATION",
        prediction_manifest.get("quality_status"),
    ),
    (
        "prediction_parser_id",
        EXPECTED_PARSER_ID,
        prediction_parser.get("parser_id"),
    ),
    (
        "prediction_parser_version",
        EXPECTED_PARSER_VERSION,
        prediction_parser.get("parser_version"),
    ),
    (
        "prediction_parser_signature",
        EXPECTED_PARSER_SIGNATURE,
        prediction_parser.get("parser_signature_sha256"),
    ),
    (
        "prediction_documents",
        EXPECTED_VALIDATION_DOCUMENTS,
        prediction_scope.get("documents"),
    ),
    (
        "prediction_templates",
        sorted(EXPECTED_VALIDATION_TEMPLATES),
        sorted(prediction_scope.get("templates", [])),
    ),
    (
        "prediction_ground_truth_loaded",
        False,
        prediction_integrity.get("ground_truth_loaded"),
    ),
    (
        "prediction_validation_gt_opened",
        0,
        prediction_integrity.get("validation_ground_truth_opened"),
    ),
    (
        "prediction_test_opened",
        0,
        prediction_integrity.get("test_opened"),
    ),
    (
        "evaluation_status",
        "PASSED",
        evaluation_manifest.get("status"),
    ),
    (
        "evaluation_stage",
        "VALIDATION_GROUND_TRUTH_EVALUATION",
        evaluation_manifest.get("stage"),
    ),
    (
        "evaluation_baseline_id",
        EXPECTED_BASELINE_ID,
        evaluation_baseline.get("baseline_id"),
    ),
    (
        "evaluation_parser_id",
        EXPECTED_PARSER_ID,
        evaluation_baseline.get("parser_id"),
    ),
    (
        "evaluation_parser_version",
        EXPECTED_PARSER_VERSION,
        evaluation_baseline.get("parser_version"),
    ),
    (
        "evaluation_parser_signature",
        EXPECTED_PARSER_SIGNATURE,
        evaluation_baseline.get("parser_signature_sha256"),
    ),
    (
        "evaluation_freeze_checksum",
        source_checksums_before["freeze_manifest"],
        evaluation_baseline.get("freeze_manifest_sha256"),
    ),
    (
        "evaluator_id",
        EXPECTED_EVALUATOR_ID,
        evaluation_evaluator.get("evaluator_id"),
    ),
    (
        "evaluator_version",
        EXPECTED_EVALUATOR_VERSION,
        evaluation_evaluator.get("evaluator_version"),
    ),
    (
        "evaluation_split",
        "validation",
        evaluation_scope.get("split"),
    ),
    (
        "evaluated_documents",
        EXPECTED_VALIDATION_DOCUMENTS,
        evaluation_scope.get("documents"),
    ),
    (
        "evaluated_templates",
        sorted(EXPECTED_VALIDATION_TEMPLATES),
        sorted(evaluation_scope.get("templates", [])),
    ),
    (
        "validation_ground_truth_opened_in_11d",
        EXPECTED_VALIDATION_DOCUMENTS,
        evaluation_scope.get("validation_ground_truth_opened"),
    ),
    (
        "test_opened_in_11d",
        0,
        evaluation_scope.get("test_opened"),
    ),
    (
        "predictions_frozen_before_validation_gt",
        True,
        evaluation_integrity.get(
            "predictions_frozen_before_ground_truth_open"
        ),
    ),
    (
        "validation_gt_used_as_prediction_input",
        False,
        evaluation_integrity.get("ground_truth_used_as_prediction_input"),
    ),
    (
        "evaluation_parser_modifications",
        0,
        evaluation_integrity.get("parser_modifications"),
    ),
    (
        "evaluation_prediction_modifications",
        0,
        evaluation_integrity.get("prediction_modifications"),
    ),
    (
        "evaluation_ground_truth_modifications",
        0,
        evaluation_integrity.get("ground_truth_modifications"),
    ),
    (
        "evaluation_dataset_modifications",
        0,
        evaluation_integrity.get("dataset_modifications"),
    ),
    (
        "technical_controls_passed",
        True,
        technical_controls_passed,
    ),
    (
        "acceptance_thresholds_consistent",
        True,
        acceptance_consistent,
    ),
]

preflight_controls = [
    {
        "control": name,
        "expected": expected,
        "actual": actual,
        "status": "VALID" if expected == actual else "INVALID",
    }
    for name, expected, actual in preflight_values
]

print("=" * 96)
print(
    f"CELL 11E — VALIDATION DECISION FREEZE — VERSION {CELL_VERSION}"
)
print(f"Decision root: {DECISION_ROOT}")
print("Validation: PASSED | Test preflight: PENDING | Test GT: LOCKED")
print("=" * 96)
print("VALIDATION DECISION PREFLIGHT")
display(pd.DataFrame(preflight_controls))

invalid_preflight = [
    record["control"]
    for record in preflight_controls
    if record["status"] != "VALID"
]
if invalid_preflight:
    raise RuntimeError(
        "CELL 11E PREFLIGHT FAILED. "
        f"Kontrol tidak valid: {invalid_preflight}. "
        "Keputusan validation belum dibekukan dan test tetap terkunci."
    )


# ============================================================
# VERIFY IMMUTABLE VALIDATION OUTPUTS WITHOUT REOPENING GT
# ============================================================

prediction_record_map = {
    str(record.get("document_id")): record
    for record in prediction_records
    if isinstance(record, dict) and record.get("document_id")
}

artifact_errors = []
verified_evaluation_records = []
recomputed_scalar_records = []
recomputed_item_records = []
recomputed_document_metrics = []

for record_index, record in enumerate(evaluation_records, start=1):
    try:
        if not isinstance(record, dict):
            raise TypeError("Evaluation manifest record bukan object.")

        document_id = str(record.get("document_id", ""))
        template_id = str(record.get("template_id", ""))
        evaluation_path = Path(str(record.get("evaluation_path", "")))
        prediction_path = Path(str(record.get("prediction_path", "")))

        if not document_id:
            raise RuntimeError("document_id kosong.")
        if template_id not in EXPECTED_VALIDATION_TEMPLATES:
            raise RuntimeError(
                f"Template evaluation di luar validation: {template_id}"
            )
        if not path_is_inside(evaluation_path, EVALUATION_ROOT):
            raise RuntimeError(
                f"Evaluation path di luar evaluation root: {document_id}"
            )

        require_file(evaluation_path, f"Evaluation {document_id}")
        evaluation_sha256 = sha256_file(evaluation_path)
        if evaluation_sha256 != record.get("evaluation_sha256"):
            raise RuntimeError(
                f"Checksum evaluation tidak cocok: {document_id}"
            )

        prediction_record = prediction_record_map.get(document_id)
        if prediction_record is None:
            raise RuntimeError(
                f"Prediction manifest record tidak ditemukan: {document_id}"
            )
        if not paths_equal(
            prediction_record.get("prediction_path"),
            prediction_path,
        ):
            raise RuntimeError(
                f"Prediction path tidak konsisten: {document_id}"
            )

        require_file(prediction_path, f"Prediction {document_id}")
        prediction_sha256 = sha256_file(prediction_path)
        if prediction_sha256 != record.get("prediction_sha256"):
            raise RuntimeError(
                f"Checksum prediction/evaluation berbeda: {document_id}"
            )
        if prediction_sha256 != prediction_record.get("prediction_sha256"):
            raise RuntimeError(
                f"Checksum prediction manifest berbeda: {document_id}"
            )

        evaluation = load_json(evaluation_path)
        evaluator = evaluation.get("evaluator", {})
        document = evaluation.get("document", {})
        inputs = evaluation.get("inputs", {})
        integrity = evaluation.get("integrity", {})
        metrics = evaluation.get("metrics", {})
        scalar_comparisons = evaluation.get("scalar_comparisons", [])
        item_comparisons = evaluation.get("item_field_comparisons", [])

        if evaluation.get("status") != "EVALUATED":
            raise RuntimeError("Status document evaluation bukan EVALUATED.")
        if evaluator.get("evaluator_id") != EXPECTED_EVALUATOR_ID:
            raise RuntimeError("Evaluator ID document tidak cocok.")
        if evaluator.get("evaluator_version") != EXPECTED_EVALUATOR_VERSION:
            raise RuntimeError("Evaluator version document tidak cocok.")
        if document.get("document_id") != document_id:
            raise RuntimeError("Document ID evaluation tidak cocok.")
        if document.get("template_id") != template_id:
            raise RuntimeError("Template ID evaluation tidak cocok.")
        if document.get("split") != "validation":
            raise RuntimeError("Document evaluation bukan validation split.")
        if inputs.get("prediction_sha256") != prediction_sha256:
            raise RuntimeError("Input prediction checksum tidak cocok.")
        if inputs.get("ground_truth_sha256") != record.get(
            "ground_truth_sha256"
        ):
            raise RuntimeError("Metadata ground-truth checksum tidak cocok.")

        ground_truth_path = Path(
            str(inputs.get("ground_truth_path", ""))
        )
        if "validation" not in {
            part.casefold() for part in ground_truth_path.parts
        }:
            raise RuntimeError("Ground-truth reference bukan validation.")
        if "test" in {
            part.casefold() for part in ground_truth_path.parts
        }:
            raise RuntimeError("Ground-truth reference mengarah ke test.")
        if integrity.get("ground_truth_split") != "validation":
            raise RuntimeError("Integrity ground-truth split tidak valid.")
        if integrity.get("ground_truth_used_as_prediction_input") is not False:
            raise RuntimeError("Ground truth terindikasi sebagai input parser.")
        if integrity.get("test_opened") != 0:
            raise RuntimeError("Document evaluation menandai test terbuka.")
        if integrity.get("prediction_modified") is not False:
            raise RuntimeError("Prediction terindikasi dimodifikasi.")
        if integrity.get("ground_truth_modified") is not False:
            raise RuntimeError("Ground truth terindikasi dimodifikasi.")
        if not isinstance(scalar_comparisons, list):
            raise TypeError("scalar_comparisons bukan list.")
        if not isinstance(item_comparisons, list):
            raise TypeError("item_field_comparisons bukan list.")

        source_checksums_before[
            f"evaluation:{document_id}"
        ] = evaluation_sha256
        source_checksums_before[
            f"prediction:{document_id}"
        ] = prediction_sha256

        recomputed_scalar_records.extend(scalar_comparisons)
        recomputed_item_records.extend(item_comparisons)
        recomputed_document_metrics.append(metrics)
        verified_evaluation_records.append(record)

    except Exception as error:
        artifact_errors.append(
            {
                "record_index": record_index,
                "document_id": (
                    record.get("document_id")
                    if isinstance(record, dict)
                    else None
                ),
                "error_type": type(error).__name__,
                "error": str(error)[:900],
            }
        )


# ============================================================
# VERIFY SUMMARY ARTIFACTS AND RECOMPUTE OVERALL METRICS
# ============================================================

artifact_descriptors = evaluation_manifest.get("artifacts", {})
summary_specs = {
    "document_summary": EXPECTED_VALIDATION_DOCUMENTS,
    "field_summary": EXPECTED_TARGET_FIELDS,
    "template_summary": len(EXPECTED_VALIDATION_TEMPLATES),
    "mismatch_details": int(
        artifact_descriptors.get("mismatch_details", {}).get(
            "records",
            -1,
        )
    ),
}
summary_tables = {}

for artifact_name, expected_rows in summary_specs.items():
    try:
        descriptor = artifact_descriptors.get(artifact_name, {})
        if not isinstance(descriptor, dict):
            raise TypeError(f"Descriptor {artifact_name} bukan object.")
        artifact_path = Path(str(descriptor.get("path", "")))
        require_file(artifact_path, artifact_name)
        if not path_is_inside(artifact_path, EVALUATION_ROOT):
            raise RuntimeError(
                f"{artifact_name} berada di luar evaluation root."
            )
        artifact_sha256 = sha256_file(artifact_path)
        if artifact_sha256 != descriptor.get("sha256"):
            raise RuntimeError(
                f"Checksum {artifact_name} tidak cocok."
            )
        table = pd.read_csv(artifact_path, keep_default_na=False)
        if len(table) != expected_rows:
            raise RuntimeError(
                f"Jumlah baris {artifact_name} tidak cocok: "
                f"expected={expected_rows}, actual={len(table)}"
            )

        source_checksums_before[
            f"summary:{artifact_name}"
        ] = artifact_sha256
        summary_tables[artifact_name] = table

    except Exception as error:
        artifact_errors.append(
            {
                "record_index": None,
                "document_id": None,
                "error_type": type(error).__name__,
                "error": f"{artifact_name}: {str(error)[:820]}",
            }
        )


def sum_bool(records: list[dict], key: str) -> int:
    return sum(record.get(key) is True for record in records)


scalar_total = len(recomputed_scalar_records)
scalar_exact = sum_bool(recomputed_scalar_records, "exact_match")
critical_scalar_records = [
    record
    for record in recomputed_scalar_records
    if record.get("critical") is True
]
critical_scalar_exact = sum_bool(
    critical_scalar_records,
    "exact_match",
)
scalar_character_errors = sum(
    int(record.get("character_errors", 0))
    for record in recomputed_scalar_records
)
scalar_reference_characters = sum(
    int(record.get("reference_characters", 0))
    for record in recomputed_scalar_records
)
scalar_word_errors = sum(
    int(record.get("word_errors", 0))
    for record in recomputed_scalar_records
)
scalar_reference_words = sum(
    int(record.get("reference_words", 0))
    for record in recomputed_scalar_records
)

item_exact_fields = sum_bool(recomputed_item_records, "exact_match")
predicted_item_fields = sum(
    int(metrics.get("predicted_item_fields", 0))
    for metrics in recomputed_document_metrics
)
reference_item_fields = sum(
    int(metrics.get("reference_item_fields", 0))
    for metrics in recomputed_document_metrics
)
row_true_positives = sum(
    int(metrics.get("row_true_positives", 0))
    for metrics in recomputed_document_metrics
)
predicted_rows = sum(
    int(metrics.get("predicted_item_count", 0))
    for metrics in recomputed_document_metrics
)
reference_rows = sum(
    int(metrics.get("reference_item_count", 0))
    for metrics in recomputed_document_metrics
)

item_precision = safe_ratio(item_exact_fields, predicted_item_fields)
item_recall = safe_ratio(item_exact_fields, reference_item_fields)
row_precision = safe_ratio(row_true_positives, predicted_rows)
row_recall = safe_ratio(row_true_positives, reference_rows)

recomputed_metrics = {
    "documents": len(recomputed_document_metrics),
    "templates": len(
        {
            record.get("template_id")
            for record in verified_evaluation_records
        }
    ),
    "scalar_field_observations": scalar_total,
    "scalar_field_exact": scalar_exact,
    "all_scalar_exact_match": safe_ratio(scalar_exact, scalar_total),
    "critical_scalar_observations": len(critical_scalar_records),
    "critical_scalar_exact": critical_scalar_exact,
    "critical_scalar_exact_match": safe_ratio(
        critical_scalar_exact,
        len(critical_scalar_records),
    ),
    "scalar_micro_cer": safe_error_rate(
        scalar_character_errors,
        scalar_reference_characters,
    ),
    "scalar_micro_wer": safe_error_rate(
        scalar_word_errors,
        scalar_reference_words,
    ),
    "predicted_item_rows": predicted_rows,
    "reference_item_rows": reference_rows,
    "matched_item_rows": row_true_positives,
    "row_precision": row_precision,
    "row_recall": row_recall,
    "row_f1": harmonic_mean(row_precision, row_recall),
    "item_field_exact": item_exact_fields,
    "predicted_item_fields": predicted_item_fields,
    "reference_item_fields": reference_item_fields,
    "line_item_field_precision": item_precision,
    "line_item_field_recall": item_recall,
    "line_item_field_f1": harmonic_mean(item_precision, item_recall),
    "item_count_exact_match": safe_ratio(
        sum_bool(recomputed_document_metrics, "item_count_exact"),
        len(recomputed_document_metrics),
    ),
    "financial_consistency": safe_ratio(
        sum_bool(recomputed_document_metrics, "financial_consistent"),
        len(recomputed_document_metrics),
    ),
    "document_exact_match": safe_ratio(
        sum_bool(recomputed_document_metrics, "document_exact_match"),
        len(recomputed_document_metrics),
    ),
}

metrics_consistent = bool(
    set(recomputed_metrics) == set(overall_metrics)
    and all(
        close_enough(recomputed_metrics[name], overall_metrics[name])
        for name in recomputed_metrics
    )
)

recomputed_mismatches = sum(
    record.get("exact_match") is not True
    for record in (
        recomputed_scalar_records + recomputed_item_records
    )
)
mismatch_descriptor = artifact_descriptors.get("mismatch_details", {})
mismatch_records_reported = int(
    mismatch_descriptor.get("records", -1)
)

document_summary_ids = set()
if "document_summary" in summary_tables:
    document_summary_ids = set(
        summary_tables["document_summary"]["document_id"].astype(str)
    )

artifact_values = [
    (
        "evaluation_records",
        EXPECTED_VALIDATION_DOCUMENTS,
        len(verified_evaluation_records),
    ),
    (
        "unique_evaluation_document_ids",
        EXPECTED_VALIDATION_DOCUMENTS,
        len(
            {
                record.get("document_id")
                for record in verified_evaluation_records
            }
        ),
    ),
    (
        "evaluation_templates",
        sorted(EXPECTED_VALIDATION_TEMPLATES),
        sorted(
            {
                record.get("template_id")
                for record in verified_evaluation_records
            }
        ),
    ),
    (
        "prediction_records",
        EXPECTED_VALIDATION_DOCUMENTS,
        len(prediction_record_map),
    ),
    (
        "document_summary_ids",
        {
            record.get("document_id")
            for record in verified_evaluation_records
        },
        document_summary_ids,
    ),
    (
        "summary_artifacts",
        4,
        len(summary_tables),
    ),
    (
        "overall_metrics_recomputed",
        True,
        metrics_consistent,
    ),
    (
        "mismatch_records_consistent",
        recomputed_mismatches,
        mismatch_records_reported,
    ),
    ("artifact_errors", 0, len(artifact_errors)),
    ("validation_ground_truth_reopened", 0, 0),
    ("test_metadata_opened", 0, 0),
    ("test_documents_opened", 0, 0),
    ("test_ground_truth_opened", 0, 0),
]

artifact_controls = [
    {
        "control": name,
        "expected": expected,
        "actual": actual,
        "status": "VALID" if expected == actual else "INVALID",
    }
    for name, expected, actual in artifact_values
]

print("\nIMMUTABLE VALIDATION ARTIFACT CONTROLS")
display(pd.DataFrame(artifact_controls))
if artifact_errors:
    print("\nARTIFACT VERIFICATION ERRORS")
    display(pd.DataFrame(artifact_errors))

invalid_artifact_controls = [
    record["control"]
    for record in artifact_controls
    if record["status"] != "VALID"
]
if invalid_artifact_controls:
    raise RuntimeError(
        "CELL 11E ARTIFACT VERIFICATION FAILED. "
        f"Kontrol tidak valid: {invalid_artifact_controls}. "
        "Test tetap terkunci."
    )


# ============================================================
# CREATE OR RECOVER THE IMMUTABLE VALIDATION DECISION
# ============================================================

if DECISION_MANIFEST_PATH.exists():
    existing_decision = load_json(DECISION_MANIFEST_PATH)
    frozen_at_utc = existing_decision.get("frozen_at_utc")
    if not isinstance(frozen_at_utc, str) or not frozen_at_utc:
        raise RuntimeError(
            "Frozen validation decision tidak memiliki frozen_at_utc."
        )
else:
    frozen_at_utc = datetime.now(timezone.utc).isoformat(
        timespec="seconds"
    )

validation_result_fingerprint = {
    "evaluation_manifest_sha256": source_checksums_before[
        "evaluation_manifest"
    ],
    "overall_metrics": overall_metrics,
    "acceptance_checks": acceptance_checks,
    "document_evaluations": [
        {
            "document_id": record["document_id"],
            "evaluation_sha256": record["evaluation_sha256"],
            "prediction_sha256": record["prediction_sha256"],
            "ground_truth_sha256": record["ground_truth_sha256"],
        }
        for record in sorted(
            verified_evaluation_records,
            key=lambda row: str(row["document_id"]),
        )
    ],
}
validation_result_sha256 = sha256_object(
    validation_result_fingerprint
)

decision_manifest = {
    "schema_version": "1.0.0",
    "cell_version": CELL_VERSION,
    "status": "FROZEN_FOR_TEST_PREFLIGHT",
    "decision": "VALIDATION_PASSED_BASELINE_ACCEPTED",
    "frozen_at_utc": frozen_at_utc,
    "baseline": {
        "baseline_id": EXPECTED_BASELINE_ID,
        "parser_id": EXPECTED_PARSER_ID,
        "parser_version": EXPECTED_PARSER_VERSION,
        "parser_signature_sha256": EXPECTED_PARSER_SIGNATURE,
        "parser_source_path": str(parser_source_path),
        "parser_source_sha256": source_checksums_before["parser_source"],
        "development_freeze_manifest_path": str(freeze_manifest_path),
        "development_freeze_manifest_sha256": source_checksums_before[
            "freeze_manifest"
        ],
    },
    "validation_evidence": {
        "evaluation_manifest_path": str(EVALUATION_MANIFEST_PATH),
        "evaluation_manifest_sha256": source_checksums_before[
            "evaluation_manifest"
        ],
        "evaluation_status": evaluation_manifest["status"],
        "documents": EXPECTED_VALIDATION_DOCUMENTS,
        "templates": sorted(EXPECTED_VALIDATION_TEMPLATES),
        "overall_metrics": overall_metrics,
        "acceptance_checks": acceptance_checks,
        "mismatch_records": mismatch_records_reported,
        "validation_result_sha256": validation_result_sha256,
    },
    "verified_artifacts": {
        "document_evaluations": len(verified_evaluation_records),
        "prediction_files": len(prediction_record_map),
        "summary_files": len(summary_tables),
        "metrics_recomputed": True,
        "all_checksums_valid": True,
    },
    "immutable_inputs": {
        "validation_baseline_pointer": {
            "path": str(BASELINE_POINTER_PATH),
            "sha256": source_checksums_before["baseline_pointer"],
        },
        "validation_prediction_manifest": {
            "path": str(PREDICTION_MANIFEST_PATH),
            "sha256": source_checksums_before["prediction_manifest"],
        },
        "validation_evaluation_manifest": {
            "path": str(EVALUATION_MANIFEST_PATH),
            "sha256": source_checksums_before["evaluation_manifest"],
        },
    },
    "split_policy": {
        "development_tuning_complete": True,
        "validation_evaluation_complete": True,
        "validation_result_may_not_tune_parser_v1_0_2": True,
        "next_allowed_stage": "TEST_COHORT_PREFLIGHT",
        "next_allowed_split": "test",
        "test_preflight_allowed": True,
        "test_ground_truth_remains_locked": True,
        "test_predictions_must_be_frozen_before_ground_truth_open": True,
        "parser_changes_require_new_version_and_new_protocol": True,
    },
    "integrity": {
        "validation_ground_truth_reopened_in_this_cell": 0,
        "test_metadata_opened": 0,
        "test_documents_opened": 0,
        "test_ground_truth_opened": 0,
        "parser_modifications": 0,
        "prediction_modifications": 0,
        "evaluation_modifications": 0,
        "dataset_modifications": 0,
        "source_modifications": 0,
    },
    "next_stage": {
        "cell": "CELL 12A",
        "action": "TEST_COHORT_PREFLIGHT_GROUND_TRUTH_CLOSED",
        "expected_documents": EXPECTED_TEST_DOCUMENTS,
        "allowed_templates": sorted(EXPECTED_TEST_TEMPLATES),
        "parser_mutation_allowed": False,
        "test_ground_truth_must_remain_closed": True,
    },
}

decision_action = save_immutable_json(
    DECISION_MANIFEST_PATH,
    decision_manifest,
)
decision_manifest_sha256 = sha256_file(DECISION_MANIFEST_PATH)

recovered_decision = load_json(DECISION_MANIFEST_PATH)
if canonical_json(recovered_decision) != canonical_json(decision_manifest):
    raise RuntimeError(
        "Validation decision berbeda setelah penulisan."
    )


# ============================================================
# CREATE OR RECOVER THE TEST-PREFLIGHT POINTER
# ============================================================

test_pointer = {
    "schema_version": "1.0.0",
    "status": "ACTIVE_FOR_TEST_PREFLIGHT",
    "baseline_id": EXPECTED_BASELINE_ID,
    "parser_id": EXPECTED_PARSER_ID,
    "parser_version": EXPECTED_PARSER_VERSION,
    "parser_signature_sha256": EXPECTED_PARSER_SIGNATURE,
    "parser_source_path": str(parser_source_path),
    "parser_source_sha256": source_checksums_before["parser_source"],
    "development_freeze_manifest_path": str(freeze_manifest_path),
    "development_freeze_manifest_sha256": source_checksums_before[
        "freeze_manifest"
    ],
    "validation_decision_manifest_path": str(DECISION_MANIFEST_PATH),
    "validation_decision_manifest_sha256": decision_manifest_sha256,
    "validation_result_sha256": validation_result_sha256,
    "next_allowed_stage": "TEST_COHORT_PREFLIGHT",
    "next_allowed_split": "test",
    "allowed_test_templates": sorted(EXPECTED_TEST_TEMPLATES),
    "expected_test_documents": EXPECTED_TEST_DOCUMENTS,
    "parser_mutation_allowed": False,
    "test_ground_truth_remains_locked": True,
    "test_predictions_must_be_frozen_before_ground_truth_open": True,
}

pointer_action = save_immutable_json(
    TEST_BASELINE_POINTER_PATH,
    test_pointer,
)
recovered_pointer = load_json(TEST_BASELINE_POINTER_PATH)
if canonical_json(recovered_pointer) != canonical_json(test_pointer):
    raise RuntimeError("Test baseline pointer berbeda setelah penulisan.")


# ============================================================
# FINAL SOURCE IMMUTABILITY AND POINTER CONTROLS
# ============================================================

source_checksums_after = {
    "baseline_pointer": sha256_file(BASELINE_POINTER_PATH),
    "freeze_manifest": sha256_file(freeze_manifest_path),
    "parser_source": sha256_file(parser_source_path),
    "prediction_manifest": sha256_file(PREDICTION_MANIFEST_PATH),
    "evaluation_manifest": sha256_file(EVALUATION_MANIFEST_PATH),
}

for record in verified_evaluation_records:
    document_id = str(record["document_id"])
    source_checksums_after[
        f"evaluation:{document_id}"
    ] = sha256_file(Path(record["evaluation_path"]))
    source_checksums_after[
        f"prediction:{document_id}"
    ] = sha256_file(Path(record["prediction_path"]))

for artifact_name in summary_tables:
    descriptor = artifact_descriptors[artifact_name]
    source_checksums_after[
        f"summary:{artifact_name}"
    ] = sha256_file(Path(descriptor["path"]))

changed_sources = [
    name
    for name, checksum in source_checksums_before.items()
    if source_checksums_after.get(name) != checksum
]
if changed_sources:
    raise RuntimeError(
        f"Source berubah selama Cell 11E: {changed_sources}"
    )

pointer_controls = [
    {
        "control": "decision_status",
        "expected": "FROZEN_FOR_TEST_PREFLIGHT",
        "actual": recovered_decision.get("status"),
    },
    {
        "control": "validation_decision",
        "expected": "VALIDATION_PASSED_BASELINE_ACCEPTED",
        "actual": recovered_decision.get("decision"),
    },
    {
        "control": "decision_checksum",
        "expected": decision_manifest_sha256,
        "actual": sha256_file(DECISION_MANIFEST_PATH),
    },
    {
        "control": "test_pointer_status",
        "expected": "ACTIVE_FOR_TEST_PREFLIGHT",
        "actual": recovered_pointer.get("status"),
    },
    {
        "control": "pointer_decision_checksum",
        "expected": decision_manifest_sha256,
        "actual": recovered_pointer.get(
            "validation_decision_manifest_sha256"
        ),
    },
    {
        "control": "next_allowed_stage",
        "expected": "TEST_COHORT_PREFLIGHT",
        "actual": recovered_pointer.get("next_allowed_stage"),
    },
    {
        "control": "next_allowed_split",
        "expected": "test",
        "actual": recovered_pointer.get("next_allowed_split"),
    },
    {
        "control": "test_ground_truth_locked",
        "expected": True,
        "actual": recovered_pointer.get(
            "test_ground_truth_remains_locked"
        ),
    },
    {
        "control": "source_modifications",
        "expected": 0,
        "actual": len(changed_sources),
    },
    {
        "control": "validation_ground_truth_reopened",
        "expected": 0,
        "actual": 0,
    },
    {
        "control": "test_ground_truth_opened",
        "expected": 0,
        "actual": 0,
    },
]

for record in pointer_controls:
    record["status"] = (
        "VALID"
        if record["expected"] == record["actual"]
        else "INVALID"
    )

print("\nFINAL FREEZE AND POINTER CONTROLS")
display(pd.DataFrame(pointer_controls))

invalid_pointer_controls = [
    record["control"]
    for record in pointer_controls
    if record["status"] != "VALID"
]
if invalid_pointer_controls:
    raise RuntimeError(
        "CELL 11E FINAL GATE FAILED. "
        f"Kontrol tidak valid: {invalid_pointer_controls}."
    )


print()
print(f"Cell version             : {CELL_VERSION}")
print(f"Baseline ID              : {EXPECTED_BASELINE_ID}")
print(f"Parser version           : {EXPECTED_PARSER_VERSION}")
print(f"Validation status        : {evaluation_manifest['status']}")
print(f"Validation documents     : {len(verified_evaluation_records)}")
print(
    "Validation templates     : "
    f"{sorted(EXPECTED_VALIDATION_TEMPLATES)}"
)
print(
    "Document exact match    : "
    f"{float(overall_metrics['document_exact_match']):.6f}"
)
print(f"Validation mismatches    : {mismatch_records_reported}")
print(f"Validation result SHA-256: {validation_result_sha256}")
print(f"Decision action          : {decision_action}")
print(f"Decision manifest        : {DECISION_MANIFEST_PATH}")
print(f"Decision SHA-256         : {decision_manifest_sha256}")
print(f"Pointer action           : {pointer_action}")
print(f"Test baseline pointer    : {TEST_BASELINE_POINTER_PATH}")
print(
    "Pointer SHA-256          : "
    f"{sha256_file(TEST_BASELINE_POINTER_PATH)}"
)
print("Validation GT reopened  : 0")
print("Test metadata opened     : 0")
print("Test documents opened    : 0")
print("Test GT opened           : 0")
print("Parser modifications     : 0")
print("Prediction modifications : 0")
print("Evaluation modifications : 0")
print("Dataset modifications    : 0")
print("Source modifications     : 0")
print()
print(
    "✅ CELL 11E PASSED — hasil validation telah dibekukan secara "
    "immutable dan baseline v1.0.2 diizinkan memasuki test cohort "
    "preflight. Ground truth test tetap terkunci; lanjutkan hanya ke "
    "Cell 12A."
)


CELL 11E — VALIDATION DECISION FREEZE — VERSION 1.0.0
Decision root: /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/field_extraction/frozen_validation_decisions/rule_based_baseline_v1_0_2
Validation: PASSED | Test preflight: PENDING | Test GT: LOCKED
VALIDATION DECISION PREFLIGHT


,control,expected,actual,status
0,baseline_pointer_status,ACTIVE_FOR_VALIDATION,ACTIVE_FOR_VALIDATION,VALID
1,baseline_id,RULE-BASED-INVOICE-PARSER-V1@1.0.2,RULE-BASED-INVOICE-PARSER-V1@1.0.2,VALID
2,baseline_parser_id,RULE-BASED-INVOICE-PARSER-V1,RULE-BASED-INVOICE-PARSER-V1,VALID
3,baseline_parser_version,1.0.2,1.0.2,VALID
4,baseline_freeze_checksum,9f234c5c8cdc10c277e9c26a1f0831d1e08f3dc6fb9503...,9f234c5c8cdc10c277e9c26a1f0831d1e08f3dc6fb9503...,VALID
5,baseline_parser_source_checksum,0a737f57a86df7d3e16eef6749e3c3a23e82fb51227d09...,0a737f57a86df7d3e16eef6749e3c3a23e82fb51227d09...,VALID
6,baseline_next_allowed_split,validation,validation,VALID
7,test_locked_by_baseline_pointer,True,True,VALID
8,freeze_status,FROZEN_FOR_VALIDATION,FROZEN_FOR_VALIDATION,VALID
9,freeze_baseline_id,RULE-BASED-INVOICE-PARSER-V1@1.0.2,RULE-BASED-INVOICE-PARSER-V1@1.0.2,VALID



IMMUTABLE VALIDATION ARTIFACT CONTROLS


,control,expected,actual,status
0,evaluation_records,40,40,VALID
1,unique_evaluation_document_ids,40,40,VALID
2,evaluation_templates,"[TPL-07, TPL-08]","[TPL-07, TPL-08]",VALID
3,prediction_records,40,40,VALID
4,document_summary_ids,"{INV-SYN-000160, INV-SYN-000146, INV-SYN-00014...","{INV-SYN-000160, INV-SYN-000146, INV-SYN-00014...",VALID
5,summary_artifacts,4,4,VALID
6,overall_metrics_recomputed,True,True,VALID
7,mismatch_records_consistent,0,0,VALID
8,artifact_errors,0,0,VALID
9,validation_ground_truth_reopened,0,0,VALID



FINAL FREEZE AND POINTER CONTROLS


,control,expected,actual,status
0,decision_status,FROZEN_FOR_TEST_PREFLIGHT,FROZEN_FOR_TEST_PREFLIGHT,VALID
1,validation_decision,VALIDATION_PASSED_BASELINE_ACCEPTED,VALIDATION_PASSED_BASELINE_ACCEPTED,VALID
2,decision_checksum,831fb82a372aaed44789afbc66386bf6ebbf073868a9c0...,831fb82a372aaed44789afbc66386bf6ebbf073868a9c0...,VALID
3,test_pointer_status,ACTIVE_FOR_TEST_PREFLIGHT,ACTIVE_FOR_TEST_PREFLIGHT,VALID
4,pointer_decision_checksum,831fb82a372aaed44789afbc66386bf6ebbf073868a9c0...,831fb82a372aaed44789afbc66386bf6ebbf073868a9c0...,VALID
5,next_allowed_stage,TEST_COHORT_PREFLIGHT,TEST_COHORT_PREFLIGHT,VALID
6,next_allowed_split,test,test,VALID
7,test_ground_truth_locked,True,True,VALID
8,source_modifications,0,0,VALID
9,validation_ground_truth_reopened,0,0,VALID



Cell version             : 1.0.0
Baseline ID              : RULE-BASED-INVOICE-PARSER-V1@1.0.2
Parser version           : 1.0.2
Validation status        : PASSED
Validation documents     : 40
Validation templates     : ['TPL-07', 'TPL-08']
Document exact match    : 1.000000
Validation mismatches    : 0
Validation result SHA-256: 31718428fcbf9d9277d3e65f1a23a9eb0fa456c4070617164434554ab556e5e6
Decision action          : CREATED
Decision manifest        : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/field_extraction/frozen_validation_decisions/rule_based_baseline_v1_0_2/validation_decision_manifest.json
Decision SHA-256         : 831fb82a372aaed44789afbc66386bf6ebbf073868a9c064f5ff6493599240cd
Pointer action           : CREATED
Test baseline pointer    : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/field_extraction/test_baseline_pointer.json
Pointer SHA-256          : d8a72f86

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import re
from collections import Counter
from pathlib import Path

import pandas as pd
from IPython.display import display


# ============================================================
# CELL 12A — TEST COHORT PREFLIGHT
#             (TEST GROUND TRUTH CLOSED)
# ============================================================

CELL_VERSION = "1.0.0"

DATA_ROOT = Path("/content/drive/MyDrive/InvoiceFlow-AI-Data")
BUILD_ROOT = (
    DATA_ROOT
    / "interim"
    / "synthetic_v1_build"
    / "20260904T150025Z"
)
RELEASE_ROOT = (
    DATA_ROOT
    / "releases"
    / "SYNTHETIC-INVOICE-V1"
    / "1.0.0"
)
FIELD_ROOT = BUILD_ROOT / "ocr_benchmark" / "field_extraction"
RENDERED_DATASET_ROOT = BUILD_ROOT / "rendered_dataset"

CURRENT_RELEASE_POINTER_PATH = (
    DATA_ROOT / "releases" / "current_release.json"
)
RELEASE_INDEX_PATH = RELEASE_ROOT / "release_index.jsonl"
RELEASE_MANIFEST_PATH = RELEASE_ROOT / "release_manifest.json"
RENDER_INDEX_PATH = (
    BUILD_ROOT / "manifests" / "batch_render_index.jsonl"
)

TEST_BASELINE_POINTER_PATH = FIELD_ROOT / "test_baseline_pointer.json"
TEST_ROOT = FIELD_ROOT / "test" / "rule_based_baseline_v1_0_2"
TEST_SELECTION_PATH = TEST_ROOT / "test_cohort_manifest.json"

EXPECTED_BASELINE_ID = "RULE-BASED-INVOICE-PARSER-V1@1.0.2"
EXPECTED_PARSER_ID = "RULE-BASED-INVOICE-PARSER-V1"
EXPECTED_PARSER_VERSION = "1.0.2"
EXPECTED_PARSER_SIGNATURE = (
    "ffe04f258fb3b1a6a55661698fa6c7dda29ddd7b714c2f210b3e6fc1fcd464b3"
)
EXPECTED_RELEASE_RECORDS = 200
EXPECTED_TEST_DOCUMENTS = 40
EXPECTED_TEST_TEMPLATES = {"TPL-09", "TPL-10"}
EXPECTED_DOCUMENTS_PER_TEMPLATE = 20
EXPECTED_LANGUAGES = {"en", "id"}
SUPPORTED_CURRENCIES = {"EUR", "GBP", "IDR", "USD"}

SHA256_PATTERN = re.compile(r"^[0-9a-f]{64}$", re.IGNORECASE)


# ============================================================
# FILE, HASH, AND IMMUTABLE-WRITE HELPERS
# ============================================================

def require_file(path: Path, label: str) -> None:
    if not path.is_file():
        raise FileNotFoundError(f"{label} tidak ditemukan: {path}")
    if path.stat().st_size <= 0:
        raise RuntimeError(f"{label} kosong: {path}")


def load_json(path: Path) -> dict:
    require_file(path, "JSON artifact")
    with path.open("r", encoding="utf-8") as file_handle:
        value = json.load(file_handle)
    if not isinstance(value, dict):
        raise TypeError(f"Root JSON bukan object: {path}")
    return value


def load_jsonl(path: Path) -> list[dict]:
    require_file(path, "JSONL artifact")
    records = []
    with path.open("r", encoding="utf-8") as file_handle:
        for line_number, line in enumerate(file_handle, start=1):
            if not line.strip():
                continue
            value = json.loads(line)
            if not isinstance(value, dict):
                raise TypeError(
                    f"Record JSONL baris {line_number} bukan object: {path}"
                )
            records.append(value)
    return records


def canonical_json(value: object) -> str:
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as file_handle:
        for chunk in iter(lambda: file_handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def sha256_object(value: object) -> str:
    return hashlib.sha256(
        canonical_json(value).encode("utf-8")
    ).hexdigest()


def atomic_write_json(path: Path, value: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(
        f".{path.name}.{os.getpid()}.tmp"
    )
    try:
        temporary_path.write_text(
            json.dumps(value, indent=2, ensure_ascii=False) + "\n",
            encoding="utf-8",
        )
        os.replace(temporary_path, path)
    finally:
        if temporary_path.exists():
            temporary_path.unlink()


def save_immutable_json(path: Path, value: dict) -> str:
    if path.exists():
        existing = load_json(path)
        if canonical_json(existing) != canonical_json(value):
            raise RuntimeError(
                f"Checkpoint sudah ada tetapi berbeda: {path}"
            )
        return "RECOVERED"

    atomic_write_json(path, value)
    return "CREATED"


def path_is_inside(path: Path, root: Path) -> bool:
    try:
        path.resolve().relative_to(root.resolve())
        return True
    except ValueError:
        return False


def paths_equal(first: object, second: Path) -> bool:
    try:
        return Path(str(first)).resolve() == second.resolve()
    except (OSError, RuntimeError, ValueError):
        return False


# ============================================================
# GENERIC MANIFEST HELPERS
# ============================================================

def walk_values(value: object):
    if isinstance(value, dict):
        for nested in value.values():
            yield from walk_values(nested)
    elif isinstance(value, list):
        for nested in value:
            yield from walk_values(nested)
    else:
        yield value


def deep_key_values(value: object, keys: set[str]) -> list[object]:
    matches = []
    if isinstance(value, dict):
        for key, nested in value.items():
            if str(key).casefold() in keys:
                matches.append(nested)
            matches.extend(deep_key_values(nested, keys))
    elif isinstance(value, list):
        for nested in value:
            matches.extend(deep_key_values(nested, keys))
    return matches


def first_scalar(
    record: dict,
    aliases: tuple[str, ...],
    label: str,
) -> object:
    alias_set = {alias.casefold() for alias in aliases}
    matches = [
        value
        for value in deep_key_values(record, alias_set)
        if isinstance(value, (str, int, float, bool))
    ]
    unique = []
    for value in matches:
        if value not in unique:
            unique.append(value)
    if len(unique) != 1:
        raise RuntimeError(
            f"{label} tidak ditemukan secara unik. Kandidat={unique}"
        )
    return unique[0]


def normalize_split(value: object) -> str:
    normalized = str(value or "").strip().casefold()
    aliases = {
        "dev": "development",
        "train": "development",
        "development": "development",
        "val": "validation",
        "valid": "validation",
        "validation": "validation",
        "test": "test",
    }
    return aliases.get(normalized, normalized)


def resolve_release_status(manifest: dict) -> str | None:
    for key in ("release_status", "status"):
        value = manifest.get(key)
        if isinstance(value, str) and value.strip():
            return value.strip().upper()

    release_section = manifest.get("release")
    if isinstance(release_section, dict):
        for key in ("release_status", "status"):
            value = release_section.get(key)
            if isinstance(value, str) and value.strip():
                return value.strip().upper()

    explicit_nested_values = [
        value
        for value in deep_key_values(manifest, {"release_status"})
        if isinstance(value, str) and value.strip()
    ]
    normalized_values = sorted(
        {value.strip().upper() for value in explicit_nested_values}
    )
    if len(normalized_values) == 1:
        return normalized_values[0]
    return None


def unique_record_by_document(
    records: list[dict],
    document_id: str,
    label: str,
) -> dict:
    matches = []
    for record in records:
        try:
            record_document_id = str(
                first_scalar(
                    record,
                    ("document_id",),
                    f"{label} document_id",
                )
            )
        except RuntimeError:
            continue
        if record_document_id == document_id:
            matches.append(record)

    if len(matches) != 1:
        raise RuntimeError(
            f"{label} {document_id} ditemukan {len(matches)} kali; "
            "seharusnya tepat satu."
        )
    return matches[0]


def locate_test_pdf_and_checksum(
    render_record: dict,
    document_id: str,
    template_id: str,
) -> tuple[Path, str]:
    artifacts = render_record.get("artifacts")
    if not isinstance(artifacts, dict):
        raise RuntimeError(
            f"artifacts tidak valid pada render record {document_id}."
        )

    pdf_artifact = artifacts.get("pdf")
    if not isinstance(pdf_artifact, dict):
        raise RuntimeError(
            f"artifacts.pdf tidak valid pada render record {document_id}."
        )

    relative_path_value = pdf_artifact.get("relative_path")
    expected_size = pdf_artifact.get("size_bytes")
    expected_checksum = pdf_artifact.get("sha256")

    if not isinstance(relative_path_value, str) or not relative_path_value:
        raise RuntimeError(f"PDF relative_path tidak valid: {document_id}")
    if not isinstance(expected_size, int) or expected_size <= 0:
        raise RuntimeError(f"PDF size_bytes tidak valid: {document_id}")
    if (
        not isinstance(expected_checksum, str)
        or not SHA256_PATTERN.fullmatch(expected_checksum)
    ):
        raise RuntimeError(f"PDF sha256 tidak valid: {document_id}")

    relative_path = Path(relative_path_value)
    if relative_path.is_absolute() or ".." in relative_path.parts:
        raise RuntimeError(
            f"PDF relative_path tidak aman: {relative_path_value}"
        )

    expected_relative_path = Path(
        "pdf",
        "test",
        template_id,
        f"{document_id}.pdf",
    )
    if relative_path != expected_relative_path:
        raise RuntimeError(
            f"Struktur path PDF tidak cocok untuk {document_id}. "
            f"Expected={expected_relative_path}, actual={relative_path}"
        )

    pdf_path = (RENDERED_DATASET_ROOT / relative_path).resolve()
    if not path_is_inside(pdf_path, RENDERED_DATASET_ROOT):
        raise RuntimeError(
            f"PDF keluar dari rendered_dataset root: {pdf_path}"
        )
    require_file(pdf_path, f"Test PDF {document_id}")
    if pdf_path.stat().st_size != expected_size:
        raise RuntimeError(
            f"Ukuran PDF {document_id} tidak cocok. "
            f"Expected={expected_size}, actual={pdf_path.stat().st_size}"
        )

    actual_checksum = sha256_file(pdf_path)
    if actual_checksum.casefold() != expected_checksum.casefold():
        raise RuntimeError(
            f"Checksum PDF {document_id} tidak cocok. "
            f"Expected={expected_checksum}, actual={actual_checksum}"
        )
    return pdf_path, actual_checksum


# ============================================================
# SOURCE PREFLIGHT — TEST GT IS STILL CLOSED
# ============================================================

for required_path, label in (
    (CURRENT_RELEASE_POINTER_PATH, "Current release pointer"),
    (RELEASE_INDEX_PATH, "Release index"),
    (RELEASE_MANIFEST_PATH, "Release manifest"),
    (RENDER_INDEX_PATH, "Batch render index"),
    (TEST_BASELINE_POINTER_PATH, "Test baseline pointer"),
):
    require_file(required_path, label)

current_release_pointer = load_json(CURRENT_RELEASE_POINTER_PATH)
release_manifest = load_json(RELEASE_MANIFEST_PATH)
release_records = load_jsonl(RELEASE_INDEX_PATH)
render_records = load_jsonl(RENDER_INDEX_PATH)
test_pointer = load_json(TEST_BASELINE_POINTER_PATH)
release_status = resolve_release_status(release_manifest)

validation_decision_path = Path(
    str(test_pointer.get("validation_decision_manifest_path", ""))
)
freeze_manifest_path = Path(
    str(test_pointer.get("development_freeze_manifest_path", ""))
)
parser_source_path = Path(
    str(test_pointer.get("parser_source_path", ""))
)
require_file(validation_decision_path, "Frozen validation decision")
require_file(freeze_manifest_path, "Frozen development baseline manifest")
require_file(parser_source_path, "Frozen parser source")

validation_decision = load_json(validation_decision_path)
freeze_manifest = load_json(freeze_manifest_path)

source_checksums_before = {
    "current_release_pointer": sha256_file(CURRENT_RELEASE_POINTER_PATH),
    "release_index": sha256_file(RELEASE_INDEX_PATH),
    "release_manifest": sha256_file(RELEASE_MANIFEST_PATH),
    "render_index": sha256_file(RENDER_INDEX_PATH),
    "test_baseline_pointer": sha256_file(TEST_BASELINE_POINTER_PATH),
    "validation_decision": sha256_file(validation_decision_path),
    "freeze_manifest": sha256_file(freeze_manifest_path),
    "parser_source": sha256_file(parser_source_path),
}

release_manifest_sha256 = source_checksums_before["release_manifest"]
pointer_strings = {
    str(value)
    for value in walk_values(current_release_pointer)
    if isinstance(value, str)
}
decision_integrity = validation_decision.get("integrity", {})
decision_split_policy = validation_decision.get("split_policy", {})
decision_validation = validation_decision.get("validation_evidence", {})
decision_baseline = validation_decision.get("baseline", {})
freeze_parser = freeze_manifest.get("parser", {})

source_values = [
    ("release_manifest_status", "FROZEN", release_status),
    (
        "release_pointer_matches_manifest",
        True,
        release_manifest_sha256 in pointer_strings,
    ),
    ("release_records", EXPECTED_RELEASE_RECORDS, len(release_records)),
    ("render_index_records", EXPECTED_RELEASE_RECORDS, len(render_records)),
    (
        "test_pointer_status",
        "ACTIVE_FOR_TEST_PREFLIGHT",
        test_pointer.get("status"),
    ),
    (
        "baseline_id",
        EXPECTED_BASELINE_ID,
        test_pointer.get("baseline_id"),
    ),
    (
        "parser_id",
        EXPECTED_PARSER_ID,
        test_pointer.get("parser_id"),
    ),
    (
        "parser_version",
        EXPECTED_PARSER_VERSION,
        test_pointer.get("parser_version"),
    ),
    (
        "parser_signature",
        EXPECTED_PARSER_SIGNATURE,
        test_pointer.get("parser_signature_sha256"),
    ),
    (
        "parser_source_path",
        True,
        paths_equal(test_pointer.get("parser_source_path"), parser_source_path),
    ),
    (
        "parser_source_checksum",
        test_pointer.get("parser_source_sha256"),
        source_checksums_before["parser_source"],
    ),
    (
        "freeze_manifest_checksum",
        test_pointer.get("development_freeze_manifest_sha256"),
        source_checksums_before["freeze_manifest"],
    ),
    (
        "validation_decision_checksum",
        test_pointer.get("validation_decision_manifest_sha256"),
        source_checksums_before["validation_decision"],
    ),
    (
        "validation_decision_status",
        "FROZEN_FOR_TEST_PREFLIGHT",
        validation_decision.get("status"),
    ),
    (
        "validation_decision",
        "VALIDATION_PASSED_BASELINE_ACCEPTED",
        validation_decision.get("decision"),
    ),
    (
        "validation_evaluation_status",
        "PASSED",
        decision_validation.get("evaluation_status"),
    ),
    (
        "decision_baseline_id",
        EXPECTED_BASELINE_ID,
        decision_baseline.get("baseline_id"),
    ),
    (
        "decision_parser_id",
        EXPECTED_PARSER_ID,
        decision_baseline.get("parser_id"),
    ),
    (
        "decision_parser_version",
        EXPECTED_PARSER_VERSION,
        decision_baseline.get("parser_version"),
    ),
    (
        "decision_parser_signature",
        EXPECTED_PARSER_SIGNATURE,
        decision_baseline.get("parser_signature_sha256"),
    ),
    (
        "decision_parser_source_checksum",
        source_checksums_before["parser_source"],
        decision_baseline.get("parser_source_sha256"),
    ),
    (
        "decision_freeze_checksum",
        source_checksums_before["freeze_manifest"],
        decision_baseline.get("development_freeze_manifest_sha256"),
    ),
    (
        "validation_result_fingerprint",
        test_pointer.get("validation_result_sha256"),
        decision_validation.get("validation_result_sha256"),
    ),
    (
        "validation_documents",
        40,
        decision_validation.get("documents"),
    ),
    (
        "validation_templates",
        ["TPL-07", "TPL-08"],
        decision_validation.get("templates"),
    ),
    (
        "decision_next_allowed_stage",
        "TEST_COHORT_PREFLIGHT",
        decision_split_policy.get("next_allowed_stage"),
    ),
    (
        "decision_next_allowed_split",
        "test",
        decision_split_policy.get("next_allowed_split"),
    ),
    (
        "test_preflight_allowed",
        True,
        decision_split_policy.get("test_preflight_allowed"),
    ),
    (
        "pointer_next_allowed_stage",
        "TEST_COHORT_PREFLIGHT",
        test_pointer.get("next_allowed_stage"),
    ),
    (
        "pointer_next_allowed_split",
        "test",
        test_pointer.get("next_allowed_split"),
    ),
    (
        "allowed_test_templates",
        sorted(EXPECTED_TEST_TEMPLATES),
        test_pointer.get("allowed_test_templates"),
    ),
    (
        "expected_test_documents",
        EXPECTED_TEST_DOCUMENTS,
        test_pointer.get("expected_test_documents"),
    ),
    (
        "parser_mutation_allowed",
        False,
        test_pointer.get("parser_mutation_allowed"),
    ),
    (
        "test_gt_locked_by_decision",
        True,
        decision_split_policy.get("test_ground_truth_remains_locked"),
    ),
    (
        "test_gt_locked_by_pointer",
        True,
        test_pointer.get("test_ground_truth_remains_locked"),
    ),
    (
        "test_predictions_must_be_frozen",
        True,
        test_pointer.get(
            "test_predictions_must_be_frozen_before_ground_truth_open"
        ),
    ),
    (
        "validation_gt_reopened_in_11e",
        0,
        decision_integrity.get(
            "validation_ground_truth_reopened_in_this_cell"
        ),
    ),
    (
        "test_gt_opened_in_11e",
        0,
        decision_integrity.get("test_ground_truth_opened"),
    ),
    (
        "freeze_status",
        "FROZEN_FOR_VALIDATION",
        freeze_manifest.get("status"),
    ),
    (
        "freeze_parser_signature",
        EXPECTED_PARSER_SIGNATURE,
        freeze_parser.get("parser_signature_sha256"),
    ),
]

source_controls = [
    {
        "control": name,
        "expected": expected,
        "actual": actual,
        "status": "VALID" if expected == actual else "INVALID",
    }
    for name, expected, actual in source_values
]

print("=" * 96)
print(
    f"CELL 12A — TEST COHORT PREFLIGHT — VERSION {CELL_VERSION}"
)
print(f"Selection root: {TEST_ROOT}")
print("Test metadata/PDF: AUTHORIZED | Test GT: CLOSED | Parser: FROZEN")
print("=" * 96)
print("SOURCE AND AUTHORIZATION CONTROLS")
display(pd.DataFrame(source_controls))

invalid_source_controls = [
    record["control"]
    for record in source_controls
    if record["status"] != "VALID"
]
if invalid_source_controls:
    raise RuntimeError(
        "CELL 12A SOURCE PREFLIGHT FAILED. "
        f"Kontrol tidak valid: {invalid_source_controls}. "
        "Test ground truth belum dibuka."
    )


# ============================================================
# BUILD COMPLETE TEST COHORT FROM FROZEN RELEASE METADATA
# ============================================================

test_release_records = []
for release_record in release_records:
    split = normalize_split(
        first_scalar(
            release_record,
            ("split", "dataset_split"),
            "release split",
        )
    )
    if split == "test":
        test_release_records.append(release_record)

cohort_records = []
mapping_errors = []

for release_record in test_release_records:
    try:
        document_id = str(
            first_scalar(
                release_record,
                ("document_id",),
                "document_id",
            )
        )
        canonical_invoice_id = str(
            first_scalar(
                release_record,
                ("canonical_invoice_id", "canonical_id"),
                "canonical_invoice_id",
            )
        )
        template_id = str(
            first_scalar(
                release_record,
                ("template_id",),
                "template_id",
            )
        )
        language = str(
            first_scalar(
                release_record,
                ("language", "locale"),
                "language",
            )
        ).casefold()
        currency = str(
            first_scalar(
                release_record,
                ("currency", "currency_code"),
                "currency",
            )
        ).upper()
        item_count = int(
            first_scalar(
                release_record,
                ("item_count", "line_item_count"),
                "item_count",
            )
        )

        render_record = unique_record_by_document(
            render_records,
            document_id,
            "Batch render record",
        )
        render_template_id = str(
            first_scalar(
                render_record,
                ("template_id",),
                "render template_id",
            )
        )
        render_split = normalize_split(
            first_scalar(
                render_record,
                ("split", "dataset_split"),
                "render split",
            )
        )
        if render_template_id != template_id:
            raise RuntimeError(
                f"Template release/render berbeda untuk {document_id}: "
                f"{template_id} != {render_template_id}"
            )
        if render_split != "test":
            raise RuntimeError(
                f"Render record bukan test split: {document_id}"
            )

        pdf_path, pdf_sha256 = locate_test_pdf_and_checksum(
            render_record,
            document_id,
            template_id,
        )

        cohort_records.append(
            {
                "sequence_number": 0,
                "canonical_invoice_id": canonical_invoice_id,
                "document_id": document_id,
                "template_id": template_id,
                "split": "test",
                "language": language,
                "currency": currency,
                "item_count": item_count,
                "pdf_path": str(pdf_path),
                "pdf_sha256": pdf_sha256,
                "release_record_sha256": sha256_object(release_record),
                "render_record_sha256": sha256_object(render_record),
            }
        )

    except Exception as error:
        mapping_errors.append(
            {
                "error_type": type(error).__name__,
                "error": str(error)[:700],
                "record_preview": str(release_record)[:300],
            }
        )

cohort_records.sort(
    key=lambda record: (
        record["template_id"],
        record["document_id"],
    )
)
for sequence_number, record in enumerate(cohort_records, start=1):
    record["sequence_number"] = sequence_number

if mapping_errors:
    print("\nTEST COHORT MAPPING ERRORS")
    display(pd.DataFrame(mapping_errors))
    raise RuntimeError(
        f"Gagal memetakan {len(mapping_errors)} test records. "
        "Test ground truth belum dibuka."
    )


# ============================================================
# TEST COHORT GATES — PDF ONLY, NO GT PAYLOADS
# ============================================================

template_counts = Counter(
    record["template_id"] for record in cohort_records
)
language_counts = Counter(
    record["language"] for record in cohort_records
)
currency_counts = Counter(
    record["currency"] for record in cohort_records
)
template_languages = {
    template_id: sorted(
        {
            record["language"]
            for record in cohort_records
            if record["template_id"] == template_id
        }
    )
    for template_id in sorted(template_counts)
}

cohort_values = [
    (
        "test_release_records",
        EXPECTED_TEST_DOCUMENTS,
        len(test_release_records),
    ),
    ("test_documents", EXPECTED_TEST_DOCUMENTS, len(cohort_records)),
    (
        "unique_document_ids",
        EXPECTED_TEST_DOCUMENTS,
        len({record["document_id"] for record in cohort_records}),
    ),
    (
        "unique_canonical_ids",
        EXPECTED_TEST_DOCUMENTS,
        len(
            {
                record["canonical_invoice_id"]
                for record in cohort_records
            }
        ),
    ),
    (
        "test_templates",
        sorted(EXPECTED_TEST_TEMPLATES),
        sorted(template_counts),
    ),
    (
        "documents_per_template",
        True,
        all(
            template_counts.get(template_id)
            == EXPECTED_DOCUMENTS_PER_TEMPLATE
            for template_id in EXPECTED_TEST_TEMPLATES
        ),
    ),
    (
        "languages_per_template",
        True,
        all(
            set(template_languages.get(template_id, []))
            == EXPECTED_LANGUAGES
            for template_id in EXPECTED_TEST_TEMPLATES
        ),
    ),
    (
        "supported_currencies",
        True,
        set(currency_counts).issubset(SUPPORTED_CURRENCIES),
    ),
    (
        "all_records_test",
        True,
        all(record["split"] == "test" for record in cohort_records),
    ),
    (
        "item_capacity_valid",
        True,
        all(2 <= record["item_count"] <= 8 for record in cohort_records),
    ),
    (
        "pdf_files",
        EXPECTED_TEST_DOCUMENTS,
        sum(Path(record["pdf_path"]).is_file() for record in cohort_records),
    ),
    (
        "unique_pdf_checksums",
        EXPECTED_TEST_DOCUMENTS,
        len({record["pdf_sha256"] for record in cohort_records}),
    ),
    (
        "sequence_numbers",
        list(range(1, EXPECTED_TEST_DOCUMENTS + 1)),
        [record["sequence_number"] for record in cohort_records],
    ),
    ("validation_ground_truth_reopened", 0, 0),
    ("test_ground_truth_loaded", False, False),
    ("test_ground_truth_opened", 0, 0),
]

cohort_controls = [
    {
        "control": name,
        "expected": expected,
        "actual": actual,
        "status": "VALID" if expected == actual else "INVALID",
    }
    for name, expected, actual in cohort_values
]

print("\nTEST COHORT CONTROLS")
display(pd.DataFrame(cohort_controls))
display(pd.DataFrame(cohort_records))

invalid_cohort_controls = [
    record["control"]
    for record in cohort_controls
    if record["status"] != "VALID"
]
if invalid_cohort_controls:
    raise RuntimeError(
        "CELL 12A TEST COHORT GATE FAILED. "
        f"Kontrol tidak valid: {invalid_cohort_controls}. "
        "Test ground truth belum dibuka."
    )


# ============================================================
# IMMUTABLE TEST SELECTION CHECKPOINT
# ============================================================

selection_manifest = {
    "schema_version": "1.0.0",
    "cell_version": CELL_VERSION,
    "status": "READY",
    "stage": "TEST_COHORT_PREFLIGHT",
    "baseline": {
        "baseline_id": test_pointer["baseline_id"],
        "parser_id": test_pointer["parser_id"],
        "parser_version": test_pointer["parser_version"],
        "parser_signature_sha256": test_pointer[
            "parser_signature_sha256"
        ],
        "parser_source_path": str(parser_source_path),
        "parser_source_sha256": source_checksums_before[
            "parser_source"
        ],
        "development_freeze_manifest_path": str(freeze_manifest_path),
        "development_freeze_manifest_sha256": source_checksums_before[
            "freeze_manifest"
        ],
        "validation_decision_manifest_path": str(
            validation_decision_path
        ),
        "validation_decision_manifest_sha256": source_checksums_before[
            "validation_decision"
        ],
        "validation_result_sha256": test_pointer[
            "validation_result_sha256"
        ],
    },
    "scope": {
        "split": "test",
        "documents": EXPECTED_TEST_DOCUMENTS,
        "templates": sorted(EXPECTED_TEST_TEMPLATES),
        "documents_per_template": dict(sorted(template_counts.items())),
        "language_distribution": dict(sorted(language_counts.items())),
        "currency_distribution": dict(sorted(currency_counts.items())),
        "template_languages": template_languages,
        "selection_policy": "ALL_FROZEN_TEST_RECORDS",
    },
    "records": cohort_records,
    "source_artifacts": {
        "current_release_pointer": {
            "path": str(CURRENT_RELEASE_POINTER_PATH),
            "sha256": source_checksums_before[
                "current_release_pointer"
            ],
        },
        "release_index": {
            "path": str(RELEASE_INDEX_PATH),
            "sha256": source_checksums_before["release_index"],
        },
        "release_manifest": {
            "path": str(RELEASE_MANIFEST_PATH),
            "sha256": source_checksums_before["release_manifest"],
        },
        "render_index": {
            "path": str(RENDER_INDEX_PATH),
            "sha256": source_checksums_before["render_index"],
        },
        "test_baseline_pointer": {
            "path": str(TEST_BASELINE_POINTER_PATH),
            "sha256": source_checksums_before[
                "test_baseline_pointer"
            ],
        },
    },
    "access_audit": {
        "release_metadata_records_loaded": len(release_records),
        "render_metadata_records_loaded": len(render_records),
        "test_release_records_selected": len(test_release_records),
        "test_pdf_files_verified": len(cohort_records),
        "validation_ground_truth_reopened": 0,
        "test_ground_truth_loaded": False,
        "test_ground_truth_opened": 0,
        "canonical_payload_loaded": False,
    },
    "integrity": {
        "parser_frozen": True,
        "validation_decision_frozen": True,
        "ground_truth_used_as_selection_input": False,
        "parser_modifications": 0,
        "dataset_modifications": 0,
        "source_modifications": 0,
    },
    "next_stage": {
        "cell": "CELL 12B",
        "action": "BUILD_TEST_UNIFIED_TEXT_LAYER",
        "parser_mutation_allowed": False,
        "test_ground_truth_must_remain_closed": True,
    },
}

selection_action = save_immutable_json(
    TEST_SELECTION_PATH,
    selection_manifest,
)
selection_sha256 = sha256_file(TEST_SELECTION_PATH)
recovered_selection = load_json(TEST_SELECTION_PATH)
if canonical_json(recovered_selection) != canonical_json(selection_manifest):
    raise RuntimeError("Test selection berbeda setelah penulisan.")


# ============================================================
# FINAL SOURCE AND PDF IMMUTABILITY CHECK
# ============================================================

source_checksums_after = {
    "current_release_pointer": sha256_file(CURRENT_RELEASE_POINTER_PATH),
    "release_index": sha256_file(RELEASE_INDEX_PATH),
    "release_manifest": sha256_file(RELEASE_MANIFEST_PATH),
    "render_index": sha256_file(RENDER_INDEX_PATH),
    "test_baseline_pointer": sha256_file(TEST_BASELINE_POINTER_PATH),
    "validation_decision": sha256_file(validation_decision_path),
    "freeze_manifest": sha256_file(freeze_manifest_path),
    "parser_source": sha256_file(parser_source_path),
}

changed_sources = [
    name
    for name, checksum in source_checksums_before.items()
    if source_checksums_after.get(name) != checksum
]
changed_pdfs = [
    record["document_id"]
    for record in cohort_records
    if sha256_file(Path(record["pdf_path"])) != record["pdf_sha256"]
]

if changed_sources or changed_pdfs:
    raise RuntimeError(
        "Source berubah selama Cell 12A: "
        f"manifests={changed_sources}, pdfs={changed_pdfs}"
    )


print()
print(f"Cell version             : {CELL_VERSION}")
print(f"Baseline ID              : {test_pointer['baseline_id']}")
print(f"Parser version           : {test_pointer['parser_version']}")
print(f"Validation decision      : {validation_decision['decision']}")
print(f"Test documents           : {len(cohort_records)}")
print(f"Templates                : {dict(sorted(template_counts.items()))}")
print(f"Languages                : {dict(sorted(language_counts.items()))}")
print(f"Currencies               : {dict(sorted(currency_counts.items()))}")
print(f"PDF files verified       : {len(cohort_records)}")
print(f"Selection action         : {selection_action}")
print(f"Selection manifest       : {TEST_SELECTION_PATH}")
print(f"Selection SHA-256        : {selection_sha256}")
print("Validation GT reopened  : 0")
print("Test GT loaded           : False")
print("Test GT opened           : 0")
print("Canonical payload loaded : False")
print("Parser modifications     : 0")
print("Dataset modifications    : 0")
print("Source modifications     : 0")
print()
print(
    "✅ CELL 12A PASSED — seluruh 40 dokumen test TPL-09 dan "
    "TPL-10 telah dipetakan serta PDF-nya diverifikasi secara "
    "immutable. Ground truth test tetap tertutup; lanjutkan hanya "
    "ke Cell 12B."
)


CELL 12A — TEST COHORT PREFLIGHT — VERSION 1.0.0
Selection root: /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/field_extraction/test/rule_based_baseline_v1_0_2
Test metadata/PDF: AUTHORIZED | Test GT: CLOSED | Parser: FROZEN
SOURCE AND AUTHORIZATION CONTROLS


,control,expected,actual,status
0,release_manifest_status,FROZEN,FROZEN,VALID
1,release_pointer_matches_manifest,True,True,VALID
2,release_records,200,200,VALID
3,render_index_records,200,200,VALID
4,test_pointer_status,ACTIVE_FOR_TEST_PREFLIGHT,ACTIVE_FOR_TEST_PREFLIGHT,VALID
5,baseline_id,RULE-BASED-INVOICE-PARSER-V1@1.0.2,RULE-BASED-INVOICE-PARSER-V1@1.0.2,VALID
6,parser_id,RULE-BASED-INVOICE-PARSER-V1,RULE-BASED-INVOICE-PARSER-V1,VALID
7,parser_version,1.0.2,1.0.2,VALID
8,parser_signature,ffe04f258fb3b1a6a55661698fa6c7dda29ddd7b714c2f...,ffe04f258fb3b1a6a55661698fa6c7dda29ddd7b714c2f...,VALID
9,parser_source_path,True,True,VALID



TEST COHORT CONTROLS


,control,expected,actual,status
0,test_release_records,40,40,VALID
1,test_documents,40,40,VALID
2,unique_document_ids,40,40,VALID
3,unique_canonical_ids,40,40,VALID
4,test_templates,"[TPL-09, TPL-10]","[TPL-09, TPL-10]",VALID
5,documents_per_template,True,True,VALID
6,languages_per_template,True,True,VALID
7,supported_currencies,True,True,VALID
8,all_records_test,True,True,VALID
9,item_capacity_valid,True,True,VALID


,sequence_number,canonical_invoice_id,document_id,template_id,split,language,currency,item_count,pdf_path,pdf_sha256,release_record_sha256,render_record_sha256
0,1,CANON-000161,INV-SYN-000161,TPL-09,test,id,IDR,5,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,c4fa21c9194a60255ef73f7663178ea82dbf5f6c7ded29...,fb5bfaad44153c308c4ef5f61fc366cece9db0509efa02...,ad158acb7d9c44b39649030e2e3be1eed2070093688f1e...
1,2,CANON-000162,INV-SYN-000162,TPL-09,test,id,IDR,3,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,7a103e4c149390b8ec49268d05ae172a3a3c61c716abc9...,0d7777f1dd827f914d2b199cf589860af5ea3a886a1cee...,09c8e4020dfe113f71dca0e7b6e8efec303226370261f4...
2,3,CANON-000163,INV-SYN-000163,TPL-09,test,id,USD,8,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,e2bfc2238ddf6c9b826666b9ffc2524a12f7ddfb0787e1...,abdaaaf3700e1a0a44515ffc7fe270c411a4e420b4c18f...,f1a73be84c59f13c49db5db229e29edcfdb01114ae0ff1...
3,4,CANON-000164,INV-SYN-000164,TPL-09,test,id,IDR,5,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,a40d76cd57b2f1903234ae88a386588398b4076394260c...,fbfb1bdc443b856106bbe67ac972c8215cbf497986adc4...,bf15cbf2a28de7a757b166829e7088e8674d67de892f2c...
4,5,CANON-000165,INV-SYN-000165,TPL-09,test,id,IDR,7,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,ef5863017633eebe3b337e4efa956b6c9562c5be33ee71...,79b174b75c8bcd073c1e92da3759483fa78444a1197ea5...,6862eb3357fb44d88f5a44b99c962d2bc020c6a9e0681c...
5,6,CANON-000166,INV-SYN-000166,TPL-09,test,id,USD,7,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,cfee31d15d59b83c53e32b121348c92ab783df21281f5a...,7b544a0e385353067bebbe80a9fc1d5276de7a09b469f2...,2c8b149aaa71246f4a02b161e6da63e5c49ad91dc00afa...
6,7,CANON-000167,INV-SYN-000167,TPL-09,test,id,IDR,4,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,a3a69702a068dfb0f399011646dfaee7100c74557d7871...,e473a023f23c32c2cfd552cb5be058fb97f450f5047d8c...,91a2d4a051300cc760ba803ecc90f6dc75286dcb5f0c02...
7,8,CANON-000168,INV-SYN-000168,TPL-09,test,id,IDR,5,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,2a00572071e6318aa9f86cdfb248ac42068fe7af81855f...,deea963cee3345f05cfbeef44c404a3c14077c8b03a156...,e7aeaeae91f79560b5c1aefe13c39932d9f37a56c2b4e6...
8,9,CANON-000169,INV-SYN-000169,TPL-09,test,id,IDR,3,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,adfc6ac77ba5f67f0460a8bd88795df98292f7effaf05b...,f2ccec3f82c4561c2473f9503074b0ecc555d3e8537ce0...,7d0b4b72d063ceb88b11fd4ccf5eaca334e665f8b3c2fa...
9,10,CANON-000170,INV-SYN-000170,TPL-09,test,id,IDR,7,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,7edf9bfa9ed6590086dcb4ce0b783c34ec477a9bd8feba...,7998a23b3d2fa07e82afb5e45cc5bb0482a9e4922e12d9...,e63dabfb04e0ff638032fae7fd8542b841a7308c69a963...



Cell version             : 1.0.0
Baseline ID              : RULE-BASED-INVOICE-PARSER-V1@1.0.2
Parser version           : 1.0.2
Validation decision      : VALIDATION_PASSED_BASELINE_ACCEPTED
Test documents           : 40
Templates                : {'TPL-09': 20, 'TPL-10': 20}
Languages                : {'en': 20, 'id': 20}
Currencies               : {'EUR': 6, 'GBP': 4, 'IDR': 16, 'USD': 14}
PDF files verified       : 40
Selection action         : CREATED
Selection manifest       : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/field_extraction/test/rule_based_baseline_v1_0_2/test_cohort_manifest.json
Selection SHA-256        : 443e7833915232760fa2025886e31285da18df1daa41a736526f4e0711ba018b
Validation GT reopened  : 0
Test GT loaded           : False
Test GT opened           : 0
Canonical payload loaded : False
Parser modifications     : 0
Dataset modifications    : 0
Source modifications     : 0

✅ CELL 12A PASSED — seluruh 40 do

In [6]:
from __future__ import annotations

import hashlib
import json
import os
from collections import Counter, defaultdict
from pathlib import Path

import fitz
import pandas as pd
from IPython.display import display


# ============================================================
# CELL 12B — TEST UNIFIED DOCUMENT TEXT LAYER
#             (TEST GROUND TRUTH CLOSED)
# ============================================================

CELL_VERSION = "1.0.0"

DATA_ROOT = Path(
    "/content/drive/MyDrive/InvoiceFlow-AI-Data"
)
BUILD_ROOT = (
    DATA_ROOT
    / "interim"
    / "synthetic_v1_build"
    / "20260904T150025Z"
)
FIELD_ROOT = BUILD_ROOT / "ocr_benchmark" / "field_extraction"
RENDERED_DATASET_ROOT = BUILD_ROOT / "rendered_dataset"

TEST_ROOT = (
    FIELD_ROOT / "test" / "rule_based_baseline_v1_0_2"
)
TEST_SELECTION_PATH = (
    TEST_ROOT / "test_cohort_manifest.json"
)
TEST_BASELINE_POINTER_PATH = (
    FIELD_ROOT / "test_baseline_pointer.json"
)

TEXT_LAYER_ROOT = TEST_ROOT / "text_layers"
SUMMARY_PATH = (
    TEST_ROOT / "test_unified_text_layer_summary.csv"
)
MANIFEST_PATH = (
    TEST_ROOT / "test_unified_text_layer_manifest.json"
)

EXPECTED_BASELINE_ID = "RULE-BASED-INVOICE-PARSER-V1@1.0.2"
EXPECTED_PARSER_ID = "RULE-BASED-INVOICE-PARSER-V1"
EXPECTED_PARSER_VERSION = "1.0.2"
EXPECTED_PARSER_SIGNATURE = (
    "ffe04f258fb3b1a6a55661698fa6c7dda29ddd7b714c2f210b3e6fc1fcd464b3"
)
EXPECTED_DOCUMENTS = 40
EXPECTED_TEMPLATES = {"TPL-09", "TPL-10"}
EXPECTED_LANGUAGES = {"en", "id"}
MINIMUM_NATIVE_WORDS = 10
MINIMUM_NATIVE_CHARACTERS = 50


# ============================================================
# FILE, HASH, AND IMMUTABLE-WRITE HELPERS
# ============================================================

def require_file(path: Path, label: str) -> None:
    if not path.is_file():
        raise FileNotFoundError(f"{label} tidak ditemukan: {path}")
    if path.stat().st_size <= 0:
        raise RuntimeError(f"{label} kosong: {path}")


def load_json(path: Path) -> dict:
    require_file(path, "JSON artifact")
    with path.open("r", encoding="utf-8") as file_handle:
        value = json.load(file_handle)
    if not isinstance(value, dict):
        raise TypeError(f"Root JSON bukan object: {path}")
    return value


def canonical_json(value: object) -> str:
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as file_handle:
        for chunk in iter(lambda: file_handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def atomic_write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(
        f".{path.name}.{os.getpid()}.tmp"
    )
    try:
        temporary_path.write_text(text, encoding="utf-8")
        os.replace(temporary_path, path)
    finally:
        if temporary_path.exists():
            temporary_path.unlink()


def atomic_write_json(path: Path, value: dict) -> None:
    atomic_write_text(
        path,
        json.dumps(value, indent=2, ensure_ascii=False) + "\n",
    )


def save_immutable_json(path: Path, value: dict) -> str:
    if path.exists():
        existing = load_json(path)
        if canonical_json(existing) != canonical_json(value):
            raise RuntimeError(
                f"Checkpoint sudah ada tetapi berbeda: {path}"
            )
        return "RECOVERED"

    atomic_write_json(path, value)
    return "CREATED"


def save_immutable_text(path: Path, text: str) -> str:
    if path.exists():
        existing = path.read_text(encoding="utf-8")
        if existing != text:
            raise RuntimeError(
                f"Checkpoint sudah ada tetapi berbeda: {path}"
            )
        return "RECOVERED"

    atomic_write_text(path, text)
    return "CREATED"


def round_float(value: float) -> float:
    return round(float(value), 6)


# ============================================================
# NATIVE PDF TEXT EXTRACTION
# Keep this representation identical to the development layer.
# ============================================================

def extract_native_layer(pdf_path: Path) -> dict:
    tokens = []
    lines = []
    page_records = []
    global_token_index = 0
    global_line_index = 0

    with fitz.open(str(pdf_path)) as document:
        if document.needs_pass:
            raise RuntimeError(f"PDF terenkripsi: {pdf_path}")

        for page_index, page in enumerate(document):
            width = float(page.rect.width)
            height = float(page.rect.height)
            page_number = page_index + 1

            if width <= 0 or height <= 0:
                raise RuntimeError(
                    f"Dimensi halaman tidak valid: {pdf_path}, "
                    f"page={page_number}"
                )

            page_records.append(
                {
                    "page_number": page_number,
                    "width_points": round_float(width),
                    "height_points": round_float(height),
                    "rotation": int(page.rotation),
                }
            )

            word_rows = page.get_text("words", sort=True)
            grouped_lines = defaultdict(list)

            for word_row in word_rows:
                if len(word_row) < 8:
                    continue

                x0, y0, x1, y1 = map(float, word_row[:4])
                text = str(word_row[4]).strip()
                block_number = int(word_row[5])
                line_number = int(word_row[6])
                word_number = int(word_row[7])

                if not text:
                    continue

                token = {
                    "token_index": global_token_index,
                    "page_number": page_number,
                    "block_number": block_number,
                    "line_number": line_number,
                    "word_number": word_number,
                    "text": text,
                    "bbox_points": [
                        round_float(x0),
                        round_float(y0),
                        round_float(x1),
                        round_float(y1),
                    ],
                    "bbox_normalized": [
                        round_float(x0 / width),
                        round_float(y0 / height),
                        round_float(x1 / width),
                        round_float(y1 / height),
                    ],
                    "confidence": 1.0,
                    "source": "PYMUPDF_NATIVE_TEXT",
                }

                tokens.append(token)
                grouped_lines[(block_number, line_number)].append(token)
                global_token_index += 1

            sorted_line_groups = sorted(
                grouped_lines.items(),
                key=lambda item: (
                    min(token["bbox_points"][1] for token in item[1]),
                    min(token["bbox_points"][0] for token in item[1]),
                ),
            )

            for (block_number, line_number), line_tokens in sorted_line_groups:
                line_tokens = sorted(
                    line_tokens,
                    key=lambda token: (
                        token["word_number"],
                        token["bbox_points"][0],
                    ),
                )

                x0 = min(token["bbox_points"][0] for token in line_tokens)
                y0 = min(token["bbox_points"][1] for token in line_tokens)
                x1 = max(token["bbox_points"][2] for token in line_tokens)
                y1 = max(token["bbox_points"][3] for token in line_tokens)

                lines.append(
                    {
                        "line_index": global_line_index,
                        "page_number": page_number,
                        "block_number": block_number,
                        "line_number": line_number,
                        "text": " ".join(
                            token["text"] for token in line_tokens
                        ),
                        "token_indexes": [
                            token["token_index"] for token in line_tokens
                        ],
                        "bbox_points": [x0, y0, x1, y1],
                        "bbox_normalized": [
                            round_float(x0 / width),
                            round_float(y0 / height),
                            round_float(x1 / width),
                            round_float(y1 / height),
                        ],
                        "confidence": 1.0,
                        "source": "PYMUPDF_NATIVE_TEXT",
                    }
                )
                global_line_index += 1

    full_text = "\n".join(line["text"] for line in lines).strip()
    nonspace_characters = sum(
        not character.isspace() for character in full_text
    )
    usable = bool(
        len(tokens) >= MINIMUM_NATIVE_WORDS
        and nonspace_characters >= MINIMUM_NATIVE_CHARACTERS
    )

    return {
        "usable": usable,
        "pages": page_records,
        "tokens": tokens,
        "lines": lines,
        "full_text": full_text,
        "metrics": {
            "page_count": len(page_records),
            "token_count": len(tokens),
            "line_count": len(lines),
            "nonspace_character_count": nonspace_characters,
        },
    }


# ============================================================
# PREFLIGHT — ONLY FROZEN METADATA; NO GROUND TRUTH
# ============================================================

for required_path, label in (
    (TEST_SELECTION_PATH, "Test cohort manifest"),
    (TEST_BASELINE_POINTER_PATH, "Test baseline pointer"),
):
    require_file(required_path, label)

selection_manifest = load_json(TEST_SELECTION_PATH)
baseline_pointer = load_json(TEST_BASELINE_POINTER_PATH)

freeze_manifest_path = Path(
    str(baseline_pointer.get("development_freeze_manifest_path", ""))
)
validation_decision_path = Path(
    str(baseline_pointer.get("validation_decision_manifest_path", ""))
)
parser_source_path = Path(
    str(baseline_pointer.get("parser_source_path", ""))
)
require_file(freeze_manifest_path, "Frozen baseline manifest")
require_file(validation_decision_path, "Frozen validation decision")
require_file(parser_source_path, "Frozen parser source")
freeze_manifest = load_json(freeze_manifest_path)
validation_decision = load_json(validation_decision_path)

selection_records = selection_manifest.get("records")
if not isinstance(selection_records, list):
    raise TypeError("records pada test cohort manifest bukan list.")

source_checksums_before = {
    "test_selection": sha256_file(TEST_SELECTION_PATH),
    "test_baseline_pointer": sha256_file(
        TEST_BASELINE_POINTER_PATH
    ),
    "freeze_manifest": sha256_file(freeze_manifest_path),
    "validation_decision": sha256_file(validation_decision_path),
    "parser_source": sha256_file(parser_source_path),
}

selection_scope = selection_manifest.get("scope", {})
selection_integrity = selection_manifest.get("integrity", {})
selection_access_audit = selection_manifest.get("access_audit", {})
selection_baseline = selection_manifest.get("baseline", {})
decision_integrity = validation_decision.get("integrity", {})
decision_validation = validation_decision.get("validation_evidence", {})
decision_baseline = validation_decision.get("baseline", {})
decision_split_policy = validation_decision.get("split_policy", {})
freeze_parser = freeze_manifest.get("parser", {})

preflight_values = [
    ("selection_status", "READY", selection_manifest.get("status")),
    (
        "selection_stage",
        "TEST_COHORT_PREFLIGHT",
        selection_manifest.get("stage"),
    ),
    ("selection_split", "test", selection_scope.get("split")),
    (
        "selection_documents",
        EXPECTED_DOCUMENTS,
        selection_scope.get("documents"),
    ),
    (
        "selection_templates",
        sorted(EXPECTED_TEMPLATES),
        sorted(selection_scope.get("templates", [])),
    ),
    (
        "selection_baseline_id",
        EXPECTED_BASELINE_ID,
        selection_baseline.get("baseline_id"),
    ),
    (
        "baseline_pointer_status",
        "ACTIVE_FOR_TEST_PREFLIGHT",
        baseline_pointer.get("status"),
    ),
    (
        "baseline_pointer_id",
        EXPECTED_BASELINE_ID,
        baseline_pointer.get("baseline_id"),
    ),
    (
        "parser_id",
        EXPECTED_PARSER_ID,
        baseline_pointer.get("parser_id"),
    ),
    (
        "parser_version",
        EXPECTED_PARSER_VERSION,
        baseline_pointer.get("parser_version"),
    ),
    (
        "parser_signature",
        EXPECTED_PARSER_SIGNATURE,
        baseline_pointer.get("parser_signature_sha256"),
    ),
    (
        "parser_source_checksum",
        baseline_pointer.get("parser_source_sha256"),
        source_checksums_before["parser_source"],
    ),
    (
        "freeze_status",
        "FROZEN_FOR_VALIDATION",
        freeze_manifest.get("status"),
    ),
    (
        "freeze_checksum",
        baseline_pointer.get("development_freeze_manifest_sha256"),
        source_checksums_before["freeze_manifest"],
    ),
    (
        "validation_decision_status",
        "FROZEN_FOR_TEST_PREFLIGHT",
        validation_decision.get("status"),
    ),
    (
        "validation_decision",
        "VALIDATION_PASSED_BASELINE_ACCEPTED",
        validation_decision.get("decision"),
    ),
    (
        "validation_decision_checksum",
        baseline_pointer.get("validation_decision_manifest_sha256"),
        source_checksums_before["validation_decision"],
    ),
    (
        "validation_evaluation_status",
        "PASSED",
        decision_validation.get("evaluation_status"),
    ),
    (
        "validation_result_fingerprint",
        baseline_pointer.get("validation_result_sha256"),
        decision_validation.get("validation_result_sha256"),
    ),
    (
        "selection_validation_result_fingerprint",
        baseline_pointer.get("validation_result_sha256"),
        selection_baseline.get("validation_result_sha256"),
    ),
    (
        "decision_baseline_id",
        EXPECTED_BASELINE_ID,
        decision_baseline.get("baseline_id"),
    ),
    (
        "decision_parser_id",
        EXPECTED_PARSER_ID,
        decision_baseline.get("parser_id"),
    ),
    (
        "decision_parser_version",
        EXPECTED_PARSER_VERSION,
        decision_baseline.get("parser_version"),
    ),
    (
        "decision_parser_signature",
        EXPECTED_PARSER_SIGNATURE,
        decision_baseline.get("parser_signature_sha256"),
    ),
    (
        "decision_parser_source_checksum",
        source_checksums_before["parser_source"],
        decision_baseline.get("parser_source_sha256"),
    ),
    (
        "decision_freeze_checksum",
        source_checksums_before["freeze_manifest"],
        decision_baseline.get("development_freeze_manifest_sha256"),
    ),
    (
        "decision_next_allowed_stage",
        "TEST_COHORT_PREFLIGHT",
        decision_split_policy.get("next_allowed_stage"),
    ),
    (
        "decision_next_allowed_split",
        "test",
        decision_split_policy.get("next_allowed_split"),
    ),
    (
        "test_preflight_allowed",
        True,
        decision_split_policy.get("test_preflight_allowed"),
    ),
    (
        "test_gt_locked_by_decision",
        True,
        decision_split_policy.get("test_ground_truth_remains_locked"),
    ),
    (
        "next_allowed_stage",
        "TEST_COHORT_PREFLIGHT",
        baseline_pointer.get("next_allowed_stage"),
    ),
    (
        "next_allowed_split",
        "test",
        baseline_pointer.get("next_allowed_split"),
    ),
    (
        "parser_mutation_allowed",
        False,
        baseline_pointer.get("parser_mutation_allowed"),
    ),
    (
        "selection_parser_id",
        EXPECTED_PARSER_ID,
        selection_baseline.get("parser_id"),
    ),
    (
        "selection_parser_version",
        EXPECTED_PARSER_VERSION,
        selection_baseline.get("parser_version"),
    ),
    (
        "selection_parser_signature",
        EXPECTED_PARSER_SIGNATURE,
        selection_baseline.get("parser_signature_sha256"),
    ),
    (
        "selection_parser_source_checksum",
        source_checksums_before["parser_source"],
        selection_baseline.get("parser_source_sha256"),
    ),
    (
        "selection_freeze_checksum",
        source_checksums_before["freeze_manifest"],
        selection_baseline.get("development_freeze_manifest_sha256"),
    ),
    (
        "selection_validation_decision_checksum",
        source_checksums_before["validation_decision"],
        selection_baseline.get("validation_decision_manifest_sha256"),
    ),
    (
        "test_ground_truth_loaded",
        False,
        selection_access_audit.get("test_ground_truth_loaded"),
    ),
    (
        "canonical_payload_loaded",
        False,
        selection_access_audit.get("canonical_payload_loaded"),
    ),
    (
        "test_ground_truth_opened",
        0,
        selection_access_audit.get("test_ground_truth_opened"),
    ),
    (
        "test_ground_truth_locked",
        True,
        baseline_pointer.get("test_ground_truth_remains_locked"),
    ),
    (
        "test_predictions_must_be_frozen",
        True,
        baseline_pointer.get(
            "test_predictions_must_be_frozen_before_ground_truth_open"
        ),
    ),
    (
        "selection_parser_frozen",
        True,
        selection_integrity.get("parser_frozen"),
    ),
    (
        "selection_validation_decision_frozen",
        True,
        selection_integrity.get("validation_decision_frozen"),
    ),
    (
        "validation_ground_truth_reopened_in_selection",
        0,
        selection_access_audit.get("validation_ground_truth_reopened"),
    ),
    (
        "decision_test_gt_opened",
        0,
        decision_integrity.get("test_ground_truth_opened"),
    ),
    (
        "freeze_parser_id",
        EXPECTED_PARSER_ID,
        freeze_parser.get("parser_id"),
    ),
    (
        "freeze_parser_version",
        EXPECTED_PARSER_VERSION,
        freeze_parser.get("parser_version"),
    ),
    (
        "freeze_parser_signature",
        EXPECTED_PARSER_SIGNATURE,
        freeze_parser.get("parser_signature_sha256"),
    ),
    (
        "freeze_parser_source_checksum",
        source_checksums_before["parser_source"],
        freeze_parser.get("source_sha256"),
    ),
]

preflight_controls = [
    {
        "control": name,
        "expected": expected,
        "actual": actual,
        "status": "VALID" if expected == actual else "INVALID",
    }
    for name, expected, actual in preflight_values
]
display(pd.DataFrame(preflight_controls))

invalid_preflight = [
    row["control"]
    for row in preflight_controls
    if row["status"] != "VALID"
]
if invalid_preflight:
    raise RuntimeError(
        "CELL 12B PREFLIGHT FAILED. "
        f"Kontrol tidak valid: {invalid_preflight}. "
        "Test ground truth belum dibuka."
    )


# ============================================================
# VERIFY THE 40 FROZEN TEST SOURCES
# ============================================================

source_records = []
source_errors = []

for record_index, source_record in enumerate(selection_records, start=1):
    try:
        if not isinstance(source_record, dict):
            raise TypeError("Cohort record bukan object.")

        document_id = str(source_record.get("document_id", "")).strip()
        template_id = str(source_record.get("template_id", "")).strip()
        split = str(source_record.get("split", "")).strip().casefold()
        language = str(source_record.get("language", "")).strip().casefold()
        pdf_path = Path(str(source_record.get("pdf_path", "")))
        expected_pdf_sha256 = str(
            source_record.get("pdf_sha256", "")
        ).casefold()

        if not document_id:
            raise RuntimeError("document_id kosong.")
        if template_id not in EXPECTED_TEMPLATES:
            raise RuntimeError(
                f"Template di luar test cohort: {template_id}"
            )
        if split != "test":
            raise RuntimeError(f"Split bukan test: {split}")
        if language not in EXPECTED_LANGUAGES:
            raise RuntimeError(f"Language tidak didukung: {language}")
        if not pdf_path.resolve().is_relative_to(
            (RENDERED_DATASET_ROOT / "pdf" / "test").resolve()
        ):
            raise RuntimeError(
                f"PDF berada di luar test PDF root: {document_id}"
            )
        if "ground_truth" in {
            part.casefold() for part in pdf_path.parts
        }:
            raise RuntimeError(
                f"Path ground truth dilarang sebagai sumber: {document_id}"
            )

        require_file(pdf_path, f"Test PDF {document_id}")
        actual_pdf_sha256 = sha256_file(pdf_path)
        if actual_pdf_sha256.casefold() != expected_pdf_sha256:
            raise RuntimeError(
                f"Checksum PDF berubah untuk {document_id}: "
                f"expected={expected_pdf_sha256}, "
                f"actual={actual_pdf_sha256}"
            )

        source_records.append(
            {
                **source_record,
                "sequence_number": int(
                    source_record.get("sequence_number", record_index)
                ),
                "document_id": document_id,
                "template_id": template_id,
                "split": split,
                "language": language,
                "pdf_path": str(pdf_path),
                "pdf_sha256": actual_pdf_sha256,
            }
        )
    except Exception as error:
        source_errors.append(
            {
                "record_index": record_index,
                "error_type": type(error).__name__,
                "error": str(error)[:700],
            }
        )

if source_errors:
    display(pd.DataFrame(source_errors))
    raise RuntimeError(
        f"CELL 12B SOURCE VERIFICATION FAILED: "
        f"{len(source_errors)} source tidak valid."
    )

source_records.sort(key=lambda row: int(row["sequence_number"]))

source_gate_values = [
    ("source_records", EXPECTED_DOCUMENTS, len(source_records)),
    (
        "unique_document_ids",
        EXPECTED_DOCUMENTS,
        len({row["document_id"] for row in source_records}),
    ),
    (
        "source_templates",
        sorted(EXPECTED_TEMPLATES),
        sorted({row["template_id"] for row in source_records}),
    ),
    (
        "source_languages",
        sorted(EXPECTED_LANGUAGES),
        sorted({row["language"] for row in source_records}),
    ),
    (
        "sequence_numbers",
        list(range(1, EXPECTED_DOCUMENTS + 1)),
        [int(row["sequence_number"]) for row in source_records],
    ),
    (
        "pdf_files",
        EXPECTED_DOCUMENTS,
        sum(Path(row["pdf_path"]).is_file() for row in source_records),
    ),
    (
        "unique_pdf_checksums",
        EXPECTED_DOCUMENTS,
        len({row["pdf_sha256"] for row in source_records}),
    ),
    (
        "all_sources_are_test_pdfs",
        True,
        all(
            Path(row["pdf_path"]).resolve().is_relative_to(
                (RENDERED_DATASET_ROOT / "pdf" / "test").resolve()
            )
            for row in source_records
        ),
    ),
    ("test_ground_truth_opened", 0, 0),
]

source_gate_controls = [
    {
        "control": name,
        "expected": expected,
        "actual": actual,
        "status": "VALID" if expected == actual else "INVALID",
    }
    for name, expected, actual in source_gate_values
]
display(pd.DataFrame(source_gate_controls))

invalid_source_gates = [
    row["control"]
    for row in source_gate_controls
    if row["status"] != "VALID"
]
if invalid_source_gates:
    raise RuntimeError(
        "CELL 12B SOURCE GATE FAILED. "
        f"Kontrol tidak valid: {invalid_source_gates}"
    )

pdf_checksums_before = {
    row["document_id"]: row["pdf_sha256"] for row in source_records
}


# ============================================================
# BUILD OR RECOVER TEST TEXT LAYERS
# ============================================================

runtime_records = []
manifest_records = []
processing_errors = []
newly_created = 0
recovered = 0

print()
print("=" * 88)
print(
    f"CELL 12B — TEST UNIFIED TEXT LAYER — "
    f"VERSION {CELL_VERSION}"
)
print(f"Text-layer root: {TEXT_LAYER_ROOT}")
print("Test PDFs: AUTHORIZED | Test GT: CLOSED | OCR executions: 0")
print("=" * 88)
print(f"Membangun text layer untuk {len(source_records)} dokumen test...\n")

for position, source in enumerate(source_records, start=1):
    document_id = source["document_id"]
    template_id = source["template_id"]
    pdf_path = Path(source["pdf_path"])
    result_path = (
        TEXT_LAYER_ROOT
        / template_id
        / f"{document_id}_text_layer.json"
    )

    try:
        native_layer = extract_native_layer(pdf_path)

        # The frozen test PDFs are digital PDFs. A non-usable native
        # layer is therefore a source/routing exception and must not be hidden
        # by changing engines during the blind test run.
        if not native_layer["usable"]:
            metrics = native_layer["metrics"]
            raise RuntimeError(
                "OCR_FALLBACK_REQUIRED: native text berada di bawah "
                f"threshold untuk {document_id}; "
                f"tokens={metrics['token_count']}, "
                f"nonspace_characters="
                f"{metrics['nonspace_character_count']}. "
                "Hentikan test dan audit routing; jangan membuka "
                "ground truth."
            )

        route_id = "NATIVE_PDF_TEXT"
        engine = "PyMuPDF"

        text_layer = {
            "schema_version": "1.0.0",
            "status": "PASSED",
            "document": {
                "canonical_invoice_id": source.get(
                    "canonical_invoice_id"
                ),
                "document_id": document_id,
                "template_id": template_id,
                "split": "test",
                "language": source.get("language"),
                "currency": source.get("currency"),
                "item_count": source.get("item_count"),
            },
            "source": {
                "pdf_path": str(pdf_path),
                "pdf_sha256": source["pdf_sha256"],
                "test_selection_path": str(
                    TEST_SELECTION_PATH
                ),
                "test_selection_sha256": source_checksums_before[
                    "test_selection"
                ],
                "fallback_artifact": None,
            },
            "routing": {
                "route_id": route_id,
                "engine": engine,
                "preprocessing": "NONE",
                "ocr_executed_in_this_cell": False,
            },
            "pages": native_layer["pages"],
            "tokens": native_layer["tokens"],
            "lines": native_layer["lines"],
            "full_text": native_layer["full_text"],
            "metrics": native_layer["metrics"],
            "integrity": {
                "test_ground_truth_loaded": False,
                "canonical_payload_loaded": False,
                "test_pdf_read": True,
                "test_ground_truth_opened": 0,
                "validation_ground_truth_reopened": 0,
                "dataset_modifications": 0,
                "source_pdf_modifications": 0,
            },
        }

        checkpoint_action = save_immutable_json(result_path, text_layer)
        if checkpoint_action == "CREATED":
            newly_created += 1
            execution = "NEW"
        else:
            recovered += 1
            execution = "RECOVERED"

        persisted = load_json(result_path)
        if canonical_json(persisted) != canonical_json(text_layer):
            raise RuntimeError(
                f"Text layer berbeda setelah penulisan: {document_id}"
            )

        metrics = text_layer["metrics"]
        manifest_record = {
            "sequence_number": int(source["sequence_number"]),
            "document_id": document_id,
            "template_id": template_id,
            "language": source.get("language"),
            "route_id": route_id,
            "engine": engine,
            "page_count": int(metrics["page_count"]),
            "token_count": int(metrics["token_count"]),
            "line_count": int(metrics["line_count"]),
            "nonspace_character_count": int(
                metrics["nonspace_character_count"]
            ),
            "pdf_path": str(pdf_path),
            "pdf_sha256": source["pdf_sha256"],
            "text_layer_path": str(result_path),
            "text_layer_sha256": sha256_file(result_path),
            "status": "PASSED",
        }
        manifest_records.append(manifest_record)
        runtime_records.append(
            {**manifest_record, "execution": execution}
        )

        print(
            f"[{position:02d}/{len(source_records):02d}] "
            f"{document_id} | route={route_id} | "
            f"tokens={metrics['token_count']} | "
            f"lines={metrics['line_count']} | {execution}"
        )

    except Exception as error:
        processing_errors.append(
            {
                "sequence_number": source.get("sequence_number"),
                "document_id": document_id,
                "template_id": template_id,
                "error_type": type(error).__name__,
                "error": str(error)[:900],
            }
        )
        print(
            f"[{position:02d}/{len(source_records):02d}] "
            f"{document_id} | ERROR: "
            f"{type(error).__name__}: {error}"
        )


# ============================================================
# RESULT GATES
# ============================================================

if processing_errors:
    print("\nPROCESSING ERRORS")
    display(pd.DataFrame(processing_errors))

runtime_table = pd.DataFrame(runtime_records)
manifest_table = pd.DataFrame(manifest_records)
route_counts = Counter(
    row["route_id"] for row in manifest_records
)

result_values = [
    ("result_records", EXPECTED_DOCUMENTS, len(manifest_records)),
    (
        "result_files",
        EXPECTED_DOCUMENTS,
        sum(
            Path(row["text_layer_path"]).is_file()
            for row in manifest_records
        ),
    ),
    (
        "unique_document_ids",
        EXPECTED_DOCUMENTS,
        len({row["document_id"] for row in manifest_records}),
    ),
    (
        "templates",
        sorted(EXPECTED_TEMPLATES),
        sorted({row["template_id"] for row in manifest_records}),
    ),
    (
        "native_text_routes",
        EXPECTED_DOCUMENTS,
        route_counts.get("NATIVE_PDF_TEXT", 0),
    ),
    (
        "ocr_fallback_routes",
        0,
        sum(
            row["route_id"] != "NATIVE_PDF_TEXT"
            for row in manifest_records
        ),
    ),
    (
        "single_page_documents",
        EXPECTED_DOCUMENTS,
        sum(row["page_count"] == 1 for row in manifest_records),
    ),
    (
        "usable_native_layers",
        EXPECTED_DOCUMENTS,
        sum(
            row["token_count"] >= MINIMUM_NATIVE_WORDS
            and row["nonspace_character_count"]
            >= MINIMUM_NATIVE_CHARACTERS
            for row in manifest_records
        ),
    ),
    ("processing_errors", 0, len(processing_errors)),
    ("test_pdf_documents_read", EXPECTED_DOCUMENTS, len(manifest_records)),
    ("validation_ground_truth_reopened", 0, 0),
    ("test_ground_truth_opened", 0, 0),
    ("ocr_executions", 0, 0),
]

result_controls = [
    {
        "control": name,
        "expected": expected,
        "actual": actual,
        "status": "VALID" if expected == actual else "INVALID",
    }
    for name, expected, actual in result_values
]

print("\nRESULT CONTROLS")
display(pd.DataFrame(result_controls))
if not runtime_table.empty:
    display(runtime_table)

invalid_result_controls = [
    row["control"]
    for row in result_controls
    if row["status"] != "VALID"
]
if invalid_result_controls:
    raise RuntimeError(
        "CELL 12B TEXT-LAYER BUILD FAILED. "
        f"Kontrol tidak valid: {invalid_result_controls}"
    )


# ============================================================
# IMMUTABLE SUMMARY AND MANIFEST
# ============================================================

summary_columns = [
    "sequence_number",
    "document_id",
    "template_id",
    "language",
    "route_id",
    "engine",
    "page_count",
    "token_count",
    "line_count",
    "nonspace_character_count",
    "pdf_path",
    "pdf_sha256",
    "text_layer_path",
    "text_layer_sha256",
    "status",
]

summary_text = manifest_table[summary_columns].to_csv(
    index=False,
    lineterminator="\n",
)
summary_action = save_immutable_text(SUMMARY_PATH, summary_text)

text_layer_manifest = {
    "schema_version": "1.0.0",
    "cell_version": CELL_VERSION,
    "status": "PASSED",
    "stage": "TEST_UNIFIED_DOCUMENT_TEXT_LAYER",
    "baseline": {
        "baseline_id": baseline_pointer["baseline_id"],
        "parser_id": baseline_pointer["parser_id"],
        "parser_version": baseline_pointer["parser_version"],
        "parser_signature_sha256": baseline_pointer[
            "parser_signature_sha256"
        ],
        "parser_source_path": str(parser_source_path),
        "parser_source_sha256": source_checksums_before[
            "parser_source"
        ],
        "development_freeze_manifest_path": str(freeze_manifest_path),
        "development_freeze_manifest_sha256": source_checksums_before[
            "freeze_manifest"
        ],
        "validation_decision_manifest_path": str(
            validation_decision_path
        ),
        "validation_decision_manifest_sha256": source_checksums_before[
            "validation_decision"
        ],
        "validation_result_sha256": baseline_pointer[
            "validation_result_sha256"
        ],
    },
    "scope": {
        "split": "test",
        "documents": EXPECTED_DOCUMENTS,
        "templates": sorted(EXPECTED_TEMPLATES),
        "languages": sorted(EXPECTED_LANGUAGES),
        "test_pdf_documents_read": EXPECTED_DOCUMENTS,
        "validation_ground_truth_reopened": 0,
        "test_ground_truth_opened": 0,
    },
    "routing": {
        "native_pdf_engine": "PyMuPDF",
        "minimum_native_words": MINIMUM_NATIVE_WORDS,
        "minimum_native_characters": MINIMUM_NATIVE_CHARACTERS,
        "route_distribution": dict(sorted(route_counts.items())),
        "ocr_fallback_policy": "STOP_AND_AUDIT",
        "ocr_executions_in_this_cell": 0,
    },
    "records": manifest_records,
    "artifacts": {
        "summary_path": str(SUMMARY_PATH),
        "summary_sha256": sha256_file(SUMMARY_PATH),
        "text_layer_root": str(TEXT_LAYER_ROOT),
    },
    "input_artifacts": {
        "test_selection": {
            "path": str(TEST_SELECTION_PATH),
            "sha256": source_checksums_before[
                "test_selection"
            ],
        },
        "test_baseline_pointer": {
            "path": str(TEST_BASELINE_POINTER_PATH),
            "sha256": source_checksums_before[
                "test_baseline_pointer"
            ],
        },
        "freeze_manifest": {
            "path": str(freeze_manifest_path),
            "sha256": source_checksums_before["freeze_manifest"],
        },
        "validation_decision": {
            "path": str(validation_decision_path),
            "sha256": source_checksums_before["validation_decision"],
        },
        "parser_source": {
            "path": str(parser_source_path),
            "sha256": source_checksums_before["parser_source"],
        },
    },
    "integrity": {
        "test_ground_truth_loaded": False,
        "canonical_payload_loaded": False,
        "test_pdf_documents_read": EXPECTED_DOCUMENTS,
        "validation_ground_truth_reopened": 0,
        "test_ground_truth_opened": 0,
        "ocr_executions": 0,
        "parser_modifications": 0,
        "dataset_modifications": 0,
        "source_pdf_modifications": 0,
        "source_manifest_modifications": 0,
    },
    "next_stage": {
        "cell": "CELL 12C",
        "action": "RUN_FROZEN_PARSER_ON_TEST_TEXT_LAYERS",
        "parser_mutation_allowed": False,
        "test_ground_truth_must_remain_closed": True,
        "test_predictions_must_be_frozen_before_ground_truth_open": True,
    },
}

manifest_action = save_immutable_json(
    MANIFEST_PATH,
    text_layer_manifest,
)

persisted_manifest = load_json(MANIFEST_PATH)
if canonical_json(persisted_manifest) != canonical_json(text_layer_manifest):
    raise RuntimeError("Manifest Cell 12B berbeda setelah penulisan.")


# ============================================================
# FINAL SOURCE IMMUTABILITY VERIFICATION
# ============================================================

source_checksums_after = {
    "test_selection": sha256_file(TEST_SELECTION_PATH),
    "test_baseline_pointer": sha256_file(
        TEST_BASELINE_POINTER_PATH
    ),
    "freeze_manifest": sha256_file(freeze_manifest_path),
    "validation_decision": sha256_file(validation_decision_path),
    "parser_source": sha256_file(parser_source_path),
}

changed_source_manifests = [
    name
    for name, checksum in source_checksums_before.items()
    if source_checksums_after.get(name) != checksum
]

changed_pdfs = []
for source in source_records:
    current_checksum = sha256_file(Path(source["pdf_path"]))
    if current_checksum != pdf_checksums_before[source["document_id"]]:
        changed_pdfs.append(source["document_id"])

if changed_source_manifests or changed_pdfs:
    raise RuntimeError(
        "Source berubah selama Cell 12B: "
        f"manifests={changed_source_manifests}, "
        f"pdfs={changed_pdfs}"
    )


print()
print(f"Cell version          : {CELL_VERSION}")
print(f"Baseline ID           : {baseline_pointer['baseline_id']}")
print(f"Parser version        : {baseline_pointer['parser_version']}")
print(f"Test documents        : {len(manifest_records)}")
print(
    "Templates             : "
    f"{dict(sorted(Counter(row['template_id'] for row in manifest_records).items()))}"
)
print(
    "Languages             : "
    f"{dict(sorted(Counter(row['language'] for row in manifest_records).items()))}"
)
print(
    "Native text routes    : "
    f"{route_counts.get('NATIVE_PDF_TEXT', 0)}"
)
print("OCR fallback routes   : 0")
print(f"New text layers       : {newly_created}")
print(f"Recovered text layers : {recovered}")
print(f"Summary action        : {summary_action}")
print(f"Manifest action       : {manifest_action}")
print(f"Text-layer root       : {TEXT_LAYER_ROOT}")
print(f"Summary               : {SUMMARY_PATH}")
print(f"Manifest              : {MANIFEST_PATH}")
print(f"Manifest SHA-256      : {sha256_file(MANIFEST_PATH)}")
print(f"Test PDFs read         : {len(manifest_records)}")
print("Validation GT reopened: 0")
print("Test GT opened        : 0")
print("OCR executions        : 0")
print("Dataset modifications : 0")
print("Source modifications  : 0")
print()
print(
    "✅ CELL 12B PASSED — unified text layer untuk 40 dokumen "
    "test berhasil dibuat secara immutable tanpa membuka "
    "ground truth test. Lanjutkan ke Cell 12C."
)


,control,expected,actual,status
0,selection_status,READY,READY,VALID
1,selection_stage,TEST_COHORT_PREFLIGHT,TEST_COHORT_PREFLIGHT,VALID
2,selection_split,test,test,VALID
3,selection_documents,40,40,VALID
4,selection_templates,"[TPL-09, TPL-10]","[TPL-09, TPL-10]",VALID
5,selection_baseline_id,RULE-BASED-INVOICE-PARSER-V1@1.0.2,RULE-BASED-INVOICE-PARSER-V1@1.0.2,VALID
6,baseline_pointer_status,ACTIVE_FOR_TEST_PREFLIGHT,ACTIVE_FOR_TEST_PREFLIGHT,VALID
7,baseline_pointer_id,RULE-BASED-INVOICE-PARSER-V1@1.0.2,RULE-BASED-INVOICE-PARSER-V1@1.0.2,VALID
8,parser_id,RULE-BASED-INVOICE-PARSER-V1,RULE-BASED-INVOICE-PARSER-V1,VALID
9,parser_version,1.0.2,1.0.2,VALID


,control,expected,actual,status
0,source_records,40,40,VALID
1,unique_document_ids,40,40,VALID
2,source_templates,"[TPL-09, TPL-10]","[TPL-09, TPL-10]",VALID
3,source_languages,"[en, id]","[en, id]",VALID
4,sequence_numbers,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",VALID
5,pdf_files,40,40,VALID
6,unique_pdf_checksums,40,40,VALID
7,all_sources_are_test_pdfs,True,True,VALID
8,test_ground_truth_opened,0,0,VALID



CELL 12B — TEST UNIFIED TEXT LAYER — VERSION 1.0.0
Text-layer root: /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/field_extraction/test/rule_based_baseline_v1_0_2/text_layers
Test PDFs: AUTHORIZED | Test GT: CLOSED | OCR executions: 0
Membangun text layer untuk 40 dokumen test...

[01/40] INV-SYN-000161 | route=NATIVE_PDF_TEXT | tokens=142 | lines=68 | NEW
[02/40] INV-SYN-000162 | route=NATIVE_PDF_TEXT | tokens=124 | lines=58 | NEW
[03/40] INV-SYN-000163 | route=NATIVE_PDF_TEXT | tokens=168 | lines=83 | NEW
[04/40] INV-SYN-000164 | route=NATIVE_PDF_TEXT | tokens=142 | lines=68 | NEW
[05/40] INV-SYN-000165 | route=NATIVE_PDF_TEXT | tokens=160 | lines=78 | NEW
[06/40] INV-SYN-000166 | route=NATIVE_PDF_TEXT | tokens=161 | lines=78 | NEW
[07/40] INV-SYN-000167 | route=NATIVE_PDF_TEXT | tokens=131 | lines=63 | NEW
[08/40] INV-SYN-000168 | route=NATIVE_PDF_TEXT | tokens=143 | lines=68 | NEW
[09/40] INV-SYN-000169 | route=NATIVE_PDF_TEXT

,control,expected,actual,status
0,result_records,40,40,VALID
1,result_files,40,40,VALID
2,unique_document_ids,40,40,VALID
3,templates,"[TPL-09, TPL-10]","[TPL-09, TPL-10]",VALID
4,native_text_routes,40,40,VALID
5,ocr_fallback_routes,0,0,VALID
6,single_page_documents,40,40,VALID
7,usable_native_layers,40,40,VALID
8,processing_errors,0,0,VALID
9,test_pdf_documents_read,40,40,VALID


,sequence_number,document_id,template_id,language,route_id,engine,page_count,token_count,line_count,nonspace_character_count,pdf_path,pdf_sha256,text_layer_path,text_layer_sha256,status,execution
0,1,INV-SYN-000161,TPL-09,id,NATIVE_PDF_TEXT,PyMuPDF,1,142,68,856,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,c4fa21c9194a60255ef73f7663178ea82dbf5f6c7ded29...,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,bda65be2cdf3ff8055d0f01951144f305fe8f3f1119abf...,PASSED,NEW
1,2,INV-SYN-000162,TPL-09,id,NATIVE_PDF_TEXT,PyMuPDF,1,124,58,763,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,7a103e4c149390b8ec49268d05ae172a3a3c61c716abc9...,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,f4dbe6ee441b7559c2a862c115911e79e2fa99b43cd057...,PASSED,NEW
2,3,INV-SYN-000163,TPL-09,id,NATIVE_PDF_TEXT,PyMuPDF,1,168,83,998,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,e2bfc2238ddf6c9b826666b9ffc2524a12f7ddfb0787e1...,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,852b50e09568b5ed28d59d1ec3a7442dad26e438836832...,PASSED,NEW
3,4,INV-SYN-000164,TPL-09,id,NATIVE_PDF_TEXT,PyMuPDF,1,142,68,850,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,a40d76cd57b2f1903234ae88a386588398b4076394260c...,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,01b7985fefc72c0c66d5bec0549f514b3b19722a69ee74...,PASSED,NEW
4,5,INV-SYN-000165,TPL-09,id,NATIVE_PDF_TEXT,PyMuPDF,1,160,78,946,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,ef5863017633eebe3b337e4efa956b6c9562c5be33ee71...,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,c72d318b229d7875a7802567ba11537c412d18fc367ca0...,PASSED,NEW
5,6,INV-SYN-000166,TPL-09,id,NATIVE_PDF_TEXT,PyMuPDF,1,161,78,955,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,cfee31d15d59b83c53e32b121348c92ab783df21281f5a...,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,24711386bf473810a53db39add40944808056510ea2651...,PASSED,NEW
6,7,INV-SYN-000167,TPL-09,id,NATIVE_PDF_TEXT,PyMuPDF,1,131,63,803,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,a3a69702a068dfb0f399011646dfaee7100c74557d7871...,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,799bd9828d51c014590cf54b6fe5279d408ac4b1b059c9...,PASSED,NEW
7,8,INV-SYN-000168,TPL-09,id,NATIVE_PDF_TEXT,PyMuPDF,1,143,68,868,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,2a00572071e6318aa9f86cdfb248ac42068fe7af81855f...,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,c74474df82d1326e50d0d984b9fc0adaa9a23ffbed5700...,PASSED,NEW
8,9,INV-SYN-000169,TPL-09,id,NATIVE_PDF_TEXT,PyMuPDF,1,124,58,762,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,adfc6ac77ba5f67f0460a8bd88795df98292f7effaf05b...,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,ff13dbd1198cf8ec9b776541334ba5f551fa23c5b5a8f5...,PASSED,NEW
9,10,INV-SYN-000170,TPL-09,id,NATIVE_PDF_TEXT,PyMuPDF,1,160,78,967,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,7edf9bfa9ed6590086dcb4ce0b783c34ec477a9bd8feba...,/content/drive/MyDrive/InvoiceFlow-AI-Data/int...,1a87673609c0b0ed8c1096817774f0f15bf32aa4fd27c6...,PASSED,NEW



Cell version          : 1.0.0
Baseline ID           : RULE-BASED-INVOICE-PARSER-V1@1.0.2
Parser version        : 1.0.2
Test documents        : 40
Templates             : {'TPL-09': 20, 'TPL-10': 20}
Languages             : {'en': 20, 'id': 20}
Native text routes    : 40
OCR fallback routes   : 0
New text layers       : 40
Recovered text layers : 0
Summary action        : CREATED
Manifest action       : CREATED
Text-layer root       : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/field_extraction/test/rule_based_baseline_v1_0_2/text_layers
Summary               : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/field_extraction/test/rule_based_baseline_v1_0_2/test_unified_text_layer_summary.csv
Manifest              : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/field_extraction/test/rule_based_baseline_v1_0_2/test_unified_te

In [7]:
from __future__ import annotations

import ast
import hashlib
import json
import os
from collections import Counter
from decimal import Decimal
from pathlib import Path

import pandas as pd
from IPython.display import display


# ============================================================
# CELL 12C — RUN FROZEN PARSER ON TEST
#             (BLIND PREDICTION; GROUND TRUTH CLOSED)
# ============================================================

CELL_VERSION = "1.0.0"

DATA_ROOT = Path(
    "/content/drive/MyDrive/InvoiceFlow-AI-Data"
)
BUILD_ROOT = (
    DATA_ROOT
    / "interim"
    / "synthetic_v1_build"
    / "20260904T150025Z"
)
FIELD_ROOT = BUILD_ROOT / "ocr_benchmark" / "field_extraction"

TEST_ROOT = (
    FIELD_ROOT / "test" / "rule_based_baseline_v1_0_2"
)
TEST_BASELINE_POINTER_PATH = (
    FIELD_ROOT / "test_baseline_pointer.json"
)
TEXT_LAYER_MANIFEST_PATH = (
    TEST_ROOT / "test_unified_text_layer_manifest.json"
)

PREDICTION_ROOT = TEST_ROOT / "predictions"
SUMMARY_PATH = TEST_ROOT / "test_prediction_summary.csv"
MANIFEST_PATH = TEST_ROOT / "test_prediction_manifest.json"

EXPECTED_BASELINE_ID = "RULE-BASED-INVOICE-PARSER-V1@1.0.2"
EXPECTED_PARSER_ID = "RULE-BASED-INVOICE-PARSER-V1"
EXPECTED_PARSER_VERSION = "1.0.2"
EXPECTED_PARSER_SIGNATURE = (
    "ffe04f258fb3b1a6a55661698fa6c7dda29ddd7b714c2f210b3e6fc1fcd464b3"
)
EXPECTED_DOCUMENTS = 40
EXPECTED_TEMPLATES = {"TPL-09", "TPL-10"}
EXPECTED_SCALAR_FIELDS = 12
PREDICTION_FREEZE_STATUS = (
    "FROZEN_BEFORE_TEST_GROUND_TRUTH_OPEN"
)


# ============================================================
# FILE, HASH, AND IMMUTABLE-WRITE HELPERS
# ============================================================

def require_file(path: Path, label: str) -> None:
    if not path.is_file():
        raise FileNotFoundError(f"{label} tidak ditemukan: {path}")
    if path.stat().st_size <= 0:
        raise RuntimeError(f"{label} kosong: {path}")


def load_json(path: Path) -> dict:
    require_file(path, "JSON artifact")
    with path.open("r", encoding="utf-8") as file_handle:
        value = json.load(file_handle)
    if not isinstance(value, dict):
        raise TypeError(f"Root JSON bukan object: {path}")
    return value


def canonical_json(value: object) -> str:
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    )


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as file_handle:
        for chunk in iter(lambda: file_handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def atomic_write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(
        f".{path.name}.{os.getpid()}.tmp"
    )
    try:
        temporary_path.write_text(text, encoding="utf-8")
        os.replace(temporary_path, path)
    finally:
        if temporary_path.exists():
            temporary_path.unlink()


def atomic_write_json(path: Path, value: dict) -> None:
    atomic_write_text(
        path,
        json.dumps(
            value,
            indent=2,
            ensure_ascii=False,
            default=str,
        )
        + "\n",
    )


def save_immutable_json(path: Path, value: dict) -> str:
    if path.exists():
        existing = load_json(path)
        if canonical_json(existing) != canonical_json(value):
            raise RuntimeError(
                f"Checkpoint sudah ada tetapi berbeda: {path}"
            )
        return "RECOVERED"

    atomic_write_json(path, value)
    return "CREATED"


def save_immutable_text(path: Path, text: str) -> str:
    if path.exists():
        existing = path.read_text(encoding="utf-8")
        if existing != text:
            raise RuntimeError(
                f"Checkpoint sudah ada tetapi berbeda: {path}"
            )
        return "RECOVERED"

    atomic_write_text(path, text)
    return "CREATED"


# ============================================================
# LOAD ONLY THE FROZEN PARSER DEFINITION PREFIX
#
# The frozen source is the complete historical Cell 10C. Importing that file
# directly would also execute its old development runner. Instead, this loader
# verifies the complete frozen file checksum, parses it, and executes exactly
# the source prefix through parse_document(). No parser rule is copied,
# rewritten, patched, or selected by test template.
# ============================================================

def load_frozen_parser(
    parser_source_path: Path,
    expected_source_sha256: str,
) -> tuple[dict, dict]:
    require_file(parser_source_path, "Frozen parser source")

    actual_source_sha256 = sha256_file(parser_source_path)
    if actual_source_sha256 != expected_source_sha256:
        raise RuntimeError(
            "Checksum frozen parser source tidak cocok. "
            f"Expected={expected_source_sha256}, "
            f"actual={actual_source_sha256}"
        )

    source_text = parser_source_path.read_text(encoding="utf-8")
    source_tree = ast.parse(
        source_text,
        filename=str(parser_source_path),
        mode="exec",
    )

    entrypoints = [
        node
        for node in source_tree.body
        if isinstance(node, ast.FunctionDef)
        and node.name == "parse_document"
    ]
    if len(entrypoints) != 1:
        raise RuntimeError(
            "Entrypoint parse_document pada frozen source tidak unik: "
            f"ditemukan {len(entrypoints)}."
        )

    entrypoint = entrypoints[0]
    entrypoint_end_line = int(entrypoint.end_lineno or entrypoint.lineno)
    prefix_nodes = [
        node
        for node in source_tree.body
        if int(node.end_lineno or node.lineno) <= entrypoint_end_line
    ]

    allowed_top_level_types = (
        ast.Import,
        ast.ImportFrom,
        ast.Assign,
        ast.AnnAssign,
        ast.FunctionDef,
    )
    disallowed_nodes = [
        {
            "node_type": type(node).__name__,
            "line": int(node.lineno),
        }
        for node in prefix_nodes
        if not isinstance(node, allowed_top_level_types)
    ]
    if disallowed_nodes:
        raise RuntimeError(
            "Frozen parser definition prefix mengandung top-level statement "
            f"yang tidak diizinkan: {disallowed_nodes}"
        )

    prefix_tree = ast.Module(
        body=prefix_nodes,
        type_ignores=source_tree.type_ignores,
    )
    ast.fix_missing_locations(prefix_tree)

    parser_namespace = {
        "__name__": "frozen_invoice_parser_v1_0_2",
        "__file__": str(parser_source_path),
    }
    compiled_prefix = compile(
        prefix_tree,
        filename=str(parser_source_path),
        mode="exec",
    )
    exec(compiled_prefix, parser_namespace)

    parse_document = parser_namespace.get("parse_document")
    if not callable(parse_document):
        raise RuntimeError(
            "Frozen parser entrypoint parse_document tidak callable."
        )

    loader_audit = {
        "method": "AST_PREFIX_THROUGH_PARSE_DOCUMENT",
        "complete_source_verified_before_load": True,
        "complete_source_sha256": actual_source_sha256,
        "entrypoint": "parse_document",
        "entrypoint_end_line": entrypoint_end_line,
        "loaded_top_level_nodes": len(prefix_nodes),
        "development_runner_executed": False,
        "parser_rule_modifications": 0,
    }
    return parser_namespace, loader_audit


def parser_configuration_from_namespace(namespace: dict) -> dict:
    return {
        "parser_id": namespace["PARSER_ID"],
        "parser_version": namespace["PARSER_VERSION"],
        "supported_currencies": sorted(
            namespace["SUPPORTED_CURRENCIES"]
        ),
        "label_aliases": {
            key: sorted(value)
            for key, value in sorted(
                namespace["LABEL_ALIASES"].items()
            )
        },
        "month_dictionary": dict(
            sorted(namespace["MONTHS"].items())
        ),
        "table_row_tolerance_points": 2.8,
        "party_legal_suffixes": sorted(
            namespace["LEGAL_ENTITY_SUFFIXES"]
        ),
        "party_suffix_geometry": {
            "same_row_tolerance_points": namespace[
                "PARTY_SUFFIX_SAME_ROW_TOLERANCE_POINTS"
            ],
            "next_line_gap_points": namespace[
                "PARTY_SUFFIX_NEXT_LINE_GAP_POINTS"
            ],
            "left_alignment_tolerance_points": namespace[
                "PARTY_SUFFIX_LEFT_ALIGNMENT_TOLERANCE_POINTS"
            ],
            "same_row_gap_points": namespace[
                "PARTY_SUFFIX_SAME_ROW_GAP_POINTS"
            ],
        },
        "ground_truth_as_prediction_input": False,
        "template_specific_branching": False,
    }


# ============================================================
# PREFLIGHT — VERIFY THE FROZEN BASELINE AND TEXT LAYERS
# ============================================================

for required_path, label in (
    (TEST_BASELINE_POINTER_PATH, "Test baseline pointer"),
    (TEXT_LAYER_MANIFEST_PATH, "Test text-layer manifest"),
):
    require_file(required_path, label)

baseline_pointer = load_json(TEST_BASELINE_POINTER_PATH)
text_layer_manifest = load_json(TEXT_LAYER_MANIFEST_PATH)

freeze_manifest_path = Path(
    str(baseline_pointer.get("development_freeze_manifest_path", ""))
)
validation_decision_path = Path(
    str(baseline_pointer.get("validation_decision_manifest_path", ""))
)
parser_source_path = Path(
    str(baseline_pointer.get("parser_source_path", ""))
)

require_file(freeze_manifest_path, "Frozen baseline manifest")
require_file(validation_decision_path, "Frozen validation decision")
require_file(parser_source_path, "Frozen parser source")

freeze_manifest = load_json(freeze_manifest_path)
validation_decision = load_json(validation_decision_path)
freeze_parser = freeze_manifest.get("parser", {})
decision_baseline = validation_decision.get("baseline", {})
decision_validation = validation_decision.get("validation_evidence", {})
decision_split_policy = validation_decision.get("split_policy", {})
decision_integrity = validation_decision.get("integrity", {})
text_scope = text_layer_manifest.get("scope", {})
text_integrity = text_layer_manifest.get("integrity", {})
text_baseline = text_layer_manifest.get("baseline", {})

source_checksums_before = {
    "test_baseline_pointer": sha256_file(
        TEST_BASELINE_POINTER_PATH
    ),
    "freeze_manifest": sha256_file(freeze_manifest_path),
    "validation_decision": sha256_file(validation_decision_path),
    "parser_source": sha256_file(parser_source_path),
    "text_layer_manifest": sha256_file(TEXT_LAYER_MANIFEST_PATH),
}

preflight_values = [
    (
        "baseline_pointer_status",
        "ACTIVE_FOR_TEST_PREFLIGHT",
        baseline_pointer.get("status"),
    ),
    (
        "baseline_id",
        EXPECTED_BASELINE_ID,
        baseline_pointer.get("baseline_id"),
    ),
    (
        "parser_id",
        EXPECTED_PARSER_ID,
        baseline_pointer.get("parser_id"),
    ),
    (
        "parser_version",
        EXPECTED_PARSER_VERSION,
        baseline_pointer.get("parser_version"),
    ),
    (
        "parser_signature",
        EXPECTED_PARSER_SIGNATURE,
        baseline_pointer.get("parser_signature_sha256"),
    ),
    (
        "next_allowed_stage",
        "TEST_COHORT_PREFLIGHT",
        baseline_pointer.get("next_allowed_stage"),
    ),
    ("next_allowed_split", "test", baseline_pointer.get("next_allowed_split")),
    (
        "parser_mutation_allowed",
        False,
        baseline_pointer.get("parser_mutation_allowed"),
    ),
    (
        "test_ground_truth_locked",
        True,
        baseline_pointer.get("test_ground_truth_remains_locked"),
    ),
    (
        "test_predictions_must_be_frozen",
        True,
        baseline_pointer.get(
            "test_predictions_must_be_frozen_before_ground_truth_open"
        ),
    ),
    (
        "freeze_status",
        "FROZEN_FOR_VALIDATION",
        freeze_manifest.get("status"),
    ),
    (
        "freeze_checksum",
        baseline_pointer.get("development_freeze_manifest_sha256"),
        source_checksums_before["freeze_manifest"],
    ),
    (
        "parser_source_path",
        str(parser_source_path),
        str(freeze_parser.get("source_path")),
    ),
    (
        "parser_source_checksum_pointer",
        baseline_pointer.get("parser_source_sha256"),
        source_checksums_before["parser_source"],
    ),
    (
        "parser_source_checksum_freeze",
        freeze_parser.get("source_sha256"),
        source_checksums_before["parser_source"],
    ),
    (
        "validation_decision_status",
        "FROZEN_FOR_TEST_PREFLIGHT",
        validation_decision.get("status"),
    ),
    (
        "validation_decision",
        "VALIDATION_PASSED_BASELINE_ACCEPTED",
        validation_decision.get("decision"),
    ),
    (
        "validation_decision_checksum",
        baseline_pointer.get("validation_decision_manifest_sha256"),
        source_checksums_before["validation_decision"],
    ),
    (
        "validation_evaluation_status",
        "PASSED",
        decision_validation.get("evaluation_status"),
    ),
    (
        "validation_result_fingerprint",
        baseline_pointer.get("validation_result_sha256"),
        decision_validation.get("validation_result_sha256"),
    ),
    (
        "decision_parser_signature",
        EXPECTED_PARSER_SIGNATURE,
        decision_baseline.get("parser_signature_sha256"),
    ),
    (
        "decision_parser_source_checksum",
        source_checksums_before["parser_source"],
        decision_baseline.get("parser_source_sha256"),
    ),
    (
        "decision_freeze_checksum",
        source_checksums_before["freeze_manifest"],
        decision_baseline.get("development_freeze_manifest_sha256"),
    ),
    (
        "decision_test_gt_locked",
        True,
        decision_split_policy.get("test_ground_truth_remains_locked"),
    ),
    (
        "decision_test_gt_opened",
        0,
        decision_integrity.get("test_ground_truth_opened"),
    ),
    (
        "template_specific_branching",
        False,
        freeze_parser.get("template_specific_branching"),
    ),
    (
        "ground_truth_as_prediction_input",
        False,
        freeze_parser.get("ground_truth_as_prediction_input"),
    ),
    (
        "text_layer_status",
        "PASSED",
        text_layer_manifest.get("status"),
    ),
    (
        "text_layer_stage",
        "TEST_UNIFIED_DOCUMENT_TEXT_LAYER",
        text_layer_manifest.get("stage"),
    ),
    ("text_layer_split", "test", text_scope.get("split")),
    (
        "text_layer_documents",
        EXPECTED_DOCUMENTS,
        text_scope.get("documents"),
    ),
    (
        "text_layer_templates",
        sorted(EXPECTED_TEMPLATES),
        sorted(text_scope.get("templates", [])),
    ),
    (
        "text_layer_baseline_id",
        EXPECTED_BASELINE_ID,
        text_baseline.get("baseline_id"),
    ),
    (
        "text_layer_parser_signature",
        EXPECTED_PARSER_SIGNATURE,
        text_baseline.get("parser_signature_sha256"),
    ),
    (
        "text_layer_parser_source_checksum",
        source_checksums_before["parser_source"],
        text_baseline.get("parser_source_sha256"),
    ),
    (
        "text_layer_freeze_checksum",
        source_checksums_before["freeze_manifest"],
        text_baseline.get("development_freeze_manifest_sha256"),
    ),
    (
        "text_layer_validation_decision_checksum",
        source_checksums_before["validation_decision"],
        text_baseline.get("validation_decision_manifest_sha256"),
    ),
    (
        "test_ground_truth_loaded",
        False,
        text_integrity.get("test_ground_truth_loaded"),
    ),
    (
        "canonical_payload_loaded",
        False,
        text_integrity.get("canonical_payload_loaded"),
    ),
    (
        "test_ground_truth_opened",
        0,
        text_integrity.get("test_ground_truth_opened"),
    ),
    (
        "validation_ground_truth_reopened",
        0,
        text_integrity.get("validation_ground_truth_reopened"),
    ),
    (
        "test_pdf_documents_read",
        EXPECTED_DOCUMENTS,
        text_integrity.get("test_pdf_documents_read"),
    ),
    (
        "text_layer_parser_modifications",
        0,
        text_integrity.get("parser_modifications"),
    ),
]

preflight_controls = [
    {
        "control": name,
        "expected": expected,
        "actual": actual,
        "status": "VALID" if expected == actual else "INVALID",
    }
    for name, expected, actual in preflight_values
]
display(pd.DataFrame(preflight_controls))

invalid_preflight = [
    row["control"]
    for row in preflight_controls
    if row["status"] != "VALID"
]
if invalid_preflight:
    raise RuntimeError(
        "CELL 12C PREFLIGHT FAILED. "
        f"Kontrol tidak valid: {invalid_preflight}"
    )


# ============================================================
# LOAD AND VERIFY THE FROZEN PARSER ENTRYPOINT
# ============================================================

parser_namespace, parser_loader_audit = load_frozen_parser(
    parser_source_path,
    baseline_pointer["parser_source_sha256"],
)
parse_document = parser_namespace["parse_document"]

parser_configuration = parser_configuration_from_namespace(
    parser_namespace
)
parser_signature = hashlib.sha256(
    canonical_json(parser_configuration).encode("utf-8")
).hexdigest()
expected_parser_signature = str(
    freeze_parser.get("parser_signature_sha256", "")
)

parser_load_values = [
    (
        "runtime_parser_id",
        EXPECTED_PARSER_ID,
        parser_namespace.get("PARSER_ID"),
    ),
    (
        "runtime_parser_version",
        EXPECTED_PARSER_VERSION,
        parser_namespace.get("PARSER_VERSION"),
    ),
    (
        "runtime_parser_signature",
        expected_parser_signature,
        parser_signature,
    ),
    (
        "parse_document_callable",
        True,
        callable(parse_document),
    ),
    (
        "development_runner_executed",
        False,
        parser_loader_audit["development_runner_executed"],
    ),
    (
        "parser_rule_modifications",
        0,
        parser_loader_audit["parser_rule_modifications"],
    ),
]

parser_load_controls = [
    {
        "control": name,
        "expected": expected,
        "actual": actual,
        "status": "VALID" if expected == actual else "INVALID",
    }
    for name, expected, actual in parser_load_values
]
print("\nFROZEN PARSER LOAD CONTROLS")
display(pd.DataFrame(parser_load_controls))

invalid_parser_load = [
    row["control"]
    for row in parser_load_controls
    if row["status"] != "VALID"
]
if invalid_parser_load:
    raise RuntimeError(
        "CELL 12C FROZEN PARSER LOAD FAILED. "
        f"Kontrol tidak valid: {invalid_parser_load}"
    )


# ============================================================
# VERIFY ALL 40 IMMUTABLE TEST TEXT LAYERS
# ============================================================

text_layer_records = text_layer_manifest.get("records")
if not isinstance(text_layer_records, list):
    raise TypeError("records pada text-layer manifest bukan list.")

source_records = sorted(
    text_layer_records,
    key=lambda row: (
        int(row.get("sequence_number", 0)),
        str(row.get("document_id", "")),
    ),
)

source_errors = []
for record_index, source_record in enumerate(source_records, start=1):
    try:
        if not isinstance(source_record, dict):
            raise TypeError("Text-layer record bukan object.")

        document_id = str(source_record.get("document_id", ""))
        template_id = str(source_record.get("template_id", ""))
        text_layer_path = Path(
            str(source_record.get("text_layer_path", ""))
        )
        expected_checksum = str(
            source_record.get("text_layer_sha256", "")
        )

        if not document_id:
            raise RuntimeError("document_id kosong.")
        if template_id not in EXPECTED_TEMPLATES:
            raise RuntimeError(
                f"Template di luar test: {template_id}"
            )
        if not text_layer_path.resolve().is_relative_to(
            (TEST_ROOT / "text_layers").resolve()
        ):
            raise RuntimeError(
                f"Text layer berada di luar test root: {document_id}"
            )
        if "ground_truth" in {
            part.casefold() for part in text_layer_path.parts
        }:
            raise RuntimeError(
                f"Path ground truth dilarang: {document_id}"
            )

        require_file(text_layer_path, f"Text layer {document_id}")
        actual_checksum = sha256_file(text_layer_path)
        if actual_checksum != expected_checksum:
            raise RuntimeError(
                f"Checksum text layer berubah: {document_id}"
            )

        text_layer = load_json(text_layer_path)
        document = text_layer.get("document", {})
        integrity = text_layer.get("integrity", {})

        if text_layer.get("status") != "PASSED":
            raise RuntimeError("Status text layer bukan PASSED.")
        if document.get("document_id") != document_id:
            raise RuntimeError("Document ID text layer tidak cocok.")
        if document.get("template_id") != template_id:
            raise RuntimeError("Template ID text layer tidak cocok.")
        if document.get("split") != "test":
            raise RuntimeError("Text layer bukan test split.")
        if integrity.get("test_ground_truth_loaded") is not False:
            raise RuntimeError(
                "Text layer menandai test ground truth telah dibuka."
            )
        if integrity.get("test_ground_truth_opened") != 0:
            raise RuntimeError("Test ground truth telah dibuka.")
        if integrity.get("validation_ground_truth_reopened") != 0:
            raise RuntimeError("Validation ground truth dibuka kembali.")
        if integrity.get("test_pdf_read") is not True:
            raise RuntimeError("Test PDF belum ditandai telah dibaca.")

        source_checksums_before[
            f"text_layer:{document_id}"
        ] = actual_checksum

    except Exception as error:
        source_errors.append(
            {
                "record_index": record_index,
                "document_id": source_record.get("document_id")
                if isinstance(source_record, dict)
                else None,
                "error_type": type(error).__name__,
                "error": str(error)[:700],
            }
        )

source_gate_values = [
    ("text_layer_records", EXPECTED_DOCUMENTS, len(source_records)),
    (
        "unique_document_ids",
        EXPECTED_DOCUMENTS,
        len({row.get("document_id") for row in source_records}),
    ),
    (
        "test_templates",
        sorted(EXPECTED_TEMPLATES),
        sorted({row.get("template_id") for row in source_records}),
    ),
    (
        "sequence_numbers",
        list(range(1, EXPECTED_DOCUMENTS + 1)),
        [int(row.get("sequence_number", 0)) for row in source_records],
    ),
    (
        "all_sources_are_test_text_layers",
        True,
        all(
            Path(row["text_layer_path"]).resolve().is_relative_to(
                (TEST_ROOT / "text_layers").resolve()
            )
            for row in source_records
        ),
    ),
    ("source_errors", 0, len(source_errors)),
    ("test_ground_truth_opened", 0, 0),
]

source_gate_controls = [
    {
        "control": name,
        "expected": expected,
        "actual": actual,
        "status": "VALID" if expected == actual else "INVALID",
    }
    for name, expected, actual in source_gate_values
]

print("\nTEXT-LAYER SOURCE CONTROLS")
display(pd.DataFrame(source_gate_controls))
if source_errors:
    display(pd.DataFrame(source_errors))

invalid_source_gates = [
    row["control"]
    for row in source_gate_controls
    if row["status"] != "VALID"
]
if invalid_source_gates:
    raise RuntimeError(
        "CELL 12C TEXT-LAYER SOURCE GATE FAILED. "
        f"Kontrol tidak valid: {invalid_source_gates}"
    )


# ============================================================
# BLIND TEST PREDICTION
# ============================================================

runtime_records = []
manifest_records = []
processing_errors = []
newly_created = 0
recovered = 0

print()
print("=" * 92)
print(
    f"CELL 12C — {EXPECTED_PARSER_ID} — VERSION "
    f"{EXPECTED_PARSER_VERSION} — BLIND TEST"
)
print(f"Prediction root: {PREDICTION_ROOT}")
print("Parser: FROZEN | Test PDFs: AUTHORIZED | Test GT: CLOSED")
print("=" * 92)
print(f"Menjalankan frozen parser untuk {len(source_records)} dokumen...\n")

for position, source_record in enumerate(source_records, start=1):
    document_id = str(source_record["document_id"])
    template_id = str(source_record["template_id"])
    text_layer_path = Path(source_record["text_layer_path"])
    prediction_path = (
        PREDICTION_ROOT
        / template_id
        / f"{document_id}_prediction.json"
    )

    try:
        text_layer = load_json(text_layer_path)
        document = text_layer["document"]

        # Only extracted lines enter the parser. Cohort metadata such as
        # template, expected item count, canonical payload, and ground truth
        # is not available to the parser entrypoint.
        parser_input = {
            "lines": text_layer.get("lines", []),
        }
        parsed = parse_document(parser_input)

        if not isinstance(parsed, dict):
            raise TypeError("Frozen parser output bukan object.")

        scalars = parsed.get("scalar_fields")
        items = parsed.get("items")
        diagnostics = parsed.get("diagnostics")

        if not isinstance(scalars, dict):
            raise TypeError("scalar_fields bukan object.")
        if not isinstance(items, list):
            raise TypeError("items bukan list.")
        if not isinstance(diagnostics, dict):
            raise TypeError("diagnostics bukan object.")
        if len(scalars) != EXPECTED_SCALAR_FIELDS:
            raise RuntimeError(
                f"Jumlah scalar field tidak valid: {len(scalars)}"
            )

        missing_required = list(
            diagnostics.get("missing_required_scalar_fields", [])
        )
        table_diagnostics = diagnostics.get("table", {})
        financial_check = diagnostics.get("financial_equation", {})

        populated_scalar_count = sum(
            isinstance(prediction, dict)
            and prediction.get("normalized_value") is not None
            for prediction in scalars.values()
        )
        party_suffix_fields = sorted(
            field_name
            for field_name, field_value in scalars.items()
            if isinstance(field_value, dict)
            and str(field_value.get("method", "")).endswith(
                ":LEGAL_SUFFIX_CONTINUATION"
            )
        )

        prediction_artifact = {
            "schema_version": "1.0.0",
            "status": "EXECUTED",
            "prediction_freeze_status": PREDICTION_FREEZE_STATUS,
            "quality_status": (
                "PENDING_TEST_GROUND_TRUTH_EVALUATION"
            ),
            "parser": {
                "baseline_id": EXPECTED_BASELINE_ID,
                "parser_id": EXPECTED_PARSER_ID,
                "parser_version": EXPECTED_PARSER_VERSION,
                "parser_signature_sha256": parser_signature,
                "parser_source_path": str(parser_source_path),
                "parser_source_sha256": source_checksums_before[
                    "parser_source"
                ],
                "freeze_status": freeze_manifest["status"],
                "approach": "DETERMINISTIC_LABEL_AND_GEOMETRY_RULES",
                "template_specific_branching": False,
                "ground_truth_used_as_prediction_input": False,
                "parser_input_fields": ["lines"],
            },
            "document": {
                "canonical_invoice_id": document.get(
                    "canonical_invoice_id"
                ),
                "document_id": document_id,
                "template_id": template_id,
                "split": "test",
                "language": document.get("language"),
            },
            "predictions": {
                "scalar_fields": scalars,
                "items": items,
            },
            "diagnostics": diagnostics,
            "source": {
                "text_layer_path": str(text_layer_path),
                "text_layer_sha256": source_record[
                    "text_layer_sha256"
                ],
                "route_id": source_record.get("route_id"),
                "engine": source_record.get("engine"),
            },
            "integrity": {
                "test_ground_truth_loaded": False,
                "canonical_payload_loaded": False,
                "ground_truth_used_as_prediction_input": False,
                "validation_ground_truth_reopened": 0,
                "test_ground_truth_opened": 0,
                "parser_modifications": 0,
                "dataset_modifications": 0,
                "source_modifications": 0,
            },
        }

        checkpoint_action = save_immutable_json(
            prediction_path,
            prediction_artifact,
        )
        if checkpoint_action == "CREATED":
            newly_created += 1
            execution = "NEW"
        else:
            recovered += 1
            execution = "RECOVERED"

        persisted = load_json(prediction_path)
        if canonical_json(persisted) != canonical_json(
            prediction_artifact
        ):
            raise RuntimeError(
                f"Prediction berbeda setelah penulisan: {document_id}"
            )

        manifest_record = {
            "sequence_number": int(
                source_record.get("sequence_number", position)
            ),
            "document_id": document_id,
            "template_id": template_id,
            "language": document.get("language"),
            "route_id": source_record.get("route_id"),
            "populated_scalar_fields": populated_scalar_count,
            "missing_required_scalar_fields": missing_required,
            "party_suffix_continuation_fields": party_suffix_fields,
            "party_suffix_continuation_count": len(
                party_suffix_fields
            ),
            "parsed_item_count": len(items),
            "table_detected": bool(
                table_diagnostics.get("table_detected", False)
            ),
            "financial_equation_executed": bool(
                financial_check.get("executed", False)
            ),
            "financial_equation_passed": bool(
                financial_check.get("passed", False)
            ),
            "prediction_path": str(prediction_path),
            "prediction_sha256": sha256_file(prediction_path),
            "text_layer_path": str(text_layer_path),
            "text_layer_sha256": source_record[
                "text_layer_sha256"
            ],
            "status": "EXECUTED",
            "prediction_freeze_status": PREDICTION_FREEZE_STATUS,
            "quality_status": (
                "PENDING_TEST_GROUND_TRUTH_EVALUATION"
            ),
        }
        manifest_records.append(manifest_record)
        runtime_records.append(
            {**manifest_record, "execution": execution}
        )

        print(
            f"[{position:02d}/{len(source_records):02d}] "
            f"{document_id} | scalars={populated_scalar_count}/12 | "
            f"items={len(items)} | "
            f"table={manifest_record['table_detected']} | "
            f"financial={manifest_record['financial_equation_passed']} | "
            f"{execution}"
        )

    except Exception as error:
        processing_errors.append(
            {
                "sequence_number": source_record.get(
                    "sequence_number", position
                ),
                "document_id": document_id,
                "template_id": template_id,
                "error_type": type(error).__name__,
                "error": str(error)[:900],
            }
        )
        print(
            f"[{position:02d}/{len(source_records):02d}] "
            f"{document_id} | ERROR: "
            f"{type(error).__name__}: {error}"
        )


# ============================================================
# EXECUTION GATES — DO NOT TURN DIAGNOSTICS INTO QUALITY CLAIMS
# ============================================================

runtime_table = pd.DataFrame(runtime_records)
manifest_table = pd.DataFrame(manifest_records)

if processing_errors:
    print("\nPROCESSING ERRORS")
    display(pd.DataFrame(processing_errors))

execution_values = [
    ("prediction_records", EXPECTED_DOCUMENTS, len(manifest_records)),
    (
        "prediction_files",
        EXPECTED_DOCUMENTS,
        sum(
            Path(row["prediction_path"]).is_file()
            for row in manifest_records
        ),
    ),
    (
        "unique_document_ids",
        EXPECTED_DOCUMENTS,
        len({row["document_id"] for row in manifest_records}),
    ),
    (
        "test_templates",
        sorted(EXPECTED_TEMPLATES),
        sorted({row["template_id"] for row in manifest_records}),
    ),
    (
        "frozen_prediction_records",
        EXPECTED_DOCUMENTS,
        sum(
            row["prediction_freeze_status"]
            == PREDICTION_FREEZE_STATUS
            for row in manifest_records
        ),
    ),
    ("processing_errors", 0, len(processing_errors)),
    (
        "parser_source_checksum",
        baseline_pointer["parser_source_sha256"],
        sha256_file(parser_source_path),
    ),
    ("parser_modifications", 0, 0),
    ("validation_ground_truth_reopened", 0, 0),
    ("test_ground_truth_opened", 0, 0),
]

execution_controls = [
    {
        "control": name,
        "expected": expected,
        "actual": actual,
        "status": "VALID" if expected == actual else "INVALID",
    }
    for name, expected, actual in execution_values
]

print("\nEXECUTION CONTROLS")
display(pd.DataFrame(execution_controls))

if not runtime_table.empty:
    display(
        runtime_table[
            [
                "sequence_number",
                "document_id",
                "template_id",
                "language",
                "populated_scalar_fields",
                "parsed_item_count",
                "table_detected",
                "financial_equation_executed",
                "financial_equation_passed",
                "party_suffix_continuation_count",
                "execution",
                "quality_status",
            ]
        ]
    )

invalid_execution_controls = [
    row["control"]
    for row in execution_controls
    if row["status"] != "VALID"
]
if invalid_execution_controls:
    raise RuntimeError(
        "CELL 12C EXECUTION FAILED. "
        f"Kontrol tidak valid: {invalid_execution_controls}"
    )


# These are observations, not acceptance gates. Their correctness can only be
# measured against the frozen test ground truth in Cell 12D.
technical_observations = [
    {
        "observation": "complete_required_scalars",
        "documents": sum(
            not row["missing_required_scalar_fields"]
            for row in manifest_records
        ),
        "status": "OBSERVED_NOT_EVALUATED",
    },
    {
        "observation": "table_detected",
        "documents": sum(
            row["table_detected"] for row in manifest_records
        ),
        "status": "OBSERVED_NOT_EVALUATED",
    },
    {
        "observation": "nonempty_items",
        "documents": sum(
            row["parsed_item_count"] > 0 for row in manifest_records
        ),
        "status": "OBSERVED_NOT_EVALUATED",
    },
    {
        "observation": "financial_equation_executed",
        "documents": sum(
            row["financial_equation_executed"]
            for row in manifest_records
        ),
        "status": "OBSERVED_NOT_EVALUATED",
    },
    {
        "observation": "financial_equation_passed",
        "documents": sum(
            row["financial_equation_passed"]
            for row in manifest_records
        ),
        "status": "OBSERVED_NOT_EVALUATED",
    },
]

print("\nTECHNICAL OBSERVATIONS — NOT QUALITY METRICS")
display(pd.DataFrame(technical_observations))


# ============================================================
# IMMUTABLE SUMMARY AND PREDICTION MANIFEST
# ============================================================

summary_columns = [
    "sequence_number",
    "document_id",
    "template_id",
    "language",
    "route_id",
    "populated_scalar_fields",
    "missing_required_scalar_fields",
    "party_suffix_continuation_count",
    "parsed_item_count",
    "table_detected",
    "financial_equation_executed",
    "financial_equation_passed",
    "prediction_path",
    "prediction_sha256",
    "text_layer_path",
    "text_layer_sha256",
    "status",
    "prediction_freeze_status",
    "quality_status",
]

summary_text = manifest_table[summary_columns].to_csv(
    index=False,
    lineterminator="\n",
)
summary_action = save_immutable_text(SUMMARY_PATH, summary_text)

prediction_manifest = {
    "schema_version": "1.0.0",
    "cell_version": CELL_VERSION,
    "status": "EXECUTED",
    "prediction_freeze_status": PREDICTION_FREEZE_STATUS,
    "quality_status": (
        "PENDING_TEST_GROUND_TRUTH_EVALUATION"
    ),
    "stage": "FROZEN_PARSER_BLIND_TEST_PREDICTION",
    "parser": {
        "baseline_id": EXPECTED_BASELINE_ID,
        **parser_configuration,
        "parser_signature_sha256": parser_signature,
        "parser_source_path": str(parser_source_path),
        "parser_source_sha256": source_checksums_before[
            "parser_source"
        ],
        "development_freeze_manifest_path": str(freeze_manifest_path),
        "development_freeze_manifest_sha256": source_checksums_before[
            "freeze_manifest"
        ],
        "validation_decision_manifest_path": str(
            validation_decision_path
        ),
        "validation_decision_manifest_sha256": source_checksums_before[
            "validation_decision"
        ],
        "validation_result_sha256": baseline_pointer[
            "validation_result_sha256"
        ],
        "loader_audit": parser_loader_audit,
    },
    "scope": {
        "split": "test",
        "documents": EXPECTED_DOCUMENTS,
        "templates": sorted(EXPECTED_TEMPLATES),
        "predictions_frozen": EXPECTED_DOCUMENTS,
        "validation_ground_truth_reopened": 0,
        "test_ground_truth_opened": 0,
    },
    "records": manifest_records,
    "execution_controls": execution_controls,
    "technical_observations": technical_observations,
    "artifacts": {
        "prediction_root": str(PREDICTION_ROOT),
        "summary_path": str(SUMMARY_PATH),
        "summary_sha256": sha256_file(SUMMARY_PATH),
    },
    "input_artifacts": {
        "test_baseline_pointer": {
            "path": str(TEST_BASELINE_POINTER_PATH),
            "sha256": source_checksums_before[
                "test_baseline_pointer"
            ],
        },
        "freeze_manifest": {
            "path": str(freeze_manifest_path),
            "sha256": source_checksums_before["freeze_manifest"],
        },
        "validation_decision": {
            "path": str(validation_decision_path),
            "sha256": source_checksums_before["validation_decision"],
        },
        "parser_source": {
            "path": str(parser_source_path),
            "sha256": source_checksums_before["parser_source"],
        },
        "text_layer_manifest": {
            "path": str(TEXT_LAYER_MANIFEST_PATH),
            "sha256": source_checksums_before[
                "text_layer_manifest"
            ],
        },
    },
    "integrity": {
        "test_ground_truth_loaded": False,
        "canonical_payload_loaded": False,
        "ground_truth_used_as_prediction_input": False,
        "test_predictions_frozen_before_ground_truth_open": True,
        "validation_ground_truth_reopened": 0,
        "test_ground_truth_opened": 0,
        "parser_modifications": 0,
        "dataset_modifications": 0,
        "source_modifications": 0,
    },
    "next_stage": {
        "cell": "CELL 12D",
        "action": "TEST_GROUND_TRUTH_EVALUATION_ONCE",
        "predictions_must_be_frozen_before_evaluation": True,
        "parser_changes_allowed_after_results": False,
        "test_ground_truth_may_open_once_in_next_cell": True,
    },
}

manifest_action = save_immutable_json(
    MANIFEST_PATH,
    prediction_manifest,
)

persisted_manifest = load_json(MANIFEST_PATH)
if canonical_json(persisted_manifest) != canonical_json(
    prediction_manifest
):
    raise RuntimeError("Manifest Cell 12C berbeda setelah penulisan.")


# ============================================================
# FINAL INPUT IMMUTABILITY VERIFICATION
# ============================================================

source_checksums_after = {
    "test_baseline_pointer": sha256_file(
        TEST_BASELINE_POINTER_PATH
    ),
    "freeze_manifest": sha256_file(freeze_manifest_path),
    "validation_decision": sha256_file(validation_decision_path),
    "parser_source": sha256_file(parser_source_path),
    "text_layer_manifest": sha256_file(TEXT_LAYER_MANIFEST_PATH),
}
for source_record in source_records:
    source_checksums_after[
        f"text_layer:{source_record['document_id']}"
    ] = sha256_file(Path(source_record["text_layer_path"]))

changed_sources = [
    name
    for name, checksum in source_checksums_before.items()
    if source_checksums_after.get(name) != checksum
]
if changed_sources:
    raise RuntimeError(
        f"Input berubah selama Cell 12C: {changed_sources}"
    )


template_counts = Counter(
    row["template_id"] for row in manifest_records
)
language_counts = Counter(
    row["language"] for row in manifest_records
)

print()
print(f"Cell version           : {CELL_VERSION}")
print(f"Baseline ID            : {EXPECTED_BASELINE_ID}")
print(f"Parser ID              : {EXPECTED_PARSER_ID}")
print(f"Parser version         : {EXPECTED_PARSER_VERSION}")
print(f"Parser signature       : {parser_signature}")
print(
    "Parser source SHA-256: "
    f"{source_checksums_before['parser_source']}"
)
print(f"Test documents   : {len(manifest_records)}")
print(f"Templates              : {dict(sorted(template_counts.items()))}")
print(f"Languages              : {dict(sorted(language_counts.items()))}")
print(f"New predictions        : {newly_created}")
print(f"Recovered predictions  : {recovered}")
print(f"Summary action         : {summary_action}")
print(f"Manifest action        : {manifest_action}")
print(f"Prediction root        : {PREDICTION_ROOT}")
print(f"Summary                : {SUMMARY_PATH}")
print(f"Manifest               : {MANIFEST_PATH}")
print(f"Manifest SHA-256       : {sha256_file(MANIFEST_PATH)}")
print(f"Prediction freeze      : {PREDICTION_FREEZE_STATUS}")
print("Validation GT reopened : 0")
print("Test GT opened         : 0")
print("Parser modifications   : 0")
print("Dataset modifications  : 0")
print("Source modifications   : 0")
print(
    "Quality status        : "
    "PENDING_TEST_GROUND_TRUTH_EVALUATION"
)
print()
print(
    "✅ CELL 12C PASSED — frozen parser v1.0.2 telah menghasilkan "
    "dan membekukan prediction checkpoint untuk 40 dokumen test "
    "tanpa membuka ground truth test. Lanjutkan ke Cell 12D untuk "
    "evaluasi test satu kali."
)


,control,expected,actual,status
0,baseline_pointer_status,ACTIVE_FOR_TEST_PREFLIGHT,ACTIVE_FOR_TEST_PREFLIGHT,VALID
1,baseline_id,RULE-BASED-INVOICE-PARSER-V1@1.0.2,RULE-BASED-INVOICE-PARSER-V1@1.0.2,VALID
2,parser_id,RULE-BASED-INVOICE-PARSER-V1,RULE-BASED-INVOICE-PARSER-V1,VALID
3,parser_version,1.0.2,1.0.2,VALID
4,parser_signature,ffe04f258fb3b1a6a55661698fa6c7dda29ddd7b714c2f...,ffe04f258fb3b1a6a55661698fa6c7dda29ddd7b714c2f...,VALID
5,next_allowed_stage,TEST_COHORT_PREFLIGHT,TEST_COHORT_PREFLIGHT,VALID
6,next_allowed_split,test,test,VALID
7,parser_mutation_allowed,False,False,VALID
8,test_ground_truth_locked,True,True,VALID
9,test_predictions_must_be_frozen,True,True,VALID



FROZEN PARSER LOAD CONTROLS


,control,expected,actual,status
0,runtime_parser_id,RULE-BASED-INVOICE-PARSER-V1,RULE-BASED-INVOICE-PARSER-V1,VALID
1,runtime_parser_version,1.0.2,1.0.2,VALID
2,runtime_parser_signature,ffe04f258fb3b1a6a55661698fa6c7dda29ddd7b714c2f...,ffe04f258fb3b1a6a55661698fa6c7dda29ddd7b714c2f...,VALID
3,parse_document_callable,True,True,VALID
4,development_runner_executed,False,False,VALID
5,parser_rule_modifications,0,0,VALID



TEXT-LAYER SOURCE CONTROLS


,control,expected,actual,status
0,text_layer_records,40,40,VALID
1,unique_document_ids,40,40,VALID
2,test_templates,"[TPL-09, TPL-10]","[TPL-09, TPL-10]",VALID
3,sequence_numbers,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",VALID
4,all_sources_are_test_text_layers,True,True,VALID
5,source_errors,0,0,VALID
6,test_ground_truth_opened,0,0,VALID


...

,control,expected,actual,status
0,prediction_records,40,40,VALID
1,prediction_files,40,40,VALID
2,unique_document_ids,40,40,VALID
3,test_templates,"[TPL-09, TPL-10]","[TPL-09, TPL-10]",VALID
4,frozen_prediction_records,40,40,VALID
5,processing_errors,0,0,VALID
6,parser_source_checksum,0a737f57a86df7d3e16eef6749e3c3a23e82fb51227d09...,0a737f57a86df7d3e16eef6749e3c3a23e82fb51227d09...,VALID
7,parser_modifications,0,0,VALID
8,validation_ground_truth_reopened,0,0,VALID
9,test_ground_truth_opened,0,0,VALID


,sequence_number,document_id,template_id,language,populated_scalar_fields,parsed_item_count,table_detected,financial_equation_executed,financial_equation_passed,party_suffix_continuation_count,execution,quality_status
0,1,INV-SYN-000161,TPL-09,id,12,5,True,True,True,0,NEW,PENDING_TEST_GROUND_TRUTH_EVALUATION
1,2,INV-SYN-000162,TPL-09,id,12,3,True,True,True,0,NEW,PENDING_TEST_GROUND_TRUTH_EVALUATION
2,3,INV-SYN-000163,TPL-09,id,12,8,True,True,True,0,NEW,PENDING_TEST_GROUND_TRUTH_EVALUATION
3,4,INV-SYN-000164,TPL-09,id,12,5,True,True,True,0,NEW,PENDING_TEST_GROUND_TRUTH_EVALUATION
4,5,INV-SYN-000165,TPL-09,id,12,7,True,True,True,0,NEW,PENDING_TEST_GROUND_TRUTH_EVALUATION
5,6,INV-SYN-000166,TPL-09,id,12,7,True,True,True,0,NEW,PENDING_TEST_GROUND_TRUTH_EVALUATION
6,7,INV-SYN-000167,TPL-09,id,12,4,True,True,True,0,NEW,PENDING_TEST_GROUND_TRUTH_EVALUATION
7,8,INV-SYN-000168,TPL-09,id,12,5,True,True,True,0,NEW,PENDING_TEST_GROUND_TRUTH_EVALUATION
8,9,INV-SYN-000169,TPL-09,id,12,3,True,True,True,0,NEW,PENDING_TEST_GROUND_TRUTH_EVALUATION
9,10,INV-SYN-000170,TPL-09,id,12,7,True,True,True,0,NEW,PENDING_TEST_GROUND_TRUTH_EVALUATION



TECHNICAL OBSERVATIONS — NOT QUALITY METRICS


,observation,documents,status
0,complete_required_scalars,30,OBSERVED_NOT_EVALUATED
1,table_detected,40,OBSERVED_NOT_EVALUATED
2,nonempty_items,40,OBSERVED_NOT_EVALUATED
3,financial_equation_executed,40,OBSERVED_NOT_EVALUATED
4,financial_equation_passed,40,OBSERVED_NOT_EVALUATED



Cell version           : 1.0.0
Baseline ID            : RULE-BASED-INVOICE-PARSER-V1@1.0.2
Parser ID              : RULE-BASED-INVOICE-PARSER-V1
Parser version         : 1.0.2
Parser signature       : ffe04f258fb3b1a6a55661698fa6c7dda29ddd7b714c2f210b3e6fc1fcd464b3
Parser source SHA-256: 0a737f57a86df7d3e16eef6749e3c3a23e82fb51227d099845908e89c8be642c
Test documents   : 40
Templates              : {'TPL-09': 20, 'TPL-10': 20}
Languages              : {'en': 20, 'id': 20}
New predictions        : 40
Recovered predictions  : 0
Summary action         : CREATED
Manifest action        : CREATED
Prediction root        : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/field_extraction/test/rule_based_baseline_v1_0_2/predictions
Summary                : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/field_extraction/test/rule_based_baseline_v1_0_2/test_prediction_summary.csv
Manifest    

In [8]:
from __future__ import annotations

import hashlib
import json
import os
import re
import unicodedata
from decimal import Decimal, InvalidOperation
from functools import lru_cache
from pathlib import Path

import pandas as pd
from IPython.display import display


# ============================================================
# CELL 12D — ONE-TIME TEST GROUND-TRUTH EVALUATION
#             (FROZEN PREDICTIONS; EVALUATION-ONLY ACCESS)
# ============================================================

CELL_VERSION = "1.0.0"
EVALUATOR_ID = "INVOICE-FIELD-EVALUATOR-V1"
EVALUATOR_VERSION = "1.0.2"

DATA_ROOT = Path("/content/drive/MyDrive/InvoiceFlow-AI-Data")
BUILD_ROOT = (
    DATA_ROOT
    / "interim"
    / "synthetic_v1_build"
    / "20260904T150025Z"
)
FIELD_ROOT = BUILD_ROOT / "ocr_benchmark" / "field_extraction"
RENDERED_DATASET_ROOT = BUILD_ROOT / "rendered_dataset"
TEST_GROUND_TRUTH_ROOT = (
    RENDERED_DATASET_ROOT / "ground_truth" / "test"
)
RENDER_INDEX_PATH = BUILD_ROOT / "manifests" / "batch_render_index.jsonl"
CONTRACT_PATH = FIELD_ROOT / "field_extraction_contract_v1.json"

TEST_ROOT = (
    FIELD_ROOT / "test" / "rule_based_baseline_v1_0_2"
)
BASELINE_POINTER_PATH = FIELD_ROOT / "test_baseline_pointer.json"
PREDICTION_MANIFEST_PATH = (
    TEST_ROOT / "test_prediction_manifest.json"
)

EVALUATION_ROOT = (
    TEST_ROOT / "evaluations" / "test_eval_v1_0_2"
)
DOCUMENT_EVALUATION_ROOT = EVALUATION_ROOT / "documents"
DOCUMENT_SUMMARY_PATH = EVALUATION_ROOT / "document_summary.csv"
FIELD_SUMMARY_PATH = EVALUATION_ROOT / "field_summary.csv"
TEMPLATE_SUMMARY_PATH = EVALUATION_ROOT / "template_summary.csv"
MISMATCH_DETAIL_PATH = EVALUATION_ROOT / "mismatch_details.csv"
EVALUATION_MANIFEST_PATH = (
    EVALUATION_ROOT / "test_evaluation_manifest.json"
)

EXPECTED_BASELINE_ID = "RULE-BASED-INVOICE-PARSER-V1@1.0.2"
EXPECTED_PARSER_ID = "RULE-BASED-INVOICE-PARSER-V1"
EXPECTED_PARSER_VERSION = "1.0.2"
EXPECTED_PARSER_SIGNATURE = (
    "ffe04f258fb3b1a6a55661698fa6c7dda29ddd7b714c2f210b3e6fc1fcd464b3"
)
EXPECTED_DOCUMENTS = 40
EXPECTED_RENDER_RECORDS = 200
EXPECTED_TEMPLATES = {"TPL-09", "TPL-10"}
EXPECTED_SCALAR_FIELDS = 12
EXPECTED_ITEM_FIELDS = 4

ITEM_FIELD_NAMES = [
    "description",
    "quantity",
    "unit_price",
    "line_total",
]

SCALAR_PATHS = {
    "invoice_number": ("invoice_number",),
    "invoice_date": ("invoice_date",),
    "due_date": ("due_date",),
    "currency": ("currency",),
    "vendor.name": ("vendor", "name"),
    "vendor.tax_identifier": ("vendor", "tax_identifier"),
    "buyer.name": ("buyer", "name"),
    "buyer.tax_identifier": ("buyer", "tax_identifier"),
    "financials.subtotal": ("financials", "subtotal"),
    "financials.tax": ("financials", "tax"),
    "financials.discount": ("financials", "discount"),
    "financials.total": ("financials", "total"),
}

MONEY_FIELDS = {
    "financials.subtotal",
    "financials.tax",
    "financials.discount",
    "financials.total",
    "items[].unit_price",
    "items[].line_total",
}

IDENTIFIER_FIELDS = {
    "invoice_number",
    "currency",
    "vendor.tax_identifier",
    "buyer.tax_identifier",
}

SHA256_PATTERN = re.compile(r"^[0-9a-f]{64}$", re.IGNORECASE)


# ============================================================
# FILE, HASH, AND IMMUTABLE-WRITE HELPERS
# ============================================================

def require_file(path: Path, label: str) -> None:
    if not path.is_file():
        raise FileNotFoundError(f"{label} tidak ditemukan: {path}")
    if path.stat().st_size <= 0:
        raise RuntimeError(f"{label} kosong: {path}")


def require_directory(path: Path, label: str) -> None:
    if not path.is_dir():
        raise FileNotFoundError(f"{label} tidak ditemukan: {path}")


def load_json(path: Path) -> dict:
    require_file(path, "JSON artifact")
    with path.open("r", encoding="utf-8") as file_handle:
        value = json.load(file_handle)
    if not isinstance(value, dict):
        raise TypeError(f"Root JSON bukan object: {path}")
    return value


def load_jsonl(path: Path) -> list[dict]:
    require_file(path, "JSONL artifact")
    records = []
    with path.open("r", encoding="utf-8") as file_handle:
        for line_number, line in enumerate(file_handle, start=1):
            if not line.strip():
                continue
            value = json.loads(line)
            if not isinstance(value, dict):
                raise TypeError(
                    f"Record JSONL baris {line_number} bukan object: {path}"
                )
            records.append(value)
    return records


def canonical_json(value: object) -> str:
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    )


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as file_handle:
        for chunk in iter(lambda: file_handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def atomic_write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(
        f".{path.name}.{os.getpid()}.tmp"
    )
    try:
        temporary_path.write_text(text, encoding="utf-8")
        os.replace(temporary_path, path)
    finally:
        if temporary_path.exists():
            temporary_path.unlink()


def atomic_write_json(path: Path, value: dict) -> None:
    atomic_write_text(
        path,
        json.dumps(
            value,
            indent=2,
            ensure_ascii=False,
            default=str,
        )
        + "\n",
    )


def save_immutable_json(path: Path, value: dict) -> str:
    if path.exists():
        existing = load_json(path)
        if canonical_json(existing) != canonical_json(value):
            raise RuntimeError(
                f"Checkpoint sudah ada tetapi berbeda: {path}"
            )
        return "RECOVERED"
    atomic_write_json(path, value)
    return "CREATED"


def save_csv_checkpoint(path: Path, table: pd.DataFrame) -> str:
    text = table.to_csv(index=False, lineterminator="\n")
    if path.exists():
        if path.read_text(encoding="utf-8") != text:
            raise RuntimeError(
                f"CSV checkpoint sudah ada tetapi berbeda: {path}"
            )
        return "RECOVERED"
    atomic_write_text(path, text)
    return "CREATED"


def nested_value(record: dict, path: tuple[str, ...]):
    value = record
    for key in path:
        if not isinstance(value, dict) or key not in value:
            return None
        value = value[key]
    return value


# ============================================================
# TYPE-AWARE NORMALIZATION — IDENTICAL TO CELL 10D
# ============================================================

def normalize_spaces(value) -> str:
    if value is None:
        return ""
    text = unicodedata.normalize("NFKC", str(value))
    return re.sub(r"\s+", " ", text).strip()


def decimal_to_string(value: Decimal) -> str:
    if value == value.to_integral():
        return str(value.quantize(Decimal("1")))
    text = format(value.normalize(), "f")
    return text.rstrip("0").rstrip(".")


def normalize_decimal(value) -> str:
    text = normalize_spaces(value)
    if not text:
        return ""
    try:
        return decimal_to_string(Decimal(text))
    except InvalidOperation:
        return text


def normalize_field_value(field_name: str, value) -> str:
    text = normalize_spaces(value)
    if field_name in MONEY_FIELDS or field_name == "items[].quantity":
        return normalize_decimal(text)
    if field_name in IDENTIFIER_FIELDS:
        return text.upper()
    return text


# ============================================================
# EDIT-DISTANCE METRICS — IDENTICAL TO CELL 10D
# ============================================================

def levenshtein_distance(reference, hypothesis) -> int:
    reference = list(reference)
    hypothesis = list(hypothesis)

    if len(reference) < len(hypothesis):
        reference, hypothesis = hypothesis, reference

    previous = list(range(len(hypothesis) + 1))
    for reference_index, reference_value in enumerate(reference, start=1):
        current = [reference_index]
        for hypothesis_index, hypothesis_value in enumerate(
            hypothesis,
            start=1,
        ):
            insertion = current[hypothesis_index - 1] + 1
            deletion = previous[hypothesis_index] + 1
            substitution = (
                previous[hypothesis_index - 1]
                + int(reference_value != hypothesis_value)
            )
            current.append(min(insertion, deletion, substitution))
        previous = current
    return previous[-1]


def safe_ratio(numerator: int | float, denominator: int | float) -> float:
    if denominator == 0:
        return 1.0 if numerator == 0 else 0.0
    return float(numerator) / float(denominator)


def safe_error_rate(
    errors: int | float,
    reference_units: int | float,
) -> float:
    if reference_units == 0:
        return 0.0 if errors == 0 else 1.0
    return float(errors) / float(reference_units)


def harmonic_mean(precision: float, recall: float) -> float:
    if precision + recall == 0:
        return 0.0
    return 2.0 * precision * recall / (precision + recall)


def string_similarity(first: str, second: str) -> float:
    first = normalize_spaces(first).casefold()
    second = normalize_spaces(second).casefold()
    denominator = max(len(first), len(second), 1)
    return 1.0 - levenshtein_distance(first, second) / denominator


# ============================================================
# GROUND-TRUTH ACCESS PLAN FROM THE FROZEN RENDER INDEX
# This section resolves paths but does not load JSON payloads.
# ============================================================

def build_render_record_map(records: list[dict]) -> dict[str, dict]:
    mapped = {}
    for record in records:
        document_id = str(record.get("document_id", ""))
        if not document_id:
            continue
        if document_id in mapped:
            raise RuntimeError(
                f"Document ID duplikat pada render index: {document_id}"
            )
        mapped[document_id] = record
    return mapped


def ground_truth_access_plan(
    prediction_records: list[dict],
    render_record_map: dict[str, dict],
) -> dict[str, dict]:
    test_root = TEST_GROUND_TRUTH_ROOT.resolve()
    plan = {}

    for prediction_record in prediction_records:
        document_id = str(prediction_record["document_id"])
        template_id = str(prediction_record["template_id"])
        render_record = render_record_map.get(document_id)
        if render_record is None:
            raise RuntimeError(
                f"Render record tidak ditemukan: {document_id}"
            )
        if render_record.get("template_id") != template_id:
            raise RuntimeError(
                f"Template render/prediction berbeda: {document_id}"
            )
        if str(render_record.get("split", "")).casefold() != "test":
            raise RuntimeError(
                f"Render record bukan test: {document_id}"
            )

        artifact = (
            render_record.get("artifacts", {}).get("ground_truth", {})
        )
        if not isinstance(artifact, dict):
            raise TypeError(
                f"Ground-truth artifact metadata tidak valid: {document_id}"
            )

        relative_path = Path(str(artifact.get("relative_path", "")))
        expected_sha256 = str(artifact.get("sha256", "")).casefold()
        expected_size = int(artifact.get("size_bytes", -1))

        if relative_path.is_absolute() or relative_path.suffix != ".json":
            raise RuntimeError(
                f"Ground-truth relative path tidak valid: {document_id}"
            )
        if not SHA256_PATTERN.fullmatch(expected_sha256):
            raise RuntimeError(
                f"Ground-truth checksum metadata tidak valid: {document_id}"
            )

        ground_truth_path = (
            RENDERED_DATASET_ROOT / relative_path
        ).resolve()
        try:
            ground_truth_path.relative_to(test_root)
        except ValueError as error:
            raise RuntimeError(
                f"Ground truth di luar test root: {document_id}"
            ) from error

        if any(
            part.casefold() in {"development", "validation"}
            for part in ground_truth_path.parts
        ):
            raise RuntimeError(
                "Ground truth split lain dilarang pada Cell 12D: "
                f"{ground_truth_path}"
            )

        require_file(ground_truth_path, f"Test GT {document_id}")
        if ground_truth_path.stat().st_size != expected_size:
            raise RuntimeError(
                f"Ukuran test GT tidak cocok: {document_id}"
            )

        plan[document_id] = {
            "document_id": document_id,
            "template_id": template_id,
            "path": str(ground_truth_path),
            "expected_sha256": expected_sha256,
            "expected_size_bytes": expected_size,
        }

    return plan


# ============================================================
# CANONICAL GROUND-TRUTH AND PREDICTION ADAPTERS
# ============================================================

def canonical_from_ground_truth(
    ground_truth: dict,
    document_id: str,
    template_id: str,
) -> dict:
    document = ground_truth.get("document", {})
    canonical = ground_truth.get("canonical")

    if not isinstance(document, dict):
        raise TypeError("ground_truth.document harus berupa object.")
    if not isinstance(canonical, dict):
        raise TypeError("ground_truth.canonical harus berupa object.")
    if document.get("document_id") != document_id:
        raise RuntimeError("Document ID ground truth tidak cocok.")
    if document.get("template_id") != template_id:
        raise RuntimeError("Template ID ground truth tidak cocok.")
    if str(document.get("split", "")).casefold() != "test":
        raise RuntimeError("Ground truth bukan test split.")
    if canonical.get("document_id") != document_id:
        raise RuntimeError("Document ID canonical tidak cocok.")
    if canonical.get("template_id") != template_id:
        raise RuntimeError("Template ID canonical tidak cocok.")
    if str(canonical.get("split", "")).casefold() != "test":
        raise RuntimeError("Canonical payload bukan test split.")

    items = canonical.get("items")
    if not isinstance(items, list) or not items:
        raise RuntimeError("Canonical items kosong atau tidak valid.")
    if not all(isinstance(item, dict) for item in items):
        raise TypeError("Setiap canonical item harus berupa object.")
    return canonical


def expected_scalars(canonical: dict) -> dict[str, str]:
    return {
        field_name: normalize_field_value(
            field_name,
            nested_value(canonical, path),
        )
        for field_name, path in SCALAR_PATHS.items()
    }


def expected_items(canonical: dict) -> list[dict[str, str]]:
    normalized_items = []
    for item in canonical["items"]:
        normalized_items.append(
            {
                field_name: normalize_field_value(
                    f"items[].{field_name}",
                    item.get(field_name),
                )
                for field_name in ITEM_FIELD_NAMES
            }
        )
    return normalized_items


def predicted_scalars(prediction: dict) -> dict[str, str]:
    scalar_predictions = (
        prediction.get("predictions", {}).get("scalar_fields", {})
    )
    if not isinstance(scalar_predictions, dict):
        raise TypeError("predictions.scalar_fields harus berupa object.")

    values = {}
    for field_name in SCALAR_PATHS:
        field_prediction = scalar_predictions.get(field_name, {})
        if not isinstance(field_prediction, dict):
            field_prediction = {}
        values[field_name] = normalize_field_value(
            field_name,
            field_prediction.get("normalized_value"),
        )
    return values


def predicted_items(prediction: dict) -> list[dict[str, str]]:
    item_predictions = prediction.get("predictions", {}).get("items", [])
    if not isinstance(item_predictions, list):
        raise TypeError("predictions.items harus berupa list.")

    normalized_items = []
    for item in item_predictions:
        if not isinstance(item, dict):
            raise TypeError("Setiap prediction item harus berupa object.")
        normalized_item = {}
        for field_name in ITEM_FIELD_NAMES:
            field_prediction = item.get(field_name, {})
            if not isinstance(field_prediction, dict):
                field_prediction = {}
            normalized_item[field_name] = normalize_field_value(
                f"items[].{field_name}",
                field_prediction.get("normalized_value"),
            )
        normalized_items.append(normalized_item)
    return normalized_items


# ============================================================
# MAXIMUM-WEIGHT BIPARTITE ITEM ALIGNMENT — IDENTICAL TO 10D
# ============================================================

def item_pair_weight(predicted: dict, expected: dict) -> int:
    description_similarity = string_similarity(
        predicted.get("description", ""),
        expected.get("description", ""),
    )
    line_total_exact = (
        predicted.get("line_total", "")
        == expected.get("line_total", "")
    )
    quantity_exact = (
        predicted.get("quantity", "")
        == expected.get("quantity", "")
    )
    unit_price_exact = (
        predicted.get("unit_price", "")
        == expected.get("unit_price", "")
    )
    return int(round(description_similarity * 10000)) + (
        10000 if line_total_exact else 0
    ) + (100 if quantity_exact else 0) + (100 if unit_price_exact else 0)


def maximum_weight_item_alignment(
    predictions: list[dict],
    references: list[dict],
) -> list[tuple[int | None, int | None]]:
    size = max(len(predictions), len(references))
    if size == 0:
        return []

    weights = []
    for prediction_index in range(size):
        row = []
        for reference_index in range(size):
            if (
                prediction_index < len(predictions)
                and reference_index < len(references)
            ):
                row.append(
                    item_pair_weight(
                        predictions[prediction_index],
                        references[reference_index],
                    )
                )
            else:
                row.append(0)
        weights.append(row)

    @lru_cache(maxsize=None)
    def solve(
        prediction_index: int,
        used_reference_mask: int,
    ) -> tuple[int, tuple[int, ...]]:
        if prediction_index == size:
            return 0, ()
        best_score = -1
        best_assignment = ()
        for reference_index in range(size):
            bit = 1 << reference_index
            if used_reference_mask & bit:
                continue
            remaining_score, remaining_assignment = solve(
                prediction_index + 1,
                used_reference_mask | bit,
            )
            score = weights[prediction_index][reference_index] + remaining_score
            assignment = (reference_index,) + remaining_assignment
            if score > best_score or (
                score == best_score and assignment < best_assignment
            ):
                best_score = score
                best_assignment = assignment
        return best_score, best_assignment

    _, assignment = solve(0, 0)
    aligned_pairs = []
    for prediction_index, reference_index in enumerate(assignment):
        real_prediction = (
            prediction_index if prediction_index < len(predictions) else None
        )
        real_reference = (
            reference_index if reference_index < len(references) else None
        )
        if real_prediction is not None or real_reference is not None:
            aligned_pairs.append((real_prediction, real_reference))
    return aligned_pairs


# ============================================================
# DOCUMENT EVALUATION — IDENTICAL METRIC DEFINITION TO 10D
# ============================================================

def evaluate_document(
    prediction: dict,
    canonical: dict,
    contract_field_map: dict[str, dict],
) -> dict:
    document = prediction.get("document", {})
    document_id = str(document["document_id"])
    template_id = str(document["template_id"])
    language = str(document.get("language", ""))

    scalar_expected = expected_scalars(canonical)
    scalar_predicted = predicted_scalars(prediction)
    scalar_records = []

    for field_name in SCALAR_PATHS:
        expected_value = scalar_expected[field_name]
        predicted_value = scalar_predicted[field_name]
        exact_match = predicted_value == expected_value
        character_errors = levenshtein_distance(
            expected_value,
            predicted_value,
        )
        word_errors = levenshtein_distance(
            expected_value.split(),
            predicted_value.split(),
        )
        scalar_records.append(
            {
                "document_id": document_id,
                "template_id": template_id,
                "language": language,
                "group": "scalar",
                "field": field_name,
                "critical": bool(
                    contract_field_map[field_name].get("critical")
                ),
                "expected": expected_value,
                "predicted": predicted_value,
                "exact_match": exact_match,
                "character_errors": character_errors,
                "reference_characters": len(expected_value),
                "word_errors": word_errors,
                "reference_words": len(expected_value.split()),
            }
        )

    item_expected = expected_items(canonical)
    item_predicted = predicted_items(prediction)
    alignment = maximum_weight_item_alignment(
        item_predicted,
        item_expected,
    )

    item_field_records = []
    item_pair_records = []
    for pair_number, (prediction_index, reference_index) in enumerate(
        alignment,
        start=1,
    ):
        predicted_item = (
            item_predicted[prediction_index]
            if prediction_index is not None
            else None
        )
        expected_item = (
            item_expected[reference_index]
            if reference_index is not None
            else None
        )

        description_exact = bool(
            predicted_item is not None
            and expected_item is not None
            and predicted_item["description"] == expected_item["description"]
        )
        line_total_exact = bool(
            predicted_item is not None
            and expected_item is not None
            and predicted_item["line_total"] == expected_item["line_total"]
        )
        accepted_row_match = description_exact and line_total_exact
        item_pair_records.append(
            {
                "pair_number": pair_number,
                "prediction_row": (
                    prediction_index + 1
                    if prediction_index is not None
                    else None
                ),
                "reference_row": (
                    reference_index + 1
                    if reference_index is not None
                    else None
                ),
                "description_exact": description_exact,
                "line_total_exact": line_total_exact,
                "accepted_row_match": accepted_row_match,
            }
        )

        for field_name in ITEM_FIELD_NAMES:
            full_field_name = f"items[].{field_name}"
            expected_value = (
                expected_item[field_name] if expected_item is not None else ""
            )
            predicted_value = (
                predicted_item[field_name]
                if predicted_item is not None
                else ""
            )
            exact_match = bool(
                predicted_item is not None
                and expected_item is not None
                and predicted_value == expected_value
            )
            item_field_records.append(
                {
                    "document_id": document_id,
                    "template_id": template_id,
                    "language": language,
                    "group": "item",
                    "field": full_field_name,
                    "critical": bool(
                        contract_field_map[full_field_name].get("critical")
                    ),
                    "prediction_row": (
                        prediction_index + 1
                        if prediction_index is not None
                        else None
                    ),
                    "reference_row": (
                        reference_index + 1
                        if reference_index is not None
                        else None
                    ),
                    "expected": expected_value,
                    "predicted": predicted_value,
                    "exact_match": exact_match,
                    "character_errors": levenshtein_distance(
                        expected_value,
                        predicted_value,
                    ),
                    "reference_characters": len(expected_value),
                    "word_errors": levenshtein_distance(
                        expected_value.split(),
                        predicted_value.split(),
                    ),
                    "reference_words": len(expected_value.split()),
                }
            )

    row_true_positives = sum(
        record["accepted_row_match"] for record in item_pair_records
    )
    row_precision = safe_ratio(row_true_positives, len(item_predicted))
    row_recall = safe_ratio(row_true_positives, len(item_expected))
    row_f1 = harmonic_mean(row_precision, row_recall)

    item_exact_fields = sum(
        record["exact_match"] for record in item_field_records
    )
    predicted_item_fields = len(item_predicted) * EXPECTED_ITEM_FIELDS
    reference_item_fields = len(item_expected) * EXPECTED_ITEM_FIELDS
    item_field_precision = safe_ratio(
        item_exact_fields,
        predicted_item_fields,
    )
    item_field_recall = safe_ratio(
        item_exact_fields,
        reference_item_fields,
    )
    item_field_f1 = harmonic_mean(
        item_field_precision,
        item_field_recall,
    )

    scalar_exact_count = sum(
        record["exact_match"] for record in scalar_records
    )
    document_exact = bool(
        scalar_exact_count == EXPECTED_SCALAR_FIELDS
        and len(item_predicted) == len(item_expected)
        and item_exact_fields == reference_item_fields
    )
    financial_check = prediction.get("diagnostics", {}).get(
        "financial_equation",
        {},
    )
    financial_consistent = bool(
        financial_check.get("executed") is True
        and financial_check.get("passed") is True
    )

    return {
        "document_id": document_id,
        "template_id": template_id,
        "language": language,
        "scalar_records": scalar_records,
        "item_field_records": item_field_records,
        "item_alignment": item_pair_records,
        "metrics": {
            "scalar_exact_count": scalar_exact_count,
            "scalar_field_count": EXPECTED_SCALAR_FIELDS,
            "scalar_exact_match": safe_ratio(
                scalar_exact_count,
                EXPECTED_SCALAR_FIELDS,
            ),
            "predicted_item_count": len(item_predicted),
            "reference_item_count": len(item_expected),
            "item_count_exact": len(item_predicted) == len(item_expected),
            "row_true_positives": row_true_positives,
            "row_precision": row_precision,
            "row_recall": row_recall,
            "row_f1": row_f1,
            "item_exact_fields": item_exact_fields,
            "predicted_item_fields": predicted_item_fields,
            "reference_item_fields": reference_item_fields,
            "item_field_precision": item_field_precision,
            "item_field_recall": item_field_recall,
            "item_field_f1": item_field_f1,
            "financial_consistent": financial_consistent,
            "document_exact_match": document_exact,
        },
    }


# ============================================================
# PREFLIGHT — FREEZE ALL PREDICTIONS BEFORE OPENING TEST GT
# ============================================================

for required_path, label in (
    (CONTRACT_PATH, "Field extraction contract"),
    (BASELINE_POINTER_PATH, "Test baseline pointer"),
    (PREDICTION_MANIFEST_PATH, "Test prediction manifest"),
    (RENDER_INDEX_PATH, "Batch render index"),
):
    require_file(required_path, label)
require_directory(
    TEST_GROUND_TRUTH_ROOT,
    "Test ground-truth root",
)

contract = load_json(CONTRACT_PATH)
baseline_pointer = load_json(BASELINE_POINTER_PATH)
prediction_manifest = load_json(PREDICTION_MANIFEST_PATH)
render_records = load_jsonl(RENDER_INDEX_PATH)

freeze_manifest_path = Path(
    str(baseline_pointer.get("development_freeze_manifest_path", ""))
)
validation_decision_path = Path(
    str(baseline_pointer.get("validation_decision_manifest_path", ""))
)
parser_source_path = Path(
    str(baseline_pointer.get("parser_source_path", ""))
)
require_file(freeze_manifest_path, "Frozen baseline manifest")
require_file(validation_decision_path, "Frozen validation decision")
require_file(parser_source_path, "Frozen parser source")
freeze_manifest = load_json(freeze_manifest_path)
validation_decision = load_json(validation_decision_path)

validation_evaluation_reference = validation_decision.get(
    "validation_evidence", {}
)
validation_evaluation_path = Path(
    str(validation_evaluation_reference.get("evaluation_manifest_path", ""))
)
require_file(
    validation_evaluation_path,
    "Frozen validation evaluation manifest",
)
validation_evaluation = load_json(validation_evaluation_path)

development_evaluation_reference = (
    freeze_manifest.get("immutable_inputs", {})
    .get("development_evaluation_manifest", {})
)
development_evaluation_path = Path(
    str(development_evaluation_reference.get("path", ""))
)
require_file(
    development_evaluation_path,
    "Frozen development evaluation manifest",
)
development_evaluation = load_json(development_evaluation_path)

field_definitions = contract.get("field_definitions", [])
contract_field_map = {
    field["field"]: field
    for field in field_definitions
    if isinstance(field, dict) and field.get("field")
}
prediction_parser = prediction_manifest.get("parser", {})
prediction_scope = prediction_manifest.get("scope", {})
prediction_integrity = prediction_manifest.get("integrity", {})
prediction_next_stage = prediction_manifest.get("next_stage", {})
freeze_parser = freeze_manifest.get("parser", {})
freeze_split_policy = freeze_manifest.get("split_policy", {})
decision_baseline = validation_decision.get("baseline", {})
decision_split_policy = validation_decision.get("split_policy", {})
decision_integrity = validation_decision.get("integrity", {})
prediction_records = prediction_manifest.get("records", [])
if not isinstance(prediction_records, list):
    raise TypeError("records pada prediction manifest bukan list.")

input_checksums_before = {
    "contract": sha256_file(CONTRACT_PATH),
    "baseline_pointer": sha256_file(BASELINE_POINTER_PATH),
    "freeze_manifest": sha256_file(freeze_manifest_path),
    "validation_decision": sha256_file(validation_decision_path),
    "validation_evaluation": sha256_file(validation_evaluation_path),
    "parser_source": sha256_file(parser_source_path),
    "prediction_manifest": sha256_file(PREDICTION_MANIFEST_PATH),
    "render_index": sha256_file(RENDER_INDEX_PATH),
    "development_evaluation": sha256_file(
        development_evaluation_path
    ),
}

frozen_acceptance_checks = (
    freeze_manifest.get("development_evidence", {})
    .get("acceptance_checks", [])
)
frozen_acceptance_minima = {
    str(check.get("metric")): float(check.get("minimum"))
    for check in frozen_acceptance_checks
    if isinstance(check, dict)
    and check.get("metric")
    and check.get("minimum") is not None
}
required_acceptance_metrics = {
    "critical_scalar_exact_match",
    "all_scalar_exact_match",
    "line_item_field_f1",
    "financial_consistency",
}

preflight_values = [
    (
        "contract_status",
        "FROZEN_FOR_DEVELOPMENT_BASELINE",
        contract.get("status"),
    ),
    ("target_fields", 16, len(contract_field_map)),
    (
        "target_field_names",
        sorted(
            set(SCALAR_PATHS)
            | {f"items[].{name}" for name in ITEM_FIELD_NAMES}
        ),
        sorted(contract_field_map),
    ),
    ("render_index_records", EXPECTED_RENDER_RECORDS, len(render_records)),
    (
        "baseline_pointer_status",
        "ACTIVE_FOR_TEST_PREFLIGHT",
        baseline_pointer.get("status"),
    ),
    ("baseline_id", EXPECTED_BASELINE_ID, baseline_pointer.get("baseline_id")),
    (
        "baseline_pointer_parser_id",
        EXPECTED_PARSER_ID,
        baseline_pointer.get("parser_id"),
    ),
    (
        "baseline_pointer_parser_version",
        EXPECTED_PARSER_VERSION,
        baseline_pointer.get("parser_version"),
    ),
    (
        "baseline_pointer_parser_signature",
        EXPECTED_PARSER_SIGNATURE,
        baseline_pointer.get("parser_signature_sha256"),
    ),
    (
        "pointer_next_allowed_stage",
        "TEST_COHORT_PREFLIGHT",
        baseline_pointer.get("next_allowed_stage"),
    ),
    ("pointer_next_allowed_split", "test", baseline_pointer.get("next_allowed_split")),
    (
        "parser_mutation_allowed",
        False,
        baseline_pointer.get("parser_mutation_allowed"),
    ),
    (
        "test_gt_locked_until_predictions_frozen",
        True,
        baseline_pointer.get("test_ground_truth_remains_locked"),
    ),
    (
        "test_predictions_must_be_frozen",
        True,
        baseline_pointer.get(
            "test_predictions_must_be_frozen_before_ground_truth_open"
        ),
    ),
    (
        "freeze_status",
        "FROZEN_FOR_VALIDATION",
        freeze_manifest.get("status"),
    ),
    ("freeze_baseline_id", EXPECTED_BASELINE_ID, freeze_manifest.get("baseline_id")),
    ("freeze_parser_id", EXPECTED_PARSER_ID, freeze_parser.get("parser_id")),
    (
        "freeze_parser_version",
        EXPECTED_PARSER_VERSION,
        freeze_parser.get("parser_version"),
    ),
    (
        "freeze_parser_signature",
        EXPECTED_PARSER_SIGNATURE,
        freeze_parser.get("parser_signature_sha256"),
    ),
    (
        "freeze_checksum",
        baseline_pointer.get("development_freeze_manifest_sha256"),
        input_checksums_before["freeze_manifest"],
    ),
    (
        "parser_source_checksum",
        baseline_pointer.get("parser_source_sha256"),
        input_checksums_before["parser_source"],
    ),
    (
        "parser_source_matches_freeze_path",
        True,
        Path(str(freeze_parser.get("source_path", ""))).resolve()
        == parser_source_path.resolve(),
    ),
    (
        "parser_source_matches_freeze_checksum",
        input_checksums_before["parser_source"],
        freeze_parser.get("source_sha256"),
    ),
    (
        "freeze_next_allowed_split",
        "validation",
        freeze_split_policy.get("next_allowed_split"),
    ),
    (
        "test_locked_by_development_freeze",
        True,
        freeze_split_policy.get("test_remains_locked"),
    ),
    (
        "validation_decision_status",
        "FROZEN_FOR_TEST_PREFLIGHT",
        validation_decision.get("status"),
    ),
    (
        "validation_decision",
        "VALIDATION_PASSED_BASELINE_ACCEPTED",
        validation_decision.get("decision"),
    ),
    (
        "validation_decision_checksum",
        baseline_pointer.get("validation_decision_manifest_sha256"),
        input_checksums_before["validation_decision"],
    ),
    (
        "decision_parser_signature",
        EXPECTED_PARSER_SIGNATURE,
        decision_baseline.get("parser_signature_sha256"),
    ),
    (
        "decision_parser_source_checksum",
        input_checksums_before["parser_source"],
        decision_baseline.get("parser_source_sha256"),
    ),
    (
        "decision_freeze_checksum",
        input_checksums_before["freeze_manifest"],
        decision_baseline.get("development_freeze_manifest_sha256"),
    ),
    (
        "validation_evaluation_status",
        "PASSED",
        validation_evaluation.get("status"),
    ),
    (
        "validation_evaluation_checksum",
        validation_evaluation_reference.get("evaluation_manifest_sha256"),
        input_checksums_before["validation_evaluation"],
    ),
    (
        "validation_result_fingerprint",
        baseline_pointer.get("validation_result_sha256"),
        validation_evaluation_reference.get("validation_result_sha256"),
    ),
    (
        "decision_next_allowed_stage",
        "TEST_COHORT_PREFLIGHT",
        decision_split_policy.get("next_allowed_stage"),
    ),
    (
        "decision_next_allowed_split",
        "test",
        decision_split_policy.get("next_allowed_split"),
    ),
    (
        "decision_test_preflight_allowed",
        True,
        decision_split_policy.get("test_preflight_allowed"),
    ),
    (
        "decision_test_gt_locked",
        True,
        decision_split_policy.get("test_ground_truth_remains_locked"),
    ),
    (
        "decision_test_gt_opened",
        0,
        decision_integrity.get("test_ground_truth_opened"),
    ),
    (
        "prediction_manifest_status",
        "EXECUTED",
        prediction_manifest.get("status"),
    ),
    (
        "prediction_freeze_status",
        "FROZEN_BEFORE_TEST_GROUND_TRUTH_OPEN",
        prediction_manifest.get("prediction_freeze_status"),
    ),
    (
        "prediction_quality_before_evaluation",
        "PENDING_TEST_GROUND_TRUTH_EVALUATION",
        prediction_manifest.get("quality_status"),
    ),
    (
        "prediction_stage",
        "FROZEN_PARSER_BLIND_TEST_PREDICTION",
        prediction_manifest.get("stage"),
    ),
    ("prediction_parser_id", EXPECTED_PARSER_ID, prediction_parser.get("parser_id")),
    (
        "prediction_parser_version",
        EXPECTED_PARSER_VERSION,
        prediction_parser.get("parser_version"),
    ),
    (
        "prediction_parser_signature",
        EXPECTED_PARSER_SIGNATURE,
        prediction_parser.get("parser_signature_sha256"),
    ),
    (
        "prediction_parser_source_path",
        True,
        Path(str(prediction_parser.get("parser_source_path", ""))).resolve()
        == parser_source_path.resolve(),
    ),
    (
        "prediction_parser_source_checksum",
        input_checksums_before["parser_source"],
        prediction_parser.get("parser_source_sha256"),
    ),
    (
        "prediction_development_freeze_checksum",
        input_checksums_before["freeze_manifest"],
        prediction_parser.get("development_freeze_manifest_sha256"),
    ),
    (
        "prediction_validation_decision_checksum",
        input_checksums_before["validation_decision"],
        prediction_parser.get("validation_decision_manifest_sha256"),
    ),
    (
        "prediction_validation_result_fingerprint",
        baseline_pointer.get("validation_result_sha256"),
        prediction_parser.get("validation_result_sha256"),
    ),
    ("prediction_split", "test", prediction_scope.get("split")),
    ("prediction_documents", EXPECTED_DOCUMENTS, prediction_scope.get("documents")),
    (
        "prediction_templates",
        sorted(EXPECTED_TEMPLATES),
        sorted(prediction_scope.get("templates", [])),
    ),
    (
        "prediction_records_frozen",
        EXPECTED_DOCUMENTS,
        prediction_scope.get("predictions_frozen"),
    ),
    (
        "prediction_integrity_frozen_before_gt",
        True,
        prediction_integrity.get(
            "test_predictions_frozen_before_ground_truth_open"
        ),
    ),
    (
        "test_ground_truth_loaded_before_evaluation",
        False,
        prediction_integrity.get("test_ground_truth_loaded"),
    ),
    (
        "ground_truth_used_as_prediction_input",
        False,
        prediction_integrity.get("ground_truth_used_as_prediction_input"),
    ),
    (
        "validation_gt_reopened_before_evaluation",
        0,
        prediction_integrity.get("validation_ground_truth_reopened"),
    ),
    (
        "test_gt_opened_before_evaluation",
        0,
        prediction_integrity.get("test_ground_truth_opened"),
    ),
    (
        "prediction_next_cell",
        "CELL 12D",
        prediction_next_stage.get("cell"),
    ),
    (
        "test_gt_one_time_access_authorized",
        True,
        prediction_next_stage.get("test_ground_truth_may_open_once_in_next_cell"),
    ),
    (
        "frozen_acceptance_metrics",
        sorted(required_acceptance_metrics),
        sorted(frozen_acceptance_minima),
    ),
    (
        "frozen_acceptance_checks_passed",
        True,
        bool(frozen_acceptance_checks)
        and all(
            check.get("status") == "PASSED"
            for check in frozen_acceptance_checks
            if isinstance(check, dict)
        ),
    ),
    (
        "development_evaluation_status",
        "PASSED",
        development_evaluation.get("status"),
    ),
    (
        "development_evaluator_id",
        EVALUATOR_ID,
        development_evaluation.get("evaluator", {}).get("evaluator_id"),
    ),
    (
        "development_evaluator_version",
        EVALUATOR_VERSION,
        development_evaluation.get("evaluator", {}).get("evaluator_version"),
    ),
    (
        "development_evaluation_checksum",
        development_evaluation_reference.get("sha256"),
        input_checksums_before["development_evaluation"],
    ),
]

preflight_controls = [
    {
        "control": name,
        "expected": expected,
        "actual": actual,
        "status": "VALID" if expected == actual else "INVALID",
    }
    for name, expected, actual in preflight_values
]

print("=" * 96)
print(
    f"CELL 12D — {EVALUATOR_ID} — VERSION {EVALUATOR_VERSION} — "
    "TEST EVALUATION"
)
print(f"Evaluation root: {EVALUATION_ROOT}")
print(
    "Predictions: FROZEN | "
    "Test GT: AUTHORIZED FOR EVALUATION ONLY"
)
print("=" * 96)
print("PREFLIGHT CONTROLS — BEFORE TEST GROUND TRUTH IS OPENED")
display(pd.DataFrame(preflight_controls))

invalid_preflight = [
    row["control"]
    for row in preflight_controls
    if row["status"] != "VALID"
]
if invalid_preflight:
    raise RuntimeError(
        "CELL 12D PREFLIGHT FAILED. "
        f"Kontrol tidak valid: {invalid_preflight}. "
        "Test ground truth belum dibuka; prediction checkpoint "
        "tetap frozen."
    )


# ============================================================
# VERIFY ALL FROZEN PREDICTIONS BEFORE BUILDING THE GT PLAN
# ============================================================

prediction_records = sorted(
    prediction_records,
    key=lambda row: (
        int(row.get("sequence_number", 0)),
        str(row.get("document_id", "")),
    ),
)
prediction_source_errors = []

for record_index, prediction_record in enumerate(
    prediction_records,
    start=1,
):
    try:
        if not isinstance(prediction_record, dict):
            raise TypeError("Prediction manifest record bukan object.")

        document_id = str(prediction_record.get("document_id", ""))
        template_id = str(prediction_record.get("template_id", ""))
        prediction_path = Path(
            str(prediction_record.get("prediction_path", ""))
        )

        if not document_id:
            raise RuntimeError("document_id kosong.")
        if template_id not in EXPECTED_TEMPLATES:
            raise RuntimeError(
                f"Template prediction di luar test: {template_id}"
            )
        if not prediction_path.resolve().is_relative_to(
            (TEST_ROOT / "predictions").resolve()
        ):
            raise RuntimeError(
                f"Prediction berada di luar test prediction root: "
                f"{document_id}"
            )
        if any(
            part.casefold() == "ground_truth"
            for part in prediction_path.parts
        ):
            raise RuntimeError(
                f"Prediction path mengarah ke ground truth: {document_id}"
            )

        require_file(prediction_path, f"Prediction {document_id}")
        actual_checksum = sha256_file(prediction_path)
        if actual_checksum != prediction_record.get("prediction_sha256"):
            raise RuntimeError(
                f"Checksum prediction tidak cocok: {document_id}"
            )

        prediction = load_json(prediction_path)
        document = prediction.get("document", {})
        parser = prediction.get("parser", {})
        integrity = prediction.get("integrity", {})

        if prediction.get("status") != "EXECUTED":
            raise RuntimeError("Status prediction bukan EXECUTED.")
        if prediction.get("prediction_freeze_status") != (
            "FROZEN_BEFORE_TEST_GROUND_TRUTH_OPEN"
        ):
            raise RuntimeError("Prediction belum dibekukan sebelum test GT.")
        if prediction_record.get("prediction_freeze_status") != (
            "FROZEN_BEFORE_TEST_GROUND_TRUTH_OPEN"
        ):
            raise RuntimeError(
                "Manifest record belum menandai prediction sebagai frozen."
            )
        if prediction.get("quality_status") != (
            "PENDING_TEST_GROUND_TRUTH_EVALUATION"
        ):
            raise RuntimeError("Quality status prediction tidak valid.")
        if document.get("document_id") != document_id:
            raise RuntimeError("Prediction document ID tidak cocok.")
        if document.get("template_id") != template_id:
            raise RuntimeError("Prediction template ID tidak cocok.")
        if document.get("split") != "test":
            raise RuntimeError("Prediction bukan test split.")
        if parser.get("parser_id") != EXPECTED_PARSER_ID:
            raise RuntimeError("Prediction parser ID tidak cocok.")
        if parser.get("parser_version") != EXPECTED_PARSER_VERSION:
            raise RuntimeError("Prediction parser version tidak cocok.")
        if parser.get("parser_signature_sha256") != (
            EXPECTED_PARSER_SIGNATURE
        ):
            raise RuntimeError("Prediction parser signature tidak cocok.")
        if integrity.get("test_ground_truth_loaded") is not False:
            raise RuntimeError("Prediction menandai ground truth telah dibuka.")
        if integrity.get("ground_truth_used_as_prediction_input") is not False:
            raise RuntimeError("Ground truth terindikasi sebagai input parser.")
        if integrity.get("test_ground_truth_opened") != 0:
            raise RuntimeError("Prediction dibuat setelah test GT dibuka.")
        if integrity.get("validation_ground_truth_reopened") != 0:
            raise RuntimeError(
                "Prediction menandai validation GT dibuka kembali."
            )

        input_checksums_before[
            f"prediction:{document_id}"
        ] = actual_checksum

    except Exception as error:
        prediction_source_errors.append(
            {
                "record_index": record_index,
                "document_id": (
                    prediction_record.get("document_id")
                    if isinstance(prediction_record, dict)
                    else None
                ),
                "error_type": type(error).__name__,
                "error": str(error)[:700],
            }
        )

prediction_gate_values = [
    ("prediction_records", EXPECTED_DOCUMENTS, len(prediction_records)),
    (
        "unique_prediction_ids",
        EXPECTED_DOCUMENTS,
        len({row.get("document_id") for row in prediction_records}),
    ),
    (
        "prediction_templates",
        sorted(EXPECTED_TEMPLATES),
        sorted({row.get("template_id") for row in prediction_records}),
    ),
    (
        "prediction_sequence_numbers",
        list(range(1, EXPECTED_DOCUMENTS + 1)),
        [int(row.get("sequence_number", 0)) for row in prediction_records],
    ),
    (
        "frozen_prediction_records",
        EXPECTED_DOCUMENTS,
        sum(
            row.get("prediction_freeze_status")
            == "FROZEN_BEFORE_TEST_GROUND_TRUTH_OPEN"
            for row in prediction_records
        ),
    ),
    ("prediction_source_errors", 0, len(prediction_source_errors)),
]

prediction_gate_controls = [
    {
        "control": name,
        "expected": expected,
        "actual": actual,
        "status": "VALID" if expected == actual else "INVALID",
    }
    for name, expected, actual in prediction_gate_values
]

print("\nFROZEN PREDICTION CONTROLS")
display(pd.DataFrame(prediction_gate_controls))
if prediction_source_errors:
    display(pd.DataFrame(prediction_source_errors))

invalid_prediction_gates = [
    row["control"]
    for row in prediction_gate_controls
    if row["status"] != "VALID"
]
if invalid_prediction_gates:
    raise RuntimeError(
        "CELL 12D PREDICTION GATE FAILED. "
        f"Kontrol tidak valid: {invalid_prediction_gates}. "
        "Test ground truth belum dibuka."
    )


# Build the access plan using metadata only. No GT JSON is loaded here.
render_record_map = build_render_record_map(render_records)
gt_plan = ground_truth_access_plan(
    prediction_records,
    render_record_map,
)

gt_plan_values = [
    ("test_gt_plan_records", EXPECTED_DOCUMENTS, len(gt_plan)),
    (
        "unique_test_gt_paths",
        EXPECTED_DOCUMENTS,
        len({record["path"] for record in gt_plan.values()}),
    ),
    (
        "test_gt_templates",
        sorted(EXPECTED_TEMPLATES),
        sorted({record["template_id"] for record in gt_plan.values()}),
    ),
    (
        "all_paths_inside_test_root",
        True,
        all(
            Path(record["path"]).resolve().is_relative_to(
                TEST_GROUND_TRUTH_ROOT.resolve()
            )
            for record in gt_plan.values()
        ),
    ),
    ("test_gt_payloads_opened_so_far", 0, 0),
    ("validation_ground_truth_reopened", 0, 0),
]

gt_plan_controls = [
    {
        "control": name,
        "expected": expected,
        "actual": actual,
        "status": "VALID" if expected == actual else "INVALID",
    }
    for name, expected, actual in gt_plan_values
]
print("\nTEST GROUND-TRUTH ACCESS PLAN")
display(pd.DataFrame(gt_plan_controls))

invalid_gt_plan = [
    row["control"]
    for row in gt_plan_controls
    if row["status"] != "VALID"
]
if invalid_gt_plan:
    raise RuntimeError(
        "CELL 12D TEST GT PLAN FAILED. "
        f"Kontrol tidak valid: {invalid_gt_plan}."
    )

print("\nFROZEN ACCEPTANCE THRESHOLDS")
display(
    pd.DataFrame(
        [
            {
                "metric": metric,
                "minimum": frozen_acceptance_minima[metric],
                "source": (
                    "DEVELOPMENT_FREEZE_MANIFEST_FROZEN_BEFORE_"
                    "VALIDATION_AND_TEST"
                ),
            }
            for metric in sorted(frozen_acceptance_minima)
        ]
    )
)


# ============================================================
# OPEN EXACTLY 40 TEST GT FILES AND EVALUATE
# ============================================================

runtime_records = []
document_manifest_records = []
all_scalar_records = []
all_item_field_records = []
evaluation_errors = []
ground_truth_paths = {}
ground_truth_checksums = {}
new_evaluations = 0
recovered_evaluations = 0

print(
    f"\nMengevaluasi {len(prediction_records)} frozen test "
    "predictions...\n"
)

for sequence_number, prediction_record in enumerate(
    prediction_records,
    start=1,
):
    document_id = str(prediction_record["document_id"])
    template_id = str(prediction_record["template_id"])
    prediction_path = Path(prediction_record["prediction_path"])
    evaluation_path = (
        DOCUMENT_EVALUATION_ROOT
        / template_id
        / f"{document_id}_evaluation.json"
    )

    try:
        prediction = load_json(prediction_path)
        gt_record = gt_plan[document_id]
        ground_truth_path = Path(gt_record["path"])

        # This is the first point at which test GT bytes and JSON are
        # opened. Only the exact path planned above is permitted.
        actual_gt_checksum = sha256_file(ground_truth_path)
        if actual_gt_checksum != gt_record["expected_sha256"]:
            raise RuntimeError(
                f"Checksum test GT tidak cocok: {document_id}"
            )
        ground_truth = load_json(ground_truth_path)
        canonical = canonical_from_ground_truth(
            ground_truth,
            document_id,
            template_id,
        )

        ground_truth_paths[document_id] = ground_truth_path
        ground_truth_checksums[document_id] = actual_gt_checksum

        result = evaluate_document(
            prediction,
            canonical,
            contract_field_map,
        )
        metrics = result["metrics"]

        evaluation_artifact = {
            "schema_version": "1.0.0",
            "status": "EVALUATED",
            "evaluator": {
                "evaluator_id": EVALUATOR_ID,
                "evaluator_version": EVALUATOR_VERSION,
                "scalar_matching": (
                    "TYPE_AWARE_NORMALIZED_EXACT_MATCH"
                ),
                "item_alignment": (
                    "MAXIMUM_WEIGHT_BIPARTITE_ROW_MATCHING"
                ),
                "acceptance_threshold_source": (
                    "FROZEN_DEVELOPMENT_FREEZE_MANIFEST"
                ),
            },
            "document": {
                "document_id": document_id,
                "template_id": template_id,
                "split": "test",
                "language": result["language"],
            },
            "metrics": metrics,
            "scalar_comparisons": result["scalar_records"],
            "item_alignment": result["item_alignment"],
            "item_field_comparisons": result["item_field_records"],
            "inputs": {
                "prediction_path": str(prediction_path),
                "prediction_sha256": prediction_record[
                    "prediction_sha256"
                ],
                "ground_truth_path": str(ground_truth_path),
                "ground_truth_sha256": actual_gt_checksum,
            },
            "integrity": {
                "predictions_frozen_before_ground_truth_open": True,
                "ground_truth_split": "test",
                "test_ground_truth_opened_in_this_evaluation": 1,
                "validation_ground_truth_reopened": 0,
                "ground_truth_used_for_evaluation_only": True,
                "ground_truth_used_as_prediction_input": False,
                "parser_modified": False,
                "prediction_modified": False,
                "ground_truth_modified": False,
                "dataset_modified": False,
            },
        }

        checkpoint_action = save_immutable_json(
            evaluation_path,
            evaluation_artifact,
        )
        if checkpoint_action == "CREATED":
            new_evaluations += 1
            execution = "NEW"
        else:
            recovered_evaluations += 1
            execution = "RECOVERED"

        persisted = load_json(evaluation_path)
        if canonical_json(persisted) != canonical_json(
            evaluation_artifact
        ):
            raise RuntimeError(
                f"Evaluation berbeda setelah penulisan: {document_id}"
            )

        record = {
            "sequence_number": sequence_number,
            "document_id": document_id,
            "template_id": template_id,
            "language": result["language"],
            "scalar_exact_match": metrics["scalar_exact_match"],
            "predicted_item_count": metrics["predicted_item_count"],
            "reference_item_count": metrics["reference_item_count"],
            "item_count_exact": metrics["item_count_exact"],
            "row_precision": metrics["row_precision"],
            "row_recall": metrics["row_recall"],
            "row_f1": metrics["row_f1"],
            "item_field_precision": metrics["item_field_precision"],
            "item_field_recall": metrics["item_field_recall"],
            "item_field_f1": metrics["item_field_f1"],
            "financial_consistent": metrics["financial_consistent"],
            "document_exact_match": metrics["document_exact_match"],
            "evaluation_path": str(evaluation_path),
            "evaluation_sha256": sha256_file(evaluation_path),
            "prediction_path": str(prediction_path),
            "prediction_sha256": prediction_record[
                "prediction_sha256"
            ],
            "ground_truth_path": str(ground_truth_path),
            "ground_truth_sha256": actual_gt_checksum,
            "status": "EVALUATED",
        }
        document_manifest_records.append(record)
        runtime_records.append({**record, "execution": execution})
        all_scalar_records.extend(result["scalar_records"])
        all_item_field_records.extend(result["item_field_records"])

        print(
            f"[{sequence_number:02d}/{len(prediction_records):02d}] "
            f"{document_id} | "
            f"scalar={metrics['scalar_exact_match']:.4f} | "
            f"items={metrics['predicted_item_count']}/"
            f"{metrics['reference_item_count']} | "
            f"item_f1={metrics['item_field_f1']:.4f} | "
            f"exact={metrics['document_exact_match']} | {execution}"
        )

    except Exception as error:
        evaluation_errors.append(
            {
                "sequence_number": sequence_number,
                "document_id": document_id,
                "template_id": template_id,
                "error_type": type(error).__name__,
                "error": str(error)[:900],
            }
        )
        print(
            f"[{sequence_number:02d}/{len(prediction_records):02d}] "
            f"{document_id} | ERROR: "
            f"{type(error).__name__}: {error}"
        )


# ============================================================
# TECHNICAL COMPLETENESS GATE
# ============================================================

if evaluation_errors:
    print("\nEVALUATION ERRORS")
    display(pd.DataFrame(evaluation_errors))

technical_values = [
    (
        "evaluated_documents",
        EXPECTED_DOCUMENTS,
        len(document_manifest_records),
    ),
    (
        "evaluation_files",
        EXPECTED_DOCUMENTS,
        sum(
            Path(record["evaluation_path"]).is_file()
            for record in document_manifest_records
        ),
    ),
    (
        "unique_document_ids",
        EXPECTED_DOCUMENTS,
        len({record["document_id"] for record in document_manifest_records}),
    ),
    (
        "templates_evaluated",
        sorted(EXPECTED_TEMPLATES),
        sorted(
            {record["template_id"] for record in document_manifest_records}
        ),
    ),
    (
        "test_ground_truth_opened",
        EXPECTED_DOCUMENTS,
        len(ground_truth_paths),
    ),
    ("validation_ground_truth_reopened", 0, 0),
    ("evaluation_errors", 0, len(evaluation_errors)),
]

technical_control_records = [
    {
        "control": name,
        "expected": expected,
        "actual": actual,
        "status": "VALID" if expected == actual else "INVALID",
    }
    for name, expected, actual in technical_values
]

print("\nTECHNICAL EVALUATION CONTROLS")
display(pd.DataFrame(technical_control_records))

invalid_technical_controls = [
    row["control"]
    for row in technical_control_records
    if row["status"] != "VALID"
]
if invalid_technical_controls:
    raise RuntimeError(
        "CELL 12D TECHNICAL EVALUATION FAILED. "
        f"Kontrol tidak valid: {invalid_technical_controls}. "
        "Hentikan evaluasi dan jangan mengubah parser atau prediction."
    )


# ============================================================
# AGGREGATE TEST METRICS
# ============================================================

scalar_total = len(all_scalar_records)
scalar_exact = sum(
    record["exact_match"] for record in all_scalar_records
)
critical_scalar_records = [
    record for record in all_scalar_records if record["critical"]
]
critical_scalar_exact = sum(
    record["exact_match"] for record in critical_scalar_records
)

all_scalar_exact_match = safe_ratio(scalar_exact, scalar_total)
critical_scalar_exact_match = safe_ratio(
    critical_scalar_exact,
    len(critical_scalar_records),
)

scalar_character_errors = sum(
    record["character_errors"] for record in all_scalar_records
)
scalar_reference_characters = sum(
    record["reference_characters"] for record in all_scalar_records
)
scalar_word_errors = sum(
    record["word_errors"] for record in all_scalar_records
)
scalar_reference_words = sum(
    record["reference_words"] for record in all_scalar_records
)
scalar_micro_cer = safe_error_rate(
    scalar_character_errors,
    scalar_reference_characters,
)
scalar_micro_wer = safe_error_rate(
    scalar_word_errors,
    scalar_reference_words,
)

item_exact_fields = sum(
    record["exact_match"] for record in all_item_field_records
)
predicted_item_field_count = sum(
    record["predicted_item_count"] * EXPECTED_ITEM_FIELDS
    for record in document_manifest_records
)
reference_item_field_count = sum(
    record["reference_item_count"] * EXPECTED_ITEM_FIELDS
    for record in document_manifest_records
)
line_item_field_precision = safe_ratio(
    item_exact_fields,
    predicted_item_field_count,
)
line_item_field_recall = safe_ratio(
    item_exact_fields,
    reference_item_field_count,
)
line_item_field_f1 = harmonic_mean(
    line_item_field_precision,
    line_item_field_recall,
)

row_true_positives = 0
predicted_rows = 0
reference_rows = 0
for record in document_manifest_records:
    evaluation = load_json(Path(record["evaluation_path"]))
    metrics = evaluation["metrics"]
    row_true_positives += int(metrics["row_true_positives"])
    predicted_rows += int(metrics["predicted_item_count"])
    reference_rows += int(metrics["reference_item_count"])

row_precision = safe_ratio(row_true_positives, predicted_rows)
row_recall = safe_ratio(row_true_positives, reference_rows)
row_f1 = harmonic_mean(row_precision, row_recall)

financial_consistency = safe_ratio(
    sum(
        record["financial_consistent"]
        for record in document_manifest_records
    ),
    len(document_manifest_records),
)
document_exact_match = safe_ratio(
    sum(
        record["document_exact_match"]
        for record in document_manifest_records
    ),
    len(document_manifest_records),
)
item_count_exact_match = safe_ratio(
    sum(
        record["item_count_exact"]
        for record in document_manifest_records
    ),
    len(document_manifest_records),
)

overall_metrics = {
    "documents": len(document_manifest_records),
    "templates": len(EXPECTED_TEMPLATES),
    "scalar_field_observations": scalar_total,
    "scalar_field_exact": scalar_exact,
    "all_scalar_exact_match": all_scalar_exact_match,
    "critical_scalar_observations": len(critical_scalar_records),
    "critical_scalar_exact": critical_scalar_exact,
    "critical_scalar_exact_match": critical_scalar_exact_match,
    "scalar_micro_cer": scalar_micro_cer,
    "scalar_micro_wer": scalar_micro_wer,
    "predicted_item_rows": predicted_rows,
    "reference_item_rows": reference_rows,
    "matched_item_rows": row_true_positives,
    "row_precision": row_precision,
    "row_recall": row_recall,
    "row_f1": row_f1,
    "item_field_exact": item_exact_fields,
    "predicted_item_fields": predicted_item_field_count,
    "reference_item_fields": reference_item_field_count,
    "line_item_field_precision": line_item_field_precision,
    "line_item_field_recall": line_item_field_recall,
    "line_item_field_f1": line_item_field_f1,
    "item_count_exact_match": item_count_exact_match,
    "financial_consistency": financial_consistency,
    "document_exact_match": document_exact_match,
}


# ============================================================
# FIELD AND TEMPLATE SUMMARIES
# ============================================================

combined_field_records = all_scalar_records + all_item_field_records
field_summary_records = []

for field_name in sorted(
    {record["field"] for record in combined_field_records}
):
    records = [
        record
        for record in combined_field_records
        if record["field"] == field_name
    ]
    exact_count = sum(record["exact_match"] for record in records)
    character_errors = sum(
        record["character_errors"] for record in records
    )
    reference_characters = sum(
        record["reference_characters"] for record in records
    )
    word_errors = sum(record["word_errors"] for record in records)
    reference_words = sum(
        record["reference_words"] for record in records
    )

    field_summary_records.append(
        {
            "field": field_name,
            "group": records[0]["group"],
            "critical": records[0]["critical"],
            "observations": len(records),
            "exact_matches": exact_count,
            "exact_match_rate": safe_ratio(exact_count, len(records)),
            "character_errors": character_errors,
            "reference_characters": reference_characters,
            "cer": safe_error_rate(
                character_errors,
                reference_characters,
            ),
            "word_errors": word_errors,
            "reference_words": reference_words,
            "wer": safe_error_rate(word_errors, reference_words),
            "status": (
                "PERFECT" if exact_count == len(records) else "HAS_ERRORS"
            ),
        }
    )

template_summary_records = []
for template_id in sorted(EXPECTED_TEMPLATES):
    documents = [
        record
        for record in document_manifest_records
        if record["template_id"] == template_id
    ]
    scalar_records = [
        record
        for record in all_scalar_records
        if record["template_id"] == template_id
    ]
    item_records = [
        record
        for record in all_item_field_records
        if record["template_id"] == template_id
    ]

    scalar_correct = sum(
        record["exact_match"] for record in scalar_records
    )
    item_correct = sum(record["exact_match"] for record in item_records)
    template_predicted_item_fields = sum(
        record["predicted_item_count"] * EXPECTED_ITEM_FIELDS
        for record in documents
    )
    template_reference_item_fields = sum(
        record["reference_item_count"] * EXPECTED_ITEM_FIELDS
        for record in documents
    )
    item_precision = safe_ratio(
        item_correct,
        template_predicted_item_fields,
    )
    item_recall = safe_ratio(
        item_correct,
        template_reference_item_fields,
    )

    template_summary_records.append(
        {
            "template_id": template_id,
            "documents": len(documents),
            "scalar_exact_match": safe_ratio(
                scalar_correct,
                len(scalar_records),
            ),
            "line_item_field_f1": harmonic_mean(
                item_precision,
                item_recall,
            ),
            "item_count_exact_match": safe_ratio(
                sum(record["item_count_exact"] for record in documents),
                len(documents),
            ),
            "financial_consistency": safe_ratio(
                sum(record["financial_consistent"] for record in documents),
                len(documents),
            ),
            "document_exact_match": safe_ratio(
                sum(record["document_exact_match"] for record in documents),
                len(documents),
            ),
        }
    )


# ============================================================
# APPLY PRE-FROZEN ACCEPTANCE THRESHOLDS
# ============================================================

metric_values = {
    "critical_scalar_exact_match": critical_scalar_exact_match,
    "all_scalar_exact_match": all_scalar_exact_match,
    "line_item_field_f1": line_item_field_f1,
    "financial_consistency": financial_consistency,
}

acceptance_checks = []
for metric_name in sorted(required_acceptance_metrics):
    minimum = frozen_acceptance_minima[metric_name]
    actual = metric_values[metric_name]
    acceptance_checks.append(
        {
            "metric": metric_name,
            "minimum": minimum,
            "actual": actual,
            "threshold_source": (
                "DEVELOPMENT_FREEZE_MANIFEST_FROZEN_BEFORE_"
                "VALIDATION_AND_TEST"
            ),
            "status": "PASSED" if actual >= minimum else "FAILED",
        }
    )

acceptance_status = (
    "PASSED"
    if all(check["status"] == "PASSED" for check in acceptance_checks)
    else "FAILED"
)


# ============================================================
# MISMATCH DETAILS AND RESULT DISPLAY
# ============================================================

mismatch_records = []
for record in combined_field_records:
    if record["exact_match"]:
        continue
    mismatch_records.append(
        {
            "document_id": record["document_id"],
            "template_id": record["template_id"],
            "language": record["language"],
            "group": record["group"],
            "field": record["field"],
            "prediction_row": record.get("prediction_row"),
            "reference_row": record.get("reference_row"),
            "expected": record["expected"],
            "predicted": record["predicted"],
            "character_errors": record["character_errors"],
            "word_errors": record["word_errors"],
        }
    )

document_table = pd.DataFrame(runtime_records)
field_table = pd.DataFrame(field_summary_records)
template_table = pd.DataFrame(template_summary_records)
acceptance_table = pd.DataFrame(acceptance_checks)

print("\nTEST ACCEPTANCE RESULTS")
display(acceptance_table)
display(field_table)
display(template_table)
display(
    document_table[
        [
            "sequence_number",
            "document_id",
            "template_id",
            "language",
            "scalar_exact_match",
            "predicted_item_count",
            "reference_item_count",
            "row_f1",
            "item_field_f1",
            "financial_consistent",
            "document_exact_match",
            "execution",
        ]
    ]
)

if mismatch_records:
    print("\nTEST MISMATCH DETAILS")
    display(pd.DataFrame(mismatch_records))


# ============================================================
# IMMUTABLE SUMMARIES AND TEST EVALUATION MANIFEST
# ============================================================

document_summary_action = save_csv_checkpoint(
    DOCUMENT_SUMMARY_PATH,
    pd.DataFrame(document_manifest_records),
)
field_summary_action = save_csv_checkpoint(
    FIELD_SUMMARY_PATH,
    field_table,
)
template_summary_action = save_csv_checkpoint(
    TEMPLATE_SUMMARY_PATH,
    template_table,
)

mismatch_columns = [
    "document_id",
    "template_id",
    "language",
    "group",
    "field",
    "prediction_row",
    "reference_row",
    "expected",
    "predicted",
    "character_errors",
    "word_errors",
]
mismatch_table = pd.DataFrame(
    mismatch_records,
    columns=mismatch_columns,
)
mismatch_action = save_csv_checkpoint(
    MISMATCH_DETAIL_PATH,
    mismatch_table,
)

evaluation_protocol = {
    "evaluator_id": EVALUATOR_ID,
    "evaluator_version": EVALUATOR_VERSION,
    "scalar_matching": "TYPE_AWARE_NORMALIZED_EXACT_MATCH",
    "item_alignment": "MAXIMUM_WEIGHT_BIPARTITE_ROW_MATCHING",
    "row_match": "DESCRIPTION_AND_LINE_TOTAL_EXACT",
    "acceptance_threshold_source": (
        "DEVELOPMENT_FREEZE_MANIFEST_FROZEN_BEFORE_"
        "VALIDATION_AND_TEST"
    ),
    "acceptance_minima": dict(sorted(frozen_acceptance_minima.items())),
}
evaluation_protocol_sha256 = hashlib.sha256(
    canonical_json(evaluation_protocol).encode("utf-8")
).hexdigest()

evaluation_manifest = {
    "schema_version": "1.0.0",
    "cell_version": CELL_VERSION,
    "status": acceptance_status,
    "stage": "TEST_GROUND_TRUTH_EVALUATION",
    "baseline": {
        "baseline_id": EXPECTED_BASELINE_ID,
        "parser_id": EXPECTED_PARSER_ID,
        "parser_version": EXPECTED_PARSER_VERSION,
        "parser_signature_sha256": EXPECTED_PARSER_SIGNATURE,
        "parser_source_path": str(parser_source_path),
        "parser_source_sha256": input_checksums_before[
            "parser_source"
        ],
        "development_freeze_manifest_path": str(freeze_manifest_path),
        "development_freeze_manifest_sha256": input_checksums_before[
            "freeze_manifest"
        ],
        "validation_decision_manifest_path": str(
            validation_decision_path
        ),
        "validation_decision_manifest_sha256": input_checksums_before[
            "validation_decision"
        ],
        "validation_result_sha256": baseline_pointer[
            "validation_result_sha256"
        ],
    },
    "evaluator": {
        **evaluation_protocol,
        "evaluation_protocol_sha256": evaluation_protocol_sha256,
        "development_evaluator_manifest_path": str(
            development_evaluation_path
        ),
        "development_evaluator_manifest_sha256": (
            input_checksums_before["development_evaluation"]
        ),
    },
    "scope": {
        "split": "test",
        "documents": EXPECTED_DOCUMENTS,
        "templates": sorted(EXPECTED_TEMPLATES),
        "test_ground_truth_opened": len(ground_truth_paths),
        "validation_ground_truth_reopened": 0,
    },
    "overall_metrics": overall_metrics,
    "acceptance_checks": acceptance_checks,
    "technical_controls": technical_control_records,
    "field_summary": field_summary_records,
    "template_summary": template_summary_records,
    "records": document_manifest_records,
    "artifacts": {
        "evaluation_root": str(EVALUATION_ROOT),
        "document_summary": {
            "path": str(DOCUMENT_SUMMARY_PATH),
            "sha256": sha256_file(DOCUMENT_SUMMARY_PATH),
        },
        "field_summary": {
            "path": str(FIELD_SUMMARY_PATH),
            "sha256": sha256_file(FIELD_SUMMARY_PATH),
        },
        "template_summary": {
            "path": str(TEMPLATE_SUMMARY_PATH),
            "sha256": sha256_file(TEMPLATE_SUMMARY_PATH),
        },
        "mismatch_details": {
            "path": str(MISMATCH_DETAIL_PATH),
            "sha256": sha256_file(MISMATCH_DETAIL_PATH),
            "records": len(mismatch_records),
        },
    },
    "inputs": {
        "contract": {
            "path": str(CONTRACT_PATH),
            "sha256": input_checksums_before["contract"],
        },
        "baseline_pointer": {
            "path": str(BASELINE_POINTER_PATH),
            "sha256": input_checksums_before["baseline_pointer"],
        },
        "development_freeze_manifest": {
            "path": str(freeze_manifest_path),
            "sha256": input_checksums_before["freeze_manifest"],
        },
        "validation_decision": {
            "path": str(validation_decision_path),
            "sha256": input_checksums_before["validation_decision"],
        },
        "validation_evaluation": {
            "path": str(validation_evaluation_path),
            "sha256": input_checksums_before["validation_evaluation"],
        },
        "parser_source": {
            "path": str(parser_source_path),
            "sha256": input_checksums_before["parser_source"],
        },
        "development_evaluation": {
            "path": str(development_evaluation_path),
            "sha256": input_checksums_before[
                "development_evaluation"
            ],
        },
        "prediction_manifest": {
            "path": str(PREDICTION_MANIFEST_PATH),
            "sha256": input_checksums_before["prediction_manifest"],
        },
        "render_index": {
            "path": str(RENDER_INDEX_PATH),
            "sha256": input_checksums_before["render_index"],
        },
        "prediction_files": {
            record["document_id"]: {
                "path": record["prediction_path"],
                "sha256": record["prediction_sha256"],
            }
            for record in document_manifest_records
        },
        "test_ground_truth_files": {
            document_id: {
                "path": str(ground_truth_paths[document_id]),
                "sha256": ground_truth_checksums[document_id],
            }
            for document_id in sorted(ground_truth_paths)
        },
    },
    "integrity": {
        "predictions_frozen_before_ground_truth_open": True,
        "test_ground_truth_loaded_for_evaluation": True,
        "ground_truth_used_for_evaluation_only": True,
        "ground_truth_used_as_prediction_input": False,
        "test_ground_truth_opened": len(ground_truth_paths),
        "validation_ground_truth_reopened": 0,
        "parser_modifications": 0,
        "prediction_modifications": 0,
        "ground_truth_modifications": 0,
        "dataset_modifications": 0,
        "source_modifications": 0,
    },
    "next_stage": {
        "cell": "CELL 12E",
        "action_if_passed": (
            "FREEZE_FINAL_PASSED_TEST_RESULT_AND_COMPLETE_BENCHMARK"
        ),
        "action_if_failed": (
            "FREEZE_FINAL_FAILED_TEST_RESULT_AND_COMPLETE_BENCHMARK"
        ),
        "silent_parser_changes_allowed": False,
        "test_result_is_final": True,
        "test_split_may_not_be_reused_for_tuning": True,
    },
}

manifest_action = save_immutable_json(
    EVALUATION_MANIFEST_PATH,
    evaluation_manifest,
)

persisted_manifest = load_json(EVALUATION_MANIFEST_PATH)
if canonical_json(persisted_manifest) != canonical_json(
    evaluation_manifest
):
    raise RuntimeError("Manifest Cell 12D berbeda setelah penulisan.")


# ============================================================
# FINAL INPUT AND GROUND-TRUTH IMMUTABILITY CHECK
# ============================================================

input_checksums_after = {
    "contract": sha256_file(CONTRACT_PATH),
    "baseline_pointer": sha256_file(BASELINE_POINTER_PATH),
    "freeze_manifest": sha256_file(freeze_manifest_path),
    "validation_decision": sha256_file(validation_decision_path),
    "validation_evaluation": sha256_file(validation_evaluation_path),
    "parser_source": sha256_file(parser_source_path),
    "prediction_manifest": sha256_file(PREDICTION_MANIFEST_PATH),
    "render_index": sha256_file(RENDER_INDEX_PATH),
    "development_evaluation": sha256_file(
        development_evaluation_path
    ),
}
for prediction_record in prediction_records:
    input_checksums_after[
        f"prediction:{prediction_record['document_id']}"
    ] = sha256_file(Path(prediction_record["prediction_path"]))

changed_inputs = [
    name
    for name, checksum in input_checksums_before.items()
    if input_checksums_after.get(name) != checksum
]
changed_ground_truth = [
    document_id
    for document_id, checksum in ground_truth_checksums.items()
    if sha256_file(ground_truth_paths[document_id]) != checksum
]

if changed_inputs or changed_ground_truth:
    raise RuntimeError(
        "Input evaluation berubah selama Cell 12D: "
        f"inputs={changed_inputs}, "
        f"ground_truth={changed_ground_truth}"
    )


print()
print(f"Cell version             : {CELL_VERSION}")
print(f"Evaluation status        : {acceptance_status}")
print(f"Documents evaluated      : {len(document_manifest_records)}")
print(f"Templates                : {len(EXPECTED_TEMPLATES)}")
print(f"All scalar exact match   : {all_scalar_exact_match:.6f}")
print(f"Critical scalar exact    : {critical_scalar_exact_match:.6f}")
print(f"Scalar micro CER         : {scalar_micro_cer:.6f}")
print(f"Scalar micro WER         : {scalar_micro_wer:.6f}")
print(f"Row precision            : {row_precision:.6f}")
print(f"Row recall               : {row_recall:.6f}")
print(f"Row F1                   : {row_f1:.6f}")
print(f"Line-item field F1       : {line_item_field_f1:.6f}")
print(f"Item-count exact match   : {item_count_exact_match:.6f}")
print(f"Financial consistency    : {financial_consistency:.6f}")
print(f"Document exact match     : {document_exact_match:.6f}")
print(f"Mismatch records         : {len(mismatch_records)}")
print(f"New evaluations          : {new_evaluations}")
print(f"Recovered evaluations    : {recovered_evaluations}")
print(f"Document summary action  : {document_summary_action}")
print(f"Field summary action     : {field_summary_action}")
print(f"Template summary action  : {template_summary_action}")
print(f"Mismatch file action     : {mismatch_action}")
print(f"Manifest action          : {manifest_action}")
print(f"Evaluation root          : {EVALUATION_ROOT}")
print(f"Evaluation manifest      : {EVALUATION_MANIFEST_PATH}")
print(f"Manifest SHA-256         : {sha256_file(EVALUATION_MANIFEST_PATH)}")
print(f"Test GT opened            : {len(ground_truth_paths)}")
print("Validation GT reopened    : 0")
print("Parser modifications     : 0")
print("Prediction modifications : 0")
print("Ground-truth modifications: 0")
print("Dataset modifications    : 0")
print("Source modifications     : 0")

if acceptance_status != "PASSED":
    failed_metrics = [
        check["metric"]
        for check in acceptance_checks
        if check["status"] != "PASSED"
    ]
    raise RuntimeError(
        "TEST BASELINE BELOW PRE-FROZEN ACCEPTANCE TARGET. "
        f"Metrik gagal: {failed_metrics}. "
        "Hasil FAILED sudah disimpan secara immutable; parser v1.0.2 "
        "tidak boleh diubah diam-diam dan test split tidak boleh "
        "digunakan ulang untuk tuning. Lanjutkan ke Cell 12E untuk "
        "membekukan keputusan final."
    )

print()
print(
    "✅ CELL 12D PASSED — frozen parser v1.0.2 memenuhi seluruh "
    "pre-frozen acceptance target pada 40 dokumen test. "
    "Test ground truth hanya digunakan untuk evaluasi. Hasil ini final; "
    "lanjutkan ke Cell 12E untuk membekukan keputusan akhir benchmark."
)


CELL 12D — INVOICE-FIELD-EVALUATOR-V1 — VERSION 1.0.2 — TEST EVALUATION
Evaluation root: /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/field_extraction/test/rule_based_baseline_v1_0_2/evaluations/test_eval_v1_0_2
Predictions: FROZEN | Test GT: AUTHORIZED FOR EVALUATION ONLY
PREFLIGHT CONTROLS — BEFORE TEST GROUND TRUTH IS OPENED


,control,expected,actual,status
0,contract_status,FROZEN_FOR_DEVELOPMENT_BASELINE,FROZEN_FOR_DEVELOPMENT_BASELINE,VALID
1,target_fields,16,16,VALID
2,target_field_names,"[buyer.name, buyer.tax_identifier, currency, d...","[buyer.name, buyer.tax_identifier, currency, d...",VALID
3,render_index_records,200,200,VALID
4,baseline_pointer_status,ACTIVE_FOR_TEST_PREFLIGHT,ACTIVE_FOR_TEST_PREFLIGHT,VALID
...,...,...,...,...
63,frozen_acceptance_checks_passed,True,True,VALID
64,development_evaluation_status,PASSED,PASSED,VALID
65,development_evaluator_id,INVOICE-FIELD-EVALUATOR-V1,INVOICE-FIELD-EVALUATOR-V1,VALID
66,development_evaluator_version,1.0.2,1.0.2,VALID



FROZEN PREDICTION CONTROLS


,control,expected,actual,status
0,prediction_records,40,40,VALID
1,unique_prediction_ids,40,40,VALID
2,prediction_templates,"[TPL-09, TPL-10]","[TPL-09, TPL-10]",VALID
3,prediction_sequence_numbers,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",VALID
4,frozen_prediction_records,40,40,VALID
5,prediction_source_errors,0,0,VALID



TEST GROUND-TRUTH ACCESS PLAN


...

,sequence_number,document_id,template_id,language,scalar_exact_match,predicted_item_count,reference_item_count,row_f1,item_field_f1,financial_consistent,document_exact_match,execution
0,1,INV-SYN-000161,TPL-09,id,1.000000,5,5,1.0,1.0,True,True,NEW
1,2,INV-SYN-000162,TPL-09,id,1.000000,3,3,1.0,1.0,True,True,NEW
2,3,INV-SYN-000163,TPL-09,id,1.000000,8,8,1.0,1.0,True,True,NEW
3,4,INV-SYN-000164,TPL-09,id,1.000000,5,5,1.0,1.0,True,True,NEW
4,5,INV-SYN-000165,TPL-09,id,1.000000,7,7,1.0,1.0,True,True,NEW
5,6,INV-SYN-000166,TPL-09,id,1.000000,7,7,1.0,1.0,True,True,NEW
6,7,INV-SYN-000167,TPL-09,id,1.000000,4,4,1.0,1.0,True,True,NEW
7,8,INV-SYN-000168,TPL-09,id,1.000000,5,5,1.0,1.0,True,True,NEW
8,9,INV-SYN-000169,TPL-09,id,1.000000,3,3,1.0,1.0,True,True,NEW
9,10,INV-SYN-000170,TPL-09,id,1.000000,7,7,1.0,1.0,True,True,NEW



TEST MISMATCH DETAILS


,document_id,template_id,language,group,field,prediction_row,reference_row,expected,predicted,character_errors,word_errors
0,INV-SYN-000181,TPL-10,id,scalar,vendor.name,None,None,CV Pijaraya Niaga Contoh,,24,4
1,INV-SYN-000181,TPL-10,id,scalar,buyer.name,None,None,CV Kiranusa Teknologi Uji,,25,4
2,INV-SYN-000182,TPL-10,id,scalar,vendor.name,None,None,PT Widyakara Kreasi Simulasi,,28,4
3,INV-SYN-000182,TPL-10,id,scalar,buyer.name,None,None,PT Lenterasa Logistik Uji,,25,4
4,INV-SYN-000183,TPL-10,id,scalar,vendor.name,None,None,CV Arunika Logistik Uji,,23,4
5,INV-SYN-000183,TPL-10,id,scalar,buyer.name,None,None,PT Cakrawana Teknologi Uji,,26,4
6,INV-SYN-000184,TPL-10,id,scalar,vendor.name,None,None,PT Kiranusa Distribusi Contoh,,29,4
7,INV-SYN-000184,TPL-10,id,scalar,buyer.name,None,None,CV Swaranusa Solusi Simulasi,,28,4
8,INV-SYN-000185,TPL-10,id,scalar,vendor.name,None,None,CV Pijaraya Niaga Contoh,,24,4
9,INV-SYN-000185,TPL-10,id,scalar,buyer.name,None,None,CV Kiranusa Teknologi Uji,,25,4



Cell version             : 1.0.0
Evaluation status        : FAILED
Documents evaluated      : 40
Templates                : 2
All scalar exact match   : 0.916667
Critical scalar exact    : 0.900000
Scalar micro CER         : 0.163233
Scalar micro WER         : 0.222222
Row precision            : 1.000000
Row recall               : 1.000000
Row F1                   : 1.000000
Line-item field F1       : 1.000000
Item-count exact match   : 1.000000
Financial consistency    : 1.000000
Document exact match     : 0.500000
Mismatch records         : 40
New evaluations          : 40
Recovered evaluations    : 0
Document summary action  : CREATED
Field summary action     : CREATED
Template summary action  : CREATED
Mismatch file action     : CREATED
Manifest action          : CREATED
Evaluation root          : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/field_extraction/test/rule_based_baseline_v1_0_2/evaluations/test_eval_v1_0_2
Evaluat

RuntimeError: TEST BASELINE BELOW PRE-FROZEN ACCEPTANCE TARGET. Metrik gagal: ['all_scalar_exact_match', 'critical_scalar_exact_match']. Hasil FAILED sudah disimpan secara immutable; parser v1.0.2 tidak boleh diubah diam-diam dan test split tidak boleh digunakan ulang untuk tuning. Lanjutkan ke Cell 12E untuk membekukan keputusan final.

In [9]:
from __future__ import annotations

import hashlib
import json
import os
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from IPython.display import display


# ============================================================
# CELL 12E — FREEZE FINAL TEST DECISION AND COMPLETE BENCHMARK
#             (NO GROUND-TRUTH PAYLOAD IS REOPENED)
# ============================================================

CELL_VERSION = "1.0.0"

DATA_ROOT = Path("/content/drive/MyDrive/InvoiceFlow-AI-Data")
BUILD_ROOT = (
    DATA_ROOT
    / "interim"
    / "synthetic_v1_build"
    / "20260904T150025Z"
)
FIELD_ROOT = BUILD_ROOT / "ocr_benchmark" / "field_extraction"
TEST_ROOT = FIELD_ROOT / "test" / "rule_based_baseline_v1_0_2"

TEST_BASELINE_POINTER_PATH = FIELD_ROOT / "test_baseline_pointer.json"
PREDICTION_MANIFEST_PATH = TEST_ROOT / "test_prediction_manifest.json"
EVALUATION_ROOT = TEST_ROOT / "evaluations" / "test_eval_v1_0_2"
EVALUATION_MANIFEST_PATH = (
    EVALUATION_ROOT / "test_evaluation_manifest.json"
)

FINAL_DECISION_ROOT = (
    FIELD_ROOT
    / "frozen_test_decisions"
    / "rule_based_baseline_v1_0_2"
)
FINAL_DECISION_MANIFEST_PATH = (
    FINAL_DECISION_ROOT / "final_test_decision_manifest.json"
)
BENCHMARK_COMPLETION_POINTER_PATH = (
    FIELD_ROOT / "benchmark_completion_pointer.json"
)

EXPECTED_BASELINE_ID = "RULE-BASED-INVOICE-PARSER-V1@1.0.2"
EXPECTED_PARSER_ID = "RULE-BASED-INVOICE-PARSER-V1"
EXPECTED_PARSER_VERSION = "1.0.2"
EXPECTED_PARSER_SIGNATURE = (
    "ffe04f258fb3b1a6a55661698fa6c7dda29ddd7b714c2f210b3e6fc1fcd464b3"
)
EXPECTED_EVALUATOR_ID = "INVOICE-FIELD-EVALUATOR-V1"
EXPECTED_EVALUATOR_VERSION = "1.0.2"
EXPECTED_DOCUMENTS = 40
EXPECTED_TEMPLATES = {"TPL-09", "TPL-10"}
EXPECTED_ACCEPTANCE_METRICS = {
    "critical_scalar_exact_match",
    "all_scalar_exact_match",
    "line_item_field_f1",
    "financial_consistency",
}


# ============================================================
# FILE, HASH, AND IMMUTABLE-WRITE HELPERS
# ============================================================

def require_file(path: Path, label: str) -> None:
    if not path.is_file():
        raise FileNotFoundError(f"{label} tidak ditemukan: {path}")
    if path.stat().st_size <= 0:
        raise RuntimeError(f"{label} kosong: {path}")


def load_json(path: Path) -> dict:
    require_file(path, "JSON artifact")
    with path.open("r", encoding="utf-8") as file_handle:
        value = json.load(file_handle)
    if not isinstance(value, dict):
        raise TypeError(f"Root JSON bukan object: {path}")
    return value


def canonical_json(value: object) -> str:
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    )


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as file_handle:
        for chunk in iter(lambda: file_handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def sha256_object(value: object) -> str:
    return hashlib.sha256(
        canonical_json(value).encode("utf-8")
    ).hexdigest()


def atomic_write_json(path: Path, value: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(
        f".{path.name}.{os.getpid()}.tmp"
    )
    try:
        temporary_path.write_text(
            json.dumps(
                value,
                indent=2,
                ensure_ascii=False,
                default=str,
            )
            + "\n",
            encoding="utf-8",
        )
        os.replace(temporary_path, path)
    finally:
        if temporary_path.exists():
            temporary_path.unlink()


def save_immutable_json(path: Path, value: dict) -> str:
    if path.exists():
        existing = load_json(path)
        if canonical_json(existing) != canonical_json(value):
            raise RuntimeError(
                f"Checkpoint sudah ada tetapi berbeda: {path}"
            )
        return "RECOVERED"
    atomic_write_json(path, value)
    return "CREATED"


def path_is_inside(path: Path, root: Path) -> bool:
    try:
        path.resolve().relative_to(root.resolve())
        return True
    except ValueError:
        return False


def close_enough(first: object, second: object) -> bool:
    try:
        return abs(float(first) - float(second)) <= 1e-12
    except (TypeError, ValueError):
        return False


def safe_ratio(numerator: int | float, denominator: int | float) -> float:
    if denominator == 0:
        return 1.0 if numerator == 0 else 0.0
    return float(numerator) / float(denominator)


def safe_error_rate(
    errors: int | float,
    reference_units: int | float,
) -> float:
    if reference_units == 0:
        return 0.0 if errors == 0 else 1.0
    return float(errors) / float(reference_units)


def harmonic_mean(precision: float, recall: float) -> float:
    if precision + recall == 0:
        return 0.0
    return 2.0 * precision * recall / (precision + recall)


# ============================================================
# PREFLIGHT — USE FROZEN EVALUATION EVIDENCE ONLY
# ============================================================

for required_path, label in (
    (TEST_BASELINE_POINTER_PATH, "Test baseline pointer"),
    (PREDICTION_MANIFEST_PATH, "Frozen test prediction manifest"),
    (EVALUATION_MANIFEST_PATH, "Test evaluation manifest"),
):
    require_file(required_path, label)

baseline_pointer = load_json(TEST_BASELINE_POINTER_PATH)
prediction_manifest = load_json(PREDICTION_MANIFEST_PATH)
evaluation_manifest = load_json(EVALUATION_MANIFEST_PATH)

freeze_manifest_path = Path(
    str(baseline_pointer.get("development_freeze_manifest_path", ""))
)
validation_decision_path = Path(
    str(baseline_pointer.get("validation_decision_manifest_path", ""))
)
parser_source_path = Path(
    str(baseline_pointer.get("parser_source_path", ""))
)
for required_path, label in (
    (freeze_manifest_path, "Development freeze manifest"),
    (validation_decision_path, "Frozen validation decision"),
    (parser_source_path, "Frozen parser source"),
):
    require_file(required_path, label)

freeze_manifest = load_json(freeze_manifest_path)
validation_decision = load_json(validation_decision_path)

source_checksums_before = {
    "test_baseline_pointer": sha256_file(TEST_BASELINE_POINTER_PATH),
    "prediction_manifest": sha256_file(PREDICTION_MANIFEST_PATH),
    "evaluation_manifest": sha256_file(EVALUATION_MANIFEST_PATH),
    "development_freeze": sha256_file(freeze_manifest_path),
    "validation_decision": sha256_file(validation_decision_path),
    "parser_source": sha256_file(parser_source_path),
}

prediction_parser = prediction_manifest.get("parser", {})
prediction_scope = prediction_manifest.get("scope", {})
prediction_integrity = prediction_manifest.get("integrity", {})
prediction_records = prediction_manifest.get("records", [])

evaluation_baseline = evaluation_manifest.get("baseline", {})
evaluation_evaluator = evaluation_manifest.get("evaluator", {})
evaluation_scope = evaluation_manifest.get("scope", {})
evaluation_integrity = evaluation_manifest.get("integrity", {})
evaluation_next_stage = evaluation_manifest.get("next_stage", {})
evaluation_records = evaluation_manifest.get("records", [])
overall_metrics = evaluation_manifest.get("overall_metrics", {})
acceptance_checks = evaluation_manifest.get("acceptance_checks", [])
technical_controls = evaluation_manifest.get("technical_controls", [])
artifacts = evaluation_manifest.get("artifacts", {})

for value, label in (
    (prediction_records, "prediction records"),
    (evaluation_records, "evaluation records"),
    (acceptance_checks, "acceptance checks"),
    (technical_controls, "technical controls"),
):
    if not isinstance(value, list):
        raise TypeError(f"{label} bukan list.")

reported_acceptance_status = (
    "PASSED"
    if acceptance_checks
    and all(row.get("status") == "PASSED" for row in acceptance_checks)
    else "FAILED"
)

preflight_values = [
    (
        "baseline_pointer_status",
        "ACTIVE_FOR_TEST_PREFLIGHT",
        baseline_pointer.get("status"),
    ),
    (
        "baseline_id",
        EXPECTED_BASELINE_ID,
        baseline_pointer.get("baseline_id"),
    ),
    (
        "parser_id",
        EXPECTED_PARSER_ID,
        baseline_pointer.get("parser_id"),
    ),
    (
        "parser_version",
        EXPECTED_PARSER_VERSION,
        baseline_pointer.get("parser_version"),
    ),
    (
        "parser_signature",
        EXPECTED_PARSER_SIGNATURE,
        baseline_pointer.get("parser_signature_sha256"),
    ),
    (
        "parser_source_checksum",
        baseline_pointer.get("parser_source_sha256"),
        source_checksums_before["parser_source"],
    ),
    (
        "development_freeze_checksum",
        baseline_pointer.get("development_freeze_manifest_sha256"),
        source_checksums_before["development_freeze"],
    ),
    (
        "validation_decision_checksum",
        baseline_pointer.get("validation_decision_manifest_sha256"),
        source_checksums_before["validation_decision"],
    ),
    (
        "validation_decision_status",
        "FROZEN_FOR_TEST_PREFLIGHT",
        validation_decision.get("status"),
    ),
    (
        "validation_decision",
        "VALIDATION_PASSED_BASELINE_ACCEPTED",
        validation_decision.get("decision"),
    ),
    (
        "prediction_manifest_status",
        "EXECUTED",
        prediction_manifest.get("status"),
    ),
    (
        "prediction_freeze_status",
        "FROZEN_BEFORE_TEST_GROUND_TRUTH_OPEN",
        prediction_manifest.get("prediction_freeze_status"),
    ),
    (
        "prediction_quality_status",
        "PENDING_TEST_GROUND_TRUTH_EVALUATION",
        prediction_manifest.get("quality_status"),
    ),
    (
        "prediction_parser_id",
        EXPECTED_PARSER_ID,
        prediction_parser.get("parser_id"),
    ),
    (
        "prediction_parser_version",
        EXPECTED_PARSER_VERSION,
        prediction_parser.get("parser_version"),
    ),
    (
        "prediction_parser_signature",
        EXPECTED_PARSER_SIGNATURE,
        prediction_parser.get("parser_signature_sha256"),
    ),
    (
        "prediction_documents",
        EXPECTED_DOCUMENTS,
        prediction_scope.get("documents"),
    ),
    (
        "prediction_templates",
        sorted(EXPECTED_TEMPLATES),
        sorted(prediction_scope.get("templates", [])),
    ),
    (
        "predictions_frozen",
        EXPECTED_DOCUMENTS,
        prediction_scope.get("predictions_frozen"),
    ),
    (
        "predictions_frozen_before_test_gt",
        True,
        prediction_integrity.get(
            "test_predictions_frozen_before_ground_truth_open"
        ),
    ),
    (
        "prediction_test_gt_opened",
        0,
        prediction_integrity.get("test_ground_truth_opened"),
    ),
    (
        "prediction_validation_gt_reopened",
        0,
        prediction_integrity.get("validation_ground_truth_reopened"),
    ),
    (
        "evaluation_status",
        reported_acceptance_status,
        evaluation_manifest.get("status"),
    ),
    (
        "evaluation_stage",
        "TEST_GROUND_TRUTH_EVALUATION",
        evaluation_manifest.get("stage"),
    ),
    (
        "evaluation_baseline_id",
        EXPECTED_BASELINE_ID,
        evaluation_baseline.get("baseline_id"),
    ),
    (
        "evaluation_parser_signature",
        EXPECTED_PARSER_SIGNATURE,
        evaluation_baseline.get("parser_signature_sha256"),
    ),
    (
        "evaluator_id",
        EXPECTED_EVALUATOR_ID,
        evaluation_evaluator.get("evaluator_id"),
    ),
    (
        "evaluator_version",
        EXPECTED_EVALUATOR_VERSION,
        evaluation_evaluator.get("evaluator_version"),
    ),
    ("evaluation_split", "test", evaluation_scope.get("split")),
    (
        "evaluated_documents",
        EXPECTED_DOCUMENTS,
        evaluation_scope.get("documents"),
    ),
    (
        "evaluated_templates",
        sorted(EXPECTED_TEMPLATES),
        sorted(evaluation_scope.get("templates", [])),
    ),
    (
        "test_ground_truth_opened_in_12d",
        EXPECTED_DOCUMENTS,
        evaluation_scope.get("test_ground_truth_opened"),
    ),
    (
        "validation_ground_truth_reopened_in_12d",
        0,
        evaluation_scope.get("validation_ground_truth_reopened"),
    ),
    (
        "evaluation_predictions_frozen_before_gt",
        True,
        evaluation_integrity.get(
            "predictions_frozen_before_ground_truth_open"
        ),
    ),
    (
        "test_gt_used_for_evaluation_only",
        True,
        evaluation_integrity.get("ground_truth_used_for_evaluation_only"),
    ),
    (
        "test_gt_used_as_prediction_input",
        False,
        evaluation_integrity.get("ground_truth_used_as_prediction_input"),
    ),
    (
        "evaluation_parser_modifications",
        0,
        evaluation_integrity.get("parser_modifications"),
    ),
    (
        "evaluation_prediction_modifications",
        0,
        evaluation_integrity.get("prediction_modifications"),
    ),
    (
        "evaluation_ground_truth_modifications",
        0,
        evaluation_integrity.get("ground_truth_modifications"),
    ),
    (
        "evaluation_dataset_modifications",
        0,
        evaluation_integrity.get("dataset_modifications"),
    ),
    (
        "evaluation_next_cell",
        "CELL 12E",
        evaluation_next_stage.get("cell"),
    ),
    (
        "test_result_is_final",
        True,
        evaluation_next_stage.get("test_result_is_final"),
    ),
    (
        "test_split_may_not_be_reused_for_tuning",
        True,
        evaluation_next_stage.get(
            "test_split_may_not_be_reused_for_tuning"
        ),
    ),
    (
        "technical_controls_passed",
        True,
        bool(technical_controls)
        and all(row.get("status") == "VALID" for row in technical_controls),
    ),
    ("test_ground_truth_reopened_in_12e", 0, 0),
    ("validation_ground_truth_reopened_in_12e", 0, 0),
]

preflight_controls = [
    {
        "control": name,
        "expected": expected,
        "actual": actual,
        "status": "VALID" if expected == actual else "INVALID",
    }
    for name, expected, actual in preflight_values
]

print("=" * 96)
print(
    f"CELL 12E — FINAL TEST DECISION FREEZE — VERSION {CELL_VERSION}"
)
print(f"Decision root: {FINAL_DECISION_ROOT}")
print(
    f"Test result: {evaluation_manifest.get('status')} | "
    "Test GT reopen: 0 | Benchmark completion: PENDING"
)
print("=" * 96)
print("FINAL TEST DECISION PREFLIGHT")
display(pd.DataFrame(preflight_controls))

invalid_preflight = [
    row["control"]
    for row in preflight_controls
    if row["status"] != "VALID"
]
if invalid_preflight:
    raise RuntimeError(
        "CELL 12E PREFLIGHT FAILED. "
        f"Kontrol tidak valid: {invalid_preflight}. "
        "Ground truth test tidak dibuka kembali."
    )


# ============================================================
# VERIFY IMMUTABLE DOCUMENT EVALUATIONS AND PREDICTIONS
# Ground-truth paths/checksums are read as metadata only.
# ============================================================

prediction_map = {
    str(record.get("document_id")): record
    for record in prediction_records
    if isinstance(record, dict) and record.get("document_id")
}
verified_records = []
scalar_records = []
item_field_records = []
artifact_errors = []

for record_index, record in enumerate(evaluation_records, start=1):
    try:
        if not isinstance(record, dict):
            raise TypeError("Evaluation record bukan object.")

        document_id = str(record.get("document_id", ""))
        template_id = str(record.get("template_id", ""))
        evaluation_path = Path(str(record.get("evaluation_path", "")))
        prediction_path = Path(str(record.get("prediction_path", "")))

        if not document_id:
            raise RuntimeError("document_id kosong.")
        if template_id not in EXPECTED_TEMPLATES:
            raise RuntimeError(f"Template tidak valid: {template_id}")
        if not path_is_inside(
            evaluation_path,
            EVALUATION_ROOT / "documents",
        ):
            raise RuntimeError("Evaluation path di luar root resmi.")
        if not path_is_inside(prediction_path, TEST_ROOT / "predictions"):
            raise RuntimeError("Prediction path di luar root resmi.")

        require_file(evaluation_path, f"Evaluation {document_id}")
        require_file(prediction_path, f"Prediction {document_id}")
        evaluation_sha256 = sha256_file(evaluation_path)
        prediction_sha256 = sha256_file(prediction_path)
        if evaluation_sha256 != record.get("evaluation_sha256"):
            raise RuntimeError("Checksum evaluation tidak cocok.")
        if prediction_sha256 != record.get("prediction_sha256"):
            raise RuntimeError("Checksum prediction tidak cocok.")

        prediction_record = prediction_map.get(document_id)
        if prediction_record is None:
            raise RuntimeError("Prediction manifest record tidak ditemukan.")
        if prediction_record.get("prediction_sha256") != prediction_sha256:
            raise RuntimeError("Checksum prediction antar-manifest berbeda.")
        if prediction_record.get("prediction_freeze_status") != (
            "FROZEN_BEFORE_TEST_GROUND_TRUTH_OPEN"
        ):
            raise RuntimeError("Prediction record tidak frozen.")

        evaluation = load_json(evaluation_path)
        document = evaluation.get("document", {})
        inputs = evaluation.get("inputs", {})
        integrity = evaluation.get("integrity", {})
        metrics = evaluation.get("metrics", {})
        current_scalars = evaluation.get("scalar_comparisons", [])
        current_items = evaluation.get("item_field_comparisons", [])

        if evaluation.get("status") != "EVALUATED":
            raise RuntimeError("Status document evaluation tidak valid.")
        if document.get("document_id") != document_id:
            raise RuntimeError("Document ID evaluation tidak cocok.")
        if document.get("template_id") != template_id:
            raise RuntimeError("Template evaluation tidak cocok.")
        if document.get("split") != "test":
            raise RuntimeError("Evaluation bukan test split.")
        if inputs.get("prediction_sha256") != prediction_sha256:
            raise RuntimeError("Input prediction checksum tidak cocok.")
        if inputs.get("ground_truth_sha256") != record.get(
            "ground_truth_sha256"
        ):
            raise RuntimeError("Ground-truth metadata checksum berbeda.")
        if integrity.get(
            "predictions_frozen_before_ground_truth_open"
        ) is not True:
            raise RuntimeError("Freeze prediction tidak terbukti.")
        if integrity.get("ground_truth_used_for_evaluation_only") is not True:
            raise RuntimeError("Ground truth tidak dibatasi untuk evaluasi.")
        if integrity.get("ground_truth_used_as_prediction_input") is not False:
            raise RuntimeError("Ground truth terindikasi sebagai input parser.")
        if integrity.get("test_ground_truth_opened_in_this_evaluation") != 1:
            raise RuntimeError("Audit akses test GT tidak valid.")
        if integrity.get("validation_ground_truth_reopened") != 0:
            raise RuntimeError("Validation GT terindikasi dibuka kembali.")
        if not isinstance(current_scalars, list):
            raise TypeError("scalar_comparisons bukan list.")
        if not isinstance(current_items, list):
            raise TypeError("item_field_comparisons bukan list.")

        metric_record_pairs = {
            "scalar_exact_match": "scalar_exact_match",
            "predicted_item_count": "predicted_item_count",
            "reference_item_count": "reference_item_count",
            "item_count_exact": "item_count_exact",
            "row_precision": "row_precision",
            "row_recall": "row_recall",
            "row_f1": "row_f1",
            "item_field_precision": "item_field_precision",
            "item_field_recall": "item_field_recall",
            "item_field_f1": "item_field_f1",
            "financial_consistent": "financial_consistent",
            "document_exact_match": "document_exact_match",
        }
        for metric_key, record_key in metric_record_pairs.items():
            expected = record.get(record_key)
            actual = metrics.get(metric_key)
            if isinstance(expected, float) or isinstance(actual, float):
                matches = close_enough(expected, actual)
            else:
                matches = expected == actual
            if not matches:
                raise RuntimeError(
                    f"Metric record/artifact berbeda: {metric_key}"
                )

        source_checksums_before[
            f"evaluation:{document_id}"
        ] = evaluation_sha256
        source_checksums_before[
            f"prediction:{document_id}"
        ] = prediction_sha256
        verified_records.append(record)
        scalar_records.extend(current_scalars)
        item_field_records.extend(current_items)

    except Exception as error:
        artifact_errors.append(
            {
                "record_index": record_index,
                "document_id": (
                    record.get("document_id")
                    if isinstance(record, dict)
                    else None
                ),
                "error_type": type(error).__name__,
                "error": str(error)[:800],
            }
        )

artifact_values = [
    ("evaluation_records", EXPECTED_DOCUMENTS, len(evaluation_records)),
    ("verified_evaluations", EXPECTED_DOCUMENTS, len(verified_records)),
    (
        "unique_evaluation_ids",
        EXPECTED_DOCUMENTS,
        len({record.get("document_id") for record in verified_records}),
    ),
    (
        "evaluation_templates",
        sorted(EXPECTED_TEMPLATES),
        sorted({record.get("template_id") for record in verified_records}),
    ),
    ("prediction_records", EXPECTED_DOCUMENTS, len(prediction_records)),
    ("artifact_errors", 0, len(artifact_errors)),
    ("test_ground_truth_reopened", 0, 0),
    ("validation_ground_truth_reopened", 0, 0),
]
artifact_controls = [
    {
        "control": name,
        "expected": expected,
        "actual": actual,
        "status": "VALID" if expected == actual else "INVALID",
    }
    for name, expected, actual in artifact_values
]

print("\nIMMUTABLE TEST ARTIFACT CONTROLS")
display(pd.DataFrame(artifact_controls))
if artifact_errors:
    display(pd.DataFrame(artifact_errors))

invalid_artifact_controls = [
    row["control"]
    for row in artifact_controls
    if row["status"] != "VALID"
]
if invalid_artifact_controls:
    raise RuntimeError(
        "CELL 12E ARTIFACT GATE FAILED. "
        f"Kontrol tidak valid: {invalid_artifact_controls}."
    )


# ============================================================
# VERIFY SUMMARY FILES AND RECOMPUTE TEST METRICS
# ============================================================

summary_errors = []
summary_tables = {}
for artifact_name in (
    "document_summary",
    "field_summary",
    "template_summary",
    "mismatch_details",
):
    try:
        artifact = artifacts.get(artifact_name, {})
        artifact_path = Path(str(artifact.get("path", "")))
        if not path_is_inside(artifact_path, EVALUATION_ROOT):
            raise RuntimeError("Summary path di luar evaluation root.")
        require_file(artifact_path, artifact_name)
        if sha256_file(artifact_path) != artifact.get("sha256"):
            raise RuntimeError("Checksum summary tidak cocok.")
        summary_tables[artifact_name] = pd.read_csv(artifact_path)
        source_checksums_before[
            f"summary:{artifact_name}"
        ] = sha256_file(artifact_path)
    except Exception as error:
        summary_errors.append(
            {
                "artifact": artifact_name,
                "error_type": type(error).__name__,
                "error": str(error)[:700],
            }
        )

scalar_total = len(scalar_records)
scalar_exact = sum(bool(row.get("exact_match")) for row in scalar_records)
critical_scalars = [
    row for row in scalar_records if bool(row.get("critical"))
]
critical_scalar_exact = sum(
    bool(row.get("exact_match")) for row in critical_scalars
)
character_errors = sum(
    int(row.get("character_errors", 0)) for row in scalar_records
)
reference_characters = sum(
    int(row.get("reference_characters", 0)) for row in scalar_records
)
word_errors = sum(
    int(row.get("word_errors", 0)) for row in scalar_records
)
reference_words = sum(
    int(row.get("reference_words", 0)) for row in scalar_records
)

item_exact = sum(
    bool(row.get("exact_match")) for row in item_field_records
)
predicted_rows = sum(
    int(record.get("predicted_item_count", 0))
    for record in verified_records
)
reference_rows = sum(
    int(record.get("reference_item_count", 0))
    for record in verified_records
)
predicted_item_fields = predicted_rows * 4
reference_item_fields = reference_rows * 4
item_precision = safe_ratio(item_exact, predicted_item_fields)
item_recall = safe_ratio(item_exact, reference_item_fields)

row_true_positives = 0
for record in verified_records:
    document_evaluation = load_json(Path(record["evaluation_path"]))
    row_true_positives += int(
        document_evaluation.get("metrics", {}).get(
            "row_true_positives",
            0,
        )
    )

recomputed_metrics = {
    "documents": len(verified_records),
    "templates": len(EXPECTED_TEMPLATES),
    "scalar_field_observations": scalar_total,
    "scalar_field_exact": scalar_exact,
    "all_scalar_exact_match": safe_ratio(scalar_exact, scalar_total),
    "critical_scalar_observations": len(critical_scalars),
    "critical_scalar_exact": critical_scalar_exact,
    "critical_scalar_exact_match": safe_ratio(
        critical_scalar_exact,
        len(critical_scalars),
    ),
    "scalar_micro_cer": safe_error_rate(
        character_errors,
        reference_characters,
    ),
    "scalar_micro_wer": safe_error_rate(word_errors, reference_words),
    "predicted_item_rows": predicted_rows,
    "reference_item_rows": reference_rows,
    "matched_item_rows": row_true_positives,
    "row_precision": safe_ratio(row_true_positives, predicted_rows),
    "row_recall": safe_ratio(row_true_positives, reference_rows),
    "row_f1": harmonic_mean(
        safe_ratio(row_true_positives, predicted_rows),
        safe_ratio(row_true_positives, reference_rows),
    ),
    "item_field_exact": item_exact,
    "predicted_item_fields": predicted_item_fields,
    "reference_item_fields": reference_item_fields,
    "line_item_field_precision": item_precision,
    "line_item_field_recall": item_recall,
    "line_item_field_f1": harmonic_mean(item_precision, item_recall),
    "item_count_exact_match": safe_ratio(
        sum(bool(row.get("item_count_exact")) for row in verified_records),
        len(verified_records),
    ),
    "financial_consistency": safe_ratio(
        sum(
            bool(row.get("financial_consistent"))
            for row in verified_records
        ),
        len(verified_records),
    ),
    "document_exact_match": safe_ratio(
        sum(
            bool(row.get("document_exact_match"))
            for row in verified_records
        ),
        len(verified_records),
    ),
}

metric_mismatches = [
    metric_name
    for metric_name, recomputed_value in recomputed_metrics.items()
    if metric_name not in overall_metrics
    or not close_enough(overall_metrics[metric_name], recomputed_value)
]

acceptance_map = {
    str(row.get("metric")): row
    for row in acceptance_checks
    if isinstance(row, dict) and row.get("metric")
}
acceptance_errors = []
for metric_name in sorted(EXPECTED_ACCEPTANCE_METRICS):
    check = acceptance_map.get(metric_name)
    if check is None:
        acceptance_errors.append(f"missing:{metric_name}")
        continue
    actual = recomputed_metrics[metric_name]
    minimum = float(check.get("minimum"))
    expected_status = "PASSED" if actual >= minimum else "FAILED"
    if not close_enough(check.get("actual"), actual):
        acceptance_errors.append(f"actual:{metric_name}")
    if check.get("status") != expected_status:
        acceptance_errors.append(f"status:{metric_name}")

recomputed_status = (
    "PASSED"
    if not acceptance_errors
    and all(
        recomputed_metrics[metric] >= float(acceptance_map[metric]["minimum"])
        for metric in EXPECTED_ACCEPTANCE_METRICS
    )
    else "FAILED"
)

mismatch_table = summary_tables.get("mismatch_details", pd.DataFrame())
document_table = summary_tables.get("document_summary", pd.DataFrame())
reported_mismatch_records = int(
    artifacts.get("mismatch_details", {}).get("records", -1)
)

summary_values = [
    ("summary_artifacts", 4, len(summary_tables)),
    ("summary_errors", 0, len(summary_errors)),
    ("overall_metrics_recomputed", [], metric_mismatches),
    ("acceptance_metrics", sorted(EXPECTED_ACCEPTANCE_METRICS), sorted(acceptance_map)),
    ("acceptance_errors", [], acceptance_errors),
    (
        "evaluation_status_recomputed",
        recomputed_status,
        evaluation_manifest.get("status"),
    ),
    (
        "document_summary_records",
        EXPECTED_DOCUMENTS,
        len(document_table),
    ),
    (
        "mismatch_records",
        reported_mismatch_records,
        len(mismatch_table),
    ),
    ("test_ground_truth_reopened", 0, 0),
]
summary_controls = [
    {
        "control": name,
        "expected": expected,
        "actual": actual,
        "status": "VALID" if expected == actual else "INVALID",
    }
    for name, expected, actual in summary_values
]

print("\nRECOMPUTED FINAL TEST RESULT CONTROLS")
display(pd.DataFrame(summary_controls))
if summary_errors:
    display(pd.DataFrame(summary_errors))

invalid_summary_controls = [
    row["control"]
    for row in summary_controls
    if row["status"] != "VALID"
]
if invalid_summary_controls:
    raise RuntimeError(
        "CELL 12E RESULT VERIFICATION FAILED. "
        f"Kontrol tidak valid: {invalid_summary_controls}."
    )


# ============================================================
# CREATE OR RECOVER THE IMMUTABLE FINAL TEST DECISION
# ============================================================

if recomputed_status == "PASSED":
    final_decision = "TEST_PASSED_BASELINE_ACCEPTED"
    deployment_recommendation = "ACCEPTED_FOR_RELEASE_CANDIDATE"
else:
    final_decision = "TEST_FAILED_BASELINE_REJECTED"
    deployment_recommendation = "REJECTED_FOR_RELEASE"

if FINAL_DECISION_MANIFEST_PATH.exists():
    existing_decision = load_json(FINAL_DECISION_MANIFEST_PATH)
    frozen_at_utc = existing_decision.get("frozen_at_utc")
    if not isinstance(frozen_at_utc, str) or not frozen_at_utc:
        raise RuntimeError("Final decision tidak memiliki frozen_at_utc.")
else:
    frozen_at_utc = datetime.now(timezone.utc).isoformat(
        timespec="seconds"
    )

test_result_fingerprint = {
    "evaluation_manifest_sha256": source_checksums_before[
        "evaluation_manifest"
    ],
    "evaluation_status": recomputed_status,
    "overall_metrics": recomputed_metrics,
    "acceptance_checks": acceptance_checks,
    "document_evaluations": [
        {
            "document_id": record["document_id"],
            "evaluation_sha256": record["evaluation_sha256"],
            "prediction_sha256": record["prediction_sha256"],
            "ground_truth_sha256": record["ground_truth_sha256"],
        }
        for record in sorted(
            verified_records,
            key=lambda row: str(row["document_id"]),
        )
    ],
}
test_result_sha256 = sha256_object(test_result_fingerprint)

final_decision_manifest = {
    "schema_version": "1.0.0",
    "cell_version": CELL_VERSION,
    "status": "FINAL_TEST_RESULT_FROZEN",
    "benchmark_status": "COMPLETED",
    "decision": final_decision,
    "deployment_recommendation": deployment_recommendation,
    "frozen_at_utc": frozen_at_utc,
    "baseline": {
        "baseline_id": EXPECTED_BASELINE_ID,
        "parser_id": EXPECTED_PARSER_ID,
        "parser_version": EXPECTED_PARSER_VERSION,
        "parser_signature_sha256": EXPECTED_PARSER_SIGNATURE,
        "parser_source_path": str(parser_source_path),
        "parser_source_sha256": source_checksums_before["parser_source"],
        "development_freeze_manifest_path": str(freeze_manifest_path),
        "development_freeze_manifest_sha256": source_checksums_before[
            "development_freeze"
        ],
        "validation_decision_manifest_path": str(
            validation_decision_path
        ),
        "validation_decision_manifest_sha256": source_checksums_before[
            "validation_decision"
        ],
    },
    "validation_evidence": {
        "decision": validation_decision.get("decision"),
        "validation_result_sha256": baseline_pointer.get(
            "validation_result_sha256"
        ),
    },
    "test_evidence": {
        "evaluation_manifest_path": str(EVALUATION_MANIFEST_PATH),
        "evaluation_manifest_sha256": source_checksums_before[
            "evaluation_manifest"
        ],
        "evaluation_status": recomputed_status,
        "documents": EXPECTED_DOCUMENTS,
        "templates": sorted(EXPECTED_TEMPLATES),
        "overall_metrics": recomputed_metrics,
        "acceptance_checks": acceptance_checks,
        "mismatch_records": reported_mismatch_records,
        "test_result_sha256": test_result_sha256,
    },
    "verified_artifacts": {
        "document_evaluations": len(verified_records),
        "prediction_files": len(prediction_map),
        "summary_files": len(summary_tables),
        "metrics_recomputed": True,
        "all_checksums_valid": True,
    },
    "immutable_inputs": {
        "test_baseline_pointer": {
            "path": str(TEST_BASELINE_POINTER_PATH),
            "sha256": source_checksums_before["test_baseline_pointer"],
        },
        "test_prediction_manifest": {
            "path": str(PREDICTION_MANIFEST_PATH),
            "sha256": source_checksums_before["prediction_manifest"],
        },
        "test_evaluation_manifest": {
            "path": str(EVALUATION_MANIFEST_PATH),
            "sha256": source_checksums_before["evaluation_manifest"],
        },
    },
    "final_policy": {
        "development_tuning_complete": True,
        "validation_evaluation_complete": True,
        "test_evaluation_complete": True,
        "test_result_is_final": True,
        "test_split_may_not_tune_parser_v1_0_2": True,
        "parser_changes_require_new_version": True,
        "new_version_requires_new_blind_evaluation_protocol": True,
        "next_allowed_stage": "BENCHMARK_COMPLETE",
        "next_allowed_split": None,
    },
    "integrity": {
        "test_ground_truth_reopened_in_this_cell": 0,
        "validation_ground_truth_reopened_in_this_cell": 0,
        "parser_modifications": 0,
        "prediction_modifications": 0,
        "evaluation_modifications": 0,
        "ground_truth_modifications": 0,
        "dataset_modifications": 0,
        "source_modifications": 0,
    },
    "next_stage": {
        "stage": "BENCHMARK_COMPLETE",
        "action": "ARCHIVE_AND_REPORT_FINAL_RESULT",
        "current_baseline_may_be_released": recomputed_status == "PASSED",
        "current_test_split_may_be_reused_for_tuning": False,
    },
}

decision_action = save_immutable_json(
    FINAL_DECISION_MANIFEST_PATH,
    final_decision_manifest,
)
decision_sha256 = sha256_file(FINAL_DECISION_MANIFEST_PATH)

benchmark_pointer = {
    "schema_version": "1.0.0",
    "status": "BENCHMARK_COMPLETE",
    "baseline_id": EXPECTED_BASELINE_ID,
    "parser_id": EXPECTED_PARSER_ID,
    "parser_version": EXPECTED_PARSER_VERSION,
    "parser_signature_sha256": EXPECTED_PARSER_SIGNATURE,
    "final_test_status": recomputed_status,
    "final_decision": final_decision,
    "deployment_recommendation": deployment_recommendation,
    "test_result_sha256": test_result_sha256,
    "final_decision_manifest_path": str(FINAL_DECISION_MANIFEST_PATH),
    "final_decision_manifest_sha256": decision_sha256,
    "completed_at_utc": frozen_at_utc,
    "next_allowed_stage": "BENCHMARK_COMPLETE",
    "next_allowed_split": None,
    "parser_mutation_allowed": False,
    "test_split_may_be_reused_for_tuning": False,
}

pointer_action = save_immutable_json(
    BENCHMARK_COMPLETION_POINTER_PATH,
    benchmark_pointer,
)


# ============================================================
# FINAL SOURCE IMMUTABILITY AND FREEZE CONTROLS
# ============================================================

source_checksums_after = {
    "test_baseline_pointer": sha256_file(TEST_BASELINE_POINTER_PATH),
    "prediction_manifest": sha256_file(PREDICTION_MANIFEST_PATH),
    "evaluation_manifest": sha256_file(EVALUATION_MANIFEST_PATH),
    "development_freeze": sha256_file(freeze_manifest_path),
    "validation_decision": sha256_file(validation_decision_path),
    "parser_source": sha256_file(parser_source_path),
}
for record in verified_records:
    document_id = str(record["document_id"])
    source_checksums_after[
        f"evaluation:{document_id}"
    ] = sha256_file(Path(record["evaluation_path"]))
    source_checksums_after[
        f"prediction:{document_id}"
    ] = sha256_file(Path(record["prediction_path"]))
for artifact_name in summary_tables:
    source_checksums_after[
        f"summary:{artifact_name}"
    ] = sha256_file(Path(artifacts[artifact_name]["path"]))

changed_sources = [
    name
    for name, checksum in source_checksums_before.items()
    if source_checksums_after.get(name) != checksum
]

recovered_decision = load_json(FINAL_DECISION_MANIFEST_PATH)
recovered_pointer = load_json(BENCHMARK_COMPLETION_POINTER_PATH)
final_values = [
    (
        "decision_status",
        "FINAL_TEST_RESULT_FROZEN",
        recovered_decision.get("status"),
    ),
    (
        "benchmark_status",
        "COMPLETED",
        recovered_decision.get("benchmark_status"),
    ),
    (
        "final_test_status",
        recomputed_status,
        recovered_pointer.get("final_test_status"),
    ),
    (
        "final_decision",
        final_decision,
        recovered_pointer.get("final_decision"),
    ),
    (
        "decision_checksum",
        decision_sha256,
        recovered_pointer.get("final_decision_manifest_sha256"),
    ),
    (
        "pointer_status",
        "BENCHMARK_COMPLETE",
        recovered_pointer.get("status"),
    ),
    (
        "next_allowed_stage",
        "BENCHMARK_COMPLETE",
        recovered_pointer.get("next_allowed_stage"),
    ),
    (
        "test_split_reusable_for_tuning",
        False,
        recovered_pointer.get("test_split_may_be_reused_for_tuning"),
    ),
    ("source_modifications", 0, len(changed_sources)),
    ("test_ground_truth_reopened", 0, 0),
    ("validation_ground_truth_reopened", 0, 0),
]
final_controls = [
    {
        "control": name,
        "expected": expected,
        "actual": actual,
        "status": "VALID" if expected == actual else "INVALID",
    }
    for name, expected, actual in final_values
]

print("\nFINAL FREEZE AND COMPLETION CONTROLS")
display(pd.DataFrame(final_controls))

invalid_final_controls = [
    row["control"]
    for row in final_controls
    if row["status"] != "VALID"
]
if invalid_final_controls:
    raise RuntimeError(
        "CELL 12E FINAL GATE FAILED. "
        f"Kontrol tidak valid: {invalid_final_controls}."
    )


print()
print(f"Cell version              : {CELL_VERSION}")
print(f"Baseline ID               : {EXPECTED_BASELINE_ID}")
print(f"Parser version            : {EXPECTED_PARSER_VERSION}")
print(f"Final test status         : {recomputed_status}")
print(f"Final decision            : {final_decision}")
print(f"Deployment recommendation : {deployment_recommendation}")
print(f"Documents evaluated       : {len(verified_records)}")
print(
    "All scalar exact match  : "
    f"{recomputed_metrics['all_scalar_exact_match']:.6f}"
)
print(
    "Critical scalar exact   : "
    f"{recomputed_metrics['critical_scalar_exact_match']:.6f}"
)
print(
    "Line-item field F1      : "
    f"{recomputed_metrics['line_item_field_f1']:.6f}"
)
print(
    "Financial consistency   : "
    f"{recomputed_metrics['financial_consistency']:.6f}"
)
print(
    "Document exact match    : "
    f"{recomputed_metrics['document_exact_match']:.6f}"
)
print(f"Mismatch records          : {reported_mismatch_records}")
print(f"Test result SHA-256       : {test_result_sha256}")
print(f"Decision action           : {decision_action}")
print(f"Decision manifest         : {FINAL_DECISION_MANIFEST_PATH}")
print(f"Decision SHA-256          : {decision_sha256}")
print(f"Pointer action            : {pointer_action}")
print(f"Completion pointer        : {BENCHMARK_COMPLETION_POINTER_PATH}")
print(
    "Pointer SHA-256         : "
    f"{sha256_file(BENCHMARK_COMPLETION_POINTER_PATH)}"
)
print("Test GT reopened          : 0")
print("Validation GT reopened    : 0")
print("Parser modifications      : 0")
print("Prediction modifications  : 0")
print("Evaluation modifications  : 0")
print("Ground-truth modifications: 0")
print("Dataset modifications     : 0")
print("Source modifications      : 0")
print()
print(
    "✅ CELL 12E PASSED — hasil test final telah dibekukan secara "
    f"immutable dengan keputusan {final_decision}. Benchmark v1.0.2 "
    "selesai; ground truth tidak dibuka kembali dan test split tidak "
    "boleh digunakan untuk tuning."
)


CELL 12E — FINAL TEST DECISION FREEZE — VERSION 1.0.0
Decision root: /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/field_extraction/frozen_test_decisions/rule_based_baseline_v1_0_2
Test result: FAILED | Test GT reopen: 0 | Benchmark completion: PENDING
FINAL TEST DECISION PREFLIGHT


,control,expected,actual,status
0,baseline_pointer_status,ACTIVE_FOR_TEST_PREFLIGHT,ACTIVE_FOR_TEST_PREFLIGHT,VALID
1,baseline_id,RULE-BASED-INVOICE-PARSER-V1@1.0.2,RULE-BASED-INVOICE-PARSER-V1@1.0.2,VALID
2,parser_id,RULE-BASED-INVOICE-PARSER-V1,RULE-BASED-INVOICE-PARSER-V1,VALID
3,parser_version,1.0.2,1.0.2,VALID
4,parser_signature,ffe04f258fb3b1a6a55661698fa6c7dda29ddd7b714c2f...,ffe04f258fb3b1a6a55661698fa6c7dda29ddd7b714c2f...,VALID
5,parser_source_checksum,0a737f57a86df7d3e16eef6749e3c3a23e82fb51227d09...,0a737f57a86df7d3e16eef6749e3c3a23e82fb51227d09...,VALID
6,development_freeze_checksum,9f234c5c8cdc10c277e9c26a1f0831d1e08f3dc6fb9503...,9f234c5c8cdc10c277e9c26a1f0831d1e08f3dc6fb9503...,VALID
7,validation_decision_checksum,831fb82a372aaed44789afbc66386bf6ebbf073868a9c0...,831fb82a372aaed44789afbc66386bf6ebbf073868a9c0...,VALID
8,validation_decision_status,FROZEN_FOR_TEST_PREFLIGHT,FROZEN_FOR_TEST_PREFLIGHT,VALID
9,validation_decision,VALIDATION_PASSED_BASELINE_ACCEPTED,VALIDATION_PASSED_BASELINE_ACCEPTED,VALID



IMMUTABLE TEST ARTIFACT CONTROLS


,control,expected,actual,status
0,evaluation_records,40,40,VALID
1,verified_evaluations,40,40,VALID
2,unique_evaluation_ids,40,40,VALID
3,evaluation_templates,"[TPL-09, TPL-10]","[TPL-09, TPL-10]",VALID
4,prediction_records,40,40,VALID
5,artifact_errors,0,0,VALID
6,test_ground_truth_reopened,0,0,VALID
7,validation_ground_truth_reopened,0,0,VALID



RECOMPUTED FINAL TEST RESULT CONTROLS


,control,expected,actual,status
0,summary_artifacts,4,4,VALID
1,summary_errors,0,0,VALID
2,overall_metrics_recomputed,[],[],VALID
3,acceptance_metrics,"[all_scalar_exact_match, critical_scalar_exact...","[all_scalar_exact_match, critical_scalar_exact...",VALID
4,acceptance_errors,[],[],VALID
5,evaluation_status_recomputed,FAILED,FAILED,VALID
6,document_summary_records,40,40,VALID
7,mismatch_records,40,40,VALID
8,test_ground_truth_reopened,0,0,VALID



FINAL FREEZE AND COMPLETION CONTROLS


,control,expected,actual,status
0,decision_status,FINAL_TEST_RESULT_FROZEN,FINAL_TEST_RESULT_FROZEN,VALID
1,benchmark_status,COMPLETED,COMPLETED,VALID
2,final_test_status,FAILED,FAILED,VALID
3,final_decision,TEST_FAILED_BASELINE_REJECTED,TEST_FAILED_BASELINE_REJECTED,VALID
4,decision_checksum,56241ba4c07328dfd53a996cf57c8fe9d3b7884a468ecf...,56241ba4c07328dfd53a996cf57c8fe9d3b7884a468ecf...,VALID
5,pointer_status,BENCHMARK_COMPLETE,BENCHMARK_COMPLETE,VALID
6,next_allowed_stage,BENCHMARK_COMPLETE,BENCHMARK_COMPLETE,VALID
7,test_split_reusable_for_tuning,False,False,VALID
8,source_modifications,0,0,VALID
9,test_ground_truth_reopened,0,0,VALID



Cell version              : 1.0.0
Baseline ID               : RULE-BASED-INVOICE-PARSER-V1@1.0.2
Parser version            : 1.0.2
Final test status         : FAILED
Final decision            : TEST_FAILED_BASELINE_REJECTED
Deployment recommendation : REJECTED_FOR_RELEASE
Documents evaluated       : 40
All scalar exact match  : 0.916667
Critical scalar exact   : 0.900000
Line-item field F1      : 1.000000
Financial consistency   : 1.000000
Document exact match    : 0.500000
Mismatch records          : 40
Test result SHA-256       : e1cee35499c1324d592e4aa332b366b61676dc3ef1b5acb7e19d574d9fc02041
Decision action           : CREATED
Decision manifest         : /content/drive/MyDrive/InvoiceFlow-AI-Data/interim/synthetic_v1_build/20260904T150025Z/ocr_benchmark/field_extraction/frozen_test_decisions/rule_based_baseline_v1_0_2/final_test_decision_manifest.json
Decision SHA-256          : 56241ba4c07328dfd53a996cf57c8fe9d3b7884a468ecff79c2644da6c7799b9
Pointer action            : CREATED
Co

In [12]:
from datetime import datetime
from pathlib import Path
import os

DRIVE_ROOT = Path("/content/drive/MyDrive")

if not DRIVE_ROOT.is_dir():
    raise FileNotFoundError(
        "Google Drive belum terpasang. Mount Drive terlebih dahulu."
    )

notebooks = []

for directory, subdirectories, filenames in os.walk(DRIVE_ROOT):
    # Dataset tidak perlu dipindai.
    subdirectories[:] = [
        name
        for name in subdirectories
        if name not in {
            "InvoiceFlow-AI-Data",
            ".Trash",
        }
    ]

    for filename in filenames:
        if not filename.casefold().endswith(".ipynb"):
            continue

        path = Path(directory) / filename
        try:
            modified = path.stat().st_mtime
            notebooks.append((modified, path))
        except OSError:
            continue

notebooks.sort(reverse=True)

print("NOTEBOOK TERBARU DI GOOGLE DRIVE\n")

for number, (modified, path) in enumerate(notebooks[:15], start=1):
    timestamp = datetime.fromtimestamp(modified).strftime(
        "%Y-%m-%d %H:%M:%S"
    )
    print(f"{number:02d}. {timestamp}")
    print(f"    {path}")

print()
print(f"Total notebook ditemukan: {len(notebooks)}")
print("✅ Pencarian selesai—belum ada file yang dipindahkan.")

NOTEBOOK TERBARU DI GOOGLE DRIVE

01. 2026-09-06 16:10:43
    /content/drive/MyDrive/Colab Notebooks/02_ocr_and_preprocessing.ipynb
02. 2026-09-05 23:52:15
    /content/drive/MyDrive/Colab Notebooks/12131.ipynb
03. 2026-09-05 10:06:01
    /content/drive/MyDrive/Colab Notebooks/01_data_collection_and_audit.ipynb
04. 2026-09-04 20:06:50
    /content/drive/MyDrive/Colab Notebooks/Untitled5.ipynb
05. 2026-09-04 14:36:28
    /content/drive/MyDrive/Colab Notebooks/00_environment_setup.ipynb
06. 2026-09-04 14:18:18
    /content/drive/MyDrive/Colab Notebooks/0121.ipynb
07. 2026-08-26 07:32:09
    /content/drive/MyDrive/Colab Notebooks/GPTAPI.ipynb
08. 2026-08-26 07:28:14
    /content/drive/MyDrive/Colab Notebooks/GptApi_tes.ipynb
09. 2026-05-15 18:18:03
    /content/drive/MyDrive/Colab Notebooks/training_camelbert.ipynb
10. 2026-05-15 17:10:51
    /content/drive/MyDrive/Colab Notebooks/training_camelbert (1).ipynb
11. 2026-05-15 16:51:14
    /content/drive/MyDrive/Colab Notebooks/Salinan dari 